In [2]:
### Creating preliminary functions and dictionaries + package loading 
cell1_start_time = time(); start_load_pkgs = time()
    using Pkg;  
    Pkg.add("Plots"); using Plots;  
    Pkg.add("JSON3"); using JSON3;
    Pkg.add("Dates"); using Dates;
    Pkg.add("JLD2"); using JLD2
    Pkg.add("HypothesisTests"); using HypothesisTests
    Pkg.add("DataFrames"); using DataFrames
    Pkg.add("CSV"); using CSV
    Pkg.add("FASTX"); using FASTX
    Pkg.add("Combinatorics"); using Combinatorics
    Pkg.add("StatsBase"); using StatsBase
    using Statistics
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
println(pwd()); cd("/Users/ryhisner"); println(pwd())
#    Pkg.add("LaTeXStrings");  using LaTeXStrings
#    Pkg.add("PyPlot"); using PyPlot
#    Pkg.add("PlotlyJS"); using PlotlyJS
#    Pkg.add("PGFPlotsX"); using PGFPlotsX
#    Pkg.add("UnicodePlots"); using UnicodePlots
#    Pkg.add("InspectDR"); using InspectDR
#    Pkg.add("GLMakie"); using GLMakie
#    Pkg.add("CairoMakie"); using CairoMakie
#    Pkg.add("WGLMakie"); using WGLMakie
#    Pkg.add("GMT"); using GMT
#####################################################################################################################################
### Turns time in seconds to hours, minutes, & seconds
function seconds_to_hrs_min_sec(t)
    hours = 0
    minutes = 0
    seconds = 0
    hours = t÷3600
    minutes = (t%3600)÷60
    if t > 0.0001
        seconds = (t%3600)%60
    end
    hours_int = Int(hours)
    minutes_int = Int(minutes)
    minutes_str = split(string(minutes_int), ".")[1]
    hours_fin = split(string(hours_int), ".")[1]
    minutes_fin = ""
    minutes_fin = lpad(minutes_str, 2, "0")
    seconds_rd = round(digits=2, seconds)
    seconds_1 = string(split(string(seconds_rd), ".")[1])
    seconds_2 = string(split(string(seconds_rd), ".")[2])
    seconds_left = lpad(seconds_1, 2, "0")
    seconds_right = rpad(seconds_2, 2, "0")
    seconds_fin = "$(seconds_left).$(seconds_right)"
    final_time = "$hours_int:$minutes_fin:$seconds_fin"
    final_time2 = "$hours_int hr, $minutes_int min, $seconds_fin sec"
    return final_time, final_time2
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
######################################################################################################################################
### Adds zero to truncated digit so all numbers have same # of digits & line up nicely E.g. if number = 9.1 & digits_rd = 3, it returns 9.100
function add_zeros_to_rounded(number::Float64, digits_rd::Int)
    if number ≥ 1
        num_final = 0.0
        num_whole = number*10^digits_rd
        num_whole_str = split(string(num_whole), ".")[1]
        len = length(num_whole_str)
        if len ≤ 2
            num_final = "."*"0"^(digits_rd-len)*num_whole_str
        else
            num_final = num_whole_str[1:end-digits_rd]*"."*num_whole_str[end-digits_rd+1:end]
        end
    return num_final
    end
end
#####################################################################################################################################
## Makes all digits go to 2 decimal places so they line up nicely
function number_to_string_to_two_decimal_places(number::Float64, decimals::Int)  
    num_str = string(number)
    before_dec = split(num_str, ".")[1]
    before_dec_str = string(before_dec)
    after_dec = split(num_str, ".")[2]
    after_dec_str = ""
    if length(after_dec) ≥ decimals
        after_dec_str = after_dec[1:decimals]
    else
        after_dec_len = length(after_dec)
        zero_array = Vector{String}()
        missing_zeros_to_fill = decimals - after_dec_len
        for i in 1:missing_zeros_to_fill
            push!(zero_array, "0")
        end
        zeros_str = join(zero_array)
        after_dec_str = after_dec[1:after_dec_len]*zeros_str
    end
    final_num_string = before_dec_str*"."*after_dec_str
    return final_num_string
end
#####################################################################################################################################
load_pkgs_runtime = time() - start_load_pkgs
load_pkgs_hms1, load_pkgs_hms2 = seconds_to_hrs_min_sec(load_pkgs_runtime)
println("Time to Load Packages = ", load_pkgs_hms1); println("Time to Load Packages = ", load_pkgs_hms2)
#####################################################################################################################################
hydrophobic_index_dict = Dict{String, Int}("L"=>97, "I"=>99, "F"=>100, "W"=>97, "V"=>76, "M"=>74, "C"=>49, "Y"=>63, "A"=>41, "T"=>13, "E"=>-31, "H"=>8, "G"=>0, "S"=>-5, "Q"=>-10, "D"=>-55, "R"=>-14, "K"=>-23, "N"=>-28, "P"=>5, "*"=>0)
#####################################################################################################################################
wuhan_ref_seq = "..."
ref_seq = wuhan_ref_seq
#####################################################################################################################################
######################################################################################################################################
clade_set_complete = Set(["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"])
clade_arr_complete = ["recombinant", "19A", "19B", "20A", "20B", "20C", "20D", "20E", "20F", "20G", "20H", "20I", "20J", "21A", "21B", "21C", "21D", "21E", "21F", "21G", "21H", "21I", "21J", "21K", "21L", "21M", "22A", "22B", "22C", "22D", "22E", "22F", "23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H", "23I", "24A", "24B", "24C", "24D", "24E", "24F", "24G", "24H", "24I", "25A", "25B", "25C", "25D", "25E", "25F", "25G", "25H", "25I"]
clade_to_pango = Dict("recombinant"=>"recombinant", "19A"=>"B", "19B"=>"A", "20A"=>"B.1", "20B"=>"B.1.1", "20C"=>"B.1", "20D"=>"B.1.1.1", "20E"=>"B.1.177", "20F"=>"D.2", "20G"=>"B.1.2", "20H"=>"B.1.351", "20I"=>"B.1.1.7", "20J"=>"P.1", "21A"=>"B.1.617.2", "21B"=>"B.1.617.1", "21C"=>"B.1.427", "21D"=>"B.1.525", "21E"=>"P.3", "21F"=>"B.1.526", "21G"=>"C.37", "21H"=>"B.1.621", "21I"=>"B.1.617.2", "21J"=>"B.1.617.2", "21K"=>"BA.1", "21L"=>"BA.2", "21M"=>"BA.3", "22A"=>"BA.4", "22B"=>"BA.5", "22C"=>"BA.2.12.1", "22D"=>"BA.2.75", "22E"=>"BQ.1", "22F"=>"XBB", "23A"=>"XBB.1.5", "23B"=>"XBB.1.16", "23C"=>"CH.1.1", "23D"=>"XBB.1.9", "23E"=>"XBB.2.3", "23F"=>"EG.5.1", "23G"=>"XBB.1.5.70", "23H"=>"HK.3", "23I"=>"BA.2.86", "24A"=>"JN.1", "24B"=>"JN.1.11.1", "24C"=>"KP.3", "24D"=>"XDV.1", "24E"=>"KP.3.1.1", "24F"=>"XEC", "24G"=>"KP.2.3", "24H"=>"LF.7", "24I"=>"MV.1", "25A"=>"LP.8.1", "25B"=>"NB.1.8.1", "25C"=>"XFG", "25D"=>"MC.10.2.1", "25E"=>"PY.1", "25F"=>"NW.1.2", "25G"=>"XFC", "25H"=>"XFJ", "25I"=>"BA.3.2")
######################################################################################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
mut_gene(n) = split(string(n), ":")[1]
######################################################################################################################################
######################################################################################################################################
AAsub_gene(n) = aa_gene_comprehensive_dict[n]
AAsub_gene_num(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
AAsub_gene_num_pos_only(n) = [aa_gene_comprehensive_dict[n], aa_pos_comprehensive_dict[n]]
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
#################################
AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
AA_gene_pos_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_gene_pos_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
######################################################################################################################################
function AA_order_key(mut)
    gene = aa_gene_comprehensive_dict[mut]
    AApos = aa_pos_comprehensive_dict[mut]
    gene_pos = gene_print_order[gene]
    return (gene_pos, AApos)
end
#####################################################################################################################################
function pango_variant_sort_key(pango::String)
    dotparts = split(pango, ".")
    k1 = string(dotparts[1])
    k2 = 0
    k3 = 0
    k4 = 0
    if length(dotparts) ≥ 2
        k2 = parse(Int, string(dotparts[2]))
    end
    if length(dotparts) ≥ 3
        k3 = parse(Int, string(dotparts[3]))
    end
    if length(dotparts) ≥ 4
        k4 = parse(Int, string(dotparts[4]))
    end
    return (k1, k2, k3, k4)
end
######################################################################################################################################
gene_print_order = Dict{String, Int}("S"=>1, "N"=>2, "E"=>3, "M"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10, "ORF1a"=>12, "ORF1b"=>13)
######################################################################################################################################
function pango_minus_X_fx(pango::String, minus::Int)
    unaliased = pango_to_pango_unaliased_v2[pango]
    dot_ct = count(".", unaliased)
    println(dot_ct)
    if dot_ct ≥ minus
        dotsplits = split(unaliased, ".")
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(pango), $(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X_pango
    else
        return pango
    end
end
####################################################################################
function pango_unaliased_minus_X_fx(unaliased::String, minus::Int)
    dot_ct = count(".", unaliased)
    if dot_ct ≥ minus 
        dotsplits = split(unaliased, ".")
        println(dotsplits)
        minus_X_unaliased = join(dotsplits[1:dot_ct+1-minus], ".")
        minus_X_pango = pango_unaliased_to_pango[minus_X_unaliased]
        println("$(unaliased), minus-$(minus) = $(minus_X_unaliased)")
        return minus_X
    else
        return minus_X_pango
    end
end
##########################################################################################################################################################################
function read_fasta(filepath::String)
    reader = FASTX.FASTA.Reader(open(filepath, "r"))
    fasta_in = [record for record in reader]
    close(reader)
    return[String(FASTX.FASTX.description(rec)) for rec in fasta_in],
    [uppercase(String(FASTX.FASTA.sequence(rec))) for rec in fasta_in]
end
##########################################################################################################################################################################
wuhan_ref_seq = "..."
refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
#ref_AA_dict = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_order_dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "ORF3a"=>4, "E"=>4, "M"=>6, "ORF6"=>7, "ORF7a"=>8, "ORF7b"=>9, "ORF8"=>10, "N"=>11, "ORF9b"=>12)
gene_AA_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
gene_AApos_sortKey(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
######################################################################################################################################
gene_AA_dict = Dict{String, String}()
gene_AA_dict["ORF1a"] = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV*"
gene_AA_dict["ORF1b"] = "RVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
gene_AA_dict["S"] = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
gene_AA_dict["E"] = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
gene_AA_dict["M"] = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
gene_AA_dict["N"] = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
gene_AA_dict["ORF3a"] = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
gene_AA_dict["ORF6"] = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
gene_AA_dict["ORF7a"] = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
gene_AA_dict["ORF7b"] = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
gene_AA_dict["ORF8"] = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
gene_AA_dict["ORF9b"] = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
######################################################################################################################################
NSP_AA_size = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP11"=>0, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>299)                                                                # "NSP12"=>BitSet(1:923), 
NSP_ranges = Dict{String, String}("NSP1"=>"ORF1a:1-180", "NSP2"=>"ORF1a:181-818", "NSP3"=>"ORF1a:819-2763", "NSP4"=>"ORF1a:2764-3263", "NSP5"=>"ORF1a:3264-3569", "NSP6"=>"ORF1a:3570-3859", "NSP7"=>"ORF1a:3860-3942", "NSP8"=>"ORF1a:3943-4140", "NSP9"=>"ORF1a:4141-4253", "NSP10"=>"ORF1a:4254-4392", "NSP11"=>"", "NSP12"=>"1a:4393-1b:923", "NSP13"=>"ORF1b:924-1524", "NSP14"=>"ORF1b:1525-2051", "NSP15"=>"ORF1b:2052-2397", "NSP16"=>"ORF1b:2398-2696", "S"=>"S:1-1274", "ORF3a"=>"ORF3a:1-276", "E"=>"E:1-76", "M"=>"M:1-223", "ORF6"=>"ORF6:1-62", "ORF7a"=>"ORF7a:1-122", "ORF7b"=>"ORF7b:1-44", "ORF8"=>"ORF8:1-122", "N"=>"N:1-420", "ORF9b"=>"ORF9b:1-98")
NSP_ranges_num_only = Dict{String, BitSet}("NSP1"=>BitSet(1:180), "NSP2"=>BitSet(181:818), "NSP3"=>BitSet(819:2763), "NSP4"=>BitSet(2764:3263), "NSP5"=>BitSet(3264:3569), "NSP6"=>BitSet(3570:3859), "NSP7"=>BitSet(3860:3942), "NSP8"=>BitSet(3943:4140), "NSP9"=>BitSet(4141:4253), "NSP10"=>BitSet(4254:4392), "NSP11"=>BitSet(), "NSP12"=>BitSet([4393:4401; 1:923]), "NSP13"=>BitSet(924:1524), "NSP14"=>BitSet(1525:2051), "NSP15"=>BitSet(2052:2397), "NSP16"=>BitSet(2398:2696), "S"=>BitSet(1:1274), "ORF3a"=>BitSet(1:276), "E"=>BitSet(1:76), "M"=>BitSet(1:223), "ORF6"=>BitSet(1:62), "ORF7a"=>BitSet(1:122), "ORF7b"=>BitSet(1:44), "ORF8"=>BitSet(1:122), "N"=>BitSet(1:420), "ORF9b"=>BitSet(1:98))
NSP_ranges1a = Dict{Int, BitSet}(1=>BitSet(1:180), 2=>BitSet(181:818), 3=>BitSet(819:2763), 4=>BitSet(2764:3263), 5=>BitSet(3264:3569), 6=>BitSet(3570:3859), 7=>BitSet(3860:3942), 8=>BitSet(3943:4140), 9=>BitSet(4141:4253), 10=>BitSet(4254:4392), 12=>BitSet(4393:4401))
NSP_ranges1b = Dict{Int, BitSet}(12=>BitSet(1:923), 13=>BitSet(924:1524), 14=>BitSet(1525:2051), 15=>BitSet(2052:2397), 16=>BitSet(2398:2696))
NSP1a_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4253, 11=>0, 12=>4392)
NSP1b_add = Dict{Int, Int}(12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
NSP1ab_add = Dict{Int, Int}(1=>0, 2=>180, 3=>818, 4=>2763, 5=>3263, 6=>3569, 7=>3859, 8=>3942, 9=>4140, 10=>4353, 11=>0, 12=>-9, 13=>923, 14=>1524, 15=>2051, 16=>2397)
######################################################################################################################################
ORF_size_dict = Dict{String, Int}()
for (orf, aaseq) in gene_AA_dict
    orf_len = length(aaseq)
    ORF_size_dict[orf] = orf_len
end
###########################################################################################################################################################################
###########################################################################################################################################################################
AA_residues = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*", "X"])
aa_pos_comprehensive_dict = Dict{String, Int}()
aa_gene_comprehensive_dict = Dict{String, String}()
aa_gene_and_pos_comprehensive_dict = Dict{String, String}()
refAA_comprehensive_dict = Dict{String, String}()
qryAA_comprehensive_dict = Dict{String, String}()
global nonspike_muts = Set{String}()
for (orf, orf_len) in ORF_size_dict
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:orf_len
                orf_mut = "$(orf):$(aa1)$(i)$(aa2)"
                orf_pos_mut = "$(orf):$(i)"
                aa_pos_comprehensive_dict[orf_mut] = i
                aa_pos_comprehensive_dict[orf_pos_mut] = i
                aa_gene_comprehensive_dict[orf_mut] = orf
                aa_gene_comprehensive_dict[orf_pos_mut] = orf
                aa_gene_and_pos_comprehensive_dict[orf_mut] = orf_pos_mut
                aa_gene_and_pos_comprehensive_dict[orf_pos_mut] = orf_pos_mut
                if orf ≠ "S"
                    push!(nonspike_muts, orf_mut)
                    push!(nonspike_muts, orf_pos_mut)
                end
                refAA_comprehensive_dict[orf_mut] = aa1
                qryAA_comprehensive_dict[orf_mut] = aa2
            end
        end
    end
end
aa_gene_comprehensive_dict["NTD_disulfide"] = "S"
aa_pos_comprehensive_dict["NTD_disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD_disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD_disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD_disulfide"] = "NTD_disulfide"
aa_gene_comprehensive_dict["NTD:disulfide"] = "S"
aa_pos_comprehensive_dict["NTD:disulfide"] = 1
aa_gene_and_pos_comprehensive_dict["NTD:disulfide"] = "S:NTD_disulfide"
refAA_comprehensive_dict["NTD:disulfide"] = "NTDdisulfide"
qryAA_comprehensive_dict["NTD:disulfide"] = "NTD_disulfide"
######################################################## Below: 100% comprehensive ref_nuc & qry_nuc dicts #################################################################
ref_nuc_comprehensive_dict = Dict{String, String}()
qry_nuc_comprehensive_dict = Dict{String, String}()
nuc_mut_int_comprehensive_dict = Dict{String, Int}()
nuc_mut_int_string_comprehensive_dict = Dict{String, String}()
nuc_residues1 = ["T", "C", "A", "G"]
nuc_residues2 = ["T", "C", "A", "G", "Y", "R", "K", "W", "M", "S", "-", "N"]
for nres1 in nuc_residues1
    for nres2 in nuc_residues2
        for i in 1:30000
            mut = "$(nres1)$(i)$(nres2)"
            nucmpos = i
            ref_nuc_comprehensive_dict[mut] = nres1
            qry_nuc_comprehensive_dict[mut] = nres2
            nuc_mut_int_comprehensive_dict[mut] = i
            nuc_mut_int_string_comprehensive_dict[mut] = string(i)
        end
    end
end
##########################################################################################################################################################################
##########################################################################################################################################################################
NSP_muts_pos_dict = Dict{String, Int}()
NSP_muts_gene_dict = Dict{String, String}()
NSP_ref_AA_dict = Dict{String, String}()
NSP_qry_AA_dict = Dict{String, String}()
NSP_set = Set(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"])
for nsp in NSP_set
    nsp_len = NSP_AA_size[nsp]
    for aa1 in AA_residues
        for aa2 in AA_residues
            for i in 1:nsp_len
                nspmut = "$(nsp):$(aa1)$(i)$(aa2)"
                nsppos = "$(nsp):$(i)"
                NSP_muts_gene_dict[nspmut] = nsp
                NSP_muts_pos_dict[nspmut] = i
                NSP_muts_gene_dict[nsppos] = nsp
                NSP_muts_pos_dict[nsppos] = i
                NSP_ref_AA_dict[nspmut] = aa1
                NSP_qry_AA_dict[nspmut] = aa2  
            end
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function AAmut_to_AApos(m)
    gn = split(m, ":")[1]
    mt = split(m, ":")[2]
    pos = mt[2:end-1]
    AApos = gn*":"*pos
    return AApos
end
########################################################################################################################################################################
########################################################################################################################################################################
########################################################################################################################################################################
function mixed2nuc(mix_mut)
    mut = mix_mut
    qry_nuc = qry_nuc_comprehensive_dict[mut]
    ref_nuc = ref_nuc_comprehensive_dict[mut]
    nuc_int = nuc_mut_int_comprehensive_dict[mut]
    ref_n_pos = "$(ref_nuc)$(nuc_int)"
    if length(mix_mut) ≥ 4
        if qry_nuc == "T"
            mut = mix_mut
        elseif qry_nuc == "C"
            mut = mix_mut
        elseif qry_nuc == "A"
            mut = mix_mut
        elseif qry_nuc == "G"
            mut = mix_mut
        elseif qry_nuc == "N"
            mut = mix_mut
        end   
        if ref_nuc == "T"
            if qry_nuc == "Y"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "C"
            if qry_nuc == "Y"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "R"
                mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "A"
            if qry_nuc == "R"
                mut = ref_n_pos*"G"
            elseif qry_nuc == "W"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "M"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "K"
                mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qry_nuc == "S"
                mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if ref_nuc == "G"
            if qry_nuc == "R"
                mut = ref_n_pos*"A"
            elseif qry_nuc == "K"
                mut = ref_n_pos*"T"
            elseif qry_nuc == "S"
                mut = ref_n_pos*"C"
            elseif qry_nuc == "Y"
                mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qry_nuc == "W"
                mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qry_nuc == "M"
                mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return mut
end
#####################################################################################################################################
####################### Making gene AA & nuc references for all designated variants #################################################
#####################################################################################################################################
gene_AA_pango_dict = Dict{String, Dict{String, String}}()
nuc_genome_pango_dict = Dict{String, String}()
pango_consensus_set = Set{String}()
headers1a, seqs1a = read_fasta("___pango_consensus_sequences/pango_consensus_AA_ORF1a_2025_06_25_NNL.fasta")
for i in 1:length(headers1a)
    pango = headers1a[i]
    push!(pango_consensus_set, pango)
end
for pango in pango_consensus_set
    gene_AA_pango_dict[pango] = Dict{String, String}()
end
################################################################################################
for gene in gene_array
    aa_file = "___pango_consensus_sequences/pango_consensus_AA_$(gene)_2025_06_25_NNL.fasta"
    headers, seqs = read_fasta(aa_file)
    for i in 1:length(headers)
        pango = headers[i]
        aa_seq = seqs[i]
        gene_AA_pango_dict[pango][gene] = aa_seq
    end
end
nuc_file = "___pango_consensus_sequences/pango_consensus_sequences_genome_nuc_2025_06_25_NNL.fasta"
nuc_headers, nuc_seqs = read_fasta(nuc_file)
for i in 1:length(nuc_headers)
    pango = nuc_headers[i]
    nuc_seq = nuc_seqs[i]
    if length(nuc_seq) ≥ 28000
        nuc_genome_pango_dict[pango] = nuc_seq
    end
end
seqs_in_nuc_genome_pango_dict = length(nuc_genome_pango_dict)
println("seqs_in_nuc_genome_pango_dict = $(seqs_in_nuc_genome_pango_dict)")
######################################################################################################################################
######################################################################################################################################
gene_hydrophobic_dict = Dict{String, Float64}()
for (gene, aaseq) in gene_AA_dict
    gene_hydrophobe_sum = 0
    for aa in aaseq
        hydrophobe_score = hydrophobic_index_dict[string(aa)]
        gene_hydrophobe_sum += hydrophobe_score
    end
    gene_hydrophobe_score = gene_hydrophobe_sum/length(aaseq)
    gene_hydrophobic_dict[gene] = gene_hydrophobe_score
end 
######################################################################################################################################
AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-", "*"])
AA_res_set_noDel = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "*"])
AA_res_pairs = Set{Tuple{String, String}}()
for res1 in AA_res_set_noDel
    for res2 in AA_res_set
        push!(AA_res_pairs, (res1, res2) )
    end
end
###########################################################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
############################################################################################################################################################################
############################################################################################################################################################################
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int, Int, Int}}()
tuple_to_index = Dict{Tuple{Int, Int, Int}, Int}()        
###########################################################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
############################################################################################################################################################################
############################################################################################################################################################################
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{String, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{String, Int}()
        end
    end
    date1_to_date2_ct = Dict{String, Int}()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function nuc_mut_pos_only_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict{Int, Dict{Int, Int}})
    for i in 1:3000
        if !haskey(all_dict, i)
            all_dict[i] = Dict{Int, Int}()
        end
    end
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, mut, 0) + count
        end
    end
    return date1_to_date2_ct
end
############################################################################################################################################################################
############################################################################################################################################################################
function ORF1abMut_to_NSP(mut::String)
    NSPmut = ""
#    NSP_muts = Dict("NSP$(i)" => Dict{String, Int}() for i in 1:16 if i ≠ 11)
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    refAA = refAA_comprehensive_dict[mut]
    qryAA = qryAA_comprehensive_dict[mut]
    if gene == "ORF1a"
        for (NSP, range) in NSP_ranges1a
            if pos in range
                NSPpos = pos - NSP1a_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    if gene == "ORF1b"
        for (NSP, range) in NSP_ranges1b
            if pos in range
                NSPpos = pos - NSP1b_add[NSP]
                NSPmut = "NSP$(NSP):$(refAA)$(NSPpos)$(qryAA)"
            end
        end
    end
    return NSPmut
end
#####################################################################################################################################
function NSPmut_to_ORF1ab(NSPmut::String)
    ORFmut = ""
    ORFpos = ""
    NSP_num = parse(Int, split(NSPmut, ":")[1][4:end])
    NSP_pos = NSP_muts_pos_dict[NSPmut]
    refAA = NSP_ref_AA_dict[NSPmut]
    qryAA = NSP_qry_AA_dict[NSPmut]
    if NSPnum in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
        ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
        ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSPnum in [13, 14, 15, 16]
        ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
        ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
    end
    if NSP_num == 12
        if NSP_pos in [1:8...]
            ORF1_pos = NSP1a_add[NSP_num] + NSP_pos
            ORFmut = "ORF1a:$(refAA)$(ORF1_pos)$(qryAA)"
        end
        if NSP_pos in [10:932...]
            ORF1_pos = NSP1b_add[NSP_num] + NSP_pos
            ORFmut = "ORF1b:$(refAA)$(ORF1_pos)$(qryAA)"
        end
    end
    ORF1pos = parse(Int, ORFpos)
    return ORFmut
end
#####################################################################################################################################
function NSP_range_to_ORF1ab_range(NSP_range::String)
    NSP = split(NSP_range, ":")[1]
    if length(NSP) < 3
        return ""
    end
    if NSP[1:3] ≠ "NSP"
        return ""
    end
    NSP_int = parse(Int, NSP[4:end])
    range = split(NSP_range, ":")[2]
    NSPpos1_int = parse(Int, split(range, "-")[1])
    NSPpos2_int = parse(Int, split(range, "-")[2])
    ORF_int1 = 0
    ORF_int2 = 0
    ORF1_AorB_pos1 = ""
    ORF1_AorB_pos2 = ""
    if NSP_int in [1:10...]
        ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1a"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int in [13:16...]
        ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
        ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
        ORF1_AorB_pos1 = "ORF1b"
        ORF1_AorB_pos2 = ""
    end
    if NSP_int == 12
        if NSPpos1_int in [1:9...]
            ORF_int1 = NSPpos1_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos1 = "1a"
        else
            ORF_int1 = NSPpos1_int + NSP1b_add[NSP_int]
            ORF1_AorB_pos1 = "ORF1b"
        end
        if NSPpos2_int in [1:9...]
            ORF_int2 = NSPpos2_int + NSP1a_add[NSP_int]
            ORF1_AorB_pos2 = "1a"
        end
        if !(NSPpos2_int in [1:9...])
            ORF_int2 = NSPpos2_int + NSP1b_add[NSP_int]
            if ORF1_AorB_pos1 == "1a"
                ORF1_AorB_pos2 = "1b:"
            else
            ORF1_AorB_pos2 = ""
            end
        end
    end
    ORF_int1_str = string(ORF_int1)
    ORF_int2_str = string(ORF_int2)
    ORF_range = ORF1_AorB_pos1*":"*ORF_int1_str*"-"*ORF1_AorB_pos2*ORF_int2_str
    return ORF_range
end
#####################################################################################################################################
function ORF1ab_range_to_NSP_range(ab_range::String)
    ab = split(ab_range, ":")[1]
    range = split(ab_range, ":")[2]
    pos1 = split(range, "-")[1]
    pos2 = split(range, "-")[2]
    pos1_int = parse(Int, pos1)
    pos2_int = parse(Int, pos2)
    NSPint1 = 0
    NSPint2 = 0
    NSPpos1 = ""
    NSPpos2 = ""
    NSPrange = ""
    NSPrange_pt1 = ""
    top_ct = 0
    if ab == "ORF1a"
        ct = 0
        for (NSP, rng) in NSP_ranges1a
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPint2 = pos2_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1a_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1a_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end
    if ab == "ORF1b"
        ct = 0
        for (NSP, rng) in NSP_ranges1b
            if pos1_int in rng && pos2_int in rng
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPint2 = pos2_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPpos2 = string(NSPint2)
                NSPstr = string(NSP)
                NSPrange = "NSP"*NSPstr*":"*NSPpos1*"-"*NSPpos2
            end
            if pos1_int in rng && !(pos2_int in rng)
                ct += 1
                NSPint1 = pos1_int - NSP1b_add[NSP]
                NSPpos1 = string(NSPint1)
                NSPstr = string(NSP)
                NSPrange_pt1 = "NSP"*NSPstr*":"*NSPpos1*"-"
            end
            if ct > 0
                top_ct += 1
                if pos2_int in rng
                    NSPint2 = pos2_int - NSP1b_add[NSP]
                    NSPpos2 = string(NSPint2)
                    NSPstr = string(NSP)
                    NSPrange_pt2 = "NSP"*NSPstr*":"*NSPpos2
                    NSPrange = NSPrange_pt1*NSPrange_pt2
                end
            end
        end
    end 
    return NSPrange
end
######################################################################################################################################
function multiepi_to_epis(multi)  
    epi_num_only_pre(n) = split(n, "_")[3]
    epi_num_only_first(n) = parse(Int, split(epi_num_only_pre(n), "-")[1])
    epi_num_only_last(n) = parse(Int, split(epi_num_only_pre(n), "-")[2])
    first = epi_num_only_first(multi)
    last = epi_num_only_last(multi)
    epi_arr = Vector{String}()
    for i in first:last
        i_str = string(i)
        epi = "EPI_ISL_"*i_str
        push!(epi_arr, epi)
    end
    return epi_arr
end
######################################################################################################################################
function stringlist_to_strings(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    arr_of_strings1 = Vector{String}()
    arr_of_strings2 = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(arr_of_strings2, mseq)
            end
        else 
            push!(arr_of_strings2, seq)
        end
    end
    sort_arr_of_strings2 = sort(collect(arr_of_strings2), by = x -> epi_sortkey(x))    
    return sort_arr_of_strings2
end
######################################################################################################################################
function stringlist_to_set(txt::String)
    epi_num_only_pre(n) = split(n, "_")[3]
    function epi_sortkey(epi)
        epinum = epi_num_only_pre(epi)
        epi_key = (length(epinum), epinum)
        return epi_key
    end
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end
    return set_of_strings
end  
######################################################################################################################################
function stringlist_to_set(txt::String)
    set_of_strings = Set{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        if '-' in seq
            multis = multiepi_to_epis(seq)
            for mseq in multis
                push!(set_of_strings, mseq)
            end
        else 
            push!(set_of_strings, seq)
        end
    end   
    return set_of_strings
end
#######################################################################################################################################
function list_to_strings(list::String)
    string_vec = string.(split(list, ", "))
    return string_vec
end
##################################################
function list_to_set(list::String)
    string_vec = string.(split(list, ", "))
    string_set = Set{String}()
    for str in string_vec
        push!(string_set, str)
    end
    return string_set
end
#####################################################################################################################################
function cross_check_set(old_seq_set::Set{String}, new_seq_set::Set{String})
    new_set2 = Set{String}()
    old_set2 = Set{String}()
    old_seq_set_len = length(old_seq_set)
    new_seq_set_len = length(new_seq_set)
    difference = new_seq_set_len - old_seq_set_len
    println("Number of Sequences in Old List = $(old_seq_set_len)")
    println("Number of Sequences in New List = $(new_seq_set_len)")
    println("Difference between Old & New Sets = $(difference)")
    println()
    for seq in new_seq_set
        if !(seq in old_seq_set)
            push!(new_set2, seq)
        end
        if seq in old_seq_set
            push!(old_set2, seq)
        end
    end
    new_len = length(new_set2)
    old_len = length(old_set2)
    if difference == new_len
        println("The numbers check out: difference between old & new sets = Number of new seqs")
    else
        println("The numbers don't check out: difference between old & new sets not equal to Number of new seqs")
    end
    println()
    println("Number of New Sequences = $(new_len)")
    println("Number of Old Sequences = $(old_len)")
    new_set2_sort = sort(collect(new_set2), by = x -> (length(x), x))
    return new_set2, new_set2_sort
end
############################################################################################################################################################################
############################################################################################################################################################################
cell1_runtime = round(digits=1, time() - cell1_start_time)
c1runtime1, c1runtime2 = seconds_to_hrs_min_sec(cell1_runtime)
println("Cell1 Runtime v0 = $(cell1_runtime) seconds")
println("Cell1 Runtime v1 = $(c1runtime1)")
println("Cell1 Runtime v2 = $(c1runtime2)"); println()
######################################################################################################################################

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

2026_03_05__644PM
/Users/ryhisner/functions_julia
/Users/ryhisner
Time to Load Packages = 0:00:32.36
Time to Load Packages = 0 hr, 0 min, 32.36 sec
seqs_in_nuc_genome_pango_dict = 3586
Cell1 Runtime v0 = 63.0 seconds
Cell1 Runtime v1 = 0:01:03.00
Cell1 Runtime v2 = 0 hr, 1 min, 03.00 sec



In [3]:
### Fx: load_all_seq_dicts, 2026_03_08   |   Loads HQCS Dataset   | many large dictionaries not needed & therefore commented out  
load_all_start = time();   date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function load_all_seq_dicts(date::String, fx_name::String, ndjson_name::String, max_AA_mut::Int, revs_thresh::Int, qc_max::Int)
    start_all = time()
    filename = "$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)"
############################################################################################################################################################################    
    global seq_ct_by_year_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
############################################################################################################################################################################
    global avg_private_AA_per_circ_seq = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq.jld2", "avg_private_AA_per_circ_seq")
    global all_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_all_seq_ct.jld2", "all_seq_ct")
    global qualifying_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_seq_ct.jld2", "qualifying_seq_ct")
    global nuc_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
    global AA_muts_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct.jld2", "AA_muts_ct")
    global AA_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
    global AA_muts_ct_pos_only_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_muts_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global gene_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density.jld2", "domain_mut_density")
    global nuc_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort.jld2", "nuc_muts_ct_sort")
    global nuc_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global nuc_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort.jld2", "nuc_muts_ct_no_dels_sort")
    global nuc_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_muts_ct_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort.jld2", "AA_muts_ct_sort")
    global AA_muts_ct_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_pos_only_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort.jld2", "AA_muts_ct_pos_only_sort")
    global AA_muts_ct_pos_only_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort.jld2", "AA_muts_ct_no_dels_sort")
    global AA_muts_ct_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort.jld2", "AA_muts_ct_pos_only_no_dels_sort")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")
    global gene_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global domain_mut_density_sort_by_gene_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global domain_mut_density_sort_by_density_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global AA_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_muts_seq.jld2", "AA_muts_seq")
    global nuc_dels_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_nuc_dels_ct.jld2", "nuc_dels_ct")
############################################################################################################################################################################
    global AA_no_dels_sub_count_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_AA_no_dels_sub_count.jld2", "AA_no_dels_sub_count")
    global total_AA_subs_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_total_AA_subs.jld2", "total_AA_subs")
#    global date_nuc_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
#    global date_nuc_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
#    global date_AA_mut_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct.jld2", "date_AA_mut_ct")
#    global date_AA_mut_ct_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
#    global date_AA_mut_ct_pos_only_no_dels_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
#    global seq_date_index_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_seq_date_index.jld2", "seq_date_index")
    global pango_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_pango_date_index_ct.jld2", "pango_date_index_ct")
    global clade_date_index_ct = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_clade_date_index_ct.jld2", "clade_date_index_ct")
############################################################################################################################################################################
######################################################## Below: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
## Changed mind about pango_date_index_ct--better to have high qc filters in place for those to weed out shitty misassigned seqs
#    global pango_date_index_ct = load("2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut7_maxRevs2_qcMax10_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global pango_date_index_ct = load("2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_pango_date_index_ct.jld2", "pango_date_index_ct")
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_pango_all = load("dictionaries/2025_08_25_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs5_qcMax8000_seq_pango.jld2", "seq_pango")



    
#####################################################################################################################################
    global country_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_country_set.jld2", "country_set")
    global clade_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_clade_set.jld2", "clade_set")
    global pango_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_pango_set.jld2", "pango_set")
#    global seq_lab_set = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################
    
    
    
    
    #    date_temp = "2025-08-03"; max_AA_mut_temp = 5; revs_thresh_temp = 1; qc_max_temp = 5; ndjson_name_temp = "EPI_ISL_400000_20080000"
#    global country_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_set.jld2", "country_set")
#    global clade_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_set.jld2", "clade_set")
#    global pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_set.jld2", "pango_set")
#    global pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_set.jld2", "pango_unaliased_set")
#    global pango_to_pango_unaliased = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_to_pango_unaliased.jld2", "pango_to_pango_unaliased")
#    global clade_pango_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_set.jld2", "clade_pango_set")
#    global clade_pango_unaliased_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_clade_pango_unaliased_set.jld2", "clade_pango_unaliased_set")
#####################################################################################################################################
#    global seq_country_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_country.jld2", "seq_country")
#    global seq_US_state_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_US_state.jld2", "seq_US_state")
#    global seq_clade_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_clade.jld2", "seq_clade")
############ Currently using low-qc seq_pango_all as it's more comprehensive ###########
#    global seq_pango_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango.jld2", "seq_pango")
#    global seq_pango_unaliased_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_pango_unaliased.jld2", "seq_pango_unaliased")
############ Currently using low-qc seq_date_index_all as it's more comprehensive ########################
#    global seq_date_index_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_index.jld2", "seq_date_index")
#    global seq_date_tuple_all = load("dictionaries/2025-11-12_all_private_muts_EPI_ISL_400000_20080000_v5_maxAAmut90_maxRevs5_qcMax8000_seq_date_tuple.jld2", "seq_date_tuple")
#    global seq_collection_date_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_collection_date.jld2", "seq_collection_date")
#    global seq_lab_dict_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_dict.jld2", "seq_lab_dict")
#    global seq_nuc_del_ranges_ct_all = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_nuc_del_ranges_ct.jld2", "seq_nuc_del_ranges_ct")
#    global seq_lab_set = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_seq_lab_set.jld2", "seq_lab_set")
#####################################################################################################################################    
#    global pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
#    global country_clade_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_clade_date_index_ct.jld2", "country_clade_date_index_ct")
#    global country_pango_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_date_index_ct.jld2", "country_pango_date_index_ct")
#    global country_pango_unaliased_date_index_ct = load("dictionaries/$(date_temp)_$(fx_name)_$(ndjson_name_temp)_maxAAmut$(max_AA_mut_temp)_maxRevs$(revs_thresh_temp)_qcMax$(qc_max_temp)_country_pango_unaliased_date_index_ct.jld2", "country_pango_unaliased_date_index_ct")
############################################################################################################################################################################
######################################################## Above: semi-permanent, unt#il new ndjson made #####################################################################
############################################################################################################################################################################
#    global ORF9b_CTD_muts_seq_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_ORF9b_CTD_muts_seq.jld2", "ORF9b_CTD_muts_seq")
    global multi_ORF9b_CTD_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD.jld2", "multi_ORF9b_CTD")
    global multi_ORF9b_CTD_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_multi_ORF9b_CTD_ct.jld2", "multi_ORF9b_CTD_ct")
#    global qualifying_ORF9b_double_ct_all = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_qualifying_ORF9b_double_ct.jld2", "qualifying_ORF9b_double_ct")
    
#    global ORF9b_CTD_muts_seq_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_ORF9b_CTD_muts_seq_relaxed_qc_10_1_30.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_relaxed_qc_10_1_30")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_multi_ORF9b_CTD_ct_relaxed_qc_10_1_30.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_10_1_30")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_10_1_30 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut8_maxRevs1_qcMax30_qualifying_ORF9b_double_ct_relaxed_qc_10_1_30.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_10_1_30")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_ORF9b_CTD_muts_seq_relaxed_qc_15_1_50.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_relaxed_qc_50_1_15")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_multi_ORF9b_CTD_ct_relaxed_qc_15_1_50.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_50_1_15")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_15_1_50 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut15_maxRevs1_qcMax50_qualifying_ORF9b_double_ct_relaxed_qc_15_1_50.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_50_1_15")

#    global ORF9b_CTD_muts_seq_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_ORF9b_CTD_muts_seq_relaxed_qc_25_3_200.jld2", "ORF9b_CTD_muts_seq_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_relaxed_qc_200_3_25")
#    global multi_ORF9b_CTD_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_multi_ORF9b_CTD_ct_relaxed_qc_25_3_200.jld2", "multi_ORF9b_CTD_ct_relaxed_qc_200_3_25")
#    global qualifying_ORF9b_double_ct_all_relaxed_qc_25_3_200 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut25_maxRevs3_qcMax200_qualifying_ORF9b_double_ct_relaxed_qc_25_3_200.jld2", "qualifying_ORF9b_double_ct_relaxed_qc_200_3_25")
#    global avg_private_AA_per_circ_seq2 = load("$(filename)/$(date)_$(fx_name)_$(ndjson_name)_maxAAmut$(max_AA_mut)_maxRevs$(revs_thresh)_qcMax$(qc_max)_avg_private_AA_per_circ_seq2.jld2", "avg_private_AA_per_circ_seq2")
    ##################### Below: Dicts for all seqs using higher (less restrictive) QC filters ##########################################
#    global AA_muts_seq_all_8000qc_filter = load("2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000/2026_01_07_all_private_muts_EPI_ISL_400001_20300000_maxAAmut90_maxRevs5_qcMax8000_AA_muts_seq.jld2", "AA_muts_seq")
#    global AA_muts_seq_all_999qc_filter = load("dictionaries/2025-08-24_all_private_muts_EPI_ISL_400000_20080000_maxAAmut90_maxRevs4_qcMax999_AA_muts_seq.jld2", "AA_muts_seq")
#####################################################################################################################################
    println("Dictionaries loaded!");   println()
    finish_all = time() - start_all;    finish_all_rd = round(digits=1, finish_all)
    load_hms1, load_hms2 = seconds_to_hrs_min_sec(finish_all)
    println("Total Time to Load ALL Dictionaries = $(finish_all_rd) seconds")  # println("Total Time to Load ALL Dictionaries = $(load_hms1)")
    println("Total Time to Load ALL Dictionaries = $(load_hms2)")
end
######################################################################################################################################
date = "2026_02_08"
clade_pct_date_index = load("dictionaries/$(date)__clade_pct_date_index.jld2", "clade_pct_date_index")
clade_pct_date_tuple = load("dictionaries/$(date)__clade_pct_date_tuple.jld2", "clade_pct_date_tuple")
clade_pct_date_string = load("dictionaries/$(date)__clade_pct_date_string.jld2", "clade_pct_date_string")
pango_pct_date_index = load("dictionaries/$(date)__pango_pct_date_index.jld2", "pango_pct_date_index")
pango_pct_date_tuple = load("dictionaries/$(date)__pango_pct_date_tuple.jld2", "pango_pct_date_tuple")
pango_pct_date_string = load("dictionaries/$(date)__pango_pct_date_string.jld2", "pango_pct_date_string")
pango_unaliased_pct_date_index = load("dictionaries/$(date)__pango_unaliased_pct_date_index.jld2", "pango_unaliased_pct_date_index")
pango_unaliased_pct_date_tuple = load("dictionaries/$(date)__pango_unaliased_pct_date_tuple.jld2", "pango_unaliased_pct_date_tuple")
pango_unaliased_pct_date_string = load("dictionaries/$(date)__pango_unaliased_pct_date_string.jld2", "pango_unaliased_pct_date_string")
pango_to_pango_unaliased_v2 = load("dictionaries/$(date)__pango_to_pango_unaliased_v2.jld2", "pango_to_pango_unaliased_v2")
pango_unaliased_to_pango_prefix = load("dictionaries/$(date)__pango_unaliased_to_pango_prefix.jld2", "pango_unaliased_to_pango_prefix")
pango_unaliased_to_pango = load("dictionaries/$(date)__pango_unaliased_to_pango.jld2", "pango_unaliased_to_pango")
pango_unaliased_predecessor_meta_dict = load("dictionaries/$(date)__pango_unaliased_predecessor_meta_dict.jld2", "pango_unaliased_predecessor_meta_dict")
pango_predecessor_meta_dict = load("dictionaries/$(date)__pango_predecessor_meta_dict.jld2", "pango_predecessor_meta_dict")
pango_unaliased_set = load("dictionaries/$(date)__pango_unaliased_set.jld2", "pango_unaliased_set")
pango_unaliased_date_index_ct = load("dictionaries/$(date)__pango_unaliased_date_index_ct.jld2", "pango_unaliased_date_index_ct")
clade_date_index_cumul = load("dictionaries/$(date)__clade_date_index_cumul.jld2", "clade_date_index_cumul")
pango_date_index_cumul = load("dictionaries/$(date)__pango_date_index_cumul.jld2", "pango_date_index_cumul")
pango_unaliased_date_index_cumul = load("dictionaries/$(date)__pango_unaliased_date_index_cumul.jld2", "pango_unaliased_date_index_cumul")
clade_total = load("dictionaries/$(date)__clade_total.jld2", "clade_total")
pango_total = load("dictionaries/$(date)__pango_total.jld2", "pango_total")
pango_unaliased_total = load("dictionaries/$(date)__pango_unaliased_total.jld2", "pango_unaliased_total")
AAmut_date_index_cumul = load("dictionaries/$(date)__AAmut_date_index_cumul.jld2", "AAmut_date_index_cumul")
nucmut_date_index_cumul = load("dictionaries/$(date)__nucmut_date_index_cumul.jld2", "nucmut_date_index_cumul")
pango_nuc_sub_WT = load("dictionaries/$(date)__pango_nuc_sub_WT.jld2", "pango_nuc_sub_WT")
pango_nuc_del_WT = load("dictionaries/$(date)__pango_nuc_del_WT.jld2", "pango_nuc_del_WT")
pango_nuc_sub_private = load("dictionaries/$(date)__pango_nuc_sub_private.jld2", "pango_nuc_sub_private")
pango_nuc_del_private = load("dictionaries/$(date)__pango_nuc_del_private.jld2", "pango_nuc_del_private")
pango_nuc_sub_revs = load("dictionaries/$(date)__pango_nuc_sub_revs.jld2", "pango_nuc_sub_revs")
pango_nuc_del_revs = load("dictionaries/$(date)__pango_nuc_del_revs.jld2", "pango_nuc_del_revs")
pango_AAsub_WT = load("dictionaries/$(date)__pango_AAsub_WT.jld2", "pango_AAsub_WT")
pango_AAsub_WT_pos_only = load("dictionaries/$(date)__pango_AAsub_WT_pos_only.jld2", "pango_AAsub_WT_pos_only")
pango_AAdel_WT = load("dictionaries/$(date)__pango_AAdel_WT.jld2", "pango_AAdel_WT")
pango_AAsub_private = load("dictionaries/$(date)__pango_AAsub_private.jld2", "pango_AAsub_private")
pango_AAdel_private = load("dictionaries/$(date)__pango_AAdel_private.jld2", "pango_AAdel_private")
pango_AAsub_revs = load("dictionaries/$(date)__pango_AAsub_revs.jld2", "pango_AAsub_revs")
pango_AAdel_revs = load("dictionaries/$(date)__pango_AAdel_revs.jld2", "pango_AAdel_revs")
pango_frameshifts_WT = load("dictionaries/$(date)__pango_frameshifts_WT.jld2", "pango_frameshifts_WT")
pango_designation_date = load("dictionaries/$(date)__pango_designation_date.jld2", "pango_designation_date")
clade_pango_set = load("dictionaries/$(date)__clade_pango_set.jld2", "clade_pango_set")
delete!(pango_AAsub_WT["B.55"], "")
delete!(pango_AAsub_WT["B"], "")
println("Done!")
load_all_fx_runtime = time() - load_all_start; load_all_fx_runtime_rd = round(digits=3, load_all_fx_runtime)
load_all_fx_hms1, load_all_fx_hms2 = seconds_to_hrs_min_sec(load_all_fx_runtime)
println("Total Time to Run load_all Fx = $(load_all_fx_runtime_rd) seconds")  # println("Total Time to Run load_all Fx = $(load_all_fx_hms1)") 
println("Total Time to Run load_all Fx = $(load_all_fx_hms2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################


2026_03_05__645PM
Done!
Total Time to Run load_all Fx = 8.257 seconds
Total Time to Run load_all Fx = 0 hr, 0 min, 08.26 sec



In [4]:
### Execute Load All Dicts Fx, EPI_ISL_400000_20300000 | 2026_02_08 | QC--5-1-5 | Runtime = 38 sec #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_load_all_dicts = time()
HQCS_date = "2026_01_06"
HQCS_fx_name = "all_private_muts"
HQCS_ndjson_name = "EPI_ISL_400001_20300000"
HQCS_max_AA_mut = 5
HQCS_revs_thresh = 1
HQCS_qc_max = 5
HQCS_qc_str = "$(HQCS_max_AA_mut)_$(HQCS_revs_thresh)_$(HQCS_qc_max)"
load_all_seq_dicts(HQCS_date, HQCS_fx_name, HQCS_ndjson_name, HQCS_max_AA_mut, HQCS_revs_thresh, HQCS_qc_max)

gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    

finish_load_all_dicts = time() - start_load_all_dicts
finish_load_all_dicts_rd = round(digits=1, finish_load_all_dicts)
println("Total Time to Load ALL Dictionaries = $(finish_load_all_dicts_rd)")
####################################################################################################################################
#################################### Create  date_index_seq_ct_all  Dictionary #####################################################
############################# Only needed for special calculations, so commented out here ##########################################
####################################################################################################################################
#date_index_seq_ct_all = Dict{Int, Int}()
#for d_index in values(seq_date_index_all)
#    date_index_seq_ct_all[d_index] = get(date_index_seq_ct_all, d_index, 0) + 1
#end
function date_index_range_seq_ct(date1::Int, date2::Int)
    date1_date2_seq_ct = 0
    for d_index in date1:date2
        date1_date2_seq_ct += date_index_seq_ct_all[d_index]
    end
    return date1_date2_seq_ct
end   
###################################################################################################################################
############################### Create  nuc_muts_ct_pos_only_no_dels_all  Dictionary ##############################################
###################################################################################################################################
dict_make_start = time()
nuc_muts_ct_pos_only_no_dels_all = Dict{Int, Int}()
for i in 1:30000
    nuc_muts_ct_pos_only_no_dels_all[i] = 0
end
for (nuc_mut, count) in nuc_muts_ct_no_dels_all
    pos = nuc_mut_int_comprehensive_dict[nuc_mut]
    nuc_muts_ct_pos_only_no_dels_all[pos] += count
end
#############################################__Create   pango_index_date_set    ###################################################
pango_index_date_set = Set{String}()
for pango in keys(pango_date_index_ct)
    push!(pango_index_date_set, pango)
end
println(length(pango_index_date_set))
#################################################################################################################################
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
dict_make_runtime = round(digits=2, time() - dict_make_start)
println("Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = $(dict_make_runtime) seconds"); print("\n"^1)
#################################################################################################################################
#################################################################################################################################

2026_03_05__645PM
Dictionaries loaded!

Total Time to Load ALL Dictionaries = 48.9 seconds
Total Time to Load ALL Dictionaries = 0 hr, 0 min, 48.90 sec
Total Time to Load ALL Dictionaries = 48.9
4297
length(keys(pango_to_pango_unaliased_v2) = 5657
Total Time to Make nuc_muts_ct_pos_only_no_dels_all Dict = 0.14 seconds



In [5]:
### Fx: chronic_load_dicts2_DQ, 2026_03_01 version |  Loads EPCI dataset  | ###############################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_chr_load_fx = time()
function chronic_load_dicts2_DQ(ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Int, abs_min_mut_thresh::Int)
    start_date = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(start_date)
######################################################################################################################################
#    global cumulative_AA_mut_ct_by_date_index
#        cumulative_AA_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_AA_mut_ct_by_date_index.jld2", "cumulative_AA_mut_ct_by_date_index")
#    global cumulative_nuc_mut_ct_by_date_index
#        cumulative_nuc_mut_ct_by_date_index = load("$(folder_name)/$(folder_name)__cumulative_nuc_mut_ct_by_date_index.jld2", "cumulative_nuc_mut_ct_by_date_index")
######################################################################################################################################
    global seq_AA_insertions_WT
        seq_AA_insertions_WT = load("$(folder_name)/$(folder_name)__seq_AA_insertions_WT.jld2", "seq_AA_insertions_WT")
    global seq_nuc_insertions_WT
        seq_nuc_insertions_WT = load("$(folder_name)/$(folder_name)__seq_nuc_insertions_WT.jld2", "seq_nuc_insertions_WT")    
######################################################################################################################################
    global total_chr_AA_subs
        total_chr_AA_subs = load("$(folder_name)/$(folder_name)__total_chr_AA_subs.jld2", "total_chr_AA_subs")
######################################################################################################################################
    global total_nuc_revs_seq
        total_nuc_revs_seq = load("$(folder_name)/$(folder_name)__total_nuc_revs_seq.jld2", "total_nuc_revs_seq")
    global seq_nuc_total_revs
        seq_nuc_total_revs = load("$(folder_name)/$(folder_name)__seq_nuc_total_revs.jld2", "seq_nuc_total_revs")
    global total_AA_revs_seq
        total_AA_revs_seq = load("$(folder_name)/$(folder_name)__total_AA_revs_seq.jld2", "total_AA_revs_seq")
    global seq_AA_total_revs
        seq_AA_total_revs = load("$(folder_name)/$(folder_name)__seq_AA_total_revs.jld2", "seq_AA_total_revs")
    global seq_AA_revs
        seq_AA_revs = load("$(folder_name)/$(folder_name)__seq_AA_revs.jld2", "seq_AA_revs")
######################################################################################################################################
    global AA_muts_ct_chr_all_ratio
        AA_muts_ct_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio.jld2", "AA_muts_ct_chr_all_ratio")
    global AA_muts_ct_no_dels_chr_all_ratio
        AA_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio.jld2", "AA_muts_ct_no_dels_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio")
    global AA_muts_ct_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_no_dels_no_revs_chr_all_ratio")
    global AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
    global nuc_muts_ct_no_dels_chr_all_ratio
        nuc_muts_ct_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio")
    global nuc_muts_ct_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_no_dels_no_revs_chr_all_ratio")
    global nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio
        nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio.jld2", "nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio")
###################################################################################################################################### 
####################################################################################################################################
    global AA_muts_ct_pos_only_adj_score
        AA_muts_ct_pos_only_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score.jld2", "AA_muts_ct_pos_only_adj_score")
    global AA_muts_ct_adj
        AA_muts_ct_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj.jld2", "AA_muts_ct_adj")
    global AA_muts_ct_pos_only_adj_score_no_dels
        AA_muts_ct_pos_only_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels.jld2", "AA_muts_ct_pos_only_adj_score_no_dels")
    global nuc_muts_ct_adj
        nuc_muts_ct_adj = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj.jld2", "nuc_muts_ct_adj")
    global AA_muts_ct_adj_score
        AA_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score.jld2", "AA_muts_ct_adj_score")
    global AA_muts_ct_adj_score_no_dels
        AA_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels.jld2", "AA_muts_ct_adj_score_no_dels")
    global AA_muts_ct_pos_only_adj
        AA_muts_ct_pos_only_adj = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj.jld2", "AA_muts_ct_pos_only_adj")
    global AA_muts_ct_pos_only_adj_no_dels
        AA_muts_ct_pos_only_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels.jld2", "AA_muts_ct_pos_only_adj_no_dels")
    global nuc_muts_ct_adj_score_no_dels
        nuc_muts_ct_adj_score_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels.jld2", "nuc_muts_ct_adj_score_no_dels")
    global nuc_muts_ct_adj_score
        nuc_muts_ct_adj_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score.jld2", "nuc_muts_ct_adj_score")
    global nuc_muts_ct_adj_no_dels
        nuc_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels.jld2", "nuc_muts_ct_adj_no_dels")
    global AA_muts_ct_adj_no_dels
        AA_muts_ct_adj_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels.jld2", "AA_muts_ct_adj_no_dels")
####################################################################################################################################
####################################################################################################################################
    global seq_ct_by_year
        seq_ct_by_year = load("$(folder_name)/$(folder_name)__seq_ct_by_year.jld2", "seq_ct_by_year")
    global seq_ct_by_year_month
        seq_ct_by_year_month = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month.jld2", "seq_ct_by_year_month")
    global seq_ct_by_year_month_day
        seq_ct_by_year_month_day = load("$(folder_name)/$(folder_name)__seq_ct_by_year_month_day.jld2", "seq_ct_by_year_month_day")
    global seq_date_tuple
        seq_date_tuple = load("$(folder_name)/$(folder_name)__seq_date_tuple.jld2", "seq_date_tuple")
    global date_nuc_mut_ct_no_dels
        date_nuc_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct_no_dels.jld2", "date_nuc_mut_ct_no_dels")
    global date_AA_mut_ct_no_dels
        date_AA_mut_ct_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_no_dels.jld2", "date_AA_mut_ct_no_dels")
    global date_AA_mut_ct
        date_AA_mut_ct = load("$(folder_name)/$(folder_name)__date_AA_mut_ct.jld2", "date_AA_mut_ct")
    global date_AA_mut_ct_pos_only_no_dels
        date_AA_mut_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__date_AA_mut_ct_pos_only_no_dels.jld2", "date_AA_mut_ct_pos_only_no_dels")
####################################################################################################################################
    global seq_collection_date
        seq_collection_date = load("$(folder_name)/$(folder_name)__seq_collection_date.jld2", "seq_collection_date")
    global seq_date_index
        seq_date_index = load("$(folder_name)/$(folder_name)__seq_date_index.jld2", "seq_date_index")
    global date_nuc_mut_ct
        date_nuc_mut_ct = load("$(folder_name)/$(folder_name)__date_nuc_mut_ct.jld2", "date_nuc_mut_ct")
####################################################################################################################################
    global seq_AA_dels
        seq_AA_dels = load("$(folder_name)/$(folder_name)__seq_AA_dels.jld2", "seq_AA_dels")
    global seq_AA_muts_pos_only_no_dels
        seq_AA_muts_pos_only_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only_no_dels.jld2", "seq_AA_muts_pos_only_no_dels")
    global seq_AA_muts_WT_pos_only
        seq_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT_pos_only.jld2", "seq_AA_muts_WT_pos_only")
    global seq_AA_dels_WT
        seq_AA_dels_WT = load("$(folder_name)/$(folder_name)__seq_AA_dels_WT.jld2", "seq_AA_dels_WT")
    global seq_AA_muts_WT
        seq_AA_muts_WT = load("$(folder_name)/$(folder_name)__seq_AA_muts_WT.jld2", "seq_AA_muts_WT")
    global seq_AA_muts
        seq_AA_muts = load("$(folder_name)/$(folder_name)__seq_AA_muts.jld2", "seq_AA_muts")
    global seq_AA_muts_no_dels
        seq_AA_muts_no_dels = load("$(folder_name)/$(folder_name)__seq_AA_muts_no_dels.jld2", "seq_AA_muts_no_dels")
    global seq_AA_muts_pos_only
        seq_AA_muts_pos_only = load("$(folder_name)/$(folder_name)__seq_AA_muts_pos_only.jld2", "seq_AA_muts_pos_only")
    global seq_mixed_AA_muts
        seq_mixed_AA_muts = load("$(folder_name)/$(folder_name)__seq_mixed_AA_muts.jld2", "seq_mixed_AA_muts")
    global seq_unknown_AA
        seq_unknown_AA = load("$(folder_name)/$(folder_name)__seq_unknown_AA.jld2", "seq_unknown_AA")
    global seq_unknown_AA_ranges
        seq_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__seq_unknown_AA_ranges.jld2", "seq_unknown_AA_ranges")
####################################################################################################################################
    global seq_nuc_muts
        seq_nuc_muts = load("$(folder_name)/$(folder_name)__seq_nuc_muts.jld2", "seq_nuc_muts")
    global seq_nuc_muts_WT
        seq_nuc_muts_WT = load("$(folder_name)/$(folder_name)__seq_nuc_muts_WT.jld2", "seq_nuc_muts_WT")
    global seq_nuc_del_ranges_WT
        seq_nuc_del_ranges_WT = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges_WT.jld2", "seq_nuc_del_ranges_WT")
    global seq_nuc_dropout
        seq_nuc_dropout = load("$(folder_name)/$(folder_name)__seq_nuc_dropout.jld2", "seq_nuc_dropout")
    global seq_nuc_del_ranges
        seq_nuc_del_ranges = load("$(folder_name)/$(folder_name)__seq_nuc_del_ranges.jld2", "seq_nuc_del_ranges")
    global seq_mixed_nucs
        seq_mixed_nucs = load("$(folder_name)/$(folder_name)__seq_mixed_nucs.jld2", "seq_mixed_nucs")
###################################################################################################################################
    global seq_clade
        seq_clade = load("$(folder_name)/$(folder_name)__seq_clade.jld2", "seq_clade")
    global seq_pango
        seq_pango = load("$(folder_name)/$(folder_name)__seq_pango.jld2", "seq_pango")
    global seq_pango_unaliased
        seq_pango_unaliased = load("$(folder_name)/$(folder_name)__seq_pango_unaliased.jld2", "seq_pango_unaliased")    
    global seq_clade_display
        seq_clade_display = load("$(folder_name)/$(folder_name)__seq_clade_display.jld2", "seq_clade_display")
    global seq_clade_ct
        seq_clade_ct = load("$(folder_name)/$(folder_name)__seq_clade_ct.jld2", "seq_clade_ct")
    global seq_pango_ct
        seq_pango_ct = load("$(folder_name)/$(folder_name)__seq_pango_ct.jld2", "seq_pango_ct")
    global seq_pango_unaliased_ct
        seq_pango_unaliased_ct = load("$(folder_name)/$(folder_name)__seq_pango_unaliased_ct.jld2", "seq_pango_unaliased_ct")
####################################################################################################################################
    global seq_US_state
        seq_US_state = load("$(folder_name)/$(folder_name)__seq_US_state.jld2", "seq_US_state")
    global seq_country
        seq_country = load("$(folder_name)/$(folder_name)__seq_country.jld2", "seq_country")
    global seq_lab_dict
        seq_lab_dict = load("$(folder_name)/$(folder_name)__seq_lab_dict.jld2", "seq_lab_dict")
####################################################################################################################################
    global gene_mut_density
        gene_mut_density = load("$(folder_name)/$(folder_name)__gene_mut_density.jld2", "gene_mut_density")
    global domain_mut_density
        domain_mut_density = load("$(folder_name)/$(folder_name)__domain_mut_density.jld2", "domain_mut_density")
####################################################################################################################################
    global nuc_muts_ct
        nuc_muts_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct.jld2", "nuc_muts_ct")
    global nuc_dels_ct
        nuc_dels_ct = load("$(folder_name)/$(folder_name)__nuc_dels_ct.jld2", "nuc_dels_ct")
    global nuc_muts_ct_no_dels
        nuc_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels.jld2", "nuc_muts_ct_no_dels")
######################
    global nuc_dels_seq
        nuc_dels_seq = load("$(folder_name)/$(folder_name)__nuc_dels_seq.jld2", "nuc_dels_seq")
    global nuc_muts_seq
        nuc_muts_seq = load("$(folder_name)/$(folder_name)__nuc_muts_seq.jld2", "nuc_muts_seq")
    global nuc_dels_seq_WT
        nuc_dels_seq_WT = load("$(folder_name)/$(folder_name)__nuc_dels_seq_WT.jld2", "nuc_dels_seq_WT")
    global nuc_muts_seq_WT
        nuc_muts_seq_WT = load("$(folder_name)/$(folder_name)__nuc_muts_seq_WT.jld2", "nuc_muts_seq_WT")
##################################################################
    global AA_muts_ct
        AA_muts_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct.jld2", "AA_muts_ct")
    global AA_muts_ct_no_dels_no_revs
        AA_muts_ct_no_dels_no_revs = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs.jld2", "AA_muts_ct_no_dels_no_revs")
    global AA_muts_ct_pos_only
        AA_muts_ct_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only.jld2", "AA_muts_ct_pos_only")
    global AA_dels_ct
        AA_dels_ct = load("$(folder_name)/$(folder_name)__AA_dels_ct.jld2", "AA_dels_ct")
    global AA_muts_ct_pos_only_no_dels
        AA_muts_ct_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels.jld2", "AA_muts_ct_pos_only_no_dels")
    global AA_muts_ct_no_dels
        AA_muts_ct_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels.jld2", "AA_muts_ct_no_dels")
############################## 
    global AA_muts_seq
        AA_muts_seq = load("$(folder_name)/$(folder_name)__AA_muts_seq.jld2", "AA_muts_seq")
    global AA_muts_seq_pos_only
        AA_muts_seq_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only.jld2", "AA_muts_seq_pos_only")
    global AA_muts_seq_pos_only_no_dels
        AA_muts_seq_pos_only_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_seq_pos_only_no_dels.jld2", "AA_muts_seq_pos_only_no_dels")
    global AA_dels_seq
        AA_dels_seq = load("$(folder_name)/$(folder_name)__AA_dels_seq.jld2", "AA_dels_seq")
##############################
    global AA_muts_seq_WT
        AA_muts_seq_WT = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT.jld2", "AA_muts_seq_WT")
    global AA_muts_seq_WT_pos_only
        AA_muts_seq_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_seq_WT_pos_only.jld2", "AA_muts_seq_WT_pos_only")
    global AA_dels_seq_WT
        AA_dels_seq_WT = load("$(folder_name)/$(folder_name)__AA_dels_seq_WT.jld2", "AA_dels_seq_WT")
#####################################################################################################################################
#####################################################################################################################################
    global NSP_muts
        NSP_muts = load("$(folder_name)/$(folder_name)__NSP_muts.jld2", "NSP_muts")
    global NSP_muts_no_dels
        NSP_muts_no_dels = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels.jld2", "NSP_muts_no_dels")
#####################################################################################################################################
    global non_rep_seqs_AA
        non_rep_seqs_AA = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA.jld2", "non_rep_seqs_AA")
    global non_rep_seqs_AA_pos_only
        non_rep_seqs_AA_pos_only = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only.jld2", "non_rep_seqs_AA_pos_only")
    global non_rep_seqs_AA_pos_only_no_dels
        non_rep_seqs_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__non_rep_seqs_AA_pos_only_no_dels.jld2", "non_rep_seqs_AA_pos_only_no_dels")
#####################################################################################################################################
    global rep_seq_grps_AA_muts_WT
        rep_seq_grps_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT.jld2", "rep_seq_grps_AA_muts_WT")
    global rep_seq_grps_AA_muts_WT_pos_only
        rep_seq_grps_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_muts_WT_pos_only.jld2", "rep_seq_grps_AA_muts_WT_pos_only")
    global rep_seq_grps_AA_dels_WT
        rep_seq_grps_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels_WT.jld2", "rep_seq_grps_AA_dels_WT")
    global rep_seq_grps_nuc_muts_WT
        rep_seq_grps_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_muts_WT.jld2", "rep_seq_grps_nuc_muts_WT")
    global rep_seq_grps_nuc_dels_WT
        rep_seq_grps_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dels_WT.jld2", "rep_seq_grps_nuc_dels_WT")
    global nuc_muts_rep_seq_grps_WT
        nuc_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_WT.jld2", "nuc_muts_rep_seq_grps_WT")
    global nuc_dels_rep_seq_grps_WT
        nuc_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps_WT.jld2", "nuc_dels_rep_seq_grps_WT")
    global AA_dels_rep_seq_grps_WT
        AA_dels_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps_WT.jld2", "AA_dels_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT
        AA_muts_rep_seq_grps_WT = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT.jld2", "AA_muts_rep_seq_grps_WT")
    global AA_muts_rep_seq_grps_WT_pos_only
        AA_muts_rep_seq_grps_WT_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_WT_pos_only.jld2", "AA_muts_rep_seq_grps_WT_pos_only")
#####################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_maxmut_pango
        rep_seq_grps_maxmut_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango.jld2", "rep_seq_grps_maxmut_pango")
    global rep_seq_grps_maxmut_clade
        rep_seq_grps_maxmut_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_clade.jld2", "rep_seq_grps_maxmut_clade")
    global rep_seq_grps_maxmut_pango_unaliased
        rep_seq_grps_maxmut_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_pango_unaliased.jld2", "rep_seq_grps_maxmut_pango_unaliased")
#####################################################################################################################################
    global rep_seq_grps_maxmut_dels
        rep_seq_grps_maxmut_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_dels.jld2", "rep_seq_grps_maxmut_dels")
    global rep_seq_grps_maxmut_AA_pos_only_no_dels
        rep_seq_grps_maxmut_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only_no_dels.jld2", "rep_seq_grps_maxmut_AA_pos_only_no_dels")
    global rep_seq_grps_maxmut_nuc
        rep_seq_grps_maxmut_nuc = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc.jld2", "rep_seq_grps_maxmut_nuc")
    global rep_seq_grps_maxmut_AA_pos_only
        rep_seq_grps_maxmut_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_pos_only.jld2", "rep_seq_grps_maxmut_AA_pos_only")
    global rep_seq_grps_maxmut_nuc_dropout
        rep_seq_grps_maxmut_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dropout.jld2", "rep_seq_grps_maxmut_nuc_dropout")
    global rep_seq_grps_maxmut_nuc_no_dels
        rep_seq_grps_maxmut_nuc_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_no_dels.jld2", "rep_seq_grps_maxmut_nuc_no_dels")
    global rep_seq_grps_maxmut_mixed_nucs
        rep_seq_grps_maxmut_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_nucs.jld2", "rep_seq_grps_maxmut_mixed_nucs")
    global rep_seq_grps_maxmut_AA_dels
        rep_seq_grps_maxmut_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels.jld2", "rep_seq_grps_maxmut_AA_dels")
    global rep_seq_grps_maxmut_del_ranges_ct
        rep_seq_grps_maxmut_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_del_ranges_ct.jld2", "rep_seq_grps_maxmut_del_ranges_ct")
    global rep_seq_grps_maxmut_AA
        rep_seq_grps_maxmut_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA.jld2", "rep_seq_grps_maxmut_AA")
    global rep_seq_grps_maxmut_mixed_AA_muts
        rep_seq_grps_maxmut_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_mixed_AA_muts.jld2", "rep_seq_grps_maxmut_mixed_AA_muts")
    global rep_seq_grps_maxmut_unknown_AA
        rep_seq_grps_maxmut_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA.jld2", "rep_seq_grps_maxmut_unknown_AA")
    global rep_seq_grps_maxmut_unknown_AA_ranges
        rep_seq_grps_maxmut_unknown_AA_ranges = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_unknown_AA_ranges.jld2", "rep_seq_grps_maxmut_unknown_AA_ranges")
    global rep_seq_grps_maxmut_seqs
        rep_seq_grps_maxmut_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_seqs.jld2", "rep_seq_grps_maxmut_seqs")
    global rep_seq_grps_maxmut_AA_no_dels
        rep_seq_grps_maxmut_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_no_dels.jld2", "rep_seq_grps_maxmut_AA_no_dels")
    global rep_seq_grps_maxmut_nuc_muts_WT
        rep_seq_grps_maxmut_nuc_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_muts_WT.jld2", "rep_seq_grps_maxmut_nuc_muts_WT")
    global rep_seq_grps_maxmut_nuc_dels_WT
        rep_seq_grps_maxmut_nuc_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_nuc_dels_WT.jld2", "rep_seq_grps_maxmut_nuc_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT
        rep_seq_grps_maxmut_AA_muts_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT.jld2", "rep_seq_grps_maxmut_AA_muts_WT")
    global rep_seq_grps_maxmut_AA_dels_WT
        rep_seq_grps_maxmut_AA_dels_WT = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_dels_WT.jld2", "rep_seq_grps_maxmut_AA_dels_WT")
    global rep_seq_grps_maxmut_AA_muts_WT_pos_only
        rep_seq_grps_maxmut_AA_muts_WT_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_maxmut_AA_muts_WT_pos_only.jld2", "rep_seq_grps_maxmut_AA_muts_WT_pos_only")
################################################################################################################################
#####################################################################################################################################
    global rep_seq_grps_seqs
        rep_seq_grps_seqs = load("$(folder_name)/$(folder_name)__rep_seq_grps_seqs.jld2", "rep_seq_grps_seqs")
    global rep_seq_grps_pango
        rep_seq_grps_pango = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango.jld2", "rep_seq_grps_pango")
    global rep_seq_grps_clade
        rep_seq_grps_clade = load("$(folder_name)/$(folder_name)__rep_seq_grps_clade.jld2", "rep_seq_grps_clade")
    global rep_seq_grps_pango_unaliased
        rep_seq_grps_pango_unaliased = load("$(folder_name)/$(folder_name)__rep_seq_grps_pango_unaliased.jld2", "rep_seq_grps_pango_unaliased")
    global rep_seq_grps_muts
        rep_seq_grps_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts.jld2", "rep_seq_grps_muts")
    global rep_seq_grps_mixed_nucs
        rep_seq_grps_mixed_nucs = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_nucs.jld2", "rep_seq_grps_mixed_nucs")
    global rep_seq_grps_muts_no_dels
        rep_seq_grps_muts_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_muts_no_dels.jld2", "rep_seq_grps_muts_no_dels")
    global rep_seq_grps_AA_pos_only
        rep_seq_grps_AA_pos_only = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only.jld2", "rep_seq_grps_AA_pos_only")
    global rep_seq_grps_AA_no_dels
        rep_seq_grps_AA_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_no_dels.jld2", "rep_seq_grps_AA_no_dels")
    global rep_seq_grps_AA_dels
        rep_seq_grps_AA_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_dels.jld2", "rep_seq_grps_AA_dels")
    global rep_seq_grps_dels
        rep_seq_grps_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_dels.jld2", "rep_seq_grps_dels")
    global rep_seq_grps_AA_pos_only_no_dels
        rep_seq_grps_AA_pos_only_no_dels = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA_pos_only_no_dels.jld2", "rep_seq_grps_AA_pos_only_no_dels")
    global rep_seq_grps_del_ranges_ct
        rep_seq_grps_del_ranges_ct = load("$(folder_name)/$(folder_name)__rep_seq_grps_del_ranges_ct.jld2", "rep_seq_grps_del_ranges_ct")
    global rep_seq_grps_AA
        rep_seq_grps_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_AA.jld2", "rep_seq_grps_AA")
    global rep_seq_grps_nuc_dropout
        rep_seq_grps_nuc_dropout = load("$(folder_name)/$(folder_name)__rep_seq_grps_nuc_dropout.jld2", "rep_seq_grps_nuc_dropout")
#    global rep_seq_grps_unknown_AA
#        rep_seq_grps_unknown_AA = load("$(folder_name)/$(folder_name)__rep_seq_grps_unknown_AA.jld2", "rep_seq_grps_unknown_AA")
#    global rep_seq_grps_mixed_AA_muts
#        rep_seq_grps_mixed_AA_muts = load("$(folder_name)/$(folder_name)__rep_seq_grps_mixed_AA_muts.jld2", "rep_seq_grps_mixed_AA_muts")
    global AA_dels_rep_seq_grps
        AA_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_dels_rep_seq_grps.jld2", "AA_dels_rep_seq_grps")
    global AA_muts_rep_seq_grps
        AA_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps.jld2", "AA_muts_rep_seq_grps")
    global AA_muts_rep_seq_grps_pos_only
        AA_muts_rep_seq_grps_pos_only = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_pos_only.jld2", "AA_muts_rep_seq_grps_pos_only")
    global AA_muts_rep_seq_grps_no_dels
        AA_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__AA_muts_rep_seq_grps_no_dels.jld2", "AA_muts_rep_seq_grps_no_dels")
    global nuc_muts_rep_seq_grps
        nuc_muts_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps.jld2", "nuc_muts_rep_seq_grps")
    global nuc_muts_rep_seq_grps_no_dels
        nuc_muts_rep_seq_grps_no_dels = load("$(folder_name)/$(folder_name)__nuc_muts_rep_seq_grps_no_dels.jld2", "nuc_muts_rep_seq_grps_no_dels")
    global nuc_dels_rep_seq_grps
        nuc_dels_rep_seq_grps = load("$(folder_name)/$(folder_name)__nuc_dels_rep_seq_grps.jld2", "nuc_dels_rep_seq_grps")    
##########################################################################################################################################################################
##########################################################################################################################################################################
################################################ Below: Used to be in ungodly @save form, now changed ####################################################################
##########################################################################################################################################################################
##########################################################################################################################################################################
    global AA_muts_ct_pos_only_adj_sort_by_site
        AA_muts_ct_pos_only_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_sort_by_site")
    global nuc_muts_ct_adj_score_no_dels_sort_by_site
        nuc_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_no_dels_sort_by_site
        nuc_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_site.jld2", "nuc_muts_ct_adj_no_dels_sort_by_site")
    global nuc_muts_ct_adj_sort_by_seq_ct
        nuc_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_sort_by_seq_ct")
    global nuc_muts_ct_adj_score_sort_by_score
        nuc_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_score.jld2", "nuc_muts_ct_adj_score_sort_by_score")
    global AA_muts_ct_adj_score_sort_by_site
        AA_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_site.jld2", "AA_muts_ct_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_sort_by_site
        AA_muts_ct_pos_only_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_site")
    global AA_muts_ct_pos_only_adj_sort_by_seq_ct
        AA_muts_ct_pos_only_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_score_no_dels_sort_by_site
        AA_muts_ct_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_site")
    global AA_muts_ct_adj_sort_by_seq_ct
        AA_muts_ct_adj_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_seq_ct.jld2", "AA_muts_ct_adj_sort_by_seq_ct")
    global AA_muts_ct_adj_no_dels_sort_by_site
        AA_muts_ct_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_adj_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_adj_score_sort_by_score
        AA_muts_ct_pos_only_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_sort_by_score")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_no_dels_sort_by_seq_ct
        nuc_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "nuc_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_adj_sort_by_site
        AA_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_sort_by_site.jld2", "AA_muts_ct_adj_sort_by_site")
    global AA_muts_ct_pos_only_adj_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_no_dels_sort_by_site")
    global AA_muts_ct_adj_score_sort_by_score
        AA_muts_ct_adj_score_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_sort_by_score.jld2", "AA_muts_ct_adj_score_sort_by_score")
    global nuc_muts_ct_adj_score_no_dels_sort_by_score
        nuc_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_no_dels_sort_by_score.jld2", "nuc_muts_ct_adj_score_no_dels_sort_by_score")
    global nuc_muts_ct_adj_sort_by_site
        nuc_muts_ct_adj_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_sort_by_site.jld2", "nuc_muts_ct_adj_sort_by_site")
    global AA_muts_ct_adj_score_no_dels_sort_by_score
        AA_muts_ct_adj_score_no_dels_sort_by_score = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_score_no_dels_sort_by_score.jld2", "AA_muts_ct_adj_score_no_dels_sort_by_score")
    global AA_muts_ct_adj_no_dels_sort_by_seq_ct
        AA_muts_ct_adj_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_adj_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_adj_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site
        AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site")
    global nuc_muts_ct_adj_score_sort_by_site
        nuc_muts_ct_adj_score_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_adj_score_sort_by_site.jld2", "nuc_muts_ct_adj_score_sort_by_site")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_sort_by_site
        AA_muts_ct_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_site.jld2", "AA_muts_ct_no_dels_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_seq_ct
        AA_muts_ct_pos_only_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_sort_by_seq_ct")
    global nuc_muts_ct_sort_by_site
        nuc_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_site.jld2", "nuc_muts_ct_sort_by_site")
    global excluded_AA
        excluded_AA = load("$(folder_name)/$(folder_name)__excluded_AA.jld2", "excluded_AA")
    global chronic_search_muts
        chronic_search_muts = load("$(folder_name)/$(folder_name)__chronic_search_muts.jld2", "chronic_search_muts")
    global excluded_pos
        excluded_pos = load("$(folder_name)/$(folder_name)__excluded_pos.jld2", "excluded_pos")
    global rep_seqs
        rep_seqs = load("$(folder_name)/$(folder_name)__rep_seqs.jld2", "rep_seqs")
    global non_rep_seqs
        non_rep_seqs = load("$(folder_name)/$(folder_name)__non_rep_seqs.jld2", "non_rep_seqs")
    global all_seqs
        all_seqs = load("$(folder_name)/$(folder_name)__all_seqs.jld2", "all_seqs")
    global domain_mut_density_sort_by_gene
        domain_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_gene.jld2", "domain_mut_density_sort_by_gene")
    global gene_mut_density_sort_by_density
        gene_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_density.jld2", "gene_mut_density_sort_by_density")
    global nuc_muts_ct_sort_by_seq_ct
        nuc_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__nuc_muts_ct_sort_by_seq_ct.jld2", "nuc_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_sort_by_seq_ct
        AA_muts_ct_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_seq_ct.jld2", "AA_muts_ct_sort_by_seq_ct")
    global AA_muts_ct_no_dels_sort_by_seq_ct
        AA_muts_ct_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_sort_by_seq_ct")
    global AA_muts_ct_pos_only_no_dels_sort_by_site
        AA_muts_ct_pos_only_no_dels_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_site.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_site")
    global gene_mut_density_sort_by_gene
        gene_mut_density_sort_by_gene = load("$(folder_name)/$(folder_name)__gene_mut_density_sort_by_gene.jld2", "gene_mut_density_sort_by_gene")
    global AA_muts_ct_sort_by_site
        AA_muts_ct_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_sort_by_site.jld2", "AA_muts_ct_sort_by_site")
    global AA_muts_ct_pos_only_sort_by_site
        AA_muts_ct_pos_only_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_sort_by_site.jld2", "AA_muts_ct_pos_only_sort_by_site")
    global domain_mut_density_sort_by_density
        domain_mut_density_sort_by_density = load("$(folder_name)/$(folder_name)__domain_mut_density_sort_by_density.jld2", "domain_mut_density_sort_by_density")
    global too_many_reversions
        too_many_reversions = load("$(folder_name)/$(folder_name)__too_many_reversions.jld2", "too_many_reversions")
    global AA_muts_ct_pos_only_no_dels_sort_by_seq_ct
        AA_muts_ct_pos_only_no_dels_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_sort_by_seq_ct.jld2", "AA_muts_ct_pos_only_no_dels_sort_by_seq_ct")

    global AA_muts_ct_chr_all_ratio_ct_sort
        AA_muts_ct_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_chr_all_ratio_ct_sort")
    global AA_muts_ct_chr_all_ratio_pos_sort
        AA_muts_ct_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_chr_all_ratio_pos_sort")    

    global AA_muts_ct_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global AA_muts_ct_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_no_dels_chr_all_ratio_pos_sort")
    
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    
    global avg_AA_subs_per_chr_seq
        avg_AA_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq.jld2", "avg_AA_subs_per_chr_seq")
    global avg_AA_subs_per_circ_seq
        avg_AA_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_circ_seq.jld2", "avg_AA_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    global NSP_muts_sortByCt_Arr
        NSP_muts_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByCt_Arr.jld2", "NSP_muts_sortByCt_Arr")
    global NSP_muts_sortByPos_Arr
        NSP_muts_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_sortByPos_Arr.jld2", "NSP_muts_sortByPos_Arr")
    global NSP_muts_no_dels_sortByCt_Arr
        NSP_muts_no_dels_sortByCt_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByCt_Arr.jld2", "NSP_muts_no_dels_sortByCt_Arr")
    global NSP_muts_no_dels_sortByPos_Arr
        NSP_muts_no_dels_sortByPos_Arr = load("$(folder_name)/$(folder_name)__NSP_muts_no_dels_sortByPos_Arr.jld2", "NSP_muts_no_dels_sortByPos_Arr")
    global NSP1_muts_sortByCt
        NSP1_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByCt.jld2", "NSP1_muts_sortByCt")
    global NSP1_muts_sortByPos
        NSP1_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_sortByPos.jld2", "NSP1_muts_sortByPos")
    global NSP2_muts_sortByCt
        NSP2_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByCt.jld2", "NSP2_muts_sortByCt")
    global NSP2_muts_sortByPos
        NSP2_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_sortByPos.jld2", "NSP2_muts_sortByPos")
    global NSP3_muts_sortByCt
        NSP3_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByCt.jld2", "NSP3_muts_sortByCt")
    global NSP3_muts_sortByPos
        NSP3_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_sortByPos.jld2", "NSP3_muts_sortByPos")
    global NSP4_muts_sortByCt
        NSP4_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByCt.jld2", "NSP4_muts_sortByCt")
    global NSP4_muts_sortByPos
        NSP4_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_sortByPos.jld2", "NSP4_muts_sortByPos")
    global NSP5_muts_sortByCt
        NSP5_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByCt.jld2", "NSP5_muts_sortByCt")
    global NSP5_muts_sortByPos
        NSP5_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_sortByPos.jld2", "NSP5_muts_sortByPos")
    global NSP6_muts_sortByCt
        NSP6_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByCt.jld2", "NSP6_muts_sortByCt")
    global NSP6_muts_sortByPos
        NSP6_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_sortByPos.jld2", "NSP6_muts_sortByPos")
    global NSP7_muts_sortByCt
        NSP7_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByCt.jld2", "NSP7_muts_sortByCt")
    global NSP7_muts_sortByPos
        NSP7_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_sortByPos.jld2", "NSP7_muts_sortByPos")
    global NSP8_muts_sortByCt
        NSP8_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByCt.jld2", "NSP8_muts_sortByCt")
    global NSP8_muts_sortByPos
        NSP8_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_sortByPos.jld2", "NSP8_muts_sortByPos")
    global NSP9_muts_sortByCt
        NSP9_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByCt.jld2", "NSP9_muts_sortByCt")
    global NSP9_muts_sortByPos
        NSP9_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_sortByPos.jld2", "NSP9_muts_sortByPos")
    global NSP10_muts_sortByCt
        NSP10_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByCt.jld2", "NSP10_muts_sortByCt")
    global NSP10_muts_sortByPos
        NSP10_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_sortByPos.jld2", "NSP10_muts_sortByPos")
    global NSP11_muts_sortByCt
        NSP11_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByCt.jld2", "NSP11_muts_sortByCt")
    global NSP11_muts_sortByPos
        NSP11_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_sortByPos.jld2", "NSP11_muts_sortByPos")
    global NSP12_muts_sortByCt
        NSP12_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByCt.jld2", "NSP12_muts_sortByCt")
    global NSP12_muts_sortByPos
        NSP12_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_sortByPos.jld2", "NSP12_muts_sortByPos")
    global NSP13_muts_sortByCt
        NSP13_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByCt.jld2", "NSP13_muts_sortByCt")
    global NSP13_muts_sortByPos
        NSP13_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_sortByPos.jld2", "NSP13_muts_sortByPos")
    global NSP14_muts_sortByCt
        NSP14_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByCt.jld2", "NSP14_muts_sortByCt")
    global NSP14_muts_sortByPos
        NSP14_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_sortByPos.jld2", "NSP14_muts_sortByPos")
    global NSP15_muts_sortByCt
        NSP15_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByCt.jld2", "NSP15_muts_sortByCt")
    global NSP15_muts_sortByPos
        NSP15_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_sortByPos.jld2", "NSP15_muts_sortByPos")
    global NSP16_muts_sortByCt
        NSP16_muts_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByCt.jld2", "NSP16_muts_sortByCt")
    global NSP16_muts_sortByPos
        NSP16_muts_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_sortByPos.jld2", "NSP16_muts_sortByPos")
    global NSP1_muts_no_dels_sortByCt
        NSP1_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByCt.jld2", "NSP1_muts_no_dels_sortByCt")
    global NSP1_muts_no_dels_sortByPos
        NSP1_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP1_muts_no_dels_sortByPos.jld2", "NSP1_muts_no_dels_sortByPos")
    global NSP2_muts_no_dels_sortByCt
        NSP2_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByCt.jld2", "NSP2_muts_no_dels_sortByCt")
    global NSP2_muts_no_dels_sortByPos
        NSP2_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP2_muts_no_dels_sortByPos.jld2", "NSP2_muts_no_dels_sortByPos")
    global NSP3_muts_no_dels_sortByCt
        NSP3_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByCt.jld2", "NSP3_muts_no_dels_sortByCt")
    global NSP3_muts_no_dels_sortByPos
        NSP3_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP3_muts_no_dels_sortByPos.jld2", "NSP3_muts_no_dels_sortByPos")
    global NSP4_muts_no_dels_sortByCt
        NSP4_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByCt.jld2", "NSP4_muts_no_dels_sortByCt")
    global NSP4_muts_no_dels_sortByPos
        NSP4_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP4_muts_no_dels_sortByPos.jld2", "NSP4_muts_no_dels_sortByPos")
    global NSP5_muts_no_dels_sortByCt
        NSP5_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByCt.jld2", "NSP5_muts_no_dels_sortByCt")
    global NSP5_muts_no_dels_sortByPos
        NSP5_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP5_muts_no_dels_sortByPos.jld2", "NSP5_muts_no_dels_sortByPos")
    global NSP6_muts_no_dels_sortByCt
        NSP6_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByCt.jld2", "NSP6_muts_no_dels_sortByCt")
    global NSP6_muts_no_dels_sortByPos
        NSP6_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP6_muts_no_dels_sortByPos.jld2", "NSP6_muts_no_dels_sortByPos")
    global NSP7_muts_no_dels_sortByCt
        NSP7_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByCt.jld2", "NSP7_muts_no_dels_sortByCt")
    global NSP7_muts_no_dels_sortByPos
        NSP7_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP7_muts_no_dels_sortByPos.jld2", "NSP7_muts_no_dels_sortByPos")
    global NSP8_muts_no_dels_sortByCt
        NSP8_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByCt.jld2", "NSP8_muts_no_dels_sortByCt")
    global NSP8_muts_no_dels_sortByPos
        NSP8_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP8_muts_no_dels_sortByPos.jld2", "NSP8_muts_no_dels_sortByPos")
    global NSP9_muts_no_dels_sortByCt
        NSP9_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByCt.jld2", "NSP9_muts_no_dels_sortByCt")
    global NSP9_muts_no_dels_sortByPos
        NSP9_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP9_muts_no_dels_sortByPos.jld2", "NSP9_muts_no_dels_sortByPos")
    global NSP10_muts_no_dels_sortByCt
        NSP10_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByCt.jld2", "NSP10_muts_no_dels_sortByCt")
    global NSP10_muts_no_dels_sortByPos
        NSP10_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP10_muts_no_dels_sortByPos.jld2", "NSP10_muts_no_dels_sortByPos")
    global NSP11_muts_no_dels_sortByCt
        NSP11_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByCt.jld2", "NSP11_muts_no_dels_sortByCt")
    global NSP11_muts_no_dels_sortByPos
        NSP11_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP11_muts_no_dels_sortByPos.jld2", "NSP11_muts_no_dels_sortByPos")
    global NSP12_muts_no_dels_sortByCt
        NSP12_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByCt.jld2", "NSP12_muts_no_dels_sortByCt")
    global NSP12_muts_no_dels_sortByPos
        NSP12_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP12_muts_no_dels_sortByPos.jld2", "NSP12_muts_no_dels_sortByPos")
    global NSP13_muts_no_dels_sortByCt
        NSP13_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByCt.jld2", "NSP13_muts_no_dels_sortByCt")
    global NSP13_muts_no_dels_sortByPos
        NSP13_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP13_muts_no_dels_sortByPos.jld2", "NSP13_muts_no_dels_sortByPos")
    global NSP14_muts_no_dels_sortByCt
        NSP14_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByCt.jld2", "NSP14_muts_no_dels_sortByCt")
    global NSP14_muts_no_dels_sortByPos
        NSP14_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP14_muts_no_dels_sortByPos.jld2", "NSP14_muts_no_dels_sortByPos")
    global NSP15_muts_no_dels_sortByCt
        NSP15_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByCt.jld2", "NSP15_muts_no_dels_sortByCt")
    global NSP15_muts_no_dels_sortByPos
        NSP15_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP15_muts_no_dels_sortByPos.jld2", "NSP15_muts_no_dels_sortByPos")
    global NSP16_muts_no_dels_sortByCt
        NSP16_muts_no_dels_sortByCt = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByCt.jld2", "NSP16_muts_no_dels_sortByCt")
    global NSP16_muts_no_dels_sortByPos
        NSP16_muts_no_dels_sortByPos = load("$(folder_name)/$(folder_name)__NSP16_muts_no_dels_sortByPos.jld2", "NSP16_muts_no_dels_sortByPos")
####################################################################################################
    global all_seqs_set
        all_seqs_set = load("$(folder_name)/$(folder_name)__all_seqs_set.jld2", "all_seqs_set")
    global all_qualifying_seqs
        all_qualifying_seqs = load("$(folder_name)/$(folder_name)__all_qualifying_seqs.jld2", "all_qualifying_seqs")
    global all_qualifying_seqs_set
        all_qualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_qualifying_seqs_set.jld2", "all_qualifying_seqs_set")
    global all_nonqualifying_seqs
        all_nonqualifying_seqs = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs.jld2", "all_nonqualifying_seqs")
    global all_nonqualifying_seqs_set
        all_nonqualifying_seqs_set = load("$(folder_name)/$(folder_name)__all_nonqualifying_seqs_set.jld2", "all_nonqualifying_seqs_set")
###########################################################################################################################################################################
    global AA_muts_ct_no_dels_no_revs_sort_by_site
        AA_muts_ct_no_dels_no_revs_sort_by_site = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_site.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_site")
    global AA_muts_ct_no_dels_no_revs_sort_by_seq_ct
        AA_muts_ct_no_dels_no_revs_sort_by_seq_ct = load("$(folder_name)/$(folder_name)__AA_muts_ct_no_dels_no_revs_sort_by_seq_ct.jld2", "AA_muts_ct_no_dels_no_revs_sort_by_seq_ct")
    global chronic_search_muts_v2
        chronic_search_muts_v2 = load("$(folder_name)/$(folder_name)__chronic_search_muts_v2.jld2", "chronic_search_muts_v2")
    global avg_AA_subs_per_chr_seq_no_revs
        avg_AA_subs_per_chr_seq_no_revs = load("$(folder_name)/$(folder_name)__avg_AA_subs_per_chr_seq_no_revs.jld2", "avg_AA_subs_per_chr_seq_no_revs")
###########################################################################################################################################################################
    global nuc_muts_ct_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_ct_sort")
    global nuc_muts_ct_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort")
    global nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort
        nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort = load("$(folder_name)/$(folder_name)__nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort.jld2", "nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort")
    global avg_nuc_subs_per_chr_seq
        avg_nuc_subs_per_chr_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_chr_seq.jld2", "avg_nuc_subs_per_chr_seq")
    global avg_nuc_subs_per_circ_seq
        avg_nuc_subs_per_circ_seq = load("$(folder_name)/$(folder_name)__avg_nuc_subs_per_circ_seq.jld2", "avg_nuc_subs_per_circ_seq")
###########################################################################################################################################################################
###########################################################################################################################################################################
    return date_nuc_mut_ct, date_nuc_mut_ct_no_dels, date_AA_mut_ct, date_AA_mut_ct_no_dels, date_AA_mut_ct_pos_only_no_dels, 
    seq_ct_by_year, seq_ct_by_year_month, seq_ct_by_year_month_day, 
    seq_collection_date, seq_date_index, seq_date_tuple, 
###################################################################################################################################
    seq_clade, seq_clade_display, seq_pango, 
    seq_pango_unaliased, seq_clade_ct, seq_pango_ct, seq_pango_unaliased_ct, 
###################################################################################################################################
    too_many_reversions, 
###################################################################################################################################
    seq_nuc_muts, seq_nuc_del_ranges, seq_nuc_dropout, seq_nuc_muts_WT, seq_nuc_del_ranges_WT, 
################################
    seq_AA_insertions_WT, seq_nuc_insertions_WT,
################################
    seq_AA_muts, seq_AA_muts_no_dels, seq_AA_muts_pos_only, seq_AA_dels, seq_mixed_AA_muts, 
    seq_AA_muts_WT, seq_AA_muts_WT_pos_only, seq_AA_dels_WT, seq_AA_muts_pos_only_no_dels, 
########################################################
    nuc_muts_seq, nuc_muts_seq_WT, 
    nuc_dels_seq, nuc_dels_seq_WT, 
    AA_muts_seq, AA_muts_seq_pos_only, AA_muts_seq_WT, AA_muts_seq_WT_pos_only, AA_muts_seq_pos_only_no_dels, 
    AA_dels_seq, AA_dels_seq_WT, 
######################################################## 
    seq_unknown_AA, seq_unknown_AA_ranges, seq_mixed_nucs, 
############################
    nuc_dels_ct, nuc_muts_ct, nuc_muts_ct_no_dels, 
###############
    AA_dels_ct, AA_muts_ct, AA_muts_ct_no_dels, AA_muts_ct_pos_only, AA_muts_ct_pos_only_no_dels, AA_muts_ct_no_dels_no_revs, 
############################
    nuc_muts_ct_sort_by_site, nuc_muts_ct_sort_by_seq_ct, 
############################ 
    AA_muts_ct_sort_by_site, AA_muts_ct_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_site, AA_muts_ct_no_dels_no_revs_sort_by_site, 
    AA_muts_ct_no_dels_no_revs_sort_by_seq_ct, AA_muts_ct_no_dels_sort_by_seq_ct,  AA_muts_ct_pos_only_sort_by_site, 
    AA_muts_ct_pos_only_sort_by_seq_ct, AA_muts_ct_pos_only_no_dels_sort_by_site, AA_muts_ct_pos_only_no_dels_sort_by_seq_ct, 
###################################################################################################################################
################################################################################################################################### 
    nuc_muts_ct_adj, nuc_muts_ct_adj_no_dels, nuc_muts_ct_adj_score, nuc_muts_ct_adj_score_no_dels,   
    nuc_muts_ct_adj_sort_by_seq_ct, nuc_muts_ct_adj_no_dels_sort_by_site, nuc_muts_ct_adj_no_dels_sort_by_seq_ct,
    nuc_muts_ct_adj_sort_by_site, nuc_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_score, 
    nuc_muts_ct_adj_score_sort_by_site, nuc_muts_ct_adj_score_sort_by_score, nuc_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_adj, AA_muts_ct_adj_no_dels, AA_muts_ct_pos_only_adj, AA_muts_ct_pos_only_adj_no_dels, 
    AA_muts_ct_adj_score, AA_muts_ct_adj_score_no_dels, AA_muts_ct_pos_only_adj_score, AA_muts_ct_pos_only_adj_score_no_dels,
    AA_muts_ct_adj_sort_by_site, AA_muts_ct_adj_sort_by_seq_ct, AA_muts_ct_adj_no_dels_sort_by_site,
    AA_muts_ct_adj_no_dels_sort_by_seq_ct, AA_muts_ct_adj_score_sort_by_score, AA_muts_ct_adj_score_sort_by_site, 
    AA_muts_ct_adj_score_no_dels_sort_by_score, AA_muts_ct_adj_score_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_sort_by_site, AA_muts_ct_pos_only_adj_sort_by_seq_ct, AA_muts_ct_pos_only_adj_no_dels_sort_by_site, 
    AA_muts_ct_pos_only_adj_no_dels_sort_by_seq_ct, AA_muts_ct_pos_only_adj_score_sort_by_site, 
    AA_muts_ct_pos_only_adj_score_sort_by_score, AA_muts_ct_pos_only_adj_score_no_dels_sort_by_site, 
####################################################################################################################################
    AA_muts_ct_chr_all_ratio, AA_muts_ct_no_dels_chr_all_ratio, AA_muts_ct_no_dels_no_revs_chr_all_ratio, AA_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
########################
    AA_muts_ct_chr_all_ratio_ct_sort, AA_muts_ct_chr_all_ratio_pos_sort, 
    AA_muts_ct_no_dels_chr_all_ratio_ct_sort, AA_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    AA_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, AA_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
###################################################### 
    nuc_muts_ct_no_dels_chr_all_ratio, nuc_muts_ct_no_dels_no_revs_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_chr_all_ratio, nuc_muts_ct_pos_only_no_dels_no_revs_chr_all_ratio, 
#######################
    nuc_muts_ct_no_dels_chr_all_ratio_ct_sort, nuc_muts_ct_no_dels_chr_all_ratio_pos_sort, 
    nuc_muts_ct_pos_only_no_dels_chr_all_ratio_pos_sort, nuc_muts_ct_pos_only_no_dels_chr_all_ratio_ct_sort, 
####################################################################################################################################
    NSP_muts, NSP_muts_sortByCt_Arr, NSP_muts_sortByPos_Arr, NSP_muts_no_dels_sortByCt_Arr, NSP_muts_no_dels_sortByPos_Arr, 
    NSP1_muts_sortByCt, NSP2_muts_sortByCt, NSP3_muts_sortByCt, NSP4_muts_sortByCt, NSP5_muts_sortByCt, NSP6_muts_sortByCt,
    NSP7_muts_sortByCt, NSP8_muts_sortByCt, NSP9_muts_sortByCt, NSP10_muts_sortByCt, NSP11_muts_sortByCt, NSP12_muts_sortByCt, 
    NSP13_muts_sortByCt, NSP14_muts_sortByCt, NSP15_muts_sortByCt, NSP16_muts_sortByCt, NSP1_muts_sortByPos, NSP2_muts_sortByPos, 
    NSP3_muts_sortByPos, NSP4_muts_sortByPos, NSP5_muts_sortByPos, NSP6_muts_sortByPos, NSP7_muts_sortByPos, NSP8_muts_sortByPos, 
    NSP9_muts_sortByPos, NSP10_muts_sortByPos, NSP11_muts_sortByPos, NSP12_muts_sortByPos, NSP13_muts_sortByPos, NSP14_muts_sortByPos, 
    NSP15_muts_sortByPos, NSP16_muts_sortByPos, NSP1_muts_no_dels_sortByCt, NSP2_muts_no_dels_sortByCt, NSP3_muts_no_dels_sortByCt, 
    NSP4_muts_no_dels_sortByCt, NSP5_muts_no_dels_sortByCt, NSP6_muts_no_dels_sortByCt, NSP7_muts_no_dels_sortByCt, 
    NSP8_muts_no_dels_sortByCt, NSP9_muts_no_dels_sortByCt, NSP10_muts_no_dels_sortByCt, NSP11_muts_no_dels_sortByCt, 
    NSP12_muts_no_dels_sortByCt, NSP13_muts_no_dels_sortByCt, NSP14_muts_no_dels_sortByCt, NSP15_muts_no_dels_sortByCt, 
    NSP16_muts_no_dels_sortByCt, NSP1_muts_no_dels_sortByPos, NSP2_muts_no_dels_sortByPos, NSP3_muts_no_dels_sortByPos, 
    NSP4_muts_no_dels_sortByPos, NSP5_muts_no_dels_sortByPos, NSP6_muts_no_dels_sortByPos, NSP7_muts_no_dels_sortByPos, 
    NSP8_muts_no_dels_sortByPos, NSP9_muts_no_dels_sortByPos, NSP10_muts_no_dels_sortByPos, NSP11_muts_no_dels_sortByPos, 
    NSP12_muts_no_dels_sortByPos, NSP13_muts_no_dels_sortByPos, NSP14_muts_no_dels_sortByPos, NSP15_muts_no_dels_sortByPos, 
    NSP16_muts_no_dels_sortByPos,
####################################################################################################################################
    chronic_search_muts, chronic_search_muts_v2, 
####################################################################################################################################
    avg_AA_subs_per_circ_seq, avg_nuc_subs_per_circ_seq, avg_AA_subs_per_chr_seq, avg_nuc_subs_per_chr_seq,
####################################################################################################################################
    nuc_muts_rep_seq_grps, nuc_dels_rep_seq_grps, 
    nuc_muts_rep_seq_grps_no_dels, rep_seq_grps_dels, rep_seq_grps_muts_no_dels, rep_seq_grps_del_ranges_ct, rep_seq_grps_nuc_dropout, 
    rep_seq_grps_mixed_nucs, rep_seq_grps_AA_dels, rep_seq_grps_AA_no_dels, AA_muts_rep_seq_grps, AA_dels_rep_seq_grps, 
    AA_muts_rep_seq_grps_no_dels, AA_muts_rep_seq_grps_pos_only,  
    rep_seq_grps_nuc_muts_WT, rep_seq_grps_nuc_dels_WT, rep_seq_grps_AA_muts_WT, rep_seq_grps_AA_dels_WT, nuc_muts_rep_seq_grps_WT, 
    nuc_dels_rep_seq_grps_WT, AA_muts_rep_seq_grps_WT, AA_dels_rep_seq_grps_WT, rep_seq_grps_AA_muts_WT_pos_only, 
    AA_muts_rep_seq_grps_WT_pos_only, rep_seq_grps_pango, rep_seq_grps_pango_unaliased, excluded_pos, excluded_AA, all_seqs, rep_seqs, 
    non_rep_seqs, rep_seq_grps_seqs, rep_seq_grps_muts, rep_seq_grps_clade, rep_seq_grps_AA, rep_seq_grps_AA_pos_only, 
    rep_seq_grps_AA_pos_only_no_dels, non_rep_seqs_AA, non_rep_seqs_AA_pos_only, non_rep_seqs_AA_pos_only_no_dels, 
####################################################################################################################################
    rep_seq_grps_maxmut_nuc_muts_WT, rep_seq_grps_maxmut_nuc_dels_WT, 
    rep_seq_grps_maxmut_AA_muts_WT, rep_seq_grps_maxmut_AA_dels_WT, rep_seq_grps_maxmut_AA_muts_WT_pos_only,
    rep_seq_grps_maxmut_seqs, rep_seq_grps_maxmut_nuc, rep_seq_grps_maxmut_nuc_no_dels, rep_seq_grps_maxmut_dels, 
    rep_seq_grps_maxmut_nuc_dropout, rep_seq_grps_maxmut_mixed_nucs, rep_seq_grps_maxmut_AA, rep_seq_grps_maxmut_AA_no_dels, 
    rep_seq_grps_maxmut_AA_dels, rep_seq_grps_maxmut_AA_pos_only, rep_seq_grps_maxmut_AA_pos_only_no_dels, 
    rep_seq_grps_maxmut_unknown_AA, rep_seq_grps_maxmut_unknown_AA_ranges, rep_seq_grps_maxmut_mixed_AA_muts, rep_seq_grps_maxmut_del_ranges_ct,
####################################################################################################################################
    gene_mut_density_sort_by_gene, gene_mut_density_sort_by_density, domain_mut_density_sort_by_gene, domain_mut_density_sort_by_density, 
    gene_mut_density, domain_mut_density, 
####################################################################################################################################
    all_seqs_set, all_qualifying_seqs, all_qualifying_seqs_set, all_nonqualifying_seqs, all_nonqualifying_seqs_set
####################################################################################################################################
    total_chr_AA_subs, total_nuc_revs_seq, seq_nuc_total_revs, total_AA_revs_seq, seq_AA_total_revs, seq_AA_revs
end
# REMOVED: rep_seq_grps_unknown_AA, rep_seq_grps_mixed_AA_muts,
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_chr(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    ORF1a_pos_ct = Dict{String, Int}()
    ORF1b_pos_ct = Dict{String, Int}()
    S_pos_ct = Dict{String, Int}()
    ORF3a_pos_ct = Dict{String, Int}()
    E_pos_ct = Dict{String, Int}()
    M_pos_ct = Dict{String, Int}()
    ORF6_pos_ct = Dict{String, Int}()
    ORF7a_pos_ct = Dict{String, Int}()
    ORF7b_pos_ct = Dict{String, Int}()
    ORF8_pos_ct = Dict{String, Int}()
    N_pos_ct = Dict{String, Int}()
    ORF9b_pos_ct = Dict{String, Int}()
    gene_pos_ct_dict_arr = [ORF1a_pos_ct, ORF1b_pos_ct, S_pos_ct, ORF3a_pos_ct, E_pos_ct, M_pos_ct, ORF6_pos_ct, ORF7a_pos_ct, ORF7b_pos_ct, ORF8_pos_ct, N_pos_ct, ORF9b_pos_ct]
######################################################
    ORF1a_pos_ct_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_v1 = Dict{Int, Int}()
    S_pos_ct_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_v1 = Dict{Int, Int}()
    E_pos_ct_v1 = Dict{Int, Int}()
    M_pos_ct_v1 = Dict{Int, Int}()
    ORF6_pos_ct_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_v1 = Dict{Int, Int}()
    ORF8_pos_ct_v1 = Dict{Int, Int}()
    N_pos_ct_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_v1 = Dict{Int, Int}()
    gene_pos_ct_v1_dict_arr = [ORF1a_pos_ct_v1, ORF1b_pos_ct_v1, S_pos_ct_v1, ORF3a_pos_ct_v1, E_pos_ct_v1, M_pos_ct_v1, ORF6_pos_ct_v1, ORF7a_pos_ct_v1, ORF7b_pos_ct_v1, ORF8_pos_ct_v1, N_pos_ct_v1, ORF9b_pos_ct_v1]
######################################################
    for i in 1:length(gene_pos_ct_dict_arr)
        dict = gene_pos_ct_dict_arr[i]
        dict_v1 = gene_pos_ct_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct[mut] = ct
            ORF1a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct[mut] = ct
            ORF1b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct[mut] = ct
            S_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct[mut] = ct
            ORF3a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct[mut] = ct
            E_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct[mut] = ct
            M_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct[mut] = ct
            ORF6_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct[mut] = ct
            ORF7a_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct[mut] = ct
            ORF7b_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct[mut] = ct
            ORF8_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct[mut] = ct
            N_pos_ct_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct[mut] = ct
            ORF9b_pos_ct_v1[pos] = ct
        end
    end
######################### 
    global ORF1a_pos_ct_sort_by_pos = sort(collect(ORF1a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_sort_by_pos = sort(collect(ORF1b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_sort_by_pos = sort(collect(S_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_sort_by_pos = sort(collect(ORF3a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_sort_by_pos = sort(collect(E_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global M_pos_ct_sort_by_pos = sort(collect(M_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_sort_by_pos = sort(collect(ORF6_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_sort_by_pos = sort(collect(ORF7a_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_sort_by_pos = sort(collect(ORF7b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_sort_by_pos = sort(collect(ORF8_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_sort_by_pos = sort(collect(N_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF9b_pos_ct_sort_by_pos = sort(collect(ORF9b_pos_ct), by = x -> aa_pos_comprehensive_dict[x[1]])
#########################
    global ORF1a_pos_ct_sort_by_ct = sort(collect(ORF1a_pos_ct), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct = sort(collect(ORF1b_pos_ct), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct = sort(collect(S_pos_ct), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct = sort(collect(ORF3a_pos_ct), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct = sort(collect(E_pos_ct), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct = sort(collect(M_pos_ct), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct = sort(collect(ORF6_pos_ct), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct = sort(collect(ORF7a_pos_ct), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct = sort(collect(ORF7b_pos_ct), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct = sort(collect(ORF8_pos_ct), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct = sort(collect(N_pos_ct), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct = sort(collect(ORF9b_pos_ct), by = x -> x[2], rev=true)
#########################
    global ORF1a_pos_ct_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[1])
    global ORF1b_pos_ct_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[1])
    global S_pos_ct_sort_by_pos_v1 = sort(collect(S_pos_ct_v1), by = x -> x[1])
    global ORF3a_pos_ct_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[1])
    global E_pos_ct_sort_by_pos_v1 = sort(collect(E_pos_ct_v1), by = x -> x[1])
    global M_pos_ct_sort_by_pos_v1 = sort(collect(M_pos_ct_v1), by = x -> x[1])
    global ORF6_pos_ct_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[1])
    global ORF7a_pos_ct_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[1])
    global ORF7b_pos_ct_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[1])
    global ORF8_pos_ct_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[1])
    global N_pos_ct_sort_by_pos_v1 = sort(collect(N_pos_ct_v1), by = x -> x[1])
    global ORF9b_pos_ct_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[1])
#########################
    global ORF1a_pos_ct_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_v1), by = x -> x[2], rev=true)
    global S_pos_ct_sort_by_ct_v1 = sort(collect(S_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_v1), by = x -> x[2], rev=true)
    global E_pos_ct_sort_by_ct_v1 = sort(collect(E_pos_ct_v1), by = x -> x[2], rev=true)
    global M_pos_ct_sort_by_ct_v1 = sort(collect(M_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_v1), by = x -> x[2], rev=true)
    global N_pos_ct_sort_by_ct_v1 = sort(collect(N_pos_ct_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_v1), by = x -> x[2], rev=true)
end
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
function gene_sub_ranks_chr(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    ORF1a_ct = Dict{String, Int}()
    ORF1b_ct = Dict{String, Int}()
    S_ct = Dict{String, Int}()
    ORF3a_ct = Dict{String, Int}()
    E_ct = Dict{String, Int}()
    M_ct = Dict{String, Int}()
    ORF6_ct = Dict{String, Int}()
    ORF7a_ct = Dict{String, Int}()
    ORF7b_ct = Dict{String, Int}()
    ORF8_ct = Dict{String, Int}()
    N_ct = Dict{String, Int}()
    ORF9b_ct = Dict{String, Int}()
#############################################################
    gene_ct_dict_arr = [ORF1a_ct, ORF1b_ct, S_ct, ORF3a_ct, E_ct, M_ct, ORF6_ct, ORF7a_ct, ORF7b_ct, ORF8_ct, N_ct, ORF9b_ct] 
#############################################################
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
########################################## 
    global ORF1a_ct_sort_by_pos = sort(collect(ORF1a_ct), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_sort_by_pos = sort(collect(ORF1b_ct), by = x -> fin_sortkey(x[1]))
    global S_ct_sort_by_pos = sort(collect(S_ct), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_sort_by_pos = sort(collect(ORF3a_ct), by = x -> fin_sortkey(x[1]))
    global E_ct_sort_by_pos = sort(collect(E_ct), by = x -> fin_sortkey(x[1]))
    global M_ct_sort_by_pos = sort(collect(M_ct), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_sort_by_pos = sort(collect(ORF6_ct), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_sort_by_pos = sort(collect(ORF7a_ct), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_sort_by_pos = sort(collect(ORF7b_ct), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_sort_by_pos = sort(collect(ORF8_ct), by = x -> fin_sortkey(x[1]))
    global N_ct_sort_by_pos = sort(collect(N_ct), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_sort_by_pos = sort(collect(ORF9b_ct), by = x -> fin_sortkey(x[1]))
##########################################
    global ORF1a_ct_sort_by_ct = sort(collect(ORF1a_ct), by = x -> x[2], rev=true)
    global ORF1b_ct_sort_by_ct = sort(collect(ORF1b_ct), by = x -> x[2], rev=true)
    global S_ct_sort_by_ct = sort(collect(S_ct), by = x -> x[2], rev=true)
    global ORF3a_ct_sort_by_ct = sort(collect(ORF3a_ct), by = x -> x[2], rev=true)
    global E_ct_sort_by_ct = sort(collect(E_ct), by = x -> x[2], rev=true)
    global M_ct_sort_by_ct = sort(collect(M_ct), by = x -> x[2], rev=true)
    global ORF6_ct_sort_by_ct = sort(collect(ORF6_ct), by = x -> x[2], rev=true)
    global ORF7a_ct_sort_by_ct = sort(collect(ORF7a_ct), by = x -> x[2], rev=true)
    global ORF7b_ct_sort_by_ct = sort(collect(ORF7b_ct), by = x -> x[2], rev=true)
    global ORF8_ct_sort_by_ct = sort(collect(ORF8_ct), by = x -> x[2], rev=true)
    global N_ct_sort_by_ct = sort(collect(N_ct), by = x -> x[2], rev=true)
    global ORF9b_ct_sort_by_ct = sort(collect(ORF9b_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels::Dict{String, Int} )
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = aa_pos_comprehensive_dict[sub]
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
#############################
    global NSP1_ct_sort_by_pos = sort(collect(NSP1_ct), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_sort_by_pos = sort(collect(NSP2_ct), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_sort_by_pos = sort(collect(NSP3_ct), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_sort_by_pos = sort(collect(NSP4_ct), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_sort_by_pos = sort(collect(NSP5_ct), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_sort_by_pos = sort(collect(NSP6_ct), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_sort_by_pos = sort(collect(NSP7_ct), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_sort_by_pos = sort(collect(NSP8_ct), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_sort_by_pos = sort(collect(NSP9_ct), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_sort_by_pos = sort(collect(NSP10_ct), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_sort_by_pos = sort(collect(NSP11_ct), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_sort_by_pos = sort(collect(NSP12_ct), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_sort_by_pos = sort(collect(NSP13_ct), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_sort_by_pos = sort(collect(NSP14_ct), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_sort_by_pos = sort(collect(NSP15_ct), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_sort_by_pos = sort(collect(NSP16_ct), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_sort_by_ct = sort(collect(NSP1_ct), by = x -> x[2], rev=true)
    global NSP2_ct_sort_by_ct = sort(collect(NSP2_ct), by = x -> x[2], rev=true)
    global NSP3_ct_sort_by_ct = sort(collect(NSP3_ct), by = x -> x[2], rev=true)
    global NSP4_ct_sort_by_ct = sort(collect(NSP4_ct), by = x -> x[2], rev=true)
    global NSP5_ct_sort_by_ct = sort(collect(NSP5_ct), by = x -> x[2], rev=true)
    global NSP6_ct_sort_by_ct = sort(collect(NSP6_ct), by = x -> x[2], rev=true)
    global NSP7_ct_sort_by_ct = sort(collect(NSP7_ct), by = x -> x[2], rev=true)
    global NSP8_ct_sort_by_ct = sort(collect(NSP8_ct), by = x -> x[2], rev=true)
    global NSP9_ct_sort_by_ct = sort(collect(NSP9_ct), by = x -> x[2], rev=true)
    global NSP10_ct_sort_by_ct = sort(collect(NSP10_ct), by = x -> x[2], rev=true)
    global NSP11_ct_sort_by_ct = sort(collect(NSP11_ct), by = x -> x[2], rev=true)
    global NSP12_ct_sort_by_ct = sort(collect(NSP12_ct), by = x -> x[2], rev=true)
    global NSP13_ct_sort_by_ct = sort(collect(NSP13_ct), by = x -> x[2], rev=true)
    global NSP14_ct_sort_by_ct = sort(collect(NSP14_ct), by = x -> x[2], rev=true)
    global NSP15_ct_sort_by_ct = sort(collect(NSP15_ct), by = x -> x[2], rev=true)
    global NSP16_ct_sort_by_ct = sort(collect(NSP16_ct), by = x -> x[2], rev=true)
end
#####################################################################################################################################
function NSP_sub_pos_ranks_chr(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_pos_only_no_dels::Dict{String, Int} )
    NSP1_pos_ct = Dict{String, Int}()
    NSP2_pos_ct = Dict{String, Int}()
    NSP3_pos_ct = Dict{String, Int}()
    NSP4_pos_ct = Dict{String, Int}()
    NSP5_pos_ct = Dict{String, Int}()
    NSP6_pos_ct = Dict{String, Int}()
    NSP7_pos_ct = Dict{String, Int}()
    NSP8_pos_ct = Dict{String, Int}()
    NSP9_pos_ct = Dict{String, Int}()
    NSP10_pos_ct = Dict{String, Int}()
    NSP11_pos_ct = Dict{String, Int}()
    NSP12_pos_ct = Dict{String, Int}()
    NSP13_pos_ct = Dict{String, Int}()
    NSP14_pos_ct = Dict{String, Int}()
    NSP15_pos_ct = Dict{String, Int}()
    NSP16_pos_ct = Dict{String, Int}()
    NSP_pos_ct_array = [NSP1_pos_ct, NSP2_pos_ct, NSP3_pos_ct, NSP4_pos_ct, NSP5_pos_ct, NSP6_pos_ct, NSP7_pos_ct, NSP8_pos_ct, NSP9_pos_ct, NSP10_pos_ct, NSP11_pos_ct, NSP12_pos_ct, NSP13_pos_ct, NSP14_pos_ct, NSP15_pos_ct, NSP16_pos_ct]
##########################################
    NSP1_pos_ct_v1 = Dict{Int, Int}()
    NSP2_pos_ct_v1 = Dict{Int, Int}()
    NSP3_pos_ct_v1 = Dict{Int, Int}()
    NSP4_pos_ct_v1 = Dict{Int, Int}()
    NSP5_pos_ct_v1 = Dict{Int, Int}()
    NSP6_pos_ct_v1 = Dict{Int, Int}()
    NSP7_pos_ct_v1 = Dict{Int, Int}()
    NSP8_pos_ct_v1 = Dict{Int, Int}()
    NSP9_pos_ct_v1 = Dict{Int, Int}()
    NSP10_pos_ct_v1 = Dict{Int, Int}()
    NSP11_pos_ct_v1 = Dict{Int, Int}()
    NSP12_pos_ct_v1 = Dict{Int, Int}()
    NSP13_pos_ct_v1 = Dict{Int, Int}()
    NSP14_pos_ct_v1 = Dict{Int, Int}()
    NSP15_pos_ct_v1 = Dict{Int, Int}()
    NSP16_pos_ct_v1 = Dict{Int, Int}()
    NSP_pos_ct_v1_array = [NSP1_pos_ct_v1, NSP2_pos_ct_v1, NSP3_pos_ct_v1, NSP4_pos_ct_v1, NSP5_pos_ct_v1, NSP6_pos_ct_v1, NSP7_pos_ct_v1, NSP8_pos_ct_v1, NSP9_pos_ct_v1, NSP10_pos_ct_v1, NSP11_pos_ct_v1, NSP12_pos_ct_v1, NSP13_pos_ct_v1, NSP14_pos_ct_v1, NSP15_pos_ct_v1, NSP16_pos_ct_v1]
#######################################################
    NSP_pos_ct = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_array)
        NSP_dict = NSP_pos_ct_array[i]
        NSP_dict_v1 = NSP_pos_ct_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_dict[NSP_pos] = 0
            NSP_dict_v1[j] = 0
            NSP_pos_ct[NSP_pos] = 0
        end
    end
##########################################################################################################################
##########################################################################################################################
    NSP1_ct = Dict{String, Int}()
    NSP2_ct = Dict{String, Int}()
    NSP3_ct = Dict{String, Int}()
    NSP4_ct = Dict{String, Int}()
    NSP5_ct = Dict{String, Int}()
    NSP6_ct = Dict{String, Int}()
    NSP7_ct = Dict{String, Int}()
    NSP8_ct = Dict{String, Int}()
    NSP9_ct = Dict{String, Int}()
    NSP10_ct = Dict{String, Int}()
    NSP11_ct = Dict{String, Int}()
    NSP12_ct = Dict{String, Int}()
    NSP13_ct = Dict{String, Int}()
    NSP14_ct = Dict{String, Int}()
    NSP15_ct = Dict{String, Int}()
    NSP16_ct = Dict{String, Int}()
#############################################################
    NSP_ct_dict_arr = [NSP1_ct, NSP2_ct, NSP3_ct, NSP4_ct, NSP5_ct, NSP6_ct, NSP7_ct, NSP8_ct, NSP9_ct, NSP10_ct, NSP11_ct, NSP12_ct, NSP13_ct, NSP14_ct, NSP15_ct, NSP16_ct]
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            NSP_num = parse(Int, split(NSP, "P")[2])
            NSP_dict = NSP_ct_dict_arr[NSP_num]
            NSP_dict[NSP_sub] = ct
        end
    end
##########################################################################################################################
##########################################################################################################################
    for i in 1:length(NSP_ct_dict_arr)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = NSP_muts_pos_dict[sub]
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct[sub_pos] += ct
        end
    end
    global NSP1_pos_ct_sort_by_pos = sort(collect(NSP1_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_sort_by_pos = sort(collect(NSP2_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_sort_by_pos = sort(collect(NSP3_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_sort_by_pos = sort(collect(NSP4_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_sort_by_pos = sort(collect(NSP5_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_sort_by_pos = sort(collect(NSP6_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_sort_by_pos = sort(collect(NSP7_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_sort_by_pos = sort(collect(NSP8_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_sort_by_pos = sort(collect(NSP9_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_sort_by_pos = sort(collect(NSP10_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_sort_by_pos = sort(collect(NSP11_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_sort_by_pos = sort(collect(NSP12_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_sort_by_pos = sort(collect(NSP13_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_sort_by_pos = sort(collect(NSP14_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_sort_by_pos = sort(collect(NSP15_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_sort_by_pos = sort(collect(NSP16_pos_ct), by = x -> NSP_muts_pos_dict[x[1]])
#########################
    global NSP1_pos_ct_sort_by_ct = sort(collect(NSP1_pos_ct), by = x -> x[2], rev=true)
    global NSP2_pos_ct_sort_by_ct = sort(collect(NSP2_pos_ct), by = x -> x[2], rev=true)
    global NSP3_pos_ct_sort_by_ct = sort(collect(NSP3_pos_ct), by = x -> x[2], rev=true)
    global NSP4_pos_ct_sort_by_ct = sort(collect(NSP4_pos_ct), by = x -> x[2], rev=true)
    global NSP5_pos_ct_sort_by_ct = sort(collect(NSP5_pos_ct), by = x -> x[2], rev=true)
    global NSP6_pos_ct_sort_by_ct = sort(collect(NSP6_pos_ct), by = x -> x[2], rev=true)
    global NSP7_pos_ct_sort_by_ct = sort(collect(NSP7_pos_ct), by = x -> x[2], rev=true)
    global NSP8_pos_ct_sort_by_ct = sort(collect(NSP8_pos_ct), by = x -> x[2], rev=true)
    global NSP9_pos_ct_sort_by_ct = sort(collect(NSP9_pos_ct), by = x -> x[2], rev=true)
    global NSP10_pos_ct_sort_by_ct = sort(collect(NSP10_pos_ct), by = x -> x[2], rev=true)
    global NSP11_pos_ct_sort_by_ct = sort(collect(NSP11_pos_ct), by = x -> x[2], rev=true)
    global NSP12_pos_ct_sort_by_ct = sort(collect(NSP12_pos_ct), by = x -> x[2], rev=true)
    global NSP13_pos_ct_sort_by_ct = sort(collect(NSP13_pos_ct), by = x -> x[2], rev=true)
    global NSP14_pos_ct_sort_by_ct = sort(collect(NSP14_pos_ct), by = x -> x[2], rev=true)
    global NSP15_pos_ct_sort_by_ct = sort(collect(NSP15_pos_ct), by = x -> x[2], rev=true)
    global NSP16_pos_ct_sort_by_ct = sort(collect(NSP16_pos_ct), by = x -> x[2], rev=true)
#########################
    global NSP1_pos_ct_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_v1), by = x -> x[1])
    global NSP2_pos_ct_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_v1), by = x -> x[1])
    global NSP3_pos_ct_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_v1), by = x -> x[1])
    global NSP4_pos_ct_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_v1), by = x -> x[1])
    global NSP5_pos_ct_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_v1), by = x -> x[1])
    global NSP6_pos_ct_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_v1), by = x -> x[1])
    global NSP7_pos_ct_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_v1), by = x -> x[1])
    global NSP8_pos_ct_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_v1), by = x -> x[1])
    global NSP9_pos_ct_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_v1), by = x -> x[1])
    global NSP10_pos_ct_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_v1), by = x -> x[1])
    global NSP11_pos_ct_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_v1), by = x -> x[1])
    global NSP12_pos_ct_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_v1), by = x -> x[1])
    global NSP13_pos_ct_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_v1), by = x -> x[1])
    global NSP14_pos_ct_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_v1), by = x -> x[1])
    global NSP15_pos_ct_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_v1), by = x -> x[1])
    global NSP16_pos_ct_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_v1), by = x -> x[1])
end
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_ranks_all(gene_array::Vector{String}, sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    ORF1a_ct_all = Dict{String, Int}()
    ORF1b_ct_all = Dict{String, Int}()
    S_ct_all = Dict{String, Int}()
    ORF3a_ct_all = Dict{String, Int}()
    E_ct_all = Dict{String, Int}()
    M_ct_all = Dict{String, Int}()
    ORF6_ct_all = Dict{String, Int}()
    ORF7a_ct_all = Dict{String, Int}()
    ORF7b_ct_all = Dict{String, Int}()
    ORF8_ct_all = Dict{String, Int}()
    N_ct_all = Dict{String, Int}()
    ORF9b_ct_all = Dict{String, Int}()
#############################################################
    #                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
    gene_ct_dict_all_arr = [ORF1a_ct_all, ORF1b_ct_all, S_ct_all, ORF3a_ct_all, E_ct_all, M_ct_all, ORF6_ct_all, ORF7a_ct_all, ORF7b_ct_all, ORF8_ct_all, N_ct_all, ORF9b_ct_all]
    for i in 1:length(gene_array)
        gene = gene_array[i]
        gene_ct_dict = gene_ct_dict_all_arr[i]
        for (AAsite, mut_type_set) in sub_types_at_every_site_combined[gene]
            for mut_type in mut_type_set
                sub = gene*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                gene_ct_dict[sub] = 0
            end
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_no_dels_all
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_ct_all[mut] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_ct_all[mut] = ct
        end
    end
    fin_sortkey(m) = (aa_pos_comprehensive_dict[m], refAA_comprehensive_dict[m]*qryAA_comprehensive_dict[m])
    global ORF1a_ct_all_sort_by_pos = sort(collect(ORF1a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF1b_ct_all_sort_by_pos = sort(collect(ORF1b_ct_all), by = x -> fin_sortkey(x[1]))
    global S_ct_all_sort_by_pos = sort(collect(S_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF3a_ct_all_sort_by_pos = sort(collect(ORF3a_ct_all), by = x -> fin_sortkey(x[1]))
    global E_ct_all_sort_by_pos = sort(collect(E_ct_all), by = x -> fin_sortkey(x[1]))
    global M_ct_all_sort_by_pos = sort(collect(M_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF6_ct_all_sort_by_pos = sort(collect(ORF6_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7a_ct_all_sort_by_pos = sort(collect(ORF7a_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF7b_ct_all_sort_by_pos = sort(collect(ORF7b_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF8_ct_all_sort_by_pos = sort(collect(ORF8_ct_all), by = x -> fin_sortkey(x[1]))
    global N_ct_all_sort_by_pos = sort(collect(N_ct_all), by = x -> fin_sortkey(x[1]))
    global ORF9b_ct_all_sort_by_pos = sort(collect(ORF9b_ct_all), by = x -> fin_sortkey(x[1]))

    global ORF1b_ct_all_sort_by_ct = sort(collect(ORF1b_ct_all), by = x -> x[2], rev=true)
    global ORF1a_ct_all_sort_by_ct = sort(collect(ORF1a_ct_all), by = x -> x[2], rev=true)
    global S_ct_all_sort_by_ct = sort(collect(S_ct_all), by = x -> x[2], rev=true)
    global ORF3a_ct_all_sort_by_ct = sort(collect(ORF3a_ct_all), by = x -> x[2], rev=true)
    global E_ct_all_sort_by_ct = sort(collect(E_ct_all), by = x -> x[2], rev=true)
    global M_ct_all_sort_by_ct = sort(collect(M_ct_all), by = x -> x[2], rev=true)
    global ORF6_ct_all_sort_by_ct = sort(collect(ORF6_ct_all), by = x -> x[2], rev=true)
    global ORF7a_ct_all_sort_by_ct = sort(collect(ORF7a_ct_all), by = x -> x[2], rev=true)
    global ORF7b_ct_all_sort_by_ct = sort(collect(ORF7b_ct_all), by = x -> x[2], rev=true)
    global ORF8_ct_all_sort_by_ct = sort(collect(ORF8_ct_all), by = x -> x[2], rev=true)
    global N_ct_all_sort_by_ct = sort(collect(N_ct_all), by = x -> x[2], rev=true)
    global ORF9b_ct_all_sort_by_ct = sort(collect(ORF9b_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function gene_sub_pos_ranks_all(gene_array::Vector{String}, gene_AA_lengths::Dict{String, Int},  AA_muts_ct_pos_only_no_dels_all::Dict{String, Int})
    ORF1a_pos_ct_all = Dict{String, Int}()
    ORF1b_pos_ct_all = Dict{String, Int}()
    S_pos_ct_all = Dict{String, Int}()
    ORF3a_pos_ct_all = Dict{String, Int}()
    E_pos_ct_all = Dict{String, Int}()
    M_pos_ct_all = Dict{String, Int}()
    ORF6_pos_ct_all = Dict{String, Int}()
    ORF7a_pos_ct_all = Dict{String, Int}()
    ORF7b_pos_ct_all = Dict{String, Int}()
    ORF8_pos_ct_all = Dict{String, Int}()
    N_pos_ct_all = Dict{String, Int}()
    ORF9b_pos_ct_all = Dict{String, Int}()
    gene_ct_pos_all_dict_arr = [ORF1a_pos_ct_all, ORF1b_pos_ct_all, S_pos_ct_all, ORF3a_pos_ct_all, E_pos_ct_all, M_pos_ct_all, ORF6_pos_ct_all, ORF7a_pos_ct_all, ORF7b_pos_ct_all, ORF8_pos_ct_all, N_pos_ct_all, ORF9b_pos_ct_all] 
###################################################### 
    ORF1a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF1b_pos_ct_all_v1 = Dict{Int, Int}()
    S_pos_ct_all_v1 = Dict{Int, Int}()
    ORF3a_pos_ct_all_v1 = Dict{Int, Int}()
    E_pos_ct_all_v1 = Dict{Int, Int}()
    M_pos_ct_all_v1 = Dict{Int, Int}()
    ORF6_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7a_pos_ct_all_v1 = Dict{Int, Int}()
    ORF7b_pos_ct_all_v1 = Dict{Int, Int}()
    ORF8_pos_ct_all_v1 = Dict{Int, Int}()
    N_pos_ct_all_v1 = Dict{Int, Int}()
    ORF9b_pos_ct_all_v1 = Dict{Int, Int}()
    gene_pos_ct_all_v1_dict_arr = [ORF1a_pos_ct_all_v1, ORF1b_pos_ct_all_v1, S_pos_ct_all_v1, ORF3a_pos_ct_all_v1, E_pos_ct_all_v1, M_pos_ct_all_v1, ORF6_pos_ct_all_v1, ORF7a_pos_ct_all_v1, ORF7b_pos_ct_all_v1, ORF8_pos_ct_all_v1, N_pos_ct_all_v1, ORF9b_pos_ct_all_v1]
###################################################### 
    for i in 1:length(gene_ct_pos_all_dict_arr)
        dict = gene_ct_pos_all_dict_arr[i]
        dict_v1 = gene_pos_ct_all_v1_dict_arr[i]
        gene = gene_array[i]
        gene_len = gene_AA_lengths[gene]
        for j in 1:gene_len
            site = gene*":"*"$(j)"
            dict[site] = 0
            dict_v1[j] = 0
        end
    end
#############################################################
    for (mut, ct) in AA_muts_ct_pos_only_no_dels_all
        pos = aa_pos_comprehensive_dict[mut]
        if aa_gene_comprehensive_dict[mut] == "ORF1a"
            ORF1a_pos_ct_all[mut] = ct
            ORF1a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF1b"
            ORF1b_pos_ct_all[mut] = ct
            ORF1b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "S"
            S_pos_ct_all[mut] = ct
            S_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF3a"
            ORF3a_pos_ct_all[mut] = ct
            ORF3a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "E"
            E_pos_ct_all[mut] = ct
            E_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "M"
            M_pos_ct_all[mut] = ct
            M_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF6"
            ORF6_pos_ct_all[mut] = ct
            ORF6_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7a"
            ORF7a_pos_ct_all[mut] = ct
            ORF7a_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF7b"
            ORF7b_pos_ct_all[mut] = ct
            ORF7b_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF8"
            ORF8_pos_ct_all[mut] = ct
            ORF8_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "N"
            N_pos_ct_all[mut] = ct
            N_pos_ct_all_v1[pos] = ct
            elseif aa_gene_comprehensive_dict[mut] == "ORF9b"
            ORF9b_pos_ct_all[mut] = ct
            ORF9b_pos_ct_all_v1[pos] = ct
        end
    end
    global ORF1a_pos_ct_all_sort_by_pos = sort(collect(ORF1a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF1b_pos_ct_all_sort_by_pos = sort(collect(ORF1b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global S_pos_ct_all_sort_by_pos = sort(collect(S_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF3a_pos_ct_all_sort_by_pos = sort(collect(ORF3a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global E_pos_ct_all_sort_by_pos = sort(collect(E_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]]) 
    global M_pos_ct_all_sort_by_pos = sort(collect(M_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF6_pos_ct_all_sort_by_pos = sort(collect(ORF6_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7a_pos_ct_all_sort_by_pos = sort(collect(ORF7a_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF7b_pos_ct_all_sort_by_pos = sort(collect(ORF7b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global ORF8_pos_ct_all_sort_by_pos = sort(collect(ORF8_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
    global N_pos_ct_all_sort_by_pos = sort(collect(N_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])   
    global ORF9b_pos_ct_all_sort_by_pos = sort(collect(ORF9b_pos_ct_all), by = x -> aa_pos_comprehensive_dict[x[1]])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct = sort(collect(ORF1a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct = sort(collect(ORF1b_pos_ct_all), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct = sort(collect(S_pos_ct_all), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct = sort(collect(ORF3a_pos_ct_all), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct = sort(collect(E_pos_ct_all), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct = sort(collect(M_pos_ct_all), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct = sort(collect(ORF6_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct = sort(collect(ORF7a_pos_ct_all), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct = sort(collect(ORF7b_pos_ct_all), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct = sort(collect(ORF8_pos_ct_all), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct = sort(collect(N_pos_ct_all), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct = sort(collect(ORF9b_pos_ct_all), by = x -> x[2], rev=true)
##################################
    global ORF1a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[1])
    global ORF1b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[1])
    global S_pos_ct_all_sort_by_pos_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[1])
    global ORF3a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[1])
    global E_pos_ct_all_sort_by_pos_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[1]) 
    global M_pos_ct_all_sort_by_pos_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[1])
    global ORF6_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[1])
    global ORF7a_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[1])
    global ORF7b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[1])
    global ORF8_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[1])
    global N_pos_ct_all_sort_by_pos_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[1])   
    global ORF9b_pos_ct_all_sort_by_pos_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[1])
################################## 
    global ORF1a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF1b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF1b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global S_pos_ct_all_sort_by_ct_v1 = sort(collect(S_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF3a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF3a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global E_pos_ct_all_sort_by_ct_v1 = sort(collect(E_pos_ct_all_v1), by = x -> x[2], rev=true)
    global M_pos_ct_all_sort_by_ct_v1 = sort(collect(M_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF6_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF6_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7a_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7a_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF7b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF7b_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF8_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF8_pos_ct_all_v1), by = x -> x[2], rev=true)
    global N_pos_ct_all_sort_by_ct_v1 = sort(collect(N_pos_ct_all_v1), by = x -> x[2], rev=true)
    global ORF9b_pos_ct_all_sort_by_ct_v1 = sort(collect(ORF9b_pos_ct_all_v1), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function NSP_sub_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
#############################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            gene = aa_gene_comprehensive_dict[sub]
            if gene == "ORF1a" || gene == "ORF1b"
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
#############################
    fin_sortkey(m) = (NSP_muts_pos_dict[m], NSP_ref_AA_dict[m]*NSP_qry_AA_dict[m])
    global NSP1_ct_all_sort_by_pos = sort(collect(NSP1_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP2_ct_all_sort_by_pos = sort(collect(NSP2_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP3_ct_all_sort_by_pos = sort(collect(NSP3_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP4_ct_all_sort_by_pos = sort(collect(NSP4_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP5_ct_all_sort_by_pos = sort(collect(NSP5_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP6_ct_all_sort_by_pos = sort(collect(NSP6_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP7_ct_all_sort_by_pos = sort(collect(NSP7_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP8_ct_all_sort_by_pos = sort(collect(NSP8_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP9_ct_all_sort_by_pos = sort(collect(NSP9_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP10_ct_all_sort_by_pos = sort(collect(NSP10_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP11_ct_all_sort_by_pos = sort(collect(NSP11_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP12_ct_all_sort_by_pos = sort(collect(NSP12_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP13_ct_all_sort_by_pos = sort(collect(NSP13_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP14_ct_all_sort_by_pos = sort(collect(NSP14_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP15_ct_all_sort_by_pos = sort(collect(NSP15_ct_all), by = x -> fin_sortkey(x[1]))
    global NSP16_ct_all_sort_by_pos = sort(collect(NSP16_ct_all), by = x -> fin_sortkey(x[1]))
#############################
    global NSP1_ct_all_sort_by_ct = sort(collect(NSP1_ct_all), by = x -> x[2], rev=true)
    global NSP2_ct_all_sort_by_ct = sort(collect(NSP2_ct_all), by = x -> x[2], rev=true)
    global NSP3_ct_all_sort_by_ct = sort(collect(NSP3_ct_all), by = x -> x[2], rev=true)
    global NSP4_ct_all_sort_by_ct = sort(collect(NSP4_ct_all), by = x -> x[2], rev=true)
    global NSP5_ct_all_sort_by_ct = sort(collect(NSP5_ct_all), by = x -> x[2], rev=true)
    global NSP6_ct_all_sort_by_ct = sort(collect(NSP6_ct_all), by = x -> x[2], rev=true)
    global NSP7_ct_all_sort_by_ct = sort(collect(NSP7_ct_all), by = x -> x[2], rev=true)
    global NSP8_ct_all_sort_by_ct = sort(collect(NSP8_ct_all), by = x -> x[2], rev=true)
    global NSP9_ct_all_sort_by_ct = sort(collect(NSP9_ct_all), by = x -> x[2], rev=true)
    global NSP10_ct_all_sort_by_ct = sort(collect(NSP10_ct_all), by = x -> x[2], rev=true)
    global NSP11_ct_all_sort_by_ct = sort(collect(NSP11_ct_all), by = x -> x[2], rev=true)
    global NSP12_ct_all_sort_by_ct = sort(collect(NSP12_ct_all), by = x -> x[2], rev=true)
    global NSP13_ct_all_sort_by_ct = sort(collect(NSP13_ct_all), by = x -> x[2], rev=true)
    global NSP14_ct_all_sort_by_ct = sort(collect(NSP14_ct_all), by = x -> x[2], rev=true)
    global NSP15_ct_all_sort_by_ct = sort(collect(NSP15_ct_all), by = x -> x[2], rev=true)
    global NSP16_ct_all_sort_by_ct = sort(collect(NSP16_ct_all), by = x -> x[2], rev=true)
end
###########################################################################################################################################################################
###########################################################################################################################################################################
# Removed from parameters: NSP_AA_size::Dict{String, Int}
function NSP_sub_pos_ranks_all(NSP_array::Vector{String}, NSP_sub_types_at_every_site_combined::Dict{String, Dict{Int, Set{String}}}, AA_muts_ct_no_dels_all::Dict{String, Int} )
    NSP1_pos_ct_all = Dict{String, Int}()
    NSP2_pos_ct_all = Dict{String, Int}()
    NSP3_pos_ct_all = Dict{String, Int}()
    NSP4_pos_ct_all = Dict{String, Int}()
    NSP5_pos_ct_all = Dict{String, Int}()
    NSP6_pos_ct_all = Dict{String, Int}()
    NSP7_pos_ct_all = Dict{String, Int}()
    NSP8_pos_ct_all = Dict{String, Int}()
    NSP9_pos_ct_all = Dict{String, Int}()
    NSP10_pos_ct_all = Dict{String, Int}()
    NSP11_pos_ct_all = Dict{String, Int}()
    NSP12_pos_ct_all = Dict{String, Int}()
    NSP13_pos_ct_all = Dict{String, Int}()
    NSP14_pos_ct_all = Dict{String, Int}()
    NSP15_pos_ct_all = Dict{String, Int}()
    NSP16_pos_ct_all = Dict{String, Int}()
    NSP_pos_ct_all_array = [NSP1_pos_ct_all, NSP2_pos_ct_all, NSP3_pos_ct_all, NSP4_pos_ct_all, NSP5_pos_ct_all, NSP6_pos_ct_all, NSP7_pos_ct_all, NSP8_pos_ct_all, NSP9_pos_ct_all, NSP10_pos_ct_all, NSP11_pos_ct_all, NSP12_pos_ct_all, NSP13_pos_ct_all, NSP14_pos_ct_all, NSP15_pos_ct_all, NSP16_pos_ct_all]
#######################################################
    NSP1_pos_ct_all_v1 = Dict{Int, Int}()
    NSP2_pos_ct_all_v1 = Dict{Int, Int}()
    NSP3_pos_ct_all_v1 = Dict{Int, Int}()
    NSP4_pos_ct_all_v1 = Dict{Int, Int}()
    NSP5_pos_ct_all_v1 = Dict{Int, Int}()
    NSP6_pos_ct_all_v1 = Dict{Int, Int}()
    NSP7_pos_ct_all_v1 = Dict{Int, Int}()
    NSP8_pos_ct_all_v1 = Dict{Int, Int}()
    NSP9_pos_ct_all_v1 = Dict{Int, Int}()
    NSP10_pos_ct_all_v1 = Dict{Int, Int}()
    NSP11_pos_ct_all_v1 = Dict{Int, Int}()
    NSP12_pos_ct_all_v1 = Dict{Int, Int}()
    NSP13_pos_ct_all_v1 = Dict{Int, Int}()
    NSP14_pos_ct_all_v1 = Dict{Int, Int}()
    NSP15_pos_ct_all_v1 = Dict{Int, Int}()
    NSP16_pos_ct_all_v1 = Dict{Int, Int}()
    NSP_pos_ct_all_v1_array = [NSP1_pos_ct_all_v1, NSP2_pos_ct_all_v1, NSP3_pos_ct_all_v1, NSP4_pos_ct_all_v1, NSP5_pos_ct_all_v1, NSP6_pos_ct_all_v1, NSP7_pos_ct_all_v1, NSP8_pos_ct_all_v1, NSP9_pos_ct_all_v1, NSP10_pos_ct_all_v1, NSP11_pos_ct_all_v1, NSP12_pos_ct_all_v1, NSP13_pos_ct_all_v1, NSP14_pos_ct_all_v1, NSP15_pos_ct_all_v1, NSP16_pos_ct_all_v1]
#######################################################
    NSP_pos_ct_all = Dict{String, Int}()
    for i in 1:length(NSP_pos_ct_all_array)
        NSP_all_dict = NSP_pos_ct_all_array[i]
        NSP_all_dict_v1 = NSP_pos_ct_all_v1_array[i]
        NSP = NSP_array[i]
        NSP_len = NSP_AA_size[NSP]
        for j in 1:NSP_len
            NSP_pos = NSP*":"*"$(j)"
            NSP_all_dict[NSP_pos] = 0
            NSP_pos_ct_all[NSP_pos] = 0
            NSP_all_dict_v1[j] = 0
        end
    end
############################################################
    NSP1_ct_all = Dict{String, Int}()
    NSP2_ct_all = Dict{String, Int}()
    NSP3_ct_all = Dict{String, Int}()
    NSP4_ct_all = Dict{String, Int}()
    NSP5_ct_all = Dict{String, Int}()
    NSP6_ct_all = Dict{String, Int}()
    NSP7_ct_all = Dict{String, Int}()
    NSP8_ct_all = Dict{String, Int}()
    NSP9_ct_all = Dict{String, Int}()
    NSP10_ct_all = Dict{String, Int}()
    NSP11_ct_all = Dict{String, Int}()
    NSP12_ct_all = Dict{String, Int}()
    NSP13_ct_all = Dict{String, Int}()
    NSP14_ct_all = Dict{String, Int}()
    NSP15_ct_all = Dict{String, Int}()
    NSP16_ct_all = Dict{String, Int}()
##################################################################################################################################
    NSP_ct_all_dict_arr = [NSP1_ct_all, NSP2_ct_all, NSP3_ct_all, NSP4_ct_all, NSP5_ct_all, NSP6_ct_all, NSP7_ct_all, NSP8_ct_all, NSP9_ct_all, NSP10_ct_all, NSP11_ct_all, NSP12_ct_all, NSP13_ct_all, NSP14_ct_all, NSP15_ct_all, NSP16_ct_all]
#                                              NSP       AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
# NSPsub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()   
    for i in 1:length(NSP_array)
        NSP = NSP_array[i]
        NSP_ct_dict = NSP_ct_all_dict_arr[i]
        for (AAsite, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
            for mut_type in mut_type_set
                sub = NSP*":"*string(mut_type[1])*"$(AAsite)"*string(mut_type[end])
                NSP_ct_dict[sub] = 0
            end
        end
    end   
    for (sub, ct) in AA_muts_ct_no_dels_all
        gene = aa_gene_comprehensive_dict[sub]
        if gene == "ORF1a" || gene == "ORF1b"
            AA_site = parse(Int, split(sub, ":")[2][2:end-1])
            if AA_site < 4402
                NSP_sub = ORF1abMut_to_NSP(sub)
                NSP = NSP_muts_gene_dict[NSP_sub]
                NSP_pos = NSP_muts_pos_dict[NSP_sub]
                NSP_num = parse(Int, split(NSP, "P")[2])
                NSP_dict = NSP_ct_all_dict_arr[NSP_num]
                NSP_dict[NSP_sub] = ct
            end
        end
    end
###################################################################################################################################
    for i in 1:length(NSP_pos_ct_all_array)
        NSP = NSP_array[i]
        NSP_dict = NSP_ct_all_dict_arr[i]
        NSP_pos_dict = NSP_pos_ct_all_array[i]
        NSP_pos_dict_v1 = NSP_pos_ct_all_v1_array[i]
        for (sub, ct) in NSP_dict
            pos = parse(Int, split(sub, ":")[2][2:end-1])
            sub_pos = NSP*":"*"$(pos)"
            NSP_pos_dict[sub_pos] += ct
            NSP_pos_dict_v1[pos] += ct
            NSP_pos_ct_all[sub_pos] += ct
        end
    end
##############################################################
    global NSP1_pos_ct_all_sort_by_pos = sort(collect(NSP1_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP2_pos_ct_all_sort_by_pos = sort(collect(NSP2_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP3_pos_ct_all_sort_by_pos = sort(collect(NSP3_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP4_pos_ct_all_sort_by_pos = sort(collect(NSP4_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP5_pos_ct_all_sort_by_pos = sort(collect(NSP5_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP6_pos_ct_all_sort_by_pos = sort(collect(NSP6_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP7_pos_ct_all_sort_by_pos = sort(collect(NSP7_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP8_pos_ct_all_sort_by_pos = sort(collect(NSP8_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP9_pos_ct_all_sort_by_pos = sort(collect(NSP9_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP10_pos_ct_all_sort_by_pos = sort(collect(NSP10_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP11_pos_ct_all_sort_by_pos = sort(collect(NSP11_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP12_pos_ct_all_sort_by_pos = sort(collect(NSP12_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP13_pos_ct_all_sort_by_pos = sort(collect(NSP13_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP14_pos_ct_all_sort_by_pos = sort(collect(NSP14_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP15_pos_ct_all_sort_by_pos = sort(collect(NSP15_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
    global NSP16_pos_ct_all_sort_by_pos = sort(collect(NSP16_pos_ct_all), by = x -> NSP_muts_pos_dict[x[1]])
#############################
    global NSP1_pos_ct_all_sort_by_ct = sort(collect(NSP1_pos_ct_all), by = x -> x[2], rev=true)
    global NSP2_pos_ct_all_sort_by_ct = sort(collect(NSP2_pos_ct_all), by = x -> x[2], rev=true)
    global NSP3_pos_ct_all_sort_by_ct = sort(collect(NSP3_pos_ct_all), by = x -> x[2], rev=true)
    global NSP4_pos_ct_all_sort_by_ct = sort(collect(NSP4_pos_ct_all), by = x -> x[2], rev=true)
    global NSP5_pos_ct_all_sort_by_ct = sort(collect(NSP5_pos_ct_all), by = x -> x[2], rev=true)
    global NSP6_pos_ct_all_sort_by_ct = sort(collect(NSP6_pos_ct_all), by = x -> x[2], rev=true)
    global NSP7_pos_ct_all_sort_by_ct = sort(collect(NSP7_pos_ct_all), by = x -> x[2], rev=true)
    global NSP8_pos_ct_all_sort_by_ct = sort(collect(NSP8_pos_ct_all), by = x -> x[2], rev=true)
    global NSP9_pos_ct_all_sort_by_ct = sort(collect(NSP9_pos_ct_all), by = x -> x[2], rev=true)
    global NSP10_pos_ct_all_sort_by_ct = sort(collect(NSP10_pos_ct_all), by = x -> x[2], rev=true)
    global NSP11_pos_ct_all_sort_by_ct = sort(collect(NSP11_pos_ct_all), by = x -> x[2], rev=true)
    global NSP12_pos_ct_all_sort_by_ct = sort(collect(NSP12_pos_ct_all), by = x -> x[2], rev=true)
    global NSP13_pos_ct_all_sort_by_ct = sort(collect(NSP13_pos_ct_all), by = x -> x[2], rev=true)
    global NSP14_pos_ct_all_sort_by_ct = sort(collect(NSP14_pos_ct_all), by = x -> x[2], rev=true)
    global NSP15_pos_ct_all_sort_by_ct = sort(collect(NSP15_pos_ct_all), by = x -> x[2], rev=true)
    global NSP16_pos_ct_all_sort_by_ct = sort(collect(NSP16_pos_ct_all), by = x -> x[2], rev=true)
###############################
    global NSP1_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP1_pos_ct_all_v1), by = x -> x[1])
    global NSP2_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP2_pos_ct_all_v1), by = x -> x[1])
    global NSP3_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP3_pos_ct_all_v1), by = x -> x[1])
    global NSP4_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP4_pos_ct_all_v1), by = x -> x[1])
    global NSP5_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP5_pos_ct_all_v1), by = x -> x[1])
    global NSP6_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP6_pos_ct_all_v1), by = x -> x[1])
    global NSP7_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP7_pos_ct_all_v1), by = x -> x[1])
    global NSP8_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP8_pos_ct_all_v1), by = x -> x[1])
    global NSP9_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP9_pos_ct_all_v1), by = x -> x[1])
    global NSP10_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP10_pos_ct_all_v1), by = x -> x[1])
    global NSP11_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP11_pos_ct_all_v1), by = x -> x[1])
    global NSP12_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP12_pos_ct_all_v1), by = x -> x[1])
    global NSP13_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP13_pos_ct_all_v1), by = x -> x[1])
    global NSP14_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP14_pos_ct_all_v1), by = x -> x[1])
    global NSP15_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP15_pos_ct_all_v1), by = x -> x[1])
    global NSP16_pos_ct_all_sort_by_pos_v1 = sort(collect(NSP16_pos_ct_all_v1), by = x -> x[1])
end
######################################################################################################################################
runtime = time() - start_chr_load_fx
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime); print("\n"^1)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)"); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_05__646PM

Runtime v0 = 0.9514789581298828 seconds
Runtime v2 = 0 hr, 0 min, 00.95 sec



In [6]:
### Execute Load Chronic Dicts DQ Fx | 2026_02_22 version | chronics_2026_02_01__6153seq | 95—8—6 | Runtime: 1 min, 00.26 sec | 2026_2_13
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
chr_dict_load_start = time()

abs_min_mut_thresh = 8
DQ_mut_thresh = 10
date_pct_DQ_thresh = 95

rep_thresh = 5
revs_thresh = 6
EPCI_qc_str = "$(abs_min_mut_thresh)_$(DQ_mut_thresh)_$(date_pct_DQ_thresh)"

print_ct_thresh = 1
date = "2026_02_20"
ndjson_name = "chronics_2026_02_01__6153seq_v2"
folder_name = "$(date)__$(ndjson_name)__EPCI_$(EPCI_qc_str)__HQCS_$(HQCS_qc_str)"
println("EPCI_qc_str = $(EPCI_qc_str) | HQCS_qc_str = $(HQCS_qc_str)")

chronic_load_dicts2_DQ(ndjson_name, folder_name, date, rep_thresh, revs_thresh, print_ct_thresh, DQ_mut_thresh, date_pct_DQ_thresh, abs_min_mut_thresh)
# ndjson_name::String, folder_name::String, date::String, rep_thresh::Int, revs_thresh::Int, print_ct_thresh::Int, DQ_mut_thresh::Int, DatePctDQThresh::Float64, abs_min_mut_thresh::Int
######################################################################################################################################
seq_AA_muts["EPI_ISL_8725398"] = Set{String}();             seq_AA_muts["EPI_ISL_949208"] = Set{String}();   seq_pango["EPI_ISL_6281381"] = "AY.4"
######################################################################################################################################
### Create  all_unique_chr_seqs  &  all_unique_chr_seqs_set 
rep_seq_grps_maxmut_seqs_arr = collect(values(rep_seq_grps_maxmut_seqs))
all_unique_chr_seqs = union(rep_seq_grps_maxmut_seqs_arr, non_rep_seqs)
all_unique_chr_seqs_ct = length(all_unique_chr_seqs)
all_unique_chr_seqs_set = Set{String}()
for seq in all_unique_chr_seqs
    push!(all_unique_chr_seqs_set, seq)
end
println("length(all_unique_chr_seqs) = $(length(all_unique_chr_seqs))")
println("length(all_unique_chr_seqs_set)  $(length(all_unique_chr_seqs_set))"); print("\n"^2)
########################################################################################################################################
chronic_pango_set = Set{String}()
for seq in all_unique_chr_seqs_set
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
for seq in all_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
end
#############################################################################################################################################################################
#### Removing clearly artifactual reversions that occur directly next to dropout (and are almost always from labs that frequently have this problem)
FCS_fake_revs = Set(["S:K679N", "S:H681P", "S:R681P"])
FCS_fake_revs_pos = Set(["S:679", "S:681"])
for seq in all_qualifying_seqs_set
    if "S:683" in seq_unknown_AA[seq] || "S:676" in seq_unknown_AA[seq]
        if seq in all_unique_chr_seqs_set
            for mut in FCS_fake_revs
                if mut in seq_AA_muts[seq]
                    mut_pos = aa_gene_and_pos_comprehensive_dict[mut]
                    AA_muts_ct[mut] -= 1
                    AA_muts_ct_no_dels[mut] -= 1
                    AA_muts_ct_pos_only[mut_pos] -= 1
                    AA_muts_ct_pos_only_no_dels[mut_pos] -= 1
                end
            end
        end
        setdiff!(seq_AA_muts[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_no_dels[seq], FCS_fake_revs)
        setdiff!(seq_AA_muts_pos_only[seq], FCS_fake_revs_pos)
        setdiff!(seq_AA_muts_pos_only_no_dels[seq], FCS_fake_revs_pos)
    end
end
######################################################################################################################################
if haskey(AA_muts_ct_pos_only_no_dels, "")
    delete!(AA_muts_ct_pos_only_no_dels, "")
end
if haskey(AA_muts_ct_no_dels, "")
    delete!(AA_muts_ct_no_dels, "")
end
######################################################################################################################################
## Deletions royally screw up any attempt to find correlated mutations since they frequently occur in bunches, which are, of course, highly correlated, but only in the most trivial way. 
## The code below is a preliminary attempt to include common deletions by only allowing the inclusion of a single AA deletion, though most contain multiple consecutive 
## AA deletions. Also included are mutations that only occur as 1-AA deletions, such as E:I13- and E:V14-. It can be removed or inserted without substantially changing the results 
deletion_exceptions = list_to_set("E:I13-, E:V14-, M:G6-, S:C15-, S:C136-, S:D138-, S:A243-, S:F371-, S:F375-, S:V483-, S:A484-, S:E484-, S:D1257-, ORF3a:T14-, ORF3a:V255-, ORF3a:V256-, ORF3a:N257-, ORF7a:I103-, ORF7a:*122-, ORF1a:N2081-, ORF1a:D2136-, ORF1a:S4398-")
pos_only_deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
del_to_del_pos = Dict{String, String}("E:I13-"=>"E:13", "E:V14-"=>"E:14", "M:G6-"=>"M:6", "S:C15-"=>"S:15", "S:C136-"=>"S:136", "S:D138-"=>"S:138", "S:A243-"=>"S:243", "S:F371-"=>"S:371", "S:F375-"=>"S:375", "S:V483-"=>"S:483", "S:A484-"=>"S:484", "S:E484-"=>"S:484", "S:D1257-"=>"S:1257", "ORF3a:T14-"=>"ORF3a:14", "ORF3a:V255-"=>"ORF3a:255", "ORF3a:V256-"=>"ORF3a:256", "ORF3a:N257-"=>"ORF3a:257", "ORF7a:I103-"=>"ORF7a:103", "ORF7a:*122-"=>"ORF7a:122", "ORF1a:N2081-"=>"ORF1a:2081", "ORF1a:D2136-"=>"ORF1a:2136", "ORF1a:S4398-"=>"ORF1a:4398")
pos_only_exception_count::Int = 0
for seq in all_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            del_pos = del_to_del_pos[del]
            push!(seq_AA_muts_pos_only_no_dels[seq], del_pos)
            pos_only_exception_count += 1
            pos_only_deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    del_pos = del_to_del_pos[del]
    if !haskey(AA_muts_ct_pos_only_no_dels, del_pos)
        AA_muts_ct_pos_only_no_dels[del_pos] = 0
    end
    AA_muts_ct_pos_only_no_dels[del_pos] += pos_only_deletion_exception_ct_dict[del]
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[del_pos] = 999
end
######################################################################################################################################
deletion_exception_ct_dict = Dict{String, Int}("E:I13-"=>0, "E:V14-"=>0, "M:G6-"=>0, "S:C15-"=>0, "S:C136-"=>0, "S:D138-"=>0, "S:A243-"=>0, "S:F371-"=>0, "S:F375-"=>0, "S:V483-"=>0, "S:A484-"=>0, "S:E484-"=>0, "S:D1257-"=>0, "ORF3a:T14-"=>0, "ORF3a:V255-"=>0, "ORF3a:V256-"=>0, "ORF3a:N257-"=>0, "ORF7a:I103-"=>0, "ORF7a:*122-"=>0, "ORF1a:N2081-"=>0, "ORF1a:D2136-"=>0, "ORF1a:S4398-"=>0)    
exception_count::Int = 0
for seq in all_unique_chr_seqs_set
    for del in deletion_exceptions
        if del in seq_AA_muts[seq]
            push!(seq_AA_muts_no_dels[seq], del)
            exception_count += 1
            deletion_exception_ct_dict[del] += 1
        end
    end
end
######################################################################################################################################
for del in deletion_exceptions
    AA_muts_ct_no_dels[del] = get(AA_muts_ct_no_dels, del, 0)
    AA_muts_ct_no_dels[del] += deletion_exception_ct_dict[del]
    AA_muts_ct_chr_all_ratio[del] = 999
end
######################################################################################################################################
blank_mutstrings = Set(["", " ", "  ", "   ", "    ", "     ", "      ", "       ", "        ", "         ", "          ", "           ", "            "])
for blnk in blank_mutstrings
    AA_muts_ct_chr_all_ratio[blnk] = 0
    AA_muts_ct_no_dels_chr_all_ratio[blnk] = 0
    AA_muts_ct_pos_only_no_dels_chr_all_ratio[blnk] = 0
end
#######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
######################################################################################################################################
mp_AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_sortKey2(n) = (n[2], mp_AA_ct_sortKey1(n))
#########
mp_AA_gene_pos_only_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_gene_pos_only_sortKey_2(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n])
mp_AA_ct_pos_only_sortKey1(n) = (1000÷mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
mp_AA_ct_pos_only_sortKey2(n) = (n[2], mp_AA_ct_pos_only_sortKey1(n))
#################################
function mp_AApos_sort_key(n)
    if haskey(aa_pos_comprehensive_dict, n)
        return (aa_pos_comprehensive_dict[n], n)
    else
        return (9999, n)
    end
end
function sel_muts_pt1_sort_key(n)
    if n == "" || isempty(n)
        return (0, 0)  # Return a default sort key for empty strings
    else
        return mp_AA_gene_sortKey_2(string(split(n, ", ")[1]))
    end
end
##################################################################
println("Number of Deletion Exceptions Made = $(exception_count)")
deletion_exception_ct_dict_sort = sort(collect(deletion_exception_ct_dict), by = x -> x[2])
for del__ct in deletion_exception_ct_dict_sort
    del = del__ct[1]
    ct = del__ct[2]
    println("$(del) = $(ct)")
end
###########################################################################################
seq_privAA_len = Dict{String, Int}()
for (seq, AAsubvec) in seq_AA_muts_no_dels
    seqAAlen = length(AAsubvec)
    seq_privAA_len[seq] = seqAAlen
end
############################################################################################################################################################################
###########################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
############################################################################################################################################################################
corrected_count = 0
noncorrected_ct = 0
for seq in all_seqs_set
    if seq_date_index[seq] > 4000
#        println("seq_date_tuple = $(seq_date_tuple[seq]); seq_date_tuple[seq][1] = $(seq_date_tuple[seq][1]); seq_date_tuple[seq][2] = $(seq_date_tuple[seq][2]); seq_date_tuple[seq][3] = $(seq_date_tuple[seq][3])")
        if seq_date_tuple[seq][1] ≠ 0 && seq_date_tuple[seq][2] ≠ 0
            new_date_tuple = (seq_date_tuple[seq][1], seq_date_tuple[seq][2], 15)
            seq_date_tuple[seq] = new_date_tuple
            seq_date_index[seq] = tuple_to_index[new_date_tuple]
            corrected_count += 1
        elseif seq_date_tuple[seq][2] == 0
            noncorrected_ct += 1
        end
    end
end
total_baddie_ct = corrected_count + noncorrected_ct
println("seq_date_index corrected = $(corrected_count)")
println("seq_date_index not corrected = $(noncorrected_ct)")
println("total_baddie_ct = $(total_baddie_ct)")
######################################################################################################################################################
######################################################################################################################################  
chr_load_runtime = time() - chr_dict_load_start
chr_load_runtime_rd = round(digits=1, chr_load_runtime)
chr_load_hms1, chr_load_hms2 = seconds_to_hrs_min_sec(chr_load_runtime)
println("Total Time to Load Chronic Dictionaries = $(chr_load_runtime_rd)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms1)")
println("Total Time to Load Chronic Dictionaries = $(chr_load_hms2)"); print("\n"^1)
println("Finished!")
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
######################################################################################################################################
######################################################################################################################################

2026_03_05__646PM
EPCI_qc_str = 8_10_95 | HQCS_qc_str = 5_1_5
2026_03_05__646PM
length(all_unique_chr_seqs) = 3297
length(all_unique_chr_seqs_set)  3297


Number of Deletion Exceptions Made = 838
S:E484- = 1
ORF1a:D2136- = 2
S:F371- = 4
ORF1a:N2081- = 6
ORF1a:S4398- = 7
E:I13- = 8
S:D1257- = 8
S:V483- = 9
ORF7a:I103- = 10
M:G6- = 12
ORF7a:*122- = 16
S:A484- = 23
S:F375- = 24
S:C136- = 32
ORF3a:V256- = 38
ORF3a:T14- = 40
ORF3a:N257- = 43
ORF3a:V255- = 51
S:D138- = 89
E:V14- = 90
S:C15- = 105
S:A243- = 220
seq_date_index corrected = 70
seq_date_index not corrected = 52
total_baddie_ct = 122
Total Time to Load Chronic Dictionaries = 19.6
Total Time to Load Chronic Dictionaries = 0:00:19.65
Total Time to Load Chronic Dictionaries = 0 hr, 0 min, 19.65 sec

Finished!
2026_03_05__646PM



In [11]:
### Calculates EPCI mutation counts for any date range, from 2020-1-1 to 2027
function mut_ct_by_date_range_all_UNIVERSAL(date1::Int, date2::Int, all_dict::Dict)
    date1_to_date2_ct = Dict()
    for i in date1:date2
        for (mut, count) in all_dict[i]
            date1_to_date2_ct[mut] = get!(date1_to_date2_ct, AAmut, 0) + count
        end
    end
end
############################################################################################################################################################################
for i in 1:3500
    if !haskey(date_AA_mut_ct_all, i)
        date_AA_mut_ct_all[i] = Dict{String, Int}()
    end
end
AA_muts_ct__dateIndex_1to731 = Dict{String, Int}()
for i in 1:731
    temp_dict = date_AA_mut_ct_all[i]
    for (mut, ct) in temp_dict
        AA_muts_ct__dateIndex_1to731[mut] = get(AA_muts_ct__dateIndex_1to731, mut, 0) + ct
    end
end

timer_ct = 0
for (mut, ct) in AA_muts_ct__dateIndex_1to731
    timer_ct += 1
    if timer_ct ≤ 20
        println("$(mut) = $(ct)")
    end
end
############################################################################################################################################################################

ORF3a:F230Y = 6
ORF1b:N1790H = 6
ORF1b:Q2556L = 34
ORF8:I121F = 530
ORF1a:Q1003P = 84
ORF1a:G2284D = 170
ORF8:L98I = 52
S:E654G = 26
N:G316A = 4
ORF1b:C1232V = 2
ORF8:F86T = 2
ORF1b:C2606S = 14
ORF7a:N43T = 92
ORF1a:A2388V = 1022
S:G1099V = 96
ORF7a:E91H = 2
ORF1a:I1045F = 8
ORF1a:F635D = 4
ORF7a:E41D = 4822
ORF1a:D3992E = 36


In [17]:
### Fx: Count number of MP sequences for any MP (given selected muts & muts_pos_only
### Note: ORF1a:T4175I/ORF1a:4175 removed from BAL list; try with & without to see how big the difference is
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd__I")
Nineb8_muts_po = [list_to_set("ORF1a:3949, ORF1a:3951, ORF1a:3952, ORF1a:3953"), list_to_set("ORF9b:64, ORF9b:65, ORF9b:66, ORF9b:67, ORF9b:68, ORF9b:70"), list_to_set("ORF9b:89, ORF9b:93, ORF9b:95, ORF9b:96")]
Nineb8_muts = [list_to_set("ORF1a:S3950T, ORF1a:L3951F, ORF1a:P3952S"), list_to_set("ORF9b:L64P, ORF9b:D66E, ORF9b:A68E, ORF9b:A68V"), list_to_set("ORF9b:D89E, ORF9b:V93L, ORF9b:V93M, ORF9b:T95A, ORF9b:T95M, ORF9b:V96A")]
eightfold_muts = [list_to_set("ORF1a:R1318G, ORF1a:T1322K, ORF1a:T1322A, ORF1a:T1322I, ORF1a:D1323G, ORF1a:D1323N, ORF1a:N1324H, ORF1a:N1324D, ORF1a:N1324S, ORF1a:I1326V, ORF1a:I1326T, ORF1a:T1328N"), list_to_set("ORF1a:T1638N, ORF1a:T1638A, ORF1a:T1638I, ORF1a:D1639A, ORF1a:D1639N"), list_to_set("ORF1a:T2274I, ORF1a:A2279V, ORF1a:T2280I"), list_to_set("ORF1a:T4087I, ORF1a:T4087A, ORF1a:T4088I, ORF1a:T4090A, ORF1a:T4090I"), list_to_set("ORF1a:T4164N, ORF1a:T4164I, ORF1a:D4165N, ORF1a:D4165G, ORF1a:D4165V, ORF1a:D4165A, ORF1a:D4166G, ORF1a:D4166A, ORF1a:D4166N, ORF1a:D4166S, ORF1a:D4166Y, ORF1a:N4167S"), list_to_set("ORF1b:T730I"), list_to_set("ORF3a:Q213K, ORF3a:Y215H")]
eightfold_muts_po = [list_to_set("ORF1a:1318, ORF1a:1322, ORF1a:1323, ORF1a:1324, ORF1a:1326, ORF1a:1328"), list_to_set("ORF1a:1637, ORF1a:1638, ORF1a:1639"), list_to_set("ORF1a:2274, ORF1a:2279, ORF1a:2280"), list_to_set("ORF1a:3441, ORF1a:3443"), list_to_set("ORF1a:4085, ORF1a:4086, ORF1a:4087, ORF1a:4088, ORF1a:4090"), list_to_set("ORF1a:4164, ORF1a:4165, ORF1a:4166, ORF1a:4167"), list_to_set("ORF1b:730, ORF1b:731, ORF1b:734"), list_to_set("ORF3a:210, ORF3a:211, ORF3a:213, ORF3a:215")]
############################################################################################################################################################################
BAL_muts = [Set(["ORF7a:*122-"]), Set(["ORF7a:*122R"]), Set(["ORF7a:*122W"]), Set(["ORF7a:*122S"]), Set(["ORF1a:T2183I"]), Set(["ORF1a:S2972P"]), Set(["ORF1a:S2972F"]), Set(["ORF1a:A3049V"]), Set(["ORF1a:T3058I"]), Set(["ORF1a:A3070V"]), Set(["ORF1a:G3072C"]), Set(["ORF1a:H3076Y"]), Set(["ORF1a:V3077A"]), Set(["ORF1a:F3089L"]), Set(["ORF1a:P3359L"]), Set(["ORF1a:K3365R"]), Set(["ORF1a:A3454V"]), Set(["ORF1a:A3456V"]), Set(["ORF1a:A3697V"]), Set(["ORF1a:Q3826R"]), Set(["ORF1a:K4176R"]), Set(["ORF1a:P4197S"]), Set(["ORF1a:I4205V"]), Set(["ORF1a:T4207A"]), Set(["ORF1a:R4387S"]), Set(["ORF1b:L314P"]), Set(["ORF1b:I1181T"]), Set(["ORF1b:Y1247C"]), Set(["ORF1b:T1424I"]), Set(["ORF1b:T2537I"]), Set(["S:K41E"]), Set(["S:S50L"]), Set(["S:T274I"]), Set(["S:P330S"]), Set(["S:V367F"]), Set(["S:Y369H"]), Set(["S:A372T"]), Set(["S:F374L"]), Set(["S:F375-"]), Set(["S:A376V"]), Set(["S:R403S"]), Set(["S:N405S"]), Set(["S:N405D"]), Set(["S:Y508H"]), Set(["S:V551I"]), Set(["S:T573I"]), Set(["S:A653V"]), Set(["S:N657K"]), Set(["S:S659P"]), Set(["S:T859I"]), Set(["S:A944T"]), Set(["S:A944V"]), Set(["S:A958D"]), Set(["S:N978D"]), Set(["S:I1169T"]), Set(["S:I1179T"]), Set(["S:I1179S"]), Set(["S:L1186F"]), Set(["E:V5A"]), Set(["E:V5F"]), Set(["E:S6L"]), Set(["E:G10S"]), Set(["E:G10C"]), Set(["E:V14-"]), Set(["E:S16N"]), Set(["E:F23S"]), Set(["E:L27S"]), Set(["E:T30I"]), Set(["E:A32V"]), Set(["E:A36V"]), Set(["E:Y42C"]), Set(["E:S68F"]), Set(["E:P71L"]), Set(["M:S4P"]), Set(["M:V10I"]), Set(["M:F28S"]), Set(["M:T77N"]), Set(["M:S94N"]), Set(["M:S94R"]), Set(["M:H125Y"]), Set(["M:H148R"]), Set(["M:A188T"]), Set(["N:S416L"]), Set(["N:T417I"]), Set(["ORF7a:T115I"]), Set(["ORF9b:Q77*"])]
BAL_muts_po = [Set(["ORF1a:1215"]), Set(["ORF1a:2972"]), Set(["ORF1a:3023"]), Set(["ORF1a:3049"]), Set(["ORF1a:3058"]), Set(["ORF1a:3070"]), Set(["ORF1a:3072"]), Set(["ORF1a:3076"]), Set(["ORF1a:3085"]), Set(["ORF1a:3089"]), Set(["ORF1a:3359"]), Set(["ORF1a:3365"]), Set(["ORF1a:3454"]), Set(["ORF1a:3456"]), Set(["ORF1a:3697"]), Set(["ORF1a:3826"]), Set(["ORF1a:4176"]), Set(["ORF1a:4197"]), Set(["ORF1a:4205"]), Set(["ORF1a:4207"]), Set(["ORF1a:4387"]), Set(["ORF1b:314"]), Set(["ORF1b:1181"]), Set(["ORF1b:1247"]), Set(["ORF1b:1424"]), Set(["ORF1b:2147"]), Set(["ORF1b:2339"]), Set(["ORF1b:2340"]), Set(["ORF1b:2537"]), Set(["S:50"]), Set(["S:330"]), Set(["S:369"]), Set(["S:374"]), Set(["S:405"]), Set(["S:508"]), Set(["S:551"]), Set(["S:573"]), Set(["S:646"]), Set(["S:647"]), Set(["S:653"]), Set(["S:657"]), Set(["S:659"]), Set(["S:944"]), Set(["S:978"]), Set(["S:1169"]), Set(["S:1175"]), Set(["S:1179"]), Set(["S:1186"]), Set(["E:5"]), Set(["E:6"]), Set(["E:10"]), Set(["E:14"]), Set(["E:15"]), Set(["E:16"]), Set(["E:26"]), Set(["E:27"]), Set(["E:30"]), Set(["E:32"]), Set(["E:36"]), Set(["E:42"]), Set(["M:10"]), Set(["M:11"]), Set(["M:28"]), Set(["M:77"]), Set(["M:94"]), Set(["M:125"]), Set(["M:173"]), Set(["M:189"]), Set(["N:416"]), Set(["N:417"]), Set(["ORF7a:115"]), Set(["ORF7a:122"])]
Cryptic_WW = [Set(["ORF1a:V38A"]), Set(["ORF1a:K1795Q"]), Set(["ORF1a:S2926F"]), Set(["ORF1a:L3606V"]), Set(["ORF1b:L314P"]), Set(["ORF1b:D870A"]), Set(["ORF1b:E2311K"]), Set(["ORF1b:F2314L"]), Set(["ORF1b:F2314V"]), Set(["S:D215G"]), Set(["S:L828F"]), Set(["S:D1153A"]), Set(["S:V1176F"]), Set(["N:S37P"]), Set(["ORF3a:I10L"]), Set(["ORF3a:W45R"]), Set(["ORF3a:H182D"])]
Cryptic_WW_po = [Set(["ORF1a:38"]), Set(["ORF1a:1188"]), Set(["ORF1a:1795"]), Set(["ORF1a:2980"]), Set(["ORF1a:3694"]), Set(["ORF1a:3729"]), Set(["ORF1a:3732"]), Set(["ORF1b:314"]), Set(["ORF1b:870"]), Set(["ORF1b:2311"]), Set(["ORF1b:2313"]), Set(["ORF1b:2314"]), Set(["S:66"]), Set(["S:215"]), Set(["S:828"]), Set(["S:1153"]), Set(["S:1176"]), Set(["S:1185"]), Set(["S:1189"]), Set(["S:1201"]), Set(["S:1231"]), Set(["S:1266"]), Set(["ORF3a:10"]), Set(["ORF3a:45"]), Set(["ORF3a:182"]), Set(["ORF9b:13"])]
#BAL_muts = list_to_set("ORF7a:*122-, ORF7a:*122R, ORF7a:*122W, ORF7a:*122S, ORF1a:T2183I, ORF1a:S2972P, ORF1a:S2972F, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:V3077A, ORF1a:F3089L, ORF1a:P3359L, ORF1a:K3365R, ORF1a:A3454V, ORF1a:A3456V, ORF1a:A3697V, ORF1a:Q3826R, ORF1a:K4176R, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:T2537I, S:K41E, S:S50L, S:T274I, S:P330S, S:V367F, S:Y369H, S:A372T, S:F374L, S:F375-, S:A376V, S:R403S, S:N405S, S:N405D, S:Y508H, S:V551I, S:T573I, S:A653V, S:N657K, S:S659P, S:T859I, S:A944T, S:A944V, S:A958D, S:N978D, S:I1169T, S:I1179T, S:I1179S, S:L1186F, E:V5A, E:V5F, E:S6L, E:G10S, E:G10C, E:V14-, E:S16N, E:F23S, E:L27S, E:T30I, E:A32V, E:A36V, E:Y42C, E:S68F, E:P71L, M:S4P, M:V10I, M:F28S, M:T77N, M:S94N, M:S94R, M:H125Y, M:H148R, M:A188T, N:S416L, N:T417I, ORF7a:T115I, ORF9b:Q77*") 
#BAL_muts_po = list_to_set("ORF1a:1215, ORF1a:2972, ORF1a:3023, ORF1a:3049, ORF1a:3058, ORF1a:3070, ORF1a:3072, ORF1a:3076, ORF1a:3085, ORF1a:3089, ORF1a:3359, ORF1a:3365, ORF1a:3454, ORF1a:3456, ORF1a:3697, ORF1a:3826, ORF1a:4176, ORF1a:4197, ORF1a:4205, ORF1a:4207, ORF1a:4387, ORF1b:314, ORF1b:1181, ORF1b:1247, ORF1b:1424, ORF1b:2147, ORF1b:2339, ORF1b:2340, ORF1b:2537, S:50, S:330, S:369, S:374, S:405, S:508, S:551, S:573, S:646, S:647, S:653, S:657, S:659, S:944, S:978, S:1169, S:1175, S:1179, S:1186, E:5, E:6, E:10, E:14, E:15, E:16, E:26, E:27, E:30, E:32, E:36, E:42, M:10, M:11, M:28, M:77, M:94, M:125, M:173, M:189, N:416, N:417, ORF7a:115, ORF7a:122")
#Cryptic_WW = list_to_set("ORF1a:V38A, ORF1a:K1795Q, ORF1a:S2926F, ORF1a:L3606V, ORF1b:L314P, ORF1b:D870A, ORF1b:E2311K, ORF1b:F2314L, ORF1b:F2314V, S:D215G, S:L828F, S:D1153A, S:V1176F, N:S37P, ORF3a:I10L, ORF3a:W45R, ORF3a:H182D")
#Cryptic_WW_po = list_to_set("ORF1a:38, ORF1a:1188, ORF1a:1795, ORF1a:2980, ORF1a:3694, ORF1a:3729, ORF1a:3732, ORF1b:314, ORF1b:870, ORF1b:2311, ORF1b:2313, ORF1b:2314, S:19, S:66, S:215, S:828, S:1153, S:1176, S:1185, S:1189, S:1201, S:1231, S:1266, ORF3a:10, ORF3a:45, ORF3a:182, ORF9b:13")
############################################################################################################################################################################
Six_7a = [list_to_set("ORF1a:E1706D, ORF1a:N1709S, ORF1a:N1709T, ORF1a:I1714A, ORF1a:I1714V, ORF1a:I1714T, ORF1a:I1714S"), list_to_set("ORF1a:M1769V, ORF1a:M1769T, ORF1a:M1769I, ORF1a:T1773A, ORF1a:T1773I"), list_to_set("ORF1a:K1795Q"), list_to_set("ORF1a:F2209L, ORF1a:S2214L, ORF1a:S2214A, ORF1a:F2215L"), list_to_set("ORF1a:V2324G, ORF1a:A2325T, ORF1a:A2325V, ORF1a:F2328V, ORF1a:F2328L, ORF1a:L2329F"), list_to_set("ORF1a:A2345V, ORF1a:M2347L, ORF1a:Q2348R, ORF1a:L2349V, ORF1a:Y2353H, ORF1a:Y2353C"), list_to_set("ORF7a:E22D"), list_to_set("ORF7a:T39I, ORF7a:N43K")]
Six_7a_po = [list_to_set("ORF1a:1706, ORF1a:1709, ORF1a:1714, ORF1a:1715"), list_to_set("ORF1a:1769, ORF1a:1770, ORF1a:1771, ORF1a:1773"), list_to_set("ORF1a:1795"), list_to_set("ORF1a:2209, ORF1a:2214, ORF1a:2215"), list_to_set("ORF1a:2324, ORF1a:2325, ORF1a:2328, ORF1a:2329"), list_to_set("ORF1a:2345, ORF1a:2348, ORF1a:2349, ORF1a:2351, ORF1a:2353"), list_to_set("ORF7a:22"), list_to_set("ORF7a:39, ORF7a:43")]
Two_macs = [list_to_set("ORF1a:R232L, ORF1a:E233K, ORF1a:E233D, ORF1a:E233A, ORF1a:E233G"), list_to_set("ORF1a:S1272G, ORF1a:D1273G, ORF1a:D1273N, ORF1a:I1274M, ORF1a:I1276T"), list_to_set("ORF1a:S1361P, ORF1a:Q1365R, ORF1a:Q1365P, ORF1a:I1367L, ORF1a:L1368I, ORF1a:G1369R, ORF1a:S1372C, ORF1a:S1372A")]
Two_macs_po = [list_to_set("ORF1a:233, ORF1a:235"), list_to_set("ORF1a:1272, ORF1a:1273, ORF1a:1274, ORF1a:1276"), list_to_set("ORF1a:1365, ORF1a:1366, ORF1a:1367, ORF1a:1368, ORF1a:1369, ORF1a:1372")] #list_to_set("ORF1a:4097, ORF1a:4100, ORF1a:4102")
Rd3N = [list_to_set("ORF1a:T1542I, ORF1a:T1543I"), list_to_set("ORF1a:R1566M, ORF1a:T1567I, ORF1a:V1570M, ORF1a:T1572I"), list_to_set("ORF1a:A4396V, ORF1a:A4396T, ORF1a:S4398P, ORF1a:S4398L"), list_to_set("ORF1b:L820F, ORF1b:P821S, ORF1b:Y822H, ORF1b:D824E, ORF1b:D824N, ORF1b:S826A"), list_to_set("N:V270I, N:V270L, N:T271I")]
Rd3N_po = [list_to_set("ORF1a:1542, ORF1a:1543"), list_to_set("ORF1a:1566, ORF1a:1567, ORF1a:1570, ORF1a:1572"), list_to_set("ORF1a:4396, ORF1a:4397, ORF1a:4398"), list_to_set("ORF1b:820, ORF1b:821, ORF1b:822, ORF1b:824, ORF1b:826")]
One_macs = [list_to_set("ORF1a:H110Y, ORF1a:I114T"), list_to_set("ORF1a:K958R, ORF1a:P959L, ORF1a:P959S"), list_to_set("ORF1a:P1220L, ORF1a:P1220S, ORF1a:S1221P, ORF1a:E1223K, ORF1a:Q1224R"), list_to_set("ORF1a:E1394D, ORF1a:T1395N, ORF1a:T1395I, ORF1a:A1397V")]
One_macs_po = [list_to_set("ORF1a:110, ORF1a:111, ORF1a:114"), list_to_set("ORF1a:958, ORF1a:959"), list_to_set("ORF1a:1220, ORF1a:1221, ORF1a:1223, ORF1a:1224, ORF1a:1225"), list_to_set("ORF1a:1394, ORF1a:1395, ORF1a:1397")]
D612 = [list_to_set("ORF1a:H1500R, ORF1a:H1500Y, ORF1a:T1504I, ORF1a:I1505L, ORF1a:L1507F"), list_to_set("ORF1a:A3648S, ORF1a:A3648V, ORF1a:F3650S, ORF1a:V3653A, ORF1a:A3657V"), list_to_set("ORF1b:Q72R, ORF1b:E75K, ORF1b:T76I")]
D612_po = [list_to_set("ORF1a:1500, ORF1a:1504, ORF1a:1505, ORF1a:1507"), list_to_set("ORF1a:3648, ORF1a:3650, ORF1a:3652, ORF1a:3653, ORF1a:3657"), list_to_set("ORF1b:72, ORF1b:75, ORF1b:76")]
Y16 = [list_to_set("ORF1a:S2466G, ORF1a:D2467E, ORF1a:E2468D, ORF1a:V2469I, ORF1a:A2470S, ORF1a:R2471G, ORF1a:R2471I, ORF1a:R2471S, ORF1a:R2471K, ORF1a:D2472E, ORF1a:L2475I"), list_to_set("ORF1a:R3802C, ORF1a:R3802H, ORF1a:R3802S, ORF1a:R3802L, ORF1a:Y3803H, ORF1a:L3808F")]
Y16_po = [list_to_set("ORF1a:2466, ORF1a:2467, ORF1a:2468, ORF1a:2469, ORF1a:2470, ORF1a:2471, ORF1a:2472, ORF1a:2475"), list_to_set("ORF1a:3802, ORF1a:3803, ORF1a:3808")]
NuBS = [list_to_set("ORF1a:V2133I, ORF1a:P2134L, ORF1a:P2134S, ORF1a:P2134H, ORF1a:D2136N, ORF1a:D2136G, ORF1a:I2138V"), list_to_set("ORF1a:T4083M, ORF1a:T4087I, ORF1a:T4090I"), list_to_set("N:P326L, N:P326H")]
NuBS_po = [list_to_set("ORF1a:2134, ORF1a:2136, ORF1a:2138, ORF1a:2139, ORF1a:2140"), list_to_set("ORF1a:4083, ORF1a:4087, ORF1a:4090"), list_to_set("N:326, N:327")]
BsHel = [list_to_set("ORF1a:C2165F, ORF1a:T2166I"), list_to_set("ORF1b:L1220F"), list_to_set("ORF9b:S10P, ORF9b:R13C")]
BsHel_po = [list_to_set("ORF1a:2165, ORF1a:2166, ORF1a:2169"), list_to_set("ORF1b:1218, ORF1b:1219, ORF1b:1220"), list_to_set("N:267, N:271")]
HR1M = [list_to_set("S:D936H"), list_to_set("M:H155N"), list_to_set("ORF3a:V256-, ORF3a:N257-"), list_to_set("ORF1a:L3201P")]
HR1M_po = [list_to_set("S:936"), list_to_set("M:155"), list_to_set("ORF3a:256, ORF3a:257, ORF3a:258"), list_to_set("ORF1a:3201")]

MP_muts_vec = [Nineb8_muts, eightfold_muts, BAL_muts, Cryptic_WW, Six_7a, Two_macs, Rd3N, One_macs, D612, Y16, NuBS, BsHel, HR1M]
MP_muts_pos_only_vec = [Nineb8_muts_po, eightfold_muts_po, BAL_muts_po, Cryptic_WW_po, Six_7a_po, Two_macs_po, Rd3N_po, One_macs_po, D612_po, Y16_po, NuBS_po, BsHel_po, HR1M_po]
MP_names_vec = ["9b8", "8fold", "BAL", "Cryptic_WW", "6-7a", "2Macs", "Rd3N", "1Macs", "D612", "Y1-6", "NuBS", "BsHel", "HR1M"]

MP_required_muts_dict = Dict{Int, Int}(0=>2, 1=>2, 2=>2, 3=>2, 4=>2, 5=>2)
for i in 6:100
    MP_required_muts_dict[i] = 3
end
#MP_required_muts_pos_only_dict = Dict{Int, Int}()
######################################################################################################################
######################## EPCI Disulfide Count = 136 sequences (calculated in special Fx below ########################
######################################################################################################################
function MP_mut_seq_ct_fx(mp_mut_name::String, mp_muts::Vector{Set{String}}, mp_muts_po::Vector{Set{String}})
    mp_mut_name_pos_only = "$(mp_mut_name)_pos_only"
    MP_mut_seq_ct_dict = Dict{Int, Int}()
    MP_mutpo_seq_ct_dict = Dict{Int, Int}()
    for i in 0:100
        MP_mut_seq_ct_dict[i] = 0
        MP_mutpo_seq_ct_dict[i] = 0
    end
    for seq in all_unique_chr_seqs_set
        region_count = 0
        AAmuts = seq_AA_muts[seq]
        for region in mp_muts
            region_overlap = length(intersect(AAmuts, region))
            if region_overlap ≥ 1
                region_count += 1
            end
        end
        region_count_po = 0
        AAmuts_po = seq_AA_muts_pos_only[seq]
        for region_po in mp_muts_po
            region_overlap_po = length(intersect(AAmuts_po, region_po))
            if region_overlap_po ≥ 1
                region_count_po += 1
            end
        end
        if mp_mut_name == "Cryptic_WW" && region_count == 3
            println(seq)
        end
        MP_mut_seq_ct_dict[region_count] = get(MP_mut_seq_ct_dict, region_count, 0) + 1
        MP_mutpo_seq_ct_dict[region_count_po] = get(MP_mutpo_seq_ct_dict, region_count_po, 0) + 1
    end
    return MP_mut_seq_ct_dict, MP_mutpo_seq_ct_dict
end
df_MP_mut_seq_ct = DataFrame(
    :MP_name => String[],
    Symbol(0) => Int[],
    Symbol(1) => Int[],
    Symbol(2) => Int[],
    Symbol(3) => Int[],
    Symbol(4) => Int[],
    Symbol(5) => Int[],
    Symbol(6) => Int[],
    Symbol(7) => Int[],
    Symbol(8) => Int[],
    Symbol(9) => Int[],
    Symbol(10) => Int[],
    Symbol(11) => Int[],
    Symbol(12) => Int[],
    Symbol(13) => Int[],
    Symbol(14) => Int[],
    Symbol(15) => Int[],
    :Total_EPCI_MP_Seqs => Int[])
df_MP_mutpo_seq_ct = DataFrame(
    :MP_name => String[],
    Symbol(0) => Int[],
    Symbol(1) => Int[],
    Symbol(2) => Int[],
    Symbol(3) => Int[],
    Symbol(4) => Int[],
    Symbol(5) => Int[],
    Symbol(6) => Int[],
    Symbol(7) => Int[],
    Symbol(8) => Int[],
    Symbol(9) => Int[],
    Symbol(10) => Int[],
    Symbol(11) => Int[],
    Symbol(12) => Int[],
    Symbol(13) => Int[],
    Symbol(14) => Int[],
    Symbol(15) => Int[],
    :Total_EPCI_MP_Seqs => Int[])

for i in 1:length(MP_muts_vec)
    MP_muts_set_vec = MP_muts_vec[i]
    MP_muts_pos_only_set_vec = MP_muts_pos_only_vec[i]
    MP_name = MP_names_vec[i]
    MP_region_tot = length(MP_muts_set_vec)
    MP_region_tot_po = length(MP_muts_pos_only_set_vec)
    required_MP_muts = MP_required_muts_dict[MP_region_tot]
    required_MP_muts_po = MP_required_muts_dict[MP_region_tot_po]
    total_qualifying_seqs = 0
    select_MP_mut_seq_ct_dict, select_MP_mutpo_seq_ct_dict = MP_mut_seq_ct_fx(MP_name, MP_muts_set_vec, MP_muts_pos_only_set_vec)
    for j in required_MP_muts:15
        reg_seq_ct = get(select_MP_mut_seq_ct_dict, j, 0)
        total_qualifying_seqs += reg_seq_ct
    end
    push!(df_MP_mut_seq_ct, (MP_name, select_MP_mut_seq_ct_dict[0], select_MP_mut_seq_ct_dict[1], select_MP_mut_seq_ct_dict[2], select_MP_mut_seq_ct_dict[3], select_MP_mut_seq_ct_dict[4], select_MP_mut_seq_ct_dict[5], select_MP_mut_seq_ct_dict[6], select_MP_mut_seq_ct_dict[7], select_MP_mut_seq_ct_dict[8], select_MP_mut_seq_ct_dict[9], select_MP_mut_seq_ct_dict[10], select_MP_mut_seq_ct_dict[11], select_MP_mut_seq_ct_dict[12], select_MP_mut_seq_ct_dict[13], select_MP_mut_seq_ct_dict[14], select_MP_mut_seq_ct_dict[15], total_qualifying_seqs))
######################
    total_qualifying_seqs_po = 0
    for j in required_MP_muts_po:15
        reg_seq_ct = get(select_MP_mutpo_seq_ct_dict, j, 0)
        total_qualifying_seqs_po += reg_seq_ct
    end
    push!(df_MP_mutpo_seq_ct, (MP_name, select_MP_mutpo_seq_ct_dict[0], select_MP_mutpo_seq_ct_dict[1], select_MP_mutpo_seq_ct_dict[2], select_MP_mutpo_seq_ct_dict[3], select_MP_mutpo_seq_ct_dict[4], select_MP_mutpo_seq_ct_dict[5], select_MP_mutpo_seq_ct_dict[6], select_MP_mutpo_seq_ct_dict[7], select_MP_mutpo_seq_ct_dict[8], select_MP_mutpo_seq_ct_dict[9], select_MP_mutpo_seq_ct_dict[10], select_MP_mutpo_seq_ct_dict[11], select_MP_mutpo_seq_ct_dict[12], select_MP_mutpo_seq_ct_dict[13], select_MP_mutpo_seq_ct_dict[14], select_MP_mutpo_seq_ct_dict[15], total_qualifying_seqs_po))
end
CSV.write("MP_seq_cts_EPCI_$(date_now).csv", df_MP_mut_seq_ct)
CSV.write("MP_pos_only_seq_cts_EPCI_$(date_now).csv", df_MP_mutpo_seq_ct)

date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
print("\n"^2)
###########################################################################################################################################################################
###########################################################################################################################################################################

2026_03_05__739PM
EPI_ISL_6977941
EPI_ISL_19431718
EPI_ISL_17796500
EPI_ISL_17244668
EPI_ISL_17885459
EPI_ISL_19850000
2026_03_05__739PM




In [7]:
### NEW | 2026_02_22 | Many, many functions (mixed_nuc, nuc_to_AA, etc) | Runtime: 1 min 33 sec 
### Timing Template
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start = time()
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
function list_to_strings_no_spaces(list::String)
    string_vec = string.(split(list, ","))
    return string_vec
end
######################################################################################################################################
function stringlist_to_strings_nonEPI(txt::String)
    arr_of_strings = Vector{String}()
    no_newlines = replace(txt, "\n" =>" ")
    for seq in split(no_newlines, ", ")
        push!(arr_of_strings, seq)
    end
    sort_arr_of_strings = sort(collect(arr_of_strings), by = x -> nuc_mut_int_comprehensive_dict[x])  
    return sort_arr_of_strings
end
######################################################################################################################################
function list_to_string_array(txt::String) # similar to stringlist_to_strings but not for EPIs
    no_newlines = replace(txt, "\n" =>" ")
    string_array = string.(split(no_newlines, ", "))
    return string_array
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function get_ref_pango_nucseq_and_geneseqs(ref_pango::String)
    ref_seq = "..."
    refAA_ORF1a = "MESLVPGFNEKTHVQLSLPVLQVRDVLVRGFGDSVEEVLSEARQHLKDGTCGLVEVEKGVLPQLEQPYVFIKRSDARTAPHGHVMVELVAELEGIQYGRSGETLGVLVPHVGEIPVAYRKVLLRKNGNKGAGGHSYGADLKSFDLGDELGTDPYEDFQENWNTKHSSGVTRELMRELNGGAYTRYVDNNFCGPDGYPLECIKDLLARAGKASCTLSEQLDFIDTKRGVYCCREHEHEIAWYTERSEKSYELQTPFEIKLAKKFDTFNGECPNFVFPLNSIIKTIQPRVEKKKLDGFMGRIRSVYPVASPNECNQMCLSTLMKCDHCGETSWQTGDFVKATCEFCGTENLTKEGATTCGYLPQNAVVKIYCPACHNSEVGPEHSLAEYHNESGLKTILRKGGRTIAFGGCVFSYVGCHNKCAYWVPRASANIGCNHTGVVGEGSEGLNDNLLEILQKEKVNINIVGDFKLNEEIAIILASFSASTSAFVETVKGLDYKAFKQIVESCGNFKVTKGKAKKGAWNIGEQKSILSPLYAFASEAARVVRSIFSRTLETAQNSVRVLQKAAITILDGISQYSLRLIDAMMFTSDLATNNLVVMAYITGGVVQLTSQWLTNIFGTVYEKLKPVLDWLEEKFKEGVEFLRDGWEIVKFISTCACEIVGGQIVTCAKEIKESVQTFFKLVNKFLALCADSIIIGGAKLKALNLGETFVTHSKGLYRKCVKSREETGLLMPLKAPKEIIFLEGETLPTEVLTEEVVLKTGDLQPLEQPTSEAVEAPLVGTPVCINGLMLLEIKDTEKYCALAPNMMVTNNTFTLKGGAPTKVTFGDDTVIEVQGYKSVNITFELDERIDKVLNEKCSAYTVELGTEVNEFACVVADAVIKTLQPVSELLTPLGIDLDEWSMATYYLFDESGEFKLASHMYCSFYPPDEDEEEGDCEEEEFEPSTQYEYGTEDDYQGKPLEFGATSAALQPEEEQEEDWLDDDSQQTVGQQDGSEDNQTTTIQTIVEVQPQLEMELTPVVQTIEVNSFSGYLKLTDNVYIKNADIVEEAKKVKPTVVVNAANVYLKHGGGVAGALNKATNNAMQVESDDYIATNGPLKVGGSCVLSGHNLAKHCLHVVGPNVNKGEDIQLLKSAYENFNQHEVLLAPLLSAGIFGADPIHSLRVCVDTVRTNVYLAVFDKNLYDKLVSSFLEMKSEKQVEQKIAEIPKEEVKPFITESKPSVEQRKQDDKKIKACVEEVTTTLEETKFLTENLLLYIDINGNLHPDSATLVSDIDITFLKKDAPYIVGDVVQEGVLTAVVIPTKKAGGTTEMLAKALRKVPTDNYITTYPGQGLNGYTVEEAKTVLKKCKSAFYILPSIISNEKQEILGTVSWNLREMLAHAEETRKLMPVCVETKAIVSTIQRKYKGIKIQEGVVDYGARFYFYTSKTTVASLINTLNDLNETLVTMPLGYVTHGLNLEEAARYMRSLKVPATVSVSSPDAVTAYNGYLTSSSKTPEEHFIETISLAGSYKDWSYSGQSTQLGIEFLKRGDKSVYYTSNPTTFHLDGEVITFDNLKTLLSLREVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIKPHNSHEGKTFYVLPNDDTLRVEAFEYYHTTDPSFLGRYMSALNHTKKWKYPQVNGLTSIKWADNNCYLATALLTLQQIELKFNPPALQDAYYRARAGEAANFCALILAYCNKTVGELGDVRETMSYLFQHANLDSCKRVLNVVCKTCGQQQTTLKGVEAVMYMGTLSYEQFKKGVQIPCTCGKQATKYLVQQESPFVMMSAPPAQYELKHGTFTCASEYTGNYQCGHYKHITSKETLYCIDGALLTKSSEYKGPITDVFYKENSYTTTIKPVTYKLDGVVCTEIDPKLDNYYKKDNSYFTEQPIDLVPNQPYPNASFDNFKFVCDNIKFADDLNQLTGYKKPASRELKVTFFPDLNGDVVAIDYKHYTPSFKKGAKLLHKPIVWHVNNATNKATYKPNTWCIRCLWSTKPVETSNSFDVLKSEDAQGMDNLACEDLKPVSEEVVENPTIQKDVLECNVKTTEVVGDIILKPANNSLKITEEVGHTDLMAAYVDNSSLTIKKPNELSRVLGLKTLATHGLAAVNSVPWDTIANYAKPFLNKVVSTTTNIVTRCLNRVCTNYMPYFFTLLLQLCTFTRSTNSRIKASMPTTIAKNTVKSVGKFCLEASFNYLKSPNFSKLINIIIWFLLLSVCLGSLIYSTAALGVLMSNLGMPSYCTGYREGYLNSTNVTIATYCTGSIPCSVCLSGLDSLDTYPSLETIQITISSFKWDLTAFGLVAEWFLAYILFTRFFYVLGLAAIMQLFFSYFAVHFISNSWLMWLIINLVQMAPISAMVRMYIFFASFYYVWKSYVHVVDGCNSSTCMMCYKRNRATRVECTTIVNGVRRSFYVYANGGKGFCKLHNWNCVNCDTFCAGSTFISDEVARDLSLQFKRPINPTDQSSYIVDSVTVKNGSIHLYFDKAGQKTYERHSLSHFVNLDNLRANNTKGSLPINVIVFDGKSKCEESSAKSASVYYSQLMCQPILLLDQALVSDVGDSAEVAVKMFDAYVNTFSSTFNVPMEKLKTLVATAEAELAKNVSLDNVLSTFISAARQGFVDSDVETKDVVECLKLSHQSDIEVTGDSCNNYMLTYNKVENMTPRDLGACIDCSARHINAQVAKSHNIALIWNVKDFMSLSEQLRKQIRSAAKKNNLPFKLTCATTRQVVNVVTTKIALKGGKIVNNWLKQLIKVTLVFLFVAAIFYLITPVHVMSKHTDFSSEIIGYKAIDGGVTRDIASTDTCFANKHADFDTWFSQRGGSYTNDKACPLIAAVITREVGFVVPGLPGTILRTTNGDFLHFLPRVFSAVGNICYTPSKLIEYTDFATSACVLAAECTIFKDASGKPVPYCYDTNVLEGSVAYESLRPDTRYVLMDGSIIQFPNTYLEGSVRVVTTFDSEYCRHGTCERSEAGVCVSTSGRWVLNNDYYRSLPGVFCGVDAVNLLTNMFTPLIQPIGALDISASIVAGGIVAIVVTCLAYYFMRFRRAFGEYSHVVAFNTLLFLMSFTVLCLTPVYSFLPGVYSVIYLYLTFYLTNDVSFLAHIQWMVMFTPLVPFWITIAYIICISTKHFYWFFSNYLKRRVVFNGVSFSTFEEAALCTFLLNKEMYLKLRSDVLLPLTQYNRYLALYNKYKYFSGAMDTTSYREAACCHLAKALNDFSNSGSDVLYQPPQTSITSAVLQSGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQSAVKRTIKGTHHWLLLTILTSLLVLVQSTQWSLFFFLYENAFLPFAMGIIAMSAFAMMFVKHKHAFLCLFLLPSLATVAYFNMVYMPASWVMRIMTWLDMVDTSLSGFKLKDCVMYASAVVLLILMTARTVYDDGARRVWTLMNVLTLVYKVYYGNALDQAISMWALIISVTSNYSGVVTTVMFLARGIVFMCVEYCPIFFITGNTLQCIMLVYCFLGYFCTCYFGLFCLLNRYFRLTLGVYDYLVSTQEFRYMNSQGLLPPKNSIDAFKLNIKLLGVGGKPCIKVATVQSKMSDVKCTSVVLLSVLQQLRVESSSKLWAQCVQLHNDILLAKDTTEAFEKMVSLLSVLLSMQGAVDINKLCEEMLDNRATLQAIASEFSSLPSYAAFATAQEAYEQAVANGDSEVVLKKLKKSLNVAKSEFDRDAAMQRKLEKMADQAMTQMYKQARSEDKRAKVTSAMQTMLFTMLRKLDNDALNNIINNARDGCVPLNIIPLTTAAKLMVVIPDYNTYKNTCDGTTFTYASALWEIQQVVDADSKIVQLSEISMDNSPNLAWPLIVTALRANSAVKLQNNELSPVALRQMSCAAGTTQTACTDDNALAYYNTTKGGRFVLALLSDLQDLKWARFPKSDGTGTIYTELEPPCRFVTDTPKGPKVKYLYFIKGLNNLNRGMVLGSLAATVRLQAGNATEVPANSTVLSFCAFAVDAAKAYKDYLASGGQPITNCVKMLCTHTGTGQAITVTPEANMDQESFGGASCCLYCRCHIDHPNPKGFCDLKGKYVQIPTTCANDPVGFTLKNTVCTVCGMWKGYGCSCDQLREPMLQSADAQSFLNGFAV"
    refAA_ORF1b = "NRVCGVSAARLTPCGTGTSTDVVYRAFDIYNDKVAGFAKFLKTNCCRFQEKDEDDNLIDSYFVVKRHTFSNYQHEETIYNLLKDCPAVAKHDFFKFRIDGDMVPHISRQRLTKYTMADLVYALRHFDEGNCDTLKEILVTYNCCDDDYFNKKDWYDFVENPDILRVYANLGERVRQALLKTVQFCDAMRNAGIVGVLTLDNQDLNGNWYDFGDFIQTTPGSGVPVVDSYYSLLMPILTLTRALTAESHVDTDLTKPYIKWDLLKYDFTEERLKLFDRYFKYWDQTYHPNCVNCLDDRCILHCANFNVLFSTVFPPTSFGPLVRKIFVDGVPFVVSTGYHFRELGVVHNQDVNLHSSRLSFKELLVYAADPAMHAASGNLLLDKRTTCFSVAALTNNVAFQTVKPGNFNKDFYDFAVSKGFFKEGSSVELKHFFFAQDGNAAISDYDYYRYNLPTMCDIRQLLFVVEVVDKYFDCYDGGCINANQVIVNNLDKSAGFPFNKWGKARLYYDSMSYEDQDALFAYTKRNVIPTITQMNLKYAISAKNRARTVAGVSICSTMTNRQFHQKLLKSIAATRGATVVIGTSKFYGGWHNMLKTVYSDVENPHLMGWDYPKCDRAMPNMLRIMASLVLARKHTTCCSLSHRFYRLANECAQVLSEMVMCGGSLYVKPGGTSSGDATTAYANSVFNICQAVTANVNALLSTDGNKIADKYVRNLQHRLYECLYRNRDVDTDFVNEFYAYLRKHFSMMILSDDAVVCFNSTYASQGLVASIKNFKSVLYYQNNVFMSEAKCWTETDLTKGPHEFCSQHTMLVKQGDDYVYLPYPDPSRILGAGCFVDDIVKTDGTLMIERFVSLAIDAYPLTKHPNQEYADVFHLYLQYIRKLHDELTGHMLDMYSVMLTNDNTSRYWEPEFYEAMYTPHTVLQAVGACVLCNSQTSLRCGACIRRPFLCCKCCYDHVISTSHKLVLSVNPYVCNAPGCDVTDVTQLYLGGMSYYCKSHKPPISFPLCANGQVFGLYKNTCVGSDNVTDFNAIATCDWTNAGDYILANTCTERLKLFAAETLKATEETFKLSYGIATVREVLSDRELHLSWEVGKPRPPLNRNYVFTGYRVTKNSKVQIGEYTFEKGDYGDAVVYRGTTTYKLNVGDYFVLTSHTVMPLSAPTLVPQEHYVRITGLYPTLNISDEFSSNVANYQKVGMQKYSTLQGPPGTGKSHFAIGLALYYPSARIVYTACSHAAVDALCEKALKYLPIDKCSRIIPARARVECFDKFKVNSTLEQYVFCTVNALPETTADIVVFDEISMATNYDLSVVNARLRAKHYVYIGDPAQLPAPRTLLTKGTLEPEYFNSVCRLMKTIGPDMFLGTCRRCPAEIVDTVSALVYDNKLKAHKDKSAQCFKMFYKGVITHDVSSAINRPQIGVVREFLTRNPAWRKAVFISPYNSQNAVASKILGLPTQTVDSSQGSEYDYVIFTQTTETAHSCNVNRFNVAITRAKVGILCIMSDRDLYDKLQFTSLEIPRRNVATLQAENVTGLFKDCSKVITGLHPTQAPTHLSVDTKFKTEGLCVDIPGIPKDMTYRRLISMMGFKMNYQVNGYPNMFITREEAIRHVRAWIGFDVEGCHATREAVGTNLPLQLGFSTGVNLVAVPTGYVDTPNNTDFSRVSAKPPPGDQFKHLIPLMYKGLPWNVVRIKIVQMLSDTLKNLSDRVVFVLWAHGFELTSMKYFVKIGPERTCCLCDRRATCFSTASDTYACWHHSIGFDYVYNPFMIDVQQWGFTGNLQSNHDLYCQVHGNAHVASCDAIMTRCLAVHECFVKRVDWTIEYPIIGDELKINAACRKVQHMVVKAALLADKFPVLHDIGNPKAIKCVPQADVEWKFYDAQPCSDKAYKIEELFYSYATHSDKFTDGVCLFWNCNVDRYPANSIVCRFDTRVLSNLNLPGCDGGSLYVNKHAFHTPAFDKSAFVNLKQLPFFYYSDSPCESHGKQVVSDIDYVPLKSATCITRCNLGGAVCRHHANEYRLYLDAYNMMISAGFSLWVYKQFDTYNLWNTFTRLQSLENVAFNVVNKGHFDGQQGEVPVSIINNTVYTKVDGVDVELFENKTTLPVNVAFELWAKRNIKPVPEVKILNNLGVDIAANTVIWDYKRDAPAHISTIGVCSMTDIAKKPTETICAPLTVFFDGRVDGQVDLFRNARNGVLITEGSVKGLQPSVGPKQASLNGVTLIGEAVKTQFNYYKKVDGVVQQLPETYFTQSRNLQEFKPRSQMEIDFLELAMDEFIERYKLEGYAFEHIVYGDFSHSQLGGLHLLIGLAKRFKESPFELEDFIPMDSTVKNYFITDAQTGSSKCVCSVIDLLLDDFVEIIKSQDLSVVSKVVKVTIDYTEISFMLWCKDGHVETFYPKLQSSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAKYTQLCQYLNTLTLAVPYNMRVIHFGAGSDKGVAPGTAVLRQWLPTGTLLVDSDLNDFVSDADSTLIGDCATVHTANKWDLIISDMYDPKTKNVTKENDSKEGFFTYICGFIQQKLALGGSVAIKITEHSWNADLYKLMGHFAWWTAFVTNVNASSSEAFLIGCNYLGKPREQIDGYVMHANYIFWRNTNPIQLSSYSLFDMSKFPLKLRGTAVMSLKEGQINDMILSLLSKGRLIIRENNRVVISSDVLVNN*"
    refAA_S = "MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPRRARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKWPWYIWLGFIAGLIAIVMVTIMLCCMTSCCSCLKGCCSCGSCCKFDEDDSEPVLKGVKLHYT*"
    refAA_ORF3a = "MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSASKIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINFVRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPISEHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEPEEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL*"
    refAA_E = "MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYSRVKNLNSSRVPDLLV*"
    refAA_M = "MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPVTLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILLNVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYKLGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ*"
    refAA_ORF6 = "MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEID*"
    refAA_ORF7a = "MKIILFLALITLATCELYHYQECVRGTTVLLKEPCSSGTYEGNSPFHPLADNKFALTCFSTQFAFACPDGVKHVYQLRARSVSPKLFIRQEEVQELYSPIFLIVAAIVFITLCFTLKRKTE*"
    refAA_ORF7b = "MIELSLIDFYLCFLAFLLFLVLIMLIIFWFSLELQDHNETCHA*"
    refAA_ORF8 = "MKFLVFLGIITTVAAFHQECSLQSCTQHQPYVVDDPCPIHFYSKWYIRVGARKSAPLIELCVDEAGSKSPIQYIDIGNYTVSCLPFTINCQEPKLGSLVVRCSFYEDFLEYHDVRVVLDFI*"
    refAA_N = "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGKEDLKFPRGQGVPINTNSSPDDQIGYYRRATRRIRGGDGKMKDLSPRWYFYYLGTGPEAGLPYGANKDGIIWVATEGALNTPKDHIGTRNPANNAAIVLQLPQGTTLPKGFYAEGSRGGSQASSRSSSRSRNSSRNSTPGSSRGTSPARMAGNGGDAALALLLLDRLNQLESKMSGKGQQQQGQTVTKKSAAEASKKPRQKRTATKAYNVTQAFGRRGPEQTQGNFGDQELIRQGTDYKHWPQIAQFAPSASAFFGMSRIGMEVTPSGTWLTYTGAIKLDDKDPNFKDQVILLNKHIDAYKTFPPTEPKKDKKKKADETQALPQRQKKQQTVTLLPAADLDDFSKQLQQSMSSADSTQA*"
    refAA_ORF9b = "MDPKISEMHPALRLVDPQIQLAVTRMENAVGRDQNNVGPKVYPIILRLGSPLSLNMARKTLNSLEDKAFQLTPIAVQMTKLATTEELPDEFVVVTVK*"
    if ref_pango ≠ "Wuhan"
        if haskey(nuc_genome_pango_dict, ref_pango)
            ref_seq = nuc_genome_pango_dict[ref_pango]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(nuc_genome_pango_dict, minus_pango_X)
                    ref_seq = nuc_genome_pango_dict[minus_pango_X]
                    break
                end
            end
        end
##########################################################################################
        if haskey(gene_AA_pango_dict, ref_pango)
            refAA_ORF1a = gene_AA_pango_dict[ref_pango]["ORF1a"]
            refAA_ORF1b = gene_AA_pango_dict[ref_pango]["ORF1b"]
            refAA_S = gene_AA_pango_dict[ref_pango]["S"]
            refAA_ORF3a = gene_AA_pango_dict[ref_pango]["ORF3a"]
            refAA_E = gene_AA_pango_dict[ref_pango]["E"]
            refAA_M = gene_AA_pango_dict[ref_pango]["M"]
            refAA_ORF6 = gene_AA_pango_dict[ref_pango]["ORF6"]
            refAA_ORF7a = gene_AA_pango_dict[ref_pango]["ORF7a"]
            refAA_ORF7b = gene_AA_pango_dict[ref_pango]["ORF7b"]
            refAA_ORF8 = gene_AA_pango_dict[ref_pango]["ORF8"]
            refAA_N = gene_AA_pango_dict[ref_pango]["N"]
            refAA_ORF9b = gene_AA_pango_dict[ref_pango]["ORF9b"]
        elseif haskey(pango_predecessor_meta_dict, ref_pango)
            minus1_pango = ""
            minus2_pango = ""
            minus3_pango = ""
            minus4_pango = ""
            if haskey(pango_predecessor_meta_dict[ref_pango], 1)
                minus1_pango = pango_predecessor_meta_dict[ref_pango][1]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 2)
                minus2_pango = pango_predecessor_meta_dict[ref_pango][2]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 3)
                minus3_pango = pango_predecessor_meta_dict[ref_pango][3]
            end
            if haskey(pango_predecessor_meta_dict[ref_pango], 4)
                minus4_pango = pango_predecessor_meta_dict[ref_pango][4]
            end
            minus_pango_vec = [minus1_pango, minus2_pango, minus3_pango, minus4_pango]
            for minus_pango_X in minus_pango_vec
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        else
            for x in 1:5
                minus_pango_X = pango_minus_X_fx(ref_pango, x)
                if haskey(gene_AA_pango_dict, minus_pango_X)
                    refAA_ORF1a = gene_AA_pango_dict[minus_pango_X]["ORF1a"]
                    refAA_ORF1b = gene_AA_pango_dict[minus_pango_X]["ORF1b"]
                    refAA_S = gene_AA_pango_dict[minus_pango_X]["S"]
                    refAA_ORF3a = gene_AA_pango_dict[minus_pango_X]["ORF3a"]
                    refAA_E = gene_AA_pango_dict[minus_pango_X]["E"]
                    refAA_M = gene_AA_pango_dict[minus_pango_X]["M"]
                    refAA_ORF6 = gene_AA_pango_dict[minus_pango_X]["ORF6"]
                    refAA_ORF7a = gene_AA_pango_dict[minus_pango_X]["ORF7a"]
                    refAA_ORF7b = gene_AA_pango_dict[minus_pango_X]["ORF7b"]
                    refAA_ORF8 = gene_AA_pango_dict[minus_pango_X]["ORF8"]
                    refAA_N = gene_AA_pango_dict[minus_pango_X]["N"]
                    refAA_ORF9b = gene_AA_pango_dict[minus_pango_X]["ORF9b"]
                    break
                end
            end
        end
    end
    return ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_nucs_filter(mut_arr)
    mixed_nuc_arr = Vector{String}()
    mixed_nuc_res_list = Set(["Y", "R", "K", "M", "W", "S"])
    for mut in mut_arr
        if string(mut[end]) in mixed_nuc_res_list
            push!(mixed_nuc_arr, mut)
        end
    end
    return mixed_nuc_arr
end
#####################################################################################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
#####################################################################################################################################
function mixed2nuc(mix_mut::String)
    nuc_mut = mix_mut
    qrynuc = qry_nuc_comprehensive_dict[mix_mut]
    refnuc = ref_nuc_comprehensive_dict[mix_mut]
    pos_str = nuc_mut_int_string_comprehensive_dict[mix_mut]
    ref_n_pos = refnuc*pos_str
    if length(mix_mut) ≥ 4
        if qrynuc == "T"
            nuc_mut = mix_mut
        elseif qrynuc == "C"
            nuc_mut = mix_mut
        elseif qrynuc == "A"
            nuc_mut = mix_mut
        elseif qrynuc == "G"
            nuc_mut = mix_mut
        elseif qrynuc == "N"
            nuc_mut = mix_mut
        end   
        if refnuc == "T"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "R"
                nuc_mut = "$(ref_n_pos)A, $(ref_n_pos)G"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            end
        end
        if refnuc == "A"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"G"
            elseif qrynuc == "W"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "M"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "K"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)G"
            elseif qrynuc == "S"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)G"
            end
        end
        if refnuc == "G"
            if qrynuc == "R"
                nuc_mut = ref_n_pos*"A"
            elseif qrynuc == "K"
                nuc_mut = ref_n_pos*"T"
            elseif qrynuc == "S"
                nuc_mut = ref_n_pos*"C"
            elseif qrynuc == "Y"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)C"
            elseif qrynuc == "W"
                nuc_mut = "$(ref_n_pos)T, $(ref_n_pos)A"
            elseif qrynuc == "M"
                nuc_mut = "$(ref_n_pos)C, $(ref_n_pos)A"
            end
        end
    end
    return nuc_mut
end
######################################################################################################################################
function muts_to_strings(mut_list_in_string_form::String)
    mut_arr = split(mut_list_in_string_form, ", ")
    mut_array = Vector{String}()
    for mut in mut_arr
        if string(mut[end]) ≠ "-"
            mutstr = string(mut)
            push!(mut_array, mutstr)
        end
    end
    sortKey(n) = (length(n), nuc_mut_int_comprehensive_dict[n])  ## Fx ##
    mut_array_sort = sort(mut_array, by = x -> sortKey(x))
#    mixed_mut_arr = mixed_nucs_filter(mut_arr)
    return mut_array_sort
end
######################################################################################################################################
function mixed_mut_to_regular_mut(mixed_nuc_muts)            ### New, 2025-1-26 (entire function)
    ct = 0
    mixed_regular_muts = Vector{String}()
    for i in 1:length(mixed_nuc_muts)
        mut = mixed_nuc_muts[i]
        qrynuc = qry_nuc_comprehensive_dict[mut]
        refnuc = ref_nuc_comprehensive_dict[mut]
        pos_str = nuc_mut_int_string_comprehensive_dict[mut]
        ref_n_pos = refnuc*pos_str
        if refnuc == "T"
            if qrynuc == "Y"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "C"
            if qrynuc == "Y"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "R"
                new_end1 = "A"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
        if refnuc == "A"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "G"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end1 = "T"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "M"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "W"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "S"
                new_end1 = "C"
                new_end2 = "G"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
        end
        if refnuc == "G"
            if qrynuc == "Y"
                new_end1 = "T"
                new_end2 = "C"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "R"
                new_end = "A"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "K"
                new_end = "T"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
            if qrynuc == "M"
                new_end1 = "C"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "W"
                new_end1 = "T"
                new_end2 = "A"
                new_mut1 = ref_n_pos*new_end1
                new_mut2 = ref_n_pos*new_end2
                push!(mixed_regular_muts, new_mut1)
                push!(mixed_regular_muts, new_mut2)
            end
            if qrynuc == "S"
                new_end = "C"
                new_mut = ref_n_pos*new_end
                push!(mixed_regular_muts, new_mut)
            end
        end
    end
    return mixed_regular_muts
end
######################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#######################################################################################################################################
function nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts_filtered = filter(!isempty, muts)
#    all_muts_sort = sort(muts_filtered, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
    nonsynonymous_nuc_to_context_dict = Dict{String, String}()
    all_nuc_to_AA_dict = Dict{String, String}()
    all_nuc_to_context_dict = Dict{String, String}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
################################################        
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nuc_pos in gene_nuc_arr[i]
            nuc_gene_num[nuc_pos] = i-1
        end
    end
    for nuc_pos in gene_nuc_arr[end]
        nuc_gene_num_9b[nuc_pos] = 11
    end

    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
    
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                                          ### FUNCTION #####################
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)                  ### FUNCTION #####################
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                                            ### FUNCTION #####################
    nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA   ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
    
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nuc_pos in coding_ranges
        gene_number = nuc_gene_num[nuc_pos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict[nuc_pos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nuc_pos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nuc_pos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nuc_pos] = codon_num
    end
#######################################################################################################################
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ",")[1])
            mut2 = string(split(mut, ",")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>6), muts)
        end
    end
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
            mut = mixed2nuc(nuc_mut)  
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end
            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            push!(AA_mut_set, AA_mut)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            all_nuc_to_AA_dict[mut] = AA_mut
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
                nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
#                push!(nonsynonymous_nuc_muts, mut)
            end
            all_nuc_to_context_dict[mut] = ref2qry_context
###################################
            for nuc_mut2 in muts
                mut2 = mixed2nuc(nuc_mut2)
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        push!(AA_mut_set, AA_mut2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        all_nuc_to_AA_dict[mut2] = AA_mut2
                        all_nuc_to_AA_dict[mut] = AA_mut2
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context 
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut2] = ref2qry_context
                            nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            nonsynonymous_nuc_to_context_dict[mut] = ref2qry_context
                        end
                        all_nuc_to_context_dict[mut] = ref2qry_context
                        all_nuc_to_context_dict[mut2] = ref2qry_context
                    end
                end
            end
        else                  
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
            ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
            full_nc_context = ref_nc_nuc_context*"|"*qry_nc_nuc_context
            noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
            for (start_end, place) in noncoding_range_dict
                frst = start_end[1]
                last = start_end[2]
                if npos ≥ frst && npos ≤ last
                    noncoding_to_noncoding_region_dict[nuc_mut] = place
                    mut_vec = [nuc_mut, place]
                    push!(noncoding_nuc_vector, nuc_mut)
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
            mut_9b = mixed2nuc(nuc_mut)
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])  *string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            push!(AA_mut_set, AA_mut_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            end
            if refAA_9b ≠ qryAA_9b
                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
#                push!(nonsynonymous_nuc_muts, mut_9b)
            end
            all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
###################################
            for nuc_mut2_9b in muts
                mut2_9b = mixed2nuc(nuc_mut2_9b)
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        push!(AA_mut_set, AA_mut2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            nonsynonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        end
                        all_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        all_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                    end
                end
            end
        end
    end
#################################################################################
    mut_vec_dict = Dict{String, Vector{String}}()
    for gene in gene_print_array
        mut_vec_dict[gene] = Vector{String}()
    end
    for mut in AA_mut_set
        jeen = aa_gene_comprehensive_dict[mut]
        mut_only = string(split(mut, ":")[2])
        push!(mut_vec_dict[jeen], mut_only) 
    end
    for gene in keys(mut_vec_dict)
        if !isempty(mut_vec_dict[gene])
            sort!(mut_vec_dict[gene], by = x -> x[2:end-1])
        end
    end
#####################################################################################################################################
    aasortkeyhere(m) = (gene_order_dict[aa_gene_comprehensive_dict[m]], aa_pos_comprehensive_dict[m])
    AA_sort = sort(collect(AA_mut_set), by = x -> aasortkeyhere(x))
    AA_sort2 = Vector{String}()
    for mut in AA_sort
        refAA = refAA_comprehensive_dict[mut]
        qryAA = qryAA_comprehensive_dict[mut]
        if !(refAA == qryAA)
            push!(AA_sort2, mut)
        end
    end
#    print("\n"^2)
#    println("##################### Amino Acid Mutations ######################")
#    for i in 1:length(AA_sort2)
#        mut = AA_sort2[i]
#        gene = aa_gene_comprehensive_dict[mut]
#        non_gene = string(split(mut, ":")[2])
#        if i == 1
#            print("               ", AA_sort2[i])
#        elseif i > 1
#            lastmut = AA_sort2[i-1]
#            last_gene = aa_gene_comprehensive_dict[lastmut]
#            last_non_gene = string(split(lastmut, ":")[2])
#            if gene == last_gene
#                print(", $(non_gene)")
#            else
#                println()
#                print("               $(mut)")
#            end
#        end
#    end
#    print("\n"^2)
#####################################################################################################################################
    all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    all_nuc_to_context_dict_sort = sort(collect(all_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    nonsynonymous_nuc_to_context_dict_sort = sort(collect(nonsynonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
####################################################################################################################################
    nc_len = length(noncoding_to_noncoding_region_dict_sort)
#    println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
    if !isempty(noncoding_to_noncoding_region_dict_sort)
        for i in 1:length(noncoding_to_noncoding_region_dict_sort)
            nc_nuc = noncoding_to_noncoding_region_dict_sort[i][1]
            nc_nuc_pad = rpad(nc_nuc, 7)
            nc_region = noncoding_to_noncoding_region_dict_sort[i][2]
            nc_region_len = length(nc_region)
            ncpad1 = (13 - nc_region_len)÷2
            ncpad12 = " "^ncpad1*nc_region
            nc_region_pad2 = rpad(ncpad12, 13)
            nc_context = noncoding_nuc_to_context_dict_sort[i][2]
            premut_context = ""
            postmut_context = ""
            context = split(nc_context, "-->")
            if !isempty(context)
                if length(context) >1
                    premut_context = split(nc_context, "-->")[1]
                    postmut_context = split(nc_context, "-->")[2]
                end
            end    
            postpad = lpad(postmut_context, 39)
#            println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#            println(postpad)
#                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
#                println(g, postpad)
        end
#        println("#"^94); println() 
#            println(g, "#"^94); println(g)
    end
######################################################################################################
    total_syn = length(synonymous_nuc_to_AA_dict_sort)
#    println("SYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(synonymous_nuc_to_AA_dict_sort)
        synnuc = synonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = synonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = synonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
#################################################################
    total_syn = length(nonsynonymous_nuc_to_AA_dict_sort)
#    println("NONSYNONYMOUS NUC MUTATIONS — Total:$(total_syn)")
    for i in 1:length(nonsynonymous_nuc_to_AA_dict_sort)
        synnuc = nonsynonymous_nuc_to_AA_dict_sort[i][1]
        synnuc_pad = rpad(synnuc, 7)
        synAA = nonsynonymous_nuc_to_AA_dict_sort[i][2]
        AAlen = length(synAA)
        pad1 = (14 - AAlen)÷2
        pad12 = " "^pad1*synAA
        synAA_pad = rpad(pad12, 14)
        syncontext = nonsynonymous_nuc_to_context_dict_sort[i][2]
        premut_context = split(syncontext, "-->")[1]
        postmut_context = split(syncontext, "-->")[2]
        postpad = lpad(postmut_context, 38)
#        println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#        println(postpad)
#            println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#            println(g, postpad)
    end
#    println()
###################################################################
    nonsynonymous_nuc_total = length(nonsynonymous_nuc_sort)
#    println("        Total Number of Non-synonymous Nuc Muts = $(nonsynonymous_nuc_total)")
    nonsynonymous_nuc_sort_join = join(nonsynonymous_nuc_sort, ", ")
#    println("################ Nonsynonymous Nuc Mutations ################")
#    println(nonsynonymous_nuc_sort_join)
#    print("\n"^1)
    for i in 1:length(nonsynonymous_nuc_sort)
#        println("               $(nonsynonymous_nuc_to_AA_dict_sort[i][1]) | $(nonsynonymous_nuc_to_AA_dict_sort[i][2])")
        premut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
        postmut_nucseq = split(nonsynonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        println("                 $(premut_nucseq)")
#        println("                 $(postmut_nucseq)")
    end
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
function mixed_muts_to_AA(ref_pango::String, muts::String)
    mut_strings = muts_to_strings(muts)
    mixed_muts = mixed_nucs_filter(mut_strings)             ### New, 2025-1-26
    mixed_muts_regular = mixed_mut_to_regular_mut(mixed_muts)    ### New, 2025-1-26
    ct = 0
    total_mixed_muts = length(mixed_muts)
#    println("Total Mixed Nucs = $(total_mixed_muts)")
#    print("\n"^2)
    for i in 1:length(mixed_muts)
        if ct == 0
#            print(mixed_muts[i])
            ct = 1
        else
#            print(", ", mixed_muts[i])
        end
    end
    ct2 = 0
#    print("\n"^2)
    for i in 1:length(mixed_muts_regular)
        if ct2 == 0
#            print(mixed_muts_regular[i])
            ct2 = 1
#        else
#            print(", ", mixed_muts_regular[i])
        end
    end
#    println()
    syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort = nuc_to_AA(ref_pango, mixed_muts_regular)
    AA__AAprintlen__vec = []
    nonsynnuc__nonsynprintlen__vec = []
    for nuc___AA in nonsyn_nuc_to_AA_dict_sort
        nucmut = nuc___AA[1]
        nucmutlen = length(nucmut)
        AAmut = nuc___AA[2]
        AAmutlen = length(AAmut)
        push!(AA__AAprintlen__vec, [AAmut, AAmutlen])
        push!(nonsynnuc__nonsynprintlen__vec, [nucmut, nucmutlen])
    end
    aa_pad_vec = String[]
    nuc_pad_vec = String[]
    for i in 1:length(AA__AAprintlen__vec)
        aa = AA__AAprintlen__vec[i][1]
        nuc = nonsynnuc__nonsynprintlen__vec[i][1]
        aapad = AA__AAprintlen__vec[i][2]
        nucpad = nonsynnuc__nonsynprintlen__vec[i][2]
        pads = [nucpad, aapad]
        pad = maximum(pads)
        push!(aa_pad_vec, rpad(aa, pad))
        push!(nuc_pad_vec, rpad(nuc, pad))
    end
    aapad_join = join(aa_pad_vec, ", ")
    nucpad_join = join(nuc_pad_vec, ", ")
#    println("\n"^1)
#    println(aapad_join)
#    println(nucpad_join)
#    println("\n"^1)
    return syn_nuc_to_AA_dict_sort, nonsyn_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
end
######################################################################################################################################
function count_nuc_mut_types(mut_strings::Vector{String})
    mut_types_arr = ["TC", "TA", "TG", "CT", "CA", "CG", "AT", "AC", "AG", "GT", "GC", "GA"]
    mut_type_cts = Dict{String, Int}(mut_nuc_type=>0 for mut_nuc_type in mut_types_arr)
    for nuc_mut in mut_strings
        ref = ref_nuc_comprehensive_dict[nuc_mut]
        qry = ref_nuc_comprehensive_dict[nuc_mut]
        if ref == "T"
            if qry == "C"
                mut_type_cts["TC"] += 1
            elseif qry == "A"
                mut_type_cts["TA"] += 1
            elseif qry == "G"
                mut_type_cts["TG"] += 1
            end
        end
        if ref == "C"
            if qry == "T"
                mut_type_cts["CT"] += 1
            elseif qry == "A"
                mut_type_cts["CA"] += 1
            elseif qry == "G"
                mut_type_cts["CG"] += 1
            end
        end 
        if ref == "A"
            if qry == "T"
                mut_type_cts["AT"] += 1
            elseif qry == "C"
                mut_type_cts["AC"] += 1
            elseif qry == "G"
                mut_type_cts["AG"] += 1
            end
        end   
        if ref == "G"
            if qry == "T"
                mut_type_cts["GT"] += 1
            elseif qry == "C"
                mut_type_cts["GC"] += 1
            elseif qry == "A"
                mut_type_cts["GA"] += 1
            end
        end
    end
    mut_type_cts_sort_by_type = sort(collect(mut_type_cts), by = x -> x[1])
    mut_type_cts_sort_by_count = sort(collect(mut_type_cts), by = x -> x[2], rev=true)
    return mut_type_cts_sort_by_count
end
######################################################################################################################################
function AA_triple(pos::Int, rem0::BitSet, rem1::BitSet, rem2::BitSet, mut::String, ref_pango::String)
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ref_triple = ""
    qry_triple = ""
    if pos in rem0 && pos%3 == 0
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem1 && pos%3 == 1
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem2 && pos%3 == 2
        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
    elseif pos in rem0 && pos%3 == 1
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem1 && pos%3 == 2
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem2 && pos%3 == 0
        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
    elseif pos in rem0 && pos%3 == 2
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem1 && pos%3 == 0
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    elseif pos in rem2 && pos%3 == 1
        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
    end
    return ref_triple, qry_triple
end
######################################################################################################################################
### Fx: SIMPLER_nuc_to_AA (no context)
function SIMPLER_nuc_to_AA(ref_pango::String, muts::Vector{String})
    muts = filter(!isempty, muts)
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
    ###############################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        nonsynonymous_nuc_to_AA_dict = Dict{String, String}()
        all_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
        N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut_set = Set{String}()
        AA_mut = ""
        for nuc_mut in muts
            pos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos in coding_ranges
                mut = mixed2nuc(nuc_mut)  
                gene_number = gene_num(mut)
                ref_triple = ""
                qry_triple = ""
                if nuc_codon_pos_dict[pos] == 1
                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                elseif nuc_codon_pos_dict[pos] == 2
                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                elseif nuc_codon_pos_dict[pos] == 3
                    ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                    qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                end
                refAA = AA_triplets[ref_triple]
                qryAA = AA_triplets[qry_triple]
                AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                push!(AA_mut_set, AA_mut)
                all_nuc_to_AA_dict[mut] = AA_mut
                if refAA == qryAA && !(pos in rem9b)
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut] = AA_mut
                else
                    nonsynonymous_nuc_to_AA_dict[mut] = AA_mut
#                    push!(nonsynonymous_nuc_muts, mut)
                end
###################################
                for nuc_mut2 in muts
                    mut2 = mixed2nuc(nuc_mut2)
                    pos2 = nuc_mut_int_comprehensive_dict[mut2]
                    if pos2 in coding_ranges
                        gene_number2 = gene_num(mut2)
                        if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                            if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                            end
                            refAA2 = AA_triplets[ref_triple]
                            qryAA2 = AA_triplets[qry_triple]
                            AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                            push!(AA_mut_set, AA_mut2)
                            delete!(AA_mut_set, AA_mut)
                            all_nuc_to_AA_dict[mut2] = AA_mut2
                            all_nuc_to_AA_dict[mut] = AA_mut2
                            if refAA2 == qryAA2 && !(pos2 in rem9b)
                                synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                nonsynonymous_nuc_to_AA_dict[mut] = AA_mut2
                            end
                        end
                    end
                end
            elseif !isempty(nuc_mut)
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if pos ≥ frst && pos ≤ last
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                push!(AA_mut_set, AA_mut_9b)
                all_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
                if refAA_9b ≠ qryAA_9b
                    nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
#                    push!(nonsynonymous_nuc_muts, mut_9b)
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            push!(AA_mut_set, AA_mut2_9b)
                            delete!(AA_mut_set, AA_mut_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            else
                                nonsynonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                nonsynonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
        AA_sort2 = Vector{String}()
        for aa in AA_sort
            if !(refAA_comprehensive_dict[aa] == qryAA_comprehensive_dict)
                push!(AA_sort2, aa)
            end
        end
#        print("\n"^1)
#        ct = 0
#        println("############# AA Mutations #############")
#        for i in 1:length(AA_sort2)
#            mut = AA_sort2[i]
#            gene = string(split(mut, ":")[1])
#            non_gene = string(split(mut, ":")[2])
#            if i == 1
#                print("        ", AA_sort2[i])
#            elseif i > 1
#                lastmut = AA_sort2[i-1]
#                last_gene = string(split(lastmut, ":")[1])
#                last_non_gene = string(split(lastmut, ":")[2])
#                if gene == last_gene
#                    print(", $(non_gene)")
#                else
#                    print("        $(mut)")
#                end
#            end
#        end
#####################################################################################################################################
        all_nuc_to_AA_dict_sort = sort(collect(all_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_to_AA_dict_sort = sort(collect(nonsynonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        nonsynonymous_nuc_sort = sort(collect(keys(nonsynonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        nonsynonymous_AA_sort = sort(collect(values(nonsynonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
        for i in 1:length(synonymous_nuc_sort)
            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
        end
#        return synonymous_nuc_to_AA_dict, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
        return synonymous_nuc_to_AA_dict_sort, nonsynonymous_nuc_to_AA_dict_sort, all_nuc_to_AA_dict_sort
    end
    if isempty(muts)
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
###########################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
function pango_minus_1_fx(pango::String)
    if '.' in pango
        dot_ct = count(".", pango)
        dotsplits = split(pango, ".")
        minus_1 = join(dotsplits(1:dot_ct-1), ".")
        return minus_1
    else
        return pango
    end
end
############################################################################################################################################################################
### Fx: SIMPLER_syn_noncoding_nonsyn_nuc (no context)
function SIMPLER_syn_noncoding_nuc(ref_pango::String, muts::Set{String})
    B_1_1_list = ["B.1.1.53", "B.1.1.273"]
    if ref_pango in B_1_1_list
        ref_pango = "B.1.1"
    end
    if ref_pango == "XBB.1.5.82"
         ref_pango = "XBB.1.5"
    end
    if !isempty(muts)
#        all_muts_sort = sort(collect(muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
###################################################################################################################################
        coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
        noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
        coding_range_9b = BitSet([28284:28577...])
        gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
        ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
        gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
        nuc_gene_num = Dict{Int, Int}()
        nuc_gene_num_9b = Dict{Int, Int}()
        synonymous_nuc_to_AA_dict = Dict{String, String}()
        noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"Octanucleotide Motif")
################################################
        noncoding_nuc_vector = Vector{String}()
################################################ 
        gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
        for i in 1:length(gene_nuc_arr)-1
            for nuc_pos in gene_nuc_arr[i]
                nuc_gene_num[nuc_pos] = i-1
            end
        end
        for nuc_pos in gene_nuc_arr[end]
            nuc_gene_num_9b[nuc_pos] = 11
        end
        rem0_gene = [5, 8, 9, 11]
        rem1_gene = [1, 3, 4, 6, 7]
        rem2_gene = [0, 2, 10]
        rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
        rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
        rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
        rem9b = BitSet([28284:28577...])
        rem7ab = BitSet([27756:27759...])

        gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]] ## Fx ##
        nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3) ## Fx ##
        nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3) ## Fx ##
        nuc2AA_ORF1a(nuc_mut, refAA, qryAA) = gene_strings[gene_num(nuc_mut)]*":"*refAA*nuc_to_AA_pos(nuc_mut)*qryAA ## Fx ##
        nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:"*refAA*nuc_to_AA_pos_9b(nuc_mut)*qryAA
        nuc_codon_pos_dict = Dict{Int, Int}()
        for nuc_pos in coding_ranges
            gene_number = nuc_gene_num[nuc_pos]
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict[nuc_pos] = codon_num
        end
        nuc_codon_pos_dict_9b = Dict{Int, Int}()
        for nuc_pos in coding_range_9b
            gene_number = 11
            gene_start = gene_nuc_starts[gene_number]
            codon_num = (nuc_pos-gene_start)%3 + 1
            nuc_codon_pos_dict_9b[nuc_pos] = codon_num
        end    
        for nuc_mut in muts
            if !isempty(muts)
                mut = mixed2nuc(nuc_mut)
                if ',' in mut
                    mut1 = string(split(mut, ",")[1])
                    mut2 = string(split(mut, ",")[2])
                    push!(muts, mut1)
                    push!(muts, mut2)
                    filter!(x -> !(length(x)>6), muts)
                end
            end
        end
        coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
        N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
        AA_mut = ""
        for nuc_mut in muts
            if !isempty(muts) && !isempty(ref_seq) && nuc_mut ≠ ""
                pos = nuc_mut_int_comprehensive_dict[nuc_mut]
                if pos in coding_ranges
                    mut = mixed2nuc(nuc_mut)
                    if isempty(mut)
                        println(nuc_mut)
                        println(mut)
                        println("$(pango)")
                    end
                    try
                        nuc_mut_int_comprehensive_dict[mut]
                    catch e
                        println("Problematic mutation string: ", repr(mut))
                        println("Length: ", length(mut))
                        println("Extracted substring: ", repr(mut[2:end-1]))
                        rethrow(e)
                    end
                    gene_number = nuc_gene_num[pos]
                    ref_triple = ""
                    qry_triple = ""
                    if nuc_codon_pos_dict[pos] == 1
                        ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                        qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                    elseif nuc_codon_pos_dict[pos] == 2
                        ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                        qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                    elseif nuc_codon_pos_dict[pos] == 3
                        ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                        qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                    end
                    refAA = AA_triplets[ref_triple]
                    qryAA = AA_triplets[qry_triple]
                    AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
                    if refAA == qryAA && !(pos in rem9b)
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                        synonymous_nuc_to_AA_dict[mut] = AA_mut
                    end
###################################
                    for nuc_mut2 in muts
                        mut2 = mixed2nuc(nuc_mut2)
                        pos2 = nuc_mut_int_comprehensive_dict[mut2]
                        if pos2 in coding_ranges
                            gene_number2 = gene_num(mut2)
                            if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                                if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                                elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                                    ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                                    qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                                elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                                elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                                    ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                                    qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                                end
                                refAA2 = AA_triplets[ref_triple]
                                qryAA2 = AA_triplets[qry_triple]
                                AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                                if refAA2 == qryAA2 && !(pos2 in rem9b)
                                    synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                                end
                            end
                        end
                    end
                else
                    qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                    for (start_end, place) in noncoding_range_dict
                        frst = start_end[1]
                        last = start_end[2]
                        if pos ≥ frst && pos ≤ last
                            mut_vec = [nuc_mut, place]
                            push!(noncoding_nuc_vector, nuc_mut)
                        end
                    end
                end
            end
        end
#########################################################################################################
        for nuc_mut in muts
            pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
            if pos_9b in rem9b && !isempty(ref_seq) && nuc_mut ≠ ""
                mut_9b = mixed2nuc(nuc_mut)
                pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
                gene_number_9b = 11
                ref_triple_9b = ""
                qry_triple_9b = ""
                if nuc_codon_pos_dict_9b[pos_9b] == 1
                    ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                    qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                    ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                    qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                    ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                    qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                end
                refAA_9b = AA_triplets[ref_triple_9b]
                qryAA_9b = AA_triplets[qry_triple_9b]
                AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
                if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                    synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                end
###################################
                for nuc_mut2_9b in muts
                    mut2_9b = mixed2nuc(nuc_mut2_9b)
                    pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                    if pos2_9b in rem9b
                        gene_number2_9b = 11
                        if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                            if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                                ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                                qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                            end
                            refAA2_9b = AA_triplets[ref_triple_9b]
                            qryAA2_9b = AA_triplets[qry_triple_9b]
                            AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                            if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                                synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            end
                        end
                    end
                end
            end
        end
#####################################################################################################################################
        synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
        synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
        noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
        synonymous_nuc_total = length(synonymous_nuc_sort)
#        for i in 1:length(synonymous_nuc_sort)
#            nucpad = rpad("$(synonymous_nuc_to_AA_dict_sort[i][1])", 10)
#        end
        return synonymous_nuc_sort, noncoding_nuc_vector_sort
    else
        aa = Vector{Pair{String, String}}()
        bb = Vector{Pair{String, String}}()
        cc = Vector{Pair{String, String}}()
        return aa, bb, cc
    end 
end
#################################################################################
function add_leading_zero(int_str::String)
    int_str2 = int_str
    if length(int_str) == 1 && int_str ≠ "0"
        int_str2 = "0"*int_str
    end
    return int_str2
end     
##############################################################################################################################
###################################################################################################################
index_to_date_str = Dict{Int, String}()
date_str_to_index = Dict{String, Int}()
function convert_date_to_date_index(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    date_index = tuple_to_index[date_tuple]
    return date_index
end
print("\n"^1)
println("Done Loading Functions (line #1618 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##############################################################################################################################################
##############################################################################################################################################
##############################################################################################################################################
for tuple in keys(tuple_to_index)
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    date_str_to_index[date_string] = convert_date_to_date_index(date_string)
end
for (index, tuple) in index_to_tuple
    one = add_leading_zero(string(tuple[1]))
    two = add_leading_zero(string(tuple[2]))
    three = add_leading_zero(string(tuple[3]))
    date_string = one*"-"*two*"-"*three
    index_to_date_str[index] = date_string
end
println("Done Making index_to_tuple & tuple_to_index (line #1641 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################
date_to_tuple = Dict{String, Tuple{Int, Int, Int}}()
tuple_to_date = Dict{Tuple{Int, Int, Int}, String}()
function convert_date_to_date_tuple(date_str::String)
    date_arr = string.(collect(date_str))
    date_tuple = nothing
### This counts the number of times "-" appears in the date_arr ---> sum(date_arr .== "-")
    if sum(date_arr .== "-") == 0
        year = parse(Int, date_str)
        date_tuple = (year, 0, 0)
    end
    if sum(date_arr .== "-") > 0
        year = parse(Int, split(date_str, "-")[1])
        month = parse(Int, split(date_str, "-")[2])
        if sum(date_arr .== "-") == 1
            date_tuple = (year, month, 0)
        else
            day = parse(Int, split(date_str, "-")[3])
            date_tuple = (year, month, day)
        end
    end
    return date_tuple
end
for date in keys(date_str_to_index)
    date_to_tuple[date] = convert_date_to_date_tuple(date)
end
for (date, date_tuple) in date_to_tuple
    tuple_to_date[date_tuple] = date
end
println("Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
##################################################################################################################
function find_X_pct_date(clade_pango::String, pct::Float64, cp_date_cumul_dict::Dict{String, Dict{Int, Int}}, cp_total_dict::Dict{String, Int})
    cp_total = cp_total_dict[clade_pango]
    pct_date_index = 0
    pct_date_tuple = nothing
    for date_index in 1:2500
        cumulative_ct = cp_date_cumul_dict[clade_pango][date_index]
        if 100*cumulative_ct/cp_total ≥ pct
            pct_date_index = date_index
            pct_date_tuple = index_to_tuple[date_index]
            break
        end
    end
    return pct_date_index, pct_date_tuple
end
##################################################################################################################
### Truly necessary stuff from old megacell | 2026_02_08
############################################################################################################################################################################
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
            if haskey(pango_predecessor_meta_dict[pango], 2)
                pango = pango_predecessor_meta_dict[pango][2]
                if !haskey(pango_AAsub_WT, pango)
                    if haskey(pango_predecessor_meta_dict, pango)
                        if haskey(pango_predecessor_meta_dict[pango], 3)
                            pango = pango_predecessor_meta_dict[pango][3]
                        end
                    end
                end
            end
        end
    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_AAsub_WT, pango)
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    elseif haskey(pango_predecessor_meta_dict, pango)
        if haskey(pango_predecessor_meta_dict[pango], 1)
            pango_pre1 = pango_predecessor_meta_dict[pango][1]
            for mut in pango_AAsub_WT[pango_pre1]
                all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
            end
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
nowtime = Dates.format(now(), "I:MM.SS_p"); println(nowtime); print("\n"^1)
############################################################################################################################################################################
runtime = time() - start
runtime_rd = round(digits=2, runtime)
runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
println("Runtime v0 = $(runtime) seconds")
println("Runtime v2 = $(runtime2)")
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
println("Finished!"); print("\n"^1)

6:46.43_PM
2026_03_05__646PM

Done Loading Functions (line #1618 as of 2022_02_22)!
6:46.43_PM

Done Making index_to_tuple & tuple_to_index (line #1641 as of 2022_02_22)!
6:46.43_PM

Done Making date_to_tuple & tuple_to_date (line #1672 as of 2022_02_22)!
6:46.44_PM

Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!
6:46.44_PM

Done Filling seq_syn_nucs, seq_noncoding_nucs!
6:48.19_PM

Runtime v0 = 96.53191184997559 seconds
Runtime v2 = 0 hr, 1 min, 36.53 sec
Finished!



In [21]:
### print_all_seq_info Fx + All gene/AA/nuc/key + synonymous_nuc_to_AA_and_noncoding_to_context #######################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
for seq in all_unique_chr_seqs_set 
    for del in seq_AA_dels[seq]
        aa_gene_comprehensive_dict[del] = string(split(del, ":")[1])
        firstdel = string(split(del, "-")[1])
        aa_pos_comprehensive_dict[del] = aa_pos_comprehensive_dict[firstdel]
    end
end
######################################################################################################################################
AA_triplets = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"X", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"X", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"X", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"X", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"X", "--C"=>"X", "--A"=>"X", "--G"=>"X", "---"=>"X", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
AA_triplet_dels = Dict{String, String}("TTT"=>"F", "TTC"=>"F", "TTA"=>"L", "TTG"=>"L", "TCT"=>"S", "TCC"=>"S", "TCA"=>"S", "TCG"=>"S", "TAT"=>"Y", "TAC"=>"Y", "TAA"=>"*", "TAG"=>"*", "TGT"=>"C", "TGC"=>"C", "TGA"=>"*", "TGG"=>"W", "CTT"=>"L", "CTC"=>"L", "CTA"=>"L", "CTG"=>"L", "CCT"=>"P", "CCC"=>"P", "CCA"=>"P", "CCG"=>"P", "CAT"=>"H", "CAC"=>"H", "CAA"=>"Q", "CAG"=>"Q", "CGT"=>"R", "CGC"=>"R", "CGA"=>"R", "CGG"=>"R", "ATT"=>"I", "ATC"=>"I", "ATA"=>"I", "ATG"=>"M", "ACT"=>"T", "ACC"=>"T", "ACA"=>"T", "ACG"=>"T", "AAT"=>"N", "AAC"=>"N", "AAA"=>"K", "AAG"=>"K", "AGT"=>"S", "AGC"=>"S", "AGA"=>"R", "AGG"=>"R", "GTT"=>"V", "GTC"=>"V", "GTA"=>"V", "GTG"=>"V", "GCT"=>"A", "GCC"=>"A", "GCA"=>"A", "GCG"=>"A", "GAT"=>"D", "GAC"=>"D", "GAA"=>"E", "GAG"=>"E", "GGT"=>"G", "GGC"=>"G", "GGA"=>"G", "GGG"=>"G", "TT-"=>"X", "TC-"=>"X", "TA-"=>"X", "TG-"=>"X", "T-T"=>"X", "T-C"=>"X", "T-A"=>"X", "T-G"=>"X", "T--"=>"-", "CT-"=>"X", "CC-"=>"X", "CA-"=>"X", "CG-"=>"X", "C-T"=>"X", "C-C"=>"X", "C-A"=>"X", "C-G"=>"X", "C--"=>"-", "AT-"=>"X", "AC-"=>"X", "AA-"=>"X", "AG-"=>"X", "A-T"=>"X", "A-C"=>"X", "A-A"=>"X", "A-G"=>"X", "A--"=>"-", "GT-"=>"X", "GC-"=>"X", "GA-"=>"X", "GG-"=>"X", "G-T"=>"X", "G-C"=>"X", "G-A"=>"X", "G-G"=>"X", "G--"=>"-", "-TT"=>"X", "-TC"=>"X", "-TA"=>"X", "-TG"=>"X", "-T-"=>"X", "-CT"=>"X", "-CC"=>"X", "-CA"=>"X", "-CG"=>"X", "-C-"=>"X", "-AT"=>"X", "-AC"=>"X", "-AA"=>"X", "-AG"=>"X", "-A-"=>"X", "-GT"=>"X", "-GC"=>"X", "-GA"=>"X", "-GG"=>"X", "-G-"=>"X", "--T"=>"-", "--C"=>"-", "--A"=>"-", "--G"=>"-", "---"=>"-", "NTT"=>"X", "TNT"=>"X", "TTN"=>"X", "NTC"=>"X", "TNC"=>"X", "TCN"=>"X", "NTA"=>"X", "TNA"=>"X", "TAN"=>"X", "NTG"=>"X", "TNG"=>"X", "TGN"=>"X", "NT-"=>"X", "TN-"=>"X", "T-N"=>"X", "NTN"=>"X", "TNN"=>"X", "NCT"=>"X", "CNT"=>"X", "CTN"=>"X", "NCC"=>"X", "CNC"=>"X", "CCN"=>"X", "NCA"=>"X", "CNA"=>"X", "CAN"=>"X", "NCG"=>"X", "CNG"=>"X", "CGN"=>"X", "NC-"=>"X", "CN-"=>"X", "C-N"=>"X", "NCN"=>"X", "CNN"=>"X", "NAT"=>"X", "ANT"=>"X", "ATN"=>"X", "NAC"=>"X", "ANC"=>"X", "ACN"=>"X", "NAA"=>"X", "ANA"=>"X", "AAN"=>"X", "NAG"=>"X", "ANG"=>"X", "AGN"=>"X", "NA-"=>"X", "AN-"=>"X", "A-N"=>"X", "NAN"=>"X", "ANN"=>"X", "NGT"=>"X", "GNT"=>"X", "GTN"=>"X", "NGC"=>"X", "GNC"=>"X", "GCN"=>"X", "NGA"=>"X", "GNA"=>"X", "GAN"=>"X", "NGG"=>"X", "GNG"=>"X", "GGN"=>"X", "NG-"=>"X", "GN-"=>"X", "G-N"=>"X", "NGN"=>"X", "GNN"=>"X", "N-T"=>"X", "-NT"=>"X", "-TN"=>"X", "N-C"=>"X", "-NC"=>"X", "-CN"=>"X", "N-A"=>"X", "-NA"=>"X", "-AN"=>"X", "N-G"=>"X", "-NG"=>"X", "-GN"=>"X", "N--"=>"X", "-N-"=>"X", "--N"=>"X", "N-N"=>"X", "-NN"=>"X", "NNT"=>"X", "NNC"=>"X", "NNA"=>"X", "NNG"=>"X", "NN-"=>"X", "NNN"=>"X")                 
######################################################################################################################################
mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
mut_gene_Dict2 = Dict{String, Int}("ORF1a"=>11, "ORF1b"=>12, "S"=>1, "E"=>2, "M"=>3, "N"=>4, "ORF3a"=>5, "ORF6"=>6, "ORF7a"=>7, "ORF7b"=>8, "ORF8"=>9, "ORF9b"=>10)
##############################################################################
nuc_del_range_sortkey(n) = parse(Int, split(n, "-")[1])
###########
AA_del_num(n) = parse(Int, split(split(n, "-")[1], ":")[2])
unknown_AA_first_pos(n) = string(split(n, "-")[1])
gene___AAnum_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAnum_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAdel_sortkey2(n) = [mut_gene_Dict2[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
gene___AAposonly_sortkey(n) = [mut_gene_Dict[aa_gene_comprehensive_dict[n]], aa_pos_comprehensive_dict[n]]
unknown_AA_ranges_sort(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[unknown_AA_first_pos(n)]], aa_pos_comprehensive_dict[unknown_AA_first_pos(n)])
##############################################################################
N3_syn = ["TCT", "TCC", "TCA", "TCG", "CTT", "CTC", "CTA", "CTG", "CCT", "CCC", "CCA", "CCG", "CGT", "CGC", "CGA", "CGG", "ACT", "ACC", "ACA", "ACG", "GTT", "GTC", "GTA", "GTG", "GCT", "GCC", "GCA", "GCG", "GGT", "GGC", "GGA", "GGG"]
N3_tv = ["TTT", "TTC", "TTA", "TTG", "TAT", "TAC", "TAA", "TAG", "AAT", "AAC", "AAA", "AAG", "AGT", "AGC", "AGA", "AGG", "GAT", "GAC", "GAA", "GAG"]
##############################################################################
EPI_ISL(n) = split(n, "|")[2]
country(n) = split(n, "/")[2]
sequence_date(n) = split(n, "|")[3]
seq_lab(n) = split(n, "/")[3]
US_state(n) = split(split(n, "/")[3], "-")[1]
##############################################################################
seq_nuc_muts["EPI_ISL_6281381"] = Set{String}()
seq_nuc_muts_no_dels = Dict{String, Set{String}}()
for (seq, mut_set) in seq_nuc_muts
    seq_nuc_muts_no_dels[seq] = Set{String}()
    for mut in mut_set
        if qry_nuc_comprehensive_dict[mut] ≠ "-"
            push!(seq_nuc_muts_no_dels[seq], mut)
        end
    end
end      
#####################################################################################################################################
gene_print_array = ["S", "N", "E", "M", "ORF3a", "ORF6", "ORF7a", "ORF7b", "ORF8", "ORF9b", "ORF1a", "ORF1b"]
#####################################################################################################################################
function print_all_seq_info(seq::String, txt_out::String)
    open(txt_out, "a") do g
        println("#"^94); println(g, "#"^94)
        cntry = seq_country[seq]
        pango = seq_pango[seq]
        USA_state = seq_US_state[seq] 
        coll_date = seq_collection_date[seq]
        date_index = seq_date_index[seq]
        lab = seq_lab_dict[seq]
#################################################################################
        aa_muts = seq_AA_muts_no_dels[seq]
        aa_del_ranges = seq_AA_dels[seq]
        nuc_muts = seq_nuc_muts[seq]
        nuc_muts_no_dels = seq_nuc_muts_no_dels[seq]
        nuc_del_ranges = seq_nuc_del_ranges[seq]
        aa_muts_sort = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey(x))
        aa_muts_sort2 = sort(collect(aa_muts), by = x -> gene___AAnum_sortkey2(x))
        aa_del_ranges_sort = sort(collect(aa_del_ranges), by = x -> gene___AAdel_sortkey2(x))
        nuc_muts_sort = sort(collect(nuc_muts), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_muts_no_dels_sort = sort(collect(nuc_muts_no_dels), by = x -> nuc_mut_int_comprehensive_dict[x])
        nuc_del_ranges_sort = sort(collect(nuc_del_ranges), by = x -> nuc_del_range_sortkey(x))
        seq_mixed_AA_muts_sort = sort(collect(seq_mixed_AA_muts[seq]), by = x -> gene___AAnum_sortkey(x))
        seq_mixed_nucs_sort = sort(collect(seq_mixed_nucs[seq]), by = x -> nuc_mut_int_comprehensive_dict[x])
        seq_unknown_AA_ranges_sort = sort(collect(seq_unknown_AA_ranges[seq]), by = x -> unknown_AA_ranges_sort(x))
        total_AA_subs_plus_del_ranges = length(aa_muts) + length(aa_del_ranges)
#################################################################################
        mut_vec_dict = Dict{String, Vector{String}}()
        for gene in gene_print_array
            mut_vec_dict[gene] = Vector{String}()
        end
        for mut in aa_muts
            gene = aa_gene_comprehensive_dict[mut]
            mut_only = string(split(mut, ":")[2])
            push!(mut_vec_dict[gene], mut_only) 
        end
        mutonly_sortkey(n) = parse(Int, n[2:end-1])
        for gene in keys(mut_vec_dict)
            if !isempty(mut_vec_dict[gene])
                sort!(mut_vec_dict[gene], by = x -> mutonly_sortkey(x))
            end
        end
#################################################################################
        if seq in rep_seqs
            for (grp_num, seq_set) in rep_seq_grps_seqs
                grp_size = length(seq_set)
                if seq in seq_set
                    seq_set_sort = sort(collect(seq_set), by = x -> (length(x), x))
                    seq_set_sort_join = join(seq_set_sort, ", ")
                    seq_set_sort_join_len = length(seq_set_sort_join)
#                    title_len = length("Group #$(grp_num), $(grp_size) related seqs--")
                        println("Group #$(grp_num), $(grp_size) related sequences")
                    println(g, "Group #$(grp_num), $(grp_size) related sequences")
                    if seq_set_sort_join_len ≤ 84
                           print("       ")
                        print(g, "       ")
                           println(seq_set_sort_join)
                        println(g, seq_set_sort_join)
                    elseif seq_set_sort_join_len > 84
                        start = 0
                        for z in 1:10
                            if seq_set_sort_join_len - start > 84
                                for i in start+85:-1:start+65
                                    if string(seq_set_sort_join[i]) == " "
                                           print("       ")
                                        print(g, "       ")
                                           println(seq_set_sort_join[start+1:i-1])
                                        println(g, seq_set_sort_join[start+1:i-1])
                                        start = i
                                        break
                                    end
                                end
                            end
                            if seq_set_sort_join_len - start ≤ 84
                                   print("       ")
                                print(g, "       ")
                                   println(seq_set_sort_join[start+1:end])
                                println(g, seq_set_sort_join[start+1:end])
                                break
                            end
                        end
                    end
                end
            end     
        else
            println("Singlet")
            println(g, "Singlet")
        end
########################################################################
           println("$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
        println(g, "$(pango)|$(seq)|$(coll_date) | $(cntry)|$(USA_state)|$(lab) | AAsubs+DelRanges = $(total_AA_subs_plus_del_ranges)")
           println("----------------------------------- AA Substitutions -------------------------------------")
        println(g, "----------------------------------- AA Substitutions -------------------------------------")
################# Next bit is to figure out how many muts are in a given gene & how much print space they take up
        gene_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_mut_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_set = Set{String}()
        for mut in aa_muts_sort2
            gene = split(mut, ":")[1]
            push!(gene_set, gene)
            gene_mut_ct[gene] += 1
            gene_print_length[gene] += length(mut) + 2
        end
        for gene in gene_set
            ## +4 is for the "--", the -2 b/c the last mut doesn't have ", " after it, & the +10 for 10 spaces btwn gene muts
            gene_print_length[gene] += length(gene) + 4 - 2
        end
################ Same thing as above but for deletions
        gene_del_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_del_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_del_set = Set{String}()
        for del in aa_del_ranges_sort
            gene = split(del, ":")[1]
            push!(gene_del_set, gene)
            del_rng = split(del, ":")[2]
            gene_del_print_length[gene] += length(del_rng) + 2
        end
        for gene in gene_del_set
            gene_del_print_length[gene] += length(gene) + 4 - 2
        end
############### Same thing as above but for unknown AA
        gene_unk_print_length = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_AA_unk_ct = Dict{String, Int}("ORF1a"=>0, "ORF1b"=>0, "ORF3a"=>0, "ORF6"=>0, "ORF7a"=>0, "ORF7b"=>0, "ORF8"=>0, "ORF9b"=>0, "S"=>0, "E"=>0, "M"=>0, "N"=>0)
        gene_unk_set = Set{String}()
        for unk in seq_unknown_AA_ranges_sort
            gene = split(unk, ":")[1]
            push!(gene_unk_set, gene)
            unk_rng = split(unk, ":")[2]
            gene_unk_print_length[gene] += length(unk_rng) + 2
        end
        for gene in gene_unk_set
            gene_unk_print_length[gene] += length(gene) + 4 - 2
        end
#####################################################################################################################################
        line_vec_dict = Dict{Int, Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        line_length = 0
        previous_gene = "S"
        midline = "no"
        for v in 1:length(gene_print_array)
            gene = gene_print_array[v]
            skip = "no"
            if !isempty(mut_vec_dict[gene])
                muts_joined = join(mut_vec_dict[gene], ", ")
                muts_line_empty = "$(gene)--$(muts_joined)"
                muts_line_nonempty = "     $(gene)--$(muts_joined)"
                muts_line_empty_len = length(muts_line_empty)
                muts_line_nonempty_len = length(muts_line_nonempty)
                if length(previous_gene) < 3 && length(gene) ≥ 4
                    skip = "yes"
                end
                previous_gene = gene
                if line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "yes"
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len ≤ (90-line_length) && skip == "no"
                    print(muts_line_nonempty); print(g, muts_line_nonempty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len ≤ 90
                    println(); println(g)
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = line_length + muts_line_nonempty_len
                    midline = "yes"
                    continue
                elseif line_length == 0 && muts_line_empty_len ≤ 90
                    print(muts_line_empty); print(g, muts_line_empty)
                    line_length = muts_line_empty_len
                    midline = "yes"
                    continue
                elseif line_length > 0 && muts_line_nonempty_len > (90-line_length) && muts_line_empty_len > 90
                    println(); println(g)
                    line_length = 0
                    midline = "no"
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                elseif line_length == 0 && muts_line_empty_len > 90
                    for i in 77:91
                        if string(muts_line_empty[i]) == " "
                            println(muts_line_empty[1:i-1]); println(g, muts_line_empty[1:i-1])
                            line_vec_dict[1] = i
                            print("      "); print(g, "      ") 
                            break
                        end
                    end
                    current_pos = line_vec_dict[1]
                    for j in 2:length(line_vec_dict)
                        line = line_vec_dict[j]
                        if muts_line_empty_len - current_pos ≤ 84
                            println(muts_line_empty[current_pos+1:end]); println(g, muts_line_empty[current_pos+1:end])
                            line_length = 0
                            midline = "no"
                            break
                        end
                        if muts_line_empty_len - current_pos > 84
                            for i in current_pos+71:current_pos+85
                                if string(muts_line_empty[i]) == " "
                                    println(muts_line_empty[current_pos+1:i-1]); println(g, muts_line_empty[current_pos+1:i-1])
                                    midline = "no"
                                    if v < length(gene_print_array) && !all(isempty(mut_vec_dict[gene_print_array[q]]) for q in v+1:length(gene_print_array))
                                        print("      "); print(g, "      ")
                                    end
                                    current_pos = i
                                    break
                                end
                            end
                        end
                    end
                end
            end
        end
        if midline == "yes"
            println(); println(g)
        end
           println("------------------------------------------------------------------------------------------")
        println(g, "------------------------------------------------------------------------------------------")
##########################################################################################
#        print("\n"^1); print(g, "\n"^1)
        if !isempty(aa_del_ranges_sort)
            aadel_caps_pad = rpad("AA DELETIONS", 14)
            print("$(aadel_caps_pad)| ")
            print(g, "$(aadel_caps_pad)| ")
            line_AA_del_print_ct = 20
            for i in 1:length(aa_del_ranges_sort)
                del = aa_del_ranges_sort[i]
                gene = split(del, ":")[1]
                del2 = split(del, ":")[2]
                if length(aa_del_ranges_sort) == 1
                    println("$(gene)--$(del2)")
                    println(g, "$(gene)--$(del2)")
                end
                if length(aa_del_ranges_sort) > 1
                    if i == 1
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2), ")
                            print(g, "$(gene)--$(del2), ")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2) + 2
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            print("$(gene)--$(del2)")
                            print(g, "$(gene)--$(del2)")
                            line_AA_del_print_ct = length(gene) + 4 + length(del2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2) + 2
                            if line_AA_del_print_ct > 102 
                                println(); println(g)
                                print("          $(del2), ")
                                print(g, "          $(del2), ")
                                line_AA_del_print_ct = 10 + length(del2) + 2
                            else
                                print("$(del2), ")
                                print(g, "$(del2), ")
                            end
                        elseif mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            line_AA_del_print_ct += length(del2)
                            if line_AA_del_print_ct > 102
                                println(); println(g)
                                println("          $(del2)")
                                println(g, "          $(del2)")
                                line_AA_del_print_ct = length(del2)
                            else
                                print("$(del2)")
                                print(g, "$(del2)")
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) == mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2) + 2
                            else
                                print("          $(gene)--$(del2), ")
                                print(g, "          $(gene)--$(del2), ")
                                line_AA_del_print_ct += 10 + length(gene) + 4 + length(del2) + 2
                            end
                        elseif mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i-1]) && mut_gene(del) ≠ mut_gene(aa_del_ranges_sort[i+1])
                            if line_AA_del_print_ct + gene_del_print_length[gene]+ 10 > 102
                                println(); println(g)
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct = 10 + length(gene) + 4 + length(del2)
                            else
                                print("          $(gene)--$(del2)")
                                print(g, "          $(gene)--$(del2)")
                                line_AA_del_print_ct += length(gene) + 4 + length(del2) + 10
                            end
                        end
                    end
                    if i == length(aa_del_ranges_sort)
                        if mut_gene(del) == mut_gene(aa_del_ranges_sort[i-1])
                            println("$(del2)")
                            println(g, "$(del2)")
                        elseif line_AA_del_print_ct + gene_del_print_length[gene] > 102
                            println(); println(g)
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        else
                            println("          $(gene)--$(del2)")
                            println(g, "          $(gene)--$(del2)")
                        end
                    end
                end
            end
        end
#        println(); println(g)
##############################################################################################
##############################################################################################
        if !isempty(seq_mixed_AA_muts_sort)
            mixed_caps_pad = rpad("MIXED AA MUTS", 14)
            print("$(mixed_caps_pad)| ")
            print(g, "$(mixed_caps_pad)| ")
            mixed_print_ct = 21
            for i in 1:length(seq_mixed_AA_muts_sort)
                mix = seq_mixed_AA_muts_sort[i]
                mix1 = split(mix, ":")[1]
                mix2 = split(mix, ":")[2]
                if length(seq_mixed_AA_muts_sort) == 1
                    println("$(mix1)--$(mix2)")
                    println(g, "$(mix1)--$(mix2)")
                end
                if length(seq_mixed_AA_muts_sort) > 1
                    if i == 1
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2), ")
                            print(g, "$(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix1)--$(mix2)")
                            print(g, "$(mix1)--$(mix2)")
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("$(mix2), ")
                            print(g, "$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            print("    $(mix1)--$(mix2), ")
                            print(g, "    $(mix1)--$(mix2), ")
                        elseif aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]] && aa_gene_comprehensive_dict[mix] ≠ aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i+1]]
                            println("    $(mix1)--$(mix2)")
                            println(g, "    $(mix1)--$(mix2)")
                        end
                    end
                    if i == length(seq_mixed_AA_muts_sort)
                        if aa_gene_comprehensive_dict[mix] == aa_gene_comprehensive_dict[seq_mixed_AA_muts_sort[i-1]]
                            println("$(mix2)")
                            println(g, "$(mix2)")
                        else
                            println("    $(mix1)--$(mix2)   ")
                            println(g, "    $(mix1)--$(mix2)   ")
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(seq_unknown_AA_ranges_sort)
            aadrop_caps_pad = rpad("AA DROPOUT", 14)
            print("$(aadrop_caps_pad)| ")
            print(g, "$(aadrop_caps_pad)| ")
            line_unk_print_ct = 18
            for i in 1:length(seq_unknown_AA_ranges_sort)
                unk = seq_unknown_AA_ranges_sort[i]
                gene = split(unk, ":")[1]
                unk2 = split(unk, ":")[2]
                if length(seq_unknown_AA_ranges_sort) == 1
                    println("$(gene)--$(unk2)")
                    println(g, "$(gene)--$(unk2)")
                end
                if length(seq_unknown_AA_ranges_sort) > 1
                    if i == 1
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2), ")
                            print(g, "$(gene)--$(unk2), ")
                            line_unk_print_ct = length(gene) + 4 + length(unk2) + 2
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            print("$(gene)--$(unk2)")
                            print(g, "$(gene)--$(unk2)")
                            line_unk_print_ct = length(gene) + 4 + length(unk2)
                        end
                    end
                    if i ≠ 1 && i ≠ length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2) + 2
                            if line_unk_print_ct > 102 
                                println(); println(g)
                                print("      $(unk2), ")
                                print(g, "      $(unk2), ")
                                line_unk_print_ct = 10 + length(unk2) + 2
                            else
                                print("$(unk2), ")
                                print(g, "$(unk2), ")
                            end
                        elseif mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            line_unk_print_ct += length(unk2)
                            if line_unk_print_ct > 102
                                println(); println(g)
                                println("      $(unk2)")
                                println(g, "      $(unk2)")
                                line_unk_print_ct = 0
                            else
                                print("$(unk2)")
                                print(g, "$(unk2)")
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println();  println(g)
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2) + 2
                            else
                                print("      $(gene)--$(unk2), ")
                                print(g, "      $(gene)--$(unk2), ")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 2 + 10
                            end
                        elseif mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i-1]) && mut_gene(unk) ≠ mut_gene(seq_unknown_AA_ranges_sort[i+1])
                            if line_unk_print_ct + gene_unk_print_length[gene] + 10 > 102
                                println(); println(g)
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct = 10 + length(gene) + 4 + length(unk2)
                            else
                                print("      $(gene)--$(unk2)")
                                print(g, "      $(gene)--$(unk2)")
                                line_unk_print_ct += length(gene) + 4 + length(unk2) + 10
                            end
                        end
                    end
                    if i == length(seq_unknown_AA_ranges_sort)
                        if mut_gene(unk) == mut_gene(seq_unknown_AA_ranges_sort[i-1])
                            println("$(unk2)")
                            println(g, "$(unk2)")
                        elseif line_unk_print_ct + gene_unk_print_length[gene] > 102
                            println(); println(g)
                            println("    $(gene)--$(unk2)")
                            println(g, "    $(gene)--$(unk2)")
                        else
                            println("      $(gene)--$(unk2)")
                            println(g, "      $(gene)--$(unk2)")
                        end
                    end
                end
            end
            println(); println(g)
        end
###############################################################################################################################
        print("NUC MUTATIONS--")
        print(g, "NUC MUTATIONS--")
        nuc_muts_join = join(nuc_muts_no_dels_sort, ", ")
        nuc_mut_len = length(nuc_muts_join)
        line_length = 0
#        line1 = 0; line2 = 0; line3 = 0; line4 = 0; line5 = 0; line6 = 0; line7 = 0; line8 = 0
#        line9 = 0; line10 = 0; line11 = 0; line12 = 0; line13 = 0; line14 = 0; line15 = 0; line16 = 0
#        line_vec = [line1, line2, line3, line4, line5, line6, line7, line8, line9, line10, line11, line12, line13, line14, line15, line16]
        line_vec_dict = Dict{Int, Int}(1=>0, 2=>0, 3=>0, 4=>0, 5=>0, 6=>0, 7=>0, 8=>0, 9=>0, 10=>0, 11=>0, 12=>0, 13=>0, 14=>0, 15=>0, 16=>0)
        done = "no"
        if nuc_mut_len ≤ 75
               println(nuc_muts_join)
            println(g, nuc_muts_join)
            done = "yes"
        end
        if nuc_mut_len > 75 && done == "no"
            for j in 69:77
                if string(nuc_muts_join[j]) == " "
                       println(nuc_muts_join[1:(j-1)])
                    println(g, nuc_muts_join[1:(j-1)])
                    line_vec_dict[1] = j
                    break
                end
            end
        end
        for i in 1:length(line_vec_dict)
            line = line_vec_dict[i]
            if nuc_mut_len - line ≤ 84 && done == "no"
                   print("      ")
                print(g, "      ")
                   println(nuc_muts_join[(line+1):end])
                println(g, nuc_muts_join[(line+1):end])
                done = "yes"
                break
            end
            if nuc_mut_len - line > 84 && done == "no"
                   print("      ")
                print(g, "      ")
                for j in line+77:line+85
                    if string(nuc_muts_join[j]) == " "
                           println(nuc_muts_join[line+1:j-1])
                        println(g, nuc_muts_join[line+1:j-1])
                        if i ≠ length(line_vec_dict)
                            line_vec_dict[i+1] = j
                        end
                    end
                end
            end
        end
##############################################################################################
        if !isempty(nuc_del_ranges_sort)
            nucdel_caps = "NUC DELETIONS--"
            print("$(nucdel_caps)")
            print(g, "$(nucdel_caps)")
            print_length = 16
            for i in 1:length(nuc_del_ranges_sort)
                nuc_del = nuc_del_ranges_sort[i]
                print_length += length(nuc_del) + 2
                if print_length > 102
                    if i == 1
                        println("$(nuc_del), ")
                        println(g, "$(nuc_del), ")
                        print_length += 8
                    end
                    if i ≠ length(nuc_del_ranges_sort) && !(i == 1)
                        println("        $(nuc_del), ")
                        println(g, "        $(nuc_del), ")
                        print_length += 8
                    elseif !(i == 1)
                        println("        $(nuc_del)")
                        println(g, "        $(nuc_del)")
                        print_length += 8
                    end
                else
                    if i ≠ length(nuc_del_ranges_sort)
                        print("$(nuc_del), ")
                        print(g, "$(nuc_del), ")
                    else
                        println("$(nuc_del)")
                        println(g, "$(nuc_del)")
                    end
                end
            end
        end
        seqnucmutsnodels = collect(seq_nuc_muts_no_dels[seq])
        synonymous_nuc__AA_sort, synonymous_nuc__context_sort, noncoding__noncoding_region_sort, noncoding_nuc__context_sort = synonymous_nuc_to_AA_and_noncoding_to_context(seq_pango[seq], seqnucmutsnodels)
#        synonymous_nucs_vec = String[]
#        synonymous_AA_vec = String[]
#        synonymous_context_vec = String[]
######################################################################################################
        if !isempty(seq_mixed_nucs_sort)
            mixed_nucs_pad = rpad("MIXED NUCS", 14)
               print("$(mixed_nucs_pad)| ")
            print(g, "$(mixed_nucs_pad)| ")
            seq_mixed_nucs_sort_join = join(seq_mixed_nucs_sort, ", ")
               println(seq_mixed_nucs_sort_join)
            println(g, seq_mixed_nucs_sort_join)
            println(); println(g)
        end
##############################################################################################
############# Below = SHORT version of syn nuc print, WITHOUT AA positions and nuc context
        if !isempty(synonymous_nuc__AA_sort)
            total_syn = length(synonymous_nuc__AA_sort)
            println("SYNONYMOUS NUC MUTATIONS - Total:$(total_syn)")
            syn_nuc_mut_vec = String[]
            for i in 1:length(synonymous_nuc__AA_sort)
                synnuc = synonymous_nuc__AA_sort[i][1]
                push!(syn_nuc_mut_vec, synnuc)
            end
            syn_nuc_string = join(syn_nuc_mut_vec, ", ")
            println("     $syn_nuc_string")
############# Below = LONG version of syn nuc print, WITH AA positions & nuc context
#            for i in 1:length(synonymous_nuc__AA_sort)
#                synnuc = synonymous_nuc__AA_sort[i][1]
#                synnuc_pad = rpad(synnuc, 7)
#                synAA = synonymous_nuc__AA_sort[i][2]
#                AAlen = length(synAA)
#                pad1 = (14 - AAlen)÷2
#                pad12 = " "^pad1*synAA
#                synAA_pad = rpad(pad12, 14)
#                syncontext = synonymous_nuc__context_sort[i][2]
#                premut_context = split(syncontext, "-->")[1]
#                postmut_context = split(syncontext, "-->")[2]
#                postpad = lpad(postmut_context, 38)
#                println("$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(postpad)
#                println(g, "$(synnuc_pad)|$(synAA_pad)|$(premut_context)")
#                println(g, postpad)
#            end
            println(); println(g)
        end
######################################################################################################
        if !isempty(noncoding__noncoding_region_sort)
            nc_len = length(noncoding__noncoding_region_sort)
            println("NONCODING NUC MUTATIONS - Total:$(nc_len)")
            for i in 1:length(noncoding__noncoding_region_sort)
                nc_nuc = noncoding__noncoding_region_sort[i][1]
                nc_nuc_pad = rpad(nc_nuc, 7)
                nc_region = noncoding__noncoding_region_sort[i][2]
                nc_region_len = length(nc_region)
                ncpad1 = (13 - nc_region_len)÷2
                ncpad12 = " "^ncpad1*nc_region
                nc_region_pad2 = rpad(ncpad12, 13)
                nc_context = noncoding_nuc__context_sort[i][2]
                premut_context = split(nc_context, "-->")[1]
                postmut_context = split(nc_context, "-->")[2]
                postpad = lpad(postmut_context, 39)
                println("$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(postpad)
                println(g, "$(nc_nuc_pad)|$(nc_region_pad2)|$(premut_context)")
                println(g, postpad)
            end
        end
        println("#"^94); println(g, "#"^94)
##############################################################################################
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### function synonymous_nuc_to_AA_and_noncoding_to_context #########################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function synonymous_nuc_to_AA_and_noncoding_to_context(ref_pango::String, muts::Vector{String})
#    all_muts_sort = sort(muts, by = x -> nuc_mut_int_comprehensive_dict[x])
    ref_seq, refAA_ORF1a, refAA_ORF1b, refAA_S, refAA_ORF3a, refAA_E, refAA_M, refAA_ORF6, refAA_ORF7a, refAA_ORF7b, refAA_ORF8, refAA_N, refAA_ORF9b = get_ref_pango_nucseq_and_geneseqs(ref_pango)
#########################################################################################################################
    coding_ranges = BitSet([266:13467..., 13469:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533..., 28284:28577...])
    noncoding_ranges = BitSet([1:265..., 21556:21562..., 25385:25392..., 26221:26244..., 26473:26522..., 27192:27201..., 27388:27393..., 27888:27893..., 28260:28273..., 29534:29830...])
    coding_range_9b = BitSet([28284:28577...])
    gene_nuc_starts = Dict{Int, Int}(0=>263, 1=>13465, 2=>21560, 3=>25390, 4=>26242, 5=>26520, 6=>27199, 7=>27391, 8=>27753, 9=>27891, 10=>28271, 11=>28281)
    ref_AA_genes = Dict{Int, String}(0=>refAA_ORF1a, 1=>refAA_ORF1b, 2=>refAA_S, 3=>refAA_ORF3a, 4=>refAA_E, 5=>refAA_M, 6=>refAA_ORF6, 7=>refAA_ORF7a, 8=>refAA_ORF7b, 9=>refAA_ORF8, 10=>refAA_N, 11=>refAA_ORF9b)
    gene_strings = Dict{Int, String}(0=>"ORF1a", 1=>"ORF1b", 2=>"S", 3=>"ORF3a", 4=>"E", 5=>"M", 6=>"ORF6", 7=>"ORF7a", 8=>"ORF7b", 9=>"ORF8", 10=>"N", 11=>"ORF9b")
    nuc_gene_num = Dict{Int, Int}()
    nuc_gene_num_9b = Dict{Int, Int}()
    synonymous_nuc_to_AA_dict = Dict{String, String}()
    synonymous_nuc_to_context_dict = Dict{String, String}()
    noncoding_nuc_to_context_dict = Dict{String, String}()
    noncoding_to_noncoding_region_dict = Dict{String, String}()
    noncoding_range_dict = Dict{Vector{Int}, String}([1, 265]=>"5' UTR", [21556, 21562]=>"Spike TRS", [25385, 25392]=>"ORF3a TRS", [26221, 26234]=>"ORF3a-E UTR", [26235, 26244]=>"E TRS", [26473, 26522]=>"E-M UTR", [27192, 27201]=>"M-ORF6 UTR", [27388, 27393]=>"ORF7a TRS", [27888, 27893]=>"ORF8 TRS", [28260, 28273]=>"N/ORF9b TRS", [29534, 29830]=>"3' UTR", [29794, 29801]=>"ONM")
    noncoding_nuc_vector = Vector{String}()
#########################################################################################################################
    gene_nuc_arr = [[266:13467...], [13469:21555...], [21563:25384...], [25393:26220...], [26245:26472...], [26523:27191...], [27202:27387...], [27394:27755...], [27760:27887...], [27894:28259...], [28274:29533...], [28284:28577...]]
    for i in 1:length(gene_nuc_arr)-1
        for nucpos in gene_nuc_arr[i]
            nuc_gene_num[nucpos] = i-1
        end
    end
    for nucpos in gene_nuc_arr[end]
        nuc_gene_num_9b[nucpos] = 11
    end
    rem0_gene = [5, 8, 9, 11]
    rem1_gene = [1, 3, 4, 6, 7]
    rem2_gene = [0, 2, 10]
    rem0 = BitSet([26523:27191..., 27760:27887..., 27894:28259..., 28284:28577...])
    rem1 = BitSet([13469:21555..., 25393:26220..., 26245:26472..., 27202:27387..., 27394:27755...])
    rem2 = BitSet([266:13467..., 21563:25384..., 28274:29533...])
    rem9b = BitSet([28284:28577...])
    rem7ab = BitSet([27756:27759...])
######################################################################################################################################
    gene_num(nuc_mut) = nuc_gene_num[nuc_mut_int_comprehensive_dict[nuc_mut]]                                           ## Fx ##
    nuc_to_AA_pos(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - gene_nuc_starts[gene_num(nuc_mut)])÷3)   ## Fx ##
    nuc_to_AA_pos_9b(nuc_mut) = string((nuc_mut_int_comprehensive_dict[nuc_mut] - 28281)÷3)                             ## Fx ##
    nuc2AA_ORF1a(nuc_mut,refAA,qryAA) = "$(gene_strings[gene_num(nuc_mut)]):$(refAA*nuc_to_AA_pos(nuc_mut))$qryAA" ### FUNCTION #####################
    nuc2AA_ORF9b(nuc_mut, refAA, qryAA) = "ORF9b:$(refAA)$(nuc_to_AA_pos_9b(nuc_mut))$(qryAA)"   ## Fx ##
######################################################################################################################################
    nuc_codon_pos_dict = Dict{Int, Int}()
    for nucpos in coding_ranges
        gene_number = nuc_gene_num[nucpos]
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict[nucpos] = codon_num
    end
    nuc_codon_pos_dict_9b = Dict{Int, Int}()
    for nucpos in coding_range_9b
        gene_number = 11
        gene_start = gene_nuc_starts[gene_number]
        codon_num = (nucpos-gene_start)%3 + 1
        nuc_codon_pos_dict_9b[nucpos] = codon_num
    end
#########################################################################
#    muts_v2 = Vector{String}()
    for nuc_mut in muts
        mut = mixed2nuc(nuc_mut)
        if ',' in mut
            mut1 = string(split(mut, ", ")[1])
            mut2 = string(split(mut, ", ")[2])
            push!(muts, mut1)
            push!(muts, mut2)
            filter!(x -> !(length(x)>9), muts)
        end
    end
######################################################################### 
    coding_ranges = BitSet([266:13467..., 13468:21555..., 21563:25384..., 25393:26220..., 26245:26472..., 26523:27191..., 27202:27387..., 27394:27755..., 27760:27887..., 27894:28259..., 28274:29533...])
    N_9b_synonymous = Set(["C28379A", "C28394A", "T28406C", "C28475A", "C28535A", "A28547C"])
    AA_mut_set = Set{String}()
    AA_mut = ""
    for nuc_mut in muts
        pos = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos in coding_ranges
#            mut = mixed2nuc(nuc_mut)  
            mut = nuc_mut
            gene_number = gene_num(mut)
            ref_triple = ""
            qry_triple = ""
            ref_triple_context = ""
            qry_triple_context = ""
            ref2qry_context = ""
            if nuc_codon_pos_dict[pos] == 1
                ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])  *string(ref_seq[pos+2])
                qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*string(ref_seq[pos+2])
                ref_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
                qry_triple_context = string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])*string(ref_seq[pos+8])
            elseif nuc_codon_pos_dict[pos] == 2
                ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]  *string(ref_seq[pos+1])
                qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                ref_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
                qry_triple_context = string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])*string(ref_seq[pos+7])
            elseif nuc_codon_pos_dict[pos] == 3
                ref_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                qry_triple = string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                ref_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
                qry_triple_context = string(ref_seq[pos-8])*string(ref_seq[pos-7])*string(ref_seq[pos-6])*string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])*string(ref_seq[pos+6])
            end

            refAA = AA_triplets[ref_triple]
            qryAA = AA_triplets[qry_triple]
            AA_mut = nuc2AA_ORF1a(mut, refAA, qryAA)
            ref2qry_context = ref_triple_context*"-->"*qry_triple_context
            if refAA == qryAA && !(pos in rem9b)
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            elseif refAA == qryAA && pos in rem9b && mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut] = AA_mut
                synonymous_nuc_to_context_dict[mut] = ref2qry_context
            else
                push!(AA_mut_set, AA_mut)
            end
###################################
            for nuc_mut2 in muts
                mut2 = nuc_mut2
                pos2 = nuc_mut_int_comprehensive_dict[mut2]
                if pos2 in coding_ranges
                    gene_number2 = gene_num(mut2)
                    if mut ≠ mut2 && gene_number == gene_number2 && nuc_to_AA_pos(mut) == nuc_to_AA_pos(mut2)
                        if nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            qry_triple = qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos+2])
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 1 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*ref_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                            qry_triple_context = string(ref_seq[pos-3])*string(ref_seq[pos-2])*string(ref_seq[pos-1])*qry_triple*string(ref_seq[pos+3])*string(ref_seq[pos+4])*string(ref_seq[pos+5])
                        elseif nuc_codon_pos_dict[pos] == 2 && nuc_codon_pos_dict[pos2] == 3
                            ref_triple = string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]*ref_nuc_comprehensive_dict[mut2]
                            qry_triple = string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]*qry_nuc_comprehensive_dict[mut2]
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 2
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]*string(ref_seq[pos+1])
                            ref_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*ref_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                            qry_triple_context = string(ref_seq[pos-4])*string(ref_seq[pos-3])*string(ref_seq[pos-2])*qry_triple*string(ref_seq[pos+2])*string(ref_seq[pos+3])*string(ref_seq[pos+4])
                        elseif nuc_codon_pos_dict[pos2] == 1 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = ref_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*ref_nuc_comprehensive_dict[mut]
                            qry_triple = qry_nuc_comprehensive_dict[mut2]*string(ref_seq[pos-1])*qry_nuc_comprehensive_dict[mut]
                            ref_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*ref_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                            qry_triple_context = string(ref_seq[pos-5])*string(ref_seq[pos-4])*string(ref_seq[pos-3])*qry_triple*string(ref_seq[pos+1])*string(ref_seq[pos+2])*string(ref_seq[pos+3])
                        elseif nuc_codon_pos_dict[pos2] == 2 && nuc_codon_pos_dict[pos] == 3
                            ref_triple = string(ref_seq[pos-2])*ref_nuc_comprehensive_dict[mut2]*ref_nuc_comprehensive_dict[mut]
                            qry_triple = string(ref_seq[pos-2])*qry_nuc_comprehensive_dict[mut2]*qry_nuc_comprehensive_dict[mut]
                        end
                        refAA2 = AA_triplets[ref_triple]
                        qryAA2 = AA_triplets[qry_triple]
                        AA_mut2 = nuc2AA_ORF1a(mut2, refAA2, qryAA2)
                        ref2qry_context = ref_triple_context*"-->"*qry_triple_context
                        if refAA2 == qryAA2 && !(pos2 in rem9b)
                            synonymous_nuc_to_AA_dict[mut2] = AA_mut2
                            synonymous_nuc_to_context_dict[mut2] = ref2qry_context
                        else
                            push!(AA_mut_set, AA_mut2)
                        end
                    end
                end
            end
        else
            npos = nuc_mut_int_comprehensive_dict[nuc_mut]
            if npos < 29890
                qry_nuc = qry_nuc_comprehensive_dict[nuc_mut]
                ref_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*string(ref_seq[npos])*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                qry_nc_nuc_context = string(ref_seq[npos-8])*string(ref_seq[npos-7])*string(ref_seq[npos-6])*string(ref_seq[npos-5])*string(ref_seq[npos-4])*string(ref_seq[npos-3])*string(ref_seq[npos-2])*string(ref_seq[npos-1])*qry_nuc*string(ref_seq[npos+1])*string(ref_seq[npos+2])*string(ref_seq[npos+3])*string(ref_seq[npos+4])*string(ref_seq[npos+5])*string(ref_seq[npos+6])*string(ref_seq[npos+7])*string(ref_seq[npos+8])
                full_nc_context = ref_nc_nuc_context*"-->"*qry_nc_nuc_context
                noncoding_nuc_to_context_dict[nuc_mut] = full_nc_context
                for (start_end, place) in noncoding_range_dict
                    frst = start_end[1]
                    last = start_end[2]
                    if npos ≥ frst && npos ≤ last
                        noncoding_to_noncoding_region_dict[nuc_mut] = place
                        mut_vec = [nuc_mut, place]
                        push!(noncoding_nuc_vector, nuc_mut)
                    end
                end
            end
        end
    end
#########################################################################################################
    for nuc_mut in muts
        pos_9b = nuc_mut_int_comprehensive_dict[nuc_mut]
        if pos_9b in rem9b
#            mut_9b = mixed2nuc(nuc_mut)
            mut_9b = nuc_mut
            pos_9b = nuc_mut_int_comprehensive_dict[mut_9b]   
            gene_number_9b = 11
            ref_triple_9b = ""
            qry_triple_9b = ""
            ref_triple_context_9b = ""
            qry_triple_context_9b = ""
            ref2qry_context_9b = ""
            if nuc_codon_pos_dict_9b[pos_9b] == 1
                ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])
                ref_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
                qry_triple_context_9b = string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])*string(ref_seq[pos_9b+8])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 2
                ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]  *string(ref_seq[pos_9b+1])
                qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                ref_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
                qry_triple_context_9b = string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])*string(ref_seq[pos_9b+7])
            elseif nuc_codon_pos_dict_9b[pos_9b] == 3
                ref_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                qry_triple_9b = string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                ref_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
                qry_triple_context_9b = string(ref_seq[pos_9b-8])*string(ref_seq[pos_9b-7])*string(ref_seq[pos_9b-6])*string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])*string(ref_seq[pos_9b+6])
            end
            refAA_9b = AA_triplets[ref_triple_9b]
            qryAA_9b = AA_triplets[qry_triple_9b]
            AA_mut_9b = nuc2AA_ORF9b(mut_9b, refAA_9b, qryAA_9b)
            ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
            if refAA_9b == qryAA_9b && nuc_mut in N_9b_synonymous
                synonymous_nuc_to_AA_dict[mut_9b] = AA_mut_9b
                synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
            else
                push!(AA_mut_set, AA_mut_9b)
            end
#############################################################################
            for nuc_mut2_9b in muts
#                mut2_9b = mixed2nuc(nuc_mut2_9b)
                mut2_9b = nuc_mut2_9b
                pos2_9b = nuc_mut_int_comprehensive_dict[mut2_9b]
                if pos2_9b in rem9b
                    gene_number2_9b = 11
                    if mut_9b ≠ mut2_9b && nuc_to_AA_pos_9b(mut_9b) == nuc_to_AA_pos_9b(mut2_9b)
                        if nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b+2])
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 1 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*ref_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                            qry_triple_context_9b = string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*string(ref_seq[pos_9b-1])*qry_triple_9b*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])*string(ref_seq[pos_9b+5])
                        elseif nuc_codon_pos_dict_9b[pos_9b] == 2 && nuc_codon_pos_dict_9b[pos2_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]*ref_nuc_comprehensive_dict[mut2_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]*qry_nuc_comprehensive_dict[mut2_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 2
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]*string(ref_seq[pos_9b+1])
                            ref_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*ref_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                            qry_triple_context_9b = string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*string(ref_seq[pos_9b-2])*qry_triple_9b*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])*string(ref_seq[pos_9b+4])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 1 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = ref_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = qry_nuc_comprehensive_dict[mut2_9b]*string(ref_seq[pos_9b-1])*qry_nuc_comprehensive_dict[mut_9b]
                            ref_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*ref_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                            qry_triple_context_9b = string(ref_seq[pos_9b-5])*string(ref_seq[pos_9b-4])*string(ref_seq[pos_9b-3])*qry_triple_9b*string(ref_seq[pos_9b+1])*string(ref_seq[pos_9b+2])*string(ref_seq[pos_9b+3])
                        elseif nuc_codon_pos_dict_9b[pos2_9b] == 2 && nuc_codon_pos_dict_9b[pos_9b] == 3
                            ref_triple_9b = string(ref_seq[pos_9b-2])*ref_nuc_comprehensive_dict[mut2_9b]*ref_nuc_comprehensive_dict[mut_9b]
                            qry_triple_9b = string(ref_seq[pos_9b-2])*qry_nuc_comprehensive_dict[mut2_9b]*qry_nuc_comprehensive_dict[mut_9b]
                        end
                        refAA2_9b = AA_triplets[ref_triple_9b]
                        qryAA2_9b = AA_triplets[qry_triple_9b]
                        AA_mut2_9b = nuc2AA_ORF9b(mut2_9b, refAA2_9b, qryAA2_9b)
                        ref2qry_context_9b = ref_triple_context_9b*"-->"*qry_triple_context_9b
                        if refAA2_9b == qryAA2_9b && nuc_mut2_9b in N_9b_synonymous 
                            synonymous_nuc_to_AA_dict[mut2_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut2_9b] = ref2qry_context_9b
                            synonymous_nuc_to_AA_dict[mut_9b] = AA_mut2_9b
                            synonymous_nuc_to_context_dict[mut_9b] = ref2qry_context_9b
                        else
                            push!(AA_mut_set, AA_mut2_9b)
                        end
                    end
                end
            end
        end
    end
#####################################################################################################################################
#    for mut in AA_mut_set
#        if !(mut in aa_mut_all_and_chr_set)
#            push!(missing_gene_pos_AA_keys, mut)
#        end
#    end
#    AA_sort = sort(collect(AA_mut_set), by = x -> AA_order_key(x))
#    AA_sort2 = Vector{String}()
#    for mut in AA_sort
#        refAA = refAA_comprehensive_dict[mut]
#        qryAA = qryAA_comprehensive_dict[mut]
#        if !(refAA == qryAA)
#            push!(AA_sort2, mut)
#        end
#    end
#####################################################################################################################################
    synonymous_nuc_to_AA_dict_sort = sort(collect(synonymous_nuc_to_AA_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    synonymous_nuc_sort = sort(collect(keys(synonymous_nuc_to_AA_dict)), by = x -> nuc_mut_int_comprehensive_dict[x])
    synonymous_AA_sort = sort(collect(values(synonymous_nuc_to_AA_dict)), by = x -> AA_gene_sortKey_2(x))
    synonymous_nuc_to_context_dict_sort = sort(collect(synonymous_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_to_context_dict_sort = sort(collect(noncoding_nuc_to_context_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
    noncoding_nuc_vector_sort = sort(collect(noncoding_nuc_vector), by = x -> nuc_mut_int_comprehensive_dict[x])
    noncoding_to_noncoding_region_dict_sort = sort(collect(noncoding_to_noncoding_region_dict), by = x -> nuc_mut_int_comprehensive_dict[x[1]])
############################################################
#    println("################## Noncoding Nuc Mutations ##################")
#    noncoding_nuc_total = length(keys(noncoding_nuc_to_context_dict))
#    println("    Total Number of Noncoding Nucs = $(noncoding_nuc_total)")
#    for nc_mut in noncoding_nuc_to_context_dict_sort
#        print("     $(nc_mut[1]) ($(nc_mut[2])), ")
#    end
#    println()
#    for i in 1:length(noncoding_nuc_to_context_dict_sort)
#        println("$(noncoding_nuc_to_context_dict_sort[i][1]) | $(noncoding_nuc_to_context_dict_sort[i][2])")
#        premut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[1]
#        postmut_nucseq = split(noncoding_nuc_to_context_dict_sort[i][2], "|")[2]
#        println("    $(premut_nucseq)")
#        println("    $(postmut_nucseq)")
#    end
#    println("################# Synonymous Nuc Mutations #################")
#    synonymous_nuc_total = length(synonymous_nuc_sort)
#    println("  Total Number of Synonymous Nuc Muts = $(synonymous_nuc_total)")
#    synonymous_nuc_sort_join = join(synonymous_nuc_sort, ", ")
#    println("  ", synonymous_nuc_sort_join)
#    print("\n"^1)
#    for i in 1:length(synonymous_nuc_sort)
#        premut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[1]
#        postmut_nucseq = split(synonymous_nuc_to_context_dict_sort[i][2], "-->")[2]
#        postmut_cntxt_pad = lpad(postmut_nucseq, 13)
#        nucpad = rpad(synonymous_nuc_to_AA_dict_sort[i][1], 7)
#        println("     $(nucpad)|$(premut_nucseq)")
#        println("$(postmut_cntxt_pad)")
#    end
#    print("\n"^1)
#synonymous_nuc_to_context_dict[mut] = ref_triple_context*"-->"*qry_triple_context
    return synonymous_nuc_to_AA_dict_sort, synonymous_nuc_to_context_dict_sort, noncoding_to_noncoding_region_dict_sort, noncoding_nuc_to_context_dict_sort
end
#############################################################################

2026_03_06__832AM
2026_03_06__832AM


synonymous_nuc_to_AA_and_noncoding_to_context (generic function with 1 method)

In [170]:
### Calculates and makes dictionaries for the density of mutations in EPCI and HQCS datasets and the ratios between the two for each gene/protein; used to make plots
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start_time_post_load_cell = time()
######################################################################################################################################
gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
domain_order = Dict{String, Int}("NSP3_Ubl1"=>1, "NSP3_HVR"=>2, "NSP3_Mac1"=>3, "NSP3_Mac2"=>4, "NSP3_Mac3"=>5, "NSP3_DPUP"=>6, "NSP3_Ubl2"=>7, "NSP3_PLpro"=>8, "NSP3_NAB"=>9, "NSP3_BSM"=>10, "NSP3_TM1"=>11, "NSP3_Ecto3"=>12, "NSP3_TM234HLX"=>13, "NSP3_Y1"=>14, "NSP3_CoVY"=>15, "NSP4_TM1"=>16, "NSP4_Ecto4"=>17, "NSP4_TM2_TM6"=>18, "NSP4_CTD"=>19, "NSP6_AmphHlx"=>20, "NSP6_MAE"=>21, "NSP6_cyto_CTD"=>22, "NSP12_NiRAN"=>23, "NSP12_intrfce"=>24, "NSP12_fingers"=>25, "NSP12_palm"=>26, "NSP12_palmLnk"=>27, "NSP12_thumb"=>28, "NSP13_ZBD"=>29, "NSP13_stalk"=>30, "NSP13_1B"=>31, "NSP13_RecA1"=>32, "NSP13_RecA2"=>33, "NSP14_nsp10"=>34, "NSP14_EXON"=>35, "NSP14_hinge1"=>36, "NSP14_hinge2"=>37, "NSP14_N7MTase"=>38, "NSP15_NTD"=>39, "NSP15_MD"=>40, "NSP15_endoU"=>41, "S_S1"=>42, "S_S2"=>43, "S_NTD"=>44, "S_N2R"=>45, "S_RBD"=>46, "S_RBM"=>47, "S_SD1"=>48, "S_SD2"=>49, "S_630_loop"=>50, "S_FCS_region"=>51, "S_Beta1"=>52, "S_3H"=>53, "S_IL770"=>54, "S_FPPR"=>55, "S_FP"=>56, "S_HR1"=>57, "S_CH"=>58, "S_CD"=>59, "S_Beta2"=>60, "S_2turnHelix"=>61, "S_HR2"=>62, "S_TM"=>63, "S_CT"=>64, "ORF3a_SignalP"=>65, "ORF3a_NTD"=>66, "ORF3a_TM1"=>67, "ORF3a_TM12Lnk"=>68, "ORF3a_TM2"=>69, "ORF3a_TM3"=>70, "ORF3a_cytosl1"=>71, "ORF3a_Loop"=>72, "ORF3a_3DB"=>73, "ORF3a_CTD"=>74, "E_TM"=>75, "E_cytosol"=>76, "E_CTD"=>77, "N_N1"=>78, "N_N2"=>79, "N_N3"=>80, "N_N4"=>81, "N_N5"=>82, "N_SR"=>83, "N_L_helix"=>84, "N_CBP"=>85, "N_9b_overlap"=>86)    
######################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#                                        Gene      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
######################################################################################################################################
##### Filling all types of NSP muts in all seqs--both circulating & chronic--for DataFrame purposes (to make # of rows the same) #####
######################################################################################################################################
sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                           Gene      AAsite sub_type_ct
sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
for gene in gene_array
    sub_types_at_every_site_combined[gene] = Dict{Int, Set{String}}()
    sub_types_at_every_site_combined_ct[gene] = Dict{Int, Int}()
end
gene_AA_lengths = Dict{String, Int}()
for (gene, AAseq) in gene_AA_dict
    gene_AA_lengths[gene] = length(AAseq)
end
for gene in gene_array
    gene_len = gene_AA_lengths[gene]
    for i in 1:gene_len
        sub_types_at_every_site_combined[gene][i] = Set{String}()
        sub_types_at_every_site_combined_ct[gene][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = aa_gene_comprehensive_dict[sub]
    pos = aa_pos_comprehensive_dict[sub]
    mut_type = refAA_comprehensive_dict[sub]*qryAA_comprehensive_dict[sub]
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for sub in keys(AA_muts_ct_no_dels)
    gene = aa_gene_comprehensive_dict[sub]
    pos = aa_pos_comprehensive_dict[sub]
    mut_type = refAA_comprehensive_dict[sub]*qryAA_comprehensive_dict[sub]
    push!(sub_types_at_every_site_combined[gene][pos], mut_type)
end
for gene in gene_array
    for (AApos, mut_type_set) in sub_types_at_every_site_combined[gene]
        sub_type_ct = length(mut_type_set)
        sub_types_at_every_site_combined_ct[gene][AApos] = sub_type_ct
    end
end
###########################################################################################################################################################################
##################### Filling all types of NSP muts in all seqs--both circulating & chronic--for DataFrame purposes (to make # of rows the same) ##########################
###########################################################################################################################################################################
#                                             NSP      AAsite  all_sub_types_at_AAsite (e.g., AV, AT, TA, etc)
NSP_sub_types_at_every_site_combined = Dict{String, Dict{Int, Set{String}}}()
#                                               NSP      AAsite sub_type_ct
NSP_sub_types_at_every_site_combined_ct = Dict{String, Dict{Int, Int}}()
NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
for NSP in NSP_array
    NSP_sub_types_at_every_site_combined[NSP] = Dict{Int, Set{String}}()
    NSP_sub_types_at_every_site_combined_ct[NSP] = Dict{Int, Int}()
end
for NSP in NSP_array
    NSP_len = NSP_AA_size[NSP]
    for i in 1:NSP_len
        NSP_sub_types_at_every_site_combined[NSP][i] = Set{String}()
        NSP_sub_types_at_every_site_combined_ct[NSP][i] = 0
    end
end
for sub in keys(AA_muts_ct_no_dels_all)
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            sub_type = NSP_ref_AA_dict[NSP_sub]*NSP_qry_AA_dict[NSP_sub]
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for sub in keys(AA_muts_ct_no_dels)
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_pos = NSP_muts_pos_dict[NSP_sub]
            sub_type = NSP_ref_AA_dict[NSP_sub]*NSP_qry_AA_dict[NSP_sub]
            push!(NSP_sub_types_at_every_site_combined[NSP][NSP_pos], sub_type)
        end
    end
end
for NSP in NSP_array
    for (NSP_pos, mut_type_set) in NSP_sub_types_at_every_site_combined[NSP]
        sub_type_ct = length(mut_type_set)
        NSP_sub_types_at_every_site_combined_ct[NSP][NSP_pos] = sub_type_ct
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
counted_seq_total_private_AA_subs = Dict{String, Int}()
total_private_AA_subs_all_counted_chronics = 0
total_counted_sequences = 0
for name in all_unique_chr_seqs_set
    total_counted_sequences += 1
    total_AA = length(seq_AA_muts_no_dels[name])
    total_private_AA_subs_all_counted_chronics += total_AA
    counted_seq_total_private_AA_subs[name] = total_AA
end 
global chronic_avg_AA_subs_per_seq = total_private_AA_subs_all_counted_chronics/total_counted_sequences
chronic_avg_AA_subs_per_seq_rd = round(digits=2, chronic_avg_AA_subs_per_seq)
avg_private_AA_per_circ_seq_rd = round(digits=3, avg_private_AA_per_circ_seq)
println()
println("Avg AA Subs per Chronic Sequence = ", chronic_avg_AA_subs_per_seq_rd)
println("Avg AA Subs per Circulating Sequence = ", avg_private_AA_per_circ_seq_rd)
date = Dates.format(today(), "yyyy_mm_dd")
fx_name = "chr_to_circ_mut_density_ratios"
chr_to_circ_gene_density_ratio = Dict{String, Float64}()
chr_to_circ_domain_density_ratio = Dict{String, Float64}()
# The adjusted dict below adjusts for the greater number of AA muts per chronic seq compared to circ seq
adj_factor = avg_private_AA_per_circ_seq/chronic_avg_AA_subs_per_seq
adj_factor_rd = round(digits=4, adj_factor)

simple_chr_to_circ_adj_factor = total_AA_subs_all/total_chr_AA_subs
simple_chr_to_circ_adj_factor_rd = round(digits=2, simple_chr_to_circ_adj_factor)

println("simple_chr_to_circ_adj_factor = $(simple_chr_to_circ_adj_factor_rd)")
println("Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = $(adj_factor_rd)")
############################################################################################################################################################################ 
############################################################################################################################################################################
tot_singlets = length(non_rep_seqs)
tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
tot_chronics = tot_singlets + tot_groups
tot_all = qualifying_seq_ct_all
total_independent_DQ_qualifying_chronic_seqs = length(all_unique_chr_seqs)
circ_to_chronic_seq_total_ratio = qualifying_seq_ct_all/tot_chronics
combined_adjustment_factor = circ_to_chronic_seq_total_ratio*adj_factor
combined_adjustment_factor_check = (avg_private_AA_per_circ_seq*qualifying_seq_ct_all)/(chronic_avg_AA_subs_per_seq*tot_chronics)
combined_adjustment_factor_rd = round(digits=3, combined_adjustment_factor)
combined_adjustment_factor_check_rd = round(digits=3, combined_adjustment_factor_check)
println()
print("Total Singlets = $(tot_singlets) | ")
print("Total Groups = $(tot_groups) | ")
println("Total Independent Chronics = $(tot_chronics)")
println("Total Qualifying Circulating Sequences = $(qualifying_seq_ct_all)")
circ_to_chronic_seq_total_ratio_rd = round(digits=2, circ_to_chronic_seq_total_ratio)
println("(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = $(circ_to_chronic_seq_total_ratio_rd)")
print("Combined Adjustment Factor = $(combined_adjustment_factor_rd)  |  ")
println("Combined Adjustment Factor Check = $(combined_adjustment_factor_check_rd)")
#####################################################################################################################################
########### Note: the 1/1000 factor below is to make up for the 1000 factor that's used in the current ALL_SEQS function ############
#####################################################################################################################################
for (gene, chr_density) in gene_mut_density
    all_density = gene_mut_density_all[gene]
    chr_circ_ratio = (chr_density/all_density)*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_gene_density_ratio[gene] = ratio_rd
end
for (domain, chr_density) in domain_mut_density
    all_density = domain_mut_density_all[domain]
    chr_circ_ratio = chr_density/all_density*simple_chr_to_circ_adj_factor
    ratio_rd = round(digits=2, chr_circ_ratio)
    chr_to_circ_domain_density_ratio[domain] = ratio_rd
end
######################################################################################################################################
chr_to_circ_gene_density_ratio_sort_by_gene = sort(collect(chr_to_circ_gene_density_ratio), by = x -> gene_protein_order[x[1]])
chr_to_circ_gene_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_gene_density_ratio), by = x -> x[2], rev=true)
chr_to_circ_domain_density_ratio_sort_by_domain = sort(collect(chr_to_circ_domain_density_ratio), by = x -> domain_order[x[1]])
chr_to_circ_domain_density_ratio_sort_by_ratio = sort(collect(chr_to_circ_domain_density_ratio), by = x -> x[2], rev=true)
######################################################################################################################################
save("$(date)_chr_to_circ_gene_density_ratio.jld2", "chr_to_circ_gene_density_ratio", chr_to_circ_gene_density_ratio)
save("$(date)_chr_to_circ_domain_density_ratio.jld2", "chr_to_circ_domain_density_ratio", chr_to_circ_domain_density_ratio)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_gene.jld2", chr_to_circ_gene_density_ratio_sort_by_gene)
@save("$(date)_chr_to_circ_gene_density_ratio_sort_by_ratio.jld2", chr_to_circ_gene_density_ratio_sort_by_ratio)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_domain.jld2", chr_to_circ_domain_density_ratio_sort_by_domain)
@save("$(date)_chr_to_circ_domain_density_ratio_sort_by_ratio.jld2", chr_to_circ_domain_density_ratio_sort_by_ratio)
######################################################################################################################################
domain_residues_NSP = Dict{String, String}("NSP3_Ubl1"=>"NSP3:1-111", "NSP3_HVR"=>"NSP3:112-206", "NSP3_Mac1"=>"NSP3:207-374", "NSP3_Mac2"=>"NSP3:375-551", "NSP3_Mac3"=>"NSP3:552-673", "NSP3_DPUP"=>"NSP3:674-743", "NSP3_Ubl2"=>"NSP3:744-803", "NSP3_PLpro"=>"NSP3:804-1052", "NSP3_NAB"=>"NSP3:1053-1203", "NSP3_BSM"=>"NSP3:1204-1334", "NSP3_TM1"=>"NSP3:1335-1440", "NSP3_Ecto3"=>"NSP3:1441-1499", "NSP3_TM234HLX"=>"NSP3:1500-1586", "NSP3_Y1"=>"NSP3:1587-1764", "NSP3_CoVY"=>"NSP3:1765-1945", "NSP4_TM1"=>"NSP4:1-31", "NSP4_Ecto4"=>"NSP4:32-271", "NSP4_TM2_TM6"=>"NSP4:272-410", "NSP4_CTD"=>"NSP4:411-500", "NSP6_AmphHlx"=>"NSP6:219-236", "NSP6_MAE"=>"NSP6:237-276", "NSP6_cyto_CTD"=>"NSP6:277-290", "NSP12_NiRAN"=>"NSP12:1-250", "NSP12_intrfce"=>"NSP12:251-398", "NSP12_fingers"=>"NSP12:399-581", "NSP12_palm"=>"NSP12:582-627", "NSP12_palmLnk"=>"NSP12:628-687", "NSP12_palm2"=>"NSP12:688-812", "NSP12_thumb"=>"NSP12:813-932", "NSP13_ZBD"=>"NSP13:1-101", "NSP13_stalk"=>"NSP13:102-150", "NSP13_1B"=>"NSP13:151-226", "NSP13_RecA1"=>"NSP13:261-442", "NSP13_RecA2"=>"NSP13:443-601", "NSP14_nsp10"=>"NSP14:1-58", "NSP14_EXON"=>"NSP14:67-282 ", "NSP14_hinge1"=>"NSP14:286-300", "NSP14_hinge2"=>"NSP14:411-434", "NSP14_N7MTase"=>"NSP14:302-527", "NSP15_NTD"=>"NSP15:1-64", "NSP15_MD"=>"NSP15:65-182", "NSP15_endoU"=>"NSP15:207-347", "S_S1"=>"S:2-686", "S_S2"=>"S:687-1273", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_N3"=>"N:176-246", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419", "N_SR"=>"N:176-206", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_9b_overlap"=>"N:4-101")
domain_residues_ORF = Dict{String, String}("ORF3a_SignalP"=>"ORF3a:1-14", "ORF3a_NTD"=>"ORF3a:15-33", "ORF3a_TM1"=>"ORF3a:34-56", "ORF3a_TM12Lnk"=>"ORF3a:57-76", "ORF3a_TM2"=>"ORF3a:77-99", "ORF3a_TM3"=>"ORF3a:103-125", "ORF3a_cytosl1"=>"ORF3a:126-169", "ORF3a_Loop"=>"ORF3a:170-183", "ORF3a_3DB"=>"ORF3a:170-222", "ORF3a_CTD"=>"ORF3a:223-275", "NSP3_Ubl1"=>"ORF1a:819-929", "NSP3_HVR"=>"ORF1a:930-1024", "NSP3_Mac1"=>"ORF1a:1025-1192", "NSP3_Mac2"=>"ORF1a:1193-1369", "NSP3_Mac3"=>"ORF1a:1370-1491", "NSP3_DPUP"=>"ORF1a:1492-1561", "NSP3_Ubl2"=>"ORF1a:1562-1621", "NSP3_PLpro"=>"ORF1a:1622-1870", "NSP3_NAB"=>"ORF1a:1871-2021", "NSP3_BSM"=>"ORF1a:2022-2152", "NSP3_TM1"=>"ORF1a:2153-2258", "NSP3_Ecto3"=>"ORF1a:2259-2317", "NSP3_TM234HLX"=>"ORF1a:2318-2404", "NSP3_Y1"=>"ORF1a:2405-2582", "NSP3_CoVY"=>"ORF1a:2583-2763", "NSP4_TM1"=>"ORF1a:2764-2794", "NSP4_Ecto4"=>"ORF1a:2795-3034", "NSP4_TM2_TM6"=>"ORF1a:3035-3173", "NSP4_CTD"=>"ORF1a:3174-3263", "NSP6_AmphHlx"=>"ORF1a:3788-3805", "NSP6_MAE"=>"ORF1a:3806-3845", "NSP6_cyto_CTD"=>"ORF1a:3846-3859", "NSP12_NiRAN"=>"1a:4393-1b:241", "NSP12_intrfce"=>"ORF1b:242-389", "NSP12_fingers"=>"ORF1b:390-572", "NSP12_palm"=>"ORF1b:573-618", "NSP12_palmLnk"=>"ORF1b:619-678", "NSP12_thumb"=>"ORF1b:804-923", "NSP13_ZBD"=>"ORF1b:924-1024", "NSP13_stalk"=>"ORF1b:1025-1073", "NSP13_1B"=>"ORF1b:1074-1149", "NSP13_RecA1"=>"ORF1b:1184-1365", "NSP13_RecA2"=>"ORF1b:1366-1524", "NSP14_nsp10"=>"ORF1b:1525-1582", "NSP14_EXON"=>"ORF1b:1591-1806", "NSP14_hinge1"=>"ORF1b:1810-1824", "NSP14_hinge2"=>"ORF1b:1935-1958", "NSP14_N7MTase"=>"ORF1b:1826-2051", "NSP15_NTD"=>"ORF1b:2052-2115", "NSP15_MD"=>"ORF1b:2116-2233", "NSP15_endoU"=>"ORF1b:2258-2398", "S_S1"=>"S:2-686", "S_NTD"=>"S:2-305", "S_N2R"=>"S:306-334", "S_RBD"=>"S:335-528", "S_RBM"=>"S:438-506", "S_SD1"=>"S:529-591", "S_SD2"=>"S:592-674", "S_630_loop"=>"S:619-642", "S_FCS_region"=>"S:675-691", "S_S2"=>"S:687-1273", "S_Beta1"=>"S:718-729", "S_3H"=>"S:747-769", "S_IL770"=>"S:770-831", "S_FPPR"=>"S:832-866", "S_FP"=>"S:867-909", "S_HR1"=>"S:910-985", "S_CH"=>"S:986-1035", "S_CD"=>"S:1036-1068", "S_Beta2"=>"S:1127-1135", "S_2turnHelix"=>"S:1148-1155", "S_HR2"=>"S:1164-1211", "S_TM"=>"S:1212-1234", "S_CT"=>"S:1235-1273", "E_TM"=>"E:8-38", "E_cytosol"=>"E:54-65", "E_CTD"=>"E:69-75", "N_9b_overlap"=>"N:4-101", "N_N1"=>"N:2-44", "N_N2"=>"N:45-175", "N_SR"=>"N:176-206", "N_N3"=>"N:176-246", "N_L_helix"=>"N:215-235", "N_CBP"=>"N:247-279", "N_N4"=>"N:247-364", "N_N5"=>"N:365-419" )
######################################################################################################################################
function NSP_to_ORF(domain::String, NSP::Int)
    ORFadd = NSP1ab_add[NSP]
    first1 = minimum(domain_residues_NSP_BitSet[domain])
    last1 = maximum(domain_residues_NSP_BitSet[domain])
    first = first1 + ORFadd
    last = last1 + ORFadd
    ORF_range_BitSet = BitSet(first:last)
    firstORF = string(first)
    lastORF = string(last)
    ORF_range = ""
    if NSP < 12
        ORF_range = "ORF1a:$(firstORF)-$(lastORF)"
    else
        ORF_range = "ORF1b:$(firstORF)-$(lastORF)"
    end
    return ORF_range, ORF_range_BitSet
end
#####################################################################################################################################
sorted_ratio_gene_dicts = [chr_to_circ_gene_density_ratio_sort_by_gene, chr_to_circ_gene_density_ratio_sort_by_ratio] 
sorted_ratio_domain_dicts = [chr_to_circ_domain_density_ratio_sort_by_domain, chr_to_circ_domain_density_ratio_sort_by_ratio]
headers_gene = ["###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Gene ######", "###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Ratio ######"]
headers_domain = ["######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Domain #########", "######### Chronic-to-Circulating Domain Mutation Density Ratios, Sorted by Ratio #########"]
open("$(date)_$(fx_name).txt", "w") do g
    ct = 0
    for sorted_dict in sorted_ratio_gene_dicts
        ct += 1
        println()
        println(headers_gene[ct])
        println(g)
        println(g, headers_gene[ct])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
            println(g, "                  ", rpad("$(gene)", 5), " = ", lpad("$(ratio)", 5), "       ", lpad("$(NSP_ranges[gene])", 16) )
        end
    end
    ct2 = 0
    for sorted_dict in sorted_ratio_domain_dicts
        ct2 += 1
        println()
        println(headers_domain[ct2])
        println(g)
        println(g, headers_domain[ct2])
        for w in sorted_dict
            gene = w[1]
            ratio1 = string(Int(w[2]÷1) )
            ratio21 = string(round(digits=2, w[2]%1) )
            ratio22 = split(ratio21, ".")[2]
            if length(ratio22) == 1
                ratio22 = ratio22*"0"
            end
            ratio = "$(ratio1).$(ratio22)"
            println("               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
            println(g, "               ", rpad("$(gene)", 13), " = ", lpad("$(ratio)", 5), "   ", rpad("$(domain_residues_NSP[gene])", 15), "  ", rpad("$(domain_residues_ORF[gene])", 17) )
        end
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
gene_sub_ranks_chr(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels)
gene_sub_ranks_all(gene_array, sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
gene_sub_pos_ranks_all(gene_array, gene_AA_lengths,  AA_muts_ct_pos_only_no_dels_all)
gene_sub_pos_ranks_chr(gene_array, gene_AA_lengths, AA_muts_ct_pos_only_no_dels )
NSP_sub_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels)
NSP_sub_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_all(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels_all)
NSP_sub_pos_ranks_chr(NSP_array, NSP_sub_types_at_every_site_combined, AA_muts_ct_no_dels); print("\n"^1)
######################################################################################################################################
print(sub_types_at_every_site_combined["ORF9b"][2])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][2]);    print("\n"^1)
print(sub_types_at_every_site_combined["ORF9b"][92])
print(" | ", sub_types_at_every_site_combined_ct["ORF9b"][92]);   print("\n"^1)
############################################################################################################################################################################ 
############################################################################################################################################################################
gene_AA_sites = Dict{String, BitSet}()
for (gene, length) in gene_AA_lengths
    gene_AA_sites[gene] = BitSet([1:length...])
end
#for (gene, bitset) in gene_AA_sites; println(gene); println(bitset); end
############################################################
gene_AA_sites_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_used[gene] = BitSet()
end
for (sub, ct) in AA_muts_ct_no_dels_all
    gene = aa_gene_comprehensive_dict[sub]
    AA_site = aa_pos_comprehensive_dict[sub]
    push!(gene_AA_sites_used[gene], AA_site)
end
############################################################
gene_AA_sites_not_used = Dict{String, BitSet}()
for gene in gene_array
    gene_AA_sites_not_used[gene] = BitSet()
end
for (gene, bitset) in gene_AA_sites
    bitset_used = gene_AA_sites_used[gene]
    unused_AA_sites = setdiff(bitset, bitset_used)
    gene_AA_sites_not_used[gene] = unused_AA_sites
end
###########################################################
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    print("$(gene) = "); print(" | ")
    for unused in unused_AA_sites
        print("$(unused) |")
    end
end
for (gene, unused_AA_sites) in gene_AA_sites_not_used
    for AA_site in unused_AA_sites
        placeholder_sub = "$(gene):W$(AA_site)W"
        AA_muts_ct_no_dels_all[placeholder_sub] = 0
    end
end
############################################################################################################################################################################ 
############################################################################################################################################################################
#NSP_array = ["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP11", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16"]
NSP1_muts_ct_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_no_dels_all = Dict{String, Int}()
NSP_muts_ct_no_dels_all_array = [NSP1_muts_ct_no_dels_all, NSP2_muts_ct_no_dels_all, NSP3_muts_ct_no_dels_all, NSP4_muts_ct_no_dels_all, NSP5_muts_ct_no_dels_all, NSP6_muts_ct_no_dels_all, NSP7_muts_ct_no_dels_all, NSP8_muts_ct_no_dels_all, NSP9_muts_ct_no_dels_all, NSP10_muts_ct_no_dels_all, NSP11_muts_ct_no_dels_all, NSP12_muts_ct_no_dels_all, NSP13_muts_ct_no_dels_all, NSP14_muts_ct_no_dels_all, NSP15_muts_ct_no_dels_all, NSP16_muts_ct_no_dels_all]       
NSP_muts_ct_no_dels_all = Dict{String, Int}()
for (sub, count) in AA_muts_ct_no_dels_all
    gene = aa_gene_comprehensive_dict[sub]
    if gene == "ORF1a" || gene == "ORF1b"
        AA_site = aa_pos_comprehensive_dict[sub]
        if AA_site < 4402
            NSP_sub = ORF1abMut_to_NSP(sub)
            NSP_muts_ct_no_dels_all[NSP_sub] = count
            NSP = NSP_muts_gene_dict[NSP_sub]
            NSP_num = parse(Int, NSP[4:end])
            NSP_dict = NSP_muts_ct_no_dels_all_array[NSP_num]
            NSP_dict[NSP_sub] = count
        end
    end
end
###################################################################################################################################
NSP1_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP2_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP3_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP4_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP5_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP6_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP7_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP8_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP9_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP10_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP11_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP12_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP13_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP14_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP15_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP16_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
NSP_muts_ct_pos_only_no_dels_all_array = [NSP1_muts_ct_pos_only_no_dels_all, NSP2_muts_ct_pos_only_no_dels_all, NSP3_muts_ct_pos_only_no_dels_all, NSP4_muts_ct_pos_only_no_dels_all, NSP5_muts_ct_pos_only_no_dels_all, NSP6_muts_ct_pos_only_no_dels_all, NSP7_muts_ct_pos_only_no_dels_all, NSP8_muts_ct_pos_only_no_dels_all, NSP9_muts_ct_pos_only_no_dels_all, NSP10_muts_ct_pos_only_no_dels_all, NSP11_muts_ct_pos_only_no_dels_all, NSP12_muts_ct_pos_only_no_dels_all, NSP13_muts_ct_pos_only_no_dels_all, NSP14_muts_ct_pos_only_no_dels_all, NSP15_muts_ct_pos_only_no_dels_all, NSP16_muts_ct_pos_only_no_dels_all]
NSP_muts_ct_pos_only_no_dels_all = Dict{String, Int}()
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP_all_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    NSP = NSP_array[i]
    NSP_len = NSP_AA_size[NSP]
    for j in 1:NSP_len
        NSP_pos = NSP*":"*"$(j)"
        NSP_all_dict[NSP_pos] = 0
        NSP_muts_ct_pos_only_no_dels_all[NSP_pos] = 0
    end
end
for i in 1:length(NSP_muts_ct_pos_only_no_dels_all_array)
    NSP = NSP_array[i]
    NSP_dict = NSP_muts_ct_no_dels_all_array[i]
    NSP_pos_dict = NSP_muts_ct_pos_only_no_dels_all_array[i]
    for (sub, ct) in NSP_dict
        pos = NSP_muts_pos_dict[sub]
        sub_pos = NSP*":"*"$(pos)"
        NSP_pos_dict[sub_pos] += ct
        NSP_muts_ct_pos_only_no_dels_all[sub_pos] += ct
    end
end
total_time_post_load_cell = time() - start_time_post_load_cell
total_time_post_load_cell_rd = round(digits=1, total_time_post_load_cell)
print("\n"^1)
println("Total Time for Post-Load Cell = $(total_time_post_load_cell_rd) Seconds"); print("\n"^2)
# Total Time for Post-Load Cell = 139.3 Seconds (2025-08-18)
######################################################################################################################

2026_03_01__904AM

Avg AA Subs per Chronic Sequence = 18.06
Avg AA Subs per Circulating Sequence = 3.744
simple_chr_to_circ_adj_factor = 400.21
Adjustment Factor for greater number of AA muts per chronic seq compared to circ seq = 0.2073

Total Singlets = 2305 | Total Groups = 992 | Total Independent Chronics = 3297
Total Qualifying Circulating Sequences = 10871228
(Total Qualifying Circulating Sequences)/(Total Independent Chronics) = 3297.31
Combined Adjustment Factor = 683.471  |  Combined Adjustment Factor Check = 683.471

###### Chronic-to-Circulating Gene Mutation Density Ratios, Sorted by Gene ######
                  NSP1  =  0.62            ORF1a:1-180
                  NSP2  =  0.27          ORF1a:181-818
                  NSP3  =  0.49         ORF1a:819-2763
                  NSP4  =  0.57        ORF1a:2764-3263
                  NSP5  =  0.37        ORF1a:3264-3569
                  NSP6  =  0.50        ORF1a:3570-3859
                  NSP7  =  0.57        ORF1a:3860-3942


In [231]:
### Making pango_to_pango_unaliased_v2 & pango consensus Dicts + MANY Others | Runtime = 154 seconds
####### Making pango_to_pango_unaliased_v2 —— using pango_variant_alias_key_2026_01_03_update.json from https://github.com/cov-lineages/pango-designation/blob/master/pango_designation/alias_key.json
start = time(); date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")

missing_AA_dict_keys = Set{String}()
############################################################################################################################################################################
#################### Making/filling index_to_tuple, tuple_to_index, index_to_date_str, date_str_to_index...etc dictionaries ################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
###################################
chronic_pango_set = Set{String}()
chronic_pango_unaliased_set = Set{String}()
chronic_clade_set = Set{String}()
for seq in all_qualifying_seqs_set
    push!(chronic_pango_set, seq_pango[seq])
    push!(chronic_pango_unaliased_set, seq_pango_unaliased[seq])
    push!(chronic_clade_set, seq_clade[seq])
end
chronic_pango_set_len = length(chronic_pango_set)
chronic_pango_unaliased_set_len = length(chronic_pango_unaliased_set)
chronic_clade_set_len = length(chronic_clade_set)
println("Number of Different Pango Lineages in Chronics = $(chronic_pango_set_len)")
println("Number of Different Pango_Unaliased Lineages in Chronics = $(chronic_pango_unaliased_set_len)")
println("Number of Different Clades in Chronics = $(chronic_clade_set_len)")
println()
#####################################################################################################################################
all_fucking_pangos = union(pango_set, pango_index_date_set, chronic_pango_set)
push!(all_fucking_pangos, "B.1.640")
push!(all_fucking_pangos, "B.1.640.1")
push!(all_fucking_pangos, "B.1.640.2")
push!(all_fucking_pangos, "B.1.258.2")
push!(all_fucking_pangos, "G.1")
#####################################################################################################################################
nonleap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>28, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
leap_month_day_dict = Dict{Int, Int}(0=>0, 1=>31, 2=>29, 3=>31, 4=>30, 5=>31, 6=>30, 7=>31, 8=>31, 9=>30, 10=>31, 11=>30, 12=>31)
###################
index_to_tuple = Dict{Int, Tuple{Int, Int, Int}}()
tuple_to_index = Dict{Tuple{Int, Int, Int}, Int}()        
###########################################################
for (seq, date) in seq_collection_date
    date_arr = string.(collect(date))
    if sum(date_arr .== "-") ≠ 2
        println("Doesn't have two dashes in date = $(seq)")
    else
        year = string(split(date, "-")[1])
        month = string(split(date, "-")[2])
        day = string(split(date, "-")[3])
        if length(month) == 1 && month ≠ "0"
            month = add_leading_zero(month)
        end
        if length(day) == 1 && day ≠ "0"
            day = add_leading_zero(day)
        end
        seq_collection_date[seq] = year*"-"*month*"-"*day
    end
end 
###################
index = 0
for year in 2020:2027
    for month in 1:12
        if year%4 == 0
            month_days = leap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        else
            month_days = nonleap_month_day_dict[month]
            for day in 1:month_days
                index += 1
                index_to_tuple[index] = (year, month, day)
            end
        end
    end
end
for (index, date) in index_to_tuple
    tuple_to_index[date] = index
end
for y in 2020:2027
    tuple_to_index[(y, 0, 0)] = y*1000000
    index_to_tuple[y*1000000] = (y, 0, 0)
    for m in 1:12
        tuple_to_index[(y, m, 0)] = y*1000000 + m*1000
        index_to_tuple[y*1000000 + m*1000] = (y, m, 0)
    end
end
tuple_to_index[(0, 0, 0)] = 1000000000
index_to_tuple[1000000000] = (0, 0, 0)
###################################################################################################################
############################################################################################################################################################################
pango_to_pango_unaliased_v2["G.1"] = "B.1.258.2.1"
pgct = 0
for (pngo, unal) in pango_to_pango_unaliased_v2
    pgct += 1
    if pgct < 5
        println("$(pngo) = $(unal)")
    end
end
println("length(keys(pango_to_pango_unaliased_v2) = $(length(keys(pango_to_pango_unaliased_v2)))")
########################################################################################################################################################################
########################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
missing_keys = String[]
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    if !(pango in pango_set)
        push!(missing_keys, pango)
    end
    push!(pango_set, pango)
end
for pango in keys(pango_date_index_ct)
    if !(pango in keys(pango_to_pango_unaliased_v2))
        push!(pango_set, pango)
    end
end
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
#pango_to_pango_unaliased = Dict{String, String}()
clade_pango_set = Dict{String, Set{String}}()
#################################################################################
pango_nuc_sub_WT = Dict{String, Set{String}}()
pango_nuc_del_WT = Dict{String, Set{String}}()
pango_nuc_sub_private = Dict{String, Set{String}}()
pango_nuc_del_private = Dict{String, Set{String}}()
pango_nuc_sub_revs = Dict{String, Set{String}}()
pango_nuc_del_revs = Dict{String, Set{String}}()

pango_AAsub_WT = Dict{String, Set{String}}()
pango_AAsub_WT_pos_only = Dict{String, Set{String}}()
pango_AAdel_WT = Dict{String, Set{String}}()
pango_AAsub_private = Dict{String, Set{String}}()
pango_AAdel_private = Dict{String, Set{String}}()
pango_AAsub_revs = Dict{String, Set{String}}()
pango_AAdel_revs = Dict{String, Set{String}}()

pango_frameshifts_WT = Dict{String, Set{String}}()
pango_designation_date = Dict{String, String}()
#####################################################
for pango in pango_set
    pango_nuc_sub_WT[pango] = Set{String}()
    pango_nuc_del_WT[pango] = Set{String}()
    pango_nuc_sub_private[pango] = Set{String}()
    pango_nuc_del_private[pango] = Set{String}()
    pango_nuc_sub_revs[pango] = Set{String}()
    pango_nuc_del_revs[pango] = Set{String}()
########
    pango_AAsub_WT[pango] = Set{String}()
    pango_AAsub_WT_pos_only[pango] = Set{String}()
    pango_AAdel_WT[pango] = Set{String}()
    pango_AAsub_private[pango] = Set{String}()
    pango_AAdel_private[pango] = Set{String}()
    pango_AAsub_revs[pango] = Set{String}()
    pango_AAdel_revs[pango] = Set{String}()
#########
    pango_frameshifts_WT[pango] = Set{String}()
end
#################################################################################
for clade in clade_set
    clade_pango_set[clade] = Set{String}()
end
ndjson = "___pango_consensus_sequences/pango_consensus_sequences_summary_2025_06_25.ndjson"
missing_keys_for_WT = Set{String}()
###################################################################
#pango_consensus_nuc_subs_WT = 
#pango_consensus_nuc_dels_WT = 
#pango_consensus_nuc_subs = 
#pango_consensus_nuc_dels = 
#pango_consensus_nuc_revs = 
#pango_consensus_nuc_del_revs = 
#pango_consensus_AA_subs_WT = 
#pango_consensus_AA_dels_WT = 
#pango_consensus_AA_subs = 
#pango_consensus_AA_dels = 
#pango_consensus_AA_revs = 
#pango_consensus_AA_del_revs = 
#pango_consensus_
###################################################################
for line in eachline(ndjson)
    j = JSON3.read(line)
    pango = j.lineage
    unaliased = j.unaliased
    clade = j.nextstrainClade
    push!(clade_pango_set[clade], pango)
#    pango_to_pango_unaliased[pango] = unaliased
####################
    parent = j.parent
    children = j.children
####################
    nuc_subs_WT = j.nucSubstitutions
    nuc_dels_WT = j.nucDeletions
    nuc_subs = j.nucSubstitutionsNew
    nuc_dels = j.nucDeletionsNew
    nuc_revs = j.nucSubstitutionsReverted
    nuc_del_revs = j.nucDeletionsReverted
#####################
    for nuc_sub in nuc_subs_WT
        push!(pango_nuc_sub_WT[pango], nuc_sub)
    end
    for ndwt in nuc_dels_WT
        push!(pango_nuc_del_WT[pango], ndwt)
    end
    for n in nuc_subs
        push!(pango_nuc_sub_private[pango], n)
    end
    for n in nuc_dels
        push!(pango_nuc_del_private[pango], n)
    end
    for n in nuc_revs
        push!(pango_nuc_sub_revs[pango], n)
    end
    for n in nuc_del_revs
        push!(pango_nuc_del_revs[pango], n)
    end
########################################################
    AA_subs_WT = j.aaSubstitutions
    AA_dels_WT = Set(j.aaDeletions)
    AA_subs = j.aaSubstitutionsNew
    AA_dels = j.aaDeletionsNew
    AA_revs = j.aaSubstitutionsReverted
    AA_del_revs = j.aaDeletionsReverted
##################################################
    AA_subs_WT_pos_only = Set{String}()
    delete!(AA_dels_WT, "")
    for sub in AA_subs_WT
        if !haskey(aa_gene_and_pos_comprehensive_dict, sub)
            push!(missing_keys_for_WT, sub)
            continue
        end
        sub_po = aa_gene_and_pos_comprehensive_dict[sub]
        push!(AA_subs_WT_pos_only, sub_po)
    end 
    for AA_sub_po in AA_subs_WT_pos_only
        push!(pango_AAsub_WT_pos_only[pango], AA_sub_po)
    end
##################################################
    for AAsub in AA_subs_WT
        push!(pango_AAsub_WT[pango], AAsub)
    end
    for AAdel in AA_dels_WT
        push!(pango_AAdel_WT[pango], AAdel)
    end
    for AAsub in AA_subs
        push!(pango_AAsub_private[pango], AAsub)
    end
    for AAdel in AA_dels
        push!(pango_AAdel_private[pango], AAdel)
    end
    for AArev in AA_revs
        push!(pango_AAsub_revs[pango], AArev)
    end
    for AAdelrev in AA_del_revs
        push!(pango_AAdel_revs[pango], AAdelrev)
    end
    frameshiftsWT = j.frameShifts
    for fs in frameshiftsWT
        push!(pango_frameshifts_WT[pango], fs)
    end
    designation_date = j.designationDate
    pango_designation_date[pango] = designation_date
end
#####################################################################################################################################################################
#####################################################################################################################################################################
push!(pango_nuc_del_WT["B.1.617.2"], "28271")
push!(pango_nuc_del_private["B.1.617.2"], "28271")
#####################################################################################################################################################################
#####################################################################################################################################################################
println("Done Making Pango Consensus Sequences! (Part 1)"); print("\n"^1)
#####################################################################################################################################################################
println("Total Missing Keys for AA_pos_only_gene_and_pos_dict = $(length(missing_keys_for_WT))")
println()
#missing_keys_for_WT_sort = sort(collect(missing_keys_for_WT), by = x -> mp_AA_gene_sortKey_2(x))
missing_keys_for_WT_join = join(collect(missing_keys_for_WT), ", ")
println("missing_keys_for_WT_join = ", "|", missing_keys_for_WT_join, "|"); print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################# NEW SECTION (2026-01-03)   #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
### Making consensus muts for new sequences (using Angie Hinrichs' tree mut TSV file) —— MUST RUN load_all_seq_dicts FIRST
df_tree_muts = CSV.read("___pango_consensus_sequences/pango_clade_mutations_from_Angie_Hinrichs_2026_01_03.tsv", DataFrame, delim='\t')
##################synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = nuc_to_AA(pango, new_muts)########################################################################################################################################
total_rows = nrow(df_tree_muts)
tree_pangos = df_tree_muts[!, 1]
tree_muts = df_tree_muts[!, 2]
#####################################################################################################################
#  Example of 1st, then 2nd column (i.e. tree_pangos & tree_muts) ——  AY.112.2 | AY.112 > G23012C > G4162T > G27516A
#####################################################################################################################
tree_pangos_set = Set(tree_pangos)
tree_muts_set = Set(tree_muts)
################################################
bad_pangos = Vector{String}()
################################################
OG_tree_pangos_set_len = length(tree_pangos_set)
println("OG_tree_pangos_set_len = $(OG_tree_pangos_set_len)")
excluded_bits = ["misc", "proposed", '_']
for tpango in tree_pangos_set
    for exclude in excluded_bits
        if occursin(exclude, tpango)
            push!(bad_pangos, tpango)
        end
    end
end
for baddy in bad_pangos
    delete!(tree_pangos_set, baddy)
end
NEW_tree_pangos_set_len = length(tree_pangos_set)
println("NEW_tree_pangos_set_len = $(NEW_tree_pangos_set_len)")
println("bad_pangos length = $(length(bad_pangos))")
bad_pangos_sort = sort(bad_pangos)
bad_pangos_sort_join = join(bad_pangos_sort, ", ")
bad_pangos_sort_join_newlines = join(bad_pangos_sort, "\n")
###########
println(bad_pangos_sort_join)
############################################################################################################################################################################
pango_unaliased_to_pango_prefix = Dict{String, String}()
pango_to_pango_unaliased_v2 = Dict{String, String}()
pango_unaliased_to_pango = Dict{String, String}()
##################################################################
pango_unaliased_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
pango_predecessor_meta_dict = Dict{String, Dict{Int, String}}()
##################################################################
for (prefix, unaliased) in pango_prefix_to_pango_unaliased
    pango_unaliased_to_pango_prefix[unaliased] = prefix
end
for pango in tree_pangos_set
    dotpts_ct = count('.', pango)
    if dotpts_ct ≥ 1
        dotsplits = split(pango, ".")
        prefix = string(dotsplits[1])
        prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
        last_pts = join(dotsplits[2:end], ".")
        pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
    else
        pango_to_pango_unaliased_v2[pango] = pango
    end
end
#####################################################################
pango_to_pango_unaliased_v2_key_set = Set(collect(keys(pango_to_pango_unaliased_v2)))
no_key_pangos = Set(["C.1.2", "B.1.1.70", "B.1.177.46", "B.1.36", "B.1.551", "B.1.1.329", "B.1.1.170", "C.26", "B.1.214.3", "B.1.609", "B.55", "A.2.5.3", "B.1.1.117", "C.13", "B.1.1.523", "B.1.470", "B.1.1.67", "B.39", "B.1.36.31", "B.1.36.22", "B.4", "B.1.1.500", "B.1.258.17", "B.1.1.101", "B.1.1.141", "B.1.260", "B.1.1.360", "AE.1", "B.1.404", "B.1.351.1", "AE.8", "B.1.1.10", "B.1.36.19", "B.1.587", "B.1.465", "B.1.177.52", "B.1.1.404", "B.1.436", "A.5", "B.1.1.221", "B.1.396", "B.1.428", "C.4", "B.1.214.2", "B.1.568", "B.1.147", "B.1.1.442", "B.1.1.47", "B.1.575.1", "B.1.177.4", "B.1.527", "B.1.1.354", "B.1.530", "B.1.361", "B.1.36.36", "B.1.126", "B.1.1.297", "B.1.420", "B.1.1.406", "B.1.36.29", "B.1.560", "B.1.214.1", "W.1", "B.1.177.12", "B.1.629", "B.1.1.37", "B.1.1.192", "N.6", "A.23", "B.1.177.54", "A.2.5.1", "A.3", "B.1.1.263", "B.1.379", "B.1.1.376", "B.1.240.1", "B.1.466.1", "B.1.1.241", "B.1.1.368", "B.1.177.32", "B.1.1.420", "B.1.1.413", "B.1.236", "B.1.2", "B.23", "B.1.1.418", "B.1.416.1", "B.1.1.54", "B.1.540", "AZ.5", "B.1.389", "B.1.433", "B.1.1.205", "B.1.1.434", "B.1.1.359", "C.2", "B.1.600", "B.1.9", "B.1.575", "B.1.177.57", "B.1.627", "B.1.36.35", "A.19", "B.1.371", "B.1.462", "B.1.22", "B.1.111", "B.1.258", "B.1.427", "B.1.177.53", "B.1.626", "B.1.1.294", "B.1.36.38", "B.1.1.332", "B.1.1.417", "B.1.1.521", "B.1.558", "B.1.36.18", "B.1.544", "B.1.1.375", "B.1.1.412", "B.1.1.485", "B.1.1.326", "B.1.375", "B.1.1.243", "B.1.1.369", "B.1.336", "B.1.577", "B.1.409", "B.1.219", "B.1.1.301", "B.1.1.433", "B.1.128", "B.1.179", "B.1.1.519", "B.1.1.307", "B.1.537", "B.1.1.265", "C.38", "B.1.177.44", "B.1.620", "AD.2", "C.23", "B.1.189", "B.1.349", "B.1.567", "B.1.398", "AZ.2", "B.1.1.25", "B.1.1.163", "B.1.497", "B.1.480", "B.1.1.198", "B.1.320", "B.1.1.121", "B.1.402", "B.6.6", "B.1.8", "B.1.298", "B.1.177.8", "B.1.1.39", "B.1.324", "B.1.1.351", "B.1.1.161", "B.1.1.279", "B.1.1.46", "B.1.177.83", "B.1.177.15", "B.1.346", "B.1.566", "B.1.78", "B.1.239", "A", "B.1.1.318", "B.1.397", "B.1.177.10", "B.1.1.416", "B.1.1.27", "AZ.3", "B.1.319", "B.1.619", "B.1.177.7", "B.1.258.3", "B.1.1.317", "B.1.605", "B.1.146", "B.1.1.61", "A.25", "C.40", "B.3", "B.1.438", "B.1.1.33", "B.1.192", "B.1.468", "C.11", "B.1.177.18", "B.1.471", "B.1.1.52", "B.1.1.171", "C.1", "B.1.1.348", "B.1.1.507", "B.1.177.82", "B.1.533", "B.1.1.306", "B.1.524", "B.1.149", "B.1.619.1", "B.1.36.9", "B.1.570", "B.1.1.50", "B.1.517", "B.1.499", "B.1.177.60", "B.1.1.378", "B.1.177.62", "C.16", "C.36.3", "B.1.1.459", "B.1.177.75", "A.27", "B.1.1.207", "B.1.618", "B.1.391", "B.1.1.115", "A.29", "B.1.1.1", "L.3", "A.28", "B.1.1.302", "B.1.3", "C.2.1", "B.1.1.135", "B.1.1.316", "B.1.393", "B.1.177.87", "R.1", "B.1.1.176", "C.32", "B.1.636", "B.1.1.133", "B.1.1.462", "N.3", "B.1.399", "B.1.1.487", "B.1.1.311", "B.1.606", "B.1.617.1", "B.1.265", "B.1.177.86", "N.9", "B.1.1.374", "B.1.1.228", "B.1.177", "B.1.177.5", "A.2.5.2", "B.1.1.57", "C.33", "B.1.538", "B.1.258.14", "B.1.1.305", "B.1.505", "B.1.1.397", "A.1", "B.1.565", "AT.1", "B.1.1.214", "B.1.177.16", "C.35", "B.6", "B.1.356", "B.1.177.21", "B.1.1.153", "B.1.1.432", "B.1.1.181", "B.1.1.312", "B.1.1.216", "B.1.1.526", "A.21", "B.1.380", "B.1.210", "B.1.1.398", "B.1.232", "B.1.1.220", "C.36", "B.1.438.1", "B.1.503", "B.1.36.7", "B.1.426", "B.1.1.159", "B.1.1.231", "A.18", "B.1.177.77", "B.1.1.99", "B.1.1.298", "B.1.453", "B.1.1.253", "N.5", "B.1.367", "B.1.549", "C.14", "B.1.400", "B.1.1.284", "AZ.4", "B.1.459", "B.1.243", "C.36.3.1", "B.1.474", "B.1.221", "B.1.630", "C.29", "B.1.1.528", "B.1.1.203", "B.1.1.269", "B.1.340", "B.1.634", "B.1.1.174", "B.1.596.1", "B.1.311", "B.1.36.24", "B.1.509", "B.1.36.39", "B.1.1.189", "B.1.36.16", "B.1.441", "B.1.91", "B.1.416", "B.1.1.56", "B.1.258.11", "B.1.417", "B.1.525", "B.1.561", "B.1.1.525", "B.1.1.277", "B.1.177.35", "B.1.240", "B.1.1.382", "B.1.234", "B.1.1.273", "B.1.1.291", "N.4", "B.1.617.3", "A.2", "B.1.523", "B.1.279", "B.1.1.63", "B.1.1.51", "B.1.1.157", "B.1.415.1", "B.1.212", "B.1.177.73", "B.1.1.371", "A.2.5", "B.1.1.44", "B.1.1.464", "B.1.362", "D.2", "B.1.241", "B.1.1.486", "B.1.377", "B.1.1.53", "B.1.588", "B.1.1.391", "B.1.1.186", "B.1.1.274", "B.1.623", "B.1.36.1", "B.1.195", "B.1.237", "B.1.110", "B.1.1.111", "AZ.1", "B.1.395", "B.1.381", "C.39", "B.1.206", "B.1.177.81", "B.1.214", "B.40", "B.1.110.3", "B.1.1.83", "B.1.411", "B.1.625", "B.1.350"])
for pango in all_fucking_pangos
    if !haskey(pango_to_pango_unaliased_v2, pango)
        dotpts_ct = count('.', pango)
        if dotpts_ct ≥ 1
            dotsplits = split(pango, ".")
            prefix = string(dotsplits[1])
            prefix_unaliased = get(pango_prefix_to_pango_unaliased, prefix, prefix)
            last_pts = join(dotsplits[2:end], ".")
            pango_to_pango_unaliased_v2[pango] = "$(prefix_unaliased).$(last_pts)"
        end
    end
end
pango_to_pango_unaliased_v2["B.1.1.529"] = "B.1.1.529"
pango_to_pango_unaliased_v2["LF.3"] = "B.1.1.529.2.86.1.1.16.1.3"
pango_to_pango_unaliased_v2["LF.3.1"] = "B.1.1.529.2.86.1.1.16.1.3.1"
pango_to_pango_unaliased_v2["B.1.469"] = "B.1.469"
pango_to_pango_unaliased_v2["A"] = "A"
pango_to_pango_unaliased_v2["B"] = "B"
pango_to_pango_unaliased_v2["B.1.1.482"] = "B.1.1.482"
pango_to_pango_unaliased_v2["B.1.1.370"] = "B.1.1.370"
pango_to_pango_unaliased_v2["B.1.596"] = "B.1.596"
pango_to_pango_unaliased_v2["B.1.415"] = "B.1.415"
####################################################################
pango_unaliased_set = Set{String}()
for (pango, unaliased) in pango_to_pango_unaliased_v2
    pango_unaliased_to_pango[unaliased] = pango
    push!(pango_unaliased_set, unaliased)
end
####################################################################
all_fucking_pangos = union(all_fucking_pangos, pango_to_pango_unaliased_v2_key_set)
for pango in all_fucking_pangos
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_predecessor_meta_dict[unaliased] = Dict{Int, String}()
#    println(pango)
    dotpts_ct = count('.', unaliased)
    if dotpts_ct ≥ 1
        dotsplits = split(unaliased, ".")
        for i in 1:dotpts_ct
            predecessor_i_unaliased = join(dotsplits[1:end-i], ".")
            pango_unaliased_predecessor_meta_dict[unaliased][i] = predecessor_i_unaliased
#            println("$(pango), $(unaliased), $(i) = $(predecessor_i_unaliased)")
        end
    end
end
pango_unaliased_predecessor_meta_dict["B.1.1.370"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.370"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.370"][3] = "B"
pango_unaliased_to_pango["B.1.1.370"] = "B.1.1.370"

pango_unaliased_predecessor_meta_dict["B.1.1.482"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.482"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.482"][3] = "B"
pango_unaliased_to_pango["B.1.1.482"] = "B.1.1.482"

pango_unaliased_predecessor_meta_dict["B.1.596"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.596"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.596"][3] = "B"
pango_unaliased_to_pango["B.1.596"] = "B.1.596"

pango_unaliased_predecessor_meta_dict["B.1.415"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.415"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.415"][3] = "B"
pango_unaliased_to_pango["B.1.415"] = "B.1.415"

for pango in all_fucking_pangos
    pango_predecessor_meta_dict[pango] = Dict{Int, String}()
    unaliased = pango_to_pango_unaliased_v2[pango]
    for (i, predecessor_i) in pango_unaliased_predecessor_meta_dict[unaliased]
        predecessor_i_pango = pango_unaliased_to_pango[predecessor_i]
        pango_predecessor_meta_dict[pango][i] = predecessor_i_pango
    end
end
#######################################################
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][1] = "B.1.1.529.2.86.1.1.16.1.3"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][2] = "B.1.1.529.2.86.1.1.16.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][3] = "B.1.1.529.2.86.1.1.16"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][4] = "B.1.1.529.2.86.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][5] = "B.1.1.529.2.86.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][6] = "B.1.1.529.2.86"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][7] = "B.1.1.529.2"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][8] = "B.1.1.529"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][9] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][10] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529.2.86.1.1.16.1.3.1"][11] = "B"
pango_predecessor_meta_dict["LF.3.1"] = Dict{Int, String}()
pango_predecessor_meta_dict["LF.3.1"][1] = "LF.3"
pango_predecessor_meta_dict["LF.3.1"][2] = "JN.1.16.1"
pango_predecessor_meta_dict["LF.3.1"][3] = "JN.1.16"
pango_predecessor_meta_dict["LF.3.1"][4] = "JN.1"
pango_predecessor_meta_dict["LF.3.1"][5] = "BA.2.86.1"
pango_predecessor_meta_dict["LF.3.1"][6] = "BA.2.86"
pango_predecessor_meta_dict["LF.3.1"][7] = "BA.2"
pango_predecessor_meta_dict["LF.3.1"][8] = "B.1.1.529"
pango_predecessor_meta_dict["LF.3.1"][9] = "B.1.1"
pango_predecessor_meta_dict["LF.3.1"][10] = "B.1"
pango_predecessor_meta_dict["LF.3.1"][11] = "B"
pango_unaliased_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_unaliased_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_unaliased_predecessor_meta_dict["B.1.1.529"][3] = "B"
pango_predecessor_meta_dict["B.1.1.529"] = Dict{Int, String}()
pango_predecessor_meta_dict["B.1.1.529"][1] = "B.1.1"
pango_predecessor_meta_dict["B.1.1.529"][2] = "B.1"
pango_predecessor_meta_dict["B.1.1.529"][3] = "B"
############################################################################################################################################################################
gene_array = ["ORF1a", "ORF1b", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"]
#######################################################################################################################################
forbidden_pangos = Set(["XCD", "XCT", "XEA", "XFR", "XFS", "XFT", "XFT.1", "XFU", "XGD"])
#######################################################################################################################################
pango_new_nuc_muts = Dict{String, Set{String}}()
pango_new_nuc_muts_WT = Dict{String, Set{String}}()
pango_new_nuc_muts_pos_only = Dict{String, Set{Int}}()
###########################
pango_new_AA_muts = Dict{String, Set{String}}()
pango_new_AA_muts_WT = Dict{String, Set{String}}()
pango_new_AA_muts_pos_only = Dict{String, Set{String}}()
###########################
nuc_mut_all_and_chr_set_missing = Set{String}()
new_pango_ct = 0
empty_string_pangos = Set{String}()
for i in 1:length(tree_muts)
    if tree_pangos[i] in tree_pangos_set
        pango = convert(String, tree_pangos[i])
        unaliased = pango_to_pango_unaliased_v2[pango]
        if ismissing(tree_muts[i])
            println("missing tree muts: $(pango)")
            continue
        end
        new_muts_pre0 = replace(tree_muts[i], r"        "=>",")
        new_muts_pre1 = replace(new_muts_pre0, r"[TCAGN] [TCAGN]"=>",")
        new_muts_pre2 = replace(new_muts_pre1, r"\s"=>"")
        new_muts_pre3 = replace(new_muts_pre2, ">"=>",")
        new_muts_pre4 = replace(new_muts_pre3, ",,"=>",")
#        println("new_muts_pre3 = $(new_muts_pre3)")
        comma_splits = string.(split(new_muts_pre3, ","))
        predecessor_vec = String[]
        predecessor = string(comma_splits[1])
        if '_' in predecessor
            predecessor = string(split(predecessor, "_")[1])
        end
#        predecessor_unaliased = pango_to_pango_unaliased_v2[predecessor]
        push!(predecessor_vec, predecessor)
#        println(predecessor)
        for j in 1:length(pango_unaliased_predecessor_meta_dict[unaliased])
            predecessor_j = pango_unaliased_predecessor_meta_dict[unaliased][j]
            predecessor_j_pango = pango_unaliased_to_pango[predecessor_j]
            push!(predecessor_vec, predecessor_j_pango)
        end
        new_muts_pre = comma_splits[2:end]
        new_muts = String[]
        for mut in new_muts_pre
            if !('N' in mut)
                push!(new_muts, mut)
            end
        end
#        println("new_muts = $(new_muts)")
#        if !haskey(pango_nuc_sub_private, pango) && !haskey(pango_AAsub_private, pango)
#            pango_nuc_sub_WT[pango] = Set{String}()
#            pango_nuc_del_WT[pango] = Set{String}()
#            pango_nuc_sub_private[pango] = Set{String}()
#            pango_nuc_del_private[pango] = Set{String}()
#            pango_AAsub_WT[pango] = Set{String}()
#            pango_AAsub_WT_pos_only[pango] = Set{String}()
#            pango_AAdel_WT[pango] = Set{String}()
#            pango_AAsub_private[pango] = Set{String}()
#            pango_AAdel_private[pango] = Set{String}()
#        end
#        new_muts = filter!(
        if !isempty(new_muts)
            for xmut in new_muts
                if xmut == ""
                    push!(empty_string_pangos, pango)
#                    println("empty string in $(pango)")
                    continue
                end
                if !(xmut in nuc_mut_all_and_chr_set)
                    push!(nuc_mut_all_and_chr_set_missing, xmut)
                    pos_int = nuc_mut_int_comprehensive_dict[xmut]
                    refnuc = ref_nuc_comprehensive_dict[xmut]
                    qrynuc = qry_nuc_comprehensive_dict[xmut]
                    nuc_mut_int_comprehensive_dict[xmut] = pos_int
                    ref_nuc_comprehensive_dict[xmut] = refnuc
                    nuc_mut_int_string_dict[xmut] = string(pos_int)
                end
            end
        end
        precedessor_vec_join = join(predecessor_vec, ", ")
        if pango == "BA.3.2"
            println("BA.3.2 new_muts = $(new_muts)")
            println("BA.3.2 precedessor_vec_join = $(precedessor_vec_join)")
        end
#        println(precedessor_vec_join)
        for predecessor_j in predecessor_vec
            if (!haskey(nuc_genome_pango_dict, pango) || !haskey(pango_AAsub_private, pango)) && haskey(nuc_genome_pango_dict, predecessor_j) && length(nuc_genome_pango_dict[predecessor_j]) ≥ 28600 # && predecessor_j in pango_consensus_set && !(pango in forbidden_pangos)
                new_pango_ct += 1
                if new_pango_ct%100 == 0
                    println("################## NEW PANGO COUNT = $(new_pango_ct) #######################")
                end
                gene_AA_pango_dict[pango] = Dict{String, String}()
                for gene in gene_array
                    gene_AA_pango_dict[pango][gene] = gene_AA_pango_dict[predecessor_j][gene]
                end
####################################################################################################################################################
#                println(new_muts)
                synonymous_nuc_AA_sorted_dict, nonsynonymous_nuc_AA_sorted_dict, all_nuc_AA_sorted_dict = SIMPLER_nuc_to_AA(predecessor_j, new_muts)
####################################################################################################################################################
                sorted_nuc_muts = String[]
                if !isempty(all_nuc_AA_sorted_dict)
                    pango_nuc_sub_private[pango] = Set{String}()
                    pango_nuc_sub_WT[pango] = Set{String}()
                    pango_new_nuc_muts_pos_only[pango] = Set{String}()
                    sorted_nuc_muts = [all_nuc_AA_sorted_dict[i][1] for i in 1:length(all_nuc_AA_sorted_dict)]
                    for mut in sorted_nuc_muts
                        nuc_mut_int = nuc_mut_int_comprehensive_dict[mut]
                        WT_ref_nuc = string(wuhan_ref_seq[nuc_mut_int])
#                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
                        predecessor_ref_nuc = ref_nuc_comprehensive_dict[mut]
#                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        qry_nuc = qry_nuc_comprehensive_dict[mut]
                        nuc_mut_WT = ""
                        if WT_ref_nuc == predecessor_ref_nuc
                            nuc_mut_WT = mut
                        else
                            nuc_mut_WT = "$(WT_ref_nuc)$(nuc_mut_int)$(qry_nuc)"
                        end
                        push!(pango_nuc_sub_private[pango], mut)
                        push!(pango_nuc_sub_WT[pango], nuc_mut_WT)
                        push!(pango_new_nuc_muts_pos_only[pango], nuc_mut_int)
                    end
                end
###################################################################
                sorted_AA_muts = String[]
                if !isempty(nonsynonymous_nuc_AA_sorted_dict)
                    pango_AAsub_private[pango] = Set{String}()
                    pango_AAsub_WT[pango] = Set{String}()
                    pango_new_AA_muts_pos_only[pango] = Set{String}()
                    sorted_AA_muts = [nonsynonymous_nuc_AA_sorted_dict[i][2] for i in 1:length(nonsynonymous_nuc_AA_sorted_dict)]
                    for mut in sorted_AA_muts
                        mutgeen = aa_gene_comprehensive_dict[mut]
                        AAmut_int = aa_pos_comprehensive_dict[mut]
                        AAmut_pos_only = aa_gene_and_pos_comprehensive_dict[mut]
                        refAA = ref_AA_dict[mut]
                        qryAA = qry_AA_dict[mut]
                        new_gene_AA_seq = gene_AA_pango_dict[pango][mutgeen][1:AAmut_int-1]*qryAA*gene_AA_pango_dict[pango][mutgeen][AAmut_int+1:end]
                        gene_AA_pango_dict[pango][mutgeen] = new_gene_AA_seq
#                        pango_new_AA_muts[pango] = Set{String}()
#                        pango_new_AA_muts_WT[pango] = Set{String}()
#                        pango_new_AA_muts_pos_only[pango] = Set{String}()
                        WT_ref_AA = string(gene_AA_pango_dict[predecessor_j][mutgeen][AAmut_int])
                        AA_mut_WT = ""
                        if WT_ref_AA == refAA
                            AA_mut_WT = mut
                        else
                            AA_mut_WT = "$(mutgeen):$(WT_ref_AA)$(AAmut_int)$(qryAA)"
                        end 
                        push!(pango_AAsub_private[pango], mut)
                        push!(pango_AAsub_WT[pango], AA_mut_WT)
                        push!(pango_new_AA_muts_pos_only[pango], AAmut_pos_only)
                    end
                end
##################################################################
                nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_j]
                if !isempty(new_muts)
                    for mut in new_muts
                        if mut ≠ ""
                            qryNuc = qry_nuc_comprehensive_dict[mut]
                            nuc_int = nuc_mut_int_comprehensive_dict[mut]
                            new_nuc_seq = nuc_genome_pango_dict[pango][1:nuc_int-1]*qryNuc*nuc_genome_pango_dict[pango][nuc_int+1:end]
                            nuc_genome_pango_dict[pango] = new_nuc_seq
                        end
                    end
                    push!(pango_consensus_set, pango)
                end
                break
            end
        end
    end
end
#for pango in all_fucking_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        unaliased = pango_to_pango_unaliased_v2[pango]
#        println("No nuc_genome_pango_dict for $(pango) | $(unaliased)")
#    end
#end
#for pango in all_fucking_pangos
#    if !haskey(nuc_genome_pango_dict, pango)
#        dot_ct = count('.', unaliased)
#        if dot_ct ≥ 1
#            for i in 1:dot_ct
#                predecessor_i = pango_unaliased_predecessor_meta_dict[pango][i]
#                if haskey(nuc_genome_pango_dict, predecessor_i)
#                    nuc_genome_pango_dict[pango] = nuc_genome_pango_dict[predecessor_i]
#                    break
#                end
#            end
#        end
#    end
#end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
#########################################################__END NEW SECTION (2026-01-03)    #################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
############################### 2025-11-13 | Filling pango/clade/unaliased date/ct/country dicts  Runtime = 41.57 seconds ##################################################
############################################################################################################################################################################
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
index_len = length(values(tuple_to_index))
println("tuple_to_index_len = $(index_len)"); println("index_to_tuple[1] = $(index_to_tuple[1])"); println("index_to_tuple[2000] = $(index_to_tuple[2000])")
###########################################################################################################
pango_unaliased_date_index_ct = Dict{String, Dict{Int64, Int64}}()    
#                          clade     date_index  cumulative_count
clade_date_index_cumul = Dict{String, Dict{Int, Int}}()
pango_date_index_cumul = Dict{String, Dict{Int, Int}}()
pango_unaliased_date_index_cumul = Dict{String, Dict{Int, Int}}()
#                                country       clade    date_index  cumulative_count  
#country_clade_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
country_pango_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
country_pango_unaliased_date_index_cumul = Dict{String, Dict{String, Dict{Int, Int}}}()
#
clade_total = Dict{String, Int}()
pango_total = Dict{String, Int}()
pango_unaliased_total = Dict{String, Int}()
####################################################################################################
AAmut_date_index_cumul = Dict{String, Dict{Int, Int}}()
nucmut_date_index_cumul = Dict{String, Dict{Int, Int}}()
###################################################################################################################################
for clade in chronic_clade_set
    clade_date_index_cumul[clade] = Dict{Int, Int}()
    cumulative_clade_ct = 0
    for date_index in 1:2500
        if !haskey(clade_date_index_ct[clade], date_index)
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
        if haskey(clade_date_index_ct[clade], date_index)
            cumulative_clade_ct += clade_date_index_ct[clade][date_index]
            clade_date_index_cumul[clade][date_index] = cumulative_clade_ct
        end
    end
    clade_total[clade] = cumulative_clade_ct
end
################################################################################################################################################################
missing_pangos = Set{String}()
for pango in all_fucking_pangos
    if !haskey(pango_date_index_ct, pango)
        push!(missing_pangos, pango)
    end
end     
################################################################################################################################################################
for pango in all_fucking_pangos
    if pango in missing_pangos
        continue
    end
    pango_date_index_cumul[pango] = Dict{Int, Int}()
    cumulative_pango_ct = 0
    for date_index in 1:2500
        if !haskey(pango_date_index_ct[pango], date_index)
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
        if haskey(pango_date_index_ct[pango], date_index)
            cumulative_pango_ct += pango_date_index_ct[pango][date_index]
            pango_date_index_cumul[pango][date_index] = cumulative_pango_ct
        end
    end
    pango_total[pango] = cumulative_pango_ct
end
################################################################################
for pango in keys(pango_date_index_cumul)
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_date_index_cumul[unaliased] = Dict{Int, Int}()
    for date_index in 1:2500
        pango_unaliased_date_index_cumul[unaliased][date_index] = pango_date_index_cumul[pango][date_index]
    end
    pango_unaliased_total[unaliased] = pango_total[pango]
end
println("Done Filling Pango Date Index Cts!")
################################################################################
clade_pct_date_index = Dict{String, Dict{Float64, Int}}()
clade_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
clade_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_pct_date_index = Dict{String, Dict{Float64, Int}}()
pango_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
pango_pct_date_string = Dict{String, Dict{Float64, String}}()
#####
pango_unaliased_pct_date_index = Dict{String, Dict{Float64, Int}}()
pango_unaliased_pct_date_tuple = Dict{String, Dict{Float64, Tuple{Int, Int, Int}}}()
pango_unaliased_pct_date_string = Dict{String, Dict{Float64, String}}()
######################################################################################################################
for clade in chronic_clade_set
    clade_pct_date_index[clade] = Dict{Float64, Int}()
    clade_pct_date_tuple[clade] = Dict{Float64, Tuple{Int, Int, Int}}()
    clade_pct_date_string[clade] = Dict{Float64, String}()
    cp_total = clade_total[clade]
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#        pct = pct/10
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = clade_date_index_cumul[clade][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                clade_pct_date_index[clade][pct] = date_index
                clade_pct_date_tuple[clade][pct] = index_to_tuple[date_index]
                clade_pct_date_string[clade][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
##########################################################
for pango in all_fucking_pangos
    if pango in missing_pangos
        continue
    end
    pango_pct_date_index[pango] = Dict{Float64, Int}()
    pango_pct_date_tuple[pango] = Dict{Float64, Tuple{Int, Int, Int}}()
    pango_pct_date_string[pango] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            cumulative_ct = pango_date_index_cumul[pango][date_index]
            if 100*cumulative_ct/cp_total ≥ pct
                pango_pct_date_index[pango][pct] = date_index
                pango_pct_date_tuple[pango][pct] = index_to_tuple[date_index]
                pango_pct_date_string[pango][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_pct_date_index/tuple/string!")
##########################################################
for pango in keys(pango_pct_date_index)
    if pango in missing_pangos
        continue
    end
    unaliased = pango_to_pango_unaliased_v2[pango]
    pango_unaliased_pct_date_index[unaliased] = Dict{Float64, Int}()
    pango_unaliased_pct_date_tuple[unaliased] = Dict{Float64, Tuple{Int, Int, Int}}()
    pango_unaliased_pct_date_string[unaliased] = Dict{Float64, String}()
    cp_total = get(pango_total, pango, 0)
    pct_date_index = 0
    pct_date_tuple = nothing
#    for pct in 950:999
#    for pct in [10.0, 25.0, 50.0, 75.0, 90.0, 95.0, 98.0, 99.0, 99.5, 99.7, 99.8, 99.9]
    for pct in [1.0, 50.0, 95.0, 99.0]
        for date_index in 1:2500
            cumulative_ct = 0
            if pango == "B.1.1.529"
                cumulative_ct = pango_date_index_cumul["BA.2"][date_index]
            elseif pango == "LF.3.1"
                cumulative_ct = pango_date_index_cumul["LF.3"][date_index]
            else
                cumulative_ct = pango_date_index_cumul[pango][date_index]
            end
            if 100*cumulative_ct/cp_total ≥ pct
                pango_unaliased_pct_date_index[unaliased][pct] = date_index
                pango_unaliased_pct_date_tuple[unaliased][pct] = index_to_tuple[date_index]
                pango_unaliased_pct_date_string[unaliased][pct] = index_to_date_str[date_index]
                break
            end
        end
    end
end
println("Done Filling pango_unaliased_pct_date_index/tuple/string!")
###########################################################################################################################################################################
###########################################################################################################################################################################
all_chr_seqs_pangos = Dict{String, Int}()
all_chr_seqs_inherited = Dict{String, Int}()
all_chr_seqs_inherited_pos_only = Dict{String, Int}()
for seq in all_unique_chr_seqs
    pango = seq_pango[seq]
    all_chr_seqs_pangos[pango] = get(all_chr_seqs_pangos, pango, 0) + 1
    if !haskey(pango_AAsub_WT, pango)
        if haskey(pango_predecessor_meta_dict, pango)
##########################################################################################
            for i in 1:5
                if haskey(pango_predecessor_meta_dict[pango], i)
                    newpango = pango_predecessor_meta_dict[pango][i]
                    if haskey(pango_AAsub_WT, newpango)
                        pango_AAsub_WT[pango] = pango_AAsub_WT[newpango]
                    end
                end
            end
        end
    end
##########################################################################################
#            if haskey(pango_predecessor_meta_dict[pango], 1)
#                pango1 = pango_predecessor_meta_dict[pango][1]
#                if haskey(pango_AAsub_WT, pango1)
#                    pango_AAsub_WT[pango] = pango_AAsub_WT[pango1]
#                elseif !haskey(pango_AAsub_WT, pango1)
#                    if haskey(pango_predecessor_meta_dict, pango1)
#                        if haskey(pango_predecessor_meta_dict[pango1], 2)
#                            pango2 = pango_predecessor_meta_dict[pango][2]
#                            if haskey(pango_AAsub_WT, pango2)
#                                pango_AAsub_WT[pango] = pango_AAsub_WT[pango2]
#                            elseif !haskey(pango_AAsub_WT, pango2)
#                                if haskey(pango_predecessor_meta_dict, pango2)
#                                    if haskey(pango_predecessor_meta_dict[pango2], 3)
#                                        pango3 = pango_predecessor_meta_dict[pango][3]
#                                        if haskey(pango_AAsub_WT, pango3)
#                                            pango_AAsub_WT[pango] = pango_AAsub_WT[pango3]
#                                        elseif !haskey(pango_AAsub_WT, pango2)
#                                            if haskey(pango_predecessor_meta_dict, pango2)
#                                                if haskey(pango_predecessor_meta_dict[pango2], 3)
#                                                    pango3 = pango_predecessor_meta_dict[pango][3]
#                                                    if haskey(pango_AAsub_WT, pango3)
#                                                        pango_AAsub_WT[pango] = pango_AAsub_WT[pango3]
#                                                    end
#                                                end
#                                            end
#                                        end
#                                    end
#                                end
#                            end
#                        end
#                    end
#                end
#            end
#        end
#    end
    if pango == "B.1.1.529"
        pango_AAsub_WT[pango] = union(pango_AAsub_WT["BA.1"], pango_AAsub_WT["BA.2"])
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    else
        for mut in pango_AAsub_WT[pango]
            all_chr_seqs_inherited[mut] = get(all_chr_seqs_inherited, mut, 0) + 1
        end
    end
end
println("Done Filling all_chr_seqs_pangos, all_chr_seqs_inherited, pango_AAsub_WT (leftovers)!")
############################################################################################################################################################################
############################################################################################################################################################################
seq_syn_nucs = Dict{String, Vector{String}}()
seq_noncoding_nucs = Dict{String, Vector{String}}()
for (seq, nuc_set) in seq_nuc_muts
    pango = seq_pango[seq]
#    if !haskey(nuc_genome_pango_dict, pango)
#        println("No nuc_genome_pango_dict key, $(pango)")
#    end
    synonymous_nucmuts, noncoding_nucmuts = SIMPLER_syn_noncoding_nuc(pango, nuc_set)
    seq_syn_nucs[seq] = synonymous_nucmuts
    seq_noncoding_nucs[seq] = noncoding_nucmuts
end
println("Done Filling seq_syn_nucs, seq_noncoding_nucs!")
############################################################################################################################################################################
nuc_genome_pango_dict["B.1.1.273"] = nuc_genome_pango_dict["B.1.1"]
nuc_genome_pango_dict["B.1.1.53"] = nuc_genome_pango_dict["B.1.1"]
############################################################################################################################################################################
println("nuc_genome_pango_dict length = $(length(nuc_genome_pango_dict))")
println("Length all_fucking_pangos = $(length(all_fucking_pangos))")
print("\n"^1)
temp_nuc_sort_key(n) = (length(n), nuc_mut_int_comprehensive_dict[n])
nuc_mut_all_and_chr_set_missing_sort = sort(collect(nuc_mut_all_and_chr_set_missing), by = x -> temp_nuc_sort_key(x))
nuc_mut_all_and_chr_set_missing_sort_join = join(nuc_mut_all_and_chr_set_missing_sort, ", ")
missing_AA_dict_keys_sort = sort(collect(missing_AA_dict_keys), by = x -> AA_gene_sortKey_2(x))
missing_AA_dict_keys_sort_join = join(missing_AA_dict_keys_sort, ", ")
empty_string_pangos_sort = sort(collect(empty_string_pangos))
empty_string_pangos_sort_join = join(empty_string_pangos_sort, ", ")
open("missing_nuc_dict_keys_$(date_now).txt", "w") do g
    println(g, "nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    println("nuc_mut_all_and_chr_set_missing_sort_join = $(nuc_mut_all_and_chr_set_missing_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    println("missing_AA_dict_keys_sort_join = $(missing_AA_dict_keys_sort_join)")
    print("\n"^1)
    print(g, "\n"^1)
    println(g, "empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
    println("empty_string_pangos_sort_join = $(empty_string_pangos_sort_join)")
end
print("\n"^1)
finish = time() - start
finish_rd = round(digits=2, finish)
println("Total time to finish = $(finish_rd)"); print("\n"^1)
println("Finished!")
print("\n"^1)
############################################################################################################################################################################
############################################################################################################################################################################

2026_02_08_615PM
nuc_genome_pango_dict length = 4730
2026_02_08__615PM
Number of Different Pango Lineages in Chronics = 834
Number of Different Pango_Unaliased Lineages in Chronics = 834
Number of Different Clades in Chronics = 49

PA.4 = B.1.1.529.2.86.1.1.11.1.3.1.1.10.1.1.4
AY.127 = B.1.617.2.127
XBB.1.5.33 = XBB.1.5.33
XES.1 = XES.1
XBY = XBY
B.1.469 = B.1.469
BA.5.2.26 = B.1.1.529.5.2.26
MC.36 = B.1.1.529.2.86.1.1.11.1.3.1.1.36
XFG.26 = XFG.26
XBB.1.5.20 = XBB.1.5.20
BA.5.2.43 = B.1.1.529.5.2.43
BQ.1.1.23 = B.1.1.529.5.3.1.1.1.1.1.1.23
XAB = XAB
XE = XE
length(keys(pango_to_pango_unaliased_v2) = 4763
2026_02_08__615PM
2026_02_08__615PM
Done Making Pango Consensus Sequences! (Part 1)

Total Missing Keys for AA_pos_only_gene_and_pos_dict = 1
missing_keys_for_WT_join = ||

OG_tree_pangos_set_len = 4432
NEW_tree_pangos_set_len = 4372
bad_pangos length = 60
AY.43.8_18086, B.1.1.529_dropout, B.1.1.7_dropout, B.1.160.16_13923, BA.1_22877_22878, BA.1_dropout, BA.2.13_revs, BA.2.3.16_alt, 

In [64]:
### FX: reversion_percentage —— Calculating reversion frequency (as % of possible reversions) in EPCI
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function reversion_percentage(mut::String, mutrev::String)
    mut_pangos = Set{String}()
    for pango in chronic_pango_set
        if mut in pango_AAsub_WT[pango]
            push!(mut_pangos, pango)
        end
    end
    mut_pangos_sort = sort(collect(mut_pangos))
    mut_pangos_sort_join = join(mut_pangos_sort, ", ")
    println(mut_pangos_sort_join)
    print("\n"^1)
    mut_chr_seq_ct = 0
    for seq in all_unique_chr_seqs_set
        if seq_pango[seq] in mut_pangos
            mut_chr_seq_ct += 1
        end
    end
    println("$(mut)_chr_seq_ct = $(mut_chr_seq_ct)")
    print("\n"^2)
    mut_rev_ct = 0
    mut_chr_seqs = String[]
    mut_dropout_ct = 0
    mut_dropout_seqs = Set{String}()
    for seq in all_unique_chr_seqs_set
        if seq_pango[seq] in mut_pangos
#            print_all_seq_info(seq, "$(mut)_lineage_chronic_seqs_$(date_hour).txt")
            if mutrev in seq_AA_muts[seq]
                mut_rev_ct += 1
                push!(mut_chr_seqs, seq)
                if aa_gene_and_pos_comprehensive_dict[mut] in seq_unknown_AA[seq]
                    mut_dropout_ct += 1
                    push!(mut_dropout_seqs, seq)
                end
            end
        end
    end
    mut_chr_seqs_sort = sort(mut_chr_seqs, by = x -> (length(x), x))
    mut_chr_seqs_sort_join = join(mut_chr_seqs_sort, ", ")
    println("$(mutrev)_rev_ct = $(mut_rev_ct)/$(mut_chr_seq_ct)")
    println("mut_dropout_ct = $(mut_dropout_ct)")
    mut_rev_prop = mut_rev_ct/(mut_chr_seq_ct-mut_dropout_ct)
    println("$(mutrev)_pct = $(mut_rev_prop)")
    print("\n"^2)
    println("###### All $(mutrev) Reversion sequences #########")
    for seq in mut_chr_seqs_sort
#        println("        $(seq)")
    end
    print("\n"^2)
    println("###### All $(mutrev) dropout sequences #########")
    for seq in mut_dropout_seqs
        println("        $(seq)")
        if !isempty(seq_mixed_AA_muts[seq])
            for mixed_AA in seq_mixed_AA_muts[seq]
                println(mixed_AA)
            end
        end
    end
    print("\n"^2)
    return (mut_rev_ct, mut_chr_seq_ct, mut_rev_prop)
end
##############################################################################################################################
##############################################################################################################################

2026_02_25__955PM


reversion_percentage (generic function with 1 method)

In [65]:
### Making reversion percentage DataFrame
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
reversion_list = ["ORF1a:R135S", "ORF1a:K556Q", "ORF1a:I842T", "ORF1a:S1307G", "ORF1a:I3090T", "ORF1a:I3255T", "ORF1b:H264Y", "ORF1b:L314P", "ORF1b:V1566I", "S:N405D", "S:I19T", "S:F371S", "S:P373S", "S:F375S", "S:A376T", "S:T444K", "S:S446G", "S:W452L", "S:S455L", "S:K478T", "S:A484E", "S:V486F", "S:P486F", "S:S486F", "S:P486L", "S:S490F", "S:R493Q", "S:S496G", "S:R498Q", "S:H505Y", "S:Y655H", "S:K764N", "S:N950D", "E:I9T", "ORF7a:Y47H", "ORF7a:A82V", "N:L13P", "ORF9b:S10P", "ORF9b:F10S", "ORF9b:G16D"]     
reversion_pct_dict = Dict{String, Tuple{Int, Int, Float64}}()
for rev in reversion_list
    refAA = refAA_comprehensive_dict[rev]
    qryAA = qryAA_comprehensive_dict[rev]
    pos = aa_pos_comprehensive_dict[rev]
    gene = aa_gene_comprehensive_dict[rev]
    antirev = "$(gene):$(qryAA)$(pos)$(refAA)"
    if rev == "S:P486L"
        antirev = "S:F486P"
    end
    if rev == "ORF9b:F10S"
        antirev = "ORF9b:P10F"
    end
    println(antirev)
    revCt_seqCt_revPct = reversion_percentage(antirev, rev)
    reversion_pct_dict[rev] = revCt_seqCt_revPct
end
reversion_pct_dict_sort = sort(collect(reversion_pct_dict), by = x -> x[2][3], rev=true)
df_reversion_pct = DataFrame(
        Reversion = String[],
        Rev_Ct = Int[],
        Total_Ct = Int[],
        Rev_Pct = Float64[])
df_reversion_pct_sort = DataFrame(
        Reversion = String[],
        Rev_Ct = Int[],
        Total_Ct = Int[],
        Rev_Pct = Float64[])
for (rev, revCt_seqCt_revPct) in reversion_pct_dict
    rev_ct = revCt_seqCt_revPct[1]
    seq_ct = revCt_seqCt_revPct[2]
    rev_pct = revCt_seqCt_revPct[3]
    push!(df_reversion_pct, (rev, rev_ct, seq_ct, rev_pct))
end
for i in 1:length(reversion_pct_dict_sort)
    mut____revCt_seqCt_revPct = reversion_pct_dict_sort[i]
    mut = mut____revCt_seqCt_revPct[1]
    rev_ct = mut____revCt_seqCt_revPct[2][1]
    seq_ct = mut____revCt_seqCt_revPct[2][2]
    rev_pct = mut____revCt_seqCt_revPct[2][3]
    push!(df_reversion_pct_sort, (mut, rev_ct, seq_ct, rev_pct))
end
    
CSV.write("reversion_percentages_$(date_now).csv", df_reversion_pct)
CSV.write("reversion_percentages_pct_sort_$(date_now).csv", df_reversion_pct_sort)
#################################################################################################################################################################
#################################################################################################################################################################

2026_02_25__955PM
ORF1a:S135R
B.1.1.529, BA.2, BA.2.1, BA.2.10, BA.2.10.1, BA.2.12.1, BA.2.12.2, BA.2.14, BA.2.15, BA.2.17, BA.2.18, BA.2.2, BA.2.22, BA.2.3, BA.2.3.10, BA.2.3.11, BA.2.3.15, BA.2.3.2, BA.2.3.20, BA.2.3.21, BA.2.3.5, BA.2.3.7, BA.2.31, BA.2.35, BA.2.36, BA.2.38, BA.2.39, BA.2.40.1, BA.2.5, BA.2.53, BA.2.54, BA.2.56, BA.2.6, BA.2.61, BA.2.62, BA.2.64, BA.2.65, BA.2.68, BA.2.7, BA.2.75.2, BA.2.75.5, BA.2.76, BA.2.77, BA.2.80, BA.2.86.1, BA.2.86.2, BA.2.9, BA.2.9.1, BA.4, BA.4.1, BA.4.1.1, BA.4.1.9, BA.4.2, BA.4.3, BA.4.4, BA.4.5, BA.4.6, BA.4.6.1, BA.5, BA.5.1, BA.5.1.1, BA.5.1.10, BA.5.1.12, BA.5.1.14, BA.5.1.15, BA.5.1.17, BA.5.1.22, BA.5.1.23, BA.5.1.24, BA.5.1.25, BA.5.1.28, BA.5.1.3, BA.5.1.30, BA.5.1.33, BA.5.1.35, BA.5.1.4, BA.5.1.5, BA.5.1.6, BA.5.1.8, BA.5.1.9, BA.5.11, BA.5.2, BA.5.2.1, BA.5.2.12, BA.5.2.13, BA.5.2.14, BA.5.2.2, BA.5.2.20, BA.5.2.21, BA.5.2.22, BA.5.2.24, BA.5.2.25, BA.5.2.26, BA.5.2.28, BA.5.2.3, BA.5.2.33, BA.5.2.34, BA.5.2.35, BA.5.2.44, BA.5

"reversion_percentages_pct_sort_2026_02_25__955PM.csv"

In [89]:
### df_chronic_search_muts | Used to choose mutations overrepresented in chronic infections, which are then used to search for additional chronic sequences
############################################################################
df_chronic_search_muts_SCORE_sort = DataFrame(
    Rank = Int[],
    Mutation = String[],
    Ratio = Float64[],
    EPCI_ct = Int[],
    EPCI_score = Float64[],
    Pango_count = Int[])
df_chronic_search_muts_ratio_sort = DataFrame(
    Rank = Int[],
    Mutation = String[],
    Ratio = Float64[],
    EPCI_ct = Int[],
    EPCI_score = Float64[],
    Pango_count = Int[])
df_chronic_search_muts_gene_sort = DataFrame(
    Mutation = String[],
    Ratio = Float64[],
    EPCI_ct = Int[],
    EPCI_score = Float64[],
    Pango_ct = Int[])
############################################################################
AA_muts_ct_no_dels_SCORE = Dict{String, Float64}()  
for (mut, ratio) in AA_muts_ct_no_dels_no_revs_chr_all_ratio
    AAmutct = AA_muts_ct[mut]
    score = AAmutct*ratio
    AA_muts_ct_no_dels_SCORE[mut] = score
end
############################################################################
Double_N_ORF9b_muts = Set(["N:L13S", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
excluded_AA = Set(["ORF1a:I2107M", "ORF1a:N2111K", "ORF1a:S2114T", "ORF1a:S2151R", "ORF1a:N2155H", "ORF1a:T2152P", "ORF1a:T2153P", "ORF1a:T2154P", "ORF1a:T2158P", "ORF1b:N498I", "ORF1b:F685Y", "ORF1b:D1914E", "ORF1b:R1915G", "ORF1b:Y1916C", "ORF1b:P1917H", "S:S27A", "S:K113N", "S:Q115H", "S:N122K", "S:D142G", "S:Y145D", "S:S408R", "S:N417K", "S:K440N", "S:I794N"])
forbidden_mutations = union(Double_N_ORF9b_muts, excluded_AA)
min_ct1 = 3
min_comb_score = 100
date_pct_DQ_thresh = 95
absminmut = 7
DQ_mut_thresh = 9
rep_thresh = 5
revs_thresh = 6
date = "2026_02_02"
ndjson_name = "2026_02_01__6153seq_v2"
AA_muts_ct_no_dels_no_revs_chr_all_ratio_sort = sort(collect(AA_muts_ct_no_dels_no_revs_chr_all_ratio), by = x -> (x[2], AA_muts_ct_no_dels[x[1]]), rev=true)
rank_ratio_ct = 0
for AAmut___ratio in AA_muts_ct_no_dels_no_revs_chr_all_ratio_sort 
    mut = string(AAmut___ratio[1])
    ratio = AAmut___ratio[2]
    qry_AA = qry_AA_dict[mut]
    ref_AA = ref_AA_dict[mut]
    AAmutct = get(AA_muts_ct_no_dels, mut, 0)
    score = AA_muts_ct_no_dels_SCORE[mut]
    mut_pango_ct = 0
    if AAmutct ≥ min_ct1 
        if !(mut in forbidden_mutations)
            if score ≥ min_comb_score || ratio > 40
    #            for pango in all_fucking_pangos
                for (pango, WT_AA_mutset) in pango_AAsub_WT
                    if mut in WT_AA_mutset
                        mut_pango_ct += 1
                    end
                end
                if mut_pango_ct < 40
                    rank_ratio_ct += 1
                    if qry_AA ≠ ref_AA
                        push!(df_chronic_search_muts_ratio_sort, (rank_ratio_ct, mut, ratio, AAmutct, score, mut_pango_ct))
                        push!(chronic_search_muts_v2, mut)
                        mutpad = rpad(mut, 12)
                    else
                        push!(chronic_search_muts_v2, mut)
                        mutpad = rpad(mut, 12)
                        push!(df_chronic_search_muts_ratio_sort, (rank_ratio_ct, mut, ratio, AAmutct, score, mut_pango_ct))
                    end
                end
            end
        end
    end
end
CSV.write("df_chronic_search_muts_minCt$(min_ct1)_minscore$(score)_ratio_sort_$(ndjson_name)_DQmutThresh$(DQ_mut_thresh)_DQpct$(date_pct_DQ_thresh)_$(date_now).csv", df_chronic_search_muts_ratio_sort)
#####################################################################################################################################################################
AA_muts_ct_no_dels_no_revs_chr_all_gene_sort = sort(collect(AA_muts_ct_no_dels_no_revs_chr_all_ratio), by = x -> (AA_gene_sortKey_2(x[1])))
for AAmut___ratio in AA_muts_ct_no_dels_no_revs_chr_all_gene_sort 
    mut = string(AAmut___ratio[1])
    ratio = AAmut___ratio[2]
    qry_AA = qry_AA_dict[mut]
    ref_AA = ref_AA_dict[mut]
    AAmutct = get(AA_muts_ct_no_dels, mut, 0)
    score = AA_muts_ct_no_dels_SCORE[mut]
    mut_pango_ct = 0
    if AAmutct ≥ min_ct1 
        if !(mut in forbidden_mutations)
            if score ≥ min_comb_score || ratio > 40
    #            for pango in all_fucking_pangos
                for (pango, WT_AA_mutset) in pango_AAsub_WT
                    if mut in WT_AA_mutset
                        mut_pango_ct += 1
                    end
                end
                if mut_pango_ct < 40
                    if qry_AA ≠ ref_AA
                        push!(df_chronic_search_muts_gene_sort, (mut, ratio, AAmutct, score, mut_pango_ct))
                        push!(chronic_search_muts_v2, mut)
                        mutpad = rpad(mut, 12)
                    else
                        push!(df_chronic_search_muts_gene_sort, mut)
                        mutpad = rpad(mut, 12)
                        push!(df_chronic_search_muts, (mut, ratio, AAmutct, score, mut_pango_ct))
                    end
                end
            end
        end
    end
end
CSV.write("df_chronic_search_muts_minCt$(min_ct1)_minscore$(score)_gene_sort_$(ndjson_name)_DQmutThresh$(DQ_mut_thresh)_DQpct$(date_pct_DQ_thresh)_$(date_now).csv", df_chronic_search_muts_gene_sort)
###################################################################################################################################################################
AA_muts_ct_no_dels_SCORE_sort = sort(collect(AA_muts_ct_no_dels_SCORE), by = x -> x[2], rev=true)
rank_score_ct = 0
for AAmut___score in AA_muts_ct_no_dels_SCORE_sort 
    mut = string(AAmut___score[1])
    ratio = AA_muts_ct_no_dels_no_revs_chr_all_ratio[mut]
    qry_AA = qry_AA_dict[mut]
    ref_AA = ref_AA_dict[mut]
    AAmutct = get(AA_muts_ct_no_dels, mut, 0)
    score = AA_muts_ct_no_dels_SCORE[mut]
    mut_pango_ct = 0
    if AAmutct ≥ min_ct1 
        if !(mut in forbidden_mutations)
            if score ≥ min_comb_score || ratio > 40
    #            for pango in all_fucking_pangos
                for (pango, WT_AA_mutset) in pango_AAsub_WT
                    if mut in WT_AA_mutset
                        mut_pango_ct += 1
                    end
                end
                if mut_pango_ct < 40
                    rank_score_ct += 1
                    if qry_AA ≠ ref_AA
                        push!(df_chronic_search_muts_SCORE_sort, (rank_score_ct, mut, ratio, AAmutct, score, mut_pango_ct))
                        push!(chronic_search_muts_v2, mut)
                        mutpad = rpad(mut, 12)
                    else
                        push!(chronic_search_muts_v2, mut)
                        mutpad = rpad(mut, 12)
                        push!(df_chronic_search_muts_SCORE_sort, (rank_score_ct, mut, ratio, AAmutct, score, mut_pango_ct))
                    end
                end
            end
        end
    end
end
CSV.write("df_chronic_search_muts_minCt$(min_ct1)_minscore$(score)_SCORE_sort_$(ndjson_name)_DQmutThresh$(DQ_mut_thresh)_DQpct$(date_pct_DQ_thresh)_$(date_now).csv", df_chronic_search_muts_SCORE_sort)
##########################################################################################################################################################################
##########################################################################################################################################################################

2026_02_06__150PM


"df_chronic_search_muts_minCt3_minscorescore_SCORE_sort_2026_02_01__6153seq_DQmutThresh9_DQpct95_2026_02_06__150PM.csv"

In [29]:
### df_EPCI_HQCS_AA_ratio # Makes CSV file containing sorted list of EPCI mutations according to their EPCI/HQCS ratio
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd__I")
chr_all_ratio_sort_mut_vec = collect(x[1] for x in AA_muts_ct_chr_all_ratio_ct_sort)
chr_all_ratio_sort_ct_vec = collect(x[2] for x in AA_muts_ct_chr_all_ratio_ct_sort)
chr_all_ratio_genesort_mut_vec = collect(x[1] for x in AA_muts_ct_chr_all_ratio_pos_sort)
chr_all_ratio_genesort_ct_vec = collect(x[2] for x in AA_muts_ct_chr_all_ratio_pos_sort)

println(typeof(chr_all_ratio_sort_mut_vec))
df_chr_all_ratios = DataFrame(Mutation = String[], EPCI_to_HQCS_ratio = Float64[], EPCI_ct = Int[])
for i in 1:length(chr_all_ratio_sort_mut_vec)
    mut = chr_all_ratio_sort_mut_vec[i]
    push!(df_chr_all_ratios, (mut, chr_all_ratio_sort_ct_vec[i], AA_muts_ct[mut]))
end
CSV.write("df_EPCI_HQCS_AA_ratio__$(date_now).csv", df_chr_all_ratios)
df_chr_all_ratios_gene_sort = DataFrame(Mutation = String[], EPCI_to_HQCS_ratio = Float64[], EPCI_ct = Int[])
for i in 1:length(chr_all_ratio_genesort_mut_vec)
    mut = chr_all_ratio_genesort_mut_vec[i]
    push!(df_chr_all_ratios_gene_sort, (mut, chr_all_ratio_genesort_ct_vec[i], AA_muts_ct[mut]))
end
CSV.write("df_EPCI_HQCS_AA_ratio_sorted_by_genome_position__$(date_now).csv", df_chr_all_ratios_gene_sort)
    

2026_02_25__818PM
Vector{String}


"df_EPCI_HQCS_AA_ratio_sorted_by_genome_position__2026_02_25__818PM.csv"

In [8]:
#### These functions make plots for AA mut density in EPCI & HQCS datasets, as well as the EPCI/HQCS ratios for each gene & region; Used to make plots
########################################################################################################################################
### Fx: Plots Stuff (Large) ####################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp")
println(date_now)
gn(n) = split(n, ":")[1]
function AA_stretch_mut_density(gene::String, AAstretch::Int)
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    date = Dates.format(today(), "yyyy_mm_dd")
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct = Dict{String, Int}()
    ORF_range = ""
    for i in 1:AAstretch÷2:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
        mut_ct = 0
        key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
        if i + AAstretch-1 ≤ gene_protein_length[gene]
            for j in i:i+AAstretch-1
                if gene[1:3] ≠ "NSP"
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only, mut_pos)
                        mut_ct += AA_muts_ct_pos_only[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                end
            end
        end
        if i + AAstretch-1 > gene_protein_length[gene]
            res1 = gene_protein_length[gene] - AAstretch + 1
            key = gene*":"*string(res1)*"-"*string(gene_protein_length[gene])
            ORF_range = NSP_range_to_ORF1ab_range(key)
            for j in res1:gene_protein_length[gene]
                if gene[1:3] ≠ "NSP"
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only, mut_pos)
                        mut_ct += AA_muts_ct_pos_only[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only, mut_pos)
                            mut_ct += AA_muts_ct_pos_only[mut_pos]
                        end
                    end
                end
            end
        end
        AA_stretch_ct[key] = mut_ct                
    end
    sort_key(n) = (length(n), n)
    date = Dates.format(now(), "yyyy_mm_dd__IMMp")
    AA_stretch_ct_sort = sort(collect(AA_stretch_ct), by = x -> sort_key(x[1]))
    open("AA_density/$(date)_AA_dens_chr_$(gene)_$(AAstretch)AA.txt", "w") do g
        for w in AA_stretch_ct_sort
            stretch = w[1]
            count = w[2]
            println("                    ", stretch, " - ", count)
            println(g, "                    ", stretch, " - ", count)
        end
    end
end
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
function AA_stretch_mut_density_no_dels(gene::String, AAstretch::Int)
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    date = Dates.format(today(), "yyyy_mm_dd")
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct_no_dels = Dict{String, Int}()
    ORF_range = ""
    for i in 1:AAstretch÷2:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
        mut_ct = 0
        key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
        ORF_range = ""
        if i + AAstretch-1 ≤ gene_protein_length[gene]
            for j in i:i+AAstretch-1
                if gene[1:3] ≠ "NSP"
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                        mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                end
            end
        end
        if i + AAstretch-1 > gene_protein_length[gene]
            res1 = gene_protein_length[gene] - AAstretch + 1
            key = gene*":"*string(res1)*"-"*string(gene_protein_length[gene])
            ORF_range = NSP_range_to_ORF1ab_range(key)
            for j in res1:gene_protein_length[gene]
                if gene[1:3] ≠ "NSP"
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                        mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                    end
                end
            end
        end
        AA_stretch_ct_no_dels[key] = mut_ct                
    end
    sort_key(n) = (length(n), n)
    AA_stretch_ct_no_dels_sort = sort(collect(AA_stretch_ct_no_dels), by = x -> sort_key(x[1]))
    open("AA_density/$(date)_AA_dens_no_dels_chr_$(gene)_$(AAstretch).txt", "w") do g
        for w in AA_stretch_ct_no_dels_sort
            stretch = w[1]
            count = w[2]
            println("                    ", stretch, " - ", count)
            println(g, "                    ", stretch, " - ", count)
        end
    end
end
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
function AA_stretch_mut_density_no_dels_all(gene::String, AAstretch::Int)
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    new_date = Dates.format(now(), "yyyy_mm_dd__IMMp")
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct_no_dels = Dict{String, Int}()
    ORF_range = ""
    for i in 1:AAstretch÷2:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
        mut_ct = 0
        key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
        ORF_range = NSP_range_to_ORF1ab_range(key)
        if i + AAstretch-1 ≤ gene_protein_length[gene]
            for j in i:i+AAstretch-1
                if gene[1:3] ≠ "NSP"
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                        mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                end
            end
        end
        if i + AAstretch-1 > gene_protein_length[gene]
            res1 = gene_protein_length[gene] - AAstretch + 1
            key = gene*":"*string(res1)*"-"*string(gene_protein_length[gene])
            ORF_range = NSP_range_to_ORF1ab_range(key)
            for j in res1:gene_protein_length[gene]
                if gene[1:3] ≠ "NSP"
                    ORF_range = NSP_range_to_ORF1ab_range(key)
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                        mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                    end
                end
                if gene[1:3] == "NSP"
                    NSPnum = parse(Int, gene[4:end])
                    if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                        ORF1a_pos = j + NSP1a_add[NSPnum]
                        ORF1a_pos_str = string(ORF1a_pos)
                        mut_pos = "ORF1a"*":"*ORF1a_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum == 12 && j > 8
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if NSPnum > 12
                        ORF1b_pos = j + NSP1b_add[NSPnum]
                        ORF1b_pos_str = string(ORF1b_pos)
                        mut_pos = "ORF1b"*":"*ORF1b_pos_str
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                end
            end
        end
        AA_stretch_ct_no_dels[key] = mut_ct                
    end
    sort_key(n) = (length(n), n)
    AA_stretch_ct_no_dels_all_sort = sort(collect(AA_stretch_ct_no_dels), by = x -> sort_key(x[1]))
    open("AA_density/$(date)_AA_dens_no_dels_ALL_$(gene)_$(AAstretch)AA.txt", "w") do g
        for w in AA_stretch_ct_no_dels_all_sort
            stretch = w[1]
            count = w[2]
            println("                    ", stretch, " - ", count)
            println(g, "                    ", stretch, " - ", count)
        end
    end
end
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
function AA_stretch_mut_density_no_dels_chr_to_all_ratio(gene::String, AAstretch::Int)
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    new_date = Dates.format(now(), "yyyy_mm_dd__IMMp")
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct_no_dels_chr = Dict{String, Int}()
    AA_stretch_ct_no_dels_all = Dict{String, Int}()
    AA_stretch_ct_no_dels_chr_to_all_ratio = Dict{String, Float64}()
    tot_singlets = length(non_rep_seqs)
    tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
    tot_chronics = tot_singlets + tot_groups
    tot_all = qualifying_seq_ct_all
    ORF_range = ""
    for i in 1:AAstretch÷2:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
        mut_ct_chr = 0
        mut_ct_all = 0
        key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
        if i + AAstretch-1 ≤ gene_protein_length[gene]
            for j in i:i+AAstretch-1
                if length(gene) == 1
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                        mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                    end
                    if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                        mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                    end
                end
                if length(gene) > 1
                    if gene[1:3] ≠ "NSP"
                        pos = string(j)
                        mut_pos = gene*":"*pos
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if gene[1:3] == "NSP"
                        ORF_range = NSP_range_to_ORF1ab_range(key)
                        NSPnum = parse(Int, gene[4:end])
                        if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j > 8
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum > 12
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                    end
                end
            end
        end
        if i + AAstretch-1 > gene_protein_length[gene]
            res1 = gene_protein_length[gene] - AAstretch + 1
            key = gene*":"*string(res1)*"-"*string(gene_protein_length[gene])
            for j in res1:gene_protein_length[gene]
                if length(gene) == 1
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                        mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                    end
                    if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                        mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                    end
                end
                if length(gene) > 2
                    if gene[1:3] ≠ "NSP"
                        ORF_range = NSP_range_to_ORF1ab_range(key)
                        pos = string(j)
                        mut_pos = gene*":"*pos
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if gene[1:3] == "NSP"
                        NSPnum = parse(Int, gene[4:end])
                        if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j > 8
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum > 12
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                    end
                end
            end
        end
        AA_stretch_ct_no_dels_chr[key] = mut_ct_chr
        AA_stretch_ct_no_dels_all[key] = mut_ct_all
        chr_all_ratio = (mut_ct_chr/mut_ct_all)*(tot_all/tot_chronics)
        chr_all_ratio_rd = round(digits=2, chr_all_ratio)
        AA_stretch_ct_no_dels_chr_to_all_ratio[key] = chr_all_ratio_rd
    end    
    sort_key(n) = (length(n), n)
    AA_stretch_ct_no_dels_chr_to_all_ratio_sort = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio), by = x -> sort_key(x[1]) )
    open("AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_$(gene)_$(AAstretch)AA.txt", "w") do g
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            if '.' in count_str
                dec = split(count_str, ".")[2]
                if length(dec) == 1
                    count_str = count_str*"0"
                end
            end
            ct_ln = length(count_str)
            println("                    ", rpad("$(stretch)", 13), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
            println(g, "                    ", rpad("$(stretch)", 13), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
        end
    end
    return AA_stretch_ct_no_dels_chr_to_all_ratio, AA_stretch_ct_no_dels_chr_to_all_ratio_sort
end
#############################################################################################################################################################################
#############################################################################################################################################################################
function AA_stretch_mut_density_no_dels_chr_to_all_ratio_multi(genes::Vector{String}, AAstretch::Int)
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    new_date = Dates.format(now(), "yyyy_mm_dd__IMMp")
#    gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
#    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>18, "ORF6"=>21, "ORF7a"=>22, "ORF7b"=>23, "ORF8"=>24, "ORF9b"=>26, "S"=>17, "E"=>19, "M"=>20, "N"=>25)
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct_no_dels_chr = Dict{String, Int}()
    AA_stretch_ct_no_dels_all = Dict{String, Int}()
    AA_stretch_ct_no_dels_chr_to_all_ratio = Dict{String, Float64}()
    tot_singlets = length(non_rep_seqs)
    tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
    tot_chronics = tot_singlets + tot_groups
    tot_all = qualifying_seq_ct_all
    ORF_range = ""
    for gene in genes
        for i in 1:AAstretch÷2:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
            mut_ct_chr = 0
            mut_ct_all = 0
            key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
            if i + AAstretch-1 ≤ gene_protein_length[gene]
                for j in i:i+AAstretch-1
                    if length(gene) == 1
                        pos = string(j)
                        mut_pos = gene*":"*pos
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            prot_mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            prot_mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if length(gene) > 1
                        if gene[1:3] ≠ "NSP"
                            pos = string(j)
                            mut_pos = gene*":"*pos
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if gene[1:3] == "NSP"
                            ORF_range = NSP_range_to_ORF1ab_range(key)
                            NSPnum = parse(Int, gene[4:end])
                            if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                                ORF1a_pos = j + NSP1a_add[NSPnum]
                                ORF1a_pos_str = string(ORF1a_pos)
                                mut_pos = "ORF1a"*":"*ORF1a_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                                ORF1a_pos = j + NSP1a_add[NSPnum]
                                ORF1a_pos_str = string(ORF1a_pos)
                                mut_pos = "ORF1a"*":"*ORF1a_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum == 12 && j > 8
                                ORF1b_pos = j + NSP1b_add[NSPnum]
                                ORF1b_pos_str = string(ORF1b_pos)
                                mut_pos = "ORF1b"*":"*ORF1b_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum > 12
                                ORF1b_pos = j + NSP1b_add[NSPnum]
                                ORF1b_pos_str = string(ORF1b_pos)
                                mut_pos = "ORF1b"*":"*ORF1b_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                        end
                    end
                end
            end
            if i + AAstretch-1 > gene_protein_length[gene]
                res1 = gene_protein_length[gene] - AAstretch + 1
                key = gene*":"*string(res1)*"-"*string(gene_protein_length[gene])
                for j in res1:gene_protein_length[gene]
                    if length(gene) == 1
                        pos = string(j)
                        mut_pos = gene*":"*pos
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if length(gene) > 2
                        if gene[1:3] ≠ "NSP"
                            pos = string(j)
                            mut_pos = gene*":"*pos
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if gene[1:3] == "NSP"
                            ORF_range = NSP_range_to_ORF1ab_range(key)
                            NSPnum = parse(Int, gene[4:end])
                            if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                                ORF1a_pos = j + NSP1a_add[NSPnum]
                                ORF1a_pos_str = string(ORF1a_pos)
                                mut_pos = "ORF1a"*":"*ORF1a_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                                ORF1a_pos = j + NSP1a_add[NSPnum]
                                ORF1a_pos_str = string(ORF1a_pos)
                                mut_pos = "ORF1a"*":"*ORF1a_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum == 12 && j > 8
                                ORF1b_pos = j + NSP1b_add[NSPnum]
                                ORF1b_pos_str = string(ORF1b_pos)
                                mut_pos = "ORF1b"*":"*ORF1b_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                            if NSPnum > 12
                                ORF1b_pos = j + NSP1b_add[NSPnum]
                                ORF1b_pos_str = string(ORF1b_pos)
                                mut_pos = "ORF1b"*":"*ORF1b_pos_str
                                if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                    mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                                end
                                if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                    mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                                end
                            end
                        end
                    end
                end
            end
            AA_stretch_ct_no_dels_chr[key] = mut_ct_chr
            AA_stretch_ct_no_dels_all[key] = mut_ct_all
            chr_all_ratio = (mut_ct_chr/mut_ct_all)*(tot_all/tot_chronics)
            chr_all_ratio_rd = round(digits=2, chr_all_ratio)
            AA_stretch_ct_no_dels_chr_to_all_ratio[key] = chr_all_ratio_rd
        end
    end
#############################################################################################################################################################################
    sort_key(n) = (gene_protein_order[gn(n)], length(n), n)
    AA_stretch_ct_no_dels_chr_to_all_ratio_sort = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio), by = x -> sort_key(x[1]) )
    AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio), by = x -> x[2], rev=true)
    open("AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_MULTI_gene_sort_$(AAstretch)AA_.txt", "w") do g
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println("                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
        end
    end
    open("$(date)/AA_density/AA_density_no_dels_chrALL_RATIO_MULTI_ratio_sort_$(AAstretch)AA_$(date).txt", "w") do g
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println("                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), lpad("$(ORF_range)", 18) )
        end
    end
    return AA_stretch_ct_no_dels_chr_to_all_ratio
end
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
function AA_stretch_mut_density_no_dels_chr_to_all_ratio_multi_slide(genes::Vector{String}, AAstretch::Int, slide::Int)
    start = time()
    AA_res_set = Set(["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "-"])
    date = Dates.format(now(), "yyyy_mm_dd__IMMp")
#    gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "ORF3a"=>17, "ORF6"=>18, "ORF7a"=>19, "ORF7b"=>20, "ORF8"=>21, "ORF9b"=>22, "S"=>23, "E"=>24, "M"=>25, "N"=>26)
#    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    gene_protein_order = Dict{String, Int}("NSP1"=>1, "NSP2"=>2, "NSP3"=>3, "NSP4"=>4, "NSP5"=>5, "NSP6"=>6, "NSP7"=>7, "NSP8"=>8, "NSP9"=>9, "NSP10"=>10, "NSP12"=>12, "NSP13"=>13, "NSP14"=>14, "NSP15"=>15, "NSP16"=>16, "S"=>17, "ORF3a"=>18, "E"=>19, "M"=>20, "ORF6"=>21, "ORF7a"=>22, "ORF7b"=>23, "ORF8"=>24, "N"=>25, "ORF9b"=>26)
    gene_protein_order_reverse = Dict{Int, String}()
    for (k, v) in gene_protein_order
        gene_protein_order_reverse[v] = k
    end
    gene_protein_length = Dict{String, Int}("NSP1"=>180, "NSP2"=>638, "NSP3"=>1945, "NSP4"=>500, "NSP5"=>306, "NSP6"=>290, "NSP7"=>83, "NSP8"=>198, "NSP9"=>113, "NSP10"=>139, "NSP12"=>932, "NSP13"=>601, "NSP14"=>527, "NSP15"=>346, "NSP16"=>298, "ORF3a"=>275, "ORF6"=>61, "ORF7a"=>121, "ORF7b"=>43, "ORF8"=>121, "ORF9b"=>97, "S"=>1273, "E"=>75, "M"=>222, "N"=>419)
    NSP1a_add = Dict{Int, Int}(1 => 0, 2 => 180, 3 => 818, 4 => 2763, 5 => 3263, 6 => 3569, 7 => 3859, 8 => 3942, 9 => 4140, 10 => 4253, 11 => 0, 12 => 4392)
    NSP1b_add = Dict{Int, Int}(12 => -9, 13 => 923, 14 => 1424, 15 => 2051, 16 => 2397)
    AA_stretch_ct_no_dels_chr = Dict{String, Int}()
    AA_stretch_ct_no_dels_all = Dict{String, Int}()
    AA_stretch_ct_no_dels_chr_to_all_ratio = Dict{String, Float64}()
#######################################################################
# The adjusted dict below adjust for the greater number of AA muts per chronic seq compared to circ seq
    adj_factor = avg_private_AA_per_circ_seq/chronic_avg_AA_subs_per_seq
    AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted = Dict{String, Float64}()
#######################################################################
    tot_singlets = length(non_rep_seqs)
    tot_groups = length(keys(rep_seq_grps_maxmut_seqs) )
    tot_chronics = tot_singlets + tot_groups
    tot_all = qualifying_seq_ct_all
    ORF_range = ""
    NSP_chr_to_all_ratio = Dict{String, Dict{String, Float64}}()
    NSP_chr_to_all_ratio_adjusted = Dict{String, Dict{String, Float64}}()
    NSP_chr = Dict{String, Dict{String, Int}}()
    NSP_all = Dict{String, Dict{String, Int}}()
    NSP_chr_per_seq = Dict{String, Dict{String, Float64}}()
    NSP_all_per_seq = Dict{String, Dict{String, Float64}}()
    NSP_ratio_dict1 = Dict{String, Float64}()
    for gene in genes
        NSP_chr_to_all_ratio[gene] = Dict{String, Float64}()
        NSP_chr_to_all_ratio_adjusted[gene] = Dict{String, Float64}()
    end
    for gene in genes
        NSP_chr[gene] = Dict{String, Int}()
        NSP_all[gene] = Dict{String, Int}()
        NSP_chr_per_seq[gene] = Dict{String, Float64}()
        NSP_all_per_seq[gene] = Dict{String, Float64}()
    end
    for gene in genes
        for i in 1:slide:gene_protein_length[gene]-AAstretch + slide
        #for i in 1:slide:gene_protein_length[gene]-gene_protein_length[gene]%(AAstretch÷2)
            mut_ct_chr = 0
            mut_ct_all = 0
            key = gene*":"*string(i)*"-"*string(i+AAstretch-1)
            NSP_rng = string(i)*"-"*string(i+AAstretch-1)
#            if i + AAstretch-1 ≤ gene_protein_length[gene]
            for j in i:min(i+AAstretch-1, gene_protein_length[gene])
                if length(gene) == 1
                    pos = string(j)
                    mut_pos = gene*":"*pos
                    if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                        mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                    end
                    if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                        mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                    end
                end
                if length(gene) > 1
                    if gene[1:3] ≠ "NSP"
                        pos = string(j)
                        mut_pos = gene*":"*pos
                        if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                            mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                        end
                        if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                            mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                        end
                    end
                    if gene[1:3] == "NSP"
                        ORF_range = NSP_range_to_ORF1ab_range(key)
                        NSPnum = parse(Int, gene[4:end])
                        if NSPnum in [1,2,3,4,5,6,7,8,9,10]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j in [1,2,3,4,5,6,7,8]
                            ORF1a_pos = j + NSP1a_add[NSPnum]
                            ORF1a_pos_str = string(ORF1a_pos)
                            mut_pos = "ORF1a"*":"*ORF1a_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum == 12 && j > 8
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                        if NSPnum > 12
                            ORF1b_pos = j + NSP1b_add[NSPnum]
                            ORF1b_pos_str = string(ORF1b_pos)
                            mut_pos = "ORF1b"*":"*ORF1b_pos_str
                            if haskey(AA_muts_ct_pos_only_no_dels, mut_pos)
                                mut_ct_chr += AA_muts_ct_pos_only_no_dels[mut_pos]
                            end
                            if haskey(AA_muts_ct_pos_only_no_dels_all, mut_pos)
                                mut_ct_all += AA_muts_ct_pos_only_no_dels_all[mut_pos]
                            end
                        end
                    end
                end
            end
            AA_stretch_ct_no_dels_chr[key] = mut_ct_chr
            AA_stretch_ct_no_dels_all[key] = mut_ct_all
            
            chr_all_ratio = (mut_ct_chr/mut_ct_all)*(tot_all/tot_chronics)
            chr_all_ratio_rd = round(digits=2, chr_all_ratio)

            chr_all_ratio_adjusted = chr_all_ratio*adj_factor
            chr_all_ratio_adjusted_rd = round(digits=2, chr_all_ratio_adjusted)
            
#            chr_all_ratio_adjusted = (mut_ct_chr/mut_ct_all)*(tot_all/tot_chronics)*(avg_private_AA_per_circ_seq/chronic_avg_AA_subs_per_seq)           

            AA_stretch_ct_no_dels_chr_to_all_ratio[key] = chr_all_ratio_rd
            NSP_chr_to_all_ratio[gene][NSP_rng] = chr_all_ratio_rd

            AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted[key] = chr_all_ratio_adjusted_rd
            NSP_chr_to_all_ratio_adjusted[gene][NSP_rng] = chr_all_ratio_adjusted_rd

            
            NSP_chr[gene][NSP_rng] = mut_ct_chr
            NSP_all[gene][NSP_rng] = mut_ct_all
            NSP_chr_per_seq[gene][NSP_rng] = mut_ct_chr/tot_chronics
            NSP_all_per_seq[gene][NSP_rng] = mut_ct_all/tot_all
        end
    end
#############################################################################################################
############################################################################################################# 
    sort_key(n) = (gene_protein_order[gn(n)], length(n), n)
    srtky(n) = (length(n), n) 
    AA_stretch_ct_no_dels_chr_to_all_ratio_sort = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio), by = x -> sort_key(x[1]))
    AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio), by = x -> x[2], rev=true)

    AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted_sort = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted), by = x -> sort_key(x[1]))
    AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted_sort_by_ratio = sort(collect(AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted), by = x -> x[2], rev=true)
############################################################################################################# 
#    log1_5(x) = log(x)/log(1.5)
    NSP_chr_vec = Vector{Tuple{String, String, Int}}()
    for (gene, inner_dict) in NSP_chr
        for (NSP_rng, mut_ct_chr) in inner_dict
            push!(NSP_chr_vec, (gene, NSP_rng, mut_ct_chr) )
        end
    end
    NSP_chr_vec_sort = sort(NSP_chr_vec, by = x -> (gene_protein_order[x[1]], srtky(x[2])) )
    NSP_chr_sort_final = Vector{Tuple{String, Int}}()
    for tupl in NSP_chr_vec_sort
        mutstring = tupl[1]*":"*tupl[2]
        push!(NSP_chr_sort_final, (mutstring, tupl[3]) )
    end
    NSP1_dens = Vector{Tuple{String, Float64}}(); NSP2_dens = Vector{Tuple{String, Float64}}(); 
    NSP3_dens = Vector{Tuple{String, Float64}}(); NSP4_dens = Vector{Tuple{String, Float64}}();
    NSP5_dens = Vector{Tuple{String, Float64}}(); NSP6_dens = Vector{Tuple{String, Float64}}()
    NSP7_dens = Vector{Tuple{String, Float64}}(); NSP8_dens = Vector{Tuple{String, Float64}}()
    NSP9_dens = Vector{Tuple{String, Float64}}(); NSP10_dens = Vector{Tuple{String, Float64}}()
    NSP11_dens = Vector{Tuple{String, Float64}}(); NSP12_dens = Vector{Tuple{String, Float64}}()
    NSP13_dens = Vector{Tuple{String, Float64}}(); NSP14_dens = Vector{Tuple{String, Float64}}()
    NSP15_dens = Vector{Tuple{String, Float64}}(); NSP16_dens = Vector{Tuple{String, Float64}}()
    S_dens = Vector{Tuple{String, Float64}}(); ORF3a_dens = Vector{Tuple{String, Float64}}()
    E_dens = Vector{Tuple{String, Float64}}(); M_dens = Vector{Tuple{String, Float64}}()
    ORF6_dens = Vector{Tuple{String, Float64}}(); ORF7a_dens = Vector{Tuple{String, Float64}}()
    ORF7b_dens = Vector{Tuple{String, Float64}}(); ORF8_dens = Vector{Tuple{String, Float64}}()
    N_dens = Vector{Tuple{String, Float64}}(); ORF9b_dens = Vector{Tuple{String, Float64}}()
    prot_dens_vec = [NSP1_dens, NSP2_dens, NSP3_dens, NSP4_dens, NSP5_dens, NSP6_dens, NSP7_dens, NSP8_dens, NSP9_dens, NSP10_dens, NSP11_dens, NSP12_dens, NSP13_dens, NSP14_dens, NSP15_dens, NSP16_dens, S_dens, ORF3a_dens, E_dens, M_dens, ORF6_dens, ORF7a_dens, ORF7b_dens, ORF8_dens, N_dens, ORF9b_dens]
    push!(NSP1_dens, NSP_chr_sort_final[1])
    dens_ct = 1
    for i in 2:length(NSP_chr_sort_final) 
        tupl = NSP_chr_sort_final[i]
        tupl_minus = NSP_chr_sort_final[i-1]
        tupl_prot = split(tupl[1], ":")[1]
        tupl_minus_prot = split(tupl_minus[1], ":")[1]
        if tupl_prot == tupl_minus_prot
            push!(prot_dens_vec[dens_ct], tupl)
        end
        if tupl_prot ≠ tupl_minus_prot
            if dens_ct == 10
                dens_ct += 2
            else
            dens_ct += 1
            push!(prot_dens_vec[dens_ct], tupl)
            end
        end
    end
    max_dens_dict = Dict{Int, Float64}(i=>0 for i in 1:length(prot_dens_vec))
    genome_max_dens = 0
    for i in 1:length(prot_dens_vec)
        for (range, v) in prot_dens_vec[i]
            if v > genome_max_dens
                genome_max_dens = v
            end
            if v > max_dens_dict[i]
                max_dens_dict[i] = v
            end
        end
    end
    println("Maximum AA-Sub Density Value in Genome = $(genome_max_dens)")
############################################################################################################# 
    NSP_chr_to_all_ratio_vec = Vector{Tuple{String, String, Float64}}()
    for (gene, inner_dict) in NSP_chr_to_all_ratio
        for (NSP_rng, chr_all_ratio_rd) in inner_dict
            push!(NSP_chr_to_all_ratio_vec, (gene, NSP_rng, chr_all_ratio_rd) )
        end
    end
    NSP_chr_to_all_ratio_vec_sort = sort(NSP_chr_to_all_ratio_vec, by = x -> (gene_protein_order[x[1]], srtky(x[2])) )
    NSP_chr_to_all_ratio_sort_final = Vector{Tuple{String, Float64}}()
    for tupl in NSP_chr_to_all_ratio_vec_sort
        mutstring = tupl[1]*":"*tupl[2]
        push!(NSP_chr_to_all_ratio_sort_final, (mutstring, tupl[3]) )
    end
    NSP1_dens_ratio = Vector{Tuple{String, Float64}}(); NSP2_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP3_dens_ratio = Vector{Tuple{String, Float64}}(); NSP4_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP5_dens_ratio = Vector{Tuple{String, Float64}}(); NSP6_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP7_dens_ratio = Vector{Tuple{String, Float64}}(); NSP8_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP9_dens_ratio = Vector{Tuple{String, Float64}}(); NSP10_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP11_dens_ratio = Vector{Tuple{String, Float64}}(); NSP12_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP13_dens_ratio = Vector{Tuple{String, Float64}}(); NSP14_dens_ratio = Vector{Tuple{String, Float64}}()
    NSP15_dens_ratio = Vector{Tuple{String, Float64}}(); NSP16_dens_ratio = Vector{Tuple{String, Float64}}()
    S_dens_ratio = Vector{Tuple{String, Float64}}(); ORF3a_dens_ratio = Vector{Tuple{String, Float64}}()
    E_dens_ratio = Vector{Tuple{String, Float64}}(); M_dens_ratio = Vector{Tuple{String, Float64}}()
    ORF6_dens_ratio = Vector{Tuple{String, Float64}}(); ORF7a_dens_ratio = Vector{Tuple{String, Float64}}()
    ORF7b_dens_ratio = Vector{Tuple{String, Float64}}(); ORF8_dens_ratio = Vector{Tuple{String, Float64}}()
    N_dens_ratio = Vector{Tuple{String, Float64}}(); ORF9b_dens_ratio = Vector{Tuple{String, Float64}}()
    prot_dens_ratio_vec = [NSP1_dens_ratio, NSP2_dens_ratio, NSP3_dens_ratio, NSP4_dens_ratio, NSP5_dens_ratio, NSP6_dens_ratio, NSP7_dens_ratio, NSP8_dens_ratio, NSP9_dens_ratio, NSP10_dens_ratio, NSP11_dens_ratio, NSP12_dens_ratio, NSP13_dens_ratio, NSP14_dens_ratio, NSP15_dens_ratio, NSP16_dens_ratio, S_dens_ratio, ORF3a_dens_ratio, E_dens_ratio, M_dens_ratio, ORF6_dens_ratio, ORF7a_dens_ratio, ORF7b_dens_ratio, ORF8_dens_ratio, N_dens_ratio, ORF9b_dens_ratio]
#############################################################################################################
    push!(NSP1_dens_ratio, NSP_chr_to_all_ratio_sort_final[1])
    dens_ratio_ct = 1
    for i in 2:length(NSP_chr_to_all_ratio_sort_final) 
        tupl = NSP_chr_to_all_ratio_sort_final[i]
        tupl_minus = NSP_chr_to_all_ratio_sort_final[i-1]
        tupl_prot = split(tupl[1], ":")[1]
        tupl_minus_prot = split(tupl_minus[1], ":")[1]
        if tupl_prot == tupl_minus_prot
            push!(prot_dens_ratio_vec[dens_ratio_ct], tupl)
        end
        if tupl_prot ≠ tupl_minus_prot
            if dens_ratio_ct == 10
                dens_ratio_ct += 2
            else
            dens_ratio_ct += 1
            push!(prot_dens_ratio_vec[dens_ratio_ct], tupl)
            end
        end
    end
    max_dens_ratio_dict = Dict{Int, Float64}(i=>0 for i in 1:length(prot_dens_ratio_vec))
    genome_max_ratio_dens = 0
    for i in 1:length(prot_dens_ratio_vec)
        for (range, v) in prot_dens_ratio_vec[i]
            if v > genome_max_ratio_dens
                genome_max_ratio_dens = v
            end
            if v > max_dens_ratio_dict[i]
                max_dens_ratio_dict[i] = v
            end
        end
    end
    println("Maximum Ratio Value in entire genome = $(genome_max_ratio_dens)")
    genome_max_ratio_dens = genome_max_ratio_dens*1.01
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#################################### Begin Adjusted Ratio Version ###########################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
#############################################################################################################
    NSP_chr_to_all_ratio_adjusted_vec = Vector{Tuple{String, String, Float64}}()
    for (gene, inner_dict) in NSP_chr_to_all_ratio_adjusted
        for (NSP_rng, chr_all_ratio_adjusted_rd) in inner_dict
            push!(NSP_chr_to_all_ratio_adjusted_vec, (gene, NSP_rng, chr_all_ratio_adjusted_rd) )
        end
    end
    NSP_chr_to_all_ratio_adjusted_vec_sort = sort(NSP_chr_to_all_ratio_adjusted_vec, by = x -> (gene_protein_order[x[1]], srtky(x[2])) )
    NSP_chr_to_all_ratio_adjusted_sort_final = Vector{Tuple{String, Float64}}()
    for tupl in NSP_chr_to_all_ratio_adjusted_vec_sort
        mutstring = tupl[1]*":"*tupl[2]
        push!(NSP_chr_to_all_ratio_adjusted_sort_final, (mutstring, tupl[3]) )
    end
    NSP1_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP2_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP3_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP4_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP5_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP6_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP7_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP8_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP9_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP10_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP11_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP12_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP13_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP14_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    NSP15_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); NSP16_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    S_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); ORF3a_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    E_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); M_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    ORF6_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); ORF7a_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    ORF7b_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); ORF8_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    N_dens_ratio_adjusted = Vector{Tuple{String, Float64}}(); ORF9b_dens_ratio_adjusted = Vector{Tuple{String, Float64}}()
    prot_dens_ratio_adjusted_vec = [NSP1_dens_ratio_adjusted, NSP2_dens_ratio_adjusted, NSP3_dens_ratio_adjusted, NSP4_dens_ratio_adjusted, NSP5_dens_ratio_adjusted, NSP6_dens_ratio_adjusted, NSP7_dens_ratio_adjusted, NSP8_dens_ratio_adjusted, NSP9_dens_ratio_adjusted, NSP10_dens_ratio_adjusted, NSP11_dens_ratio_adjusted, NSP12_dens_ratio_adjusted, NSP13_dens_ratio_adjusted, NSP14_dens_ratio_adjusted, NSP15_dens_ratio_adjusted, NSP16_dens_ratio_adjusted, S_dens_ratio_adjusted, ORF3a_dens_ratio_adjusted, E_dens_ratio_adjusted, M_dens_ratio_adjusted, ORF6_dens_ratio_adjusted, ORF7a_dens_ratio_adjusted, ORF7b_dens_ratio_adjusted, ORF8_dens_ratio_adjusted, N_dens_ratio_adjusted, ORF9b_dens_ratio_adjusted]
#############################################################################################################
    push!(NSP1_dens_ratio_adjusted, NSP_chr_to_all_ratio_adjusted_sort_final[1])
    dens_ratio_adjusted_ct = 1
    for i in 2:length(NSP_chr_to_all_ratio_adjusted_sort_final) 
        tupl = NSP_chr_to_all_ratio_adjusted_sort_final[i]
        tupl_minus = NSP_chr_to_all_ratio_adjusted_sort_final[i-1]
        tupl_prot = split(tupl[1], ":")[1]
        tupl_minus_prot = split(tupl_minus[1], ":")[1]
        if tupl_prot == tupl_minus_prot
            push!(prot_dens_ratio_adjusted_vec[dens_ratio_adjusted_ct], tupl)
        end
        if tupl_prot ≠ tupl_minus_prot
            if dens_ratio_adjusted_ct == 10
                dens_ratio_adjusted_ct += 2
            else
            dens_ratio_adjusted_ct += 1
            push!(prot_dens_ratio_adjusted_vec[dens_ratio_adjusted_ct], tupl)
            end
        end
    end
    max_dens_ratio_adjusted_dict = Dict{Int, Float64}(i=>0 for i in 1:length(prot_dens_ratio_adjusted_vec))
    genome_max_ratio_adjusted_dens = 0
    for i in 1:length(prot_dens_ratio_adjusted_vec)
        for (range, v) in prot_dens_ratio_adjusted_vec[i]
            if v > genome_max_ratio_adjusted_dens
                genome_max_ratio_adjusted_dens = v
            end
            if v > max_dens_ratio_adjusted_dict[i]
                max_dens_ratio_adjusted_dict[i] = v
            end
        end
    end
    println("Maximum ratio_adjusted Value in entire genome = $(genome_max_ratio_adjusted_dens)")
    genome_max_ratio_adjusted_dens = genome_max_ratio_adjusted_dens*1.01
#############################################################################################################
#############################################################################################################
#############################################################################################################
##################################### End Adjusted Ratio Version ############################################
#############################################################################################################
#############################################################################################################
############################################################################################################# 
    function xval_xtick_label(prot_rng)
#        prot = split(prot_rng, ":")[1]
        rng = split(prot_rng, ":")[2]
        site1 = parse(Int, split(rng, "-")[1])
        site2 = parse(Int, split(rng, "-")[2])
        mid = (site1 + site2)÷2
        mid_str = string(mid)
#        xval_label = prot*":"*mid_str
        xval_label = mid_str
        return xval_label
    end
    function xtik_font_size(prot_AA_length::Int)
        size = 11
        if prot_AA_length ≥ 1 && prot_AA_length < 100
            size = 15
        end
        if prot_AA_length ≥ 101 && prot_AA_length < 160
            size = 14
        end
        if prot_AA_length ≥ 161 && prot_AA_length < 295
            size = 13
        end
        if prot_AA_length ≥ 296 && prot_AA_length < 501
            size = 12
        end
        if prot_AA_length ≥ 502 && prot_AA_length < 2000
            size = 11
        end
        if prot_AA_length ≥ 2001
            size = 4
        end
        return size
    end
#############################################################################################################
#$$    for i in 1:length(prot_dens_ratio_vec)
#$$        if i ≠ 11
#$$            xval = [x[1] for x in prot_dens_ratio_vec[i]]
#$$            yval = [max(y[2], 0) for y in prot_dens_ratio_vec[i]]
#$$            xind = 1:length(xval)
#$$            xinterval = max(2, (length(xval)÷100)+2)
#$$            xtick_ind = collect(1:xinterval:length(xval))
#$$            xfont = xtik_font_size(length(xval))
#$$            xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$            y_max = max(genome_max_ratio_dens÷3, max_dens_ratio_dict[i])
#$$        p = Plots.plot(xind, yval,  
#$$            seriestype=:line, linewidth=2, size=(2000, 800), dpi=600, xticks=(xtick_ind, xtick_labels),
#$$            xlabel="Protein + AA Site", ylabel="EPCI:HQCS AA-Sub Density Ratio", 
#$$            xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),
#$$
#$$            title="$(gene_protein_order_reverse[i])", label="EPCI:HQCS AA-Sub Density Ratio",
#$$            legend=:topleft, background_color_inside=:white, background_color_outside=:white, 
#$$            linecolor=:darkred, xrotation=90, ytickfontsize=15, xtickfontsize=xfont,
#$$            ylims=(0, y_max), tickfontfamily="Helvetica Bold", ygrid=false, 
#$$            gridcolor=:gray, gridalpha=0.5)
#$$            Plots.plot!(p, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_$(gene_protein_order_reverse[i])__$(AAstretch)AA.png")
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_$(gene_protein_order_reverse[i])__$(AAstretch)AA.svg")
#            display(p)
#$$        end
#$$    end
#############################################################################################################
#$$    for i in 1:length(prot_dens_vec)
#$$        if i ≠ 11            
#$$            xval = [x[1] for x in prot_dens_vec[i]]
#$$            yval = [max(y[2], 0) for y in prot_dens_vec[i]]
#$$            xinterval = max(2, (length(xval)÷100 + 2) )
#$$            xtick_ind = collect(1:xinterval:length(xval))
#$$            xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$            xind = 1:length(xval)
#$$            xfont = xtik_font_size(length(xval))
#$$            y_max = max(genome_max_dens÷3, max_dens_dict[i])
#$$        p = Plots.plot(xind, yval, 
#$$            seriestype=:line, linewidth=2, size=(2000, 800), dpi=600, xticks=(xtick_ind, xtick_labels), 
#$$            xlabel="Protein + AA Site", ylabel="EPCI AA-sub density", xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),
#$$            title="$(gene_protein_order_reverse[i])", label="EPCI AA-sub density",  titlefont=("Helvetica Bold", 28),
#$$
#$$            background_color_inside=:white, background_color_outside=:white, linecolor=:darkblue, 
#$$            xrotation=90, ylims=(0, y_max), tickfontfamily="Helvetica Bold", 
#$$
#$$            ytickfontsize=15, xtickfontsize=xfont, ygrid=false, gridcolor=:gray, gridalpha=0.5)
#$$            Plots.plot!(p, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chr_$(gene_protein_order_reverse[i])__$(AAstretch)AA.png")
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chr_$(gene_protein_order_reverse[i])__$(AAstretch)AA.svg")
#            display(p)
#$$        end
#$$    end
#############################################################################################################
    xval = [x[1] for x in NSP_chr_sort_final]
    yval = [max(y[2], 0) for y in NSP_chr_sort_final]
    xinterval = 20
    xtick_ind = collect(1:xinterval:length(xval))
    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
    x_indices = 1:length(xval)
    xfont = xtik_font_size(length(xval))
    y_max = genome_max_dens
    q = Plots.plot(x_indices, yval, 
        seriestype=:line, linewidth=1, linecolor=:darkblue, fill = (0, :blue), size=(4000, 800), dpi=810,
        title="AA-sub density by $(AAstretch)-AA sliding window", label="EPCI AA-sub density", titlefont=("Helvetica Bold", 28),
        xticks=(xtick_ind, xtick_labels), xrotation=90, xlabel="Amino acid site", ylabel="EPCI AA-sub density", legend=:topright, 
        guidefont=font("Helvetica Bold", 20), tickfontfamily="Helvetica Bold", ytickfontsize=15, xtickfontsize=xfont,   
        ylims=(0, y_max), titlefontcolor=:black, xguidefontcolor=:black, yguidefontcolor=:black, tickfontcolor=:black,
        ygrid=false, yminorgrid=false, gridcolor=:gray, gridalpha=0.4)
    Plots.plot!(q, bottom_margin=20Plots.mm, left_margin=30Plots.mm, right_margin=5Plots.mm)
    Plots.savefig(q, "AA_density/$(date)_AA_dens_no_dels_chr_FULL_GENOME__$(AAstretch)AA.png")
    Plots.savefig(q, "AA_density/$(date)_AA_dens_no_dels_chr_FULL_GENOME__$(AAstretch)AA.svg")
#    display(q)
#############################################################################################################
#$$    xval = [x[1] for x in NSP_chr_sort_final]
#$$    yval = [max(log2.(y[2]), 1) for y in NSP_chr_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval))
#$$    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$    x_indices = 1:length(xval)
#$$    xfont = xtik_font_size(length(xval))
#$$    y_max = log2(genome_max_dens)
#$$    q = Plots.plot(x_indices, yval, 
#$$        seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=810,
#$$        xticks=(xtick_ind, xtick_labels), xlabel="Amino acid site", ylabel="EPCI AA-sub density",
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),  titlefont=("Helvetica Bold", 28),
#$$        title="AA-sub density by $(AAstretch)-AA sliding window", label="EPCI AA-sub density", 
#$$        background_color_inside=:white, background_color_outside=:white, linecolor=:darkblue,
#$$
#$$        xrotation=90, ylims=(0, y_max), tickfontfamily="Helvetica Bold", ytickfontsize=15, 
#$$        xtickfontsize=xfont, legend=:topright, ygrid=false, gridcolor=:gray, gridalpha=0.5)
#$$    Plots.plot!(q, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#    Plots.savefig(q, "AA_density/$(date)_AA_dens_no_dels_LOG2_chr_FULL_GENOME__$(AAstretch)AA.png")
#    Plots.savefig(q, "AA_density/$(date)_AA_dens_no_dels_LOG2_chr_FULL_GENOME__$(AAstretch)AA.svg")
#    display(q)
#############################################################################################################
#$$    for i in 1:length(prot_dens_ratio_vec)
#$$        if i ≠ 11
#$$        xval = [x[1] for x in prot_dens_ratio_vec[i]]
#$$        yval = [max(y[2], 0) for y in prot_dens_ratio_vec[i]]
#$$        xind = 1:length(xval)
#$$        xinterval = max(2, (length(xval)÷100)+2)
#$$        xtick_ind = collect(1:xinterval:length(xval))
#$$        xfont = xtik_font_size(length(xval))
#$$        xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$        y_max = max(genome_max_ratio_dens÷2, max_dens_ratio_dict[i])
#$$    combined_prot_plots = Plots.plot(xind, yval, 
#$$        seriestype=:line, linewidth=2, size=(2000, 800), dpi=600, xticks=(xtick_ind, xtick_labels), 
#$$        xlabel="Amino acid site", ylabel="EPCI:HQCS AA-Sub Density Ratio", 
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),  titlefont=("Helvetica Bold", 28),  
#$$        title="$(gene_protein_order_reverse[i])", label="EPCI:HQCS AA-Sub Density Ratio", 
#$$        legend=:topleft, background_color_inside=:white, background_color_outside=:white, 
#$$
#$$        linecolor=:darkred, xrotation=90, ytickfontsize=15, xtickfontsize=xfont, 
#$$        ylims=(0, y_max), tickfontfamily="Helvetica Bold", ygrid=false, 
#$$        gridcolor=:gray, gridalpha=0.5) 
#$$        Plots.plot!(combined_prot_plots, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#$$    yval2 = [max(y[2], 0) for y in prot_dens_vec[i]]
#$$        y_max = max(genome_max_dens÷3, max_dens_dict[i])
#$$    Plots.plot!(Plots.twinx(), xind, yval2, 
#$$        linecolor=:darkblue, xrotation=90, xtickfontsize=xfont, ytickfontsize=15, 
#$$        ylims=(0, y_max), tickfontfamily="Helvetica Bold", 
#$$        label="EPCI AA-sub density, $(AAstretch)-AA sliding window", linewidth=2, 
#$$
#$$        ylabel="EPCI AA-sub density, $(AAstretch)-AA sliding window", legend=:topright )
#    Plots.savefig(combined_prot_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_$(gene_protein_order_reverse[i])__$(AAstretch)AA.png")
#    Plots.savefig(combined_prot_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_$(gene_protein_order_reverse[i])__$(AAstretch)AA.svg")
#        Plots.savefig(combined_prot_plots, "$(date)/AA_density/AA_stretch_mut_density_no_dels_chrALL_RATIO_MULTI_SLIDE_$(gene_protein_order_reverse[i])__$(date)_$(AAstretch)AAsize_plot.pdf")
#        display(combined_prot_plots)
#$$        end
#$$    end
#############################################################################################################
#$$    xval = [x[1] for x in NSP_chr_to_all_ratio_sort_final]
#$$    yval = [max(y[2], 0) for y in NSP_chr_to_all_ratio_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval))
#$$    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$    x_indices = 1:length(xval)
#$$    xfont = xtik_font_size(length(xval))
#$$    y_max = genome_max_ratio_dens
#$$    s = Plots.plot(x_indices, yval, 
#$$        seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=810, xticks=(xtick_ind, xtick_labels), 
#$$        xlabel="Amino acid sites", ylabel="EPCI:HQCS AA-Sub Density Ratio",
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),  titlefont=("Helvetica Bold", 28),
#$$
#$$        title="AA-Sub Density Ratio: EPCI/HQCS Sequences", tickfontfamily="Helvetica Bold",
#$$        label="EPCI:HQCS AA-Sub Density Ratio", background_color_inside=:white, ylims=(0, y_max),
#$$        background_color_outside=:white, linecolor=:darkred, xrotation=90,  xtickfontsize=xfont, 
#$$        ytickfontsize=15, legend=:topleft, ygrid=false, gridcolor=:gray, gridalpha=0.5)
#$$        
#$$        Plots.plot!(s, bottom_margin=20Plots.mm)
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_FULL_GENOME__$(AAstretch)AA.png")
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_FULL_GENOME__$(AAstretch)AA.svg")
#    display(s)
#############################################################################################################
#$$    xval = [x[1] for x in NSP_chr_to_all_ratio_sort_final]
#$$    yval = [max(log2.(y[2]), 1) for y in NSP_chr_to_all_ratio_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval))
#$$    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$    x_indices = 1:length(xval)
#$$    xfont = xtik_font_size(length(xval))
#$$    y_max = log2(genome_max_ratio_dens)
#$$    s = Plots.plot(x_indices, yval, 
#$$        seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=800, xticks=(xtick_ind, xtick_labels), 
#$$        xlabel="Amino acid sites", ylabel="EPCI:HQCS AA-Sub Density Ratio",
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20),  titlefont=("Helvetica Bold", 28),
#$$
#$$        title="AA-Sub Density Ratio: EPCI/HQCS Sequences", tickfontfamily="Helvetica Bold",
#$$        label="EPCI:HQCS AA-Sub Density Ratio", background_color_inside=:white, 
#$$        background_color_outside=:white, linecolor=:darkred, xrotation=90,  xtickfontsize=xfont, 
#$$        ytickfontsize=15, legend=:topleft, ylims=(0, y_max), ygrid=false, 
#$$        gridcolor=:gray, gridalpha=0.5)
#$$        Plots.plot!(s, bottom_margin=20Plots.mm)
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_LOG2_chrALL_RATIO_FULL_GENOME__$(AAstretch)AA.png")
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_LOG2_chrALL_RATIO_FULL_GENOME__$(AAstretch)AA.svg")
#    display(s)
#############################################################################################################
#$$    xval1 = [x[1] for x in NSP_chr_sort_final]
#$$    yval1 = [max(y[2], 0) for y in NSP_chr_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval1))
#$$    xtick_labels = xval_xtick_label.(xval1[1:xinterval:length(xval1)])
#$$    x_indices = 1:length(xval1)
#$$    xfont = xtik_font_size(length(xval1))
#$$    println(xfont)
#$$    y_max = genome_max_dens
#$$    combined_plots = Plots.plot(x_indices, yval1, 
#$$        label="chr_mut_density", seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=810, 
#$$        xticks=(xtick_ind, xtick_labels), xlabel="Amino acid site", ylabel="EPCI AA-sub density", 
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 20), titlefont=("Helvetica Bold", 28), 
#$$
#$$        title="AA-sub density by $(AAstretch)-AA sliding window", background_color_inside=:white, 
#$$        background_color_outside=:white, linecolor=:darkblue, xrotation=90, ylims=(0, y_max), 
#$$        tickfontfamily="Helvetica Bold", ytickfontsize=15, xtickfontsize=xfont, ygrid=false, gridcolor=:gray, 
#$$        gridalpha=0.5)        
#$$    xval2 = [x[1] for x in NSP_chr_to_all_ratio_sort_final]
#$$    yval2 = [max(y[2], 0) for y in NSP_chr_to_all_ratio_sort_final]
#$$    xtick_ind = collect(1:xinterval:length(xval2))
#$$    xtick_labels = xval_xtick_label.(xval2[1:xinterval:length(xval2)-AAstretch÷2+1])
#$$    x_indices2 = 1:length(xval2)
#$$    y_max = genome_max_ratio_dens
#$$    Plots.plot!(Plots.twinx(), x_indices2, yval2, 
#$$        seriestype=:line, linecolor=:darkred, label="EPCI:HQCS AA-Sub Ratio", linewidth=1.5,
#$$        tickfontfamily="Helvetica Bold", ytickfontsize=15, 
#$$        ylabel="EPCI:HQCS AA-Sub Ratio", ylims=(0, y_max) )
#$$    Plots.plot!(combined_plots, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#    Plots.savefig(combined_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_FULL_GENOME__$(AAstretch)AA.png")
#    Plots.savefig(combined_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_FULL_GENOME__$(AAstretch)AA.svg")
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
############################################## Begin Adjusted Ratio Versions ##########################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#############################################################################################################################################################################
#$$    for i in 1:length(prot_dens_ratio_adjusted_vec)
#$$        if i ≠ 11
#$$            xval = [x[1] for x in prot_dens_ratio_adjusted_vec[i]]
#$$            yval = [max(y[2], 0) for y in prot_dens_ratio_adjusted_vec[i]]
#$$            xind = 1:length(xval)
#$$            xinterval = max(2, (length(xval)÷100)+2)
#$$            xtick_ind = collect(1:xinterval:length(xval))
#$$            xfont = xtik_font_size(length(xval))
#$$            xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$            y_max = max(genome_max_ratio_dens÷3, max_dens_ratio_adjusted_dict[i])
#$$        p = Plots.plot(xind, yval,  
#$$            seriestype=:line, linewidth=2, size=(2000, 800), dpi=600, xticks=(xtick_ind, xtick_labels),
#$$            xlabel="Protein + AA Site", ylabel="EPCI:HQCS AA-sub ratio, normalized for AA subs/seq", 
#$$            xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 16), titlefont=("Helvetica Bold", 28),
#$$            title="$(gene_protein_order_reverse[i])", label="EPCI/HQCS AA-sub ratio, normalized for AA subs/seq",
#$$            legend=:topleft, background_color_inside=:white, background_color_outside=:white, 
#$$            linecolor=:darkred, xrotation=90, ytickfontsize=15, xtickfontsize=xfont, 
#$$
#$$            ylims=(0, y_max), tickfontfamily="Helvetica Bold", 
#$$            ygrid=false, gridcolor=:gray, gridalpha=0.5)
#$$            Plots.plot!(p, bottom_margin=20Plots.mm, left_margin=20Plots.mm, right_margin=20Plots.mm)
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_ADJ_$(gene_protein_order_reverse[i])__$(AAstretch)AA.png")
#            Plots.savefig(p, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_ADJ_$(gene_protein_order_reverse[i])__$(AAstretch)AA.svg")
#            display(p)
#$$        end
#$$    end
#############################################################################################################
#$$    xval = [x[1] for x in NSP_chr_to_all_ratio_adjusted_sort_final]
#$$    yval = [max(y[2], 0) for y in NSP_chr_to_all_ratio_adjusted_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval))
#$$    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$    x_indices = 1:length(xval)
#$$    xfont = xtik_font_size(length(xval))
#$$    y_max = genome_max_ratio_adjusted_dens
#$$    s = Plots.plot(x_indices, yval, 
#$$        seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=810, xticks=(xtick_ind, xtick_labels), 
#$$        xlabel="Amino acid sites", ylabel="EPCI:HQCS AA-sub ratio, normalized for AA subs/seq",
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 16),  titlefont=("Helvetica Bold", 28),
#$$        title="AA-Sub Density Ratio: EPCI/HQCS Sequences, normalized for AA subs/seq", tickfontfamily="Helvetica Bold",
#$$        label="EPCI/HQCS AA-sub ratio, normalized for AA subs/seq", background_color_inside=:white, 
#$$        background_color_outside=:white, linecolor=:darkred, xrotation=90,  xtickfontsize=xfont, 
#$$    
#$$        ytickfontsize=15, ylims=(0, y_max), legend=:topleft, ygrid=false, gridcolor=:gray, 
#$$        gridalpha=0.5)
        
#$$        Plots.plot!(s, bottom_margin=20Plots.mm)
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_ADJ_FULL_GENOME__$(AAstretch)AA.png")
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_chrALL_RATIO_ADJ_FULL_GENOME__$(AAstretch)AA.svg")
#    display(s)
#############################################################################################################
#$$    xval = [x[1] for x in NSP_chr_to_all_ratio_adjusted_sort_final]
#$$    yval = [max(log2.(y[2]), -1) for y in NSP_chr_to_all_ratio_adjusted_sort_final]
#$$    xinterval = 20
#$$    xtick_ind = collect(1:xinterval:length(xval))
#$$    xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)])
#$$    x_indices = 1:length(xval)
#$$    xfont = xtik_font_size(length(xval))
#$$    y_max = log2(genome_max_ratio_adjusted_dens)
#$$    s = Plots.plot(x_indices, yval, 
#$$        seriestype=:line, linewidth=1.5, size=(4000, 800), dpi=810, xticks=(xtick_ind, xtick_labels), 
#$$        xlabel="Amino acid sites", ylabel="EPCI:HQCS AA-sub ratio, normalized for AA subs/seq",
#$$        xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 16),  titlefont=("Helvetica Bold", 28),
#$$        title="AA-Sub Density Ratio: EPCI/HQCS Sequences, normalized for AA subs/seq", tickfontfamily="Helvetica Bold",
#$$        label="EPCI/HQCS AA-sub ratio, normalized for AA subs/seq", background_color_inside=:white, 
#$$        background_color_outside=:white, linecolor=:darkred, xrotation=90,  xtickfontsize=xfont, 
#$$    
#$$        ytickfontsize=15, legend=:topleft, ylims=(-1, y_max), ygrid=false, gridcolor=:gray, 
#$$        gridalpha=0.5)
#$$        Plots.plot!(s, bottom_margin=20Plots.mm)
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_LOG2_chrALL_RATIO_ADJ_FULL_GENOME__$(AAstretch)AA.png")
#        Plots.savefig(s, "AA_density/$(date)_AA_dens_no_dels_LOG2_chrALL_RATIO_ADJ_FULL_GENOME__$(AAstretch)AA.svg")
#    display(s)
#####################################################################################################################################
    for i in 1:length(prot_dens_ratio_adjusted_vec)
        if i ≠ 11
            xval = [x[1] for x in prot_dens_ratio_adjusted_vec[i]]
            yval = [max(y[2], 0) for y in prot_dens_ratio_adjusted_vec[i]]
            xind = 1:length(xval); xinterval = max(2, (length(xval)÷100)+2); xtick_ind = collect(1:xinterval:length(xval))
            xtick_labels = xval_xtick_label.(xval[1:xinterval:length(xval)]); xfont = xtik_font_size(length(xval))
            y_max = max(genome_max_ratio_adjusted_dens÷4, 1.1*max_dens_ratio_adjusted_dict[i])
            yinterval = max(5, (y_max÷30)*5); ytick_ind = collect(1:length(yval)); ytick_labels = 1:yinterval:y_max
        combined_prot_plots = Plots.plot(xind, yval, 
            seriestype=:line, linecolor=:darkred, linewidth=3, size=(2000, 800), dpi=300,  
            xticks=(xtick_ind, xtick_labels),
            title="$(gene_protein_order_reverse[i])", label="EPCI/HQCS AA-sub ratio, normalized for AA subs/seq",
            xlabel="Amino acid site", ylabel="EPCI:HQCS AA-sub ratio, normalized for AA subs/seq", 
            xguidefont=font("Helvetica Bold", 20), yguidefont=font("Helvetica Bold", 16), titlefont=("Helvetica Bold", 28),
            legend=:topleft, legendfont=font("Helvetica Bold", 12), 
            background_color_inside=:white, background_color_outside=:white, 
            xrotation=90, ytickfontsize=15, xtickfontsize=xfont, tickfontfamily="Helvetica Bold",  
            xgrid=true, ygrid=false, gridcolor=:gray, gridalpha=0.5, gridstyle=:solid, yminorgrid=false, minorgridstyle=:solid, minorgridalpha=0.3,
    
            ylims=(0, y_max)) 
            Plots.plot!(combined_prot_plots, bottom_margin=11Plots.mm, left_margin=15Plots.mm, right_margin=11Plots.mm)
        yval2 = [max(y[2], 0) for y in prot_dens_vec[i]]
            y_max = max(genome_max_dens÷3, 1.1*max_dens_dict[i])
        Plots.plot!(Plots.twinx(), xind, yval2, 
            linecolor=:darkblue, linewidth=3, label="EPCI AA-sub density, $(AAstretch)-AA sliding window", 
            legend=:topright, legendfont=font("Helvetica Bold", 12), 
            xrotation=90, xtickfontsize=xfont, ytickfontsize=15, tickfontfamily="Helvetica Bold", xguidefont=font("Helvetica Bold", 16), 
            yguidefont=font("Helvetica Bold", 16), ylabel="EPCI AA-sub density, $(AAstretch)-AA window",
            ylims=(0, y_max))
            if prot_dens_ratio_adjusted_vec[i] == ORF9b_dens_ratio_adjusted
                println("YES")
                last = length(xval)
                println("last index = $(last)")
                first_index_to_highlight = last-9+AAstretch÷2
                println("first_index_to_highlight = $(first_index_to_highlight)")
                last_AA = 97 - AAstretch÷2
                Plots.vspan!(combined_prot_plots, [first_index_to_highlight, last], fill = :yellow, fillalpha = 0.3, label = "ORF9b:89-97")
            end
Plots.savefig(combined_prot_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_RATIO_ADJ_$(gene_protein_order_reverse[i])__$(AAstretch)AA.png")
Plots.savefig(combined_prot_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_RATIO_ADJ_$(gene_protein_order_reverse[i])__$(AAstretch)AA.svg")
#        display(combined_prot_plots)
        end
    end
#####################################################################################################################################
    xval1 = [x[1] for x in NSP_chr_sort_final]
    yval1 = [max(y[2], 0) for y in NSP_chr_sort_final]
    xinterval = 20
    xtick_ind = collect(1:xinterval:length(xval1))
    xtick_labels = xval_xtick_label.(xval1[1:xinterval:length(xval1)])
    x_indices = 1:length(xval1)
    xfont = xtik_font_size(length(xval1))
    println(xfont)
    y_max = 1.1*genome_max_dens
    combined_plots = Plots.plot(x_indices, yval1, 
        xticks=(xtick_ind, xtick_labels), seriestype=:line, size=(2500, 800), dpi=810, linewidth=1, linecolor=:darkblue, 
        title="AA-sub density by $(AAstretch)-AA sliding window", titlefont=("Helvetica Bold", 28), label="EPCI AA-sub density", 
        xlabel="Amino acid site", xguidefont=font("Helvetica Bold", 20), xrotation=90, ylabel="EPCI AA-sub density", 
        yguidefont=font("Helvetica Bold", 20), tickfontfamily="Helvetica Bold", ytickfontsize=15, xtickfontsize=xfont, 
        background_color_inside=:white, background_color_outside=:white, legendfont=font("Helvetica Bold", 14), 
        xgrid=true, ygrid=false, yminorgrid=false, gridcolor=:gray, gridalpha=0.4,
        ylims=(0, y_max))                 
        #gridstyle=:solid, (Make grid lines dashed), minorgrid=false, (Turn off minor gridlines)  yticks=nothing (Remove y-axis tick labels) 
    xval2 = [x[1] for x in NSP_chr_to_all_ratio_adjusted_sort_final]
    yval2 = [max(y[2], 0) for y in NSP_chr_to_all_ratio_adjusted_sort_final]
    xtick_ind = collect(1:xinterval:length(xval2))
    xtick_labels = xval_xtick_label.(xval2[1:xinterval:length(xval2)-AAstretch÷2+1])
    x_indices2 = 1:length(xval2)
    y_max = genome_max_ratio_adjusted_dens
    Plots.plot!(Plots.twinx(), x_indices2, yval2, 
        seriestype=:line, linecolor=:darkred, label="EPCI/HQCS AA-sub ratio, normalized for AA subs/seq", linewidth=1,
        legend=:topright, legendfont=font("Helvetica Bold", 14), 
        ylabel="EPCI:HQCS AA-sub ratio, normalized for AA subs/seq", yguidefont=font("Helvetica Bold", 16),
        tickfontfamily="Helvetica Bold", ytickfontsize=15, 
        ylims=(0, y_max))
    Plots.plot!(combined_plots, bottom_margin=11Plots.mm, left_margin=16Plots.mm, right_margin=16Plots.mm)
    Plots.savefig(combined_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_FULL_GENOME__$(AAstretch)AA.png")
    Plots.savefig(combined_plots, "AA_density/$(date)_AA_dens_no_dels_twinplot_FULL_GENOME__$(AAstretch)AA.svg")
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
############################################## END Adjusted Ratio Versions ############################################################
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
#####################################################################################################################################
    open("AA_density/$(date)_AA_density_no_dels_chrALL_RATIO_gene_sort__$(AAstretch)AA.txt", "w") do g
        println(g, "\n"^3)
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), "  $(ORF_range)" )
        end
    end
    open("AA_density/$(date)_AA_density_no_dels_chrALL_RATIO_ratio_sort__$(AAstretch)AA.txt", "w") do g
        println(g, "\n"^3)
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), "  $(ORF_range)" )
        end
   end
#####################################################################################################################################
############################################ Begin Adjusted Ratio Print Versions ######################################################
#####################################################################################################################################
    open("AA_density/$(date)_AA_density_no_dels_chrALL_RATIO_ADJ_gene_sort__$(AAstretch)AA.txt", "w") do g
        println(g, "\n"^3)
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), "  $(ORF_range)" )
        end
    end
    open("AA_density/$(date)_AA_density_no_dels_chrALL_RATIO_ADJ_ratio_sort__$(AAstretch)AA.txt", "w") do g
        println(g, "\n"^3)
        for w in AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio
            stretch = w[1]
            ORF_range = NSP_range_to_ORF1ab_range(stretch)
            count = w[2]
            count_str = string(count)
            dec = split(count_str, ".")[2]
            if length(dec) == 1
                count_str = count_str*"0"
            end
            ct_ln = length(count_str)
            println(g, "                    ", rpad("$(stretch)", 14), " - ", lpad("$(count_str)", 6), "  $(ORF_range)" )
        end
    end
#####################################################################################################################################
############################################### End Adjusted Ratio Print Versions #####################################################
#####################################################################################################################################
    total_time = time() - start
    hms1, hms2 = seconds_to_hrs_min_sec(total_time)
    println(hms1)
    println(hms2)
#    return AA_stretch_ct_no_dels_chr_to_all_ratio, AA_stretch_ct_no_dels_chr_to_all_ratio_sort_by_ratio, AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted, AA_stretch_ct_no_dels_chr_to_all_ratio_adjusted_sort_by_ratio
end

2026_03_05__648PM


AA_stretch_mut_density_no_dels_chr_to_all_ratio_multi_slide (generic function with 1 method)

In [161]:
### Execute Plots, 9-AA Sliding Window ####################
AA_stretch_mut_density_no_dels_chr_to_all_ratio_multi_slide(["NSP1", "NSP2", "NSP3", "NSP4", "NSP5", "NSP6", "NSP7", "NSP8", "NSP9", "NSP10", "NSP12", "NSP13", "NSP14", "NSP15", "NSP16", "S", "ORF3a", "E", "M", "ORF6", "ORF7a", "ORF7b", "ORF8", "N", "ORF9b"], 9, 1)
# AA_stretch_mut_density_no_dels_chr_to_all_ratio_multi(genes::Vector{String}, AAstretch::Int)
################################################################################################
################################################################################################

Maximum AA-Sub Density Value in Genome = 1246.0
Maximum Ratio Value in entire genome = 412.16
Maximum ratio_adjusted Value in entire genome = 85.43
YES
last index = 89
first_index_to_highlight = 84
4
0:00:25.96
0 hr, 0 min, 25.96 sec


In [225]:
### Fx: pango_chr_mut_cts(pango::String) ## Calculates the private EPCI mutations for each pango lineage and its descendants
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now);  date = Dates.format(today(), "yyyy_mm_dd")
function pango_chr_mut_cts(pango::String)
    pango_chr_mut_counts = Dict{String, Int}()
    pango_chr_mut_counts_ct_sort = Vector{Pair{String, Int64}}()
    date = Dates.format(today(), "yyyy_mm_dd")
    unaliased = pango_to_pango_unaliased_v2[pango]
    unal_len = length(unaliased)
    qualified_ct = 0
    for seq in all_unique_chr_seqs_set
        fx_seq_pango = seq_pango[seq]
        fx_seq_unal = pango_to_pango_unaliased_v2[fx_seq_pango]
        fx_seq_unal_len = length(fx_seq_unal)
        if fx_seq_unal_len ≥ unal_len
            qualified_ct += 1
            if string(fx_seq_unal[1:unal_len]) == unaliased
                for mut in seq_AA_muts[seq]
                    pango_chr_mut_counts[mut] = get!(pango_chr_mut_counts, mut, 0) + 1
                end
            end
        end
    end
    pango_chr_mut_counts_ct_sort = sort(collect(pango_chr_mut_counts), by = x -> x[2], rev=true)
    pango_chr_mut_counts_gene_sort = sort(collect(pango_chr_mut_counts), by = x -> gene___AAnum_sortkey(x[1]))
    open("pango_AA_mut_cts/pango_AA_muts_ct_$(pango)_$(date).txt", "w") do g
        for mut___ct in pango_chr_mut_counts_ct_sort
            mut = mut___ct[1]
            ct = mut___ct[2]
            mutpad = rpad(mut, 12)
            ct_pad = lpad(ct, 2)
#            println("$(mutpad) = $(ct_pad)")
            println(g, "$(mutpad) = $(ct_pad)")
        end
    end
    pango_chr_mut_counts_type = typeof(pango_chr_mut_counts)
    pango_chr_mut_counts_ct_sort_type = typeof(pango_chr_mut_counts_ct_sort)
#    println("pango_chr_mut_counts_type = $(pango_chr_mut_counts_type)")
#    println("pango_chr_mut_counts_ct_sort_type = $(pango_chr_mut_counts_ct_sort_type)")
    return pango_chr_mut_counts, pango_chr_mut_counts_ct_sort
end
#########################################################################################################################################################
#########################################################################################################################################################

2026_02_08_524PM


pango_chr_mut_cts (generic function with 1 method)

In [327]:
### Execute pango_chr_mut_cts  
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now);   date = Dates.format(today(), "yyyy_mm_dd")
pango_chr_mut_cts("BA.2.86")
# pango_chr_mut_cts(pango::String)
#########################################################################################################################################################

2026_01_31_611PM
unaliased = B.1.1.529.2.86
unal_len = 14
ORF1a:T1638I = 29
ORF1a:T1322I = 26
S:S455L      = 25
ORF7a:T39I   = 23
ORF1a:K1795Q = 21
E:T30I       = 19
ORF1a:S4398L = 15
ORF1a:H3580Y = 15
ORF6:P57S    = 13
S:T385I      = 13
N:P326L      = 12
ORF1a:L3606F = 12
S:K481-      = 11
S:T573I      = 11
ORF1a:P1640L = 11
S:S31-       = 11
S:V367A      = 11
S:T299I      = 10
N:T271I      = 10
S:I19T       = 10
ORF9b:D89E   =  9
ORF1a:D1323G =  9
ORF1a:T1543I =  9
ORF1a:I4098T =  9
S:A243-      =  8
ORF1a:T1638N =  8
S:K484E      =  8
ORF1a:H1500Y =  8
S:H445R      =  8
S:G482K      =  8
S:D936H      =  8
ORF3a:Q213K  =  8
S:L242-      =  8
S:T21-       =  8
S:K478E      =  8
N:S37P       =  7
S:P384S      =  7
ORF7a:A105V  =  7
ORF1b:T730I  =  7
ORF1a:A3648V =  7
ORF1b:G662S  =  7
S:D142-      =  7
ORF9b:L64P   =  7
S:V143-      =  7
ORF1a:D1639N =  7
ORF1a:E4097K =  7
ORF1a:T4090I =  7
S:R681H      =  7
M:H155N      =  7
ORF1a:A2325V =  6
E:S55F       =  6
ORF7a:L96P   =  6
S:K403

2075-element Vector{Pair{String, Int64}}:
 "ORF1a:T1638I" => 29
 "ORF1a:T1322I" => 26
      "S:S455L" => 25
   "ORF7a:T39I" => 23
 "ORF1a:K1795Q" => 21
       "E:T30I" => 19
 "ORF1a:S4398L" => 15
 "ORF1a:H3580Y" => 15
    "ORF6:P57S" => 13
      "S:T385I" => 13
      "N:P326L" => 12
 "ORF1a:L3606F" => 12
      "S:K481-" => 11
                ⋮
 "ORF1a:M2169T" => 1
    "ORF8:T11K" => 1
 "ORF1b:P1567L" => 1
   "ORF3a:K21G" => 1
 "ORF1a:G1073E" => 1
      "S:V227I" => 1
  "ORF1b:A414V" => 1
  "ORF1b:E427K" => 1
 "ORF1b:A1428V" => 1
 "ORF1a:K2120E" => 1
      "S:V963F" => 1
 "ORF1a:T3461A" => 1

In [226]:
### Making pango_AA_muts_ct_metadict: making a dictionary that holds the mut counts for all pango lineages (& descendants) in EPCI
# -----must FIRST run: NEW | 2026_02_06 | Many, many functions 
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
start = time()
#                                Pango         Mut    Count 
pango_AA_muts_ct_metadict = Dict{String, Dict{String, Int}}()
pango_AA_muts_ct_meta_vec_sort = Dict{String, Vector{Pair{String, Int64}}}()
png_d_ct = 0
print("\n"^1)
for pango in chronic_pango_set
    pango_AA_muts_ct_meta_vec_sort[pango] = Vector{Pair{String, Int64}}()
    pango_ct_dictionary, pango_ct_vec_sort = pango_chr_mut_cts(pango)
    pango_ct_dictionary_type = typeof(pango_ct_dictionary)
    pango_ct_vec_sort_type = typeof(pango_ct_vec_sort)
#    println("pango_ct_dictionary_type = $(pango_ct_dictionary_type)")
#    println("pango_ct_vec_sort = $(pango_ct_vec_sort_type)")
#    println("pango_AA_muts_ct_meta_vec_sort[pango] Type = $(typeof(pango_AA_muts_ct_meta_vec_sort[pango]))")
    pango_AA_muts_ct_metadict[pango] = pango_ct_dictionary
    pango_AA_muts_ct_meta_vec_sort[pango] = pango_ct_vec_sort
    png_d_ct += 1
    if png_d_ct%50 == 0
        runtime = time() - start
        runtime_rd = round(digits=2, runtime)
        println("$(png_d_ct) Pango Ct Dicts completed. Runtime = $(runtime_rd)")
    end
end
runtime = time() - start
runtime_rd = round(digits=2, runtime); print("\n"^1)
println("Total Runtime = $(runtime_rd) seconds")
println("Finished!"); print("\n"^2)
#########################################################################################################################################################
#########################################################################################################################################################

2026_02_08__524PM

50 Pango Ct Dicts completed. Runtime = 0.67
100 Pango Ct Dicts completed. Runtime = 0.96
150 Pango Ct Dicts completed. Runtime = 3.71
200 Pango Ct Dicts completed. Runtime = 3.85
250 Pango Ct Dicts completed. Runtime = 4.14
300 Pango Ct Dicts completed. Runtime = 4.28
350 Pango Ct Dicts completed. Runtime = 4.84
400 Pango Ct Dicts completed. Runtime = 4.98
450 Pango Ct Dicts completed. Runtime = 5.19
500 Pango Ct Dicts completed. Runtime = 5.32
550 Pango Ct Dicts completed. Runtime = 5.64
600 Pango Ct Dicts completed. Runtime = 5.74
650 Pango Ct Dicts completed. Runtime = 5.81
700 Pango Ct Dicts completed. Runtime = 5.93
750 Pango Ct Dicts completed. Runtime = 6.57
800 Pango Ct Dicts completed. Runtime = 6.67

Total Runtime = 6.7 seconds
Finished!




In [238]:
### ORF7a:H47Y lineage search + count in chronics
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd__I")
ORF7a_H47Y_pangos = Set{String}()
pango_AAsub_WT["B.1.1.529"] = union(pango_AAsub_WT["BA.1.1"], pango_AAsub_WT["BA.2"])
pango_AAsub_WT["LF.3.1"] = pango_AAsub_WT["LF.3"]
for pango in chronic_pango_set
    if "ORF7a:H47Y" in pango_AAsub_WT[pango]
        push!(ORF7a_H47Y_pangos, pango)
    end
end
ORF7a_H47Y_pangos_sort = sort(collect(ORF7a_H47Y_pangos))
ORF7a_H47Y_pangos_sort_join = join(ORF7a_H47Y_pangos_sort, ", ")
println(ORF7a_H47Y_pangos_sort_join)
print("\n"^1)
ORF7a_H47Y_chr_seq_ct = 0
for seq in all_unique_chr_seqs_set
    if seq_pango[seq] in ORF7a_H47Y_pangos
        ORF7a_H47Y_chr_seq_ct += 1
    end
end
println("ORF7a_H47Y_chr_seq_ct = $(ORF7a_H47Y_chr_seq_ct)")
print("\n"^2)
ORF7a_Y47H_rev_ct = 0
ORF7a_H47Y_chr_seqs = String[]
for seq in all_unique_chr_seqs_set
    if seq_pango[seq] in ORF7a_H47Y_pangos
        print_all_seq_info(seq, "ORF7a_H47Y_lineage_chronic_seqs_$(date_hour).txt")
        if "ORF7a:Y47H" in seq_AA_muts[seq]
            ORF7a_Y47H_rev_ct += 1
            push!(ORF7a_H47Y_chr_seqs, seq)
        end
    end
end
ORF7a_H47Y_chr_seqs_sort = sort(ORF7a_H47Y_chr_seqs, by = x -> (length(x), x))
ORF7a_H47Y_chr_seqs_sort_join = join(ORF7a_H47Y_chr_seqs_sort, ", ")
println("ORF7a_Y47H_rev_ct = $(ORF7a_Y47H_rev_ct)/$(ORF7a_H47Y_chr_seq_ct)")
ORF7a_Y47H_pct = round(digits=2, 100*ORF7a_Y47H_rev_ct/ORF7a_H47Y_chr_seq_ct)
println("ORF7a_Y47H_pct = $(ORF7a_Y47H_pct)")
print("\n"^2)
for seq in ORF7a_H47Y_chr_seqs_sort
    println(seq)
end
print("\n"^2)
#############################################################################################################################################################################
#############################################################################################################################################################################

2026_02_08__646PM
BF.5, BF.7.14, BF.7.14.1, BF.7.14.4, BF.7.14.5

ORF7a_H47Y_chr_seq_ct = 16


##############################################################################################
Singlet
BF.7.14|EPI_ISL_17199743|2023-02-14 | Heilongjiang||HLJCDC-0290 | AAsubs+DelRanges = 9
----------------------------------- AA Substitutions -------------------------------------
S--F79L, A348P, R403K, K440R
ORF6--H3Y     ORF1a--K722R     ORF1b--V329I, G662S, L2184S
------------------------------------------------------------------------------------------
NUC MUTATIONS--A2430G, A4783G, T8803C, G14452A, G15451A, T20018C, A21282G, T21797C,
      G22604C, G22770A, A22881G, C23896G, G25644C, C27208T
SYNONYMOUS NUC MUTATIONS - Total:5
     A4783G, T8803C, A21282G, C23896G, G25644C

##############################################################################################
##############################################################################################
Singlet
BF.7.14.5|EPI_ISL_176

In [52]:
### Checking NSP2:232-233,235 seqs with & without Mac2-Mac3-linker muts
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
############################################################
NSP2__Mac2_Mac3_Linker_cts_dict = Dict{String, Int}()
############################################################
NSP2_muts_pos_only = Set(["ORF1a:233"])
NSP2_muts = Set(["ORF1a:R232L", "ORF1a:E233A", "ORF1a:E233D", "ORF1a:E233G", "ORF1a:E233K"])
Mac2_Mac3_Linker = Set(["ORF1a:S1361F", "ORF1a:S1361P", "ORF1a:S1361A", "ORF1a:E1363V", "ORF1a:E1363G", "ORF1a:E1363A", "ORF1a:E1363D", "ORF1a:E1363K", "ORF1a:Q1365L", "ORF1a:Q1365K", "ORF1a:Q1365E", "ORF1a:Q1365R", "ORF1a:Q1365P", "ORF1a:E1366G", "ORF1a:E1366D", "ORF1a:E1366K", "ORF1a:E1366A", "ORF1a:I1367L", "ORF1a:I1367M", "ORF1a:I1367V", "ORF1a:L1368I", "ORF1a:G1369R", "ORF1a:S1372A", "ORF1a:S1372C", "ORF1a:S1372V", "ORF1a:S1372P"])
Mac2_Mac3_Linker_pos_only = Set(["ORF1a:1361", "ORF1a:1362", "ORF1a:1363", "ORF1a:1364", "ORF1a:1365", "ORF1a:1366", "ORF1a:1367", "ORF1a:1368", "ORF1a:1369", "ORF1a:1370", "ORF1a:1371", "ORF1a:1372"])
##############################
NSP2_seqs_set = Set{String}()
##############################
for seq in all_unique_chr_seqs_set
    for mut in NSP2_muts
        if mut in seq_AA_muts[seq]
            push!(NSP2_seqs_set, seq)
        end
    end
end
############################################################
NSP2_and_Mac2_Mac3_seqs = Set{String}()
NSP2_and_not_Mac2_Mac3_seqs = Set{String}()
NSP2_and_Mac2_Mac3_ct = 0
NSP2_and_not_Mac2_Mac3_ct = 0
total_NSP2_mut_seqs = length(NSP2_seqs_set)
for seq in NSP2_seqs_set
    amino_A_muts = seq_AA_muts_no_dels[seq]
    M23L_muts = intersect(amino_A_muts, Mac2_Mac3_Linker)
    amino_A_muts_pos_only = AAmut_to_AApos.(collect(amino_A_muts))
#    M23L_muts_pos_only = intersect(amino_A_muts_pos_only, Mac2_Mac3_Linker_pos_only)
    if !isempty(intersect(amino_A_muts_pos_only, Mac2_Mac3_Linker_pos_only))
        NSP2_and_Mac2_Mac3_ct += 1
        push!(NSP2_and_Mac2_Mac3_seqs, seq)
        for mut in M23L_muts
            NSP2__Mac2_Mac3_Linker_cts_dict[mut] = get!(NSP2__Mac2_Mac3_Linker_cts_dict, mut, 0) + 1
        end
    else
        NSP2_and_not_Mac2_Mac3_ct += 1
        push!(NSP2_and_not_Mac2_Mac3_seqs, seq)
    end
end
NSP2_and_Mac2_Mac3_pct = round(digits=1, 100*NSP2_and_Mac2_Mac3_ct/total_NSP2_mut_seqs)
############################################################
println("NSP2_and_Mac2_Mac3_ct = $(NSP2_and_Mac2_Mac3_ct)")
println("NSP2_and_not_Mac2_Mac3_ct = $(NSP2_and_not_Mac2_Mac3_ct)")
println("$(NSP2_and_Mac2_Mac3_ct)/$(total_NSP2_mut_seqs) NSP2-mut seqs with ≥1 Mac2-Mac3-linker muts (ORF1a:1361-1372) = $(NSP2_and_Mac2_Mac3_pct)%")
####################################################################################################
####################################################################################################
#NSP2_and_not_Mac2_Mac3_seqs_plus_related = Set(["EPI_ISL_18924428", "EPI_ISL_18948422", "EPI_ISL_19017499", "EPI_ISL_19159918", "EPI_ISL_19159923"])
#NSP2_and_not_Mac2_Mac3_seqs_plus_related_sort = sort(collect(NSP2_and_not_Mac2_Mac3_seqs_plus_related), by = x -> (length(x), x))
#for seq in NSP2_and_not_Mac2_Mac3_seqs_plus_related_sort
#    txtout = "all_seq_info/all_seq_info_$(seq)_$(date_now).txt"
#    print_all_seq_info(seq, txtout)
#end
###########################################################################################################################
#############################################################################################################################################################################
println("##########################################################################################################################")
NSP2__Mac2_Mac3_Linker_ct_total = sum(values(NSP2__Mac2_Mac3_Linker_cts_dict))
Mac2_Mac3_muts_per_NSP2_seq = round(digits=2, NSP2__Mac2_Mac3_Linker_ct_total/total_NSP2_mut_seqs)
println("Total Mac2_Mac2_Linker muts in NSP2 sequences = $(NSP2__Mac2_Mac3_Linker_ct_total)")
println("Mac2_Mac2_Linker muts per NSP2 sequence = $(Mac2_Mac3_muts_per_NSP2_seq)")
#########################
NSP2__Mac2_Mac3_Linker_cts_dict_sort = sort(collect(NSP2__Mac2_Mac3_Linker_cts_dict), by = x -> gene_AA_sortKey(x[1]))
for mut___ct in NSP2__Mac2_Mac3_Linker_cts_dict_sort
    mut = mut___ct[1]
    ct = mut___ct[2]
    pct = round(digits=2, 100*ct/NSP2__Mac2_Mac3_Linker_ct_total)
    ct_pad = lpad(ct, 3)
    mut_pad = rpad(mut, 12)
    pct_pad = lpad(pct, 5)
    println("$(mut) = $(ct_pad) | $(pct_pad)%")
end
#############################################################################################################################################################################
NSP2__Mac2_Mac3_Linker_cts_dict_count_sort = sort(collect(NSP2__Mac2_Mac3_Linker_cts_dict), by = x -> x[2], rev=true)
println("##########################################################################################################################")
for mut___ct in NSP2__Mac2_Mac3_Linker_cts_dict_count_sort
    mut = mut___ct[1]
    ct = mut___ct[2]
    pct = round(digits=2, 100*ct/NSP2__Mac2_Mac3_Linker_ct_total)
    ct_pad = lpad(ct, 3)
    mut_pad = rpad(mut, 12)
    pct_pad = lpad(pct, 5)
    println("$(mut) = $(ct_pad) | $(pct_pad)%")
end
#println(NSP2__Mac2_Mac3_Linker_cts_dict)
#########################
println("Finished!")
####################################################################################################
####################################################################################################
###########################################################################################################################
###########################################################################################################################

2026_01_27_631PM
NSP2_and_Mac2_Mac3_ct = 27
NSP2_and_not_Mac2_Mac3_ct = 2
27/29 NSP2-mut seqs with ≥1 Mac2-Mac3-linker muts (ORF1a:1361-1372) = 93.1%
##########################################################################################################################
Total Mac2_Mac2_Linker muts in NSP2 sequences = 33
Mac2_Mac2_Linker muts per NSP2 sequence = 1.14
ORF1a:S1361P =   3 |  9.09%
ORF1a:E1363V =   1 |  3.03%
ORF1a:Q1365R =   1 |  3.03%
ORF1a:Q1365P =   1 |  3.03%
ORF1a:Q1365E =   1 |  3.03%
ORF1a:I1367L =   5 | 15.15%
ORF1a:I1367V =   1 |  3.03%
ORF1a:L1368I =   1 |  3.03%
ORF1a:G1369R =   6 | 18.18%
ORF1a:S1372C =   5 | 15.15%
ORF1a:S1372A =   7 | 21.21%
ORF1a:S1372V =   1 |  3.03%
##########################################################################################################################
ORF1a:S1372A =   7 | 21.21%
ORF1a:G1369R =   6 | 18.18%
ORF1a:S1372C =   5 | 15.15%
ORF1a:I1367L =   5 | 15.15%
ORF1a:S1361P =   3 |  9.09%
ORF1a:Q1365R =   1 |  3.03%
OR

In [51]:
### Checking Mac2 (ORF1a:1272-1273, 1276) seqs with & without Mac2-Mac3-linker muts (ORF1a:1361-1372)
date_now = Dates.format(now(), "yyyy_mm_dd_IMMp"); println(date_now)
############################################################
Mac2__Mac2Mac3Linker_cts_dict = Dict{String, Int}()
############################################################
Mac2_muts_pos_only = Set(["ORF1a:1272", "ORF1a:1273", "ORF1a:1274", "ORF1a:1275", "ORF1a:1276"])
Mac2_muts = Set(["ORF1a:S1272G", "ORF1a:S1272T", "ORF1a:S1272N", "ORF1a:S1272R", "ORF1a:D1273N", "ORF1a:D1273H", "ORF1a:D1273G", "ORF1a:D1273E", "ORF1a:D1273A", "ORF1a:I1274V", "ORF1a:I1274M", "ORF1a:I1274F", "ORF1a:I1274T", "ORF1a:D1275A", "ORF1a:D1275N", "ORF1a:I1276S", "ORF1a:I1276T", "ORF1a:I1276V", "ORF1a:I1276L"])
Mac2Mac3Linker = Set(["ORF1a:S1361F", "ORF1a:S1361P", "ORF1a:S1361A", "ORF1a:E1363V", "ORF1a:E1363G", "ORF1a:E1363A", "ORF1a:E1363D", "ORF1a:E1363K", "ORF1a:Q1365L", "ORF1a:Q1365K", "ORF1a:Q1365E", "ORF1a:Q1365R", "ORF1a:Q1365P", "ORF1a:E1366G", "ORF1a:E1366D", "ORF1a:E1366K", "ORF1a:E1366A", "ORF1a:I1367L", "ORF1a:I1367M", "ORF1a:I1367V", "ORF1a:L1368I", "ORF1a:G1369R", "ORF1a:S1372A", "ORF1a:S1372C", "ORF1a:S1372V", "ORF1a:S1372P"])
Mac2Mac3Linker_pos_only = Set(["ORF1a:1361", "ORF1a:1362", "ORF1a:1363", "ORF1a:1364", "ORF1a:1365", "ORF1a:1366", "ORF1a:1367", "ORF1a:1368", "ORF1a:1369", "ORF1a:1370", "ORF1a:1371", "ORF1a:1372"])
##############################
Mac2_seqs_set = Set{String}()  
##########
for seq in all_unique_chr_seqs_set
    for mut in Mac2_muts
        if mut in seq_AA_muts[seq]
            push!(Mac2_seqs_set, seq)
        end
    end
end
############################################################
Mac2_and_Mac2Mac3_seqs = Set{String}()
Mac2_and_not_Mac2Mac3_seqs = Set{String}()
Mac2_and_Mac2Mac3_ct = 0
Mac2_and_not_Mac2Mac3_ct = 0
total_Mac2_mut_seqs = length(Mac2_seqs_set)
for seq in Mac2_seqs_set
    amino_A_muts = seq_AA_muts_no_dels[seq]
    M23L_muts = intersect(amino_A_muts, Mac2Mac3Linker)
    amino_A_muts_pos_only = AAmut_to_AApos.(collect(amino_A_muts))
    if !isempty(intersect(amino_A_muts_pos_only, Mac2Mac3Linker_pos_only))
        Mac2_and_Mac2Mac3_ct += 1
        push!(Mac2_and_Mac2Mac3_seqs, seq)
        for mut in M23L_muts
            Mac2__Mac2Mac3Linker_cts_dict[mut] = get!(Mac2__Mac2Mac3Linker_cts_dict, mut, 0) + 1
        end
    else
        Mac2_and_not_Mac2Mac3_ct += 1
        push!(Mac2_and_not_Mac2Mac3_seqs, seq)
    end
end
Mac2_and_Mac2Mac3_pct = round(digits=1, 100*Mac2_and_Mac2Mac3_ct/total_Mac2_mut_seqs)
############################################################
println("Mac2_and_Mac2Mac3_ct = $(Mac2_and_Mac2Mac3_ct)")
println("Mac2_and_not_Mac2Mac3_ct = $(Mac2_and_not_Mac2Mac3_ct)")
println("$(Mac2_and_Mac2Mac3_ct)/$(total_Mac2_mut_seqs) Mac2-mut seqs with ≥1 Mac2-Mac3-linker muts (ORF1a:1361-1372) = $(Mac2_and_Mac2Mac3_pct)%")
####################################################################################################
####################################################################################################
#Mac2_and_not_Mac2Mac3_seqs_plus_related = Set(["", "", "", "", ""])
#Mac2_and_not_Mac2Mac3_seqs_plus_related_sort = sort(collect(Mac2_and_not_Mac2Mac3_seqs_plus_related), by = x -> (length(x), x))
#for seq in Mac2_and_not_Mac2Mac3_seqs_plus_related_sort
#    txtout = "all_seq_info/all_seq_info_$(seq)_$(date_now).txt"
#    print_all_seq_info(seq, txtout)
#end
#############################################################################################################################################################################
println("##########################################################################################################################")
Mac2__Mac2Mac3Linker_ct_total = sum(values(Mac2__Mac2Mac3Linker_cts_dict))
Mac2Mac3_muts_per_Mac2_seq = round(digits=2, Mac2__Mac2Mac3Linker_ct_total/total_Mac2_mut_seqs)
println("Total Mac2_Mac2_Linker muts in Mac2 sequences = $(Mac2__Mac2Mac3Linker_ct_total)")
println("Mac2_Mac2_Linker muts per Mac2 sequence = $(Mac2Mac3_muts_per_Mac2_seq)")
#########################
Mac2__Mac2Mac3Linker_cts_dict_sort = sort(collect(Mac2__Mac2Mac3Linker_cts_dict), by = x -> gene_AA_sortKey(x[1]))
for mut___ct in Mac2__Mac2Mac3Linker_cts_dict_sort
    mut = mut___ct[1]
    ct = mut___ct[2]
    pct = round(digits=2, 100*ct/Mac2__Mac2Mac3Linker_ct_total)
    ct_pad = lpad(ct, 3)
    mut_pad = rpad(mut, 12)
    pct_pad = lpad(pct, 5)
    println("$(mut) = $(ct_pad) | $(pct_pad)%")
end
#############################################################################################################################################################################
Mac2__Mac2Mac3Linker_cts_dict_count_sort = sort(collect(Mac2__Mac2Mac3Linker_cts_dict), by = x -> x[2], rev=true)
println("##########################################################################################################################")
for mut___ct in Mac2__Mac2Mac3Linker_cts_dict_count_sort
    mut = mut___ct[1]
    ct = mut___ct[2]
    pct = round(digits=2, 100*ct/Mac2__Mac2Mac3Linker_ct_total)
    ct_pad = lpad(ct, 3)
    mut_pad = rpad(mut, 12)
    pct_pad = lpad(pct, 5)
    println("$(mut) = $(ct_pad) | $(pct_pad)%")
end
println(Mac2__Mac2Mac3Linker_cts_dict)
#########################
println("Finished!")
####################################################################################################
####################################################################################################

2026_01_27_630PM
Mac2_and_Mac2Mac3_ct = 69
Mac2_and_not_Mac2Mac3_ct = 69
69/138 Mac2-mut seqs with ≥1 Mac2-Mac3-linker muts (ORF1a:1361-1372) = 50.0%
##########################################################################################################################
Total Mac2_Mac2_Linker muts in Mac2 sequences = 85
Mac2_Mac2_Linker muts per Mac2 sequence = 0.62
ORF1a:S1361P =   4 |  4.71%
ORF1a:E1363V =   1 |  1.18%
ORF1a:E1363D =   1 |  1.18%
ORF1a:Q1365K =   1 |  1.18%
ORF1a:Q1365R =   6 |  7.06%
ORF1a:Q1365P =   6 |  7.06%
ORF1a:Q1365E =   1 |  1.18%
ORF1a:E1366D =   1 |  1.18%
ORF1a:E1366A =   4 |  4.71%
ORF1a:E1366G =   1 |  1.18%
ORF1a:I1367L =  10 | 11.76%
ORF1a:I1367V =   2 |  2.35%
ORF1a:L1368I =   9 | 10.59%
ORF1a:G1369R =   9 | 10.59%
ORF1a:S1372C =   7 |  8.24%
ORF1a:S1372A =  20 | 23.53%
ORF1a:S1372V =   2 |  2.35%
##########################################################################################################################
ORF1a:S1372A =  20 | 23.53%
OR

In [181]:
### 8fold Wide | Counting number of seqs w/X muts from select list of muts (e.g. 8fold)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
search_moots = Set(["ORF1a:R1318G", "ORF1a:T1322R", "ORF1a:T1322A", "ORF1a:T1322V", "ORF1a:T1322P", "ORF1a:T1322I", "ORF1a:T1322L", "ORF1a:T1322K", "ORF1a:D1323N", "ORF1a:D1323G", "ORF1a:D1323A", "ORF1a:D1323E", "ORF1a:D1323Y", "ORF1a:N1324I", "ORF1a:N1324Y", "ORF1a:N1324K", "ORF1a:N1324D", "ORF1a:N1324H", "ORF1a:N1324T", "ORF1a:N1324E", "ORF1a:N1324S", "ORF1a:N1324G", "ORF1a:I1326L", "ORF1a:I1326V", "ORF1a:I1326T", "ORF1a:T1328N", "ORF1a:T1638I", "ORF1a:T1638A", "ORF1a:T1638L", "ORF1a:T1638V", "ORF1a:T1638Y", "ORF1a:T1638N", "ORF1a:T1638P", "ORF1a:D1639E", "ORF1a:D1639H", "ORF1a:D1639N", "ORF1a:D1639A", "ORF1a:P1640F", "ORF1a:L1640F", "ORF1a:S1640F", "ORF1a:T2274I", "ORF1a:V2276A", "ORF1a:T2277I", "ORF1a:T2280I", "ORF1a:L3440F", "ORF1a:E3441D", "ORF1a:E3441N", "ORF1a:N3443K", "ORF1a:N3443V", "ORF1a:N3443Y", "ORF1a:N3443S", "ORF1a:N3443D", "ORF1a:D4085A", "ORF1a:D4085N", "ORF1a:D4085G", "ORF1a:D4085E", "ORF1a:G4086C", "ORF1a:G4086V", "ORF1a:G4086S", "ORF1a:T4087M", "ORF1a:T4087A", "ORF1a:T4087V", "ORF1a:T4087I", "ORF1a:T4087R", "ORF1a:T4087K", "ORF1a:T4088I", "ORF1a:T4090S", "ORF1a:T4090I", "ORF1a:T4090N", "ORF1a:T4090A", "ORF1a:T4164S", "ORF1a:T4164I", "ORF1a:T4164P", "ORF1a:T4164N", "ORF1a:D4165G", "ORF1a:D4165Y", "ORF1a:D4165T", "ORF1a:D4165V", "ORF1a:D4165A", "ORF1a:D4165N", "ORF1a:D4166E", "ORF1a:D4166G", "ORF1a:D4166S", "ORF1a:D4166A", "ORF1a:D4166Y", "ORF1a:D4166C", "ORF1a:D4166N", "ORF1a:N4167T", "ORF1a:N4167S", "ORF1a:N4167K", "ORF1a:A4168T", "ORF1a:L4169S", "ORF1a:Y4172H", "ORF1b:D729E", "ORF1b:T730I", "ORF1b:D731E", "ORF1b:N734S", "ORF3a:D210N", "ORF3a:D210Y", "ORF3a:D210G", "ORF3a:D210E", "ORF3a:D210A", "ORF3a:Y211H", "ORF3a:Y211C", "ORF3a:Q213K", "ORF3a:Q213L", "ORF3a:Q213R", "ORF3a:Y215C", "ORF3a:Y215H"])
#pattern_ct = 0
counts = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
count_dict = Dict{Int, Int}(i => 0 for i in 1:length(counts))
min_ct = 2
all_unique_chr_seqs_set_len = length(all_unique_chr_seqs_set)
for seq in all_unique_chr_seqs_set
    mutct = 0
    for mut in seq_AA_muts[seq]
        if mut in search_moots
            mutct += 1
        end
    end
    for i in 1:length(counts)
        min_ct = counts[i]
        if mutct ≥ min_ct
            count_dict[i] += 1
#            pattern_ct += 1
#            txtout = "temp_$(seq).txt"
#            println(seq)
#            print_all_seq_info(seq, txtout)
        end
    end
end
for i in 1:length(counts)
    min_ct = i
    pattern_ct = count_dict[i]
    pattern_pct = round(digits=2, 100*pattern_ct/all_unique_chr_seqs_set_len)
    println("Total with ≥$(min_ct) muts in search moots = $(pattern_ct)")
    println("$(min_ct)x-search moots pct = $(pattern_pct)"); println()
end

2026_03_04__823PM
Total with ≥1 muts in search moots = 3297
1x-search moots pct = 100.0

Total with ≥2 muts in search moots = 1038
2x-search moots pct = 31.48

Total with ≥3 muts in search moots = 707
3x-search moots pct = 21.44

Total with ≥4 muts in search moots = 448
4x-search moots pct = 13.59

Total with ≥5 muts in search moots = 252
5x-search moots pct = 7.64

Total with ≥6 muts in search moots = 110
6x-search moots pct = 3.34

Total with ≥7 muts in search moots = 43
7x-search moots pct = 1.3

Total with ≥8 muts in search moots = 18
8x-search moots pct = 0.55

Total with ≥9 muts in search moots = 5
9x-search moots pct = 0.15

Total with ≥10 muts in search moots = 2
10x-search moots pct = 0.06

Total with ≥11 muts in search moots = 0
11x-search moots pct = 0.0

Total with ≥12 muts in search moots = 0
12x-search moots pct = 0.0



In [106]:
### 8fold Wide | Counting number of seqs w/X pos_only_muts from select list of muts (e.g. 8fold——wide borders below)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
search_moots = Set(["ORF1a:1318", "ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1326", "ORF1a:1328", "ORF1a:1638", "ORF1a:1639", "ORF1a:2274", "ORF1a:2275", "ORF1a:2276", "ORF1a:2277", "ORF1a:2278", "ORF1a:2279", "ORF1a:2280", "ORF1a:3440", "ORF1a:3441", "ORF1a:3443", "ORF1a:4085", "ORF1a:4086", "ORF1a:4087", "ORF1a:4088", "ORF1a:4089", "ORF1a:4090", "ORF1a:4092", "ORF1a:4162", "ORF1a:4163", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1a:4168", "ORF1a:4169", "ORF1a:4170", "ORF1a:4171", "ORF1a:4172", "ORF1b:730", "ORF1b:731", "ORF1b:732", "ORF1b:733", "ORF1b:734", "ORF1b:735", "ORF3a:210", "ORF3a:211", "ORF3a:212", "ORF3a:213", "ORF3a:214", "ORF3a:215", "ORF3a:216"])
#pattern_ct = 0
z1 = ["ORF1a:1318", "ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1326", "ORF1a:1328"]
z2 = ["ORF1a:1638", "ORF1a:1639"]
z3 = ["ORF1a:2274", "ORF1a:2275", "ORF1a:2276", "ORF1a:2277", "ORF1a:2278", "ORF1a:2279", "ORF1a:2280"]
z4 = ["ORF1a:3440", "ORF1a:3441", "ORF1a:3443"]
z5 = ["ORF1a:4085", "ORF1a:4086", "ORF1a:4087", "ORF1a:4088", "ORF1a:4089", "ORF1a:4090", "ORF1a:4092"]
z6 = ["ORF1a:4162", "ORF1a:4163", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1a:4168", "ORF1a:4169", "ORF1a:4170", "ORF1a:4171", "ORF1a:4172"]
z7 = ["ORF1b:730", "ORF1b:731", "ORF1b:732", "ORF1b:733", "ORF1b:734", "ORF1b:735"]
z8 = ["ORF3a:210", "ORF3a:211", "ORF3a:212", "ORF3a:213", "ORF3a:214", "ORF3a:215", "ORF3a:216"]
#################################################################################
function pct_of_EPCI(seq_count::Int)
    pct = round(digits=2, 100*seq_count/length(all_unique_chr_seqs_set))
    return pct
end
#################################################################################
println("######################### 8fold stats WIDE BORDERS version ###########################")
zones = [z1, z2, z3, z4, z5, z6, z7, z8]
counts = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
#####################
count_dict = Dict{Int, Int}(i => 0 for i in 1:length(counts))
exact_count_dict = Dict{Int, Int}(i => 0 for i in 1:length(counts))
#####################
count_dict_max_1_per_zone_per_seq = Dict{Int, Int}(i => 0 for i in 0:length(counts))
total_zones_count_dict = Dict{Int, Int}(i => 0 for i in 0:length(counts))
#####################
zones_count_dict = Dict{Int, Int}(i => 0 for i in 1:8)
zones_count_dict_max_1_per_zone_per_seq = Dict{Int, Int}(i => 0 for i in 1:8)
#####################
zone_pattern_dict = Dict{Vector{Int}, Int}()
zone_pattern_dict_max_1_per_zone = Dict{Vector{Int}, Int}()
#####################
min_ct = 2
all_unique_chr_seqs_set_len = length(all_unique_chr_seqs_set)
zone1_not_zone2_seqs = String[]
zone1_and_zone2_seqs = String[]
z1_only_seqs = String[]
z1_only_seqs_ct = 0
#################################################################################
for seq in all_unique_chr_seqs_set
    mutct = 0
    seq_zones_count_dict = Dict{Int, Int}(i => 0 for i in 1:8)
    seq_zones_ct = 0
    seq_zones_vec = Int[]
    seq_zones_max_1_per_zone_ct = 0
    seq_zones_vec_max_1_per_zone = Int[]
    for mut in seq_AA_muts_pos_only_no_dels[seq]
        if mut in search_moots
            mutct += 1
            for i in 1:8
                if mut in zones[i]
                    zones_count_dict[i] += 1
                    seq_zones_count_dict[i] += 1
                    push!(seq_zones_vec, i)
                    if !(i in seq_zones_vec_max_1_per_zone)
                        push!(seq_zones_vec_max_1_per_zone, i)
                        seq_zones_max_1_per_zone_ct += 1
                    end
                end
            end 
        end
    end
    tot_zones = seq_zones_max_1_per_zone_ct
    count_dict_max_1_per_zone_per_seq[tot_zones] += 1
    if tot_zones ≥ 2
        if 1 in seq_zones_vec_max_1_per_zone && !(2 in seq_zones_vec_max_1_per_zone)
            push!(zone1_not_zone2_seqs, seq)
        end
        if 1 in seq_zones_vec_max_1_per_zone && 2 in seq_zones_vec_max_1_per_zone
            push!(zone1_and_zone2_seqs, seq)
        end
    end
    for i in 1:8
        if seq_zones_count_dict[i] ≥ 1
            zones_count_dict_max_1_per_zone_per_seq[i] += 1
        end
    end
    for i in 1:length(counts)
        min_ct = i
        if mutct ≥ min_ct
            count_dict[i] += 1
        end
        if mutct == i
            exact_count_dict[i] += 1
        end
#            txtout = "temp_$(seq).txt";   println(seq)
#            print_all_seq_info(seq, txtout)
    end
    if mutct ≥ 7
        println("$(seq), $(seq_collection_date[seq]), $(seq_pango[seq]), $(seq_country[seq])")
    end
    if seq_zones_vec_max_1_per_zone == [1]
        push!(z1_only_seqs, seq)
        z1_only_seqs_ct += 1
    end
    seq_zones_vec_sort = sort(seq_zones_vec)
    zone_pattern_dict[seq_zones_vec_sort] = get!(zone_pattern_dict, seq_zones_vec_sort, 0) + 1
    seq_zones_vec_max_1_per_zone_sort = sort(seq_zones_vec_max_1_per_zone)
    zone_pattern_dict_max_1_per_zone[seq_zones_vec_max_1_per_zone_sort] = get!(zone_pattern_dict_max_1_per_zone, seq_zones_vec_max_1_per_zone_sort, 0) + 1
end
##########################################################################################################################################
z2_region = ["ORF1a:1628", "ORF1a:1629", "ORF1a:1630", "ORF1a:1631", "ORF1a:1632", "ORF1a:1633", "ORF1a:1634", "ORF1a:1635", "ORF1a:1636", "ORF1a:1637", "ORF1a:1638", "ORF1a:1639", "ORF1a:1640", "ORF1a:1641", "ORF1a:1642", "ORF1a:1643", "ORF1a:1644", "ORF1a:1645", "ORF1a:1646", "ORF1a:1647", "ORF1a:1648", "ORF1a:1649"]
z1_only_z2_dropout_seqs = String[]
z1_only_z2_dropout_seqs_ct = 0
for seq in z1_only_seqs
    for mutpo in z2_region
        if mutpo in seq_unknown_AA[seq]
            if !(seq in z1_only_z2_dropout_seqs)
                push!(z1_only_z2_dropout_seqs, seq)
                z1_only_z2_dropout_seqs_ct += 1
            end
        end
    end
end
total_z1_only_seqs = length(z1_only_seqs)
total_z1_only_z2_dropout_seqs = length(z1_only_z2_dropout_seqs)
print("\n"^2)
println("Total z1_only_seqs = $(total_z1_only_seqs)")
println("Total z1_only_z2_dropout_seqs = $(total_z1_only_z2_dropout_seqs)")
print("\n"^2)          
######################################################################################################
open("Eightfold_pos_only_cts_zoneCts_zonePatternCts_WIDE_BORDERS_$(date_now).txt", "w") do g
    println("######################### 8fold stats WIDE BORDERS version ###########################"); println()
    println(g, "######################### 8fold stats WIDE BORDERS version ###########################"); println(g)
    for i in 1:length(counts)
        min_ct = i
        exact_count = exact_count_dict[i]
        exact_count_pct = pct_of_EPCI(exact_count)
################
        println("Total with exactly $(min_ct) muts = $(exact_count)")
        println("       Exactly-$(min_ct) muts pct = $(exact_count_pct)%") #; println()
        println(g, "Total with exactly $(min_ct) muts = $(exact_count)")
        println(g, "       Exactly-$(min_ct) muts pct = $(exact_count_pct)%") #; println(g)
    end
    print("\n"^3)
################################################################
    for i in 1:length(counts)
        min_ct = i
        cumulative_pattern_ct = count_dict[i]
        cumulative_pattern_pct = pct_of_EPCI(cumulative_pattern_ct)
################
        println("Total with ≥$(min_ct) muts in search moots = $(cumulative_pattern_ct)")
        println("            $(min_ct)x-search muts pct = $(cumulative_pattern_pct)%") #; println()
        println(g, "Total with ≥$(min_ct) muts in search moots = $(cumulative_pattern_ct)")
        println(g, "            $(min_ct)x-search muts pct = $(cumulative_pattern_pct)%") #; println(g)
    end
    print("\n"^3)
######################################################################################################
    for i in 1:length(counts)
        min_ct = i
        exact_pattern_ct = count_dict_max_1_per_zone_per_seq[i]
        exact_pattern_pct = pct_of_EPCI(exact_pattern_ct)
################
        println("Total with exactly $(min_ct) zone-muts = $(exact_pattern_ct)")
        println("      Exactly-$(min_ct) muts pct = $(exact_pattern_pct)%") #; println()
        println(g, "Total with exactly $(min_ct) zone-muts = $(exact_pattern_ct)")
        println(g, "      Exactly-$(min_ct) muts pct = $(exact_pattern_pct)%") #; println(g)
    end
    print("\n"^4)
################################################################
    cumulative_zone_count = 0
    for i in 1:length(counts)
        min_ct = i
        exact_pattern_ct = count_dict_max_1_per_zone_per_seq[i]
        cumulative_zone_count += exact_pattern_ct
        cumulative_pattern_pct = pct_of_EPCI(cumulative_zone_count)
################
        println("Total with ≥$(min_ct) zone-muts in search moots = $(cumulative_zone_count)")
        println("            ≥$(min_ct)x-zone muts pct = $(cumulative_pattern_pct)%") #; println()
        println(g, "Total with ≥$(min_ct) zone-muts in search moots = $(cumulative_zone_count)")
        println(g, "            ≥$(min_ct)x-zone muts pct = $(cumulative_pattern_pct)%") #; println(g)
    end
######################################################################################################
    print("\n"^3)
    zone_pattern_dict_sort = sort(collect(zone_pattern_dict), by = x -> x[1])
    zone_pattern_dict_max_1_per_zone_sort = sort(collect(zone_pattern_dict_max_1_per_zone), by = x -> x[1])
    for zone__ct in zone_pattern_dict_sort
        zone = zone__ct[1]
        ct = zone__ct[2]
        zone_join = join(zone, "-")
        println("$(zone_join) = $(ct)")
        println(g, "$(zone_join) = $(ct)")
    end
    print("\n"^3)
    for zone__ct in zone_pattern_dict_max_1_per_zone_sort
        zone = zone__ct[1]
        ct = zone__ct[2]
        zone_join = join(zone, "-")
        println("$(zone_join) = $(ct)")
        println(g, "$(zone_join) = $(ct)")
    end
    print("\n"^3)
    print(g, "\n"^3)
    zone1_not_zone2_seqs_sort = sort(zone1_not_zone2_seqs, by = x-> (length(x), x))
    zone1_and_zone2_seqs_sort = sort(zone1_and_zone2_seqs, by = x-> (length(x), x))
    total_z1_not_z2_with_2_or_more_8fold_muts = length(zone1_not_zone2_seqs_sort)
    total_z1_and_z2 = length(zone1_and_zone2_seqs_sort)
    z1_not_z2_pct = pct_of_EPCI(total_z1_not_z2_with_2_or_more_8fold_muts)
    z1_and_z2_pct = pct_of_EPCI(total_z1_and_z2)
    total_z1_not_z2_including_z1_singlets = total_z1_not_z2_with_2_or_more_8fold_muts + z1_only_seqs_ct
    total_z1_not_z2_including_z1_singlets_EPCI_pct = pct_of_EPCI(total_z1_not_z2_including_z1_singlets)
################################################################
    println("Total EPCI seqs with mut in z1 but NOT z2 = $(total_z1_not_z2_including_z1_singlets); Pct of all EPCI = $(total_z1_not_z2_including_z1_singlets_EPCI_pct)%")
    println("Total EPCI seqs with ≥2 8fold muts & mut in z1 but NOT z2 = $(total_z1_not_z2_with_2_or_more_8fold_muts); Pct of all EPCI = $(z1_not_z2_pct)%")
    println("Total EPCI seqs with ≥2 8fold muts & mut in z1 AND z2 = $(total_z1_and_z2); Pct of all EPCI = $(z1_and_z2_pct)%")
################################################################
    println(g, "Total EPCI seqs with mut in z1 but NOT z2 = $(total_z1_not_z2_including_z1_singlets); Pct of all EPCI = $(total_z1_not_z2_including_z1_singlets_EPCI_pct)%")
    println(g, "Total EPCI seqs with ≥2 8fold muts & mut in z1 but NOT z2 = $(total_z1_not_z2_with_2_or_more_8fold_muts); Pct of all EPCI = $(z1_not_z2_pct)%")
    println(g, "Total EPCI seqs with ≥2 8fold muts & mut in z1 AND z2 = $(total_z1_and_z2); Pct of all EPCI = $(z1_and_z2_pct)%")
################################################################
    zone1_not_zone2_seqs_with_z2_dropout = String[]
    for seq in zone1_not_zone2_seqs_sort
        for mutpo in z2_region
            if mutpo in seq_unknown_AA[seq]
                if !(seq in zone1_not_zone2_seqs_with_z2_dropout)
                    push!(zone1_not_zone2_seqs_with_z2_dropout, seq)
                end
            end
        end
    end
    z2_dropouts = length(zone1_not_zone2_seqs_with_z2_dropout)
    final_total_z1_not_z2_with_2_or_more_8fold_muts = total_z1_not_z2_with_2_or_more_8fold_muts - z2_dropouts
    final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts = round(digits=2, 100*(total_z1_and_z2/(total_z1_and_z2 + final_total_z1_not_z2_with_2_or_more_8fold_muts)))
    println("#######################################################################################################################")
    println("Final adjusted pct of 8fold seqs (≥2 8fold muts) with z1 that also have z2 muts = $(final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts)")
    println(g, "#######################################################################################################################")
    println(g, "Final adjusted pct of 8fold seqs (≥2 8fold muts) with z1 that also have z2 muts = $(final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts)")
    final_adjusted_z1_not_z2_including_z1_singlets_ct = total_z1_not_z2_with_2_or_more_8fold_muts + z1_only_seqs_ct - z2_dropouts - z1_only_z2_dropout_seqs_ct
    final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets = round(digits=2, 100*total_z1_and_z2/(total_z1_and_z2 + final_adjusted_z1_not_z2_including_z1_singlets_ct))
    println("#######################################################################################################################")
    println("Final adjusted number of z1 but not z2 seqs including z1 singlets = $(final_adjusted_z1_not_z2_including_z1_singlets_ct)")
    println("Final adjusted total z1 EPCI seqs = $(final_adjusted_z1_not_z2_including_z1_singlets_ct + total_z1_and_z2)")
    println("Final adjusted pct of EPCI z1-mut seqs that also have a z2-mut = $(final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets)%")
    println("#######################################################################################################################")
    println(g, "#######################################################################################################################")
    println(g, "Final adjusted number of z1 but not z2 seqs including z1 singlets = $(final_adjusted_z1_not_z2_including_z1_singlets_ct)")
    println(g, "Final adjusted total z1 EPCI seqs = $(final_adjusted_z1_not_z2_including_z1_singlets_ct + total_z1_and_z2)")
    println(g, "Final adjusted pct of EPCI z1-mut seqs that also have a z2-mut = $(final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets)%")
    println(g, "#######################################################################################################################")
    for seq in zone1_not_zone2_seqs_with_z2_dropout
        println("Seq with z1 & not z2 mut with z2 dropout = $(seq)")
        println(g, "Seq with z1 & not z2 mut with z2 dropout = $(seq)")
    end
    print("\n"^2); print(g, "\n"^2)
end
###############################################################################################################################################
###############################################################################################################################################

2026_01_05__651AM
######################### 8fold stats WIDE BORDERS version ###########################
EPI_ISL_15218165, 2022-09-14, BA.1.1.18, USA
EPI_ISL_16585286, 2022-12-28, BA.2, Netherlands
EPI_ISL_19642771, 2024-01-0, BA.2, Saudi_Arabia
EPI_ISL_15845778, 2022-10-26, BA.2.12.1, USA
EPI_ISL_19561072, 2024-11-10, BA.1.1, Israel
EPI_ISL_17997917, 2023-05-22, BA.5.11, USA
EPI_ISL_18290989, 2023-04-25, AY.122, Romania
EPI_ISL_17257608, 2023-03-09, BA.1.1, Italy
EPI_ISL_11437359, 2022-01-17, B.1.2, USA
EPI_ISL_17782148, 2023-05-24, BA.2.12.1, USA
EPI_ISL_18952873, 2024-02-07, BN.1.5.2, Netherlands
EPI_ISL_13741330, 2022-06-27, BA.1.1, England
EPI_ISL_18359328, 2023-08-23, BA.2, Spain
EPI_ISL_5323016, 2021-09-29, B.1.1.7, Sweden
EPI_ISL_18134315, 2023-08-05, BA.5.2.2, Norway
EPI_ISL_17780726, 2023-05-26, AY.122, Italy
EPI_ISL_18450249, 2023-10-19, BA.5.2.35, England
EPI_ISL_20019468, 2023-02-23, BA.4.4, Canada
EPI_ISL_18470400, 2023-10-25, EG.6.1, Canada
EPI_ISL_19512930, 2024-10-17, 

In [ ]:
search_moots = Set(["ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1638", "ORF1a:1639", "ORF1a:2274", "ORF1a:2279", "ORF1a:2280", "ORF1a:3441", "ORF1a:3443", "ORF1a:4087", "ORF1a:4088", "ORF1a:4090", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1b:730", "ORF3a:211", "ORF3a:213", "ORF3a:215"])
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
z9 = ["S:865", "S:866", "S:867", "S:868", "S:869", "S:870", "S:871", "S:872", "S:873"]
z9_seqs = String[]
z9_seq_ct = 0
z9_8fold_ct_dict = Dict{String, Int}()
total_z9seq_8fold_ct = 0
total_z9_mut_ct = 0
for seq in all_unique_chr_seqs_set
    seq_z9mutct = 0
    seq_z9_8fold_ct = 0
    for mut in seq_AA_muts_pos_only_no_dels[seq]
        if mut in z9
            seq_z9mutct += 1
            if !(seq in z9seqs)
                push!(z9_seqs, seq)
                z9_seq_ct += 1
            end
        end
        if mut in search_moots
            seq_z9_8fold_ct += 1
        end
    end
    z9_8fold_ct_dict[seq_z9_8fold_ct] = get!(z9_8fold_ct_dict, seq_z9_8fold_ct, 0) + 1
    total_z9seq_8fold_ct += seq_z9_8fold_ct
end
avg_8fold_muts_per_z9seq = round(digits=2, total_z9seq_8fold_ct/z9_seq_ct)
print("\n"^1)
println("Total z9_seq_ct = $(z9_seq_ct)")
println("Total z9_seq_8fold_ct = $(total_z9seq_8fold_ct)")
println("Avg 8fold muts per z9_seq = $(avg_8fold_muts_per_z9seq)")
for i in 1:10
    println("Total z9_seqs with ≥")
println("")
println("")
println("")
println("")

        


In [160]:
seq_8fold_dropout_zones_wide = Dict{String, Vector{Int}}()
## Below is 8fold in which a seq has dropout and no mutations in that zone
seq_8fold_dropout_zones_effective_wide = Dict{String, Vector{Int}}()
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
z1 = ["ORF1a:1318", "ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1326", "ORF1a:1328"]
z2 = ["ORF1a:1638", "ORF1a:1639"]
z3 = ["ORF1a:2274", "ORF1a:2275", "ORF1a:2276", "ORF1a:2277", "ORF1a:2278", "ORF1a:2279", "ORF1a:2280"]
z4 = ["ORF1a:3440", "ORF1a:3441", "ORF1a:3443"]
z5 = ["ORF1a:4085", "ORF1a:4086", "ORF1a:4087", "ORF1a:4088", "ORF1a:4089", "ORF1a:4090", "ORF1a:4092"]
z6 = ["ORF1a:4162", "ORF1a:4163", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1a:4168", "ORF1a:4169", "ORF1a:4170", "ORF1a:4171", "ORF1a:4172"]
z7 = ["ORF1b:730", "ORF1b:731", "ORF1b:732", "ORF1b:733", "ORF1b:734", "ORF1b:735"]
z8 = ["ORF3a:210", "ORF3a:211", "ORF3a:212", "ORF3a:213", "ORF3a:214", "ORF3a:215", "ORF3a:216"]
z_vec_wide = [z1, z2, z3, z4, z5, z6, z7, z8]
####                  8fold_zone   8fold_zone_dropout_sites (zone sites ± 10 AA sites)
dropout_zones_dict_wide = Dict{Int, Vector{String}}()
no_unknown_AA_key_seq_set = Set{String}()
no_unknown_AA_key_ct = 0
for i in 1:8
    dropout_zone = String[]
    dropout_zones_dict_wide[i] = String[]
    zone = z_vec_wide[i]
    zone_first_mut = zone[1]
    zone_last_mut = zone[end]
    zone_first_int = aa_pos_comprehensive_dict[zone_first_mut]
    zone_last_int = aa_pos_comprehensive_dict[zone_last_mut]
    gene = aa_gene_comprehensive_dict[zone_first_mut]
    dropout_zone_first_int = zone_first_int - 10
    dropout_zone_last_int = zone_last_int + 10
    for j in dropout_zone_first_int:dropout_zone_last_int
        drop_site = "$(gene):$(j)"
        push!(dropout_zones_dict_wide[i], drop_site)
    end
end
for seq in all_seqs_set
    seq_8fold_dropout_zones_wide[seq] = Vector{Int}()
    for i in 1:8
        dropout_zone = dropout_zones_dict_wide[i]
        if !haskey(seq_unknown_AA, seq)
            if !(seq in no_unknown_AA_key_seq_set)
#                println(seq)
                no_unknown_AA_key_ct += 1
            end
            push!(no_unknown_AA_key_seq_set, seq)
        elseif !isempty(intersect(dropout_zone, seq_unknown_AA[seq]))
            push!(seq_8fold_dropout_zones_wide[seq], i)
        end
    end
end
seq_unknown_AA_no_key_unique_ct = 0
for seq in no_unknown_AA_key_seq_set
    if seq in all_unique_chr_seqs_set
        seq_unknown_AA_no_key_unique_ct += 1
    end
end 
println("Total seqs w/no seq_unknown_AA key = $(no_unknown_AA_key_ct)")
println("Total seqs on all_unique_chr_seqs_set list w/no seq_unknown_AA key = $(seq_unknown_AA_no_key_unique_ct)"); println()
#####################################################################################################################################################################
#####################################################################################################################################################################
seq_8fold_mut_ct_wide = Dict{String, Int}()
seq_8fold_zones_ct_wide = Dict{String, Int}()
seq_8fold_zone_array_wide = Dict{String, Vector{Int}}()
seq_8fold_unique_zone_array_wide = Dict{String, Vector{Int}}()
NoKey_seqs_for_seq_AA_muts_pos_only_no_dels = Set(["EPI_ISL_8725398", "EPI_ISL_949208", "EPI_ISL_17820258"])
for seq in all_seqs_set
    seq_8fold_mut_ct_wide[seq] = 0
    seq_8fold_zones_ct_wide[seq] = 0
    seq_8fold_zone_array_wide[seq] = Int[]
    seq_8fold_unique_zone_array_wide[seq] = Int[]
    for i in 1:length(z_vec_wide)
        zone = z_vec_wide[i]
        if !(seq in NoKey_seqs_for_seq_AA_muts_pos_only_no_dels)
            for mut in seq_AA_muts_pos_only_no_dels[seq]
                if mut in zone
                    seq_8fold_mut_ct_wide[seq] += 1
                    push!(seq_8fold_zone_array_wide[seq], i)
                    if !(i in seq_8fold_unique_zone_array_wide[seq])
                        push!(seq_8fold_unique_zone_array_wide[seq], i)
                        seq_8fold_zones_ct_wide[seq] += 1
                    end
                end
            end
        end
    end
    sequence_dropout_zones = seq_8fold_dropout_zones_wide[seq]
    sequence_8fold_zones = seq_8fold_unique_zone_array_wide[seq]
    effective_sequence_dropout_zones = setdiff(sequence_dropout_zones, sequence_8fold_zones)
    effective_seq_dropout_sort = sort(effective_sequence_dropout_zones)
    seq_8fold_dropout_zones_effective_wide[seq] = effective_seq_dropout_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
good_ct = 0
bad_ct = 0
for seq in all_unique_chr_seqs_set
    if seq_8fold_zones_ct_wide[seq] ≥ 2
        effective_drop_zones_for_sequence = seq_8fold_dropout_zones_effective_wide[seq]
        unique_8fold_zones_for_seq = seq_8fold_unique_zone_array_wide[seq]
        sequence_dropout_zones = seq_8fold_dropout_zones_wide[seq]
        effective_sequence_dropout_zones = setdiff(sequence_dropout_zones, unique_8fold_zones_for_seq)
        if effective_drop_zones_for_sequence == effective_sequence_dropout_zones
            good_ct += 1
        else
            bad_ct += 1
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
############################################################################################################################################################################
four_8fold_mut_ct = 0
three_8fold_mut_ct = 0
four_8fold_zone_ct = 0
three_8fold_zone_ct = 0
for seq in all_unique_chr_seqs_set
    if seq_8fold_zones_ct_wide[seq] ≥ 4
        four_8fold_zone_ct += 1
    end
    if seq_8fold_mut_ct_wide[seq] ≥ 4
        four_8fold_mut_ct += 1
    end
    if seq_8fold_zones_ct_wide[seq] ≥ 3
        three_8fold_zone_ct += 1
    end
    if seq_8fold_mut_ct_wide[seq] ≥ 3
        three_8fold_mut_ct += 1
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_z1 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_z1_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_z2 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_z2_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct = 0
################################################################################
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 4 && !(1 in dropout_plus_mut_8fold_zones) && !(2 in dropout_plus_mut_8fold_zones)
        four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2, seq)
    end
end
################################################
println("############ Seq lacking z1 but w/muts in 4 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 4 && !(1 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        four_8fold_muts_and_no_mut_or_dropout_in_z1_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_z1, seq)
    end
end; print("\n"^1)
println("############ Seq lacking z2 but w/muts in 4 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 4 && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        four_8fold_muts_and_no_mut_or_dropout_in_z2_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_z2, seq)
    end
end; print("\n"^2)
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = union(four_8fold_muts_and_no_mut_or_dropout_in_z1, four_8fold_muts_and_no_mut_or_dropout_in_z2)
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = length(four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_z1 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_z1_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_z2 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_z2_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct = 0
################################################################################
println("############# seqs with ≥3 zones with 8fold muts but NEITHER z1 NOR z2 ##############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 3 && !(1 in dropout_plus_mut_8fold_zones) && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2, seq)
    end
end; print("\n"^1)
################################################
println("############ Seq lacking z1 but w/muts in 3 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 3 && !(1 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        three_8fold_muts_and_no_mut_or_dropout_in_z1_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_z1, seq)
    end
end; print("\n"^1)
println("############ Seq lacking z2 but w/muts in 3 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_wide[seq], seq_8fold_unique_zone_array_wide[seq])
    if seq_8fold_zones_ct_wide[seq] ≥ 3 && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        three_8fold_muts_and_no_mut_or_dropout_in_z2_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_z2, seq)
    end
end; print("\n"^2)
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = union(three_8fold_muts_and_no_mut_or_dropout_in_z1, three_8fold_muts_and_no_mut_or_dropout_in_z2)
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = length(three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both)
###########################################################################################################################################################################
println("Total Number of Seqs with ≥ 4 zones with 8fold muts = $(four_8fold_zone_ct)")
println("Total Number of Seqs with ≥ 4 8fold muts = $(four_8fold_mut_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in both z1 & z2 = $(four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in z1 = $(four_8fold_muts_and_no_mut_or_dropout_in_z1_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in z2 = $(four_8fold_muts_and_no_mut_or_dropout_in_z2_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in either z1 or z2 or both = $(four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct)"); print("\n"^1)
###########################################################################################################################################################################
println("Total Number of Seqs with ≥ 3 zones with 8fold muts = $(three_8fold_zone_ct)")
println("Total Number of Seqs with ≥ 3 8fold muts = $(three_8fold_mut_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in both z1 & z2 = $(three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in z1 = $(three_8fold_muts_and_no_mut_or_dropout_in_z1_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in z2 = $(three_8fold_muts_and_no_mut_or_dropout_in_z2_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in either z1 or z2 or both = $(three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct)"); print("\n"^1)
###########################################################################################################################################################################
###########################################################################################################################################################################

2026_01_07__856PM
Total seqs w/no seq_unknown_AA key = 42
Total seqs on all_unique_chr_seqs_set list w/no seq_unknown_AA key = 0

############ Seq lacking z1 but w/muts in 4 other regions ############
EPI_ISL_19613481, EPI_ISL_19299051, EPI_ISL_15218165, EPI_ISL_19761402, EPI_ISL_14509715, EPI_ISL_17113114, EPI_ISL_15958934, 
############ Seq lacking z2 but w/muts in 4 other regions ############
EPI_ISL_19369713, 

############# seqs with ≥3 zones with 8fold muts but NEITHER z1 NOR z2 ##############
EPI_ISL_19892080, 
############ Seq lacking z1 but w/muts in 3 other regions ############
EPI_ISL_19613481, EPI_ISL_19299051, EPI_ISL_15218165, EPI_ISL_19560838, EPI_ISL_16868654, EPI_ISL_10230612, EPI_ISL_15040867, EPI_ISL_15743318, EPI_ISL_15797751, EPI_ISL_19761402, EPI_ISL_19892080, EPI_ISL_2096935, EPI_ISL_14509715, EPI_ISL_17113114, EPI_ISL_15958934, EPI_ISL_15145892, 
############ Seq lacking z2 but w/muts in 3 other regions ############
EPI_ISL_19135505, EPI_ISL_19369713, EPI_ISL_19

In [155]:
seq_8fold_dropout_zones_narrow = Dict{String, Vector{Int}}()
## Below is 8fold in which a seq has dropout and no mutations in that zone
seq_8fold_dropout_zones_effective_narrow = Dict{String, Vector{Int}}()
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now); print("\n"^1)
z1 = ["ORF1a:1322", "ORF1a:1323", "ORF1a:1324"]
z2 = ["ORF1a:1638", "ORF1a:1639"]
z3 = ["ORF1a:2274", "ORF1a:2279", "ORF1a:2280"]
z4 = ["ORF1a:3441", "ORF1a:3443"]
z5 = ["ORF1a:4087", "ORF1a:4088", "ORF1a:4090", "ORF1a:4092"]
z6 = ["ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167"]
z7 = ["ORF1b:730"]
z8 = ["ORF3a:211", "ORF3a:213", "ORF3a:215"]
z_vec_narrow = [z1, z2, z3, z4, z5, z6, z7, z8]
####                  8fold_zone   8fold_zone_dropout_sites (zone sites ± 10 AA sites)
dropout_zones_dict_narrow = Dict{Int, Vector{String}}()
no_unknown_AA_key_seq_set = Set{String}()
no_unknown_AA_key_ct = 0
for i in 1:8
    dropout_zone = String[]
    dropout_zones_dict_narrow[i] = String[]
    zone = z_vec_narrow[i]
    zone_first_mut = zone[1]
    zone_last_mut = zone[end]
    zone_first_int = aa_pos_comprehensive_dict[zone_first_mut]
    zone_last_int = aa_pos_comprehensive_dict[zone_last_mut]
    gene = aa_gene_comprehensive_dict[zone_first_mut]
    dropout_zone_first_int = zone_first_int - 10
    dropout_zone_last_int = zone_last_int + 10
    for j in dropout_zone_first_int:dropout_zone_last_int
        drop_site = "$(gene):$(j)"
        push!(dropout_zones_dict_narrow[i], drop_site)
    end
end
for seq in all_seqs_set
    seq_8fold_dropout_zones_narrow[seq] = Vector{Int}()
    for i in 1:8
        dropout_zone = dropout_zones_dict_narrow[i]
        if !haskey(seq_unknown_AA, seq)
            if !(seq in no_unknown_AA_key_seq_set)
#                println(seq)
                no_unknown_AA_key_ct += 1
            end
            push!(no_unknown_AA_key_seq_set, seq)
        elseif !isempty(intersect(dropout_zone, seq_unknown_AA[seq]))
            push!(seq_8fold_dropout_zones_narrow[seq], i)
        end
    end
end
seq_unknown_AA_no_key_unique_ct = 0
for seq in no_unknown_AA_key_seq_set
    if seq in all_unique_chr_seqs_set
        seq_unknown_AA_no_key_unique_ct += 1
    end
end 
#println("Total seqs w/no seq_unknown_AA key = $(no_unknown_AA_key_ct)")
#println("Total seqs on all_unique_chr_seqs_set list w/no seq_unknown_AA key = $(seq_unknown_AA_no_key_unique_ct)"); println()
#####################################################################################################################################################################
#####################################################################################################################################################################
seq_8fold_mut_ct_narrow = Dict{String, Int}()
seq_8fold_zones_ct_narrow = Dict{String, Int}()
seq_8fold_zone_array_narrow = Dict{String, Vector{Int}}()
seq_8fold_unique_zone_array_narrow = Dict{String, Vector{Int}}()
NoKey_seqs_for_seq_AA_muts_pos_only_no_dels = Set(["EPI_ISL_8725398", "EPI_ISL_949208", "EPI_ISL_17820258"])
for seq in all_seqs_set
    seq_8fold_mut_ct_narrow[seq] = 0
    seq_8fold_zones_ct_narrow[seq] = 0
    seq_8fold_zone_array_narrow[seq] = Int[]
    seq_8fold_unique_zone_array_narrow[seq] = Int[]
    for i in 1:length(z_vec_narrow)
        zone = z_vec_narrow[i]
        if !(seq in NoKey_seqs_for_seq_AA_muts_pos_only_no_dels)
            for mut in seq_AA_muts_pos_only_no_dels[seq]
                if mut in zone
                    seq_8fold_mut_ct_narrow[seq] += 1
                    push!(seq_8fold_zone_array_narrow[seq], i)
                    if !(i in seq_8fold_unique_zone_array_narrow[seq])
                        push!(seq_8fold_unique_zone_array_narrow[seq], i)
                        seq_8fold_zones_ct_narrow[seq] += 1
                    end
                end
            end
        end
    end
    sequence_dropout_zones = seq_8fold_dropout_zones_narrow[seq]
    sequence_8fold_zones = seq_8fold_unique_zone_array_narrow[seq]
    effective_sequence_dropout_zones = setdiff(sequence_dropout_zones, sequence_8fold_zones)
    effective_seq_dropout_sort = sort(effective_sequence_dropout_zones)
    seq_8fold_dropout_zones_effective_narrow[seq] = effective_seq_dropout_sort
end
###########################################################################################################################################################################
###########################################################################################################################################################################
good_ct = 0
bad_ct = 0
for seq in all_unique_chr_seqs_set
    if seq_8fold_zones_ct_narrow[seq] ≥ 2
        effective_drop_zones_for_sequence = seq_8fold_dropout_zones_effective_narrow[seq]
        unique_8fold_zones_for_seq = seq_8fold_unique_zone_array_narrow[seq]
        sequence_dropout_zones = seq_8fold_dropout_zones_narrow[seq]
        effective_sequence_dropout_zones = setdiff(sequence_dropout_zones, unique_8fold_zones_for_seq)
        if effective_drop_zones_for_sequence == effective_sequence_dropout_zones
            good_ct += 1
        else
            bad_ct += 1
        end
    end
end
###########################################################################################################################################################################
###########################################################################################################################################################################
############################################################################################################################################################################
four_8fold_mut_ct = 0
three_8fold_mut_ct = 0
four_8fold_zone_ct = 0
three_8fold_zone_ct = 0
for seq in all_unique_chr_seqs_set
    if seq_8fold_zones_ct_narrow[seq] ≥ 4
        four_8fold_zone_ct += 1
    end
    if seq_8fold_mut_ct_narrow[seq] ≥ 4
        four_8fold_mut_ct += 1
    end
    if seq_8fold_zones_ct_narrow[seq] ≥ 3
        three_8fold_zone_ct += 1
    end
    if seq_8fold_mut_ct_narrow[seq] ≥ 3
        three_8fold_mut_ct += 1
    end
end
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_z1 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_z1_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_z2 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_z2_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = 0
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2 = Set{String}()
four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct = 0
################################################################################
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 4 && !(1 in dropout_plus_mut_8fold_zones) && !(2 in dropout_plus_mut_8fold_zones)
        four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2, seq)
    end
end
################################################
println("############ Seqs lacking z1 but w/muts in 4 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 4 && !(1 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        four_8fold_muts_and_no_mut_or_dropout_in_z1_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_z1, seq)
    end
end; print("\n")
println("############ Seqs lacking z2 but w/muts in 4 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 4 && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        four_8fold_muts_and_no_mut_or_dropout_in_z2_ct += 1
        push!(four_8fold_muts_and_no_mut_or_dropout_in_z2, seq)
    end
end; print("\n")
################################################################################
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = union(four_8fold_muts_and_no_mut_or_dropout_in_z1, four_8fold_muts_and_no_mut_or_dropout_in_z2)
four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = length(four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both)
############################################################################################################################################################################
############################################################################################################################################################################
############################################################################################################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_z1 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_z1_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_z2 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_z2_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = 0
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2 = Set{String}()
three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct = 0
################################################################################
println("############ Seqs with ≥3 zones with 8fold muts but NEITHER z1 NOR z2 #########")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 3 && !(1 in dropout_plus_mut_8fold_zones) && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq)"); print("\n"^1)
        three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2, seq)
    end
end; print("\n"^1)
################################################
println("############ Seq lacking z1 but w/muts in 3 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 3 && !(1 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        three_8fold_muts_and_no_mut_or_dropout_in_z1_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_z1, seq)
    end
end; print("\n"^1)
println("############ Seq lacking z2 but w/muts in 3 other regions ############")
for seq in all_unique_chr_seqs_set
    dropout_plus_mut_8fold_zones = union(seq_8fold_dropout_zones_narrow[seq], seq_8fold_unique_zone_array_narrow[seq])
    if seq_8fold_zones_ct_narrow[seq] ≥ 3 && !(2 in dropout_plus_mut_8fold_zones)
        print("$(seq), ")
        three_8fold_muts_and_no_mut_or_dropout_in_z2_ct += 1
        push!(three_8fold_muts_and_no_mut_or_dropout_in_z2, seq)
    end
end; print("\n"^2)
################################################################################
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both = union(three_8fold_muts_and_no_mut_or_dropout_in_z1, three_8fold_muts_and_no_mut_or_dropout_in_z2)
three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct = length(three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both)
###########################################################################################################################################################################
println("Total Number of Seqs with ≥ 4 zones with 8fold muts = $(four_8fold_zone_ct)")
println("Total Number of Seqs with ≥ 4 8fold muts = $(four_8fold_mut_ct)")  #; print("\n"^1)
println("Seqs with ≥4 8fold zones but not dropout or mut in both z1 & z2 = $(four_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in z1 = $(four_8fold_muts_and_no_mut_or_dropout_in_z1_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in z2 = $(four_8fold_muts_and_no_mut_or_dropout_in_z2_ct)")
println("Seqs with ≥4 8fold zones but not dropout or mut in either z1 or z2 or both = $(four_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct)"); print("\n"^1)
###########################################################################################################################################################################
println("Total Number of Seqs with ≥ 3 zones with 8fold muts = $(three_8fold_zone_ct)")
println("Total Number of Seqs with ≥ 3 8fold muts = $(three_8fold_mut_ct)")  #; print("\n"^1)
println("Seqs with ≥3 8fold zones but not dropout or mut in both z1 & z2 = $(three_8fold_muts_and_no_mut_or_dropout_in_both_z1_and_z2_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in z1 = $(three_8fold_muts_and_no_mut_or_dropout_in_z1_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in z2 = $(three_8fold_muts_and_no_mut_or_dropout_in_z2_ct)")
println("Seqs with ≥3 8fold zones but not dropout or mut in either z1 or z2 or both = $(three_8fold_muts_and_no_mut_or_dropout_in_either_z1_or_z2_or_both_ct)"); print("\n"^1)
###########################################################################################################################################################################
###########################################################################################################################################################################

2026_01_07__853PM

############ Seqs lacking z1 but w/muts in 4 other regions ############
EPI_ISL_19511239, EPI_ISL_19613481, EPI_ISL_19299051, EPI_ISL_15218165, EPI_ISL_14062117, EPI_ISL_17265160, EPI_ISL_19434973, EPI_ISL_14509715, EPI_ISL_17113114, EPI_ISL_15958934, EPI_ISL_19627737, 
############ Seqs lacking z2 but w/muts in 4 other regions ############
EPI_ISL_19369713, 
############ Seqs with ≥3 zones with 8fold muts but NEITHER z1 NOR z2 #########
EPI_ISL_19892080
############ Seq lacking z1 but w/muts in 3 other regions ############
EPI_ISL_19511239, EPI_ISL_19613481, EPI_ISL_19299051, EPI_ISL_15218165, EPI_ISL_19560838, EPI_ISL_16868654, EPI_ISL_10230612, EPI_ISL_18398259, EPI_ISL_15040867, EPI_ISL_16244923, EPI_ISL_14062117, EPI_ISL_15743318, EPI_ISL_15797751, EPI_ISL_15874567, EPI_ISL_17265160, EPI_ISL_13483538, EPI_ISL_18053315, EPI_ISL_19434973, EPI_ISL_19946231, EPI_ISL_12168418, EPI_ISL_19761402, EPI_ISL_19892080, EPI_ISL_2096935, EPI_ISL_14509715, EPI_ISL_17113114, EP

In [105]:
### Counting number of seqs w/X pos_only_muts from select list of muts (e.g. 8fold——narrow borders below)
search_moots = Set(["ORF1a:1322", "ORF1a:1323", "ORF1a:1324", "ORF1a:1638", "ORF1a:1639", "ORF1a:2274", "ORF1a:2279", "ORF1a:2280", "ORF1a:3441", "ORF1a:3443", "ORF1a:4087", "ORF1a:4088", "ORF1a:4090", "ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167", "ORF1b:730", "ORF3a:211", "ORF3a:213", "ORF3a:215"])
z1 = ["ORF1a:1322", "ORF1a:1323", "ORF1a:1324"]
z2 = ["ORF1a:1638", "ORF1a:1639"]
z3 = ["ORF1a:2274", "ORF1a:2279", "ORF1a:2280"]
z4 = ["ORF1a:3441", "ORF1a:3443"]
z5 = ["ORF1a:4087", "ORF1a:4088", "ORF1a:4090"]
z6 = ["ORF1a:4164", "ORF1a:4165", "ORF1a:4166", "ORF1a:4167"]
z7 = ["ORF1b:730"]
z8 = ["ORF3a:211", "ORF3a:213", "ORF3a:215"]
z_vec = [z1, z2, z3, z4, z5, z6, z7, z8]
println("######################### 8fold stats NARROW BORDERS version ###########################")
#################################################################################
function pct_of_EPCI(seq_count::Int)
    pct = round(digits=2, 100*seq_count/length(all_unique_chr_seqs_set))
    return pct
end
#################################################################################
zones = [z1, z2, z3, z4, z5, z6, z7, z8]
counts = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
#####################
count_dict = Dict{Int, Int}(i => 0 for i in 1:length(counts))
exact_count_dict = Dict{Int, Int}(i => 0 for i in 1:length(counts))
#####################
count_dict_max_1_per_zone_per_seq = Dict{Int, Int}(i => 0 for i in 0:length(counts))
total_zones_count_dict = Dict{Int, Int}(i => 0 for i in 0:length(counts))
#####################
zones_count_dict = Dict{Int, Int}(i => 0 for i in 1:8)
zones_count_dict_max_1_per_zone_per_seq = Dict{Int, Int}(i => 0 for i in 1:8)
#####################
zone_pattern_dict = Dict{Vector{Int}, Int}()
zone_pattern_dict_max_1_per_zone = Dict{Vector{Int}, Int}()
#####################
zone_effective_dropout_dict = Dict{}
four_8fold_muts_zone_effective_dropout_dict = Dict{}
#####################
min_ct = 2
all_unique_chr_seqs_set_len = length(all_unique_chr_seqs_set)
zone1_not_zone2_seqs = String[]
zone1_and_zone2_seqs = String[]
z1_only_seqs = String[]
z1_only_seqs_ct = 0
#################################################################################
for seq in all_unique_chr_seqs_set
    mutct = 0
    seq_zones_count_dict = Dict{Int, Int}(i => 0 for i in 1:8)
    seq_zones_ct = 0
    seq_zones_vec = Int[]
    seq_zones_max_1_per_zone_ct = 0
    seq_zones_vec_max_1_per_zone = Int[]
    for mut in seq_AA_muts_pos_only_no_dels[seq]
        if mut in search_moots
            mutct += 1
            for i in 1:8
                if mut in zones[i]
                    zones_count_dict[i] += 1
                    seq_zones_count_dict[i] += 1
                    push!(seq_zones_vec, i)
                    if !(i in seq_zones_vec_max_1_per_zone)
                        push!(seq_zones_vec_max_1_per_zone, i)
                        seq_zones_max_1_per_zone_ct += 1
                    end
                end
            end 
        end
    end
    tot_zones = seq_zones_max_1_per_zone_ct
    count_dict_max_1_per_zone_per_seq[tot_zones] += 1
    if tot_zones ≥ 2
        if 1 in seq_zones_vec_max_1_per_zone && !(2 in seq_zones_vec_max_1_per_zone)
            push!(zone1_not_zone2_seqs, seq)
        end
        if 1 in seq_zones_vec_max_1_per_zone && 2 in seq_zones_vec_max_1_per_zone
            push!(zone1_and_zone2_seqs, seq)
        end
    end
###########################################################
    
###########################################################
    for i in 1:8
        if seq_zones_count_dict[i] ≥ 1
            zones_count_dict_max_1_per_zone_per_seq[i] += 1
        end
    end
    for i in 1:length(counts)
        min_ct = i
        if mutct ≥ min_ct
            count_dict[i] += 1
        end
        if mutct == i
            exact_count_dict[i] += 1
        end
#            txtout = "temp_$(seq).txt";   println(seq)
#            print_all_seq_info(seq, txtout)
    end
    if mutct ≥ 7
        println("$(seq), $(seq_collection_date[seq]), $(seq_pango[seq]), $(seq_country[seq])")
    end
    if seq_zones_vec_max_1_per_zone == [1]
        push!(z1_only_seqs, seq)
        z1_only_seqs_ct += 1
    end
    seq_zones_vec_sort = sort(seq_zones_vec)
    zone_pattern_dict[seq_zones_vec_sort] = get!(zone_pattern_dict, seq_zones_vec_sort, 0) + 1
    seq_zones_vec_max_1_per_zone_sort = sort(seq_zones_vec_max_1_per_zone)
    zone_pattern_dict_max_1_per_zone[seq_zones_vec_max_1_per_zone_sort] = get!(zone_pattern_dict_max_1_per_zone, seq_zones_vec_max_1_per_zone_sort, 0) + 1
end
##########################################################################################################################################
z2_region = ["ORF1a:1628", "ORF1a:1629", "ORF1a:1630", "ORF1a:1631", "ORF1a:1632", "ORF1a:1633", "ORF1a:1634", "ORF1a:1635", "ORF1a:1636", "ORF1a:1637", "ORF1a:1638", "ORF1a:1639", "ORF1a:1640", "ORF1a:1641", "ORF1a:1642", "ORF1a:1643", "ORF1a:1644", "ORF1a:1645", "ORF1a:1646", "ORF1a:1647", "ORF1a:1648", "ORF1a:1649"]
z1_only_z2_dropout_seqs = String[]
z1_only_z2_dropout_seqs_ct = 0
for seq in z1_only_seqs
    for mutpo in z2_region
        if mutpo in seq_unknown_AA[seq]
            if !(seq in z1_only_z2_dropout_seqs)
                push!(z1_only_z2_dropout_seqs, seq)
                z1_only_z2_dropout_seqs_ct += 1
            end
        end
    end
end
total_z1_only_seqs = length(z1_only_seqs)
total_z1_only_z2_dropout_seqs = length(z1_only_z2_dropout_seqs)
print("\n"^2)
println("Total z1_only_seqs = $(total_z1_only_seqs)")
println("Total z1_only_z2_dropout_seqs = $(total_z1_only_z2_dropout_seqs)")
print("\n"^2)          
######################################################################################################
open("Eightfold_pos_only_cts_zoneCts_zonePatternCts_ NARROW_BORDERS_$(date_now).txt", "w") do g
    println("######################### 8fold stats NARROW BORDERS version ###########################"); println()
    println(g, "######################### 8fold stats NARROW BORDERS version ###########################"); println(g)
    for i in 1:length(counts)
        min_ct = i
        exact_count = exact_count_dict[i]
        exact_count_pct = pct_of_EPCI(exact_count)
################
        println("Total with exactly $(min_ct) muts = $(exact_count)")
        println("       Exactly-$(min_ct) muts pct = $(exact_count_pct)%") #; println()
        println(g, "Total with exactly $(min_ct) muts = $(exact_count)")
        println(g, "       Exactly-$(min_ct) muts pct = $(exact_count_pct)%") #; println(g)
    end
    print("\n"^3)
################################################################
    for i in 1:length(counts)
        min_ct = i
        cumulative_pattern_ct = count_dict[i]
        cumulative_pattern_pct = pct_of_EPCI(cumulative_pattern_ct)
################
        println("Total with ≥$(min_ct) muts in search moots = $(cumulative_pattern_ct)")
        println("            $(min_ct)x-search muts pct = $(cumulative_pattern_pct)%") #; println()
        println(g, "Total with ≥$(min_ct) muts in search moots = $(cumulative_pattern_ct)")
        println(g, "            $(min_ct)x-search muts pct = $(cumulative_pattern_pct)%") #; println(g)
    end
    print("\n"^3)
######################################################################################################
    for i in 1:length(counts)
        min_ct = i
        exact_pattern_ct = count_dict_max_1_per_zone_per_seq[i]
        exact_pattern_pct = pct_of_EPCI(exact_pattern_ct)
################
        println("Total with exactly $(min_ct) zone-muts = $(exact_pattern_ct)")
        println("      Exactly-$(min_ct) muts pct = $(exact_pattern_pct)%") #; println()
        println(g, "Total with exactly $(min_ct) zone-muts = $(exact_pattern_ct)")
        println(g, "      Exactly-$(min_ct) muts pct = $(exact_pattern_pct)%") #; println(g)
    end
    print("\n"^4)
################################################################
    cumulative_zone_count = 0
    for i in 1:length(counts)
        min_ct = i
        exact_pattern_ct = count_dict_max_1_per_zone_per_seq[i]
        cumulative_zone_count += exact_pattern_ct
        cumulative_pattern_pct = pct_of_EPCI(cumulative_zone_count)
################
        println("Total with ≥$(min_ct) zone-muts in search moots = $(cumulative_zone_count)")
        println("            ≥$(min_ct)x-zone muts pct = $(cumulative_pattern_pct)%") #; println()
        println(g, "Total with ≥$(min_ct) zone-muts in search moots = $(cumulative_zone_count)")
        println(g, "            ≥$(min_ct)x-zone muts pct = $(cumulative_pattern_pct)%") #; println(g)
    end
######################################################################################################
    print("\n"^3)
    zone_pattern_dict_sort = sort(collect(zone_pattern_dict), by = x -> x[1])
    zone_pattern_dict_max_1_per_zone_sort = sort(collect(zone_pattern_dict_max_1_per_zone), by = x -> x[1])
    for zone__ct in zone_pattern_dict_sort
        zone = zone__ct[1]
        ct = zone__ct[2]
        zone_join = join(zone, "-")
        println("$(zone_join) = $(ct)")
        println(g, "$(zone_join) = $(ct)")
    end
    print("\n"^3)
    for zone__ct in zone_pattern_dict_max_1_per_zone_sort
        zone = zone__ct[1]
        ct = zone__ct[2]
        zone_join = join(zone, "-")
        println("$(zone_join) = $(ct)")
        println(g, "$(zone_join) = $(ct)")
    end
    print("\n"^3)
    print(g, "\n"^3)
    zone1_not_zone2_seqs_sort = sort(zone1_not_zone2_seqs, by = x-> (length(x), x))
    zone1_and_zone2_seqs_sort = sort(zone1_and_zone2_seqs, by = x-> (length(x), x))
    total_z1_not_z2_with_2_or_more_8fold_muts = length(zone1_not_zone2_seqs_sort)
    total_z1_and_z2 = length(zone1_and_zone2_seqs_sort)
    z1_not_z2_pct = pct_of_EPCI(total_z1_not_z2_with_2_or_more_8fold_muts)
    z1_and_z2_pct = pct_of_EPCI(total_z1_and_z2)
    total_z1_not_z2_including_z1_singlets = total_z1_not_z2_with_2_or_more_8fold_muts + z1_only_seqs_ct
    total_z1_not_z2_including_z1_singlets_EPCI_pct = pct_of_EPCI(total_z1_not_z2_including_z1_singlets)
################################################################
    println("Total EPCI seqs with mut in z1 but NOT z2 = $(total_z1_not_z2_including_z1_singlets); Pct of all EPCI = $(total_z1_not_z2_including_z1_singlets_EPCI_pct)%")
    println("Total EPCI seqs with ≥2 8fold muts & mut in z1 but NOT z2 = $(total_z1_not_z2_with_2_or_more_8fold_muts); Pct of all EPCI = $(z1_not_z2_pct)%")
    println("Total EPCI seqs with ≥2 8fold muts & mut in z1 AND z2 = $(total_z1_and_z2); Pct of all EPCI = $(z1_and_z2_pct)%")
################################################################
    println(g, "Total EPCI seqs with mut in z1 but NOT z2 = $(total_z1_not_z2_including_z1_singlets); Pct of all EPCI = $(total_z1_not_z2_including_z1_singlets_EPCI_pct)%")
    println(g, "Total EPCI seqs with ≥2 8fold muts & mut in z1 but NOT z2 = $(total_z1_not_z2_with_2_or_more_8fold_muts); Pct of all EPCI = $(z1_not_z2_pct)%")
    println(g, "Total EPCI seqs with ≥2 8fold muts & mut in z1 AND z2 = $(total_z1_and_z2); Pct of all EPCI = $(z1_and_z2_pct)%")
################################################################
    zone1_not_zone2_seqs_with_z2_dropout = String[]
    for seq in zone1_not_zone2_seqs_sort
        for mutpo in z2_region
            if mutpo in seq_unknown_AA[seq]
                if !(seq in zone1_not_zone2_seqs_with_z2_dropout)
                    push!(zone1_not_zone2_seqs_with_z2_dropout, seq)
                end
            end
        end
    end
    z2_dropouts = length(zone1_not_zone2_seqs_with_z2_dropout)
    final_total_z1_not_z2_with_2_or_more_8fold_muts = total_z1_not_z2_with_2_or_more_8fold_muts - z2_dropouts
    final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts = round(digits=2, 100*(total_z1_and_z2/(total_z1_and_z2 + final_total_z1_not_z2_with_2_or_more_8fold_muts)))
    println("#######################################################################################################################")
    println("Final adjusted pct of 8fold seqs (≥2 8fold muts) with z1 that also have z2 muts = $(final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts)")
    println(g, "#######################################################################################################################")
    println(g, "Final adjusted pct of 8fold seqs (≥2 8fold muts) with z1 that also have z2 muts = $(final_z1_and_z2_pct_of_all_z1_seqs_with_2_or_more_8fold_muts)")
    final_adjusted_z1_not_z2_including_z1_singlets_ct = total_z1_not_z2_with_2_or_more_8fold_muts + z1_only_seqs_ct - z2_dropouts - z1_only_z2_dropout_seqs_ct
    final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets = round(digits=2, 100*total_z1_and_z2/(total_z1_and_z2 + final_adjusted_z1_not_z2_including_z1_singlets_ct))
    println("#######################################################################################################################")
    println("Final adjusted number of z1 but not z2 seqs including z1 singlets = $(final_adjusted_z1_not_z2_including_z1_singlets_ct)")
    println("Final adjusted total z1 EPCI seqs = $(final_adjusted_z1_not_z2_including_z1_singlets_ct + total_z1_and_z2)")
    println("Final adjusted pct of EPCI z1-mut seqs that also have a z2-mut = $(final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets)%")
    println("#######################################################################################################################")
    println(g, "#######################################################################################################################")
    println(g, "Final adjusted number of z1 but not z2 seqs including z1 singlets = $(final_adjusted_z1_not_z2_including_z1_singlets_ct)")
    println(g, "Final adjusted total z1 EPCI seqs = $(final_adjusted_z1_not_z2_including_z1_singlets_ct + total_z1_and_z2)")
    println(g, "Final adjusted pct of EPCI z1-mut seqs that also have a z2-mut = $(final_adjusted_z1_and_z2_pct_of_all_z1_EPCI_seqs_including_z1_singlets)%")
    println(g, "#######################################################################################################################")
    for seq in zone1_not_zone2_seqs_with_z2_dropout
        println("Seq with z1 & not z2 mut with z2 dropout = $(seq)")
        println(g, "Seq with z1 & not z2 mut with z2 dropout = $(seq)")
    end
    print("\n"^2); print(g, "\n"^2)
end
###############################################################################################################################################
###############################################################################################################################################

EPI_ISL_16585286, 2022-12-28, BA.2, Netherlands
EPI_ISL_15845778, 2022-10-26, BA.2.12.1, USA
EPI_ISL_19561072, 2024-11-10, BA.1.1, Israel
EPI_ISL_17997917, 2023-05-22, BA.5.11, USA
EPI_ISL_17257608, 2023-03-09, BA.1.1, Italy
EPI_ISL_17782148, 2023-05-24, BA.2.12.1, USA
EPI_ISL_18359328, 2023-08-23, BA.2, Spain
EPI_ISL_17780726, 2023-05-26, AY.122, Italy
EPI_ISL_20019468, 2023-02-23, BA.4.4, Canada
EPI_ISL_18470400, 2023-10-25, EG.6.1, Canada
EPI_ISL_16722270, 2023-01-13, BA.1, England
EPI_ISL_20227249, 2025-09-15, XCH.1, Luxembourg


Total z1_only_seqs = 36
Total z1_only_z2_dropout_seqs = 1


Total with exactly 1 muts = 327
       Exactly-1 muts pct = 9.96%
Total with exactly 2 muts = 249
       Exactly-2 muts pct = 7.58%
Total with exactly 3 muts = 217
       Exactly-3 muts pct = 6.61%
Total with exactly 4 muts = 127
       Exactly-4 muts pct = 3.87%
Total with exactly 5 muts = 54
       Exactly-5 muts pct = 1.64%
Total with exactly 6 muts = 24
       Exactly-6 muts pct = 0.73%
Total 

In [119]:
### Checking all muts in posited major A*01:01 HLA epitope to see how often they show up in 8fold sequences
fold8_moots = Set(["ORF1a:R1318G", "ORF1a:T1322R", "ORF1a:T1322A", "ORF1a:T1322V", "ORF1a:T1322P", "ORF1a:T1322I", "ORF1a:T1322L", "ORF1a:T1322K", "ORF1a:D1323N", "ORF1a:D1323G", "ORF1a:D1323A", "ORF1a:D1323E", "ORF1a:D1323Y", "ORF1a:N1324I", "ORF1a:N1324Y", "ORF1a:N1324K", "ORF1a:N1324D", "ORF1a:N1324H", "ORF1a:N1324T", "ORF1a:N1324E", "ORF1a:N1324S", "ORF1a:N1324G", "ORF1a:I1326L", "ORF1a:I1326V", "ORF1a:I1326T", "ORF1a:T1328N", "ORF1a:T1638I", "ORF1a:T1638A", "ORF1a:T1638L", "ORF1a:T1638V", "ORF1a:T1638Y", "ORF1a:T1638N", "ORF1a:T1638P", "ORF1a:D1639E", "ORF1a:D1639H", "ORF1a:D1639N", "ORF1a:D1639A", "ORF1a:P1640H", "ORF1a:P1640F", "ORF1a:L1640F", "ORF1a:S1640F", "ORF1a:T2274I", "ORF1a:V2276A", "ORF1a:T2277I", "ORF1a:T2280I", "ORF1a:L3440F", "ORF1a:E3441D", "ORF1a:E3441N", "ORF1a:N3443K", "ORF1a:N3443V", "ORF1a:N3443Y", "ORF1a:N3443S", "ORF1a:N3443D", "ORF1a:D4085A", "ORF1a:D4085N", "ORF1a:D4085G", "ORF1a:D4085E", "ORF1a:G4086C", "ORF1a:G4086V", "ORF1a:G4086S", "ORF1a:T4087M", "ORF1a:T4087A", "ORF1a:T4087V", "ORF1a:T4087I", "ORF1a:T4087R", "ORF1a:T4087K", "ORF1a:T4088I", "ORF1a:T4090S", "ORF1a:T4090I", "ORF1a:T4090N", "ORF1a:T4090A", "ORF1a:A4162T", "ORF1a:A4162S", "ORF1a:T4164S", "ORF1a:T4164I", "ORF1a:T4164P", "ORF1a:T4164N", "ORF1a:D4165G", "ORF1a:D4165Y", "ORF1a:D4165T", "ORF1a:D4165V", "ORF1a:D4165A", "ORF1a:D4165N", "ORF1a:D4166E", "ORF1a:D4166G", "ORF1a:D4166S", "ORF1a:D4166A", "ORF1a:D4166Y", "ORF1a:D4166C", "ORF1a:D4166N", "ORF1a:N4167T", "ORF1a:N4167S", "ORF1a:N4167K", "ORF1a:A4168T", "ORF1a:L4169S", "ORF1a:Y4172H", "ORF1b:D729E", "ORF1b:T730I", "ORF1b:D731E", "ORF1b:N734S", "ORF3a:D210N", "ORF3a:D210Y", "ORF3a:D210G", "ORF3a:D210E", "ORF3a:D210A", "ORF3a:Y211H", "ORF3a:Y211C", "ORF3a:Q213K", "ORF3a:Q213L", "ORF3a:Q213R", "ORF3a:Y215C", "ORF3a:Y215H"])
mut_chk_vec = ["N:S78G", "N:S79N", "N:S79G", "N:S79T", "N:P80S", "N:P80R", "N:P80L", "N:P80Q", "N:D81H", "N:D81N", "N:D81G", "N:D81Y", "N:Q83R", "N:Q83N", "N:Q83E", "N:I84V", "N:Y87F"]
for mut in mut_chk_vec
    total_mut_seq_ct = 0
    qualified_seq_ct = 0
    epi_vec = String[]
    for seq in AA_muts_seq[mut]
        mutct = 0
        if seq in all_unique_chr_seqs_set
            total_mut_seq_ct += 1
            for mut in seq_AA_muts[seq]
                if mut in fold8_moots
                    mutct += 1
                end
            end
#            println("$(seq) = $(mutct)")
            if mutct ≥ 2
#                println(seq)
                seq_num = "$(seq) [x$(mutct)]"
                push!(epi_vec, seq_num)
                qualified_seq_ct += 1
            end
    #        if mutct ≥ 3
    #            txtout = "temp_$(seq).txt"
    #            print_all_seq_info(seq, txtout)
    #        end
        end
    #    total_mut_seqs = length(AA_muts_seq[mut])
    end
    qualified_pct = round(digits=1, 100*qualified_seq_ct/total_mut_seq_ct)
    seq_qual_pad = rpad("$(qualified_seq_ct)/$(total_mut_seq_ct)", 5)
    pct_pad = lpad("$(qualified_pct)%", 6)
    print("$(mut) = $(seq_qual_pad) | $(pct_pad)   ")
    epi_vec_str = join(epi_vec, ", ")
    print(epi_vec_str)
    println()
end



N:S78G = 2/2   | 100.0%   EPI_ISL_18590820 [x5], EPI_ISL_19561072 [x7]
N:S79N = 0/1   |   0.0%   
N:S79G = 2/2   | 100.0%   EPI_ISL_17592236 [x3], EPI_ISL_16818471 [x3]
N:S79T = 0/1   |   0.0%   
N:P80S = 1/5   |  20.0%   EPI_ISL_17813049 [x4]
N:P80R = 0/6   |   0.0%   
N:P80L = 3/10  |  30.0%   EPI_ISL_18398259 [x5], EPI_ISL_15476724 [x2], EPI_ISL_15761543 [x2]
N:P80Q = 1/2   |  50.0%   EPI_ISL_18580750 [x5]
N:D81H = 4/6   |  66.7%   EPI_ISL_17997917 [x9], EPI_ISL_17389779 [x5], EPI_ISL_11437359 [x7], EPI_ISL_17813049 [x4]
N:D81N = 1/1   | 100.0%   EPI_ISL_17780726 [x7]
N:D81G = 1/1   | 100.0%   EPI_ISL_18398259 [x5]
N:D81Y = 1/1   | 100.0%   EPI_ISL_19555115 [x2]
N:Q83R = 1/6   |  16.7%   EPI_ISL_14509715 [x5]
N:Q83N = 0/1   |   0.0%   
N:Q83E = 1/1   | 100.0%   EPI_ISL_18053315 [x6]
N:I84V = 1/6   |  16.7%   EPI_ISL_20047831 [x5]
N:Y87F = 1/1   | 100.0%   EPI_ISL_18687895 [x6]


In [11]:
### Finding seqs in EPCI list with given search mutations #################
search_mutations = ["S:L822F", "S:T791I", "S:T791P"]
search_mut_seqs = String[]
for seq in all_unique_chr_seqs
    search_mut_ct = 0
    for mut in search_mutations
        if mut in seq_AA_muts[seq]
            search_mut_ct += 1
        end
    end
    if search_mut_ct ≥ 2
        push!(search_mut_seqs, seq)
    end
end
println()
for seq in search_mut_seqs
    print("$(seq), ")
end




EPI_ISL_18094476, EPI_ISL_17988396, EPI_ISL_19804491, EPI_ISL_19977768, EPI_ISL_18139400, EPI_ISL_16766196, 

In [36]:
### Printing Mutations in given range with high Chronic-to-Circulating ratios
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
new_chronic_2025_09_27_new = stringlist_to_set("EPI_ISL_779969, EPI_ISL_1379391, EPI_ISL_1379435, EPI_ISL_1554931, EPI_ISL_1554932, EPI_ISL_1554933, EPI_ISL_1554934, EPI_ISL_1582755, EPI_ISL_2228938, EPI_ISL_2476401, EPI_ISL_2484216, EPI_ISL_2484217, EPI_ISL_2484218, EPI_ISL_2484219, EPI_ISL_2566726, EPI_ISL_2983871, EPI_ISL_5692576, EPI_ISL_5692832, EPI_ISL_5693164, EPI_ISL_7473129, EPI_ISL_7473130, EPI_ISL_7473131, EPI_ISL_7473132, EPI_ISL_7473133, EPI_ISL_7473134, EPI_ISL_7473135, EPI_ISL_7473136, EPI_ISL_7473137, EPI_ISL_7473138, EPI_ISL_7473139, EPI_ISL_7473140, EPI_ISL_7473141, EPI_ISL_7473142, EPI_ISL_7473143, EPI_ISL_7473144, EPI_ISL_14178690, EPI_ISL_14253197, EPI_ISL_15405688, EPI_ISL_16003220, EPI_ISL_16979306, EPI_ISL_17481521, EPI_ISL_17602334, EPI_ISL_17621115, EPI_ISL_17803784, EPI_ISL_17990203, EPI_ISL_18234763, EPI_ISL_18373754, EPI_ISL_18512418, EPI_ISL_18790918, EPI_ISL_18943910, EPI_ISL_19081421, EPI_ISL_19116728, EPI_ISL_19408911, EPI_ISL_19438699, EPI_ISL_19451639, EPI_ISL_19454521, EPI_ISL_19506574, EPI_ISL_19694042, EPI_ISL_19761928, EPI_ISL_19872997, EPI_ISL_19904802, EPI_ISL_20043225, EPI_ISL_20131508, EPI_ISL_20132435, EPI_ISL_20136586, EPI_ISL_20137604, EPI_ISL_20138079, EPI_ISL_20138923, EPI_ISL_20139405, EPI_ISL_20140153, EPI_ISL_20140838, EPI_ISL_20140845, EPI_ISL_20141945, EPI_ISL_20141955, EPI_ISL_20141982, EPI_ISL_20143974, EPI_ISL_20146793, EPI_ISL_20146795, EPI_ISL_20146858, EPI_ISL_20148475, EPI_ISL_20155076, EPI_ISL_20155864, EPI_ISL_20157513, EPI_ISL_20166121, EPI_ISL_20167020, EPI_ISL_20175070, EPI_ISL_20176704, EPI_ISL_20176761, EPI_ISL_20178658, EPI_ISL_20179076, EPI_ISL_20179179, EPI_ISL_20181458, EPI_ISL_20181461, EPI_ISL_20182683, EPI_ISL_20182757, EPI_ISL_20184766, EPI_ISL_20185296, EPI_ISL_20185471, EPI_ISL_20185528")
chronics_2025_10_28_update = stringlist_to_set("EPI_ISL_474824, EPI_ISL_508674, EPI_ISL_510148, EPI_ISL_516999, EPI_ISL_517000, EPI_ISL_528438, EPI_ISL_529236, EPI_ISL_534720, EPI_ISL_539541, EPI_ISL_539542, EPI_ISL_540582, EPI_ISL_593478, EPI_ISL_593479, EPI_ISL_593480, EPI_ISL_593553, EPI_ISL_593554, EPI_ISL_593555, EPI_ISL_593556, EPI_ISL_593557, EPI_ISL_593558, EPI_ISL_596228, EPI_ISL_602912, EPI_ISL_654166, EPI_ISL_654170, EPI_ISL_654172, EPI_ISL_654173, EPI_ISL_654182, EPI_ISL_654186, EPI_ISL_654191, EPI_ISL_654194, EPI_ISL_678289, EPI_ISL_686537, EPI_ISL_732971, EPI_ISL_753676, EPI_ISL_776770, EPI_ISL_779969, EPI_ISL_801876, EPI_ISL_812862, EPI_ISL_831496, EPI_ISL_872778, EPI_ISL_941340, EPI_ISL_949208, EPI_ISL_959309, EPI_ISL_979349, EPI_ISL_996554, EPI_ISL_1014810, EPI_ISL_1039159, EPI_ISL_1039839, EPI_ISL_1048102, EPI_ISL_1048104, EPI_ISL_1048105, EPI_ISL_1059094, EPI_ISL_1090851, EPI_ISL_1103757, EPI_ISL_1104882, EPI_ISL_1105146, EPI_ISL_1105179, EPI_ISL_1105235, EPI_ISL_1158169, EPI_ISL_1200867, EPI_ISL_1200912, EPI_ISL_1209365, EPI_ISL_1225620, EPI_ISL_1241756, EPI_ISL_1248458, EPI_ISL_1248485, EPI_ISL_1248497, EPI_ISL_1257978, EPI_ISL_1261009, EPI_ISL_1295569, EPI_ISL_1295575, EPI_ISL_1308671, EPI_ISL_1309105, EPI_ISL_1366562, EPI_ISL_1366563, EPI_ISL_1366564, EPI_ISL_1366565, EPI_ISL_1366566, EPI_ISL_1366567, EPI_ISL_1366568, EPI_ISL_1366569, EPI_ISL_1366570, EPI_ISL_1366571, EPI_ISL_1366572, EPI_ISL_1366573, EPI_ISL_1366792, EPI_ISL_1372287, EPI_ISL_1372288, EPI_ISL_1378739, EPI_ISL_1379391, EPI_ISL_1379435, EPI_ISL_1403404, EPI_ISL_1469973, EPI_ISL_1470396, EPI_ISL_1473700, EPI_ISL_1477334, EPI_ISL_1483302, EPI_ISL_1490655, EPI_ISL_1490669, EPI_ISL_1495749, EPI_ISL_1517099, EPI_ISL_1522107, EPI_ISL_1534324, EPI_ISL_1547461, EPI_ISL_1554931, EPI_ISL_1554932, EPI_ISL_1554933, EPI_ISL_1554934, EPI_ISL_1575358, EPI_ISL_1578495, EPI_ISL_1582755, EPI_ISL_1593324, EPI_ISL_1626185, EPI_ISL_1637040, EPI_ISL_1668821, EPI_ISL_1668822, EPI_ISL_1668823, EPI_ISL_1668824, EPI_ISL_1668825, EPI_ISL_1670378, EPI_ISL_1671330, EPI_ISL_1675190, EPI_ISL_1675203, EPI_ISL_1678377, EPI_ISL_1738308, EPI_ISL_1743263, EPI_ISL_1744401, EPI_ISL_1756179, EPI_ISL_1756180, EPI_ISL_1792929, EPI_ISL_1829054, EPI_ISL_1829108, EPI_ISL_1840893, EPI_ISL_1853568, EPI_ISL_1855854, EPI_ISL_1904901, EPI_ISL_1904903, EPI_ISL_1908476, EPI_ISL_1909055, EPI_ISL_1935116, EPI_ISL_1935282, EPI_ISL_1941336, EPI_ISL_1941816, EPI_ISL_1965009, EPI_ISL_1965714, EPI_ISL_1965722, EPI_ISL_2001260, EPI_ISL_2001292, EPI_ISL_2030332, EPI_ISL_2035047, EPI_ISL_2036230, EPI_ISL_2080876, EPI_ISL_2096935, EPI_ISL_2098974, EPI_ISL_2140680, EPI_ISL_2142447, EPI_ISL_2159603, EPI_ISL_2170618, EPI_ISL_2179080, EPI_ISL_2179597, EPI_ISL_2179598, EPI_ISL_2179600, EPI_ISL_2179601, EPI_ISL_2179635, EPI_ISL_2193387, EPI_ISL_2193781, EPI_ISL_2193790, EPI_ISL_2228938, EPI_ISL_2229473, EPI_ISL_2232987, EPI_ISL_2232988, EPI_ISL_2245655, EPI_ISL_2246946, EPI_ISL_2272316, EPI_ISL_2281463, EPI_ISL_2285732, EPI_ISL_2289324, EPI_ISL_2331631, EPI_ISL_2335139, EPI_ISL_2355027, EPI_ISL_2373667, EPI_ISL_2373676, EPI_ISL_2373689, EPI_ISL_2373915, EPI_ISL_2373976, EPI_ISL_2376734, EPI_ISL_2385134, EPI_ISL_2397307, EPI_ISL_2397308, EPI_ISL_2397309, EPI_ISL_2397310, EPI_ISL_2397311, EPI_ISL_2397312, EPI_ISL_2397313, EPI_ISL_2403056, EPI_ISL_2408213, EPI_ISL_2443102, EPI_ISL_2443306, EPI_ISL_2451852, EPI_ISL_2453771, EPI_ISL_2456706, EPI_ISL_2466638, EPI_ISL_2476401, EPI_ISL_2482552, EPI_ISL_2482891, EPI_ISL_2484152, EPI_ISL_2484216, EPI_ISL_2484217, EPI_ISL_2484218, EPI_ISL_2484219, EPI_ISL_2485156, EPI_ISL_2492266, EPI_ISL_2501697, EPI_ISL_2504017, EPI_ISL_2510252, EPI_ISL_2526835, EPI_ISL_2536904, EPI_ISL_2537393, EPI_ISL_2545260, EPI_ISL_2566726, EPI_ISL_2567482, EPI_ISL_2567516, EPI_ISL_2598472, EPI_ISL_2621566, EPI_ISL_2622092, EPI_ISL_2626505, EPI_ISL_2629070, EPI_ISL_2629071, EPI_ISL_2629072, EPI_ISL_2629073, EPI_ISL_2629074, EPI_ISL_2629075, EPI_ISL_2646107, EPI_ISL_2652487, EPI_ISL_2658958, EPI_ISL_2658962, EPI_ISL_2658963, EPI_ISL_2658970, EPI_ISL_2658971, EPI_ISL_2658972, EPI_ISL_2674459, EPI_ISL_2681259, EPI_ISL_2686814, EPI_ISL_2686837, EPI_ISL_2696945, EPI_ISL_2709225, EPI_ISL_2713004, EPI_ISL_2716246, EPI_ISL_2731960, EPI_ISL_2732319, EPI_ISL_2733379, EPI_ISL_2733797, EPI_ISL_2741595, EPI_ISL_2749506, EPI_ISL_2749718, EPI_ISL_2758178, EPI_ISL_2758179, EPI_ISL_2768315, EPI_ISL_2776212, EPI_ISL_2788712, EPI_ISL_2790083, EPI_ISL_2791260, EPI_ISL_2810326, EPI_ISL_2811857, EPI_ISL_2817504, EPI_ISL_2820526, EPI_ISL_2828407, EPI_ISL_2833904, EPI_ISL_2858161, EPI_ISL_2858877, EPI_ISL_2860316, EPI_ISL_2868572, EPI_ISL_2868616, EPI_ISL_2876377, EPI_ISL_2894033, EPI_ISL_2903438, EPI_ISL_2928168, EPI_ISL_2931876, EPI_ISL_2931884, EPI_ISL_2931896, EPI_ISL_2931903, EPI_ISL_2955288, EPI_ISL_2955320, EPI_ISL_2966985, EPI_ISL_2978243, EPI_ISL_2978352, EPI_ISL_2983871, EPI_ISL_2984725, EPI_ISL_2990101, EPI_ISL_2993722, EPI_ISL_3010321, EPI_ISL_3029841, EPI_ISL_3030114, EPI_ISL_3030118, EPI_ISL_3030145, EPI_ISL_3030738, EPI_ISL_3032627, EPI_ISL_3033635, EPI_ISL_3039352, EPI_ISL_3045789, EPI_ISL_3045809, EPI_ISL_3053903, EPI_ISL_3061061, EPI_ISL_3063770, EPI_ISL_3064314, EPI_ISL_3123372, EPI_ISL_3129808, EPI_ISL_3130077, EPI_ISL_3130081, EPI_ISL_3130177, EPI_ISL_3130302, EPI_ISL_3132631, EPI_ISL_3133023, EPI_ISL_3152200, EPI_ISL_3153240, EPI_ISL_3164424, EPI_ISL_3185346, EPI_ISL_3208304, EPI_ISL_3209041, EPI_ISL_3212959, EPI_ISL_3215721, EPI_ISL_3215722, EPI_ISL_3215726, EPI_ISL_3246237, EPI_ISL_3251444, EPI_ISL_3251605, EPI_ISL_3259560, EPI_ISL_3275376, EPI_ISL_3339536, EPI_ISL_3356734, EPI_ISL_3358574, EPI_ISL_3370176, EPI_ISL_3394321, EPI_ISL_3396491, EPI_ISL_3414767, EPI_ISL_3414889, EPI_ISL_3415104, EPI_ISL_3415226, EPI_ISL_3426474, EPI_ISL_3446827, EPI_ISL_3447712, EPI_ISL_3453279, EPI_ISL_3457423, EPI_ISL_3459118, EPI_ISL_3460897, EPI_ISL_3471360, EPI_ISL_3475993, EPI_ISL_3497667, EPI_ISL_3503811, EPI_ISL_3504036, EPI_ISL_3509013, EPI_ISL_3556945, EPI_ISL_3634003, EPI_ISL_3634004, EPI_ISL_3635506, EPI_ISL_3657112, EPI_ISL_3666069, EPI_ISL_3712919, EPI_ISL_3771882, EPI_ISL_3779849, EPI_ISL_3813731, EPI_ISL_3838306, EPI_ISL_3891136, EPI_ISL_3933252, EPI_ISL_3937027, EPI_ISL_3957778, EPI_ISL_3958461, EPI_ISL_3958994, EPI_ISL_3982251, EPI_ISL_4029567, EPI_ISL_4051642, EPI_ISL_4052911, EPI_ISL_4072038, EPI_ISL_4096626, EPI_ISL_4096639, EPI_ISL_4114033, EPI_ISL_4124532, EPI_ISL_4178790, EPI_ISL_4193135, EPI_ISL_4198270, EPI_ISL_4203869, EPI_ISL_4251611, EPI_ISL_4261403, EPI_ISL_4261411, EPI_ISL_4281194, EPI_ISL_4284228, EPI_ISL_4295678, EPI_ISL_4298277, EPI_ISL_4298278, EPI_ISL_4298279, EPI_ISL_4309817, EPI_ISL_4415808, EPI_ISL_4440075, EPI_ISL_4515444, EPI_ISL_4525691, EPI_ISL_4525698, EPI_ISL_4525700, EPI_ISL_4531313, EPI_ISL_4536418, EPI_ISL_4576991, EPI_ISL_4577474, EPI_ISL_4625101, EPI_ISL_4652284, EPI_ISL_4769360, EPI_ISL_4769386, EPI_ISL_4775547, EPI_ISL_4875939, EPI_ISL_4930863, EPI_ISL_4935777, EPI_ISL_4935949, EPI_ISL_4936095, EPI_ISL_4936533, EPI_ISL_4949584, EPI_ISL_5018695, EPI_ISL_5033183, EPI_ISL_5056434, EPI_ISL_5059980, EPI_ISL_5099310, EPI_ISL_5132437, EPI_ISL_5132595, EPI_ISL_5145656, EPI_ISL_5196003, EPI_ISL_5263400, EPI_ISL_5265214, EPI_ISL_5280146, EPI_ISL_5307398, EPI_ISL_5323016, EPI_ISL_5332877, EPI_ISL_5332878, EPI_ISL_5395558, EPI_ISL_5446154, EPI_ISL_5463914, EPI_ISL_5532714, EPI_ISL_5592605, EPI_ISL_5594441, EPI_ISL_5620309, EPI_ISL_5621224, EPI_ISL_5627313, EPI_ISL_5628248, EPI_ISL_5639913, EPI_ISL_5640459, EPI_ISL_5649323, EPI_ISL_5650474, EPI_ISL_5655562, EPI_ISL_5680241, EPI_ISL_5692576, EPI_ISL_5692774, EPI_ISL_5692832, EPI_ISL_5693164, EPI_ISL_5749185, EPI_ISL_5780324, EPI_ISL_5814411, EPI_ISL_5865553, EPI_ISL_5892132, EPI_ISL_5922347, EPI_ISL_5935407, EPI_ISL_5944665, EPI_ISL_5944669, EPI_ISL_5944842, EPI_ISL_5944948, EPI_ISL_5946914, EPI_ISL_6017746, EPI_ISL_6017747, EPI_ISL_6076460, EPI_ISL_6113246, EPI_ISL_6208674, EPI_ISL_6208675, EPI_ISL_6208676, EPI_ISL_6227177, EPI_ISL_6227208, EPI_ISL_6262165, EPI_ISL_6262536, EPI_ISL_6281381, EPI_ISL_6324366, EPI_ISL_6327943, EPI_ISL_6381841, EPI_ISL_6384755, EPI_ISL_6397259, EPI_ISL_6574278, EPI_ISL_6584511, EPI_ISL_6604686, EPI_ISL_6605003, EPI_ISL_6605659, EPI_ISL_6628662, EPI_ISL_6642599, EPI_ISL_6660906, EPI_ISL_6666037, EPI_ISL_6698637, EPI_ISL_6710721, EPI_ISL_6735468, EPI_ISL_6737833, EPI_ISL_6757093, EPI_ISL_6783610, EPI_ISL_6810267, EPI_ISL_6814822, EPI_ISL_6826536, EPI_ISL_6842893, EPI_ISL_6863316, EPI_ISL_6863457, EPI_ISL_6865741, EPI_ISL_6930836, EPI_ISL_6938691, EPI_ISL_6976497, EPI_ISL_6977941, EPI_ISL_7000663, EPI_ISL_7015624, EPI_ISL_7015625, EPI_ISL_7135374, EPI_ISL_7159687, EPI_ISL_7175748, EPI_ISL_7204318, EPI_ISL_7205152, EPI_ISL_7334013, EPI_ISL_7361483, EPI_ISL_7361527, EPI_ISL_7452581, EPI_ISL_7452603, EPI_ISL_7456462, EPI_ISL_7458003, EPI_ISL_7473129, EPI_ISL_7473130, EPI_ISL_7473131, EPI_ISL_7473132, EPI_ISL_7473133, EPI_ISL_7473134, EPI_ISL_7473135, EPI_ISL_7473136, EPI_ISL_7473137, EPI_ISL_7473138, EPI_ISL_7473139, EPI_ISL_7473140, EPI_ISL_7473141, EPI_ISL_7473142, EPI_ISL_7473143, EPI_ISL_7473144, EPI_ISL_7502990, EPI_ISL_7503221, EPI_ISL_7507393, EPI_ISL_7592661, EPI_ISL_7592687, EPI_ISL_7652115, EPI_ISL_7660179, EPI_ISL_7707631, EPI_ISL_7711813, EPI_ISL_7729239, EPI_ISL_7806535, EPI_ISL_7806536, EPI_ISL_7806549, EPI_ISL_7813798, EPI_ISL_7813896, EPI_ISL_7861580, EPI_ISL_7876524, EPI_ISL_7908114, EPI_ISL_7961502, EPI_ISL_7976931, EPI_ISL_7980711, EPI_ISL_8001538, EPI_ISL_8035582, EPI_ISL_8057418, EPI_ISL_8131224, EPI_ISL_8132253, EPI_ISL_8151798, EPI_ISL_8153087, EPI_ISL_8166542, EPI_ISL_8189765, EPI_ISL_8189775, EPI_ISL_8204828, EPI_ISL_8205040, EPI_ISL_8207600, EPI_ISL_8215783, EPI_ISL_8215787, EPI_ISL_8251200, EPI_ISL_8263463, EPI_ISL_8350537, EPI_ISL_8376888, EPI_ISL_8479639, EPI_ISL_8479640, EPI_ISL_8563217, EPI_ISL_8563218, EPI_ISL_8563219, EPI_ISL_8615077, EPI_ISL_8627379, EPI_ISL_8669281, EPI_ISL_8675368, EPI_ISL_8712661, EPI_ISL_8725398, EPI_ISL_8725399, EPI_ISL_8725400, EPI_ISL_8725401, EPI_ISL_8725402, EPI_ISL_8725403, EPI_ISL_8725404, EPI_ISL_8725405, EPI_ISL_8725406, EPI_ISL_8725407, EPI_ISL_8725408, EPI_ISL_8725409, EPI_ISL_8732699, EPI_ISL_8732807, EPI_ISL_8732841, EPI_ISL_8750545, EPI_ISL_8754305, EPI_ISL_8766992, EPI_ISL_8800409, EPI_ISL_8806077, EPI_ISL_8806082, EPI_ISL_8806084, EPI_ISL_8819629, EPI_ISL_8825833, EPI_ISL_8828662, EPI_ISL_8887845, EPI_ISL_8887874, EPI_ISL_8889632, EPI_ISL_9021214, EPI_ISL_9141923, EPI_ISL_9155607, EPI_ISL_9201951, EPI_ISL_9242265, EPI_ISL_9242269, EPI_ISL_9319180, EPI_ISL_9570633, EPI_ISL_9630717, EPI_ISL_9637481, EPI_ISL_9637840, EPI_ISL_9702285, EPI_ISL_9735644, EPI_ISL_9844246, EPI_ISL_9869512, EPI_ISL_9873278, EPI_ISL_9907655, EPI_ISL_9949797, EPI_ISL_9949799, EPI_ISL_10124646, EPI_ISL_10127751, EPI_ISL_10128185, EPI_ISL_10128193, EPI_ISL_10166426, EPI_ISL_10185453, EPI_ISL_10195257, EPI_ISL_10195262, EPI_ISL_10195263, EPI_ISL_10195264, EPI_ISL_10195305, EPI_ISL_10230612, EPI_ISL_10239201, EPI_ISL_10251304, EPI_ISL_10329391, EPI_ISL_10329558, EPI_ISL_10352747, EPI_ISL_10381820, EPI_ISL_10397517, EPI_ISL_10451205, EPI_ISL_10451252, EPI_ISL_10548912, EPI_ISL_10548913, EPI_ISL_10548915, EPI_ISL_10548916, EPI_ISL_10548917, EPI_ISL_10548918, EPI_ISL_10548919, EPI_ISL_10548920, EPI_ISL_10548921, EPI_ISL_10548922, EPI_ISL_10549162, EPI_ISL_10549163, EPI_ISL_10549164, EPI_ISL_10549165, EPI_ISL_10549166, EPI_ISL_10590270, EPI_ISL_10590760, EPI_ISL_10591808, EPI_ISL_10657890, EPI_ISL_10681118, EPI_ISL_10704797, EPI_ISL_10706292, EPI_ISL_10712909, EPI_ISL_10717525, EPI_ISL_10787994, EPI_ISL_10815044, EPI_ISL_10816730, EPI_ISL_10816731, EPI_ISL_10816732, EPI_ISL_10816733, EPI_ISL_10816734, EPI_ISL_10816735, EPI_ISL_10816736, EPI_ISL_10816737, EPI_ISL_10816738, EPI_ISL_10816739, EPI_ISL_10816741, EPI_ISL_10816742, EPI_ISL_10816743, EPI_ISL_10816744, EPI_ISL_10824028, EPI_ISL_10876034, EPI_ISL_10876749, EPI_ISL_10899907, EPI_ISL_10942195, EPI_ISL_10942196, EPI_ISL_10981395, EPI_ISL_10995323, EPI_ISL_10998528, EPI_ISL_11025821, EPI_ISL_11025897, EPI_ISL_11030507, EPI_ISL_11036385, EPI_ISL_11036386, EPI_ISL_11036389, EPI_ISL_11036399, EPI_ISL_11036451, EPI_ISL_11036666, EPI_ISL_11036688, EPI_ISL_11036712, EPI_ISL_11036915, EPI_ISL_11036917, EPI_ISL_11050902, EPI_ISL_11055609, EPI_ISL_11106543, EPI_ISL_11110730, EPI_ISL_11173072, EPI_ISL_11219235, EPI_ISL_11219236, EPI_ISL_11221773, EPI_ISL_11221782, EPI_ISL_11222620, EPI_ISL_11229672, EPI_ISL_11230305, EPI_ISL_11239958, EPI_ISL_11242266, EPI_ISL_11248919, EPI_ISL_11256669, EPI_ISL_11290054, EPI_ISL_11295642, EPI_ISL_11296415, EPI_ISL_11349763, EPI_ISL_11356267, EPI_ISL_11356269, EPI_ISL_11403393, EPI_ISL_11424263, EPI_ISL_11424578, EPI_ISL_11437359, EPI_ISL_11482304, EPI_ISL_11503909, EPI_ISL_11504189, EPI_ISL_11517385, EPI_ISL_11565840, EPI_ISL_11576757, EPI_ISL_11621597, EPI_ISL_11621883, EPI_ISL_11657715, EPI_ISL_11661806, EPI_ISL_11695384, EPI_ISL_11742572, EPI_ISL_11742812, EPI_ISL_11747289, EPI_ISL_11787443, EPI_ISL_11789090, EPI_ISL_11798407, EPI_ISL_11798458, EPI_ISL_11816904, EPI_ISL_11825798, EPI_ISL_11826326, EPI_ISL_11826898, EPI_ISL_11871462, EPI_ISL_11886470, EPI_ISL_11886479, EPI_ISL_11886624, EPI_ISL_11897546, EPI_ISL_11898000, EPI_ISL_11919916, EPI_ISL_11961223, EPI_ISL_11968830, EPI_ISL_11968876, EPI_ISL_11970393, EPI_ISL_11976211, EPI_ISL_11992954, EPI_ISL_11995938, EPI_ISL_12001179, EPI_ISL_12001180, EPI_ISL_12006455, EPI_ISL_12021469, EPI_ISL_12039060, EPI_ISL_12060087, EPI_ISL_12063598, EPI_ISL_12063600, EPI_ISL_12063601, EPI_ISL_12063602, EPI_ISL_12079999, EPI_ISL_12083619, EPI_ISL_12089943, EPI_ISL_12097355, EPI_ISL_12108965, EPI_ISL_12127282, EPI_ISL_12139045, EPI_ISL_12139066, EPI_ISL_12145506, EPI_ISL_12146396, EPI_ISL_12146579, EPI_ISL_12146859, EPI_ISL_12146865, EPI_ISL_12146872, EPI_ISL_12147521, EPI_ISL_12148419, EPI_ISL_12148503, EPI_ISL_12150077, EPI_ISL_12150259, EPI_ISL_12150361, EPI_ISL_12150484, EPI_ISL_12155759, EPI_ISL_12155809, EPI_ISL_12157165, EPI_ISL_12157166, EPI_ISL_12157187, EPI_ISL_12158960, EPI_ISL_12162547, EPI_ISL_12168418, EPI_ISL_12171333, EPI_ISL_12171674, EPI_ISL_12172132, EPI_ISL_12172842, EPI_ISL_12173486, EPI_ISL_12173730, EPI_ISL_12173879, EPI_ISL_12174612, EPI_ISL_12174734, EPI_ISL_12174735, EPI_ISL_12174736, EPI_ISL_12174739, EPI_ISL_12174942, EPI_ISL_12175020, EPI_ISL_12175024, EPI_ISL_12175203, EPI_ISL_12176184, EPI_ISL_12207682, EPI_ISL_12220762, EPI_ISL_12240087, EPI_ISL_12240983, EPI_ISL_12251561, EPI_ISL_12259859, EPI_ISL_12261629, EPI_ISL_12268492, EPI_ISL_12278477, EPI_ISL_12278678, EPI_ISL_12278997, EPI_ISL_12284821, EPI_ISL_12296790, EPI_ISL_12296891, EPI_ISL_12310532, EPI_ISL_12323992, EPI_ISL_12325408, EPI_ISL_12350967, EPI_ISL_12351281, EPI_ISL_12355622, EPI_ISL_12371147, EPI_ISL_12397635, EPI_ISL_12401928, EPI_ISL_12422504, EPI_ISL_12423062, EPI_ISL_12425033, EPI_ISL_12430022, EPI_ISL_12444737, EPI_ISL_12447557, EPI_ISL_12454820, EPI_ISL_12454840, EPI_ISL_12456735, EPI_ISL_12464077, EPI_ISL_12467081, EPI_ISL_12467157, EPI_ISL_12473693, EPI_ISL_12475004, EPI_ISL_12480203, EPI_ISL_12486436, EPI_ISL_12486761, EPI_ISL_12488441, EPI_ISL_12495863, EPI_ISL_12501519, EPI_ISL_12511246, EPI_ISL_12530780, EPI_ISL_12531462, EPI_ISL_12535815, EPI_ISL_12539663, EPI_ISL_12568162, EPI_ISL_12568208, EPI_ISL_12575298, EPI_ISL_12579825, EPI_ISL_12581914, EPI_ISL_12590958, EPI_ISL_12602903, EPI_ISL_12611697, EPI_ISL_12611721, EPI_ISL_12616586, EPI_ISL_12622901, EPI_ISL_12622902, EPI_ISL_12623284, EPI_ISL_12628520, EPI_ISL_12639714, EPI_ISL_12639788, EPI_ISL_12639917, EPI_ISL_12645823, EPI_ISL_12646116, EPI_ISL_12646361, EPI_ISL_12646785, EPI_ISL_12647336, EPI_ISL_12652423, EPI_ISL_12654179, EPI_ISL_12656485, EPI_ISL_12661097, EPI_ISL_12663222, EPI_ISL_12664764, EPI_ISL_12667249, EPI_ISL_12680798, EPI_ISL_12685124, EPI_ISL_12685126, EPI_ISL_12691923, EPI_ISL_12698937, EPI_ISL_12701772, EPI_ISL_12701820, EPI_ISL_12701858, EPI_ISL_12701871, EPI_ISL_12701895, EPI_ISL_12703517, EPI_ISL_12707647, EPI_ISL_12715944, EPI_ISL_12717870, EPI_ISL_12739425, EPI_ISL_12741485, EPI_ISL_12742126, EPI_ISL_12742222, EPI_ISL_12754976, EPI_ISL_12763802, EPI_ISL_12765459, EPI_ISL_12784028, EPI_ISL_12789812, EPI_ISL_12789846, EPI_ISL_12808264, EPI_ISL_12809016, EPI_ISL_12809684, EPI_ISL_12822481, EPI_ISL_12822483, EPI_ISL_12829036, EPI_ISL_12830215, EPI_ISL_12830827, EPI_ISL_12844170, EPI_ISL_12851188, EPI_ISL_12851233, EPI_ISL_12851285, EPI_ISL_12862705, EPI_ISL_12862970, EPI_ISL_12871249, EPI_ISL_12879252, EPI_ISL_12881932, EPI_ISL_12892238, EPI_ISL_12892312, EPI_ISL_12892482, EPI_ISL_12896994, EPI_ISL_12903760, EPI_ISL_12906172, EPI_ISL_12911895, EPI_ISL_12911898, EPI_ISL_12914019, EPI_ISL_12926955, EPI_ISL_12932770, EPI_ISL_12942108, EPI_ISL_12955795, EPI_ISL_12958668, EPI_ISL_12961699, EPI_ISL_12968380, EPI_ISL_12980420, EPI_ISL_12993020, EPI_ISL_12995123, EPI_ISL_12995422, EPI_ISL_12995533, EPI_ISL_13000815, EPI_ISL_13001148, EPI_ISL_13002153, EPI_ISL_13002802, EPI_ISL_13002811, EPI_ISL_13011225, EPI_ISL_13011226, EPI_ISL_13019919, EPI_ISL_13026163, EPI_ISL_13028133, EPI_ISL_13040401, EPI_ISL_13044133, EPI_ISL_13047387, EPI_ISL_13050078, EPI_ISL_13050793, EPI_ISL_13051431, EPI_ISL_13051740, EPI_ISL_13051970, EPI_ISL_13052096, EPI_ISL_13052204, EPI_ISL_13055324, EPI_ISL_13056142, EPI_ISL_13056190, EPI_ISL_13065554, EPI_ISL_13066396, EPI_ISL_13066472, EPI_ISL_13066603, EPI_ISL_13073691, EPI_ISL_13085784, EPI_ISL_13085991, EPI_ISL_13086417, EPI_ISL_13086737, EPI_ISL_13086826, EPI_ISL_13086831, EPI_ISL_13088769, EPI_ISL_13088942, EPI_ISL_13089020, EPI_ISL_13089384, EPI_ISL_13091908, EPI_ISL_13091912, EPI_ISL_13091925, EPI_ISL_13091929, EPI_ISL_13092725, EPI_ISL_13092906, EPI_ISL_13093369, EPI_ISL_13093922, EPI_ISL_13094368, EPI_ISL_13108591, EPI_ISL_13110029, EPI_ISL_13110031, EPI_ISL_13123133, EPI_ISL_13129387, EPI_ISL_13131091, EPI_ISL_13131118, EPI_ISL_13131514, EPI_ISL_13132070, EPI_ISL_13133128, EPI_ISL_13133359, EPI_ISL_13152570, EPI_ISL_13157537, EPI_ISL_13157638, EPI_ISL_13158312, EPI_ISL_13158314, EPI_ISL_13160040, EPI_ISL_13166402, EPI_ISL_13166803, EPI_ISL_13172328, EPI_ISL_13172329, EPI_ISL_13176279, EPI_ISL_13176281, EPI_ISL_13178754, EPI_ISL_13183984, EPI_ISL_13187136, EPI_ISL_13192066, EPI_ISL_13192072, EPI_ISL_13192202, EPI_ISL_13199746, EPI_ISL_13199759, EPI_ISL_13202578, EPI_ISL_13203425, EPI_ISL_13210987, EPI_ISL_13214299, EPI_ISL_13215742, EPI_ISL_13230467, EPI_ISL_13242111, EPI_ISL_13242155, EPI_ISL_13244164, EPI_ISL_13244243, EPI_ISL_13244707, EPI_ISL_13251406, EPI_ISL_13251514, EPI_ISL_13251524, EPI_ISL_13253132, EPI_ISL_13253164, EPI_ISL_13253244, EPI_ISL_13253296, EPI_ISL_13253344, EPI_ISL_13253353, EPI_ISL_13253416, EPI_ISL_13253427, EPI_ISL_13253489, EPI_ISL_13253493, EPI_ISL_13253550, EPI_ISL_13253551, EPI_ISL_13253570, EPI_ISL_13264387, EPI_ISL_13271922, EPI_ISL_13272001, EPI_ISL_13272223, EPI_ISL_13273550, EPI_ISL_13284168, EPI_ISL_13289213, EPI_ISL_13289619, EPI_ISL_13289774, EPI_ISL_13294595, EPI_ISL_13299108, EPI_ISL_13299118, EPI_ISL_13299119, EPI_ISL_13299122, EPI_ISL_13301357, EPI_ISL_13301368, EPI_ISL_13304429, EPI_ISL_13304451, EPI_ISL_13312837, EPI_ISL_13314960, EPI_ISL_13317150, EPI_ISL_13317160, EPI_ISL_13318832, EPI_ISL_13319769, EPI_ISL_13320180, EPI_ISL_13322028, EPI_ISL_13322954, EPI_ISL_13322975, EPI_ISL_13323290, EPI_ISL_13328732, EPI_ISL_13329792, EPI_ISL_13332433, EPI_ISL_13332459, EPI_ISL_13333047, EPI_ISL_13339980, EPI_ISL_13345908, EPI_ISL_13349432, EPI_ISL_13350581, EPI_ISL_13351788, EPI_ISL_13354243, EPI_ISL_13354366, EPI_ISL_13356160, EPI_ISL_13356194, EPI_ISL_13358749, EPI_ISL_13358809, EPI_ISL_13358893, EPI_ISL_13358894, EPI_ISL_13361313, EPI_ISL_13361419, EPI_ISL_13362130, EPI_ISL_13368501, EPI_ISL_13368552, EPI_ISL_13369326, EPI_ISL_13370722, EPI_ISL_13371824, EPI_ISL_13376289, EPI_ISL_13385215, EPI_ISL_13386427, EPI_ISL_13389863, EPI_ISL_13389864, EPI_ISL_13394010, EPI_ISL_13398372, EPI_ISL_13398391, EPI_ISL_13400530, EPI_ISL_13405240, EPI_ISL_13406133, EPI_ISL_13407391, EPI_ISL_13407748, EPI_ISL_13407802, EPI_ISL_13408380, EPI_ISL_13410054, EPI_ISL_13410128, EPI_ISL_13412509, EPI_ISL_13417637, EPI_ISL_13422063, EPI_ISL_13425805, EPI_ISL_13426860, EPI_ISL_13426861, EPI_ISL_13426862, EPI_ISL_13440246, EPI_ISL_13440269, EPI_ISL_13440488, EPI_ISL_13443257, EPI_ISL_13449356, EPI_ISL_13455972, EPI_ISL_13463270, EPI_ISL_13466629, EPI_ISL_13466644, EPI_ISL_13467321, EPI_ISL_13467676, EPI_ISL_13470130, EPI_ISL_13470158, EPI_ISL_13476507, EPI_ISL_13477158, EPI_ISL_13478276, EPI_ISL_13478413, EPI_ISL_13478421, EPI_ISL_13478425, EPI_ISL_13478448, EPI_ISL_13479636, EPI_ISL_13480851, EPI_ISL_13481704, EPI_ISL_13482848, EPI_ISL_13483067, EPI_ISL_13483538, EPI_ISL_13502856, EPI_ISL_13502894, EPI_ISL_13503354, EPI_ISL_13503362, EPI_ISL_13503413, EPI_ISL_13504084, EPI_ISL_13504103, EPI_ISL_13504568, EPI_ISL_13518485, EPI_ISL_13519541, EPI_ISL_13522554, EPI_ISL_13522707, EPI_ISL_13529086, EPI_ISL_13535844, EPI_ISL_13536846, EPI_ISL_13536992, EPI_ISL_13538620, EPI_ISL_13538621, EPI_ISL_13539746, EPI_ISL_13552269, EPI_ISL_13560139, EPI_ISL_13563849, EPI_ISL_13563900, EPI_ISL_13564453, EPI_ISL_13564901, EPI_ISL_13566717, EPI_ISL_13571026, EPI_ISL_13571629, EPI_ISL_13572579, EPI_ISL_13572829, EPI_ISL_13573707, EPI_ISL_13577432, EPI_ISL_13578717, EPI_ISL_13585850, EPI_ISL_13592668, EPI_ISL_13605240, EPI_ISL_13605930, EPI_ISL_13608346, EPI_ISL_13611373, EPI_ISL_13612632, EPI_ISL_13614328, EPI_ISL_13615804, EPI_ISL_13615826, EPI_ISL_13616458, EPI_ISL_13617390, EPI_ISL_13617475, EPI_ISL_13617493, EPI_ISL_13619646, EPI_ISL_13622657, EPI_ISL_13622831, EPI_ISL_13622874, EPI_ISL_13624440, EPI_ISL_13624441, EPI_ISL_13626226, EPI_ISL_13633558, EPI_ISL_13633729, EPI_ISL_13636934, EPI_ISL_13637141, EPI_ISL_13637734, EPI_ISL_13638494, EPI_ISL_13653660, EPI_ISL_13665096, EPI_ISL_13665110, EPI_ISL_13666764, EPI_ISL_13677872, EPI_ISL_13688333, EPI_ISL_13691966, EPI_ISL_13692397, EPI_ISL_13694663, EPI_ISL_13698648, EPI_ISL_13700128, EPI_ISL_13700243, EPI_ISL_13700756, EPI_ISL_13701029, EPI_ISL_13701810, EPI_ISL_13715129, EPI_ISL_13715746, EPI_ISL_13732799, EPI_ISL_13734474, EPI_ISL_13734683, EPI_ISL_13738059, EPI_ISL_13740111, EPI_ISL_13740674, EPI_ISL_13741330, EPI_ISL_13744798, EPI_ISL_13744799, EPI_ISL_13745638, EPI_ISL_13745641, EPI_ISL_13748166, EPI_ISL_13750726, EPI_ISL_13750730, EPI_ISL_13750760, EPI_ISL_13750771, EPI_ISL_13750936, EPI_ISL_13750937, EPI_ISL_13751254, EPI_ISL_13757795, EPI_ISL_13757902, EPI_ISL_13757914, EPI_ISL_13759674, EPI_ISL_13759811, EPI_ISL_13760929, EPI_ISL_13762803, EPI_ISL_13764852, EPI_ISL_13765234, EPI_ISL_13776103, EPI_ISL_13776118, EPI_ISL_13778691, EPI_ISL_13788917, EPI_ISL_13794761, EPI_ISL_13795106, EPI_ISL_13795308, EPI_ISL_13802466, EPI_ISL_13805350, EPI_ISL_13806026, EPI_ISL_13806197, EPI_ISL_13812067, EPI_ISL_13826362, EPI_ISL_13830194, EPI_ISL_13830195, EPI_ISL_13830196, EPI_ISL_13830197, EPI_ISL_13830445, EPI_ISL_13839105, EPI_ISL_13839285, EPI_ISL_13842068, EPI_ISL_13844161, EPI_ISL_13850726, EPI_ISL_13855446, EPI_ISL_13856866, EPI_ISL_13858143, EPI_ISL_13858664, EPI_ISL_13860426, EPI_ISL_13860879, EPI_ISL_13866687, EPI_ISL_13866688, EPI_ISL_13866691, EPI_ISL_13871326, EPI_ISL_13873100, EPI_ISL_13875348, EPI_ISL_13875677, EPI_ISL_13875711, EPI_ISL_13876318, EPI_ISL_13876612, EPI_ISL_13876760, EPI_ISL_13881123, EPI_ISL_13884353, EPI_ISL_13884360, EPI_ISL_13884439, EPI_ISL_13889482, EPI_ISL_13891697, EPI_ISL_13892755, EPI_ISL_13893481, EPI_ISL_13896135, EPI_ISL_13896156, EPI_ISL_13896578, EPI_ISL_13900930, EPI_ISL_13905704, EPI_ISL_13907795, EPI_ISL_13908668, EPI_ISL_13913047, EPI_ISL_13915527, EPI_ISL_13915530, EPI_ISL_13925703, EPI_ISL_13931117, EPI_ISL_13931975, EPI_ISL_13931997, EPI_ISL_13932039, EPI_ISL_13932080, EPI_ISL_13937613, EPI_ISL_13939223, EPI_ISL_13947566, EPI_ISL_13948007, EPI_ISL_13951654, EPI_ISL_13958481, EPI_ISL_13958566, EPI_ISL_13963776, EPI_ISL_13963832, EPI_ISL_13970237, EPI_ISL_13970242, EPI_ISL_13970249, EPI_ISL_13970257, EPI_ISL_13970276, EPI_ISL_13975822, EPI_ISL_13981101, EPI_ISL_13984460, EPI_ISL_13986491, EPI_ISL_13986492, EPI_ISL_13986494, EPI_ISL_13986497, EPI_ISL_13986501, EPI_ISL_13991375, EPI_ISL_13993708, EPI_ISL_14000155, EPI_ISL_14004519, EPI_ISL_14005794, EPI_ISL_14010460, EPI_ISL_14010462, EPI_ISL_14011475, EPI_ISL_14015047, EPI_ISL_14019093, EPI_ISL_14019109, EPI_ISL_14019330, EPI_ISL_14020697, EPI_ISL_14022780, EPI_ISL_14022892, EPI_ISL_14023662, EPI_ISL_14027304, EPI_ISL_14027788, EPI_ISL_14027974, EPI_ISL_14028215, EPI_ISL_14030175, EPI_ISL_14032717, EPI_ISL_14036069, EPI_ISL_14041069, EPI_ISL_14044698, EPI_ISL_14044704, EPI_ISL_14046291, EPI_ISL_14047361, EPI_ISL_14050544, EPI_ISL_14051041, EPI_ISL_14057503, EPI_ISL_14062117, EPI_ISL_14064598, EPI_ISL_14064601, EPI_ISL_14066591, EPI_ISL_14066852, EPI_ISL_14071587, EPI_ISL_14071795, EPI_ISL_14097542, EPI_ISL_14097629, EPI_ISL_14124074, EPI_ISL_14128514, EPI_ISL_14134678, EPI_ISL_14135854, EPI_ISL_14147202, EPI_ISL_14155218, EPI_ISL_14158264, EPI_ISL_14161024, EPI_ISL_14170603, EPI_ISL_14171301, EPI_ISL_14172905, EPI_ISL_14173767, EPI_ISL_14175092, EPI_ISL_14175097, EPI_ISL_14175103, EPI_ISL_14175182, EPI_ISL_14178690, EPI_ISL_14193000, EPI_ISL_14196068, EPI_ISL_14197724, EPI_ISL_14198080, EPI_ISL_14200801, EPI_ISL_14203613, EPI_ISL_14208740, EPI_ISL_14209372, EPI_ISL_14209934, EPI_ISL_14210619, EPI_ISL_14215818, EPI_ISL_14216595, EPI_ISL_14222817, EPI_ISL_14223595, EPI_ISL_14224871, EPI_ISL_14225747, EPI_ISL_14228030, EPI_ISL_14229584, EPI_ISL_14230544, EPI_ISL_14231739, EPI_ISL_14231749, EPI_ISL_14231751, EPI_ISL_14232221, EPI_ISL_14234017, EPI_ISL_14241722, EPI_ISL_14243471, EPI_ISL_14243503, EPI_ISL_14243509, EPI_ISL_14253197, EPI_ISL_14257394, EPI_ISL_14257902, EPI_ISL_14257905, EPI_ISL_14259114, EPI_ISL_14259905, EPI_ISL_14260215, EPI_ISL_14261704, EPI_ISL_14262778, EPI_ISL_14263077, EPI_ISL_14277057, EPI_ISL_14287370, EPI_ISL_14288695, EPI_ISL_14289901, EPI_ISL_14289906, EPI_ISL_14292615, EPI_ISL_14292645, EPI_ISL_14292727, EPI_ISL_14292796, EPI_ISL_14296586, EPI_ISL_14298637, EPI_ISL_14301376, EPI_ISL_14305792, EPI_ISL_14311909, EPI_ISL_14311965, EPI_ISL_14312743, EPI_ISL_14321842, EPI_ISL_14340832, EPI_ISL_14353536, EPI_ISL_14359010, EPI_ISL_14382623, EPI_ISL_14387989, EPI_ISL_14389788, EPI_ISL_14389796, EPI_ISL_14391372, EPI_ISL_14393120, EPI_ISL_14393933, EPI_ISL_14409468, EPI_ISL_14416474, EPI_ISL_14425116, EPI_ISL_14425894, EPI_ISL_14426235, EPI_ISL_14426336, EPI_ISL_14430592, EPI_ISL_14433737, EPI_ISL_14436225, EPI_ISL_14437098, EPI_ISL_14439513, EPI_ISL_14439514, EPI_ISL_14439530, EPI_ISL_14439649, EPI_ISL_14439686, EPI_ISL_14440238, EPI_ISL_14448667, EPI_ISL_14455168, EPI_ISL_14459779, EPI_ISL_14462783, EPI_ISL_14464386, EPI_ISL_14467169, EPI_ISL_14470997, EPI_ISL_14471721, EPI_ISL_14478172, EPI_ISL_14478208, EPI_ISL_14479735, EPI_ISL_14483275, EPI_ISL_14485890, EPI_ISL_14487304, EPI_ISL_14487315, EPI_ISL_14490179, EPI_ISL_14493139, EPI_ISL_14493608, EPI_ISL_14493822, EPI_ISL_14493989, EPI_ISL_14494098, EPI_ISL_14494322, EPI_ISL_14496822, EPI_ISL_14497316, EPI_ISL_14498244, EPI_ISL_14503169, EPI_ISL_14503437, EPI_ISL_14504973, EPI_ISL_14505974, EPI_ISL_14507199, EPI_ISL_14507200, EPI_ISL_14509715, EPI_ISL_14513137, EPI_ISL_14518038, EPI_ISL_14518039, EPI_ISL_14518040, EPI_ISL_14518101, EPI_ISL_14518137, EPI_ISL_14523885, EPI_ISL_14527351, EPI_ISL_14527924, EPI_ISL_14535112, EPI_ISL_14540192, EPI_ISL_14544667, EPI_ISL_14545270, EPI_ISL_14551066, EPI_ISL_14552116, EPI_ISL_14553998, EPI_ISL_14554016, EPI_ISL_14554020, EPI_ISL_14554042, EPI_ISL_14554047, EPI_ISL_14556650, EPI_ISL_14560721, EPI_ISL_14560826, EPI_ISL_14560873, EPI_ISL_14561036, EPI_ISL_14561487, EPI_ISL_14561987, EPI_ISL_14562000, EPI_ISL_14562820, EPI_ISL_14571366, EPI_ISL_14572777, EPI_ISL_14574572, EPI_ISL_14577541, EPI_ISL_14577981, EPI_ISL_14578599, EPI_ISL_14588127, EPI_ISL_14596883, EPI_ISL_14599772, EPI_ISL_14599778, EPI_ISL_14599779, EPI_ISL_14599780, EPI_ISL_14602583, EPI_ISL_14602992, EPI_ISL_14606016, EPI_ISL_14608770, EPI_ISL_14610722, EPI_ISL_14610723, EPI_ISL_14610724, EPI_ISL_14610725, EPI_ISL_14610726, EPI_ISL_14610727, EPI_ISL_14610728, EPI_ISL_14610729, EPI_ISL_14610730, EPI_ISL_14610731, EPI_ISL_14610732, EPI_ISL_14610733, EPI_ISL_14610734, EPI_ISL_14613632, EPI_ISL_14613671, EPI_ISL_14616144, EPI_ISL_14616443, EPI_ISL_14616543, EPI_ISL_14616681, EPI_ISL_14618772, EPI_ISL_14619952, EPI_ISL_14623561, EPI_ISL_14623599, EPI_ISL_14624407, EPI_ISL_14625263, EPI_ISL_14630170, EPI_ISL_14647032, EPI_ISL_14652006, EPI_ISL_14665394, EPI_ISL_14666760, EPI_ISL_14666761, EPI_ISL_14666762, EPI_ISL_14666764, EPI_ISL_14666765, EPI_ISL_14666766, EPI_ISL_14667834, EPI_ISL_14676287, EPI_ISL_14681429, EPI_ISL_14683500, EPI_ISL_14687471, EPI_ISL_14691921, EPI_ISL_14694460, EPI_ISL_14695466, EPI_ISL_14699555, EPI_ISL_14700183, EPI_ISL_14700285, EPI_ISL_14701161, EPI_ISL_14701325, EPI_ISL_14702629, EPI_ISL_14706169, EPI_ISL_14707196, EPI_ISL_14707197, EPI_ISL_14707948, EPI_ISL_14710821, EPI_ISL_14710834, EPI_ISL_14711613, EPI_ISL_14711614, EPI_ISL_14715522, EPI_ISL_14721837, EPI_ISL_14721894, EPI_ISL_14725600, EPI_ISL_14727457, EPI_ISL_14728608, EPI_ISL_14728814, EPI_ISL_14732990, EPI_ISL_14744620, EPI_ISL_14744804, EPI_ISL_14744809, EPI_ISL_14745146, EPI_ISL_14746124, EPI_ISL_14746196, EPI_ISL_14746271, EPI_ISL_14746905, EPI_ISL_14746964, EPI_ISL_14747246, EPI_ISL_14747247, EPI_ISL_14747621, EPI_ISL_14752384, EPI_ISL_14754570, EPI_ISL_14755727, EPI_ISL_14755766, EPI_ISL_14755933, EPI_ISL_14760819, EPI_ISL_14762572, EPI_ISL_14763447, EPI_ISL_14763711, EPI_ISL_14765653, EPI_ISL_14766331, EPI_ISL_14766361, EPI_ISL_14766363, EPI_ISL_14766433, EPI_ISL_14770484, EPI_ISL_14771903, EPI_ISL_14772260, EPI_ISL_14773345, EPI_ISL_14773569, EPI_ISL_14773570, EPI_ISL_14773839, EPI_ISL_14780448, EPI_ISL_14785887, EPI_ISL_14788048, EPI_ISL_14789391, EPI_ISL_14789392, EPI_ISL_14789508, EPI_ISL_14791420, EPI_ISL_14793146, EPI_ISL_14793618, EPI_ISL_14796633, EPI_ISL_14804570, EPI_ISL_14806018, EPI_ISL_14806413, EPI_ISL_14809350, EPI_ISL_14811078, EPI_ISL_14812412, EPI_ISL_14813068, EPI_ISL_14813161, EPI_ISL_14813215, EPI_ISL_14813300, EPI_ISL_14813995, EPI_ISL_14815433, EPI_ISL_14816346, EPI_ISL_14817985, EPI_ISL_14832276, EPI_ISL_14832977, EPI_ISL_14836181, EPI_ISL_14837867, EPI_ISL_14838049, EPI_ISL_14841625, EPI_ISL_14842196, EPI_ISL_14845057, EPI_ISL_14846399, EPI_ISL_14846400, EPI_ISL_14847727, EPI_ISL_14856139, EPI_ISL_14859457, EPI_ISL_14859713, EPI_ISL_14859716, EPI_ISL_14859829, EPI_ISL_14862263, EPI_ISL_14863183, EPI_ISL_14886333, EPI_ISL_14888736, EPI_ISL_14890020, EPI_ISL_14891391, EPI_ISL_14891763, EPI_ISL_14891765, EPI_ISL_14892114, EPI_ISL_14892395, EPI_ISL_14892970, EPI_ISL_14896074, EPI_ISL_14901195, EPI_ISL_14901198, EPI_ISL_14901439, EPI_ISL_14901672, EPI_ISL_14901693, EPI_ISL_14903212, EPI_ISL_14904331, EPI_ISL_14907633, EPI_ISL_14912863, EPI_ISL_14919989, EPI_ISL_14920419, EPI_ISL_14921805, EPI_ISL_14922117, EPI_ISL_14922118, EPI_ISL_14922929, EPI_ISL_14922954, EPI_ISL_14924448, EPI_ISL_14925471, EPI_ISL_14925480, EPI_ISL_14925483, EPI_ISL_14925487, EPI_ISL_14926371, EPI_ISL_14931103, EPI_ISL_14934229, EPI_ISL_14934234, EPI_ISL_14935361, EPI_ISL_14935895, EPI_ISL_14935908, EPI_ISL_14935930, EPI_ISL_14935931, EPI_ISL_14937654, EPI_ISL_14937852, EPI_ISL_14937864, EPI_ISL_14942094, EPI_ISL_14942184, EPI_ISL_14942530, EPI_ISL_14943208, EPI_ISL_14943290, EPI_ISL_14946958, EPI_ISL_14949065, EPI_ISL_14950282, EPI_ISL_14951595, EPI_ISL_14951609, EPI_ISL_14951892, EPI_ISL_14952059, EPI_ISL_14952205, EPI_ISL_14952220, EPI_ISL_14953487, EPI_ISL_14955378, EPI_ISL_14959923, EPI_ISL_14960752, EPI_ISL_14960911, EPI_ISL_14961972, EPI_ISL_14962212, EPI_ISL_14962429, EPI_ISL_14962617, EPI_ISL_14971640, EPI_ISL_14975894, EPI_ISL_14977865, EPI_ISL_14980656, EPI_ISL_14992324, EPI_ISL_14993023, EPI_ISL_14994380, EPI_ISL_14995884, EPI_ISL_14995958, EPI_ISL_14997830, EPI_ISL_14999886, EPI_ISL_15003867, EPI_ISL_15005560, EPI_ISL_15010697, EPI_ISL_15012784, EPI_ISL_15013151, EPI_ISL_15013344, EPI_ISL_15014035, EPI_ISL_15014516, EPI_ISL_15015307, EPI_ISL_15017244, EPI_ISL_15018502, EPI_ISL_15022788, EPI_ISL_15024848, EPI_ISL_15026124, EPI_ISL_15026704, EPI_ISL_15030291, EPI_ISL_15030370, EPI_ISL_15032101, EPI_ISL_15036387, EPI_ISL_15038137, EPI_ISL_15040430, EPI_ISL_15040463, EPI_ISL_15040845, EPI_ISL_15040855, EPI_ISL_15040867, EPI_ISL_15047139, EPI_ISL_15048524, EPI_ISL_15050379, EPI_ISL_15051633, EPI_ISL_15058728, EPI_ISL_15071928, EPI_ISL_15072261, EPI_ISL_15072510, EPI_ISL_15072543, EPI_ISL_15072550, EPI_ISL_15072553, EPI_ISL_15072554, EPI_ISL_15072999, EPI_ISL_15075043, EPI_ISL_15075836, EPI_ISL_15076071, EPI_ISL_15077422, EPI_ISL_15078481, EPI_ISL_15083622, EPI_ISL_15084091, EPI_ISL_15085357, EPI_ISL_15085883, EPI_ISL_15085910, EPI_ISL_15086100, EPI_ISL_15086132, EPI_ISL_15086246, EPI_ISL_15088435, EPI_ISL_15088854, EPI_ISL_15093244, EPI_ISL_15093817, EPI_ISL_15093818, EPI_ISL_15094085, EPI_ISL_15096597, EPI_ISL_15096672, EPI_ISL_15098367, EPI_ISL_15101602, EPI_ISL_15107059, EPI_ISL_15107248, EPI_ISL_15107529, EPI_ISL_15108940, EPI_ISL_15108982, EPI_ISL_15109913, EPI_ISL_15111369, EPI_ISL_15114528, EPI_ISL_15114696, EPI_ISL_15115237, EPI_ISL_15116712, EPI_ISL_15118472, EPI_ISL_15118484, EPI_ISL_15119416, EPI_ISL_15125352, EPI_ISL_15126616, EPI_ISL_15129252, EPI_ISL_15135632, EPI_ISL_15137908, EPI_ISL_15137948, EPI_ISL_15140027, EPI_ISL_15140068, EPI_ISL_15145892, EPI_ISL_15145981, EPI_ISL_15156399, EPI_ISL_15160596, EPI_ISL_15161674, EPI_ISL_15169761, EPI_ISL_15169791, EPI_ISL_15170512, EPI_ISL_15171102, EPI_ISL_15172949, EPI_ISL_15173621, EPI_ISL_15175083, EPI_ISL_15177304, EPI_ISL_15177330, EPI_ISL_15177334, EPI_ISL_15178067, EPI_ISL_15178215, EPI_ISL_15184076, EPI_ISL_15184282, EPI_ISL_15184330, EPI_ISL_15191475, EPI_ISL_15191490, EPI_ISL_15191491, EPI_ISL_15191505, EPI_ISL_15191642, EPI_ISL_15191714, EPI_ISL_15191804, EPI_ISL_15192502, EPI_ISL_15193406, EPI_ISL_15195342, EPI_ISL_15195410, EPI_ISL_15195634, EPI_ISL_15195645, EPI_ISL_15195724, EPI_ISL_15198987, EPI_ISL_15210372, EPI_ISL_15210373, EPI_ISL_15211295, EPI_ISL_15211305, EPI_ISL_15213088, EPI_ISL_15215446, EPI_ISL_15215974, EPI_ISL_15216639, EPI_ISL_15217187, EPI_ISL_15218165, EPI_ISL_15222709, EPI_ISL_15229199, EPI_ISL_15231108, EPI_ISL_15236061, EPI_ISL_15236355, EPI_ISL_15241555, EPI_ISL_15242970, EPI_ISL_15247582, EPI_ISL_15248681, EPI_ISL_15250551, EPI_ISL_15251240, EPI_ISL_15251241, EPI_ISL_15251242, EPI_ISL_15251243, EPI_ISL_15254725, EPI_ISL_15256010, EPI_ISL_15257404, EPI_ISL_15268715, EPI_ISL_15268834, EPI_ISL_15273578, EPI_ISL_15275240, EPI_ISL_15277675, EPI_ISL_15278730, EPI_ISL_15278787, EPI_ISL_15279546, EPI_ISL_15279743, EPI_ISL_15284364, EPI_ISL_15284373, EPI_ISL_15284374, EPI_ISL_15284586, EPI_ISL_15286527, EPI_ISL_15287330, EPI_ISL_15292331, EPI_ISL_15294656, EPI_ISL_15296403, EPI_ISL_15305368, EPI_ISL_15305862, EPI_ISL_15306067, EPI_ISL_15307010, EPI_ISL_15307651, EPI_ISL_15310561, EPI_ISL_15312119, EPI_ISL_15314949, EPI_ISL_15317884, EPI_ISL_15322076, EPI_ISL_15322793, EPI_ISL_15325687, EPI_ISL_15325946, EPI_ISL_15328668, EPI_ISL_15330077, EPI_ISL_15330418, EPI_ISL_15331994, EPI_ISL_15332912, EPI_ISL_15333287, EPI_ISL_15335656, EPI_ISL_15335800, EPI_ISL_15338015, EPI_ISL_15338081, EPI_ISL_15340355, EPI_ISL_15341321, EPI_ISL_15347054, EPI_ISL_15348674, EPI_ISL_15348926, EPI_ISL_15348927, EPI_ISL_15348928, EPI_ISL_15348929, EPI_ISL_15349852, EPI_ISL_15352149, EPI_ISL_15354679, EPI_ISL_15354775, EPI_ISL_15357057, EPI_ISL_15357960, EPI_ISL_15362650, EPI_ISL_15363544, EPI_ISL_15367191, EPI_ISL_15368893, EPI_ISL_15370137, EPI_ISL_15370885, EPI_ISL_15370889, EPI_ISL_15376124, EPI_ISL_15376348, EPI_ISL_15376376, EPI_ISL_15379719, EPI_ISL_15380518, EPI_ISL_15384507, EPI_ISL_15384545, EPI_ISL_15385232, EPI_ISL_15385971, EPI_ISL_15387248, EPI_ISL_15387296, EPI_ISL_15387509, EPI_ISL_15387686, EPI_ISL_15388463, EPI_ISL_15389278, EPI_ISL_15392529, EPI_ISL_15393294, EPI_ISL_15393302, EPI_ISL_15396381, EPI_ISL_15403852, EPI_ISL_15403865, EPI_ISL_15403898, EPI_ISL_15405688, EPI_ISL_15408697, EPI_ISL_15409673, EPI_ISL_15415649, EPI_ISL_15420040, EPI_ISL_15420138, EPI_ISL_15420212, EPI_ISL_15420431, EPI_ISL_15420631, EPI_ISL_15423234, EPI_ISL_15424211, EPI_ISL_15424276, EPI_ISL_15424884, EPI_ISL_15427653, EPI_ISL_15434634, EPI_ISL_15434919, EPI_ISL_15435185, EPI_ISL_15436140, EPI_ISL_15436494, EPI_ISL_15436498, EPI_ISL_15436499, EPI_ISL_15442625, EPI_ISL_15442735, EPI_ISL_15446553, EPI_ISL_15456143, EPI_ISL_15462878, EPI_ISL_15463120, EPI_ISL_15471419, EPI_ISL_15471420, EPI_ISL_15472394, EPI_ISL_15472759, EPI_ISL_15473942, EPI_ISL_15473981, EPI_ISL_15473994, EPI_ISL_15476105, EPI_ISL_15476158, EPI_ISL_15476180, EPI_ISL_15476315, EPI_ISL_15476724, EPI_ISL_15479591, EPI_ISL_15481002, EPI_ISL_15486348, EPI_ISL_15490572, EPI_ISL_15490728, EPI_ISL_15492743, EPI_ISL_15492887, EPI_ISL_15494260, EPI_ISL_15494897, EPI_ISL_15495028, EPI_ISL_15495125, EPI_ISL_15496641, EPI_ISL_15501222, EPI_ISL_15502864, EPI_ISL_15505215, EPI_ISL_15505985, EPI_ISL_15506333, EPI_ISL_15507204, EPI_ISL_15507296, EPI_ISL_15507616, EPI_ISL_15509746, EPI_ISL_15509755, EPI_ISL_15509992, EPI_ISL_15511841, EPI_ISL_15511842, EPI_ISL_15511843, EPI_ISL_15513583, EPI_ISL_15513588, EPI_ISL_15513663, EPI_ISL_15514216, EPI_ISL_15514265, EPI_ISL_15514302, EPI_ISL_15514923, EPI_ISL_15523591, EPI_ISL_15528152, EPI_ISL_15528328, EPI_ISL_15528329, EPI_ISL_15528330, EPI_ISL_15528331, EPI_ISL_15528332, EPI_ISL_15528333, EPI_ISL_15528334, EPI_ISL_15535800, EPI_ISL_15537619, EPI_ISL_15538513, EPI_ISL_15538645, EPI_ISL_15541751, EPI_ISL_15542503, EPI_ISL_15549778, EPI_ISL_15549981, EPI_ISL_15550525, EPI_ISL_15557544, EPI_ISL_15558837, EPI_ISL_15568780, EPI_ISL_15579728, EPI_ISL_15579786, EPI_ISL_15580359, EPI_ISL_15580699, EPI_ISL_15581446, EPI_ISL_15581681, EPI_ISL_15581727, EPI_ISL_15581931, EPI_ISL_15581932, EPI_ISL_15581939, EPI_ISL_15582076, EPI_ISL_15582517, EPI_ISL_15585338, EPI_ISL_15587950, EPI_ISL_15588132, EPI_ISL_15594682, EPI_ISL_15595518, EPI_ISL_15598104, EPI_ISL_15598966, EPI_ISL_15602198, EPI_ISL_15604595, EPI_ISL_15607872, EPI_ISL_15608835, EPI_ISL_15609106, EPI_ISL_15609107, EPI_ISL_15610726, EPI_ISL_15610881, EPI_ISL_15612047, EPI_ISL_15612048, EPI_ISL_15612190, EPI_ISL_15614101, EPI_ISL_15614116, EPI_ISL_15614150, EPI_ISL_15614383, EPI_ISL_15614456, EPI_ISL_15614490, EPI_ISL_15616889, EPI_ISL_15617621, EPI_ISL_15617635, EPI_ISL_15619675, EPI_ISL_15626705, EPI_ISL_15626859, EPI_ISL_15626955, EPI_ISL_15628252, EPI_ISL_15630041, EPI_ISL_15631537, EPI_ISL_15635022, EPI_ISL_15636641, EPI_ISL_15637121, EPI_ISL_15637182, EPI_ISL_15638597, EPI_ISL_15639067, EPI_ISL_15642936, EPI_ISL_15642980, EPI_ISL_15649157, EPI_ISL_15650076, EPI_ISL_15650225, EPI_ISL_15653695, EPI_ISL_15654640, EPI_ISL_15656922, EPI_ISL_15659716, EPI_ISL_15659847, EPI_ISL_15659982, EPI_ISL_15661609, EPI_ISL_15666581, EPI_ISL_15666595, EPI_ISL_15666598, EPI_ISL_15667047, EPI_ISL_15669004, EPI_ISL_15671244, EPI_ISL_15671388, EPI_ISL_15671577, EPI_ISL_15671878, EPI_ISL_15671888, EPI_ISL_15673934, EPI_ISL_15675248, EPI_ISL_15678339, EPI_ISL_15681826, EPI_ISL_15685722, EPI_ISL_15687965, EPI_ISL_15688500, EPI_ISL_15688701, EPI_ISL_15692625, EPI_ISL_15693169, EPI_ISL_15693174, EPI_ISL_15693676, EPI_ISL_15700160, EPI_ISL_15705061, EPI_ISL_15712450, EPI_ISL_15712747, EPI_ISL_15719141, EPI_ISL_15719142, EPI_ISL_15719143, EPI_ISL_15721137, EPI_ISL_15721185, EPI_ISL_15721190, EPI_ISL_15723589, EPI_ISL_15725799, EPI_ISL_15728467, EPI_ISL_15728613, EPI_ISL_15728673, EPI_ISL_15729287, EPI_ISL_15729288, EPI_ISL_15729308, EPI_ISL_15729309, EPI_ISL_15729310, EPI_ISL_15729311, EPI_ISL_15729315, EPI_ISL_15729341, EPI_ISL_15729358, EPI_ISL_15731233, EPI_ISL_15731409, EPI_ISL_15732413, EPI_ISL_15736424, EPI_ISL_15739498, EPI_ISL_15739617, EPI_ISL_15742450, EPI_ISL_15743318, EPI_ISL_15743816, EPI_ISL_15754145, EPI_ISL_15754794, EPI_ISL_15760224, EPI_ISL_15760382, EPI_ISL_15760554, EPI_ISL_15760609, EPI_ISL_15760812, EPI_ISL_15761520, EPI_ISL_15761543, EPI_ISL_15761663, EPI_ISL_15763216, EPI_ISL_15764573, EPI_ISL_15765022, EPI_ISL_15768827, EPI_ISL_15773132, EPI_ISL_15773881, EPI_ISL_15773907, EPI_ISL_15776989, EPI_ISL_15779724, EPI_ISL_15780387, EPI_ISL_15781197, EPI_ISL_15781220, EPI_ISL_15785782, EPI_ISL_15786114, EPI_ISL_15786255, EPI_ISL_15789537, EPI_ISL_15790179, EPI_ISL_15790657, EPI_ISL_15790738, EPI_ISL_15791223, EPI_ISL_15791252, EPI_ISL_15792351, EPI_ISL_15793981, EPI_ISL_15797751, EPI_ISL_15798331, EPI_ISL_15801425, EPI_ISL_15801499, EPI_ISL_15801515, EPI_ISL_15815337, EPI_ISL_15815889, EPI_ISL_15818486, EPI_ISL_15820055, EPI_ISL_15820336, EPI_ISL_15822919, EPI_ISL_15824080, EPI_ISL_15824099, EPI_ISL_15824207, EPI_ISL_15826867, EPI_ISL_15829108, EPI_ISL_15832444, EPI_ISL_15837686, EPI_ISL_15837751, EPI_ISL_15837827, EPI_ISL_15838124, EPI_ISL_15839941, EPI_ISL_15840082, EPI_ISL_15843473, EPI_ISL_15843671, EPI_ISL_15844032, EPI_ISL_15844165, EPI_ISL_15845753, EPI_ISL_15845778, EPI_ISL_15845946, EPI_ISL_15846023, EPI_ISL_15846264, EPI_ISL_15849690, EPI_ISL_15850759, EPI_ISL_15850872, EPI_ISL_15853809, EPI_ISL_15853943, EPI_ISL_15856103, EPI_ISL_15856463, EPI_ISL_15856822, EPI_ISL_15857383, EPI_ISL_15860163, EPI_ISL_15864217, EPI_ISL_15864218, EPI_ISL_15865301, EPI_ISL_15865421, EPI_ISL_15865482, EPI_ISL_15866887, EPI_ISL_15874567, EPI_ISL_15878818, EPI_ISL_15882637, EPI_ISL_15883009, EPI_ISL_15887656, EPI_ISL_15895625, EPI_ISL_15896804, EPI_ISL_15896923, EPI_ISL_15897067, EPI_ISL_15897092, EPI_ISL_15898992, EPI_ISL_15900796, EPI_ISL_15910198, EPI_ISL_15911160, EPI_ISL_15912221, EPI_ISL_15912222, EPI_ISL_15912223, EPI_ISL_15912224, EPI_ISL_15914119, EPI_ISL_15916592, EPI_ISL_15920181, EPI_ISL_15920505, EPI_ISL_15920753, EPI_ISL_15920754, EPI_ISL_15920755, EPI_ISL_15924323, EPI_ISL_15924325, EPI_ISL_15926083, EPI_ISL_15926723, EPI_ISL_15928909, EPI_ISL_15929002, EPI_ISL_15929151, EPI_ISL_15932554, EPI_ISL_15934274, EPI_ISL_15937718, EPI_ISL_15938074, EPI_ISL_15941879, EPI_ISL_15941880, EPI_ISL_15945504, EPI_ISL_15949533, EPI_ISL_15953870, EPI_ISL_15958934, EPI_ISL_15959369, EPI_ISL_15961456, EPI_ISL_15962045, EPI_ISL_15965716, EPI_ISL_15966131, EPI_ISL_15966527, EPI_ISL_15969420, EPI_ISL_15969421, EPI_ISL_15969437, EPI_ISL_15969438, EPI_ISL_15970088, EPI_ISL_15970187, EPI_ISL_15976036, EPI_ISL_15982641, EPI_ISL_15984958, EPI_ISL_15992803, EPI_ISL_15998627, EPI_ISL_16001974, EPI_ISL_16001995, EPI_ISL_16003220, EPI_ISL_16003255, EPI_ISL_16005246, EPI_ISL_16005457, EPI_ISL_16006665, EPI_ISL_16007931, EPI_ISL_16008877, EPI_ISL_16012424, EPI_ISL_16012845, EPI_ISL_16013074, EPI_ISL_16013086, EPI_ISL_16015099, EPI_ISL_16017107, EPI_ISL_16018930, EPI_ISL_16019056, EPI_ISL_16019744, EPI_ISL_16019756, EPI_ISL_16024407, EPI_ISL_16024682, EPI_ISL_16027431, EPI_ISL_16027937, EPI_ISL_16029135, EPI_ISL_16029382, EPI_ISL_16029654, EPI_ISL_16030181, EPI_ISL_16033087, EPI_ISL_16033186, EPI_ISL_16034832, EPI_ISL_16036598, EPI_ISL_16039444, EPI_ISL_16043974, EPI_ISL_16044428, EPI_ISL_16045410, EPI_ISL_16045424, EPI_ISL_16046711, EPI_ISL_16050095, EPI_ISL_16054133, EPI_ISL_16054451, EPI_ISL_16054953, EPI_ISL_16054963, EPI_ISL_16055434, EPI_ISL_16055461, EPI_ISL_16055527, EPI_ISL_16055697, EPI_ISL_16055721, EPI_ISL_16057031, EPI_ISL_16060790, EPI_ISL_16062229, EPI_ISL_16064284, EPI_ISL_16066333, EPI_ISL_16068281, EPI_ISL_16068583, EPI_ISL_16068914, EPI_ISL_16073080, EPI_ISL_16073469, EPI_ISL_16073474, EPI_ISL_16074871, EPI_ISL_16075086, EPI_ISL_16075127, EPI_ISL_16076496, EPI_ISL_16077353, EPI_ISL_16079016, EPI_ISL_16080170, EPI_ISL_16080871, EPI_ISL_16082206, EPI_ISL_16091870, EPI_ISL_16099360, EPI_ISL_16101516, EPI_ISL_16102479, EPI_ISL_16102480, EPI_ISL_16104542, EPI_ISL_16109108, EPI_ISL_16111875, EPI_ISL_16113331, EPI_ISL_16113603, EPI_ISL_16113806, EPI_ISL_16114505, EPI_ISL_16114631, EPI_ISL_16115703, EPI_ISL_16116190, EPI_ISL_16116234, EPI_ISL_16116659, EPI_ISL_16116707, EPI_ISL_16119498, EPI_ISL_16119508, EPI_ISL_16119512, EPI_ISL_16119517, EPI_ISL_16119519, EPI_ISL_16119805, EPI_ISL_16131965, EPI_ISL_16131986, EPI_ISL_16131997, EPI_ISL_16136901, EPI_ISL_16137616, EPI_ISL_16151030, EPI_ISL_16151463, EPI_ISL_16151651, EPI_ISL_16153650, EPI_ISL_16153658, EPI_ISL_16153800, EPI_ISL_16157875, EPI_ISL_16158109, EPI_ISL_16158197, EPI_ISL_16158326, EPI_ISL_16158363, EPI_ISL_16158374, EPI_ISL_16160252, EPI_ISL_16160296, EPI_ISL_16160313, EPI_ISL_16165250, EPI_ISL_16167761, EPI_ISL_16178283, EPI_ISL_16178634, EPI_ISL_16178844, EPI_ISL_16179355, EPI_ISL_16180564, EPI_ISL_16180574, EPI_ISL_16181797, EPI_ISL_16181828, EPI_ISL_16181950, EPI_ISL_16183022, EPI_ISL_16190977, EPI_ISL_16191476, EPI_ISL_16196150, EPI_ISL_16196167, EPI_ISL_16197958, EPI_ISL_16201173, EPI_ISL_16204855, EPI_ISL_16215808, EPI_ISL_16218191, EPI_ISL_16219709, EPI_ISL_16219753, EPI_ISL_16221691, EPI_ISL_16224408, EPI_ISL_16230801, EPI_ISL_16231330, EPI_ISL_16231873, EPI_ISL_16233000, EPI_ISL_16233650, EPI_ISL_16233865, EPI_ISL_16234790, EPI_ISL_16235313, EPI_ISL_16235462, EPI_ISL_16235523, EPI_ISL_16235930, EPI_ISL_16240521, EPI_ISL_16244367, EPI_ISL_16244373, EPI_ISL_16244408, EPI_ISL_16244923, EPI_ISL_16245232, EPI_ISL_16245289, EPI_ISL_16245329, EPI_ISL_16245400, EPI_ISL_16245414, EPI_ISL_16245433, EPI_ISL_16245601, EPI_ISL_16245627, EPI_ISL_16247208, EPI_ISL_16247263, EPI_ISL_16247490, EPI_ISL_16247545, EPI_ISL_16247708, EPI_ISL_16251186, EPI_ISL_16251466, EPI_ISL_16251507, EPI_ISL_16251579, EPI_ISL_16257294, EPI_ISL_16259227, EPI_ISL_16264400, EPI_ISL_16265325, EPI_ISL_16268074, EPI_ISL_16270258, EPI_ISL_16271245, EPI_ISL_16271444, EPI_ISL_16271604, EPI_ISL_16273936, EPI_ISL_16281373, EPI_ISL_16284103, EPI_ISL_16284207, EPI_ISL_16284311, EPI_ISL_16287253, EPI_ISL_16287690, EPI_ISL_16290877, EPI_ISL_16293662, EPI_ISL_16312661, EPI_ISL_16327295, EPI_ISL_16331524, EPI_ISL_16334679, EPI_ISL_16336940, EPI_ISL_16338847, EPI_ISL_16338862, EPI_ISL_16343084, EPI_ISL_16343221, EPI_ISL_16344593, EPI_ISL_16348840, EPI_ISL_16348868, EPI_ISL_16351967, EPI_ISL_16354229, EPI_ISL_16355537, EPI_ISL_16356453, EPI_ISL_16356458, EPI_ISL_16356910, EPI_ISL_16358915, EPI_ISL_16359990, EPI_ISL_16360495, EPI_ISL_16365715, EPI_ISL_16368903, EPI_ISL_16369869, EPI_ISL_16370037, EPI_ISL_16374806, EPI_ISL_16375356, EPI_ISL_16378181, EPI_ISL_16379359, EPI_ISL_16380313, EPI_ISL_16381332, EPI_ISL_16381679, EPI_ISL_16384522, EPI_ISL_16385377, EPI_ISL_16385455, EPI_ISL_16385456, EPI_ISL_16391752, EPI_ISL_16394844, EPI_ISL_16394922, EPI_ISL_16395667, EPI_ISL_16398472, EPI_ISL_16399824, EPI_ISL_16400033, EPI_ISL_16400342, EPI_ISL_16402001, EPI_ISL_16402612, EPI_ISL_16413336, EPI_ISL_16414127, EPI_ISL_16414190, EPI_ISL_16422834, EPI_ISL_16424130, EPI_ISL_16425691, EPI_ISL_16428100, EPI_ISL_16428101, EPI_ISL_16428102, EPI_ISL_16429066, EPI_ISL_16434308, EPI_ISL_16435903, EPI_ISL_16436533, EPI_ISL_16439413, EPI_ISL_16440696, EPI_ISL_16443129, EPI_ISL_16443688, EPI_ISL_16443874, EPI_ISL_16444178, EPI_ISL_16444600, EPI_ISL_16445144, EPI_ISL_16446802, EPI_ISL_16446804, EPI_ISL_16452054, EPI_ISL_16454044, EPI_ISL_16460278, EPI_ISL_16460823, EPI_ISL_16461302, EPI_ISL_16464657, EPI_ISL_16467436, EPI_ISL_16470710, EPI_ISL_16470892, EPI_ISL_16471153, EPI_ISL_16471554, EPI_ISL_16471730, EPI_ISL_16471943, EPI_ISL_16473435, EPI_ISL_16473762, EPI_ISL_16474400, EPI_ISL_16475138, EPI_ISL_16475139, EPI_ISL_16479826, EPI_ISL_16482060, EPI_ISL_16489594, EPI_ISL_16491494, EPI_ISL_16492150, EPI_ISL_16492585, EPI_ISL_16492756, EPI_ISL_16493396, EPI_ISL_16493785, EPI_ISL_16495643, EPI_ISL_16497702, EPI_ISL_16498515, EPI_ISL_16503949, EPI_ISL_16507701, EPI_ISL_16507896, EPI_ISL_16507927, EPI_ISL_16520597, EPI_ISL_16520598, EPI_ISL_16520637, EPI_ISL_16520640, EPI_ISL_16520641, EPI_ISL_16520711, EPI_ISL_16520763, EPI_ISL_16520788, EPI_ISL_16521101, EPI_ISL_16524906, EPI_ISL_16528641, EPI_ISL_16528645, EPI_ISL_16528786, EPI_ISL_16528903, EPI_ISL_16535376, EPI_ISL_16536212, EPI_ISL_16539692, EPI_ISL_16541774, EPI_ISL_16542381, EPI_ISL_16542553, EPI_ISL_16543095, EPI_ISL_16544506, EPI_ISL_16553339, EPI_ISL_16553795, EPI_ISL_16567779, EPI_ISL_16574574, EPI_ISL_16577679, EPI_ISL_16577762, EPI_ISL_16581578, EPI_ISL_16583782, EPI_ISL_16584104, EPI_ISL_16585126, EPI_ISL_16585286, EPI_ISL_16586683, EPI_ISL_16586702, EPI_ISL_16587574, EPI_ISL_16597363, EPI_ISL_16598790, EPI_ISL_16598814, EPI_ISL_16599846, EPI_ISL_16607452, EPI_ISL_16607500, EPI_ISL_16607535, EPI_ISL_16611498, EPI_ISL_16611571, EPI_ISL_16613287, EPI_ISL_16613482, EPI_ISL_16615201, EPI_ISL_16615597, EPI_ISL_16615617, EPI_ISL_16615668, EPI_ISL_16616642, EPI_ISL_16625690, EPI_ISL_16626611, EPI_ISL_16626666, EPI_ISL_16627067, EPI_ISL_16628854, EPI_ISL_16630260, EPI_ISL_16630261, EPI_ISL_16636917, EPI_ISL_16637089, EPI_ISL_16637607, EPI_ISL_16637631, EPI_ISL_16638190, EPI_ISL_16638453, EPI_ISL_16640597, EPI_ISL_16643406, EPI_ISL_16647535, EPI_ISL_16647537, EPI_ISL_16647544, EPI_ISL_16647740, EPI_ISL_16649988, EPI_ISL_16650006, EPI_ISL_16650011, EPI_ISL_16653618, EPI_ISL_16669313, EPI_ISL_16669829, EPI_ISL_16672282, EPI_ISL_16672301, EPI_ISL_16672327, EPI_ISL_16672352, EPI_ISL_16676267, EPI_ISL_16677015, EPI_ISL_16678917, EPI_ISL_16678946, EPI_ISL_16679654, EPI_ISL_16681451, EPI_ISL_16681917, EPI_ISL_16682342, EPI_ISL_16682837, EPI_ISL_16688219, EPI_ISL_16688525, EPI_ISL_16688688, EPI_ISL_16688713, EPI_ISL_16689887, EPI_ISL_16691397, EPI_ISL_16691487, EPI_ISL_16694176, EPI_ISL_16695383, EPI_ISL_16695435, EPI_ISL_16697861, EPI_ISL_16700160, EPI_ISL_16702838, EPI_ISL_16705882, EPI_ISL_16706498, EPI_ISL_16708209, EPI_ISL_16708798, EPI_ISL_16711038, EPI_ISL_16711095, EPI_ISL_16711417, EPI_ISL_16711531, EPI_ISL_16716967, EPI_ISL_16721930, EPI_ISL_16722183, EPI_ISL_16722215, EPI_ISL_16722270, EPI_ISL_16722970, EPI_ISL_16723215, EPI_ISL_16725887, EPI_ISL_16727241, EPI_ISL_16728257, EPI_ISL_16728383, EPI_ISL_16728411, EPI_ISL_16729998, EPI_ISL_16730248, EPI_ISL_16731753, EPI_ISL_16735576, EPI_ISL_16735633, EPI_ISL_16736400, EPI_ISL_16738641, EPI_ISL_16739045, EPI_ISL_16739452, EPI_ISL_16740104, EPI_ISL_16740406, EPI_ISL_16741567, EPI_ISL_16741573, EPI_ISL_16743490, EPI_ISL_16749999, EPI_ISL_16750878, EPI_ISL_16751721, EPI_ISL_16751722, EPI_ISL_16751789, EPI_ISL_16751791, EPI_ISL_16751977, EPI_ISL_16752073, EPI_ISL_16752138, EPI_ISL_16757168, EPI_ISL_16757210, EPI_ISL_16758981, EPI_ISL_16760201, EPI_ISL_16764861, EPI_ISL_16765888, EPI_ISL_16766196, EPI_ISL_16811091, EPI_ISL_16812565, EPI_ISL_16815494, EPI_ISL_16816293, EPI_ISL_16818458, EPI_ISL_16818471, EPI_ISL_16825124, EPI_ISL_16825157, EPI_ISL_16825222, EPI_ISL_16825237, EPI_ISL_16825238, EPI_ISL_16825239, EPI_ISL_16828876, EPI_ISL_16828896, EPI_ISL_16829188, EPI_ISL_16831507, EPI_ISL_16832953, EPI_ISL_16833893, EPI_ISL_16834974, EPI_ISL_16834985, EPI_ISL_16835399, EPI_ISL_16842787, EPI_ISL_16842790, EPI_ISL_16847425, EPI_ISL_16847642, EPI_ISL_16847674, EPI_ISL_16847675, EPI_ISL_16847676, EPI_ISL_16847677, EPI_ISL_16847696, EPI_ISL_16847697, EPI_ISL_16849243, EPI_ISL_16849244, EPI_ISL_16849245, EPI_ISL_16853227, EPI_ISL_16853229, EPI_ISL_16853597, EPI_ISL_16856355, EPI_ISL_16856565, EPI_ISL_16856637, EPI_ISL_16856833, EPI_ISL_16857514, EPI_ISL_16857776, EPI_ISL_16858617, EPI_ISL_16858667, EPI_ISL_16861084, EPI_ISL_16862130, EPI_ISL_16863260, EPI_ISL_16863802, EPI_ISL_16866580, EPI_ISL_16867553, EPI_ISL_16868647, EPI_ISL_16868654, EPI_ISL_16868655, EPI_ISL_16868993, EPI_ISL_16869007, EPI_ISL_16875565, EPI_ISL_16875752, EPI_ISL_16876039, EPI_ISL_16876784, EPI_ISL_16877428, EPI_ISL_16878720, EPI_ISL_16880439, EPI_ISL_16883240, EPI_ISL_16883873, EPI_ISL_16884622, EPI_ISL_16890808, EPI_ISL_16893283, EPI_ISL_16894717, EPI_ISL_16895138, EPI_ISL_16895290, EPI_ISL_16895522, EPI_ISL_16897500, EPI_ISL_16899216, EPI_ISL_16903492, EPI_ISL_16903494, EPI_ISL_16904444, EPI_ISL_16904536, EPI_ISL_16908446, EPI_ISL_16908447, EPI_ISL_16908450, EPI_ISL_16908472, EPI_ISL_16909804, EPI_ISL_16910025, EPI_ISL_16910165, EPI_ISL_16910272, EPI_ISL_16911226, EPI_ISL_16911909, EPI_ISL_16913144, EPI_ISL_16921530, EPI_ISL_16921542, EPI_ISL_16925257, EPI_ISL_16927736, EPI_ISL_16931901, EPI_ISL_16939487, EPI_ISL_16941750, EPI_ISL_16941980, EPI_ISL_16942000, EPI_ISL_16942017, EPI_ISL_16945429, EPI_ISL_16946783, EPI_ISL_16947592, EPI_ISL_16947625, EPI_ISL_16951592, EPI_ISL_16953425, EPI_ISL_16953741, EPI_ISL_16954486, EPI_ISL_16955471, EPI_ISL_16957015, EPI_ISL_16966140, EPI_ISL_16966997, EPI_ISL_16967082, EPI_ISL_16967083, EPI_ISL_16967084, EPI_ISL_16967085, EPI_ISL_16967086, EPI_ISL_16969756, EPI_ISL_16969757, EPI_ISL_16970279, EPI_ISL_16973343, EPI_ISL_16977317, EPI_ISL_16977653, EPI_ISL_16977749, EPI_ISL_16979306, EPI_ISL_16979482, EPI_ISL_16980075, EPI_ISL_16980683, EPI_ISL_16981030, EPI_ISL_16981047, EPI_ISL_16981048, EPI_ISL_16981102, EPI_ISL_16985060, EPI_ISL_16987088, EPI_ISL_16987375, EPI_ISL_16987376, EPI_ISL_16988527, EPI_ISL_16989157, EPI_ISL_16989160, EPI_ISL_16995247, EPI_ISL_16995491, EPI_ISL_16995525, EPI_ISL_16995951, EPI_ISL_16997638, EPI_ISL_17001974, EPI_ISL_17001987, EPI_ISL_17002442, EPI_ISL_17006258, EPI_ISL_17008393, EPI_ISL_17008502, EPI_ISL_17016219, EPI_ISL_17016344, EPI_ISL_17018731, EPI_ISL_17020636, EPI_ISL_17021187, EPI_ISL_17022063, EPI_ISL_17022081, EPI_ISL_17024099, EPI_ISL_17025560, EPI_ISL_17025998, EPI_ISL_17026052, EPI_ISL_17026537, EPI_ISL_17027430, EPI_ISL_17032070, EPI_ISL_17032664, EPI_ISL_17035345, EPI_ISL_17036551, EPI_ISL_17037388, EPI_ISL_17040117, EPI_ISL_17040120, EPI_ISL_17040127, EPI_ISL_17040130, EPI_ISL_17040131, EPI_ISL_17040132, EPI_ISL_17040133, EPI_ISL_17040134, EPI_ISL_17040184, EPI_ISL_17041089, EPI_ISL_17041105, EPI_ISL_17041117, EPI_ISL_17041143, EPI_ISL_17044002, EPI_ISL_17046406, EPI_ISL_17047368, EPI_ISL_17047667, EPI_ISL_17050958, EPI_ISL_17051743, EPI_ISL_17056159, EPI_ISL_17057279, EPI_ISL_17061716, EPI_ISL_17062081, EPI_ISL_17065016, EPI_ISL_17067007, EPI_ISL_17068616, EPI_ISL_17068621, EPI_ISL_17073286, EPI_ISL_17074720, EPI_ISL_17075299, EPI_ISL_17076011, EPI_ISL_17076926, EPI_ISL_17077233, EPI_ISL_17077446, EPI_ISL_17079150, EPI_ISL_17079151, EPI_ISL_17079427, EPI_ISL_17080036, EPI_ISL_17080146, EPI_ISL_17080283, EPI_ISL_17080510, EPI_ISL_17081567, EPI_ISL_17084330, EPI_ISL_17086936, EPI_ISL_17086958, EPI_ISL_17090268, EPI_ISL_17092242, EPI_ISL_17092260, EPI_ISL_17099321, EPI_ISL_17099446, EPI_ISL_17101049, EPI_ISL_17101231, EPI_ISL_17104807, EPI_ISL_17105674, EPI_ISL_17105677, EPI_ISL_17105765, EPI_ISL_17105777, EPI_ISL_17105786, EPI_ISL_17105789, EPI_ISL_17105804, EPI_ISL_17106895, EPI_ISL_17109738, EPI_ISL_17109787, EPI_ISL_17112198, EPI_ISL_17113114, EPI_ISL_17118740, EPI_ISL_17119370, EPI_ISL_17126699, EPI_ISL_17126727, EPI_ISL_17127510, EPI_ISL_17129671, EPI_ISL_17139969, EPI_ISL_17144489, EPI_ISL_17149673, EPI_ISL_17149697, EPI_ISL_17150312, EPI_ISL_17152522, EPI_ISL_17152602, EPI_ISL_17152816, EPI_ISL_17154843, EPI_ISL_17154893, EPI_ISL_17158601, EPI_ISL_17158659, EPI_ISL_17158660, EPI_ISL_17158661, EPI_ISL_17158662, EPI_ISL_17158663, EPI_ISL_17158664, EPI_ISL_17158665, EPI_ISL_17164529, EPI_ISL_17165387, EPI_ISL_17165528, EPI_ISL_17170921, EPI_ISL_17173754, EPI_ISL_17174263, EPI_ISL_17174278, EPI_ISL_17174323, EPI_ISL_17175107, EPI_ISL_17180776, EPI_ISL_17182281, EPI_ISL_17188691, EPI_ISL_17188772, EPI_ISL_17188836, EPI_ISL_17189286, EPI_ISL_17189360, EPI_ISL_17189372, EPI_ISL_17189865, EPI_ISL_17190813, EPI_ISL_17191784, EPI_ISL_17193988, EPI_ISL_17194121, EPI_ISL_17194564, EPI_ISL_17195276, EPI_ISL_17195807, EPI_ISL_17198415, EPI_ISL_17199165, EPI_ISL_17199381, EPI_ISL_17199743, EPI_ISL_17200348, EPI_ISL_17200520, EPI_ISL_17201694, EPI_ISL_17202051, EPI_ISL_17205892, EPI_ISL_17206016, EPI_ISL_17206140, EPI_ISL_17207424, EPI_ISL_17210230, EPI_ISL_17210689, EPI_ISL_17214413, EPI_ISL_17214693, EPI_ISL_17214774, EPI_ISL_17214805, EPI_ISL_17214933, EPI_ISL_17215427, EPI_ISL_17215676, EPI_ISL_17215686, EPI_ISL_17215790, EPI_ISL_17216822, EPI_ISL_17216978, EPI_ISL_17221710, EPI_ISL_17222365, EPI_ISL_17223438, EPI_ISL_17226531, EPI_ISL_17226637, EPI_ISL_17232350, EPI_ISL_17232448, EPI_ISL_17237921, EPI_ISL_17239049, EPI_ISL_17239405, EPI_ISL_17239499, EPI_ISL_17241376, EPI_ISL_17244630, EPI_ISL_17244668, EPI_ISL_17245140, EPI_ISL_17245198, EPI_ISL_17245226, EPI_ISL_17245255, EPI_ISL_17246876, EPI_ISL_17246931, EPI_ISL_17247186, EPI_ISL_17247325, EPI_ISL_17247333, EPI_ISL_17247834, EPI_ISL_17248435, EPI_ISL_17251028, EPI_ISL_17252934, EPI_ISL_17253364, EPI_ISL_17253589, EPI_ISL_17257608, EPI_ISL_17262137, EPI_ISL_17265160, EPI_ISL_17270165, EPI_ISL_17270215, EPI_ISL_17270470, EPI_ISL_17270618, EPI_ISL_17270950, EPI_ISL_17270964, EPI_ISL_17270974, EPI_ISL_17271226, EPI_ISL_17271272, EPI_ISL_17272946, EPI_ISL_17275616, EPI_ISL_17275984, EPI_ISL_17276025, EPI_ISL_17276030, EPI_ISL_17276098, EPI_ISL_17276962, EPI_ISL_17284010, EPI_ISL_17284045, EPI_ISL_17284573, EPI_ISL_17284790, EPI_ISL_17284791, EPI_ISL_17284792, EPI_ISL_17284793, EPI_ISL_17284794, EPI_ISL_17284795, EPI_ISL_17284796, EPI_ISL_17285690, EPI_ISL_17288589, EPI_ISL_17290740, EPI_ISL_17292666, EPI_ISL_17292834, EPI_ISL_17297993, EPI_ISL_17298321, EPI_ISL_17298323, EPI_ISL_17299688, EPI_ISL_17300150, EPI_ISL_17304801, EPI_ISL_17304899, EPI_ISL_17305358, EPI_ISL_17319411, EPI_ISL_17319528, EPI_ISL_17319601, EPI_ISL_17321362, EPI_ISL_17322993, EPI_ISL_17334027, EPI_ISL_17342544, EPI_ISL_17344004, EPI_ISL_17344178, EPI_ISL_17344660, EPI_ISL_17345445, EPI_ISL_17347577, EPI_ISL_17348219, EPI_ISL_17349770, EPI_ISL_17349983, EPI_ISL_17350301, EPI_ISL_17352192, EPI_ISL_17358767, EPI_ISL_17359772, EPI_ISL_17370155, EPI_ISL_17371048, EPI_ISL_17374170, EPI_ISL_17374605, EPI_ISL_17374609, EPI_ISL_17374807, EPI_ISL_17376230, EPI_ISL_17381216, EPI_ISL_17387122, EPI_ISL_17389140, EPI_ISL_17389210, EPI_ISL_17389223, EPI_ISL_17389779, EPI_ISL_17390660, EPI_ISL_17390743, EPI_ISL_17390873, EPI_ISL_17391460, EPI_ISL_17394837, EPI_ISL_17397497, EPI_ISL_17397582, EPI_ISL_17408352, EPI_ISL_17409157, EPI_ISL_17411543, EPI_ISL_17414235, EPI_ISL_17414543, EPI_ISL_17418615, EPI_ISL_17421962, EPI_ISL_17423074, EPI_ISL_17424014, EPI_ISL_17429770, EPI_ISL_17430458, EPI_ISL_17430487, EPI_ISL_17431229, EPI_ISL_17431238, EPI_ISL_17434223, EPI_ISL_17434227, EPI_ISL_17437940, EPI_ISL_17440507, EPI_ISL_17441169, EPI_ISL_17441208, EPI_ISL_17441815, EPI_ISL_17445401, EPI_ISL_17446132, EPI_ISL_17464711, EPI_ISL_17466081, EPI_ISL_17470229, EPI_ISL_17470269, EPI_ISL_17471181, EPI_ISL_17471185, EPI_ISL_17471619, EPI_ISL_17471674, EPI_ISL_17471824, EPI_ISL_17472531, EPI_ISL_17475799, EPI_ISL_17476568, EPI_ISL_17476871, EPI_ISL_17477106, EPI_ISL_17480516, EPI_ISL_17481180, EPI_ISL_17481517, EPI_ISL_17481521, EPI_ISL_17481597, EPI_ISL_17482811, EPI_ISL_17482813, EPI_ISL_17482815, EPI_ISL_17482819, EPI_ISL_17493613, EPI_ISL_17494372, EPI_ISL_17494568, EPI_ISL_17494731, EPI_ISL_17497461, EPI_ISL_17497688, EPI_ISL_17497854, EPI_ISL_17497868, EPI_ISL_17501536, EPI_ISL_17501576, EPI_ISL_17501763, EPI_ISL_17502219, EPI_ISL_17502972, EPI_ISL_17503268, EPI_ISL_17503711, EPI_ISL_17504816, EPI_ISL_17504835, EPI_ISL_17505072, EPI_ISL_17508749, EPI_ISL_17509597, EPI_ISL_17510495, EPI_ISL_17510856, EPI_ISL_17511096, EPI_ISL_17511836, EPI_ISL_17512412, EPI_ISL_17512500, EPI_ISL_17512876, EPI_ISL_17512968, EPI_ISL_17513218, EPI_ISL_17513312, EPI_ISL_17514540, EPI_ISL_17514694, EPI_ISL_17515086, EPI_ISL_17515177, EPI_ISL_17516651, EPI_ISL_17516658, EPI_ISL_17516659, EPI_ISL_17517664, EPI_ISL_17517834, EPI_ISL_17517844, EPI_ISL_17521302, EPI_ISL_17521772, EPI_ISL_17521778, EPI_ISL_17522610, EPI_ISL_17522687, EPI_ISL_17522934, EPI_ISL_17523535, EPI_ISL_17523620, EPI_ISL_17523782, EPI_ISL_17523873, EPI_ISL_17524106, EPI_ISL_17524432, EPI_ISL_17524502, EPI_ISL_17524503, EPI_ISL_17535664, EPI_ISL_17535979, EPI_ISL_17541088, EPI_ISL_17541797, EPI_ISL_17543006, EPI_ISL_17544283, EPI_ISL_17545970, EPI_ISL_17546923, EPI_ISL_17547529, EPI_ISL_17547545, EPI_ISL_17548526, EPI_ISL_17549129, EPI_ISL_17550129, EPI_ISL_17550538, EPI_ISL_17553063, EPI_ISL_17553974, EPI_ISL_17554057, EPI_ISL_17556705, EPI_ISL_17559150, EPI_ISL_17559165, EPI_ISL_17559166, EPI_ISL_17559167, EPI_ISL_17559168, EPI_ISL_17559246, EPI_ISL_17563568, EPI_ISL_17565071, EPI_ISL_17565211, EPI_ISL_17565212, EPI_ISL_17566854, EPI_ISL_17579120, EPI_ISL_17580420, EPI_ISL_17583157, EPI_ISL_17584277, EPI_ISL_17585020, EPI_ISL_17585021, EPI_ISL_17585022, EPI_ISL_17585023, EPI_ISL_17585036, EPI_ISL_17585039, EPI_ISL_17586115, EPI_ISL_17587423, EPI_ISL_17587656, EPI_ISL_17587657, EPI_ISL_17587665, EPI_ISL_17587671, EPI_ISL_17587859, EPI_ISL_17588127, EPI_ISL_17588216, EPI_ISL_17588460, EPI_ISL_17589845, EPI_ISL_17590449, EPI_ISL_17590486, EPI_ISL_17591005, EPI_ISL_17591014, EPI_ISL_17591015, EPI_ISL_17591028, EPI_ISL_17592236, EPI_ISL_17592618, EPI_ISL_17593692, EPI_ISL_17593865, EPI_ISL_17595116, EPI_ISL_17595117, EPI_ISL_17595967, EPI_ISL_17595980, EPI_ISL_17597954, EPI_ISL_17598384, EPI_ISL_17599326, EPI_ISL_17599427, EPI_ISL_17600948, EPI_ISL_17600958, EPI_ISL_17600978, EPI_ISL_17600988, EPI_ISL_17601066, EPI_ISL_17601144, EPI_ISL_17601196, EPI_ISL_17601219, EPI_ISL_17601261, EPI_ISL_17601276, EPI_ISL_17601933, EPI_ISL_17602334, EPI_ISL_17602469, EPI_ISL_17602756, EPI_ISL_17605514, EPI_ISL_17608762, EPI_ISL_17612035, EPI_ISL_17612050, EPI_ISL_17612051, EPI_ISL_17612052, EPI_ISL_17612398, EPI_ISL_17615127, EPI_ISL_17616259, EPI_ISL_17616354, EPI_ISL_17616355, EPI_ISL_17616356, EPI_ISL_17617168, EPI_ISL_17617538, EPI_ISL_17618306, EPI_ISL_17621115, EPI_ISL_17621930, EPI_ISL_17623470, EPI_ISL_17623785, EPI_ISL_17623810, EPI_ISL_17626289, EPI_ISL_17628376, EPI_ISL_17628383, EPI_ISL_17628855, EPI_ISL_17630096, EPI_ISL_17632950, EPI_ISL_17633442, EPI_ISL_17634290, EPI_ISL_17634585, EPI_ISL_17634799, EPI_ISL_17637409, EPI_ISL_17637499, EPI_ISL_17637946, EPI_ISL_17640029, EPI_ISL_17640079, EPI_ISL_17642112, EPI_ISL_17642765, EPI_ISL_17643093, EPI_ISL_17644186, EPI_ISL_17644989, EPI_ISL_17644991, EPI_ISL_17645081, EPI_ISL_17645416, EPI_ISL_17646422, EPI_ISL_17651803, EPI_ISL_17652508, EPI_ISL_17652513, EPI_ISL_17654325, EPI_ISL_17654831, EPI_ISL_17655018, EPI_ISL_17656002, EPI_ISL_17657287, EPI_ISL_17658392, EPI_ISL_17658574, EPI_ISL_17659247, EPI_ISL_17659794, EPI_ISL_17660857, EPI_ISL_17661435, EPI_ISL_17661709, EPI_ISL_17661736, EPI_ISL_17661772, EPI_ISL_17662111, EPI_ISL_17664370, EPI_ISL_17665676, EPI_ISL_17666708, EPI_ISL_17667360, EPI_ISL_17669304, EPI_ISL_17669306, EPI_ISL_17669441, EPI_ISL_17669457, EPI_ISL_17671157, EPI_ISL_17671162, EPI_ISL_17671689, EPI_ISL_17677128, EPI_ISL_17677325, EPI_ISL_17678395, EPI_ISL_17679253, EPI_ISL_17679612, EPI_ISL_17680172, EPI_ISL_17683135, EPI_ISL_17683747, EPI_ISL_17683879, EPI_ISL_17683882, EPI_ISL_17683902, EPI_ISL_17683926, EPI_ISL_17684194, EPI_ISL_17685960, EPI_ISL_17685982, EPI_ISL_17686409, EPI_ISL_17686485, EPI_ISL_17686694, EPI_ISL_17686736, EPI_ISL_17688072, EPI_ISL_17689247, EPI_ISL_17695348, EPI_ISL_17696086, EPI_ISL_17696551, EPI_ISL_17696575, EPI_ISL_17697616, EPI_ISL_17699149, EPI_ISL_17699879, EPI_ISL_17699884, EPI_ISL_17700051, EPI_ISL_17700270, EPI_ISL_17701083, EPI_ISL_17701278, EPI_ISL_17701782, EPI_ISL_17703815, EPI_ISL_17704713, EPI_ISL_17705564, EPI_ISL_17706013, EPI_ISL_17706030, EPI_ISL_17708288, EPI_ISL_17709926, EPI_ISL_17710268, EPI_ISL_17710278, EPI_ISL_17710307, EPI_ISL_17710673, EPI_ISL_17710974, EPI_ISL_17711012, EPI_ISL_17711646, EPI_ISL_17712964, EPI_ISL_17713423, EPI_ISL_17713709, EPI_ISL_17714880, EPI_ISL_17714882, EPI_ISL_17714902, EPI_ISL_17714948, EPI_ISL_17715122, EPI_ISL_17715974, EPI_ISL_17716296, EPI_ISL_17718358, EPI_ISL_17718497, EPI_ISL_17719162, EPI_ISL_17721620, EPI_ISL_17721941, EPI_ISL_17722060, EPI_ISL_17722142, EPI_ISL_17722884, EPI_ISL_17726746, EPI_ISL_17727194, EPI_ISL_17728144, EPI_ISL_17728250, EPI_ISL_17729454, EPI_ISL_17731387, EPI_ISL_17731388, EPI_ISL_17732098, EPI_ISL_17733269, EPI_ISL_17734236, EPI_ISL_17735972, EPI_ISL_17736284, EPI_ISL_17736388, EPI_ISL_17737562, EPI_ISL_17739108, EPI_ISL_17739543, EPI_ISL_17741957, EPI_ISL_17743681, EPI_ISL_17744022, EPI_ISL_17747309, EPI_ISL_17759354, EPI_ISL_17759925, EPI_ISL_17760279, EPI_ISL_17761215, EPI_ISL_17762387, EPI_ISL_17762760, EPI_ISL_17763721, EPI_ISL_17764011, EPI_ISL_17764066, EPI_ISL_17764072, EPI_ISL_17764496, EPI_ISL_17765796, EPI_ISL_17766060, EPI_ISL_17766100, EPI_ISL_17766112, EPI_ISL_17767434, EPI_ISL_17767435, EPI_ISL_17767436, EPI_ISL_17767437, EPI_ISL_17769030, EPI_ISL_17769081, EPI_ISL_17769169, EPI_ISL_17769170, EPI_ISL_17769216, EPI_ISL_17769229, EPI_ISL_17769310, EPI_ISL_17769888, EPI_ISL_17770725, EPI_ISL_17770729, EPI_ISL_17770732, EPI_ISL_17770736, EPI_ISL_17770779, EPI_ISL_17770815, EPI_ISL_17771047, EPI_ISL_17771051, EPI_ISL_17772012, EPI_ISL_17775344, EPI_ISL_17776736, EPI_ISL_17777061, EPI_ISL_17777067, EPI_ISL_17777729, EPI_ISL_17777748, EPI_ISL_17778593, EPI_ISL_17778602, EPI_ISL_17780724, EPI_ISL_17780726, EPI_ISL_17780860, EPI_ISL_17780886, EPI_ISL_17781122, EPI_ISL_17781585, EPI_ISL_17781712, EPI_ISL_17782148, EPI_ISL_17782366, EPI_ISL_17782502, EPI_ISL_17783358, EPI_ISL_17784545, EPI_ISL_17784546, EPI_ISL_17784547, EPI_ISL_17784552, EPI_ISL_17784558, EPI_ISL_17784569, EPI_ISL_17784585, EPI_ISL_17784593, EPI_ISL_17784775, EPI_ISL_17784803, EPI_ISL_17784804, EPI_ISL_17786165, EPI_ISL_17786546, EPI_ISL_17786769, EPI_ISL_17786827, EPI_ISL_17787009, EPI_ISL_17787597, EPI_ISL_17787864, EPI_ISL_17788384, EPI_ISL_17788516, EPI_ISL_17789385, EPI_ISL_17789475, EPI_ISL_17789808, EPI_ISL_17790033, EPI_ISL_17790116, EPI_ISL_17791306, EPI_ISL_17791796, EPI_ISL_17792172, EPI_ISL_17792191, EPI_ISL_17794816, EPI_ISL_17796500, EPI_ISL_17796537, EPI_ISL_17796598, EPI_ISL_17796704, EPI_ISL_17797704, EPI_ISL_17798165, EPI_ISL_17799068, EPI_ISL_17799108, EPI_ISL_17802597, EPI_ISL_17803325, EPI_ISL_17803653, EPI_ISL_17803784, EPI_ISL_17806504, EPI_ISL_17806524, EPI_ISL_17809334, EPI_ISL_17809574, EPI_ISL_17810512, EPI_ISL_17812915, EPI_ISL_17813049, EPI_ISL_17813537, EPI_ISL_17813637, EPI_ISL_17813862, EPI_ISL_17815222, EPI_ISL_17816174, EPI_ISL_17817657, EPI_ISL_17817985, EPI_ISL_17818039, EPI_ISL_17819921, EPI_ISL_17820257, EPI_ISL_17820258, EPI_ISL_17820602, EPI_ISL_17821850, EPI_ISL_17823538, EPI_ISL_17824292, EPI_ISL_17824608, EPI_ISL_17824611, EPI_ISL_17824670, EPI_ISL_17826285, EPI_ISL_17830134, EPI_ISL_17830169, EPI_ISL_17830573, EPI_ISL_17830591, EPI_ISL_17830762, EPI_ISL_17831005, EPI_ISL_17831639, EPI_ISL_17831941, EPI_ISL_17833161, EPI_ISL_17833549, EPI_ISL_17837092, EPI_ISL_17837097, EPI_ISL_17837134, EPI_ISL_17837135, EPI_ISL_17837188, EPI_ISL_17837432, EPI_ISL_17837459, EPI_ISL_17837460, EPI_ISL_17837914, EPI_ISL_17837915, EPI_ISL_17838109, EPI_ISL_17838506, EPI_ISL_17839137, EPI_ISL_17850070, EPI_ISL_17850078, EPI_ISL_17851276, EPI_ISL_17853355, EPI_ISL_17853579, EPI_ISL_17855226, EPI_ISL_17856975, EPI_ISL_17857949, EPI_ISL_17857950, EPI_ISL_17859477, EPI_ISL_17859958, EPI_ISL_17860390, EPI_ISL_17860984, EPI_ISL_17862677, EPI_ISL_17871595, EPI_ISL_17879222, EPI_ISL_17884376, EPI_ISL_17884518, EPI_ISL_17885064, EPI_ISL_17885128, EPI_ISL_17885331, EPI_ISL_17885459, EPI_ISL_17891004, EPI_ISL_17899627, EPI_ISL_17949029, EPI_ISL_17949339, EPI_ISL_17949978, EPI_ISL_17950840, EPI_ISL_17952015, EPI_ISL_17952019, EPI_ISL_17953343, EPI_ISL_17953610, EPI_ISL_17954106, EPI_ISL_17954662, EPI_ISL_17954669, EPI_ISL_17954940, EPI_ISL_17956164, EPI_ISL_17958015, EPI_ISL_17959424, EPI_ISL_17960600, EPI_ISL_17960747, EPI_ISL_17960749, EPI_ISL_17961252, EPI_ISL_17961253, EPI_ISL_17964403, EPI_ISL_17964415, EPI_ISL_17964828, EPI_ISL_17965635, EPI_ISL_17965636, EPI_ISL_17966200, EPI_ISL_17966205, EPI_ISL_17968777, EPI_ISL_17968962, EPI_ISL_17969108, EPI_ISL_17971223, EPI_ISL_17971936, EPI_ISL_17972242, EPI_ISL_17972372, EPI_ISL_17973367, EPI_ISL_17974574, EPI_ISL_17974688, EPI_ISL_17974927, EPI_ISL_17974952, EPI_ISL_17974980, EPI_ISL_17975003, EPI_ISL_17975174, EPI_ISL_17976105, EPI_ISL_17976113, EPI_ISL_17976114, EPI_ISL_17976116, EPI_ISL_17976531, EPI_ISL_17977985, EPI_ISL_17978344, EPI_ISL_17978693, EPI_ISL_17978839, EPI_ISL_17978964, EPI_ISL_17979017, EPI_ISL_17979965, EPI_ISL_17979979, EPI_ISL_17979981, EPI_ISL_17980588, EPI_ISL_17982411, EPI_ISL_17982453, EPI_ISL_17982543, EPI_ISL_17985757, EPI_ISL_17988396, EPI_ISL_17989190, EPI_ISL_17989433, EPI_ISL_17989516, EPI_ISL_17989740, EPI_ISL_17989749, EPI_ISL_17989792, EPI_ISL_17989829, EPI_ISL_17989858, EPI_ISL_17989860, EPI_ISL_17990203, EPI_ISL_17990304, EPI_ISL_17993966, EPI_ISL_17994784, EPI_ISL_17994786, EPI_ISL_17995488, EPI_ISL_17995513, EPI_ISL_17995955, EPI_ISL_17996897, EPI_ISL_17997249, EPI_ISL_17997251, EPI_ISL_17997917, EPI_ISL_17997982, EPI_ISL_17998406, EPI_ISL_18000155, EPI_ISL_18000245, EPI_ISL_18000414, EPI_ISL_18000654, EPI_ISL_18000825, EPI_ISL_18001789, EPI_ISL_18001862, EPI_ISL_18008246, EPI_ISL_18008262, EPI_ISL_18008673, EPI_ISL_18009591, EPI_ISL_18009602, EPI_ISL_18010720, EPI_ISL_18011449, EPI_ISL_18011518, EPI_ISL_18012526, EPI_ISL_18012547, EPI_ISL_18012806, EPI_ISL_18014237, EPI_ISL_18014700, EPI_ISL_18015682, EPI_ISL_18016999, EPI_ISL_18019246, EPI_ISL_18028785, EPI_ISL_18029979, EPI_ISL_18030390, EPI_ISL_18030391, EPI_ISL_18030395, EPI_ISL_18031842, EPI_ISL_18032297, EPI_ISL_18032322, EPI_ISL_18032338, EPI_ISL_18033013, EPI_ISL_18033516, EPI_ISL_18033631, EPI_ISL_18034079, EPI_ISL_18034109, EPI_ISL_18037119, EPI_ISL_18037474, EPI_ISL_18037476, EPI_ISL_18037744, EPI_ISL_18038269, EPI_ISL_18039728, EPI_ISL_18040070, EPI_ISL_18041130, EPI_ISL_18041968, EPI_ISL_18042110, EPI_ISL_18044024, EPI_ISL_18044164, EPI_ISL_18044400, EPI_ISL_18044754, EPI_ISL_18044755, EPI_ISL_18044759, EPI_ISL_18046466, EPI_ISL_18048708, EPI_ISL_18048972, EPI_ISL_18048978, EPI_ISL_18049009, EPI_ISL_18049160, EPI_ISL_18049161, EPI_ISL_18049174, EPI_ISL_18049809, EPI_ISL_18049917, EPI_ISL_18050065, EPI_ISL_18050520, EPI_ISL_18050523, EPI_ISL_18051914, EPI_ISL_18051918, EPI_ISL_18052440, EPI_ISL_18052776, EPI_ISL_18052929, EPI_ISL_18053022, EPI_ISL_18053315, EPI_ISL_18054466, EPI_ISL_18054899, EPI_ISL_18056643, EPI_ISL_18056644, EPI_ISL_18056759, EPI_ISL_18056769, EPI_ISL_18058525, EPI_ISL_18058567, EPI_ISL_18058881, EPI_ISL_18059074, EPI_ISL_18059075, EPI_ISL_18059076, EPI_ISL_18059726, EPI_ISL_18060973, EPI_ISL_18062475, EPI_ISL_18064366, EPI_ISL_18064383, EPI_ISL_18064405, EPI_ISL_18064413, EPI_ISL_18064431, EPI_ISL_18064456, EPI_ISL_18064519, EPI_ISL_18070310, EPI_ISL_18071883, EPI_ISL_18071901, EPI_ISL_18072068, EPI_ISL_18072343, EPI_ISL_18073924, EPI_ISL_18074072, EPI_ISL_18075985, EPI_ISL_18076065, EPI_ISL_18076069, EPI_ISL_18076165, EPI_ISL_18076251, EPI_ISL_18076473, EPI_ISL_18077275, EPI_ISL_18078878, EPI_ISL_18079417, EPI_ISL_18080566, EPI_ISL_18083488, EPI_ISL_18091808, EPI_ISL_18092412, EPI_ISL_18092435, EPI_ISL_18093840, EPI_ISL_18094397, EPI_ISL_18094429, EPI_ISL_18094476, EPI_ISL_18094560, EPI_ISL_18094842, EPI_ISL_18095157, EPI_ISL_18095961, EPI_ISL_18097327, EPI_ISL_18097349, EPI_ISL_18097786, EPI_ISL_18098270, EPI_ISL_18098276, EPI_ISL_18098299, EPI_ISL_18098479, EPI_ISL_18098976, EPI_ISL_18099952, EPI_ISL_18100455, EPI_ISL_18100457, EPI_ISL_18100607, EPI_ISL_18104072, EPI_ISL_18106416, EPI_ISL_18106460, EPI_ISL_18106464, EPI_ISL_18106662, EPI_ISL_18106788, EPI_ISL_18106910, EPI_ISL_18106912, EPI_ISL_18106920, EPI_ISL_18106929, EPI_ISL_18106930, EPI_ISL_18106931, EPI_ISL_18106933, EPI_ISL_18106934, EPI_ISL_18106950, EPI_ISL_18106951, EPI_ISL_18107900, EPI_ISL_18109285, EPI_ISL_18110014, EPI_ISL_18110496, EPI_ISL_18110776, EPI_ISL_18111020, EPI_ISL_18111021, EPI_ISL_18111040, EPI_ISL_18111041, EPI_ISL_18111086, EPI_ISL_18112015, EPI_ISL_18115442, EPI_ISL_18115451, EPI_ISL_18115956, EPI_ISL_18116015, EPI_ISL_18116176, EPI_ISL_18118264, EPI_ISL_18118289, EPI_ISL_18118388, EPI_ISL_18118556, EPI_ISL_18118855, EPI_ISL_18119265, EPI_ISL_18120201, EPI_ISL_18123396, EPI_ISL_18124840, EPI_ISL_18125049, EPI_ISL_18125050, EPI_ISL_18126021, EPI_ISL_18126834, EPI_ISL_18127203, EPI_ISL_18127526, EPI_ISL_18127527, EPI_ISL_18127685, EPI_ISL_18127834, EPI_ISL_18128931, EPI_ISL_18129019, EPI_ISL_18129038, EPI_ISL_18129213, EPI_ISL_18129656, EPI_ISL_18129944, EPI_ISL_18131053, EPI_ISL_18131109, EPI_ISL_18134315, EPI_ISL_18134392, EPI_ISL_18134395, EPI_ISL_18134442, EPI_ISL_18134610, EPI_ISL_18134691, EPI_ISL_18134700, EPI_ISL_18134706, EPI_ISL_18134984, EPI_ISL_18135040, EPI_ISL_18135429, EPI_ISL_18136392, EPI_ISL_18136968, EPI_ISL_18139400, EPI_ISL_18139409, EPI_ISL_18141686, EPI_ISL_18141739, EPI_ISL_18141844, EPI_ISL_18142202, EPI_ISL_18142317, EPI_ISL_18142357, EPI_ISL_18142525, EPI_ISL_18142978, EPI_ISL_18142994, EPI_ISL_18146885, EPI_ISL_18147456, EPI_ISL_18147966, EPI_ISL_18151975, EPI_ISL_18151976, EPI_ISL_18151977, EPI_ISL_18153000, EPI_ISL_18154903, EPI_ISL_18159587, EPI_ISL_18160510, EPI_ISL_18160518, EPI_ISL_18160519, EPI_ISL_18160530, EPI_ISL_18160538, EPI_ISL_18162567, EPI_ISL_18163680, EPI_ISL_18163890, EPI_ISL_18163891, EPI_ISL_18163892, EPI_ISL_18163893, EPI_ISL_18163894, EPI_ISL_18163895, EPI_ISL_18163896, EPI_ISL_18163897, EPI_ISL_18163898:, EPI_ISL_18164441, EPI_ISL_18166642, EPI_ISL_18166643, EPI_ISL_18168780, EPI_ISL_18169117, EPI_ISL_18205057, EPI_ISL_18207613, EPI_ISL_18210510, EPI_ISL_18212559, EPI_ISL_18213104, EPI_ISL_18215123, EPI_ISL_18215226, EPI_ISL_18215482, EPI_ISL_18215552, EPI_ISL_18217564, EPI_ISL_18217995, EPI_ISL_18218776, EPI_ISL_18219915, EPI_ISL_18219916, EPI_ISL_18219931, EPI_ISL_18219970, EPI_ISL_18220073, EPI_ISL_18220498, EPI_ISL_18221521, EPI_ISL_18221524, EPI_ISL_18221527, EPI_ISL_18221982, EPI_ISL_18221985, EPI_ISL_18222367, EPI_ISL_18224410, EPI_ISL_18224514, EPI_ISL_18225473, EPI_ISL_18227366, EPI_ISL_18227596, EPI_ISL_18227611, EPI_ISL_18227624, EPI_ISL_18227629, EPI_ISL_18228307, EPI_ISL_18232124, EPI_ISL_18233906, EPI_ISL_18234431, EPI_ISL_18234763, EPI_ISL_18236180, EPI_ISL_18236291, EPI_ISL_18238117, EPI_ISL_18241087, EPI_ISL_18241705, EPI_ISL_18241707, EPI_ISL_18241719, EPI_ISL_18245571, EPI_ISL_18247259, EPI_ISL_18248695, EPI_ISL_18249682, EPI_ISL_18253248, EPI_ISL_18253249, EPI_ISL_18255994, EPI_ISL_18256173, EPI_ISL_18256714, EPI_ISL_18256980, EPI_ISL_18258766, EPI_ISL_18259784, EPI_ISL_18260202, EPI_ISL_18263919, EPI_ISL_18263945, EPI_ISL_18263946, EPI_ISL_18263981, EPI_ISL_18269234, EPI_ISL_18271265, EPI_ISL_18273982, EPI_ISL_18274346, EPI_ISL_18276415, EPI_ISL_18277439, EPI_ISL_18277736, EPI_ISL_18278627, EPI_ISL_18278909, EPI_ISL_18279614, EPI_ISL_18281186, EPI_ISL_18281259, EPI_ISL_18281287, EPI_ISL_18281288, EPI_ISL_18281494, EPI_ISL_18281574, EPI_ISL_18282077, EPI_ISL_18282082, EPI_ISL_18286773, EPI_ISL_18287351, EPI_ISL_18289877, EPI_ISL_18290890, EPI_ISL_18290989, EPI_ISL_18291808, EPI_ISL_18292038, EPI_ISL_18292398, EPI_ISL_18294574, EPI_ISL_18295441, EPI_ISL_18295450, EPI_ISL_18298019, EPI_ISL_18299948, EPI_ISL_18300587, EPI_ISL_18301587, EPI_ISL_18302636, EPI_ISL_18303012, EPI_ISL_18303206, EPI_ISL_18303444, EPI_ISL_18303592, EPI_ISL_18303595, EPI_ISL_18303758, EPI_ISL_18306254, EPI_ISL_18306922, EPI_ISL_18308642, EPI_ISL_18311951, EPI_ISL_18313683, EPI_ISL_18315747, EPI_ISL_18315789, EPI_ISL_18319306, EPI_ISL_18319903, EPI_ISL_18319904, EPI_ISL_18319906, EPI_ISL_18319907, EPI_ISL_18320079, EPI_ISL_18321271, EPI_ISL_18322273, EPI_ISL_18322420, EPI_ISL_18322438, EPI_ISL_18323536, EPI_ISL_18324107, EPI_ISL_18324168, EPI_ISL_18324491, EPI_ISL_18325145, EPI_ISL_18325563, EPI_ISL_18326430, EPI_ISL_18326597, EPI_ISL_18326806, EPI_ISL_18326807, EPI_ISL_18330957, EPI_ISL_18330966, EPI_ISL_18331347, EPI_ISL_18334903, EPI_ISL_18334945, EPI_ISL_18334986, EPI_ISL_18335084, EPI_ISL_18335526, EPI_ISL_18336165, EPI_ISL_18336602, EPI_ISL_18336862, EPI_ISL_18337738, EPI_ISL_18338137, EPI_ISL_18338143, EPI_ISL_18338144, EPI_ISL_18338502, EPI_ISL_18338504, EPI_ISL_18338709, EPI_ISL_18342412, EPI_ISL_18343598, EPI_ISL_18345777, EPI_ISL_18345926, EPI_ISL_18346109, EPI_ISL_18351588, EPI_ISL_18352473, EPI_ISL_18352485, EPI_ISL_18352489, EPI_ISL_18359229, EPI_ISL_18359328, EPI_ISL_18359679, EPI_ISL_18360507, EPI_ISL_18360944, EPI_ISL_18361202, EPI_ISL_18362265, EPI_ISL_18362515, EPI_ISL_18363170, EPI_ISL_18363300, EPI_ISL_18365170, EPI_ISL_18365256, EPI_ISL_18367086, EPI_ISL_18367563, EPI_ISL_18367586, EPI_ISL_18367599, EPI_ISL_18367615, EPI_ISL_18367908, EPI_ISL_18367992, EPI_ISL_18370898, EPI_ISL_18370960, EPI_ISL_18370967, EPI_ISL_18371749, EPI_ISL_18373201, EPI_ISL_18373754, EPI_ISL_18377021, EPI_ISL_18377214, EPI_ISL_18377245, EPI_ISL_18377248, EPI_ISL_18378384, EPI_ISL_18380731, EPI_ISL_18381066, EPI_ISL_18383121, EPI_ISL_18383423, EPI_ISL_18384846, EPI_ISL_18384936, EPI_ISL_18385358, EPI_ISL_18385924, EPI_ISL_18386091, EPI_ISL_18386403, EPI_ISL_18387037, EPI_ISL_18388509, EPI_ISL_18388585, EPI_ISL_18389783, EPI_ISL_18389797, EPI_ISL_18391451, EPI_ISL_18391597, EPI_ISL_18392259, EPI_ISL_18392502, EPI_ISL_18392841, EPI_ISL_18393366, EPI_ISL_18395551, EPI_ISL_18398210, EPI_ISL_18398259, EPI_ISL_18400843, EPI_ISL_18400856, EPI_ISL_18400946, EPI_ISL_18400976, EPI_ISL_18400987, EPI_ISL_18401313, EPI_ISL_18403047, EPI_ISL_18403051, EPI_ISL_18403054, EPI_ISL_18403509, EPI_ISL_18403523, EPI_ISL_18404585, EPI_ISL_18405535, EPI_ISL_18405621, EPI_ISL_18406394, EPI_ISL_18408561, EPI_ISL_18410987, EPI_ISL_18414567, EPI_ISL_18414568, EPI_ISL_18414808, EPI_ISL_18415823, EPI_ISL_18415832, EPI_ISL_18415834, EPI_ISL_18415840, EPI_ISL_18415854, EPI_ISL_18416870, EPI_ISL_18417129, EPI_ISL_18417211, EPI_ISL_18419485, EPI_ISL_18419748, EPI_ISL_18421674, EPI_ISL_18422693, EPI_ISL_18422715, EPI_ISL_18422771, EPI_ISL_18423785, EPI_ISL_18423814, EPI_ISL_18423907, EPI_ISL_18424281, EPI_ISL_18424468, EPI_ISL_18426836, EPI_ISL_18428844, EPI_ISL_18429684, EPI_ISL_18429702, EPI_ISL_18429725, EPI_ISL_18429773, EPI_ISL_18429797, EPI_ISL_18432077, EPI_ISL_18433350, EPI_ISL_18434194, EPI_ISL_18435557, EPI_ISL_18435892, EPI_ISL_18435949, EPI_ISL_18437342, EPI_ISL_18438723, EPI_ISL_18439733, EPI_ISL_18440037, EPI_ISL_18440370, EPI_ISL_18440660, EPI_ISL_18440866, EPI_ISL_18441868, EPI_ISL_18443784, EPI_ISL_18443944, EPI_ISL_18446092, EPI_ISL_18448894, EPI_ISL_18449647, EPI_ISL_18449794, EPI_ISL_18449820, EPI_ISL_18449892, EPI_ISL_18450249, EPI_ISL_18450812, EPI_ISL_18451678, EPI_ISL_18453400, EPI_ISL_18455292, EPI_ISL_18455564, EPI_ISL_18455706, EPI_ISL_18455950, EPI_ISL_18457808, EPI_ISL_18459512, EPI_ISL_18461774, EPI_ISL_18462852, EPI_ISL_18463490, EPI_ISL_18463766, EPI_ISL_18466251, EPI_ISL_18468149, EPI_ISL_18470400, EPI_ISL_18472311, EPI_ISL_18472312, EPI_ISL_18473559, EPI_ISL_18474126, EPI_ISL_18474555, EPI_ISL_18474665, EPI_ISL_18475072, EPI_ISL_18475534, EPI_ISL_18475535, EPI_ISL_18480741, EPI_ISL_18486919, EPI_ISL_18487225, EPI_ISL_18489646, EPI_ISL_18489793, EPI_ISL_18489829, EPI_ISL_18491841, EPI_ISL_18491844, EPI_ISL_18492277, EPI_ISL_18492305, EPI_ISL_18492307, EPI_ISL_18492412, EPI_ISL_18492455, EPI_ISL_18493129, EPI_ISL_18495416, EPI_ISL_18496252, EPI_ISL_18496585, EPI_ISL_18498001, EPI_ISL_18498420, EPI_ISL_18498499, EPI_ISL_18500316, EPI_ISL_18500771, EPI_ISL_18501087, EPI_ISL_18503287, EPI_ISL_18509817, EPI_ISL_18512418, EPI_ISL_18512421, EPI_ISL_18512438, EPI_ISL_18513936, EPI_ISL_18514552, EPI_ISL_18515280, EPI_ISL_18515316, EPI_ISL_18515328, EPI_ISL_18515343, EPI_ISL_18515511, EPI_ISL_18515749, EPI_ISL_18516916, EPI_ISL_18518769, EPI_ISL_18518932, EPI_ISL_18519113, EPI_ISL_18520677, EPI_ISL_18520678, EPI_ISL_18521575, EPI_ISL_18521765, EPI_ISL_18522020, EPI_ISL_18522184, EPI_ISL_18522580, EPI_ISL_18523129, EPI_ISL_18524926, EPI_ISL_18525067, EPI_ISL_18525840, EPI_ISL_18526641, EPI_ISL_18528453, EPI_ISL_18529555, EPI_ISL_18530445, EPI_ISL_18530449, EPI_ISL_18536853, EPI_ISL_18537013, EPI_ISL_18537032, EPI_ISL_18537373, EPI_ISL_18537428, EPI_ISL_18537814, EPI_ISL_18538015, EPI_ISL_18543268, EPI_ISL_18543705, EPI_ISL_18546112, EPI_ISL_18546287, EPI_ISL_18546551, EPI_ISL_18546715, EPI_ISL_18551440, EPI_ISL_18552697, EPI_ISL_18553587, EPI_ISL_18554053, EPI_ISL_18556084, EPI_ISL_18556539, EPI_ISL_18557145, EPI_ISL_18558360, EPI_ISL_18558385, EPI_ISL_18558412, EPI_ISL_18558468, EPI_ISL_18558477, EPI_ISL_18559317, EPI_ISL_18560556, EPI_ISL_18560725, EPI_ISL_18560872, EPI_ISL_18561098, EPI_ISL_18563181, EPI_ISL_18563821, EPI_ISL_18564403, EPI_ISL_18564755, EPI_ISL_18566696, EPI_ISL_18567985, EPI_ISL_18568124, EPI_ISL_18576266, EPI_ISL_18576754, EPI_ISL_18577842, EPI_ISL_18577862, EPI_ISL_18577966, EPI_ISL_18578195, EPI_ISL_18579981, EPI_ISL_18580011, EPI_ISL_18580750, EPI_ISL_18580874, EPI_ISL_18581347, EPI_ISL_18584141, EPI_ISL_18588773, EPI_ISL_18589012, EPI_ISL_18589243, EPI_ISL_18589475, EPI_ISL_18589669, EPI_ISL_18590820, EPI_ISL_18591717, EPI_ISL_18593579, EPI_ISL_18594183, EPI_ISL_18594233, EPI_ISL_18594266, EPI_ISL_18595212, EPI_ISL_18598503, EPI_ISL_18598525, EPI_ISL_18598964, EPI_ISL_18603922, EPI_ISL_18604501, EPI_ISL_18604502, EPI_ISL_18604503, EPI_ISL_18604504, EPI_ISL_18604505, EPI_ISL_18606460, EPI_ISL_18607149, EPI_ISL_18607150, EPI_ISL_18608694, EPI_ISL_18609973, EPI_ISL_18612246, EPI_ISL_18615968, EPI_ISL_18622139, EPI_ISL_18624843, EPI_ISL_18625316, EPI_ISL_18626713, EPI_ISL_18626714, EPI_ISL_18626750, EPI_ISL_18630930, EPI_ISL_18633829, EPI_ISL_18634703, EPI_ISL_18635526, EPI_ISL_18635546, EPI_ISL_18635599, EPI_ISL_18635961, EPI_ISL_18640058, EPI_ISL_18641470, EPI_ISL_18641499, EPI_ISL_18642608, EPI_ISL_18646912, EPI_ISL_18646945, EPI_ISL_18648209, EPI_ISL_18652556, EPI_ISL_18653898, EPI_ISL_18654501, EPI_ISL_18672102, EPI_ISL_18674382, EPI_ISL_18677704, EPI_ISL_18681665, EPI_ISL_18681942, EPI_ISL_18681975, EPI_ISL_18683430, EPI_ISL_18687895, EPI_ISL_18689964, EPI_ISL_18693860, EPI_ISL_18694187, EPI_ISL_18702737, EPI_ISL_18712665, EPI_ISL_18712806, EPI_ISL_18712994, EPI_ISL_18713198, EPI_ISL_18714360, EPI_ISL_18714857, EPI_ISL_18715065, EPI_ISL_18715624, EPI_ISL_18717261, EPI_ISL_18727247, EPI_ISL_18727294, EPI_ISL_18729637, EPI_ISL_18730563, EPI_ISL_18734528, EPI_ISL_18740049, EPI_ISL_18742836, EPI_ISL_18743266, EPI_ISL_18743442, EPI_ISL_18743504, EPI_ISL_18743721, EPI_ISL_18758247, EPI_ISL_18758251, EPI_ISL_18759772, EPI_ISL_18760241, EPI_ISL_18763780, EPI_ISL_18766752, EPI_ISL_18772294, EPI_ISL_18777179, EPI_ISL_18778198, EPI_ISL_18778941, EPI_ISL_18782000, EPI_ISL_18782699, EPI_ISL_18784365, EPI_ISL_18784385, EPI_ISL_18784404, EPI_ISL_18784438, EPI_ISL_18787342, EPI_ISL_18787343, EPI_ISL_18790918, EPI_ISL_18792770, EPI_ISL_18796930, EPI_ISL_18797833, EPI_ISL_18798193, EPI_ISL_18798198, EPI_ISL_18798202, EPI_ISL_18798204, EPI_ISL_18798234, EPI_ISL_18799019, EPI_ISL_18801405, EPI_ISL_18803784, EPI_ISL_18805014, EPI_ISL_18809509, EPI_ISL_18809652, EPI_ISL_18811047, EPI_ISL_18811325, EPI_ISL_18811375, EPI_ISL_18811389, EPI_ISL_18814271, EPI_ISL_18814272, EPI_ISL_18814273, EPI_ISL_18815556, EPI_ISL_18816278, EPI_ISL_18816628, EPI_ISL_18817415, EPI_ISL_18818949, EPI_ISL_18820996, EPI_ISL_18824608, EPI_ISL_18825438, EPI_ISL_18828912, EPI_ISL_18829708, EPI_ISL_18832540, EPI_ISL_18832845, EPI_ISL_18839074, EPI_ISL_18839275, EPI_ISL_18839824, EPI_ISL_18839850, EPI_ISL_18839932, EPI_ISL_18841696, EPI_ISL_18842709, EPI_ISL_18846230, EPI_ISL_18851473, EPI_ISL_18853389, EPI_ISL_18853925, EPI_ISL_18854197, EPI_ISL_18854790, EPI_ISL_18854974, EPI_ISL_18856190, EPI_ISL_18856328, EPI_ISL_18859868, EPI_ISL_18860821, EPI_ISL_18863734, EPI_ISL_18864357, EPI_ISL_18864823, EPI_ISL_18865364, EPI_ISL_18865427, EPI_ISL_18868940, EPI_ISL_18869806, EPI_ISL_18870297, EPI_ISL_18872609, EPI_ISL_18873437, EPI_ISL_18873899, EPI_ISL_18873907, EPI_ISL_18874096, EPI_ISL_18874749, EPI_ISL_18875008, EPI_ISL_18876158, EPI_ISL_18876159, EPI_ISL_18876160, EPI_ISL_18876161, EPI_ISL_18876162, EPI_ISL_18876164, EPI_ISL_18876166, EPI_ISL_18876167, EPI_ISL_18876171, EPI_ISL_18876172, EPI_ISL_18876174, EPI_ISL_18876176, EPI_ISL_18876178, EPI_ISL_18876356, EPI_ISL_18877288, EPI_ISL_18877358, EPI_ISL_18877379, EPI_ISL_18877954, EPI_ISL_18878665, EPI_ISL_18880075, EPI_ISL_18882888, EPI_ISL_18884114, EPI_ISL_18885028, EPI_ISL_18885545, EPI_ISL_18886798, EPI_ISL_18888145, EPI_ISL_18888340, EPI_ISL_18889879, EPI_ISL_18892766, EPI_ISL_18896020, EPI_ISL_18897180, EPI_ISL_18897181, EPI_ISL_18899656, EPI_ISL_18899840, EPI_ISL_18900159, EPI_ISL_18901224, EPI_ISL_18901411, EPI_ISL_18901513, EPI_ISL_18901762, EPI_ISL_18901997, EPI_ISL_18904234, EPI_ISL_18906411, EPI_ISL_18906577, EPI_ISL_18908924, EPI_ISL_18914858, EPI_ISL_18915560, EPI_ISL_18916107, EPI_ISL_18916798, EPI_ISL_18916914, EPI_ISL_18917364, EPI_ISL_18917482, EPI_ISL_18917496, EPI_ISL_18917541, EPI_ISL_18918931, EPI_ISL_18918932, EPI_ISL_18918944, EPI_ISL_18919348, EPI_ISL_18919477, EPI_ISL_18919544, EPI_ISL_18924428, EPI_ISL_18927022, EPI_ISL_18927251, EPI_ISL_18927258, EPI_ISL_18927292, EPI_ISL_18927500, EPI_ISL_18927514, EPI_ISL_18927587, EPI_ISL_18930149, EPI_ISL_18930177, EPI_ISL_18930182, EPI_ISL_18930190, EPI_ISL_18930681, EPI_ISL_18931503, EPI_ISL_18931504, EPI_ISL_18931535, EPI_ISL_18931538, EPI_ISL_18931542, EPI_ISL_18932507, EPI_ISL_18933211, EPI_ISL_18933315, EPI_ISL_18933410, EPI_ISL_18933606, EPI_ISL_18933647, EPI_ISL_18935609, EPI_ISL_18935631, EPI_ISL_18936396, EPI_ISL_18939566, EPI_ISL_18939949, EPI_ISL_18943203, EPI_ISL_18945392, EPI_ISL_18945748, EPI_ISL_18947086, EPI_ISL_18947189, EPI_ISL_18947944, EPI_ISL_18948077, EPI_ISL_18948078, EPI_ISL_18948422, EPI_ISL_18948613, EPI_ISL_18948756, EPI_ISL_18948757, EPI_ISL_18949161, EPI_ISL_18949911, EPI_ISL_18952873, EPI_ISL_18953132, EPI_ISL_18953134, EPI_ISL_18953234, EPI_ISL_18953461, EPI_ISL_18953493, EPI_ISL_18953503, EPI_ISL_18953522, EPI_ISL_18955186, EPI_ISL_18956070, EPI_ISL_18956087, EPI_ISL_18956161, EPI_ISL_18958247, EPI_ISL_18958295, EPI_ISL_18958383, EPI_ISL_18960149, EPI_ISL_18960183, EPI_ISL_18960224, EPI_ISL_18962896, EPI_ISL_18962928, EPI_ISL_18963535, EPI_ISL_18963652, EPI_ISL_18965082, EPI_ISL_18965451, EPI_ISL_18966035, EPI_ISL_18966186, EPI_ISL_18968116, EPI_ISL_18968120, EPI_ISL_18969119, EPI_ISL_18969697, EPI_ISL_18969735, EPI_ISL_18970692, EPI_ISL_18971336, EPI_ISL_18972187, EPI_ISL_18972699, EPI_ISL_18972708, EPI_ISL_18972719, EPI_ISL_18972725, EPI_ISL_18972734, EPI_ISL_18972995, EPI_ISL_18975064, EPI_ISL_18975148, EPI_ISL_18975189, EPI_ISL_18975333, EPI_ISL_18977928, EPI_ISL_18977929, EPI_ISL_18979948, EPI_ISL_18980164, EPI_ISL_18981298, EPI_ISL_18982763, EPI_ISL_18986641, EPI_ISL_18986642, EPI_ISL_18986747, EPI_ISL_18987173, EPI_ISL_18987258, EPI_ISL_18987259, EPI_ISL_18987545, EPI_ISL_18990023, EPI_ISL_18993521, EPI_ISL_18998415, EPI_ISL_18998583, EPI_ISL_18999516, EPI_ISL_19000455, EPI_ISL_19002243, EPI_ISL_19002832, EPI_ISL_19003837, EPI_ISL_19004571, EPI_ISL_19005468, EPI_ISL_19005470, EPI_ISL_19005472, EPI_ISL_19005481, EPI_ISL_19006725, EPI_ISL_19008875, EPI_ISL_19009123, EPI_ISL_19009131, EPI_ISL_19010135, EPI_ISL_19012038, EPI_ISL_19012114, EPI_ISL_19012460, EPI_ISL_19012472, EPI_ISL_19012554, EPI_ISL_19012663, EPI_ISL_19013418, EPI_ISL_19014517, EPI_ISL_19014601, EPI_ISL_19015115, EPI_ISL_19015882, EPI_ISL_19016044, EPI_ISL_19016053, EPI_ISL_19016370, EPI_ISL_19017499, EPI_ISL_19019122, EPI_ISL_19019127, EPI_ISL_19019128, EPI_ISL_19019136, EPI_ISL_19019141, EPI_ISL_19021045, EPI_ISL_19021046, EPI_ISL_19022071, EPI_ISL_19022479, EPI_ISL_19024260, EPI_ISL_19026222, EPI_ISL_19026577, EPI_ISL_19026932, EPI_ISL_19029499, EPI_ISL_19030027, EPI_ISL_19030112, EPI_ISL_19030117, EPI_ISL_19030129, EPI_ISL_19030183, EPI_ISL_19031583, EPI_ISL_19032054, EPI_ISL_19032056, EPI_ISL_19032179, EPI_ISL_19032598, EPI_ISL_19033249, EPI_ISL_19035801, EPI_ISL_19036115, EPI_ISL_19036116, EPI_ISL_19036117, EPI_ISL_19036644, EPI_ISL_19036878, EPI_ISL_19036921, EPI_ISL_19041379, EPI_ISL_19041967, EPI_ISL_19042675, EPI_ISL_19043835, EPI_ISL_19043836, EPI_ISL_19044085, EPI_ISL_19044119, EPI_ISL_19044163, EPI_ISL_19044218, EPI_ISL_19044280, EPI_ISL_19044474, EPI_ISL_19044644, EPI_ISL_19044737, EPI_ISL_19046891, EPI_ISL_19046892, EPI_ISL_19047408, EPI_ISL_19049407, EPI_ISL_19049586, EPI_ISL_19050509, EPI_ISL_19051602, EPI_ISL_19051976, EPI_ISL_19052024, EPI_ISL_19052034, EPI_ISL_19052090, EPI_ISL_19052102, EPI_ISL_19052117, EPI_ISL_19053803, EPI_ISL_19053806, EPI_ISL_19053808, EPI_ISL_19053809, EPI_ISL_19054784, EPI_ISL_19055364, EPI_ISL_19055368, EPI_ISL_19055382, EPI_ISL_19055909, EPI_ISL_19058120, EPI_ISL_19060002, EPI_ISL_19060327, EPI_ISL_19060922, EPI_ISL_19061139, EPI_ISL_19061646, EPI_ISL_19062162, EPI_ISL_19062523, EPI_ISL_19062524, EPI_ISL_19064175, EPI_ISL_19065060, EPI_ISL_19065882, EPI_ISL_19066171, EPI_ISL_19066817, EPI_ISL_19067782, EPI_ISL_19067788, EPI_ISL_19070471, EPI_ISL_19071618, EPI_ISL_19073169, EPI_ISL_19073731, EPI_ISL_19073732, EPI_ISL_19073800, EPI_ISL_19073807, EPI_ISL_19074673, EPI_ISL_19075261, EPI_ISL_19075267, EPI_ISL_19075350, EPI_ISL_19076088, EPI_ISL_19081415, EPI_ISL_19081416, EPI_ISL_19081419, EPI_ISL_19081421, EPI_ISL_19081422, EPI_ISL_19081425, EPI_ISL_19082080, EPI_ISL_19082092, EPI_ISL_19082200, EPI_ISL_19082275, EPI_ISL_19082487, EPI_ISL_19085083, EPI_ISL_19085329, EPI_ISL_19085381, EPI_ISL_19085539, EPI_ISL_19085540, EPI_ISL_19085542, EPI_ISL_19086378, EPI_ISL_19086393, EPI_ISL_19086523, EPI_ISL_19091018, EPI_ISL_19091019, EPI_ISL_19091036, EPI_ISL_19091144, EPI_ISL_19094153, EPI_ISL_19094369, EPI_ISL_19095154, EPI_ISL_19095156, EPI_ISL_19095501, EPI_ISL_19095606, EPI_ISL_19095734, EPI_ISL_19095768, EPI_ISL_19100255, EPI_ISL_19100981, EPI_ISL_19105055, EPI_ISL_19106368, EPI_ISL_19106842, EPI_ISL_19106844, EPI_ISL_19106999, EPI_ISL_19108198, EPI_ISL_19108199, EPI_ISL_19108200, EPI_ISL_19108208, EPI_ISL_19108340, EPI_ISL_19108650, EPI_ISL_19108704, EPI_ISL_19108706, EPI_ISL_19108725, EPI_ISL_19109579, EPI_ISL_19116728, EPI_ISL_19117393, EPI_ISL_19131415, EPI_ISL_19131416, EPI_ISL_19131417, EPI_ISL_19131418, EPI_ISL_19131419, EPI_ISL_19132785, EPI_ISL_19132846, EPI_ISL_19133800, EPI_ISL_19135448, EPI_ISL_19135481, EPI_ISL_19135505, EPI_ISL_19135510, EPI_ISL_19135511, EPI_ISL_19137782, EPI_ISL_19137844, EPI_ISL_19140762, EPI_ISL_19141756, EPI_ISL_19141912, EPI_ISL_19142916, EPI_ISL_19143060, EPI_ISL_19143411, EPI_ISL_19143412, EPI_ISL_19143551, EPI_ISL_19143854, EPI_ISL_19143866, EPI_ISL_19143953, EPI_ISL_19143954, EPI_ISL_19144318, EPI_ISL_19145335, EPI_ISL_19146181, EPI_ISL_19146282, EPI_ISL_19146283, EPI_ISL_19146308, EPI_ISL_19146747, EPI_ISL_19147578, EPI_ISL_19148775, EPI_ISL_19151438, EPI_ISL_19153859, EPI_ISL_19158778, EPI_ISL_19158782, EPI_ISL_19159275, EPI_ISL_19159918, EPI_ISL_19159922, EPI_ISL_19159923, EPI_ISL_19161805, EPI_ISL_19164072, EPI_ISL_19164956, EPI_ISL_19165347, EPI_ISL_19165721, EPI_ISL_19167420, EPI_ISL_19167714, EPI_ISL_19169236, EPI_ISL_19169239, EPI_ISL_19169240, EPI_ISL_19169597, EPI_ISL_19169598, EPI_ISL_19173640, EPI_ISL_19173786, EPI_ISL_19175160, EPI_ISL_19175165, EPI_ISL_19175555, EPI_ISL_19175939, EPI_ISL_19176642, EPI_ISL_19176853, EPI_ISL_19176930, EPI_ISL_19177237, EPI_ISL_19177366, EPI_ISL_19177549, EPI_ISL_19177604, EPI_ISL_19177633, EPI_ISL_19178043, EPI_ISL_19178311, EPI_ISL_19179820, EPI_ISL_19182830, EPI_ISL_19182893, EPI_ISL_19183095, EPI_ISL_19183664, EPI_ISL_19183910, EPI_ISL_19183917, EPI_ISL_19183924, EPI_ISL_19184000, EPI_ISL_19184031, EPI_ISL_19184858, EPI_ISL_19185391, EPI_ISL_19186372, EPI_ISL_19187674, EPI_ISL_19187980, EPI_ISL_19188471, EPI_ISL_19189250, EPI_ISL_19190171, EPI_ISL_19190915, EPI_ISL_19191046, EPI_ISL_19192623, EPI_ISL_19192806, EPI_ISL_19192819, EPI_ISL_19193007, EPI_ISL_19193543, EPI_ISL_19193606, EPI_ISL_19193607, EPI_ISL_19193617, EPI_ISL_19195977, EPI_ISL_19195978, EPI_ISL_19195979, EPI_ISL_19195980, EPI_ISL_19195981, EPI_ISL_19195982, EPI_ISL_19196018, EPI_ISL_19196019, EPI_ISL_19196020, EPI_ISL_19196021, EPI_ISL_19197906, EPI_ISL_19198127, EPI_ISL_19198259, EPI_ISL_19199717, EPI_ISL_19199719, EPI_ISL_19200980, EPI_ISL_19202065, EPI_ISL_19202066, EPI_ISL_19203296, EPI_ISL_19205927, EPI_ISL_19209656, EPI_ISL_19210836, EPI_ISL_19211459, EPI_ISL_19213222, EPI_ISL_19213351, EPI_ISL_19214251, EPI_ISL_19214614, EPI_ISL_19216679, EPI_ISL_19216828, EPI_ISL_19217899, EPI_ISL_19219811, EPI_ISL_19221626, EPI_ISL_19223353, EPI_ISL_19225911, EPI_ISL_19226495, EPI_ISL_19227219, EPI_ISL_19228129, EPI_ISL_19229097, EPI_ISL_19229565, EPI_ISL_19230757, EPI_ISL_19230859, EPI_ISL_19230867, EPI_ISL_19230971, EPI_ISL_19232972, EPI_ISL_19234843, EPI_ISL_19237942, EPI_ISL_19239019, EPI_ISL_19239628, EPI_ISL_19239713, EPI_ISL_19243158, EPI_ISL_19243166, EPI_ISL_19243171, EPI_ISL_19243231, EPI_ISL_19243516, EPI_ISL_19243810, EPI_ISL_19244002, EPI_ISL_19245397, EPI_ISL_19245398, EPI_ISL_19245399, EPI_ISL_19251154, EPI_ISL_19254798, EPI_ISL_19256132, EPI_ISL_19256137, EPI_ISL_19256138, EPI_ISL_19256151, EPI_ISL_19256152, EPI_ISL_19257013, EPI_ISL_19257014, EPI_ISL_19257017, EPI_ISL_19257018, EPI_ISL_19257108, EPI_ISL_19257977, EPI_ISL_19259365, EPI_ISL_19259369, EPI_ISL_19259383, EPI_ISL_19260849, EPI_ISL_19260850, EPI_ISL_19261083, EPI_ISL_19261711, EPI_ISL_19264977, EPI_ISL_19265076, EPI_ISL_19267631, EPI_ISL_19268349, EPI_ISL_19268541, EPI_ISL_19269083, EPI_ISL_19269483, EPI_ISL_19271166, EPI_ISL_19271182, EPI_ISL_19271183, EPI_ISL_19271567, EPI_ISL_19271746, EPI_ISL_19273074, EPI_ISL_19277027, EPI_ISL_19277033, EPI_ISL_19280260, EPI_ISL_19281017, EPI_ISL_19282183, EPI_ISL_19282184, EPI_ISL_19283996, EPI_ISL_19286133, EPI_ISL_19286135, EPI_ISL_19286138, EPI_ISL_19286139, EPI_ISL_19286260, EPI_ISL_19287266, EPI_ISL_19287684, EPI_ISL_19288763, EPI_ISL_19290906, EPI_ISL_19292343, EPI_ISL_19292838, EPI_ISL_19292841, EPI_ISL_19297116, EPI_ISL_19297576, EPI_ISL_19298817, EPI_ISL_19299051, EPI_ISL_19300378, EPI_ISL_19300385, EPI_ISL_19300400, EPI_ISL_19301040, EPI_ISL_19302363, EPI_ISL_19302407, EPI_ISL_19304883, EPI_ISL_19308329, EPI_ISL_19308670, EPI_ISL_19308767, EPI_ISL_19308875, EPI_ISL_19308876, EPI_ISL_19308877, EPI_ISL_19308906, EPI_ISL_19309906, EPI_ISL_19310234, EPI_ISL_19311057, EPI_ISL_19311769, EPI_ISL_19312520, EPI_ISL_19318476, EPI_ISL_19319006, EPI_ISL_19320511, EPI_ISL_19322462, EPI_ISL_19324934, EPI_ISL_19326359, EPI_ISL_19331433, EPI_ISL_19332294, EPI_ISL_19333087, EPI_ISL_19335441, EPI_ISL_19340140, EPI_ISL_19341128, EPI_ISL_19341144, EPI_ISL_19344211, EPI_ISL_19344746, EPI_ISL_19345465, EPI_ISL_19345845, EPI_ISL_19346102, EPI_ISL_19346538, EPI_ISL_19348673, EPI_ISL_19351027, EPI_ISL_19351032, EPI_ISL_19351033, EPI_ISL_19351648, EPI_ISL_19351927, EPI_ISL_19359999, EPI_ISL_19360941, EPI_ISL_19362874, EPI_ISL_19362955, EPI_ISL_19363093, EPI_ISL_19363366, EPI_ISL_19364322, EPI_ISL_19364675, EPI_ISL_19365917, EPI_ISL_19369553, EPI_ISL_19369713, EPI_ISL_19374392, EPI_ISL_19374843, EPI_ISL_19374844, EPI_ISL_19374845, EPI_ISL_19380467, EPI_ISL_19381264, EPI_ISL_19381428, EPI_ISL_19381638, EPI_ISL_19381992, EPI_ISL_19381994, EPI_ISL_19382602, EPI_ISL_19383694, EPI_ISL_19384121, EPI_ISL_19385914, EPI_ISL_19385980, EPI_ISL_19387703, EPI_ISL_19388165, EPI_ISL_19388758, EPI_ISL_19391206, EPI_ISL_19391216, EPI_ISL_19393434, EPI_ISL_19393708, EPI_ISL_19398369, EPI_ISL_19405517, EPI_ISL_19405918, EPI_ISL_19408692, EPI_ISL_19408693, EPI_ISL_19408911, EPI_ISL_19409171, EPI_ISL_19410044, EPI_ISL_19410056, EPI_ISL_19410058, EPI_ISL_19411869, EPI_ISL_19412054, EPI_ISL_19412418, EPI_ISL_19414059, EPI_ISL_19414060, EPI_ISL_19414061, EPI_ISL_19414452, EPI_ISL_19414669, EPI_ISL_19414842, EPI_ISL_19415183, EPI_ISL_19415272, EPI_ISL_19415273, EPI_ISL_19415333, EPI_ISL_19417987, EPI_ISL_19418385, EPI_ISL_19418789, EPI_ISL_19419991, EPI_ISL_19425659, EPI_ISL_19427049, EPI_ISL_19427050, EPI_ISL_19427051, EPI_ISL_19428450, EPI_ISL_19428673, EPI_ISL_19431719, EPI_ISL_19433335, EPI_ISL_19434973, EPI_ISL_19438222, EPI_ISL_19438699, EPI_ISL_19441794, EPI_ISL_19446721, EPI_ISL_19446726, EPI_ISL_19447859, EPI_ISL_19450094, EPI_ISL_19451639, EPI_ISL_19452022, EPI_ISL_19454521, EPI_ISL_19456759, EPI_ISL_19457982, EPI_ISL_19458104, EPI_ISL_19458866, EPI_ISL_19459469, EPI_ISL_19463531, EPI_ISL_19463787, EPI_ISL_19463811, EPI_ISL_19464534, EPI_ISL_19465468, EPI_ISL_19467713, EPI_ISL_19467725, EPI_ISL_19468710, EPI_ISL_19473728, EPI_ISL_19474606, EPI_ISL_19474613, EPI_ISL_19477107, EPI_ISL_19478383, EPI_ISL_19478598, EPI_ISL_19479514, EPI_ISL_19480237, EPI_ISL_19482235, EPI_ISL_19483178, EPI_ISL_19483184, EPI_ISL_19483313, EPI_ISL_19486141, EPI_ISL_19488917, EPI_ISL_19491347, EPI_ISL_19495855, EPI_ISL_19498393, EPI_ISL_19499089, EPI_ISL_19499133, EPI_ISL_19499199, EPI_ISL_19499228, EPI_ISL_19499640, EPI_ISL_19499789, EPI_ISL_19500407, EPI_ISL_19502648, EPI_ISL_19503869, EPI_ISL_19506293, EPI_ISL_19506337, EPI_ISL_19506387, EPI_ISL_19506391, EPI_ISL_19506410, EPI_ISL_19506412, EPI_ISL_19506489, EPI_ISL_19506517, EPI_ISL_19506572, EPI_ISL_19506574, EPI_ISL_19508878, EPI_ISL_19510518, EPI_ISL_19511239, EPI_ISL_19511242, EPI_ISL_19511245, EPI_ISL_19511259, EPI_ISL_19511353, EPI_ISL_19512930, EPI_ISL_19513189, EPI_ISL_19513373, EPI_ISL_19513376, EPI_ISL_19517945, EPI_ISL_19519860, EPI_ISL_19521406, EPI_ISL_19522101, EPI_ISL_19526975, EPI_ISL_19527216, EPI_ISL_19527499, EPI_ISL_19529679, EPI_ISL_19529699, EPI_ISL_19530201, EPI_ISL_19530213, EPI_ISL_19530220, EPI_ISL_19532518, EPI_ISL_19535971, EPI_ISL_19536358, EPI_ISL_19537504, EPI_ISL_19537517, EPI_ISL_19537806, EPI_ISL_19539586, EPI_ISL_19539602, EPI_ISL_19541318, EPI_ISL_19543648, EPI_ISL_19543801, EPI_ISL_19546384, EPI_ISL_19550039, EPI_ISL_19550107, EPI_ISL_19551038, EPI_ISL_19551134, EPI_ISL_19555102, EPI_ISL_19555114, EPI_ISL_19555115, EPI_ISL_19557993, EPI_ISL_19558034, EPI_ISL_19560436, EPI_ISL_19560828, EPI_ISL_19560838, EPI_ISL_19561072, EPI_ISL_19561079, EPI_ISL_19562666, EPI_ISL_19575844, EPI_ISL_19575892, EPI_ISL_19577900, EPI_ISL_19578009, EPI_ISL_19578384, EPI_ISL_19585411, EPI_ISL_19585472, EPI_ISL_19588050, EPI_ISL_19588617, EPI_ISL_19588630, EPI_ISL_19589209, EPI_ISL_19589226, EPI_ISL_19589644, EPI_ISL_19591990, EPI_ISL_19600727, EPI_ISL_19601813, EPI_ISL_19603076, EPI_ISL_19603396, EPI_ISL_19603766, EPI_ISL_19606872, EPI_ISL_19606873, EPI_ISL_19606893, EPI_ISL_19613477, EPI_ISL_19613481, EPI_ISL_19613618, EPI_ISL_19615536, EPI_ISL_19615541, EPI_ISL_19615759, EPI_ISL_19616027, EPI_ISL_19616479, EPI_ISL_19618593, EPI_ISL_19619689, EPI_ISL_19619734, EPI_ISL_19619757, EPI_ISL_19620010, EPI_ISL_19621085, EPI_ISL_19621954, EPI_ISL_19622266, EPI_ISL_19623358, EPI_ISL_19623503, EPI_ISL_19627737, EPI_ISL_19627776, EPI_ISL_19627778, EPI_ISL_19630975, EPI_ISL_19631111, EPI_ISL_19632962, EPI_ISL_19634022, EPI_ISL_19636173, EPI_ISL_19638051, EPI_ISL_19639319, EPI_ISL_19640624, EPI_ISL_19641389, EPI_ISL_19641840, EPI_ISL_19642771, EPI_ISL_19643417, EPI_ISL_19643444, EPI_ISL_19643542, EPI_ISL_19643616, EPI_ISL_19643666, EPI_ISL_19645977, EPI_ISL_19646339, EPI_ISL_19648297, EPI_ISL_19655163, EPI_ISL_19656451, EPI_ISL_19657328, EPI_ISL_19657519, EPI_ISL_19657528, EPI_ISL_19660180, EPI_ISL_19661881, EPI_ISL_19661997, EPI_ISL_19665170, EPI_ISL_19666574, EPI_ISL_19671026, EPI_ISL_19671598, EPI_ISL_19671605, EPI_ISL_19671606, EPI_ISL_19671607, EPI_ISL_19671887, EPI_ISL_19673194, EPI_ISL_19673201, EPI_ISL_19676136, EPI_ISL_19680534, EPI_ISL_19683226, EPI_ISL_19683235, EPI_ISL_19683886, EPI_ISL_19685790, EPI_ISL_19685811, EPI_ISL_19685815, EPI_ISL_19685993, EPI_ISL_19688052, EPI_ISL_19689294, EPI_ISL_19690695, EPI_ISL_19693383, EPI_ISL_19693416, EPI_ISL_19693419, EPI_ISL_19694042, EPI_ISL_19696986, EPI_ISL_19703902, EPI_ISL_19704129, EPI_ISL_19704643, EPI_ISL_19704646, EPI_ISL_19706712, EPI_ISL_19706713, EPI_ISL_19710331, EPI_ISL_19710359, EPI_ISL_19711118, EPI_ISL_19711122, EPI_ISL_19713810, EPI_ISL_19713821, EPI_ISL_19713832, EPI_ISL_19714066, EPI_ISL_19715063, EPI_ISL_19715322, EPI_ISL_19717002, EPI_ISL_19719176, EPI_ISL_19719216, EPI_ISL_19719218, EPI_ISL_19720736, EPI_ISL_19720749, EPI_ISL_19720751, EPI_ISL_19720757, EPI_ISL_19720765, EPI_ISL_19720774, EPI_ISL_19720781, EPI_ISL_19721494, EPI_ISL_19721498, EPI_ISL_19729765, EPI_ISL_19729811, EPI_ISL_19729840, EPI_ISL_19730527, EPI_ISL_19735517, EPI_ISL_19736097, EPI_ISL_19736266, EPI_ISL_19736299, EPI_ISL_19736349, EPI_ISL_19736527, EPI_ISL_19739887, EPI_ISL_19742521, EPI_ISL_19743439, EPI_ISL_19743558, EPI_ISL_19748970, EPI_ISL_19749286, EPI_ISL_19749348, EPI_ISL_19750081, EPI_ISL_19750087, EPI_ISL_19750479, EPI_ISL_19751001, EPI_ISL_19752288, EPI_ISL_19755631, EPI_ISL_19760571, EPI_ISL_19761402, EPI_ISL_19761928, EPI_ISL_19765520, EPI_ISL_19765527, EPI_ISL_19766581, EPI_ISL_19766606, EPI_ISL_19770265, EPI_ISL_19770267, EPI_ISL_19771106, EPI_ISL_19773769, EPI_ISL_19773770, EPI_ISL_19775232, EPI_ISL_19775886, EPI_ISL_19776542, EPI_ISL_19782005, EPI_ISL_19783524, EPI_ISL_19786156, EPI_ISL_19788439, EPI_ISL_19791384, EPI_ISL_19794365, EPI_ISL_19795466, EPI_ISL_19797371, EPI_ISL_19798087, EPI_ISL_19799368, EPI_ISL_19799392, EPI_ISL_19799712, EPI_ISL_19799810, EPI_ISL_19800009, EPI_ISL_19800089, EPI_ISL_19800094, EPI_ISL_19800098, EPI_ISL_19800151, EPI_ISL_19801979, EPI_ISL_19803172, EPI_ISL_19803215, EPI_ISL_19803225, EPI_ISL_19803382, EPI_ISL_19804293, EPI_ISL_19804491, EPI_ISL_19805234, EPI_ISL_19806441, EPI_ISL_19806502, EPI_ISL_19806836, EPI_ISL_19806981, EPI_ISL_19807128, EPI_ISL_19807229, EPI_ISL_19808706, EPI_ISL_19810078, EPI_ISL_19810444, EPI_ISL_19810684, EPI_ISL_19812168, EPI_ISL_19814897, EPI_ISL_19815994, EPI_ISL_19816020, EPI_ISL_19816100, EPI_ISL_19816126, EPI_ISL_19816143, EPI_ISL_19816182, EPI_ISL_19816207, EPI_ISL_19816266, EPI_ISL_19816276, EPI_ISL_19816297, EPI_ISL_19816344, EPI_ISL_19816356, EPI_ISL_19816434, EPI_ISL_19819753, EPI_ISL_19819801, EPI_ISL_19820321, EPI_ISL_19822739, EPI_ISL_19824345, EPI_ISL_19826284, EPI_ISL_19826623, EPI_ISL_19828130, EPI_ISL_19828714, EPI_ISL_19831372, EPI_ISL_19831976, EPI_ISL_19836303, EPI_ISL_19838390, EPI_ISL_19841148, EPI_ISL_19845799, EPI_ISL_19845890, EPI_ISL_19846096, EPI_ISL_19846112, EPI_ISL_19846130, EPI_ISL_19846377, EPI_ISL_19846620, EPI_ISL_19846657, EPI_ISL_19846658, EPI_ISL_19848250, EPI_ISL_19848532, EPI_ISL_19848571, EPI_ISL_19850000, EPI_ISL_19851510, EPI_ISL_19852484, EPI_ISL_19853694, EPI_ISL_19853731, EPI_ISL_19854645, EPI_ISL_19856109, EPI_ISL_19856123, EPI_ISL_19859249, EPI_ISL_19859991, EPI_ISL_19860001, EPI_ISL_19860628, EPI_ISL_19860631, EPI_ISL_19860648, EPI_ISL_19860650, EPI_ISL_19860663, EPI_ISL_19862012, EPI_ISL_19867834, EPI_ISL_19867960, EPI_ISL_19869240, EPI_ISL_19869461, EPI_ISL_19869479, EPI_ISL_19870045, EPI_ISL_19872997, EPI_ISL_19873015, EPI_ISL_19874592, EPI_ISL_19875482, EPI_ISL_19875516, EPI_ISL_19875907, EPI_ISL_19875923, EPI_ISL_19875929, EPI_ISL_19876467, EPI_ISL_19876468, EPI_ISL_19876475, EPI_ISL_19876688, EPI_ISL_19878325, EPI_ISL_19878543, EPI_ISL_19878655, EPI_ISL_19881774, EPI_ISL_19881782, EPI_ISL_19885459, EPI_ISL_19888289, EPI_ISL_19890790, EPI_ISL_19893141, EPI_ISL_19895139, EPI_ISL_19896098, EPI_ISL_19900414, EPI_ISL_19904802, EPI_ISL_19905384, EPI_ISL_19906764, EPI_ISL_19907172, EPI_ISL_19907176, EPI_ISL_19908300, EPI_ISL_19909161, EPI_ISL_19909472, EPI_ISL_19912125, EPI_ISL_19913944, EPI_ISL_19913962, EPI_ISL_19914005, EPI_ISL_19914095, EPI_ISL_19914209, EPI_ISL_19914518, EPI_ISL_19915549, EPI_ISL_19916900, EPI_ISL_19921952, EPI_ISL_19922099, EPI_ISL_19922152, EPI_ISL_19922153, EPI_ISL_19923571, EPI_ISL_19924741, EPI_ISL_19925769, EPI_ISL_19927172, EPI_ISL_19927619, EPI_ISL_19930618, EPI_ISL_19931371, EPI_ISL_19931603, EPI_ISL_19931761, EPI_ISL_19932828, EPI_ISL_19932862, EPI_ISL_19933840, EPI_ISL_19937235, EPI_ISL_19939096, EPI_ISL_19939840, EPI_ISL_19940375, EPI_ISL_19941809, EPI_ISL_19942866, EPI_ISL_19944411, EPI_ISL_19945844, EPI_ISL_19946231, EPI_ISL_19946695, EPI_ISL_19948061, EPI_ISL_19964231, EPI_ISL_19971003, EPI_ISL_19972226, EPI_ISL_19977768, EPI_ISL_19978422, EPI_ISL_19981086, EPI_ISL_19981113, EPI_ISL_19981563, EPI_ISL_19987040, EPI_ISL_19988382, EPI_ISL_19988544, EPI_ISL_19988982, EPI_ISL_19988985, EPI_ISL_19989313, EPI_ISL_19989447, EPI_ISL_19989546, EPI_ISL_19989696, EPI_ISL_19989888, EPI_ISL_19991141, EPI_ISL_19993030, EPI_ISL_19996562, EPI_ISL_19998366, EPI_ISL_19998531, EPI_ISL_20000997, EPI_ISL_20001882, EPI_ISL_20003020, EPI_ISL_20003783, EPI_ISL_20004263, EPI_ISL_20004532, EPI_ISL_20005159, EPI_ISL_20005257, EPI_ISL_20005829, EPI_ISL_20006380, EPI_ISL_20007517, EPI_ISL_20007659, EPI_ISL_20010416, EPI_ISL_20011218, EPI_ISL_20012094, EPI_ISL_20012599, EPI_ISL_20013003, EPI_ISL_20013131, EPI_ISL_20013626, EPI_ISL_20013761, EPI_ISL_20013799, EPI_ISL_20013935, EPI_ISL_20014644, EPI_ISL_20014744, EPI_ISL_20014843, EPI_ISL_20015117, EPI_ISL_20015176, EPI_ISL_20015192, EPI_ISL_20015246, EPI_ISL_20015340, EPI_ISL_20015538, EPI_ISL_20016529, EPI_ISL_20016537, EPI_ISL_20016564, EPI_ISL_20016684, EPI_ISL_20016782, EPI_ISL_20016815, EPI_ISL_20016838, EPI_ISL_20017364, EPI_ISL_20017606, EPI_ISL_20017773, EPI_ISL_20018061, EPI_ISL_20019468, EPI_ISL_20024326, EPI_ISL_20025108, EPI_ISL_20025341, EPI_ISL_20025539, EPI_ISL_20026339, EPI_ISL_20027365, EPI_ISL_20027593, EPI_ISL_20027953, EPI_ISL_20030503, EPI_ISL_20040387, EPI_ISL_20040730, EPI_ISL_20043225, EPI_ISL_20043488, EPI_ISL_20045296, EPI_ISL_20046476, EPI_ISL_20047831, EPI_ISL_20047892, EPI_ISL_20048939, EPI_ISL_20048957, EPI_ISL_20053144, EPI_ISL_20053774, EPI_ISL_20054078, EPI_ISL_20056146, EPI_ISL_20062254, EPI_ISL_20063868, EPI_ISL_20066350, EPI_ISL_20066351, EPI_ISL_20066352, EPI_ISL_20066353, EPI_ISL_20066354, EPI_ISL_20066355, EPI_ISL_20066356, EPI_ISL_20066357, EPI_ISL_20066358, EPI_ISL_20066359, EPI_ISL_20066360, EPI_ISL_20066487, EPI_ISL_20070801, EPI_ISL_20070849, EPI_ISL_20074733, EPI_ISL_20076185, EPI_ISL_20080050, EPI_ISL_20081356, EPI_ISL_20081602, EPI_ISL_20081628, EPI_ISL_20082525, EPI_ISL_20088904, EPI_ISL_20096211, EPI_ISL_20096224, EPI_ISL_20097661, EPI_ISL_20097746, EPI_ISL_20097747, EPI_ISL_20098386, EPI_ISL_20099967, EPI_ISL_20113880, EPI_ISL_20114442, EPI_ISL_20114467, EPI_ISL_20114468, EPI_ISL_20114469, EPI_ISL_20115114, EPI_ISL_20127947, EPI_ISL_20131508, EPI_ISL_20132435, EPI_ISL_20136586, EPI_ISL_20137604, EPI_ISL_20138079, EPI_ISL_20138923, EPI_ISL_20139405, EPI_ISL_20140153, EPI_ISL_20140838, EPI_ISL_20140845, EPI_ISL_20141945, EPI_ISL_20141955, EPI_ISL_20141982, EPI_ISL_20143974, EPI_ISL_20146793, EPI_ISL_20146795, EPI_ISL_20146858, EPI_ISL_20148475, EPI_ISL_20155076, EPI_ISL_20155864, EPI_ISL_20157513, EPI_ISL_20166121, EPI_ISL_20167020, EPI_ISL_20175070, EPI_ISL_20176704, EPI_ISL_20176761, EPI_ISL_20178658, EPI_ISL_20179076, EPI_ISL_20179179, EPI_ISL_20181458, EPI_ISL_20181461, EPI_ISL_20182683, EPI_ISL_20182757, EPI_ISL_20184766, EPI_ISL_20185296, EPI_ISL_20185471, EPI_ISL_20185528, EPI_ISL_20193978, EPI_ISL_20193979, EPI_ISL_20193993, EPI_ISL_20193997, EPI_ISL_20194032, EPI_ISL_20196214, EPI_ISL_20196709, EPI_ISL_20196755, EPI_ISL_20196768, EPI_ISL_20196838, EPI_ISL_20198727, EPI_ISL_20212488, EPI_ISL_20223218, EPI_ISL_20223363, EPI_ISL_20223369, EPI_ISL_20225608, EPI_ISL_20227249")
chronics_2025_11_09_update = stringlist_to_set("EPI_ISL_474824, EPI_ISL_508674, EPI_ISL_510148, EPI_ISL_516999, EPI_ISL_517000, EPI_ISL_528438, EPI_ISL_529236, EPI_ISL_534720, EPI_ISL_539541, EPI_ISL_539542, EPI_ISL_540582, EPI_ISL_593478, EPI_ISL_593479, EPI_ISL_593480, EPI_ISL_593553, EPI_ISL_593554, EPI_ISL_593555, EPI_ISL_593556, EPI_ISL_593557, EPI_ISL_593558, EPI_ISL_596228, EPI_ISL_602912, EPI_ISL_654166, EPI_ISL_654170, EPI_ISL_654172, EPI_ISL_654173, EPI_ISL_654182, EPI_ISL_654186, EPI_ISL_654191, EPI_ISL_654194, EPI_ISL_678289, EPI_ISL_686537, EPI_ISL_732971, EPI_ISL_753676, EPI_ISL_776770, EPI_ISL_779969, EPI_ISL_801876, EPI_ISL_812862, EPI_ISL_831496, EPI_ISL_872778, EPI_ISL_941340, EPI_ISL_949208, EPI_ISL_959309, EPI_ISL_979349, EPI_ISL_996554, EPI_ISL_1014810, EPI_ISL_1039159, EPI_ISL_1039839, EPI_ISL_1048102, EPI_ISL_1048104, EPI_ISL_1048105, EPI_ISL_1059094, EPI_ISL_1090851, EPI_ISL_1103757, EPI_ISL_1104882, EPI_ISL_1105146, EPI_ISL_1105179, EPI_ISL_1105235, EPI_ISL_1158169, EPI_ISL_1200867, EPI_ISL_1200912, EPI_ISL_1209365, EPI_ISL_1225620, EPI_ISL_1241756, EPI_ISL_1248458, EPI_ISL_1248485, EPI_ISL_1248497, EPI_ISL_1257978, EPI_ISL_1261009, EPI_ISL_1295569, EPI_ISL_1295575, EPI_ISL_1308671, EPI_ISL_1309105, EPI_ISL_1366562, EPI_ISL_1366563, EPI_ISL_1366564, EPI_ISL_1366565, EPI_ISL_1366566, EPI_ISL_1366567, EPI_ISL_1366568, EPI_ISL_1366569, EPI_ISL_1366570, EPI_ISL_1366571, EPI_ISL_1366572, EPI_ISL_1366573, EPI_ISL_1366792, EPI_ISL_1372287, EPI_ISL_1372288, EPI_ISL_1378739, EPI_ISL_1379391, EPI_ISL_1379435, EPI_ISL_1403404, EPI_ISL_1469973, EPI_ISL_1470396, EPI_ISL_1473700, EPI_ISL_1477334, EPI_ISL_1483302, EPI_ISL_1490655, EPI_ISL_1490669, EPI_ISL_1495749, EPI_ISL_1517099, EPI_ISL_1522107, EPI_ISL_1534324, EPI_ISL_1547461, EPI_ISL_1554931, EPI_ISL_1554932, EPI_ISL_1554933, EPI_ISL_1554934, EPI_ISL_1575358, EPI_ISL_1578495, EPI_ISL_1582755, EPI_ISL_1593324, EPI_ISL_1626185, EPI_ISL_1637040, EPI_ISL_1668821, EPI_ISL_1668822, EPI_ISL_1668823, EPI_ISL_1668824, EPI_ISL_1668825, EPI_ISL_1670378, EPI_ISL_1671330, EPI_ISL_1675190, EPI_ISL_1675203, EPI_ISL_1678377, EPI_ISL_1738308, EPI_ISL_1743263, EPI_ISL_1744401, EPI_ISL_1756179, EPI_ISL_1756180, EPI_ISL_1792929, EPI_ISL_1829054, EPI_ISL_1829108, EPI_ISL_1840893, EPI_ISL_1853568, EPI_ISL_1855854, EPI_ISL_1904901, EPI_ISL_1904903, EPI_ISL_1908476, EPI_ISL_1909055, EPI_ISL_1935116, EPI_ISL_1935282, EPI_ISL_1941336, EPI_ISL_1941816, EPI_ISL_1965009, EPI_ISL_1965714, EPI_ISL_1965722, EPI_ISL_2001260, EPI_ISL_2001292, EPI_ISL_2030332, EPI_ISL_2035047, EPI_ISL_2036230, EPI_ISL_2080876, EPI_ISL_2096935, EPI_ISL_2098974, EPI_ISL_2140680, EPI_ISL_2142447, EPI_ISL_2159603, EPI_ISL_2170618, EPI_ISL_2179080, EPI_ISL_2179597, EPI_ISL_2179598, EPI_ISL_2179600, EPI_ISL_2179601, EPI_ISL_2179635, EPI_ISL_2193387, EPI_ISL_2193781, EPI_ISL_2193790, EPI_ISL_2228938, EPI_ISL_2229473, EPI_ISL_2232987, EPI_ISL_2232988, EPI_ISL_2245655, EPI_ISL_2246946, EPI_ISL_2272316, EPI_ISL_2281463, EPI_ISL_2285732, EPI_ISL_2289324, EPI_ISL_2331631, EPI_ISL_2335139, EPI_ISL_2355027, EPI_ISL_2373667, EPI_ISL_2373676, EPI_ISL_2373689, EPI_ISL_2373915, EPI_ISL_2373976, EPI_ISL_2376734, EPI_ISL_2385134, EPI_ISL_2397307, EPI_ISL_2397308, EPI_ISL_2397309, EPI_ISL_2397310, EPI_ISL_2397311, EPI_ISL_2397312, EPI_ISL_2397313, EPI_ISL_2403056, EPI_ISL_2408213, EPI_ISL_2443102, EPI_ISL_2443306, EPI_ISL_2451852, EPI_ISL_2453771, EPI_ISL_2456706, EPI_ISL_2466638, EPI_ISL_2476401, EPI_ISL_2482552, EPI_ISL_2482891, EPI_ISL_2484152, EPI_ISL_2484216, EPI_ISL_2484217, EPI_ISL_2484218, EPI_ISL_2484219, EPI_ISL_2485156, EPI_ISL_2492266, EPI_ISL_2501697, EPI_ISL_2504017, EPI_ISL_2510252, EPI_ISL_2526835, EPI_ISL_2536904, EPI_ISL_2537393, EPI_ISL_2545260, EPI_ISL_2566726, EPI_ISL_2567482, EPI_ISL_2567516, EPI_ISL_2598472, EPI_ISL_2621566, EPI_ISL_2622092, EPI_ISL_2626505, EPI_ISL_2629070, EPI_ISL_2629071, EPI_ISL_2629072, EPI_ISL_2629073, EPI_ISL_2629074, EPI_ISL_2629075, EPI_ISL_2646107, EPI_ISL_2652487, EPI_ISL_2658958, EPI_ISL_2658962, EPI_ISL_2658963, EPI_ISL_2658970, EPI_ISL_2658971, EPI_ISL_2658972, EPI_ISL_2674459, EPI_ISL_2681259, EPI_ISL_2686814, EPI_ISL_2686837, EPI_ISL_2696945, EPI_ISL_2709225, EPI_ISL_2713004, EPI_ISL_2716246, EPI_ISL_2731960, EPI_ISL_2732319, EPI_ISL_2733379, EPI_ISL_2733797, EPI_ISL_2741595, EPI_ISL_2749506, EPI_ISL_2749718, EPI_ISL_2758178, EPI_ISL_2758179, EPI_ISL_2768315, EPI_ISL_2776212, EPI_ISL_2788712, EPI_ISL_2790083, EPI_ISL_2791260, EPI_ISL_2810326, EPI_ISL_2811857, EPI_ISL_2817504, EPI_ISL_2820526, EPI_ISL_2828407, EPI_ISL_2833904, EPI_ISL_2858161, EPI_ISL_2858877, EPI_ISL_2860316, EPI_ISL_2868572, EPI_ISL_2868616, EPI_ISL_2876377, EPI_ISL_2894033, EPI_ISL_2903438, EPI_ISL_2928168, EPI_ISL_2931876, EPI_ISL_2931884, EPI_ISL_2931896, EPI_ISL_2931903, EPI_ISL_2955288, EPI_ISL_2955320, EPI_ISL_2966985, EPI_ISL_2978243, EPI_ISL_2978352, EPI_ISL_2983871, EPI_ISL_2984725, EPI_ISL_2990101, EPI_ISL_2993722, EPI_ISL_3010321, EPI_ISL_3029841, EPI_ISL_3030114, EPI_ISL_3030118, EPI_ISL_3030145, EPI_ISL_3030738, EPI_ISL_3032627, EPI_ISL_3033635, EPI_ISL_3039352, EPI_ISL_3045789, EPI_ISL_3045809, EPI_ISL_3053903, EPI_ISL_3061061, EPI_ISL_3063770, EPI_ISL_3064314, EPI_ISL_3123372, EPI_ISL_3129808, EPI_ISL_3130077, EPI_ISL_3130081, EPI_ISL_3130177, EPI_ISL_3130302, EPI_ISL_3132631, EPI_ISL_3133023, EPI_ISL_3152200, EPI_ISL_3153240, EPI_ISL_3164424, EPI_ISL_3185346, EPI_ISL_3208304, EPI_ISL_3209041, EPI_ISL_3212959, EPI_ISL_3215721, EPI_ISL_3215722, EPI_ISL_3215726, EPI_ISL_3246237, EPI_ISL_3251444, EPI_ISL_3251605, EPI_ISL_3259560, EPI_ISL_3275376, EPI_ISL_3339536, EPI_ISL_3356734, EPI_ISL_3358574, EPI_ISL_3370176, EPI_ISL_3394321, EPI_ISL_3396491, EPI_ISL_3414767, EPI_ISL_3414889, EPI_ISL_3415104, EPI_ISL_3415226, EPI_ISL_3426474, EPI_ISL_3446827, EPI_ISL_3447712, EPI_ISL_3453279, EPI_ISL_3457423, EPI_ISL_3459118, EPI_ISL_3460897, EPI_ISL_3471360, EPI_ISL_3475993, EPI_ISL_3497667, EPI_ISL_3503811, EPI_ISL_3504036, EPI_ISL_3509013, EPI_ISL_3556945, EPI_ISL_3634003, EPI_ISL_3634004, EPI_ISL_3635506, EPI_ISL_3657112, EPI_ISL_3666069, EPI_ISL_3712919, EPI_ISL_3771882, EPI_ISL_3779849, EPI_ISL_3813731, EPI_ISL_3838306, EPI_ISL_3891136, EPI_ISL_3933252, EPI_ISL_3937027, EPI_ISL_3957778, EPI_ISL_3958461, EPI_ISL_3958994, EPI_ISL_3982251, EPI_ISL_4029567, EPI_ISL_4051642, EPI_ISL_4052911, EPI_ISL_4072038, EPI_ISL_4096626, EPI_ISL_4096639, EPI_ISL_4114033, EPI_ISL_4124532, EPI_ISL_4178790, EPI_ISL_4193135, EPI_ISL_4198270, EPI_ISL_4203869, EPI_ISL_4251611, EPI_ISL_4261403, EPI_ISL_4261411, EPI_ISL_4281194, EPI_ISL_4284228, EPI_ISL_4295678, EPI_ISL_4298277, EPI_ISL_4298278, EPI_ISL_4298279, EPI_ISL_4309817, EPI_ISL_4415808, EPI_ISL_4440075, EPI_ISL_4515444, EPI_ISL_4525691, EPI_ISL_4525698, EPI_ISL_4525700, EPI_ISL_4531313, EPI_ISL_4536418, EPI_ISL_4576991, EPI_ISL_4577474, EPI_ISL_4625101, EPI_ISL_4652284, EPI_ISL_4769360, EPI_ISL_4769386, EPI_ISL_4775547, EPI_ISL_4875939, EPI_ISL_4930863, EPI_ISL_4935777, EPI_ISL_4935949, EPI_ISL_4936095, EPI_ISL_4936533, EPI_ISL_4949584, EPI_ISL_5018695, EPI_ISL_5033183, EPI_ISL_5056434, EPI_ISL_5059980, EPI_ISL_5099310, EPI_ISL_5132437, EPI_ISL_5132595, EPI_ISL_5145656, EPI_ISL_5196003, EPI_ISL_5263400, EPI_ISL_5265214, EPI_ISL_5280146, EPI_ISL_5307398, EPI_ISL_5323016, EPI_ISL_5332877, EPI_ISL_5332878, EPI_ISL_5395558, EPI_ISL_5446154, EPI_ISL_5463914, EPI_ISL_5532714, EPI_ISL_5592605, EPI_ISL_5594441, EPI_ISL_5620309, EPI_ISL_5621224, EPI_ISL_5627313, EPI_ISL_5628248, EPI_ISL_5639913, EPI_ISL_5640459, EPI_ISL_5649323, EPI_ISL_5650474, EPI_ISL_5655562, EPI_ISL_5680241, EPI_ISL_5692576, EPI_ISL_5692774, EPI_ISL_5692832, EPI_ISL_5693164, EPI_ISL_5749185, EPI_ISL_5780324, EPI_ISL_5814411, EPI_ISL_5865553, EPI_ISL_5892132, EPI_ISL_5922347, EPI_ISL_5935407, EPI_ISL_5944665, EPI_ISL_5944669, EPI_ISL_5944842, EPI_ISL_5944948, EPI_ISL_5946914, EPI_ISL_6017746, EPI_ISL_6017747, EPI_ISL_6076460, EPI_ISL_6113246, EPI_ISL_6208674, EPI_ISL_6208675, EPI_ISL_6208676, EPI_ISL_6227177, EPI_ISL_6227208, EPI_ISL_6262165, EPI_ISL_6262536, EPI_ISL_6281381, EPI_ISL_6324366, EPI_ISL_6327943, EPI_ISL_6381841, EPI_ISL_6384755, EPI_ISL_6397259, EPI_ISL_6574278, EPI_ISL_6584511, EPI_ISL_6604686, EPI_ISL_6605003, EPI_ISL_6605659, EPI_ISL_6628662, EPI_ISL_6642599, EPI_ISL_6660906, EPI_ISL_6666037, EPI_ISL_6698637, EPI_ISL_6710721, EPI_ISL_6735468, EPI_ISL_6737833, EPI_ISL_6757093, EPI_ISL_6783610, EPI_ISL_6810267, EPI_ISL_6814822, EPI_ISL_6826536, EPI_ISL_6842893, EPI_ISL_6863316, EPI_ISL_6863457, EPI_ISL_6865741, EPI_ISL_6930836, EPI_ISL_6938691, EPI_ISL_6976497, EPI_ISL_6977941, EPI_ISL_7000663, EPI_ISL_7015624, EPI_ISL_7015625, EPI_ISL_7135374, EPI_ISL_7159687, EPI_ISL_7175748, EPI_ISL_7204318, EPI_ISL_7205152, EPI_ISL_7334013, EPI_ISL_7361483, EPI_ISL_7361527, EPI_ISL_7452581, EPI_ISL_7452603, EPI_ISL_7456462, EPI_ISL_7458003, EPI_ISL_7473129, EPI_ISL_7473130, EPI_ISL_7473131, EPI_ISL_7473132, EPI_ISL_7473133, EPI_ISL_7473134, EPI_ISL_7473135, EPI_ISL_7473136, EPI_ISL_7473137, EPI_ISL_7473138, EPI_ISL_7473139, EPI_ISL_7473140, EPI_ISL_7473141, EPI_ISL_7473142, EPI_ISL_7473143, EPI_ISL_7473144, EPI_ISL_7502990, EPI_ISL_7503221, EPI_ISL_7507393, EPI_ISL_7592661, EPI_ISL_7592687, EPI_ISL_7652115, EPI_ISL_7660179, EPI_ISL_7707631, EPI_ISL_7711813, EPI_ISL_7729239, EPI_ISL_7806535, EPI_ISL_7806536, EPI_ISL_7806549, EPI_ISL_7813798, EPI_ISL_7813896, EPI_ISL_7861580, EPI_ISL_7876524, EPI_ISL_7908114, EPI_ISL_7961502, EPI_ISL_7976931, EPI_ISL_7980711, EPI_ISL_8001538, EPI_ISL_8035582, EPI_ISL_8057418, EPI_ISL_8131224, EPI_ISL_8132253, EPI_ISL_8151798, EPI_ISL_8153087, EPI_ISL_8166542, EPI_ISL_8189765, EPI_ISL_8189775, EPI_ISL_8204828, EPI_ISL_8205040, EPI_ISL_8207600, EPI_ISL_8215783, EPI_ISL_8215787, EPI_ISL_8251200, EPI_ISL_8263463, EPI_ISL_8350537, EPI_ISL_8376888, EPI_ISL_8479639, EPI_ISL_8479640, EPI_ISL_8563217, EPI_ISL_8563218, EPI_ISL_8563219, EPI_ISL_8615077, EPI_ISL_8627379, EPI_ISL_8669281, EPI_ISL_8675368, EPI_ISL_8712661, EPI_ISL_8725398, EPI_ISL_8725399, EPI_ISL_8725400, EPI_ISL_8725401, EPI_ISL_8725402, EPI_ISL_8725403, EPI_ISL_8725404, EPI_ISL_8725405, EPI_ISL_8725406, EPI_ISL_8725407, EPI_ISL_8725408, EPI_ISL_8725409, EPI_ISL_8732699, EPI_ISL_8732807, EPI_ISL_8732841, EPI_ISL_8750545, EPI_ISL_8754305, EPI_ISL_8766992, EPI_ISL_8800409, EPI_ISL_8806077, EPI_ISL_8806082, EPI_ISL_8806084, EPI_ISL_8819629, EPI_ISL_8825833, EPI_ISL_8828662, EPI_ISL_8887845, EPI_ISL_8887874, EPI_ISL_8889632, EPI_ISL_9021214, EPI_ISL_9141923, EPI_ISL_9155607, EPI_ISL_9201951, EPI_ISL_9242265, EPI_ISL_9242269, EPI_ISL_9319180, EPI_ISL_9570633, EPI_ISL_9630717, EPI_ISL_9637481, EPI_ISL_9637840, EPI_ISL_9702285, EPI_ISL_9735644, EPI_ISL_9844246, EPI_ISL_9869512, EPI_ISL_9873278, EPI_ISL_9907655, EPI_ISL_9949797, EPI_ISL_9949799, EPI_ISL_10124646, EPI_ISL_10127751, EPI_ISL_10128185, EPI_ISL_10128193, EPI_ISL_10166426, EPI_ISL_10185453, EPI_ISL_10195257, EPI_ISL_10195262, EPI_ISL_10195263, EPI_ISL_10195264, EPI_ISL_10195305, EPI_ISL_10230612, EPI_ISL_10239201, EPI_ISL_10251304, EPI_ISL_10329391, EPI_ISL_10329558, EPI_ISL_10352747, EPI_ISL_10381820, EPI_ISL_10397517, EPI_ISL_10451205, EPI_ISL_10451252, EPI_ISL_10548912, EPI_ISL_10548913, EPI_ISL_10548915, EPI_ISL_10548916, EPI_ISL_10548917, EPI_ISL_10548918, EPI_ISL_10548919, EPI_ISL_10548920, EPI_ISL_10548921, EPI_ISL_10548922, EPI_ISL_10549162, EPI_ISL_10549163, EPI_ISL_10549164, EPI_ISL_10549165, EPI_ISL_10549166, EPI_ISL_10590270, EPI_ISL_10590760, EPI_ISL_10591808, EPI_ISL_10657890, EPI_ISL_10681118, EPI_ISL_10704797, EPI_ISL_10706292, EPI_ISL_10712909, EPI_ISL_10717525, EPI_ISL_10787994, EPI_ISL_10815044, EPI_ISL_10816730, EPI_ISL_10816731, EPI_ISL_10816732, EPI_ISL_10816733, EPI_ISL_10816734, EPI_ISL_10816735, EPI_ISL_10816736, EPI_ISL_10816737, EPI_ISL_10816738, EPI_ISL_10816739, EPI_ISL_10816741, EPI_ISL_10816742, EPI_ISL_10816743, EPI_ISL_10816744, EPI_ISL_10824028, EPI_ISL_10876034, EPI_ISL_10876749, EPI_ISL_10899907, EPI_ISL_10942195, EPI_ISL_10942196, EPI_ISL_10981395, EPI_ISL_10995323, EPI_ISL_10998528, EPI_ISL_11025821, EPI_ISL_11025897, EPI_ISL_11030507, EPI_ISL_11036385, EPI_ISL_11036386, EPI_ISL_11036389, EPI_ISL_11036399, EPI_ISL_11036451, EPI_ISL_11036666, EPI_ISL_11036688, EPI_ISL_11036712, EPI_ISL_11036915, EPI_ISL_11036917, EPI_ISL_11050902, EPI_ISL_11055609, EPI_ISL_11106543, EPI_ISL_11110730, EPI_ISL_11173072, EPI_ISL_11219235, EPI_ISL_11219236, EPI_ISL_11221773, EPI_ISL_11221782, EPI_ISL_11222620, EPI_ISL_11229672, EPI_ISL_11230305, EPI_ISL_11239958, EPI_ISL_11242266, EPI_ISL_11248919, EPI_ISL_11256669, EPI_ISL_11290054, EPI_ISL_11295642, EPI_ISL_11296415, EPI_ISL_11349763, EPI_ISL_11356267, EPI_ISL_11356269, EPI_ISL_11403393, EPI_ISL_11424263, EPI_ISL_11424578, EPI_ISL_11437359, EPI_ISL_11482304, EPI_ISL_11503909, EPI_ISL_11504189, EPI_ISL_11517385, EPI_ISL_11565840, EPI_ISL_11576757, EPI_ISL_11621597, EPI_ISL_11621883, EPI_ISL_11657715, EPI_ISL_11661806, EPI_ISL_11695384, EPI_ISL_11742572, EPI_ISL_11742812, EPI_ISL_11747289, EPI_ISL_11787443, EPI_ISL_11789090, EPI_ISL_11798407, EPI_ISL_11798458, EPI_ISL_11816904, EPI_ISL_11825798, EPI_ISL_11826326, EPI_ISL_11826898, EPI_ISL_11871462, EPI_ISL_11886470, EPI_ISL_11886479, EPI_ISL_11886624, EPI_ISL_11897546, EPI_ISL_11898000, EPI_ISL_11919916, EPI_ISL_11961223, EPI_ISL_11968830, EPI_ISL_11968876, EPI_ISL_11970393, EPI_ISL_11976211, EPI_ISL_11992954, EPI_ISL_11995938, EPI_ISL_12001179, EPI_ISL_12001180, EPI_ISL_12006455, EPI_ISL_12021469, EPI_ISL_12039060, EPI_ISL_12060087, EPI_ISL_12063598, EPI_ISL_12063600, EPI_ISL_12063601, EPI_ISL_12063602, EPI_ISL_12079999, EPI_ISL_12083619, EPI_ISL_12089943, EPI_ISL_12097355, EPI_ISL_12108965, EPI_ISL_12127282, EPI_ISL_12139045, EPI_ISL_12139066, EPI_ISL_12145506, EPI_ISL_12146396, EPI_ISL_12146579, EPI_ISL_12146859, EPI_ISL_12146865, EPI_ISL_12146872, EPI_ISL_12147521, EPI_ISL_12148419, EPI_ISL_12148503, EPI_ISL_12150077, EPI_ISL_12150259, EPI_ISL_12150361, EPI_ISL_12150484, EPI_ISL_12155759, EPI_ISL_12155809, EPI_ISL_12157165, EPI_ISL_12157166, EPI_ISL_12157187, EPI_ISL_12158960, EPI_ISL_12162547, EPI_ISL_12168418, EPI_ISL_12171333, EPI_ISL_12171674, EPI_ISL_12172132, EPI_ISL_12172842, EPI_ISL_12173486, EPI_ISL_12173730, EPI_ISL_12173879, EPI_ISL_12174612, EPI_ISL_12174734, EPI_ISL_12174735, EPI_ISL_12174736, EPI_ISL_12174739, EPI_ISL_12174942, EPI_ISL_12175020, EPI_ISL_12175024, EPI_ISL_12175203, EPI_ISL_12176184, EPI_ISL_12207682, EPI_ISL_12220762, EPI_ISL_12240087, EPI_ISL_12240983, EPI_ISL_12251561, EPI_ISL_12259859, EPI_ISL_12261629, EPI_ISL_12268492, EPI_ISL_12278477, EPI_ISL_12278678, EPI_ISL_12278997, EPI_ISL_12284821, EPI_ISL_12296790, EPI_ISL_12296891, EPI_ISL_12310532, EPI_ISL_12323992, EPI_ISL_12325408, EPI_ISL_12350967, EPI_ISL_12351281, EPI_ISL_12355622, EPI_ISL_12371147, EPI_ISL_12397635, EPI_ISL_12401928, EPI_ISL_12422504, EPI_ISL_12423062, EPI_ISL_12425033, EPI_ISL_12430022, EPI_ISL_12444737, EPI_ISL_12447557, EPI_ISL_12454820, EPI_ISL_12454840, EPI_ISL_12456735, EPI_ISL_12464077, EPI_ISL_12467081, EPI_ISL_12467157, EPI_ISL_12473693, EPI_ISL_12475004, EPI_ISL_12480203, EPI_ISL_12486436, EPI_ISL_12486761, EPI_ISL_12488441, EPI_ISL_12495863, EPI_ISL_12501519, EPI_ISL_12511246, EPI_ISL_12530780, EPI_ISL_12531462, EPI_ISL_12535815, EPI_ISL_12539663, EPI_ISL_12568162, EPI_ISL_12568208, EPI_ISL_12575298, EPI_ISL_12579825, EPI_ISL_12581914, EPI_ISL_12590958, EPI_ISL_12602903, EPI_ISL_12611697, EPI_ISL_12611721, EPI_ISL_12616586, EPI_ISL_12622901, EPI_ISL_12622902, EPI_ISL_12623284, EPI_ISL_12628520, EPI_ISL_12639714, EPI_ISL_12639788, EPI_ISL_12639917, EPI_ISL_12645823, EPI_ISL_12646116, EPI_ISL_12646361, EPI_ISL_12646785, EPI_ISL_12647336, EPI_ISL_12652423, EPI_ISL_12654179, EPI_ISL_12656485, EPI_ISL_12661097, EPI_ISL_12663222, EPI_ISL_12664764, EPI_ISL_12667249, EPI_ISL_12680798, EPI_ISL_12685124, EPI_ISL_12685126, EPI_ISL_12691923, EPI_ISL_12698937, EPI_ISL_12701772, EPI_ISL_12701820, EPI_ISL_12701858, EPI_ISL_12701871, EPI_ISL_12701895, EPI_ISL_12703517, EPI_ISL_12707647, EPI_ISL_12715944, EPI_ISL_12717870, EPI_ISL_12739425, EPI_ISL_12741485, EPI_ISL_12742126, EPI_ISL_12742222, EPI_ISL_12754976, EPI_ISL_12763802, EPI_ISL_12765459, EPI_ISL_12784028, EPI_ISL_12789812, EPI_ISL_12789846, EPI_ISL_12808264, EPI_ISL_12809016, EPI_ISL_12809684, EPI_ISL_12822481, EPI_ISL_12822483, EPI_ISL_12829036, EPI_ISL_12830215, EPI_ISL_12830827, EPI_ISL_12844170, EPI_ISL_12851188, EPI_ISL_12851233, EPI_ISL_12851285, EPI_ISL_12862705, EPI_ISL_12862970, EPI_ISL_12871249, EPI_ISL_12879252, EPI_ISL_12881932, EPI_ISL_12892238, EPI_ISL_12892312, EPI_ISL_12892482, EPI_ISL_12896994, EPI_ISL_12903760, EPI_ISL_12906172, EPI_ISL_12911895, EPI_ISL_12911898, EPI_ISL_12914019, EPI_ISL_12926955, EPI_ISL_12932770, EPI_ISL_12942108, EPI_ISL_12955795, EPI_ISL_12958668, EPI_ISL_12961699, EPI_ISL_12968380, EPI_ISL_12980420, EPI_ISL_12993020, EPI_ISL_12995123, EPI_ISL_12995422, EPI_ISL_12995533, EPI_ISL_13000815, EPI_ISL_13001148, EPI_ISL_13002153, EPI_ISL_13002802, EPI_ISL_13002811, EPI_ISL_13011225, EPI_ISL_13011226, EPI_ISL_13019919, EPI_ISL_13026163, EPI_ISL_13028133, EPI_ISL_13040401, EPI_ISL_13044133, EPI_ISL_13047387, EPI_ISL_13050078, EPI_ISL_13050793, EPI_ISL_13051431, EPI_ISL_13051740, EPI_ISL_13051970, EPI_ISL_13052096, EPI_ISL_13052204, EPI_ISL_13055324, EPI_ISL_13056142, EPI_ISL_13056190, EPI_ISL_13065554, EPI_ISL_13066396, EPI_ISL_13066472, EPI_ISL_13066603, EPI_ISL_13073691, EPI_ISL_13085784, EPI_ISL_13085991, EPI_ISL_13086417, EPI_ISL_13086737, EPI_ISL_13086826, EPI_ISL_13086831, EPI_ISL_13088769, EPI_ISL_13088942, EPI_ISL_13089020, EPI_ISL_13089384, EPI_ISL_13091908, EPI_ISL_13091912, EPI_ISL_13091925, EPI_ISL_13091929, EPI_ISL_13092725, EPI_ISL_13092906, EPI_ISL_13093369, EPI_ISL_13093922, EPI_ISL_13094368, EPI_ISL_13108591, EPI_ISL_13110029, EPI_ISL_13110031, EPI_ISL_13123133, EPI_ISL_13129387, EPI_ISL_13131091, EPI_ISL_13131118, EPI_ISL_13131514, EPI_ISL_13132070, EPI_ISL_13133128, EPI_ISL_13133359, EPI_ISL_13152570, EPI_ISL_13157537, EPI_ISL_13157638, EPI_ISL_13158312, EPI_ISL_13158314, EPI_ISL_13160040, EPI_ISL_13166402, EPI_ISL_13166803, EPI_ISL_13172328, EPI_ISL_13172329, EPI_ISL_13176279, EPI_ISL_13176281, EPI_ISL_13178754, EPI_ISL_13183984, EPI_ISL_13187136, EPI_ISL_13192066, EPI_ISL_13192072, EPI_ISL_13192202, EPI_ISL_13199746, EPI_ISL_13199759, EPI_ISL_13202578, EPI_ISL_13203425, EPI_ISL_13210987, EPI_ISL_13214299, EPI_ISL_13215742, EPI_ISL_13230467, EPI_ISL_13242111, EPI_ISL_13242155, EPI_ISL_13244164, EPI_ISL_13244243, EPI_ISL_13244707, EPI_ISL_13251406, EPI_ISL_13251514, EPI_ISL_13251524, EPI_ISL_13253132, EPI_ISL_13253164, EPI_ISL_13253244, EPI_ISL_13253296, EPI_ISL_13253344, EPI_ISL_13253353, EPI_ISL_13253416, EPI_ISL_13253427, EPI_ISL_13253489, EPI_ISL_13253493, EPI_ISL_13253550, EPI_ISL_13253551, EPI_ISL_13253570, EPI_ISL_13264387, EPI_ISL_13271922, EPI_ISL_13272001, EPI_ISL_13272223, EPI_ISL_13273550, EPI_ISL_13284168, EPI_ISL_13289213, EPI_ISL_13289619, EPI_ISL_13289774, EPI_ISL_13294595, EPI_ISL_13299108, EPI_ISL_13299118, EPI_ISL_13299119, EPI_ISL_13299122, EPI_ISL_13301357, EPI_ISL_13301368, EPI_ISL_13304429, EPI_ISL_13304451, EPI_ISL_13312837, EPI_ISL_13314960, EPI_ISL_13317150, EPI_ISL_13317160, EPI_ISL_13318832, EPI_ISL_13319769, EPI_ISL_13320180, EPI_ISL_13322028, EPI_ISL_13322954, EPI_ISL_13322975, EPI_ISL_13323290, EPI_ISL_13328732, EPI_ISL_13329792, EPI_ISL_13332433, EPI_ISL_13332459, EPI_ISL_13333047, EPI_ISL_13339980, EPI_ISL_13345908, EPI_ISL_13349432, EPI_ISL_13350581, EPI_ISL_13351788, EPI_ISL_13354243, EPI_ISL_13354366, EPI_ISL_13356160, EPI_ISL_13356194, EPI_ISL_13358749, EPI_ISL_13358809, EPI_ISL_13358893, EPI_ISL_13358894, EPI_ISL_13361313, EPI_ISL_13361419, EPI_ISL_13362130, EPI_ISL_13368501, EPI_ISL_13368552, EPI_ISL_13369326, EPI_ISL_13370722, EPI_ISL_13371824, EPI_ISL_13376289, EPI_ISL_13385215, EPI_ISL_13386427, EPI_ISL_13389863, EPI_ISL_13389864, EPI_ISL_13394010, EPI_ISL_13398372, EPI_ISL_13398391, EPI_ISL_13400530, EPI_ISL_13405240, EPI_ISL_13406133, EPI_ISL_13407391, EPI_ISL_13407748, EPI_ISL_13407802, EPI_ISL_13408380, EPI_ISL_13410054, EPI_ISL_13410128, EPI_ISL_13412509, EPI_ISL_13417637, EPI_ISL_13422063, EPI_ISL_13425805, EPI_ISL_13426860, EPI_ISL_13426861, EPI_ISL_13426862, EPI_ISL_13440246, EPI_ISL_13440269, EPI_ISL_13440488, EPI_ISL_13443257, EPI_ISL_13449356, EPI_ISL_13455972, EPI_ISL_13463270, EPI_ISL_13466629, EPI_ISL_13466644, EPI_ISL_13467321, EPI_ISL_13467676, EPI_ISL_13470130, EPI_ISL_13470158, EPI_ISL_13476507, EPI_ISL_13477158, EPI_ISL_13478276, EPI_ISL_13478413, EPI_ISL_13478421, EPI_ISL_13478425, EPI_ISL_13478448, EPI_ISL_13479636, EPI_ISL_13480851, EPI_ISL_13481704, EPI_ISL_13482848, EPI_ISL_13483067, EPI_ISL_13483538, EPI_ISL_13502856, EPI_ISL_13502894, EPI_ISL_13503354, EPI_ISL_13503362, EPI_ISL_13503413, EPI_ISL_13504084, EPI_ISL_13504103, EPI_ISL_13504568, EPI_ISL_13518485, EPI_ISL_13519541, EPI_ISL_13522554, EPI_ISL_13522707, EPI_ISL_13529086, EPI_ISL_13535844, EPI_ISL_13536846, EPI_ISL_13536992, EPI_ISL_13538620, EPI_ISL_13538621, EPI_ISL_13539746, EPI_ISL_13552269, EPI_ISL_13560139, EPI_ISL_13563849, EPI_ISL_13563900, EPI_ISL_13564453, EPI_ISL_13564901, EPI_ISL_13566717, EPI_ISL_13571026, EPI_ISL_13571629, EPI_ISL_13572579, EPI_ISL_13572829, EPI_ISL_13573707, EPI_ISL_13577432, EPI_ISL_13578717, EPI_ISL_13585850, EPI_ISL_13592668, EPI_ISL_13605240, EPI_ISL_13605930, EPI_ISL_13608346, EPI_ISL_13611373, EPI_ISL_13612632, EPI_ISL_13614328, EPI_ISL_13615804, EPI_ISL_13615826, EPI_ISL_13616458, EPI_ISL_13617390, EPI_ISL_13617475, EPI_ISL_13617493, EPI_ISL_13619646, EPI_ISL_13622657, EPI_ISL_13622831, EPI_ISL_13622874, EPI_ISL_13624440, EPI_ISL_13624441, EPI_ISL_13626226, EPI_ISL_13633558, EPI_ISL_13633729, EPI_ISL_13636934, EPI_ISL_13637141, EPI_ISL_13637734, EPI_ISL_13638494, EPI_ISL_13653660, EPI_ISL_13665096, EPI_ISL_13665110, EPI_ISL_13666764, EPI_ISL_13677872, EPI_ISL_13688333, EPI_ISL_13691966, EPI_ISL_13692397, EPI_ISL_13694663, EPI_ISL_13698648, EPI_ISL_13700128, EPI_ISL_13700243, EPI_ISL_13700756, EPI_ISL_13701029, EPI_ISL_13701810, EPI_ISL_13715129, EPI_ISL_13715746, EPI_ISL_13732799, EPI_ISL_13734474, EPI_ISL_13734683, EPI_ISL_13738059, EPI_ISL_13740111, EPI_ISL_13740674, EPI_ISL_13741330, EPI_ISL_13744798, EPI_ISL_13744799, EPI_ISL_13745638, EPI_ISL_13745641, EPI_ISL_13748166, EPI_ISL_13750726, EPI_ISL_13750730, EPI_ISL_13750760, EPI_ISL_13750771, EPI_ISL_13750936, EPI_ISL_13750937, EPI_ISL_13751254, EPI_ISL_13757795, EPI_ISL_13757902, EPI_ISL_13757914, EPI_ISL_13759674, EPI_ISL_13759811, EPI_ISL_13760929, EPI_ISL_13762803, EPI_ISL_13764852, EPI_ISL_13765234, EPI_ISL_13776103, EPI_ISL_13776118, EPI_ISL_13778691, EPI_ISL_13788917, EPI_ISL_13794761, EPI_ISL_13795106, EPI_ISL_13795308, EPI_ISL_13802466, EPI_ISL_13805350, EPI_ISL_13806026, EPI_ISL_13806197, EPI_ISL_13812067, EPI_ISL_13826362, EPI_ISL_13830194, EPI_ISL_13830195, EPI_ISL_13830196, EPI_ISL_13830197, EPI_ISL_13830445, EPI_ISL_13839105, EPI_ISL_13839285, EPI_ISL_13842068, EPI_ISL_13844161, EPI_ISL_13850726, EPI_ISL_13855446, EPI_ISL_13856866, EPI_ISL_13858143, EPI_ISL_13858664, EPI_ISL_13860426, EPI_ISL_13860879, EPI_ISL_13866687, EPI_ISL_13866688, EPI_ISL_13866691, EPI_ISL_13871326, EPI_ISL_13873100, EPI_ISL_13875348, EPI_ISL_13875677, EPI_ISL_13875711, EPI_ISL_13876318, EPI_ISL_13876612, EPI_ISL_13876760, EPI_ISL_13881123, EPI_ISL_13884353, EPI_ISL_13884360, EPI_ISL_13884439, EPI_ISL_13889482, EPI_ISL_13891697, EPI_ISL_13892755, EPI_ISL_13893481, EPI_ISL_13896135, EPI_ISL_13896156, EPI_ISL_13896578, EPI_ISL_13900930, EPI_ISL_13905704, EPI_ISL_13907795, EPI_ISL_13908668, EPI_ISL_13913047, EPI_ISL_13915527, EPI_ISL_13915530, EPI_ISL_13925703, EPI_ISL_13931117, EPI_ISL_13931975, EPI_ISL_13931997, EPI_ISL_13932039, EPI_ISL_13932080, EPI_ISL_13937613, EPI_ISL_13939223, EPI_ISL_13947566, EPI_ISL_13948007, EPI_ISL_13951654, EPI_ISL_13958481, EPI_ISL_13958566, EPI_ISL_13963776, EPI_ISL_13963832, EPI_ISL_13970237, EPI_ISL_13970242, EPI_ISL_13970249, EPI_ISL_13970257, EPI_ISL_13970276, EPI_ISL_13975822, EPI_ISL_13981101, EPI_ISL_13984460, EPI_ISL_13986491, EPI_ISL_13986492, EPI_ISL_13986494, EPI_ISL_13986497, EPI_ISL_13986501, EPI_ISL_13991375, EPI_ISL_13993708, EPI_ISL_14000155, EPI_ISL_14004519, EPI_ISL_14005794, EPI_ISL_14010460, EPI_ISL_14010462, EPI_ISL_14011475, EPI_ISL_14015047, EPI_ISL_14019093, EPI_ISL_14019109, EPI_ISL_14019330, EPI_ISL_14020697, EPI_ISL_14022780, EPI_ISL_14022892, EPI_ISL_14023662, EPI_ISL_14027304, EPI_ISL_14027788, EPI_ISL_14027974, EPI_ISL_14028215, EPI_ISL_14030175, EPI_ISL_14032717, EPI_ISL_14036069, EPI_ISL_14041069, EPI_ISL_14044698, EPI_ISL_14044704, EPI_ISL_14046291, EPI_ISL_14047361, EPI_ISL_14050544, EPI_ISL_14051041, EPI_ISL_14057503, EPI_ISL_14062117, EPI_ISL_14064598, EPI_ISL_14064601, EPI_ISL_14066591, EPI_ISL_14066852, EPI_ISL_14071587, EPI_ISL_14071795, EPI_ISL_14097542, EPI_ISL_14097629, EPI_ISL_14124074, EPI_ISL_14128514, EPI_ISL_14134678, EPI_ISL_14135854, EPI_ISL_14147202, EPI_ISL_14155218, EPI_ISL_14158264, EPI_ISL_14161024, EPI_ISL_14170603, EPI_ISL_14171301, EPI_ISL_14172905, EPI_ISL_14173767, EPI_ISL_14175092, EPI_ISL_14175097, EPI_ISL_14175103, EPI_ISL_14175182, EPI_ISL_14178690, EPI_ISL_14193000, EPI_ISL_14196068, EPI_ISL_14197724, EPI_ISL_14198080, EPI_ISL_14200801, EPI_ISL_14203613, EPI_ISL_14208740, EPI_ISL_14209372, EPI_ISL_14209934, EPI_ISL_14210619, EPI_ISL_14215818, EPI_ISL_14216595, EPI_ISL_14222817, EPI_ISL_14223595, EPI_ISL_14224871, EPI_ISL_14225747, EPI_ISL_14228030, EPI_ISL_14229584, EPI_ISL_14230544, EPI_ISL_14231739, EPI_ISL_14231749, EPI_ISL_14231751, EPI_ISL_14232221, EPI_ISL_14234017, EPI_ISL_14241722, EPI_ISL_14243471, EPI_ISL_14243503, EPI_ISL_14243509, EPI_ISL_14253197, EPI_ISL_14257394, EPI_ISL_14257902, EPI_ISL_14257905, EPI_ISL_14259114, EPI_ISL_14259905, EPI_ISL_14260215, EPI_ISL_14261704, EPI_ISL_14262778, EPI_ISL_14263077, EPI_ISL_14277057, EPI_ISL_14287370, EPI_ISL_14288695, EPI_ISL_14289901, EPI_ISL_14289906, EPI_ISL_14292615, EPI_ISL_14292645, EPI_ISL_14292727, EPI_ISL_14292796, EPI_ISL_14296586, EPI_ISL_14298637, EPI_ISL_14301376, EPI_ISL_14305792, EPI_ISL_14311909, EPI_ISL_14311965, EPI_ISL_14312743, EPI_ISL_14321842, EPI_ISL_14340832, EPI_ISL_14353536, EPI_ISL_14359010, EPI_ISL_14382623, EPI_ISL_14387989, EPI_ISL_14389788, EPI_ISL_14389796, EPI_ISL_14391372, EPI_ISL_14393120, EPI_ISL_14393933, EPI_ISL_14409468, EPI_ISL_14416474, EPI_ISL_14425116, EPI_ISL_14425894, EPI_ISL_14426235, EPI_ISL_14426336, EPI_ISL_14430592, EPI_ISL_14433737, EPI_ISL_14436225, EPI_ISL_14437098, EPI_ISL_14439513, EPI_ISL_14439514, EPI_ISL_14439530, EPI_ISL_14439649, EPI_ISL_14439686, EPI_ISL_14440238, EPI_ISL_14448667, EPI_ISL_14455168, EPI_ISL_14459779, EPI_ISL_14462783, EPI_ISL_14464386, EPI_ISL_14467169, EPI_ISL_14470997, EPI_ISL_14471721, EPI_ISL_14478172, EPI_ISL_14478208, EPI_ISL_14479735, EPI_ISL_14483275, EPI_ISL_14485890, EPI_ISL_14487304, EPI_ISL_14487315, EPI_ISL_14490179, EPI_ISL_14493139, EPI_ISL_14493608, EPI_ISL_14493822, EPI_ISL_14493989, EPI_ISL_14494098, EPI_ISL_14494322, EPI_ISL_14496822, EPI_ISL_14497316, EPI_ISL_14498244, EPI_ISL_14503169, EPI_ISL_14503437, EPI_ISL_14504973, EPI_ISL_14505974, EPI_ISL_14507199, EPI_ISL_14507200, EPI_ISL_14509715, EPI_ISL_14513137, EPI_ISL_14518038, EPI_ISL_14518039, EPI_ISL_14518040, EPI_ISL_14518101, EPI_ISL_14518137, EPI_ISL_14523885, EPI_ISL_14527351, EPI_ISL_14527924, EPI_ISL_14535112, EPI_ISL_14540192, EPI_ISL_14544667, EPI_ISL_14545270, EPI_ISL_14551066, EPI_ISL_14552116, EPI_ISL_14553998, EPI_ISL_14554016, EPI_ISL_14554020, EPI_ISL_14554042, EPI_ISL_14554047, EPI_ISL_14556650, EPI_ISL_14560721, EPI_ISL_14560826, EPI_ISL_14560873, EPI_ISL_14561036, EPI_ISL_14561487, EPI_ISL_14561987, EPI_ISL_14562000, EPI_ISL_14562820, EPI_ISL_14571366, EPI_ISL_14571802, EPI_ISL_14572777, EPI_ISL_14574572, EPI_ISL_14577541, EPI_ISL_14577981, EPI_ISL_14578599, EPI_ISL_14588127, EPI_ISL_14596883, EPI_ISL_14599772, EPI_ISL_14599778, EPI_ISL_14599779, EPI_ISL_14599780, EPI_ISL_14602583, EPI_ISL_14602992, EPI_ISL_14606016, EPI_ISL_14608770, EPI_ISL_14610722, EPI_ISL_14610723, EPI_ISL_14610724, EPI_ISL_14610725, EPI_ISL_14610726, EPI_ISL_14610727, EPI_ISL_14610728, EPI_ISL_14610729, EPI_ISL_14610730, EPI_ISL_14610731, EPI_ISL_14610732, EPI_ISL_14610733, EPI_ISL_14610734, EPI_ISL_14613632, EPI_ISL_14613671, EPI_ISL_14616144, EPI_ISL_14616443, EPI_ISL_14616543, EPI_ISL_14616681, EPI_ISL_14618772, EPI_ISL_14619952, EPI_ISL_14623561, EPI_ISL_14623599, EPI_ISL_14624407, EPI_ISL_14625263, EPI_ISL_14630170, EPI_ISL_14647032, EPI_ISL_14652006, EPI_ISL_14665394, EPI_ISL_14666760, EPI_ISL_14666761, EPI_ISL_14666762, EPI_ISL_14666764, EPI_ISL_14666765, EPI_ISL_14666766, EPI_ISL_14667834, EPI_ISL_14676287, EPI_ISL_14681429, EPI_ISL_14683500, EPI_ISL_14687471, EPI_ISL_14691921, EPI_ISL_14694460, EPI_ISL_14695466, EPI_ISL_14699555, EPI_ISL_14700183, EPI_ISL_14700285, EPI_ISL_14701161, EPI_ISL_14701325, EPI_ISL_14702629, EPI_ISL_14706169, EPI_ISL_14707196, EPI_ISL_14707197, EPI_ISL_14707948, EPI_ISL_14710821, EPI_ISL_14710834, EPI_ISL_14711613, EPI_ISL_14711614, EPI_ISL_14715522, EPI_ISL_14721837, EPI_ISL_14721894, EPI_ISL_14725600, EPI_ISL_14727457, EPI_ISL_14728608, EPI_ISL_14728814, EPI_ISL_14732990, EPI_ISL_14744620, EPI_ISL_14744804, EPI_ISL_14744809, EPI_ISL_14745146, EPI_ISL_14746124, EPI_ISL_14746196, EPI_ISL_14746271, EPI_ISL_14746905, EPI_ISL_14746964, EPI_ISL_14747246, EPI_ISL_14747247, EPI_ISL_14747621, EPI_ISL_14752384, EPI_ISL_14754570, EPI_ISL_14755727, EPI_ISL_14755766, EPI_ISL_14755933, EPI_ISL_14760819, EPI_ISL_14762572, EPI_ISL_14763447, EPI_ISL_14763711, EPI_ISL_14765653, EPI_ISL_14766331, EPI_ISL_14766361, EPI_ISL_14766363, EPI_ISL_14766433, EPI_ISL_14770484, EPI_ISL_14771903, EPI_ISL_14772260, EPI_ISL_14773345, EPI_ISL_14773569, EPI_ISL_14773570, EPI_ISL_14773839, EPI_ISL_14780448, EPI_ISL_14785887, EPI_ISL_14788048, EPI_ISL_14789391, EPI_ISL_14789392, EPI_ISL_14789508, EPI_ISL_14791420, EPI_ISL_14793146, EPI_ISL_14793618, EPI_ISL_14796633, EPI_ISL_14804570, EPI_ISL_14806018, EPI_ISL_14806413, EPI_ISL_14809350, EPI_ISL_14811078, EPI_ISL_14812412, EPI_ISL_14813068, EPI_ISL_14813161, EPI_ISL_14813215, EPI_ISL_14813300, EPI_ISL_14813995, EPI_ISL_14815433, EPI_ISL_14816346, EPI_ISL_14817985, EPI_ISL_14832276, EPI_ISL_14832977, EPI_ISL_14836181, EPI_ISL_14837867, EPI_ISL_14838049, EPI_ISL_14841625, EPI_ISL_14842196, EPI_ISL_14845057, EPI_ISL_14846399, EPI_ISL_14846400, EPI_ISL_14847727, EPI_ISL_14856139, EPI_ISL_14859457, EPI_ISL_14859713, EPI_ISL_14859716, EPI_ISL_14859829, EPI_ISL_14862263, EPI_ISL_14863183, EPI_ISL_14886333, EPI_ISL_14888736, EPI_ISL_14890020, EPI_ISL_14891391, EPI_ISL_14891763, EPI_ISL_14891765, EPI_ISL_14892114, EPI_ISL_14892395, EPI_ISL_14892970, EPI_ISL_14896074, EPI_ISL_14901195, EPI_ISL_14901198, EPI_ISL_14901439, EPI_ISL_14901672, EPI_ISL_14901693, EPI_ISL_14903212, EPI_ISL_14904331, EPI_ISL_14907633, EPI_ISL_14912863, EPI_ISL_14919989, EPI_ISL_14920419, EPI_ISL_14921805, EPI_ISL_14922117, EPI_ISL_14922118, EPI_ISL_14922929, EPI_ISL_14922954, EPI_ISL_14924448, EPI_ISL_14925471, EPI_ISL_14925480, EPI_ISL_14925483, EPI_ISL_14925487, EPI_ISL_14926371, EPI_ISL_14931103, EPI_ISL_14934229, EPI_ISL_14934234, EPI_ISL_14935361, EPI_ISL_14935895, EPI_ISL_14935908, EPI_ISL_14935930, EPI_ISL_14935931, EPI_ISL_14937654, EPI_ISL_14937852, EPI_ISL_14937864, EPI_ISL_14942094, EPI_ISL_14942184, EPI_ISL_14942530, EPI_ISL_14943208, EPI_ISL_14943290, EPI_ISL_14946958, EPI_ISL_14949065, EPI_ISL_14950282, EPI_ISL_14951595, EPI_ISL_14951609, EPI_ISL_14951892, EPI_ISL_14952059, EPI_ISL_14952205, EPI_ISL_14952220, EPI_ISL_14953487, EPI_ISL_14955378, EPI_ISL_14959923, EPI_ISL_14960752, EPI_ISL_14960911, EPI_ISL_14961972, EPI_ISL_14962212, EPI_ISL_14962429, EPI_ISL_14962617, EPI_ISL_14971640, EPI_ISL_14975894, EPI_ISL_14977865, EPI_ISL_14980656, EPI_ISL_14992324, EPI_ISL_14993023, EPI_ISL_14994380, EPI_ISL_14995884, EPI_ISL_14995958, EPI_ISL_14997830, EPI_ISL_14999886, EPI_ISL_15003867, EPI_ISL_15005560, EPI_ISL_15010697, EPI_ISL_15012784, EPI_ISL_15013151, EPI_ISL_15013344, EPI_ISL_15014035, EPI_ISL_15014516, EPI_ISL_15015307, EPI_ISL_15017244, EPI_ISL_15018502, EPI_ISL_15022788, EPI_ISL_15024848, EPI_ISL_15026124, EPI_ISL_15026704, EPI_ISL_15030291, EPI_ISL_15030370, EPI_ISL_15032101, EPI_ISL_15036387, EPI_ISL_15038137, EPI_ISL_15040430, EPI_ISL_15040463, EPI_ISL_15040845, EPI_ISL_15040855, EPI_ISL_15040867, EPI_ISL_15047139, EPI_ISL_15048524, EPI_ISL_15050379, EPI_ISL_15051633, EPI_ISL_15058728, EPI_ISL_15071928, EPI_ISL_15072261, EPI_ISL_15072510, EPI_ISL_15072543, EPI_ISL_15072550, EPI_ISL_15072553, EPI_ISL_15072554, EPI_ISL_15072999, EPI_ISL_15075043, EPI_ISL_15075836, EPI_ISL_15076071, EPI_ISL_15077422, EPI_ISL_15078481, EPI_ISL_15083622, EPI_ISL_15084091, EPI_ISL_15085357, EPI_ISL_15085883, EPI_ISL_15085910, EPI_ISL_15086100, EPI_ISL_15086132, EPI_ISL_15086246, EPI_ISL_15088435, EPI_ISL_15088854, EPI_ISL_15093244, EPI_ISL_15093817, EPI_ISL_15093818, EPI_ISL_15094085, EPI_ISL_15096597, EPI_ISL_15096672, EPI_ISL_15098367, EPI_ISL_15101602, EPI_ISL_15107059, EPI_ISL_15107248, EPI_ISL_15107529, EPI_ISL_15108940, EPI_ISL_15108982, EPI_ISL_15109913, EPI_ISL_15111369, EPI_ISL_15114528, EPI_ISL_15114696, EPI_ISL_15115237, EPI_ISL_15116712, EPI_ISL_15118472, EPI_ISL_15118484, EPI_ISL_15119416, EPI_ISL_15125352, EPI_ISL_15126616, EPI_ISL_15129252, EPI_ISL_15135632, EPI_ISL_15137908, EPI_ISL_15137948, EPI_ISL_15140027, EPI_ISL_15140068, EPI_ISL_15145892, EPI_ISL_15145981, EPI_ISL_15156399, EPI_ISL_15160596, EPI_ISL_15161674, EPI_ISL_15169761, EPI_ISL_15169791, EPI_ISL_15170512, EPI_ISL_15171102, EPI_ISL_15172949, EPI_ISL_15173621, EPI_ISL_15175083, EPI_ISL_15177304, EPI_ISL_15177330, EPI_ISL_15177334, EPI_ISL_15178067, EPI_ISL_15178215, EPI_ISL_15184076, EPI_ISL_15184282, EPI_ISL_15184330, EPI_ISL_15191475, EPI_ISL_15191490, EPI_ISL_15191491, EPI_ISL_15191505, EPI_ISL_15191642, EPI_ISL_15191714, EPI_ISL_15191804, EPI_ISL_15192502, EPI_ISL_15193406, EPI_ISL_15195342, EPI_ISL_15195410, EPI_ISL_15195634, EPI_ISL_15195645, EPI_ISL_15195724, EPI_ISL_15198987, EPI_ISL_15210372, EPI_ISL_15210373, EPI_ISL_15211295, EPI_ISL_15211305, EPI_ISL_15213088, EPI_ISL_15215446, EPI_ISL_15215974, EPI_ISL_15216639, EPI_ISL_15217187, EPI_ISL_15218165, EPI_ISL_15222709, EPI_ISL_15229199, EPI_ISL_15231108, EPI_ISL_15236061, EPI_ISL_15236355, EPI_ISL_15241555, EPI_ISL_15242970, EPI_ISL_15247582, EPI_ISL_15248681, EPI_ISL_15250551, EPI_ISL_15251240, EPI_ISL_15251241, EPI_ISL_15251242, EPI_ISL_15251243, EPI_ISL_15254725, EPI_ISL_15256010, EPI_ISL_15257404, EPI_ISL_15268715, EPI_ISL_15268834, EPI_ISL_15273578, EPI_ISL_15275120, EPI_ISL_15275240, EPI_ISL_15277675, EPI_ISL_15278730, EPI_ISL_15278787, EPI_ISL_15279546, EPI_ISL_15279743, EPI_ISL_15284364, EPI_ISL_15284373, EPI_ISL_15284374, EPI_ISL_15284586, EPI_ISL_15286527, EPI_ISL_15287330, EPI_ISL_15292331, EPI_ISL_15294656, EPI_ISL_15296403, EPI_ISL_15305368, EPI_ISL_15305862, EPI_ISL_15306067, EPI_ISL_15307010, EPI_ISL_15307651, EPI_ISL_15310561, EPI_ISL_15312119, EPI_ISL_15314949, EPI_ISL_15317884, EPI_ISL_15322076, EPI_ISL_15322793, EPI_ISL_15325687, EPI_ISL_15325946, EPI_ISL_15328668, EPI_ISL_15330077, EPI_ISL_15330418, EPI_ISL_15331994, EPI_ISL_15332912, EPI_ISL_15333287, EPI_ISL_15335656, EPI_ISL_15335800, EPI_ISL_15338015, EPI_ISL_15338081, EPI_ISL_15340355, EPI_ISL_15341321, EPI_ISL_15347054, EPI_ISL_15348674, EPI_ISL_15348926, EPI_ISL_15348927, EPI_ISL_15348928, EPI_ISL_15348929, EPI_ISL_15349852, EPI_ISL_15352149, EPI_ISL_15354679, EPI_ISL_15354775, EPI_ISL_15357057, EPI_ISL_15357960, EPI_ISL_15362650, EPI_ISL_15363544, EPI_ISL_15367191, EPI_ISL_15368893, EPI_ISL_15370137, EPI_ISL_15370885, EPI_ISL_15370889, EPI_ISL_15376124, EPI_ISL_15376348, EPI_ISL_15376376, EPI_ISL_15379719, EPI_ISL_15380518, EPI_ISL_15384507, EPI_ISL_15384545, EPI_ISL_15385232, EPI_ISL_15385971, EPI_ISL_15387248, EPI_ISL_15387296, EPI_ISL_15387509, EPI_ISL_15387686, EPI_ISL_15388463, EPI_ISL_15389278, EPI_ISL_15392529, EPI_ISL_15393294, EPI_ISL_15393302, EPI_ISL_15396381, EPI_ISL_15403852, EPI_ISL_15403865, EPI_ISL_15403898, EPI_ISL_15405688, EPI_ISL_15408697, EPI_ISL_15409673, EPI_ISL_15415649, EPI_ISL_15420040, EPI_ISL_15420138, EPI_ISL_15420212, EPI_ISL_15420431, EPI_ISL_15420631, EPI_ISL_15423234, EPI_ISL_15424211, EPI_ISL_15424276, EPI_ISL_15424884, EPI_ISL_15427653, EPI_ISL_15434634, EPI_ISL_15434919, EPI_ISL_15435185, EPI_ISL_15436140, EPI_ISL_15436494, EPI_ISL_15436498, EPI_ISL_15436499, EPI_ISL_15442625, EPI_ISL_15442735, EPI_ISL_15446553, EPI_ISL_15456143, EPI_ISL_15462878, EPI_ISL_15463120, EPI_ISL_15471419, EPI_ISL_15471420, EPI_ISL_15472394, EPI_ISL_15472759, EPI_ISL_15473942, EPI_ISL_15473981, EPI_ISL_15473994, EPI_ISL_15476105, EPI_ISL_15476158, EPI_ISL_15476180, EPI_ISL_15476315, EPI_ISL_15476724, EPI_ISL_15479591, EPI_ISL_15481002, EPI_ISL_15486348, EPI_ISL_15490572, EPI_ISL_15490728, EPI_ISL_15492743, EPI_ISL_15492887, EPI_ISL_15494260, EPI_ISL_15494897, EPI_ISL_15495028, EPI_ISL_15495125, EPI_ISL_15496641, EPI_ISL_15501222, EPI_ISL_15502864, EPI_ISL_15505215, EPI_ISL_15505985, EPI_ISL_15506333, EPI_ISL_15507204, EPI_ISL_15507296, EPI_ISL_15507616, EPI_ISL_15509746, EPI_ISL_15509755, EPI_ISL_15509992, EPI_ISL_15511841, EPI_ISL_15511842, EPI_ISL_15511843, EPI_ISL_15513583, EPI_ISL_15513588, EPI_ISL_15513663, EPI_ISL_15514216, EPI_ISL_15514265, EPI_ISL_15514302, EPI_ISL_15514923, EPI_ISL_15523591, EPI_ISL_15528152, EPI_ISL_15528328, EPI_ISL_15528329, EPI_ISL_15528330, EPI_ISL_15528331, EPI_ISL_15528332, EPI_ISL_15528333, EPI_ISL_15528334, EPI_ISL_15535800, EPI_ISL_15537619, EPI_ISL_15538513, EPI_ISL_15538645, EPI_ISL_15541751, EPI_ISL_15542503, EPI_ISL_15549778, EPI_ISL_15549981, EPI_ISL_15550525, EPI_ISL_15557544, EPI_ISL_15558837, EPI_ISL_15568780, EPI_ISL_15579728, EPI_ISL_15579786, EPI_ISL_15580359, EPI_ISL_15580699, EPI_ISL_15581446, EPI_ISL_15581681, EPI_ISL_15581727, EPI_ISL_15581931, EPI_ISL_15581932, EPI_ISL_15581939, EPI_ISL_15582076, EPI_ISL_15582517, EPI_ISL_15585338, EPI_ISL_15587950, EPI_ISL_15588132, EPI_ISL_15594682, EPI_ISL_15595518, EPI_ISL_15598104, EPI_ISL_15598966, EPI_ISL_15602198, EPI_ISL_15604595, EPI_ISL_15607872, EPI_ISL_15608835, EPI_ISL_15609106, EPI_ISL_15609107, EPI_ISL_15610726, EPI_ISL_15610881, EPI_ISL_15612047, EPI_ISL_15612048, EPI_ISL_15612190, EPI_ISL_15614101, EPI_ISL_15614116, EPI_ISL_15614150, EPI_ISL_15614383, EPI_ISL_15614456, EPI_ISL_15614490, EPI_ISL_15616889, EPI_ISL_15617621, EPI_ISL_15617635, EPI_ISL_15619675, EPI_ISL_15626705, EPI_ISL_15626859, EPI_ISL_15626955, EPI_ISL_15628252, EPI_ISL_15630041, EPI_ISL_15631537, EPI_ISL_15635022, EPI_ISL_15636641, EPI_ISL_15637121, EPI_ISL_15637182, EPI_ISL_15638597, EPI_ISL_15639067, EPI_ISL_15642936, EPI_ISL_15642980, EPI_ISL_15649157, EPI_ISL_15650076, EPI_ISL_15650225, EPI_ISL_15653695, EPI_ISL_15654640, EPI_ISL_15656922, EPI_ISL_15659716, EPI_ISL_15659847, EPI_ISL_15659982, EPI_ISL_15661609, EPI_ISL_15666581, EPI_ISL_15666595, EPI_ISL_15666598, EPI_ISL_15667047, EPI_ISL_15669004, EPI_ISL_15671244, EPI_ISL_15671388, EPI_ISL_15671577, EPI_ISL_15671878, EPI_ISL_15671888, EPI_ISL_15673934, EPI_ISL_15675248, EPI_ISL_15678339, EPI_ISL_15681826, EPI_ISL_15685722, EPI_ISL_15687965, EPI_ISL_15688500, EPI_ISL_15688701, EPI_ISL_15692625, EPI_ISL_15693169, EPI_ISL_15693174, EPI_ISL_15693676, EPI_ISL_15700160, EPI_ISL_15705061, EPI_ISL_15712450, EPI_ISL_15712747, EPI_ISL_15719141, EPI_ISL_15719142, EPI_ISL_15719143, EPI_ISL_15721137, EPI_ISL_15721185, EPI_ISL_15721190, EPI_ISL_15723589, EPI_ISL_15725799, EPI_ISL_15728467, EPI_ISL_15728613, EPI_ISL_15728673, EPI_ISL_15729287, EPI_ISL_15729288, EPI_ISL_15729308, EPI_ISL_15729309, EPI_ISL_15729310, EPI_ISL_15729311, EPI_ISL_15729315, EPI_ISL_15729341, EPI_ISL_15729358, EPI_ISL_15731233, EPI_ISL_15731409, EPI_ISL_15732413, EPI_ISL_15736424, EPI_ISL_15739498, EPI_ISL_15739617, EPI_ISL_15742450, EPI_ISL_15743318, EPI_ISL_15743816, EPI_ISL_15754145, EPI_ISL_15754794, EPI_ISL_15760224, EPI_ISL_15760382, EPI_ISL_15760554, EPI_ISL_15760609, EPI_ISL_15760812, EPI_ISL_15761520, EPI_ISL_15761543, EPI_ISL_15761663, EPI_ISL_15763216, EPI_ISL_15764573, EPI_ISL_15765022, EPI_ISL_15768827, EPI_ISL_15773132, EPI_ISL_15773881, EPI_ISL_15773907, EPI_ISL_15776989, EPI_ISL_15779724, EPI_ISL_15780387, EPI_ISL_15781197, EPI_ISL_15781220, EPI_ISL_15785782, EPI_ISL_15786114, EPI_ISL_15786255, EPI_ISL_15789537, EPI_ISL_15790179, EPI_ISL_15790657, EPI_ISL_15790738, EPI_ISL_15791223, EPI_ISL_15791252, EPI_ISL_15792351, EPI_ISL_15793981, EPI_ISL_15797751, EPI_ISL_15798331, EPI_ISL_15801425, EPI_ISL_15801499, EPI_ISL_15801515, EPI_ISL_15815337, EPI_ISL_15815889, EPI_ISL_15818486, EPI_ISL_15820055, EPI_ISL_15820336, EPI_ISL_15822919, EPI_ISL_15824080, EPI_ISL_15824099, EPI_ISL_15824207, EPI_ISL_15826867, EPI_ISL_15829108, EPI_ISL_15832444, EPI_ISL_15837686, EPI_ISL_15837751, EPI_ISL_15837827, EPI_ISL_15838124, EPI_ISL_15839941, EPI_ISL_15840082, EPI_ISL_15843473, EPI_ISL_15843671, EPI_ISL_15844032, EPI_ISL_15844165, EPI_ISL_15845753, EPI_ISL_15845778, EPI_ISL_15845946, EPI_ISL_15846023, EPI_ISL_15846264, EPI_ISL_15849690, EPI_ISL_15850759, EPI_ISL_15850872, EPI_ISL_15853809, EPI_ISL_15853943, EPI_ISL_15856103, EPI_ISL_15856463, EPI_ISL_15856822, EPI_ISL_15857383, EPI_ISL_15860163, EPI_ISL_15864217, EPI_ISL_15864218, EPI_ISL_15865301, EPI_ISL_15865421, EPI_ISL_15865482, EPI_ISL_15866887, EPI_ISL_15874567, EPI_ISL_15878818, EPI_ISL_15882637, EPI_ISL_15883009, EPI_ISL_15887656, EPI_ISL_15895625, EPI_ISL_15896804, EPI_ISL_15896923, EPI_ISL_15897067, EPI_ISL_15897092, EPI_ISL_15898992, EPI_ISL_15900796, EPI_ISL_15910198, EPI_ISL_15911160, EPI_ISL_15912221, EPI_ISL_15912222, EPI_ISL_15912223, EPI_ISL_15912224, EPI_ISL_15914119, EPI_ISL_15916592, EPI_ISL_15920181, EPI_ISL_15920505, EPI_ISL_15920753, EPI_ISL_15920754, EPI_ISL_15920755, EPI_ISL_15924323, EPI_ISL_15924325, EPI_ISL_15926083, EPI_ISL_15926723, EPI_ISL_15928909, EPI_ISL_15929002, EPI_ISL_15929151, EPI_ISL_15932554, EPI_ISL_15934274, EPI_ISL_15937718, EPI_ISL_15938074, EPI_ISL_15941879, EPI_ISL_15941880, EPI_ISL_15945504, EPI_ISL_15949533, EPI_ISL_15953870, EPI_ISL_15958934, EPI_ISL_15959369, EPI_ISL_15961456, EPI_ISL_15962045, EPI_ISL_15965716, EPI_ISL_15966131, EPI_ISL_15966527, EPI_ISL_15969420, EPI_ISL_15969421, EPI_ISL_15969437, EPI_ISL_15969438, EPI_ISL_15970088, EPI_ISL_15970187, EPI_ISL_15976036, EPI_ISL_15982641, EPI_ISL_15984958, EPI_ISL_15992803, EPI_ISL_15998627, EPI_ISL_16001974, EPI_ISL_16001995, EPI_ISL_16003220, EPI_ISL_16003255, EPI_ISL_16005246, EPI_ISL_16005457, EPI_ISL_16006665, EPI_ISL_16007931, EPI_ISL_16008877, EPI_ISL_16012424, EPI_ISL_16012845, EPI_ISL_16013074, EPI_ISL_16013086, EPI_ISL_16015099, EPI_ISL_16017107, EPI_ISL_16018930, EPI_ISL_16019056, EPI_ISL_16019744, EPI_ISL_16019756, EPI_ISL_16024407, EPI_ISL_16024682, EPI_ISL_16027431, EPI_ISL_16027937, EPI_ISL_16029135, EPI_ISL_16029382, EPI_ISL_16029654, EPI_ISL_16030181, EPI_ISL_16033087, EPI_ISL_16033186, EPI_ISL_16034832, EPI_ISL_16036598, EPI_ISL_16039444, EPI_ISL_16043974, EPI_ISL_16044428, EPI_ISL_16045410, EPI_ISL_16045424, EPI_ISL_16046711, EPI_ISL_16050095, EPI_ISL_16054133, EPI_ISL_16054451, EPI_ISL_16054953, EPI_ISL_16054963, EPI_ISL_16055434, EPI_ISL_16055461, EPI_ISL_16055527, EPI_ISL_16055697, EPI_ISL_16055721, EPI_ISL_16057031, EPI_ISL_16060790, EPI_ISL_16062229, EPI_ISL_16064284, EPI_ISL_16066333, EPI_ISL_16068281, EPI_ISL_16068583, EPI_ISL_16068914, EPI_ISL_16073080, EPI_ISL_16073469, EPI_ISL_16073474, EPI_ISL_16074871, EPI_ISL_16075086, EPI_ISL_16075127, EPI_ISL_16076496, EPI_ISL_16077353, EPI_ISL_16079016, EPI_ISL_16080170, EPI_ISL_16080871, EPI_ISL_16082206, EPI_ISL_16091870, EPI_ISL_16099360, EPI_ISL_16101516, EPI_ISL_16102479, EPI_ISL_16102480, EPI_ISL_16104542, EPI_ISL_16109108, EPI_ISL_16111875, EPI_ISL_16113331, EPI_ISL_16113603, EPI_ISL_16113806, EPI_ISL_16114505, EPI_ISL_16114631, EPI_ISL_16115703, EPI_ISL_16116190, EPI_ISL_16116234, EPI_ISL_16116659, EPI_ISL_16116707, EPI_ISL_16119498, EPI_ISL_16119508, EPI_ISL_16119512, EPI_ISL_16119517, EPI_ISL_16119519, EPI_ISL_16119805, EPI_ISL_16131965, EPI_ISL_16131986, EPI_ISL_16131997, EPI_ISL_16136901, EPI_ISL_16137616, EPI_ISL_16151030, EPI_ISL_16151463, EPI_ISL_16151651, EPI_ISL_16153650, EPI_ISL_16153658, EPI_ISL_16153800, EPI_ISL_16157875, EPI_ISL_16158109, EPI_ISL_16158197, EPI_ISL_16158326, EPI_ISL_16158363, EPI_ISL_16158374, EPI_ISL_16160252, EPI_ISL_16160296, EPI_ISL_16160313, EPI_ISL_16165250, EPI_ISL_16167761, EPI_ISL_16178283, EPI_ISL_16178634, EPI_ISL_16178844, EPI_ISL_16179355, EPI_ISL_16180564, EPI_ISL_16180574, EPI_ISL_16181797, EPI_ISL_16181828, EPI_ISL_16181950, EPI_ISL_16183022, EPI_ISL_16190977, EPI_ISL_16191476, EPI_ISL_16196150, EPI_ISL_16196167, EPI_ISL_16197958, EPI_ISL_16201173, EPI_ISL_16204855, EPI_ISL_16215808, EPI_ISL_16218191, EPI_ISL_16219709, EPI_ISL_16219753, EPI_ISL_16221691, EPI_ISL_16224408, EPI_ISL_16230801, EPI_ISL_16231330, EPI_ISL_16231873, EPI_ISL_16233000, EPI_ISL_16233650, EPI_ISL_16233865, EPI_ISL_16234790, EPI_ISL_16235313, EPI_ISL_16235462, EPI_ISL_16235523, EPI_ISL_16235930, EPI_ISL_16240521, EPI_ISL_16244367, EPI_ISL_16244373, EPI_ISL_16244408, EPI_ISL_16244923, EPI_ISL_16245232, EPI_ISL_16245289, EPI_ISL_16245329, EPI_ISL_16245400, EPI_ISL_16245414, EPI_ISL_16245433, EPI_ISL_16245601, EPI_ISL_16245627, EPI_ISL_16247208, EPI_ISL_16247263, EPI_ISL_16247490, EPI_ISL_16247545, EPI_ISL_16247708, EPI_ISL_16251186, EPI_ISL_16251466, EPI_ISL_16251507, EPI_ISL_16251579, EPI_ISL_16257294, EPI_ISL_16259227, EPI_ISL_16264400, EPI_ISL_16265325, EPI_ISL_16268074, EPI_ISL_16270258, EPI_ISL_16271245, EPI_ISL_16271444, EPI_ISL_16271604, EPI_ISL_16273936, EPI_ISL_16281373, EPI_ISL_16284103, EPI_ISL_16284207, EPI_ISL_16284311, EPI_ISL_16287253, EPI_ISL_16287690, EPI_ISL_16290877, EPI_ISL_16293662, EPI_ISL_16312661, EPI_ISL_16327295, EPI_ISL_16331524, EPI_ISL_16334679, EPI_ISL_16336940, EPI_ISL_16338847, EPI_ISL_16338862, EPI_ISL_16343084, EPI_ISL_16343221, EPI_ISL_16344593, EPI_ISL_16348840, EPI_ISL_16348868, EPI_ISL_16351967, EPI_ISL_16354229, EPI_ISL_16355537, EPI_ISL_16356453, EPI_ISL_16356458, EPI_ISL_16356910, EPI_ISL_16358915, EPI_ISL_16359990, EPI_ISL_16360495, EPI_ISL_16365715, EPI_ISL_16368903, EPI_ISL_16369869, EPI_ISL_16370037, EPI_ISL_16374806, EPI_ISL_16375356, EPI_ISL_16378181, EPI_ISL_16379359, EPI_ISL_16380313, EPI_ISL_16381332, EPI_ISL_16381679, EPI_ISL_16384522, EPI_ISL_16385377, EPI_ISL_16385455, EPI_ISL_16385456, EPI_ISL_16391752, EPI_ISL_16394844, EPI_ISL_16394922, EPI_ISL_16395667, EPI_ISL_16398472, EPI_ISL_16399824, EPI_ISL_16400033, EPI_ISL_16400342, EPI_ISL_16402001, EPI_ISL_16402612, EPI_ISL_16413336, EPI_ISL_16414127, EPI_ISL_16414190, EPI_ISL_16422834, EPI_ISL_16424130, EPI_ISL_16425691, EPI_ISL_16428100, EPI_ISL_16428101, EPI_ISL_16428102, EPI_ISL_16429066, EPI_ISL_16434308, EPI_ISL_16435903, EPI_ISL_16436533, EPI_ISL_16439413, EPI_ISL_16440696, EPI_ISL_16443129, EPI_ISL_16443688, EPI_ISL_16443874, EPI_ISL_16444178, EPI_ISL_16444600, EPI_ISL_16445144, EPI_ISL_16446802, EPI_ISL_16446804, EPI_ISL_16452054, EPI_ISL_16454044, EPI_ISL_16460278, EPI_ISL_16460823, EPI_ISL_16461302, EPI_ISL_16464657, EPI_ISL_16467436, EPI_ISL_16470710, EPI_ISL_16470892, EPI_ISL_16471153, EPI_ISL_16471554, EPI_ISL_16471730, EPI_ISL_16471943, EPI_ISL_16473435, EPI_ISL_16473762, EPI_ISL_16474400, EPI_ISL_16475138, EPI_ISL_16475139, EPI_ISL_16479826, EPI_ISL_16482060, EPI_ISL_16489594, EPI_ISL_16491494, EPI_ISL_16492150, EPI_ISL_16492585, EPI_ISL_16492756, EPI_ISL_16493396, EPI_ISL_16493785, EPI_ISL_16495643, EPI_ISL_16497702, EPI_ISL_16498515, EPI_ISL_16503949, EPI_ISL_16507701, EPI_ISL_16507896, EPI_ISL_16507927, EPI_ISL_16520597, EPI_ISL_16520598, EPI_ISL_16520637, EPI_ISL_16520640, EPI_ISL_16520641, EPI_ISL_16520711, EPI_ISL_16520763, EPI_ISL_16520788, EPI_ISL_16521101, EPI_ISL_16524906, EPI_ISL_16528641, EPI_ISL_16528645, EPI_ISL_16528786, EPI_ISL_16528903, EPI_ISL_16535376, EPI_ISL_16536212, EPI_ISL_16539692, EPI_ISL_16541774, EPI_ISL_16542381, EPI_ISL_16542553, EPI_ISL_16543095, EPI_ISL_16544506, EPI_ISL_16553339, EPI_ISL_16553795, EPI_ISL_16567779, EPI_ISL_16574574, EPI_ISL_16577679, EPI_ISL_16577762, EPI_ISL_16581578, EPI_ISL_16583782, EPI_ISL_16584104, EPI_ISL_16585126, EPI_ISL_16585286, EPI_ISL_16586683, EPI_ISL_16586702, EPI_ISL_16587574, EPI_ISL_16597363, EPI_ISL_16598790, EPI_ISL_16598814, EPI_ISL_16599846, EPI_ISL_16607452, EPI_ISL_16607500, EPI_ISL_16607535, EPI_ISL_16611498, EPI_ISL_16611571, EPI_ISL_16613287, EPI_ISL_16613482, EPI_ISL_16615201, EPI_ISL_16615597, EPI_ISL_16615617, EPI_ISL_16615668, EPI_ISL_16616642, EPI_ISL_16625690, EPI_ISL_16626611, EPI_ISL_16626666, EPI_ISL_16627067, EPI_ISL_16628854, EPI_ISL_16630260, EPI_ISL_16630261, EPI_ISL_16636917, EPI_ISL_16637089, EPI_ISL_16637607, EPI_ISL_16637631, EPI_ISL_16638190, EPI_ISL_16638453, EPI_ISL_16640597, EPI_ISL_16643406, EPI_ISL_16647535, EPI_ISL_16647537, EPI_ISL_16647544, EPI_ISL_16647740, EPI_ISL_16649988, EPI_ISL_16650006, EPI_ISL_16650011, EPI_ISL_16653618, EPI_ISL_16669313, EPI_ISL_16669829, EPI_ISL_16672282, EPI_ISL_16672301, EPI_ISL_16672327, EPI_ISL_16672352, EPI_ISL_16676267, EPI_ISL_16677015, EPI_ISL_16678917, EPI_ISL_16678946, EPI_ISL_16679654, EPI_ISL_16681451, EPI_ISL_16681917, EPI_ISL_16682342, EPI_ISL_16682837, EPI_ISL_16688219, EPI_ISL_16688525, EPI_ISL_16688688, EPI_ISL_16688713, EPI_ISL_16689887, EPI_ISL_16691397, EPI_ISL_16691487, EPI_ISL_16694176, EPI_ISL_16695383, EPI_ISL_16695435, EPI_ISL_16697861, EPI_ISL_16700160, EPI_ISL_16702838, EPI_ISL_16705882, EPI_ISL_16706498, EPI_ISL_16708209, EPI_ISL_16708798, EPI_ISL_16711038, EPI_ISL_16711095, EPI_ISL_16711417, EPI_ISL_16711531, EPI_ISL_16716967, EPI_ISL_16721930, EPI_ISL_16722183, EPI_ISL_16722215, EPI_ISL_16722270, EPI_ISL_16722970, EPI_ISL_16723215, EPI_ISL_16725887, EPI_ISL_16727241, EPI_ISL_16728257, EPI_ISL_16728383, EPI_ISL_16728411, EPI_ISL_16729998, EPI_ISL_16730248, EPI_ISL_16731753, EPI_ISL_16735576, EPI_ISL_16735633, EPI_ISL_16736400, EPI_ISL_16738641, EPI_ISL_16739045, EPI_ISL_16739452, EPI_ISL_16740104, EPI_ISL_16740406, EPI_ISL_16741567, EPI_ISL_16741573, EPI_ISL_16743490, EPI_ISL_16749999, EPI_ISL_16750878, EPI_ISL_16751721, EPI_ISL_16751722, EPI_ISL_16751789, EPI_ISL_16751791, EPI_ISL_16751977, EPI_ISL_16752073, EPI_ISL_16752138, EPI_ISL_16757168, EPI_ISL_16757210, EPI_ISL_16758981, EPI_ISL_16760201, EPI_ISL_16764861, EPI_ISL_16765888, EPI_ISL_16766196, EPI_ISL_16811091, EPI_ISL_16812565, EPI_ISL_16815494, EPI_ISL_16816293, EPI_ISL_16818458, EPI_ISL_16818471, EPI_ISL_16825124, EPI_ISL_16825157, EPI_ISL_16825222, EPI_ISL_16825237, EPI_ISL_16825238, EPI_ISL_16825239, EPI_ISL_16828876, EPI_ISL_16828896, EPI_ISL_16829188, EPI_ISL_16831507, EPI_ISL_16832953, EPI_ISL_16833893, EPI_ISL_16834974, EPI_ISL_16834985, EPI_ISL_16835399, EPI_ISL_16842787, EPI_ISL_16842790, EPI_ISL_16847425, EPI_ISL_16847642, EPI_ISL_16847674, EPI_ISL_16847675, EPI_ISL_16847676, EPI_ISL_16847677, EPI_ISL_16847696, EPI_ISL_16847697, EPI_ISL_16849243, EPI_ISL_16849244, EPI_ISL_16849245, EPI_ISL_16853227, EPI_ISL_16853229, EPI_ISL_16853597, EPI_ISL_16856355, EPI_ISL_16856565, EPI_ISL_16856637, EPI_ISL_16856833, EPI_ISL_16857514, EPI_ISL_16857776, EPI_ISL_16858617, EPI_ISL_16858667, EPI_ISL_16861084, EPI_ISL_16862130, EPI_ISL_16863260, EPI_ISL_16863802, EPI_ISL_16866580, EPI_ISL_16867553, EPI_ISL_16868647, EPI_ISL_16868654, EPI_ISL_16868655, EPI_ISL_16868993, EPI_ISL_16869007, EPI_ISL_16875565, EPI_ISL_16875752, EPI_ISL_16876039, EPI_ISL_16876784, EPI_ISL_16877428, EPI_ISL_16878720, EPI_ISL_16880439, EPI_ISL_16883240, EPI_ISL_16883873, EPI_ISL_16884622, EPI_ISL_16890808, EPI_ISL_16893283, EPI_ISL_16894717, EPI_ISL_16895138, EPI_ISL_16895290, EPI_ISL_16895522, EPI_ISL_16897500, EPI_ISL_16899216, EPI_ISL_16903492, EPI_ISL_16903494, EPI_ISL_16904444, EPI_ISL_16904536, EPI_ISL_16908446, EPI_ISL_16908447, EPI_ISL_16908450, EPI_ISL_16908472, EPI_ISL_16909804, EPI_ISL_16910025, EPI_ISL_16910165, EPI_ISL_16910272, EPI_ISL_16911226, EPI_ISL_16911909, EPI_ISL_16913144, EPI_ISL_16921530, EPI_ISL_16921542, EPI_ISL_16925257, EPI_ISL_16927736, EPI_ISL_16931901, EPI_ISL_16939487, EPI_ISL_16941750, EPI_ISL_16941980, EPI_ISL_16942000, EPI_ISL_16942017, EPI_ISL_16945429, EPI_ISL_16946783, EPI_ISL_16947592, EPI_ISL_16947625, EPI_ISL_16951592, EPI_ISL_16953425, EPI_ISL_16953741, EPI_ISL_16954486, EPI_ISL_16955471, EPI_ISL_16957015, EPI_ISL_16966140, EPI_ISL_16966997, EPI_ISL_16967082, EPI_ISL_16967083, EPI_ISL_16967084, EPI_ISL_16967085, EPI_ISL_16967086, EPI_ISL_16969756, EPI_ISL_16969757, EPI_ISL_16970279, EPI_ISL_16973343, EPI_ISL_16977317, EPI_ISL_16977653, EPI_ISL_16977749, EPI_ISL_16979306, EPI_ISL_16979482, EPI_ISL_16980075, EPI_ISL_16980683, EPI_ISL_16981030, EPI_ISL_16981047, EPI_ISL_16981048, EPI_ISL_16981102, EPI_ISL_16985060, EPI_ISL_16987088, EPI_ISL_16987375, EPI_ISL_16987376, EPI_ISL_16988527, EPI_ISL_16989157, EPI_ISL_16989160, EPI_ISL_16995247, EPI_ISL_16995491, EPI_ISL_16995525, EPI_ISL_16995951, EPI_ISL_16997638, EPI_ISL_17001974, EPI_ISL_17001987, EPI_ISL_17002442, EPI_ISL_17006258, EPI_ISL_17008393, EPI_ISL_17008502, EPI_ISL_17016219, EPI_ISL_17016344, EPI_ISL_17018731, EPI_ISL_17020636, EPI_ISL_17021187, EPI_ISL_17022063, EPI_ISL_17022081, EPI_ISL_17024099, EPI_ISL_17025560, EPI_ISL_17025998, EPI_ISL_17026052, EPI_ISL_17026537, EPI_ISL_17027430, EPI_ISL_17032070, EPI_ISL_17032664, EPI_ISL_17035345, EPI_ISL_17036551, EPI_ISL_17037388, EPI_ISL_17040117, EPI_ISL_17040120, EPI_ISL_17040127, EPI_ISL_17040130, EPI_ISL_17040131, EPI_ISL_17040132, EPI_ISL_17040133, EPI_ISL_17040134, EPI_ISL_17040184, EPI_ISL_17041089, EPI_ISL_17041105, EPI_ISL_17041117, EPI_ISL_17041143, EPI_ISL_17044002, EPI_ISL_17046406, EPI_ISL_17047368, EPI_ISL_17047667, EPI_ISL_17050958, EPI_ISL_17051743, EPI_ISL_17056159, EPI_ISL_17057279, EPI_ISL_17061716, EPI_ISL_17062081, EPI_ISL_17065016, EPI_ISL_17067007, EPI_ISL_17068616, EPI_ISL_17068621, EPI_ISL_17073286, EPI_ISL_17074720, EPI_ISL_17075299, EPI_ISL_17076011, EPI_ISL_17076926, EPI_ISL_17077233, EPI_ISL_17077446, EPI_ISL_17079150, EPI_ISL_17079151, EPI_ISL_17079427, EPI_ISL_17080036, EPI_ISL_17080146, EPI_ISL_17080283, EPI_ISL_17080510, EPI_ISL_17081567, EPI_ISL_17084330, EPI_ISL_17086936, EPI_ISL_17086958, EPI_ISL_17090268, EPI_ISL_17092242, EPI_ISL_17092260, EPI_ISL_17099321, EPI_ISL_17099446, EPI_ISL_17101049, EPI_ISL_17101231, EPI_ISL_17104807, EPI_ISL_17105674, EPI_ISL_17105677, EPI_ISL_17105765, EPI_ISL_17105777, EPI_ISL_17105786, EPI_ISL_17105789, EPI_ISL_17105804, EPI_ISL_17106895, EPI_ISL_17109738, EPI_ISL_17109787, EPI_ISL_17112198, EPI_ISL_17113114, EPI_ISL_17118740, EPI_ISL_17119370, EPI_ISL_17126699, EPI_ISL_17126727, EPI_ISL_17127510, EPI_ISL_17129671, EPI_ISL_17139969, EPI_ISL_17144489, EPI_ISL_17149673, EPI_ISL_17149697, EPI_ISL_17150312, EPI_ISL_17152522, EPI_ISL_17152602, EPI_ISL_17152816, EPI_ISL_17154843, EPI_ISL_17154893, EPI_ISL_17158601, EPI_ISL_17158659, EPI_ISL_17158660, EPI_ISL_17158661, EPI_ISL_17158662, EPI_ISL_17158663, EPI_ISL_17158664, EPI_ISL_17158665, EPI_ISL_17164529, EPI_ISL_17165387, EPI_ISL_17165528, EPI_ISL_17170921, EPI_ISL_17173754, EPI_ISL_17174263, EPI_ISL_17174278, EPI_ISL_17174323, EPI_ISL_17175107, EPI_ISL_17180776, EPI_ISL_17182281, EPI_ISL_17188691, EPI_ISL_17188772, EPI_ISL_17188836, EPI_ISL_17189286, EPI_ISL_17189360, EPI_ISL_17189372, EPI_ISL_17189865, EPI_ISL_17190813, EPI_ISL_17191784, EPI_ISL_17193988, EPI_ISL_17194121, EPI_ISL_17194564, EPI_ISL_17195276, EPI_ISL_17195807, EPI_ISL_17198415, EPI_ISL_17199165, EPI_ISL_17199381, EPI_ISL_17199743, EPI_ISL_17200348, EPI_ISL_17200520, EPI_ISL_17201694, EPI_ISL_17202051, EPI_ISL_17205892, EPI_ISL_17206016, EPI_ISL_17206140, EPI_ISL_17207424, EPI_ISL_17210230, EPI_ISL_17210689, EPI_ISL_17214413, EPI_ISL_17214693, EPI_ISL_17214774, EPI_ISL_17214805, EPI_ISL_17214933, EPI_ISL_17215427, EPI_ISL_17215676, EPI_ISL_17215686, EPI_ISL_17215790, EPI_ISL_17216822, EPI_ISL_17216978, EPI_ISL_17221710, EPI_ISL_17222365, EPI_ISL_17223438, EPI_ISL_17226531, EPI_ISL_17226637, EPI_ISL_17232350, EPI_ISL_17232448, EPI_ISL_17237921, EPI_ISL_17239049, EPI_ISL_17239405, EPI_ISL_17239499, EPI_ISL_17241376, EPI_ISL_17244630, EPI_ISL_17244668, EPI_ISL_17245140, EPI_ISL_17245198, EPI_ISL_17245226, EPI_ISL_17245255, EPI_ISL_17246876, EPI_ISL_17246931, EPI_ISL_17247186, EPI_ISL_17247325, EPI_ISL_17247333, EPI_ISL_17247834, EPI_ISL_17248435, EPI_ISL_17251028, EPI_ISL_17252934, EPI_ISL_17253364, EPI_ISL_17253589, EPI_ISL_17257608, EPI_ISL_17262137, EPI_ISL_17265160, EPI_ISL_17270165, EPI_ISL_17270215, EPI_ISL_17270470, EPI_ISL_17270618, EPI_ISL_17270950, EPI_ISL_17270964, EPI_ISL_17270974, EPI_ISL_17271226, EPI_ISL_17271272, EPI_ISL_17272946, EPI_ISL_17275616, EPI_ISL_17275984, EPI_ISL_17276025, EPI_ISL_17276030, EPI_ISL_17276098, EPI_ISL_17276962, EPI_ISL_17284010, EPI_ISL_17284045, EPI_ISL_17284573, EPI_ISL_17284790, EPI_ISL_17284791, EPI_ISL_17284792, EPI_ISL_17284793, EPI_ISL_17284794, EPI_ISL_17284795, EPI_ISL_17284796, EPI_ISL_17285690, EPI_ISL_17288589, EPI_ISL_17290740, EPI_ISL_17292666, EPI_ISL_17292834, EPI_ISL_17297993, EPI_ISL_17298321, EPI_ISL_17298323, EPI_ISL_17299688, EPI_ISL_17300150, EPI_ISL_17304801, EPI_ISL_17304899, EPI_ISL_17305358, EPI_ISL_17319411, EPI_ISL_17319528, EPI_ISL_17319601, EPI_ISL_17321362, EPI_ISL_17322993, EPI_ISL_17334027, EPI_ISL_17342544, EPI_ISL_17344004, EPI_ISL_17344178, EPI_ISL_17344660, EPI_ISL_17345445, EPI_ISL_17347577, EPI_ISL_17348219, EPI_ISL_17349770, EPI_ISL_17349983, EPI_ISL_17350301, EPI_ISL_17352192, EPI_ISL_17358767, EPI_ISL_17359772, EPI_ISL_17370155, EPI_ISL_17371048, EPI_ISL_17374170, EPI_ISL_17374605, EPI_ISL_17374609, EPI_ISL_17374807, EPI_ISL_17376230, EPI_ISL_17381216, EPI_ISL_17387122, EPI_ISL_17389140, EPI_ISL_17389210, EPI_ISL_17389223, EPI_ISL_17389779, EPI_ISL_17390660, EPI_ISL_17390743, EPI_ISL_17390873, EPI_ISL_17391460, EPI_ISL_17394837, EPI_ISL_17396870, EPI_ISL_17397497, EPI_ISL_17397582, EPI_ISL_17408352, EPI_ISL_17409157, EPI_ISL_17411543, EPI_ISL_17414235, EPI_ISL_17414543, EPI_ISL_17418615, EPI_ISL_17421962, EPI_ISL_17423074, EPI_ISL_17424014, EPI_ISL_17429770, EPI_ISL_17430458, EPI_ISL_17430487, EPI_ISL_17431229, EPI_ISL_17431238, EPI_ISL_17434223, EPI_ISL_17434227, EPI_ISL_17437940, EPI_ISL_17440507, EPI_ISL_17441169, EPI_ISL_17441208, EPI_ISL_17441815, EPI_ISL_17445401, EPI_ISL_17446132, EPI_ISL_17464711, EPI_ISL_17466081, EPI_ISL_17470229, EPI_ISL_17470269, EPI_ISL_17471181, EPI_ISL_17471185, EPI_ISL_17471619, EPI_ISL_17471674, EPI_ISL_17471824, EPI_ISL_17472531, EPI_ISL_17475799, EPI_ISL_17476568, EPI_ISL_17476871, EPI_ISL_17477106, EPI_ISL_17480516, EPI_ISL_17481180, EPI_ISL_17481517, EPI_ISL_17481521, EPI_ISL_17481597, EPI_ISL_17482811, EPI_ISL_17482813, EPI_ISL_17482815, EPI_ISL_17482819, EPI_ISL_17493613, EPI_ISL_17494372, EPI_ISL_17494568, EPI_ISL_17494731, EPI_ISL_17497461, EPI_ISL_17497688, EPI_ISL_17497854, EPI_ISL_17497868, EPI_ISL_17501536, EPI_ISL_17501576, EPI_ISL_17501763, EPI_ISL_17502219, EPI_ISL_17502972, EPI_ISL_17503268, EPI_ISL_17503711, EPI_ISL_17504816, EPI_ISL_17504835, EPI_ISL_17505072, EPI_ISL_17508749, EPI_ISL_17509597, EPI_ISL_17510495, EPI_ISL_17510856, EPI_ISL_17511096, EPI_ISL_17511836, EPI_ISL_17512412, EPI_ISL_17512500, EPI_ISL_17512876, EPI_ISL_17512968, EPI_ISL_17513218, EPI_ISL_17513312, EPI_ISL_17514540, EPI_ISL_17514694, EPI_ISL_17515086, EPI_ISL_17515177, EPI_ISL_17516651, EPI_ISL_17516658, EPI_ISL_17516659, EPI_ISL_17517664, EPI_ISL_17517834, EPI_ISL_17517844, EPI_ISL_17521302, EPI_ISL_17521772, EPI_ISL_17521778, EPI_ISL_17522610, EPI_ISL_17522687, EPI_ISL_17522934, EPI_ISL_17523535, EPI_ISL_17523620, EPI_ISL_17523782, EPI_ISL_17523873, EPI_ISL_17524106, EPI_ISL_17524432, EPI_ISL_17524502, EPI_ISL_17524503, EPI_ISL_17535664, EPI_ISL_17535979, EPI_ISL_17541088, EPI_ISL_17541797, EPI_ISL_17543006, EPI_ISL_17544283, EPI_ISL_17545970, EPI_ISL_17546923, EPI_ISL_17547529, EPI_ISL_17547545, EPI_ISL_17548526, EPI_ISL_17549129, EPI_ISL_17550129, EPI_ISL_17550538, EPI_ISL_17553063, EPI_ISL_17553974, EPI_ISL_17554057, EPI_ISL_17556705, EPI_ISL_17559150, EPI_ISL_17559165, EPI_ISL_17559166, EPI_ISL_17559167, EPI_ISL_17559168, EPI_ISL_17559246, EPI_ISL_17563568, EPI_ISL_17565071, EPI_ISL_17565211, EPI_ISL_17565212, EPI_ISL_17566854, EPI_ISL_17579120, EPI_ISL_17580420, EPI_ISL_17583157, EPI_ISL_17584277, EPI_ISL_17585020, EPI_ISL_17585021, EPI_ISL_17585022, EPI_ISL_17585023, EPI_ISL_17585036, EPI_ISL_17585039, EPI_ISL_17586115, EPI_ISL_17587423, EPI_ISL_17587656, EPI_ISL_17587657, EPI_ISL_17587665, EPI_ISL_17587671, EPI_ISL_17587859, EPI_ISL_17588127, EPI_ISL_17588216, EPI_ISL_17588460, EPI_ISL_17589845, EPI_ISL_17590449, EPI_ISL_17590486, EPI_ISL_17591005, EPI_ISL_17591014, EPI_ISL_17591015, EPI_ISL_17591028, EPI_ISL_17592236, EPI_ISL_17592618, EPI_ISL_17593692, EPI_ISL_17593865, EPI_ISL_17595116, EPI_ISL_17595117, EPI_ISL_17595967, EPI_ISL_17595980, EPI_ISL_17597954, EPI_ISL_17598384, EPI_ISL_17599326, EPI_ISL_17599427, EPI_ISL_17600948, EPI_ISL_17600958, EPI_ISL_17600978, EPI_ISL_17600988, EPI_ISL_17601066, EPI_ISL_17601144, EPI_ISL_17601196, EPI_ISL_17601219, EPI_ISL_17601261, EPI_ISL_17601276, EPI_ISL_17601933, EPI_ISL_17602334, EPI_ISL_17602469, EPI_ISL_17602756, EPI_ISL_17605514, EPI_ISL_17608762, EPI_ISL_17612035, EPI_ISL_17612050, EPI_ISL_17612051, EPI_ISL_17612052, EPI_ISL_17612398, EPI_ISL_17615127, EPI_ISL_17616259, EPI_ISL_17616354, EPI_ISL_17616355, EPI_ISL_17616356, EPI_ISL_17617168, EPI_ISL_17617538, EPI_ISL_17618306, EPI_ISL_17621115, EPI_ISL_17621930, EPI_ISL_17623470, EPI_ISL_17623785, EPI_ISL_17623810, EPI_ISL_17626289, EPI_ISL_17628376, EPI_ISL_17628383, EPI_ISL_17628855, EPI_ISL_17630096, EPI_ISL_17632950, EPI_ISL_17633442, EPI_ISL_17634290, EPI_ISL_17634585, EPI_ISL_17634799, EPI_ISL_17637409, EPI_ISL_17637499, EPI_ISL_17637946, EPI_ISL_17640029, EPI_ISL_17640079, EPI_ISL_17642112, EPI_ISL_17642765, EPI_ISL_17643093, EPI_ISL_17644186, EPI_ISL_17644989, EPI_ISL_17644991, EPI_ISL_17645081, EPI_ISL_17645416, EPI_ISL_17646422, EPI_ISL_17651803, EPI_ISL_17652508, EPI_ISL_17652513, EPI_ISL_17654325, EPI_ISL_17654831, EPI_ISL_17655018, EPI_ISL_17656002, EPI_ISL_17657287, EPI_ISL_17658392, EPI_ISL_17658574, EPI_ISL_17659247, EPI_ISL_17659794, EPI_ISL_17660857, EPI_ISL_17661435, EPI_ISL_17661709, EPI_ISL_17661736, EPI_ISL_17661772, EPI_ISL_17662111, EPI_ISL_17664370, EPI_ISL_17665676, EPI_ISL_17666708, EPI_ISL_17667360, EPI_ISL_17669304, EPI_ISL_17669306, EPI_ISL_17669441, EPI_ISL_17669457, EPI_ISL_17671157, EPI_ISL_17671162, EPI_ISL_17671689, EPI_ISL_17677128, EPI_ISL_17677325, EPI_ISL_17678395, EPI_ISL_17679253, EPI_ISL_17679612, EPI_ISL_17680172, EPI_ISL_17683135, EPI_ISL_17683747, EPI_ISL_17683879, EPI_ISL_17683882, EPI_ISL_17683902, EPI_ISL_17683926, EPI_ISL_17684194, EPI_ISL_17685960, EPI_ISL_17685982, EPI_ISL_17686409, EPI_ISL_17686485, EPI_ISL_17686694, EPI_ISL_17686736, EPI_ISL_17688072, EPI_ISL_17689247, EPI_ISL_17695348, EPI_ISL_17696086, EPI_ISL_17696551, EPI_ISL_17696575, EPI_ISL_17697616, EPI_ISL_17699149, EPI_ISL_17699879, EPI_ISL_17699884, EPI_ISL_17700051, EPI_ISL_17700270, EPI_ISL_17701083, EPI_ISL_17701278, EPI_ISL_17701782, EPI_ISL_17703815, EPI_ISL_17704713, EPI_ISL_17705564, EPI_ISL_17706013, EPI_ISL_17706030, EPI_ISL_17708288, EPI_ISL_17709926, EPI_ISL_17710268, EPI_ISL_17710278, EPI_ISL_17710307, EPI_ISL_17710673, EPI_ISL_17710974, EPI_ISL_17711012, EPI_ISL_17711646, EPI_ISL_17712964, EPI_ISL_17713423, EPI_ISL_17713709, EPI_ISL_17714880, EPI_ISL_17714882, EPI_ISL_17714902, EPI_ISL_17714948, EPI_ISL_17715122, EPI_ISL_17715974, EPI_ISL_17716296, EPI_ISL_17718358, EPI_ISL_17718497, EPI_ISL_17719162, EPI_ISL_17721620, EPI_ISL_17721941, EPI_ISL_17722060, EPI_ISL_17722142, EPI_ISL_17722884, EPI_ISL_17726746, EPI_ISL_17727194, EPI_ISL_17728144, EPI_ISL_17728250, EPI_ISL_17729454, EPI_ISL_17731387, EPI_ISL_17731388, EPI_ISL_17732098, EPI_ISL_17733269, EPI_ISL_17734236, EPI_ISL_17735972, EPI_ISL_17736284, EPI_ISL_17736388, EPI_ISL_17737562, EPI_ISL_17739108, EPI_ISL_17739543, EPI_ISL_17741957, EPI_ISL_17743681, EPI_ISL_17744022, EPI_ISL_17747309, EPI_ISL_17759354, EPI_ISL_17759925, EPI_ISL_17760279, EPI_ISL_17761215, EPI_ISL_17762387, EPI_ISL_17762760, EPI_ISL_17763721, EPI_ISL_17764011, EPI_ISL_17764066, EPI_ISL_17764072, EPI_ISL_17764496, EPI_ISL_17765796, EPI_ISL_17766060, EPI_ISL_17766100, EPI_ISL_17766112, EPI_ISL_17767434, EPI_ISL_17767435, EPI_ISL_17767436, EPI_ISL_17767437, EPI_ISL_17769030, EPI_ISL_17769081, EPI_ISL_17769169, EPI_ISL_17769170, EPI_ISL_17769216, EPI_ISL_17769229, EPI_ISL_17769310, EPI_ISL_17769888, EPI_ISL_17770725, EPI_ISL_17770729, EPI_ISL_17770732, EPI_ISL_17770736, EPI_ISL_17770779, EPI_ISL_17770815, EPI_ISL_17771047, EPI_ISL_17771051, EPI_ISL_17772012, EPI_ISL_17775344, EPI_ISL_17776736, EPI_ISL_17777061, EPI_ISL_17777067, EPI_ISL_17777729, EPI_ISL_17777748, EPI_ISL_17778593, EPI_ISL_17778602, EPI_ISL_17780724, EPI_ISL_17780726, EPI_ISL_17780860, EPI_ISL_17780886, EPI_ISL_17781122, EPI_ISL_17781585, EPI_ISL_17781712, EPI_ISL_17782148, EPI_ISL_17782366, EPI_ISL_17782502, EPI_ISL_17783358, EPI_ISL_17784545, EPI_ISL_17784546, EPI_ISL_17784547, EPI_ISL_17784552, EPI_ISL_17784558, EPI_ISL_17784569, EPI_ISL_17784585, EPI_ISL_17784593, EPI_ISL_17784775, EPI_ISL_17784803, EPI_ISL_17784804, EPI_ISL_17786165, EPI_ISL_17786546, EPI_ISL_17786769, EPI_ISL_17786827, EPI_ISL_17787009, EPI_ISL_17787597, EPI_ISL_17787864, EPI_ISL_17788384, EPI_ISL_17788516, EPI_ISL_17789385, EPI_ISL_17789475, EPI_ISL_17789808, EPI_ISL_17790033, EPI_ISL_17790116, EPI_ISL_17791306, EPI_ISL_17791796, EPI_ISL_17792172, EPI_ISL_17792191, EPI_ISL_17794816, EPI_ISL_17796500, EPI_ISL_17796537, EPI_ISL_17796598, EPI_ISL_17796704, EPI_ISL_17797704, EPI_ISL_17798165, EPI_ISL_17799068, EPI_ISL_17799108, EPI_ISL_17802597, EPI_ISL_17803325, EPI_ISL_17803653, EPI_ISL_17803784, EPI_ISL_17806504, EPI_ISL_17806524, EPI_ISL_17809334, EPI_ISL_17809574, EPI_ISL_17810512, EPI_ISL_17812915, EPI_ISL_17813049, EPI_ISL_17813537, EPI_ISL_17813637, EPI_ISL_17813862, EPI_ISL_17815222, EPI_ISL_17816174, EPI_ISL_17817657, EPI_ISL_17817985, EPI_ISL_17818039, EPI_ISL_17819921, EPI_ISL_17820257, EPI_ISL_17820258, EPI_ISL_17820602, EPI_ISL_17821850, EPI_ISL_17823538, EPI_ISL_17824292, EPI_ISL_17824608, EPI_ISL_17824611, EPI_ISL_17824670, EPI_ISL_17826285, EPI_ISL_17830134, EPI_ISL_17830169, EPI_ISL_17830573, EPI_ISL_17830591, EPI_ISL_17830762, EPI_ISL_17831005, EPI_ISL_17831639, EPI_ISL_17831941, EPI_ISL_17833161, EPI_ISL_17833549, EPI_ISL_17837092, EPI_ISL_17837097, EPI_ISL_17837134, EPI_ISL_17837135, EPI_ISL_17837188, EPI_ISL_17837432, EPI_ISL_17837459, EPI_ISL_17837460, EPI_ISL_17837914, EPI_ISL_17837915, EPI_ISL_17838109, EPI_ISL_17838506, EPI_ISL_17839137, EPI_ISL_17850070, EPI_ISL_17850078, EPI_ISL_17851276, EPI_ISL_17853355, EPI_ISL_17853579, EPI_ISL_17855226, EPI_ISL_17856975, EPI_ISL_17857949, EPI_ISL_17857950, EPI_ISL_17859477, EPI_ISL_17859958, EPI_ISL_17860390, EPI_ISL_17860984, EPI_ISL_17862677, EPI_ISL_17871595, EPI_ISL_17879222, EPI_ISL_17884376, EPI_ISL_17884518, EPI_ISL_17885064, EPI_ISL_17885128, EPI_ISL_17885331, EPI_ISL_17885459, EPI_ISL_17891004, EPI_ISL_17899627, EPI_ISL_17949029, EPI_ISL_17949339, EPI_ISL_17949978, EPI_ISL_17950840, EPI_ISL_17952015, EPI_ISL_17952019, EPI_ISL_17953343, EPI_ISL_17953610, EPI_ISL_17954106, EPI_ISL_17954662, EPI_ISL_17954669, EPI_ISL_17954940, EPI_ISL_17956164, EPI_ISL_17958015, EPI_ISL_17959424, EPI_ISL_17960600, EPI_ISL_17960747, EPI_ISL_17960749, EPI_ISL_17961252, EPI_ISL_17961253, EPI_ISL_17964403, EPI_ISL_17964415, EPI_ISL_17964828, EPI_ISL_17965635, EPI_ISL_17965636, EPI_ISL_17966200, EPI_ISL_17966205, EPI_ISL_17968777, EPI_ISL_17968962, EPI_ISL_17969108, EPI_ISL_17971223, EPI_ISL_17971936, EPI_ISL_17972242, EPI_ISL_17972372, EPI_ISL_17973367, EPI_ISL_17974574, EPI_ISL_17974688, EPI_ISL_17974927, EPI_ISL_17974952, EPI_ISL_17974980, EPI_ISL_17975003, EPI_ISL_17975174, EPI_ISL_17976105, EPI_ISL_17976113, EPI_ISL_17976114, EPI_ISL_17976116, EPI_ISL_17976531, EPI_ISL_17977985, EPI_ISL_17978344, EPI_ISL_17978693, EPI_ISL_17978839, EPI_ISL_17978964, EPI_ISL_17979017, EPI_ISL_17979965, EPI_ISL_17979979, EPI_ISL_17979981, EPI_ISL_17980588, EPI_ISL_17982411, EPI_ISL_17982453, EPI_ISL_17982543, EPI_ISL_17985757, EPI_ISL_17988396, EPI_ISL_17989190, EPI_ISL_17989433, EPI_ISL_17989516, EPI_ISL_17989740, EPI_ISL_17989749, EPI_ISL_17989792, EPI_ISL_17989829, EPI_ISL_17989858, EPI_ISL_17989860, EPI_ISL_17990203, EPI_ISL_17990304, EPI_ISL_17993966, EPI_ISL_17994784, EPI_ISL_17994786, EPI_ISL_17995488, EPI_ISL_17995513, EPI_ISL_17995955, EPI_ISL_17996897, EPI_ISL_17997249, EPI_ISL_17997251, EPI_ISL_17997917, EPI_ISL_17997982, EPI_ISL_17998406, EPI_ISL_18000155, EPI_ISL_18000245, EPI_ISL_18000414, EPI_ISL_18000654, EPI_ISL_18000825, EPI_ISL_18001789, EPI_ISL_18001862, EPI_ISL_18008246, EPI_ISL_18008262, EPI_ISL_18008673, EPI_ISL_18009591, EPI_ISL_18009602, EPI_ISL_18010720, EPI_ISL_18011449, EPI_ISL_18011518, EPI_ISL_18012526, EPI_ISL_18012547, EPI_ISL_18012806, EPI_ISL_18014237, EPI_ISL_18014700, EPI_ISL_18015682, EPI_ISL_18016999, EPI_ISL_18019246, EPI_ISL_18028785, EPI_ISL_18029979, EPI_ISL_18030390, EPI_ISL_18030391, EPI_ISL_18030395, EPI_ISL_18031842, EPI_ISL_18032297, EPI_ISL_18032322, EPI_ISL_18032338, EPI_ISL_18033013, EPI_ISL_18033516, EPI_ISL_18033631, EPI_ISL_18034079, EPI_ISL_18034109, EPI_ISL_18037119, EPI_ISL_18037474, EPI_ISL_18037476, EPI_ISL_18037744, EPI_ISL_18038269, EPI_ISL_18039728, EPI_ISL_18040070, EPI_ISL_18041130, EPI_ISL_18041968, EPI_ISL_18042110, EPI_ISL_18044024, EPI_ISL_18044164, EPI_ISL_18044400, EPI_ISL_18044754, EPI_ISL_18044755, EPI_ISL_18044759, EPI_ISL_18046466, EPI_ISL_18048708, EPI_ISL_18048972, EPI_ISL_18048978, EPI_ISL_18049009, EPI_ISL_18049160, EPI_ISL_18049161, EPI_ISL_18049174, EPI_ISL_18049809, EPI_ISL_18049917, EPI_ISL_18050065, EPI_ISL_18050520, EPI_ISL_18050523, EPI_ISL_18051914, EPI_ISL_18051918, EPI_ISL_18052440, EPI_ISL_18052776, EPI_ISL_18052929, EPI_ISL_18053022, EPI_ISL_18053315, EPI_ISL_18054466, EPI_ISL_18054899, EPI_ISL_18056643, EPI_ISL_18056644, EPI_ISL_18056759, EPI_ISL_18056769, EPI_ISL_18058525, EPI_ISL_18058567, EPI_ISL_18058881, EPI_ISL_18059074, EPI_ISL_18059075, EPI_ISL_18059076, EPI_ISL_18059726, EPI_ISL_18060973, EPI_ISL_18062475, EPI_ISL_18064366, EPI_ISL_18064383, EPI_ISL_18064405, EPI_ISL_18064413, EPI_ISL_18064431, EPI_ISL_18064456, EPI_ISL_18064519, EPI_ISL_18070310, EPI_ISL_18071883, EPI_ISL_18071901, EPI_ISL_18072068, EPI_ISL_18072343, EPI_ISL_18073924, EPI_ISL_18074072, EPI_ISL_18075985, EPI_ISL_18076065, EPI_ISL_18076069, EPI_ISL_18076165, EPI_ISL_18076251, EPI_ISL_18076473, EPI_ISL_18077275, EPI_ISL_18078878, EPI_ISL_18079417, EPI_ISL_18080566, EPI_ISL_18083488, EPI_ISL_18091808, EPI_ISL_18092412, EPI_ISL_18092435, EPI_ISL_18093840, EPI_ISL_18094397, EPI_ISL_18094429, EPI_ISL_18094476, EPI_ISL_18094560, EPI_ISL_18094842, EPI_ISL_18095157, EPI_ISL_18095961, EPI_ISL_18097327, EPI_ISL_18097349, EPI_ISL_18097786, EPI_ISL_18098270, EPI_ISL_18098276, EPI_ISL_18098299, EPI_ISL_18098479, EPI_ISL_18098976, EPI_ISL_18099952, EPI_ISL_18100455, EPI_ISL_18100457, EPI_ISL_18100607, EPI_ISL_18104072, EPI_ISL_18106416, EPI_ISL_18106460, EPI_ISL_18106464, EPI_ISL_18106662, EPI_ISL_18106788, EPI_ISL_18106910, EPI_ISL_18106912, EPI_ISL_18106920, EPI_ISL_18106929, EPI_ISL_18106930, EPI_ISL_18106931, EPI_ISL_18106933, EPI_ISL_18106934, EPI_ISL_18106950, EPI_ISL_18106951, EPI_ISL_18107900, EPI_ISL_18109285, EPI_ISL_18110014, EPI_ISL_18110496, EPI_ISL_18110776, EPI_ISL_18111020, EPI_ISL_18111021, EPI_ISL_18111040, EPI_ISL_18111041, EPI_ISL_18111086, EPI_ISL_18112015, EPI_ISL_18115442, EPI_ISL_18115451, EPI_ISL_18115956, EPI_ISL_18116015, EPI_ISL_18116176, EPI_ISL_18118264, EPI_ISL_18118289, EPI_ISL_18118388, EPI_ISL_18118556, EPI_ISL_18118855, EPI_ISL_18119265, EPI_ISL_18120201, EPI_ISL_18123396, EPI_ISL_18124840, EPI_ISL_18125049, EPI_ISL_18125050, EPI_ISL_18126021, EPI_ISL_18126834, EPI_ISL_18127203, EPI_ISL_18127526, EPI_ISL_18127527, EPI_ISL_18127685, EPI_ISL_18127834, EPI_ISL_18128931, EPI_ISL_18129019, EPI_ISL_18129038, EPI_ISL_18129213, EPI_ISL_18129656, EPI_ISL_18129944, EPI_ISL_18131053, EPI_ISL_18131109, EPI_ISL_18134315, EPI_ISL_18134392, EPI_ISL_18134395, EPI_ISL_18134442, EPI_ISL_18134610, EPI_ISL_18134691, EPI_ISL_18134700, EPI_ISL_18134706, EPI_ISL_18134984, EPI_ISL_18135040, EPI_ISL_18135429, EPI_ISL_18136392, EPI_ISL_18136968, EPI_ISL_18139400, EPI_ISL_18139409, EPI_ISL_18141686, EPI_ISL_18141739, EPI_ISL_18141844, EPI_ISL_18142202, EPI_ISL_18142317, EPI_ISL_18142357, EPI_ISL_18142525, EPI_ISL_18142978, EPI_ISL_18142994, EPI_ISL_18146885, EPI_ISL_18147456, EPI_ISL_18147966, EPI_ISL_18151975, EPI_ISL_18151976, EPI_ISL_18151977, EPI_ISL_18153000, EPI_ISL_18154903, EPI_ISL_18159587, EPI_ISL_18160510, EPI_ISL_18160518, EPI_ISL_18160519, EPI_ISL_18160530, EPI_ISL_18160538, EPI_ISL_18162567, EPI_ISL_18163680, EPI_ISL_18163890, EPI_ISL_18163891, EPI_ISL_18163892, EPI_ISL_18163893, EPI_ISL_18163894, EPI_ISL_18163895, EPI_ISL_18163896, EPI_ISL_18163897, EPI_ISL_18163898, EPI_ISL_18163898:, EPI_ISL_18164441, EPI_ISL_18166642, EPI_ISL_18166643, EPI_ISL_18168780, EPI_ISL_18169117, EPI_ISL_18205057, EPI_ISL_18207613, EPI_ISL_18210510, EPI_ISL_18212559, EPI_ISL_18213104, EPI_ISL_18215123, EPI_ISL_18215226, EPI_ISL_18215482, EPI_ISL_18215552, EPI_ISL_18217564, EPI_ISL_18217995, EPI_ISL_18218776, EPI_ISL_18219915, EPI_ISL_18219916, EPI_ISL_18219931, EPI_ISL_18219970, EPI_ISL_18220073, EPI_ISL_18220498, EPI_ISL_18221521, EPI_ISL_18221524, EPI_ISL_18221527, EPI_ISL_18221982, EPI_ISL_18221985, EPI_ISL_18222367, EPI_ISL_18224410, EPI_ISL_18224514, EPI_ISL_18225473, EPI_ISL_18227366, EPI_ISL_18227596, EPI_ISL_18227611, EPI_ISL_18227624, EPI_ISL_18227629, EPI_ISL_18228307, EPI_ISL_18232124, EPI_ISL_18233906, EPI_ISL_18234431, EPI_ISL_18234763, EPI_ISL_18236180, EPI_ISL_18236291, EPI_ISL_18238117, EPI_ISL_18241087, EPI_ISL_18241705, EPI_ISL_18241707, EPI_ISL_18241719, EPI_ISL_18245571, EPI_ISL_18247259, EPI_ISL_18248695, EPI_ISL_18249682, EPI_ISL_18253248, EPI_ISL_18253249, EPI_ISL_18255994, EPI_ISL_18256173, EPI_ISL_18256714, EPI_ISL_18256980, EPI_ISL_18258766, EPI_ISL_18259784, EPI_ISL_18260202, EPI_ISL_18263919, EPI_ISL_18263945, EPI_ISL_18263946, EPI_ISL_18263981, EPI_ISL_18269234, EPI_ISL_18271265, EPI_ISL_18273982, EPI_ISL_18274346, EPI_ISL_18276415, EPI_ISL_18277439, EPI_ISL_18277736, EPI_ISL_18278627, EPI_ISL_18278909, EPI_ISL_18279614, EPI_ISL_18281186, EPI_ISL_18281259, EPI_ISL_18281287, EPI_ISL_18281288, EPI_ISL_18281494, EPI_ISL_18281574, EPI_ISL_18282077, EPI_ISL_18282082, EPI_ISL_18284607, EPI_ISL_18286773, EPI_ISL_18287351, EPI_ISL_18289877, EPI_ISL_18290890, EPI_ISL_18290989, EPI_ISL_18291808, EPI_ISL_18292038, EPI_ISL_18292398, EPI_ISL_18294574, EPI_ISL_18295441, EPI_ISL_18295450, EPI_ISL_18298019, EPI_ISL_18299948, EPI_ISL_18300587, EPI_ISL_18301587, EPI_ISL_18302636, EPI_ISL_18303012, EPI_ISL_18303206, EPI_ISL_18303444, EPI_ISL_18303592, EPI_ISL_18303595, EPI_ISL_18303758, EPI_ISL_18306254, EPI_ISL_18306922, EPI_ISL_18308642, EPI_ISL_18311951, EPI_ISL_18313683, EPI_ISL_18315747, EPI_ISL_18315789, EPI_ISL_18319306, EPI_ISL_18319903, EPI_ISL_18319904, EPI_ISL_18319906, EPI_ISL_18319907, EPI_ISL_18320079, EPI_ISL_18321271, EPI_ISL_18322273, EPI_ISL_18322420, EPI_ISL_18322438, EPI_ISL_18323536, EPI_ISL_18324107, EPI_ISL_18324168, EPI_ISL_18324491, EPI_ISL_18325145, EPI_ISL_18325563, EPI_ISL_18326430, EPI_ISL_18326597, EPI_ISL_18326806, EPI_ISL_18326807, EPI_ISL_18330957, EPI_ISL_18330966, EPI_ISL_18331347, EPI_ISL_18334903, EPI_ISL_18334945, EPI_ISL_18334986, EPI_ISL_18335084, EPI_ISL_18335526, EPI_ISL_18336165, EPI_ISL_18336602, EPI_ISL_18336862, EPI_ISL_18337738, EPI_ISL_18338137, EPI_ISL_18338143, EPI_ISL_18338144, EPI_ISL_18338502, EPI_ISL_18338504, EPI_ISL_18338709, EPI_ISL_18342412, EPI_ISL_18343598, EPI_ISL_18345777, EPI_ISL_18345926, EPI_ISL_18346109, EPI_ISL_18351588, EPI_ISL_18352473, EPI_ISL_18352485, EPI_ISL_18352489, EPI_ISL_18359229, EPI_ISL_18359328, EPI_ISL_18359679, EPI_ISL_18360507, EPI_ISL_18360944, EPI_ISL_18361202, EPI_ISL_18362265, EPI_ISL_18362515, EPI_ISL_18363170, EPI_ISL_18363300, EPI_ISL_18365170, EPI_ISL_18365256, EPI_ISL_18367086, EPI_ISL_18367563, EPI_ISL_18367586, EPI_ISL_18367599, EPI_ISL_18367615, EPI_ISL_18367908, EPI_ISL_18367992, EPI_ISL_18370898, EPI_ISL_18370960, EPI_ISL_18370967, EPI_ISL_18371749, EPI_ISL_18373201, EPI_ISL_18373754, EPI_ISL_18377021, EPI_ISL_18377214, EPI_ISL_18377245, EPI_ISL_18377248, EPI_ISL_18378384, EPI_ISL_18380731, EPI_ISL_18381066, EPI_ISL_18381403, EPI_ISL_18383121, EPI_ISL_18383423, EPI_ISL_18384846, EPI_ISL_18384936, EPI_ISL_18385358, EPI_ISL_18385924, EPI_ISL_18386091, EPI_ISL_18386403, EPI_ISL_18387037, EPI_ISL_18388509, EPI_ISL_18388585, EPI_ISL_18389783, EPI_ISL_18389797, EPI_ISL_18391451, EPI_ISL_18391597, EPI_ISL_18392259, EPI_ISL_18392502, EPI_ISL_18392841, EPI_ISL_18393366, EPI_ISL_18395551, EPI_ISL_18398210, EPI_ISL_18398259, EPI_ISL_18400843, EPI_ISL_18400856, EPI_ISL_18400946, EPI_ISL_18400976, EPI_ISL_18400987, EPI_ISL_18401313, EPI_ISL_18403047, EPI_ISL_18403051, EPI_ISL_18403054, EPI_ISL_18403509, EPI_ISL_18403523, EPI_ISL_18404585, EPI_ISL_18405535, EPI_ISL_18405621, EPI_ISL_18406394, EPI_ISL_18408561, EPI_ISL_18410987, EPI_ISL_18414567, EPI_ISL_18414568, EPI_ISL_18414808, EPI_ISL_18415823, EPI_ISL_18415832, EPI_ISL_18415834, EPI_ISL_18415840, EPI_ISL_18415854, EPI_ISL_18416870, EPI_ISL_18417129, EPI_ISL_18417211, EPI_ISL_18419485, EPI_ISL_18419748, EPI_ISL_18421674, EPI_ISL_18422693, EPI_ISL_18422715, EPI_ISL_18422771, EPI_ISL_18423785, EPI_ISL_18423814, EPI_ISL_18423907, EPI_ISL_18424281, EPI_ISL_18424468, EPI_ISL_18426836, EPI_ISL_18428844, EPI_ISL_18429684, EPI_ISL_18429702, EPI_ISL_18429725, EPI_ISL_18429773, EPI_ISL_18429797, EPI_ISL_18432077, EPI_ISL_18433350, EPI_ISL_18434194, EPI_ISL_18435050, EPI_ISL_18435557, EPI_ISL_18435892, EPI_ISL_18435949, EPI_ISL_18437342, EPI_ISL_18438723, EPI_ISL_18439733, EPI_ISL_18440037, EPI_ISL_18440370, EPI_ISL_18440660, EPI_ISL_18440866, EPI_ISL_18441868, EPI_ISL_18443784, EPI_ISL_18443944, EPI_ISL_18446092, EPI_ISL_18448894, EPI_ISL_18449647, EPI_ISL_18449794, EPI_ISL_18449820, EPI_ISL_18449892, EPI_ISL_18450249, EPI_ISL_18450812, EPI_ISL_18451678, EPI_ISL_18453400, EPI_ISL_18455292, EPI_ISL_18455564, EPI_ISL_18455706, EPI_ISL_18455950, EPI_ISL_18457808, EPI_ISL_18459512, EPI_ISL_18461774, EPI_ISL_18462852, EPI_ISL_18463490, EPI_ISL_18463766, EPI_ISL_18466251, EPI_ISL_18468149, EPI_ISL_18470400, EPI_ISL_18472311, EPI_ISL_18472312, EPI_ISL_18473559, EPI_ISL_18474126, EPI_ISL_18474555, EPI_ISL_18474665, EPI_ISL_18475072, EPI_ISL_18475534, EPI_ISL_18475535, EPI_ISL_18480741, EPI_ISL_18486919, EPI_ISL_18487225, EPI_ISL_18489646, EPI_ISL_18489793, EPI_ISL_18489829, EPI_ISL_18491841, EPI_ISL_18491844, EPI_ISL_18492277, EPI_ISL_18492305, EPI_ISL_18492307, EPI_ISL_18492412, EPI_ISL_18492455, EPI_ISL_18493129, EPI_ISL_18495416, EPI_ISL_18496252, EPI_ISL_18496585, EPI_ISL_18498001, EPI_ISL_18498420, EPI_ISL_18498499, EPI_ISL_18500316, EPI_ISL_18500771, EPI_ISL_18501087, EPI_ISL_18503287, EPI_ISL_18509817, EPI_ISL_18512418, EPI_ISL_18512421, EPI_ISL_18512438, EPI_ISL_18513936, EPI_ISL_18514552, EPI_ISL_18515280, EPI_ISL_18515316, EPI_ISL_18515328, EPI_ISL_18515343, EPI_ISL_18515511, EPI_ISL_18515749, EPI_ISL_18516916, EPI_ISL_18518769, EPI_ISL_18518932, EPI_ISL_18519113, EPI_ISL_18520677, EPI_ISL_18520678, EPI_ISL_18521575, EPI_ISL_18521765, EPI_ISL_18522020, EPI_ISL_18522184, EPI_ISL_18522580, EPI_ISL_18523129, EPI_ISL_18524926, EPI_ISL_18525067, EPI_ISL_18525840, EPI_ISL_18526641, EPI_ISL_18528453, EPI_ISL_18529555, EPI_ISL_18530445, EPI_ISL_18530449, EPI_ISL_18536853, EPI_ISL_18537013, EPI_ISL_18537032, EPI_ISL_18537373, EPI_ISL_18537428, EPI_ISL_18537814, EPI_ISL_18538015, EPI_ISL_18543268, EPI_ISL_18543705, EPI_ISL_18546112, EPI_ISL_18546287, EPI_ISL_18546551, EPI_ISL_18546715, EPI_ISL_18551440, EPI_ISL_18552697, EPI_ISL_18553587, EPI_ISL_18554053, EPI_ISL_18556084, EPI_ISL_18556539, EPI_ISL_18557145, EPI_ISL_18558360, EPI_ISL_18558385, EPI_ISL_18558412, EPI_ISL_18558468, EPI_ISL_18558477, EPI_ISL_18559317, EPI_ISL_18560556, EPI_ISL_18560725, EPI_ISL_18560872, EPI_ISL_18561098, EPI_ISL_18563181, EPI_ISL_18563821, EPI_ISL_18564403, EPI_ISL_18564755, EPI_ISL_18566696, EPI_ISL_18567985, EPI_ISL_18568124, EPI_ISL_18576266, EPI_ISL_18576754, EPI_ISL_18577842, EPI_ISL_18577862, EPI_ISL_18577966, EPI_ISL_18578195, EPI_ISL_18579981, EPI_ISL_18580011, EPI_ISL_18580750, EPI_ISL_18580874, EPI_ISL_18581347, EPI_ISL_18584141, EPI_ISL_18588773, EPI_ISL_18589012, EPI_ISL_18589243, EPI_ISL_18589475, EPI_ISL_18589669, EPI_ISL_18590820, EPI_ISL_18591717, EPI_ISL_18593579, EPI_ISL_18594183, EPI_ISL_18594233, EPI_ISL_18594266, EPI_ISL_18595212, EPI_ISL_18598503, EPI_ISL_18598525, EPI_ISL_18598964, EPI_ISL_18603922, EPI_ISL_18604501, EPI_ISL_18604502, EPI_ISL_18604503, EPI_ISL_18604504, EPI_ISL_18604505, EPI_ISL_18606460, EPI_ISL_18607149, EPI_ISL_18607150, EPI_ISL_18608694, EPI_ISL_18609973, EPI_ISL_18612246, EPI_ISL_18615968, EPI_ISL_18622139, EPI_ISL_18624843, EPI_ISL_18625316, EPI_ISL_18626713, EPI_ISL_18626714, EPI_ISL_18626750, EPI_ISL_18630930, EPI_ISL_18633829, EPI_ISL_18634703, EPI_ISL_18635526, EPI_ISL_18635546, EPI_ISL_18635599, EPI_ISL_18635961, EPI_ISL_18640058, EPI_ISL_18641470, EPI_ISL_18641499, EPI_ISL_18642608, EPI_ISL_18646912, EPI_ISL_18646945, EPI_ISL_18648209, EPI_ISL_18652556, EPI_ISL_18653898, EPI_ISL_18654501, EPI_ISL_18672102, EPI_ISL_18674382, EPI_ISL_18677704, EPI_ISL_18681665, EPI_ISL_18681942, EPI_ISL_18681975, EPI_ISL_18683430, EPI_ISL_18687895, EPI_ISL_18689964, EPI_ISL_18693860, EPI_ISL_18694187, EPI_ISL_18702737, EPI_ISL_18712665, EPI_ISL_18712806, EPI_ISL_18712994, EPI_ISL_18713198, EPI_ISL_18714360, EPI_ISL_18714857, EPI_ISL_18715065, EPI_ISL_18715624, EPI_ISL_18717261, EPI_ISL_18727247, EPI_ISL_18727294, EPI_ISL_18729637, EPI_ISL_18730563, EPI_ISL_18734528, EPI_ISL_18740049, EPI_ISL_18742836, EPI_ISL_18743266, EPI_ISL_18743442, EPI_ISL_18743504, EPI_ISL_18743721, EPI_ISL_18758247, EPI_ISL_18758251, EPI_ISL_18759772, EPI_ISL_18760241, EPI_ISL_18763780, EPI_ISL_18766752, EPI_ISL_18772294, EPI_ISL_18777179, EPI_ISL_18778198, EPI_ISL_18778941, EPI_ISL_18782000, EPI_ISL_18782699, EPI_ISL_18784365, EPI_ISL_18784385, EPI_ISL_18784404, EPI_ISL_18784438, EPI_ISL_18787342, EPI_ISL_18787343, EPI_ISL_18790918, EPI_ISL_18792770, EPI_ISL_18796930, EPI_ISL_18797833, EPI_ISL_18798193, EPI_ISL_18798198, EPI_ISL_18798202, EPI_ISL_18798204, EPI_ISL_18798234, EPI_ISL_18799019, EPI_ISL_18801405, EPI_ISL_18803784, EPI_ISL_18805014, EPI_ISL_18809509, EPI_ISL_18809652, EPI_ISL_18811047, EPI_ISL_18811325, EPI_ISL_18811375, EPI_ISL_18811389, EPI_ISL_18814271, EPI_ISL_18814272, EPI_ISL_18814273, EPI_ISL_18815556, EPI_ISL_18816278, EPI_ISL_18816628, EPI_ISL_18817415, EPI_ISL_18818949, EPI_ISL_18820996, EPI_ISL_18824608, EPI_ISL_18825438, EPI_ISL_18828912, EPI_ISL_18829708, EPI_ISL_18832540, EPI_ISL_18832845, EPI_ISL_18839074, EPI_ISL_18839275, EPI_ISL_18839824, EPI_ISL_18839850, EPI_ISL_18839932, EPI_ISL_18841696, EPI_ISL_18842709, EPI_ISL_18846230, EPI_ISL_18851473, EPI_ISL_18853389, EPI_ISL_18853925, EPI_ISL_18854197, EPI_ISL_18854790, EPI_ISL_18854974, EPI_ISL_18856190, EPI_ISL_18856328, EPI_ISL_18859868, EPI_ISL_18860821, EPI_ISL_18863734, EPI_ISL_18864357, EPI_ISL_18864823, EPI_ISL_18865364, EPI_ISL_18865427, EPI_ISL_18868940, EPI_ISL_18869806, EPI_ISL_18870297, EPI_ISL_18872609, EPI_ISL_18873437, EPI_ISL_18873899, EPI_ISL_18873907, EPI_ISL_18874096, EPI_ISL_18874749, EPI_ISL_18875008, EPI_ISL_18876158, EPI_ISL_18876159, EPI_ISL_18876160, EPI_ISL_18876161, EPI_ISL_18876162, EPI_ISL_18876164, EPI_ISL_18876166, EPI_ISL_18876167, EPI_ISL_18876171, EPI_ISL_18876172, EPI_ISL_18876174, EPI_ISL_18876176, EPI_ISL_18876178, EPI_ISL_18876356, EPI_ISL_18877288, EPI_ISL_18877358, EPI_ISL_18877379, EPI_ISL_18877954, EPI_ISL_18878665, EPI_ISL_18880075, EPI_ISL_18882888, EPI_ISL_18884114, EPI_ISL_18885028, EPI_ISL_18885545, EPI_ISL_18886798, EPI_ISL_18888145, EPI_ISL_18888340, EPI_ISL_18889879, EPI_ISL_18892766, EPI_ISL_18896020, EPI_ISL_18897180, EPI_ISL_18897181, EPI_ISL_18899656, EPI_ISL_18899840, EPI_ISL_18900159, EPI_ISL_18901224, EPI_ISL_18901411, EPI_ISL_18901513, EPI_ISL_18901762, EPI_ISL_18901997, EPI_ISL_18904234, EPI_ISL_18906411, EPI_ISL_18906577, EPI_ISL_18908924, EPI_ISL_18914858, EPI_ISL_18915560, EPI_ISL_18916107, EPI_ISL_18916798, EPI_ISL_18916914, EPI_ISL_18917364, EPI_ISL_18917482, EPI_ISL_18917496, EPI_ISL_18917541, EPI_ISL_18918931, EPI_ISL_18918932, EPI_ISL_18918944, EPI_ISL_18919348, EPI_ISL_18919477, EPI_ISL_18919544, EPI_ISL_18924428, EPI_ISL_18927022, EPI_ISL_18927251, EPI_ISL_18927258, EPI_ISL_18927292, EPI_ISL_18927500, EPI_ISL_18927514, EPI_ISL_18927587, EPI_ISL_18930149, EPI_ISL_18930177, EPI_ISL_18930182, EPI_ISL_18930190, EPI_ISL_18930681, EPI_ISL_18931503, EPI_ISL_18931504, EPI_ISL_18931535, EPI_ISL_18931538, EPI_ISL_18931542, EPI_ISL_18932507, EPI_ISL_18933211, EPI_ISL_18933315, EPI_ISL_18933410, EPI_ISL_18933606, EPI_ISL_18933647, EPI_ISL_18935609, EPI_ISL_18935631, EPI_ISL_18936396, EPI_ISL_18939566, EPI_ISL_18939949, EPI_ISL_18943203, EPI_ISL_18945392, EPI_ISL_18945748, EPI_ISL_18947086, EPI_ISL_18947189, EPI_ISL_18947944, EPI_ISL_18948077, EPI_ISL_18948078, EPI_ISL_18948422, EPI_ISL_18948613, EPI_ISL_18948756, EPI_ISL_18948757, EPI_ISL_18949161, EPI_ISL_18949911, EPI_ISL_18952873, EPI_ISL_18953132, EPI_ISL_18953134, EPI_ISL_18953234, EPI_ISL_18953461, EPI_ISL_18953493, EPI_ISL_18953503, EPI_ISL_18953522, EPI_ISL_18955186, EPI_ISL_18956070, EPI_ISL_18956087, EPI_ISL_18956161, EPI_ISL_18958247, EPI_ISL_18958295, EPI_ISL_18958383, EPI_ISL_18960149, EPI_ISL_18960183, EPI_ISL_18960224, EPI_ISL_18962896, EPI_ISL_18962928, EPI_ISL_18963535, EPI_ISL_18963652, EPI_ISL_18965082, EPI_ISL_18965451, EPI_ISL_18966035, EPI_ISL_18966186, EPI_ISL_18968116, EPI_ISL_18968120, EPI_ISL_18969119, EPI_ISL_18969697, EPI_ISL_18969735, EPI_ISL_18970692, EPI_ISL_18971336, EPI_ISL_18972187, EPI_ISL_18972699, EPI_ISL_18972708, EPI_ISL_18972719, EPI_ISL_18972725, EPI_ISL_18972734, EPI_ISL_18972995, EPI_ISL_18975064, EPI_ISL_18975148, EPI_ISL_18975189, EPI_ISL_18975333, EPI_ISL_18977928, EPI_ISL_18977929, EPI_ISL_18979948, EPI_ISL_18980164, EPI_ISL_18981298, EPI_ISL_18982763, EPI_ISL_18986641, EPI_ISL_18986642, EPI_ISL_18986747, EPI_ISL_18987173, EPI_ISL_18987258, EPI_ISL_18987259, EPI_ISL_18987545, EPI_ISL_18990023, EPI_ISL_18993521, EPI_ISL_18998415, EPI_ISL_18998583, EPI_ISL_18999516, EPI_ISL_19000455, EPI_ISL_19002243, EPI_ISL_19002832, EPI_ISL_19003837, EPI_ISL_19004571, EPI_ISL_19005468, EPI_ISL_19005470, EPI_ISL_19005472, EPI_ISL_19005481, EPI_ISL_19006725, EPI_ISL_19008875, EPI_ISL_19009123, EPI_ISL_19009131, EPI_ISL_19010135, EPI_ISL_19012038, EPI_ISL_19012114, EPI_ISL_19012460, EPI_ISL_19012472, EPI_ISL_19012554, EPI_ISL_19012663, EPI_ISL_19013418, EPI_ISL_19014517, EPI_ISL_19014601, EPI_ISL_19015115, EPI_ISL_19015882, EPI_ISL_19016044, EPI_ISL_19016053, EPI_ISL_19016370, EPI_ISL_19017499, EPI_ISL_19019122, EPI_ISL_19019127, EPI_ISL_19019128, EPI_ISL_19019136, EPI_ISL_19019141, EPI_ISL_19021045, EPI_ISL_19021046, EPI_ISL_19022071, EPI_ISL_19022479, EPI_ISL_19024260, EPI_ISL_19026222, EPI_ISL_19026577, EPI_ISL_19026932, EPI_ISL_19029499, EPI_ISL_19030027, EPI_ISL_19030112, EPI_ISL_19030117, EPI_ISL_19030129, EPI_ISL_19030183, EPI_ISL_19031583, EPI_ISL_19032054, EPI_ISL_19032056, EPI_ISL_19032179, EPI_ISL_19032268, EPI_ISL_19032269, EPI_ISL_19032270, EPI_ISL_19032271, EPI_ISL_19032272, EPI_ISL_19032273, EPI_ISL_19032274, EPI_ISL_19032275, EPI_ISL_19032276, EPI_ISL_19032277, EPI_ISL_19032278, EPI_ISL_19032279, EPI_ISL_19032280, EPI_ISL_19032281, EPI_ISL_19032282, EPI_ISL_19032283, EPI_ISL_19032284, EPI_ISL_19032598, EPI_ISL_19033249, EPI_ISL_19035801, EPI_ISL_19036115, EPI_ISL_19036116, EPI_ISL_19036117, EPI_ISL_19036644, EPI_ISL_19036878, EPI_ISL_19036921, EPI_ISL_19041379, EPI_ISL_19041967, EPI_ISL_19042675, EPI_ISL_19043835, EPI_ISL_19043836, EPI_ISL_19044085, EPI_ISL_19044119, EPI_ISL_19044163, EPI_ISL_19044218, EPI_ISL_19044280, EPI_ISL_19044474, EPI_ISL_19044644, EPI_ISL_19044737, EPI_ISL_19046891, EPI_ISL_19046892, EPI_ISL_19047408, EPI_ISL_19049407, EPI_ISL_19049586, EPI_ISL_19050509, EPI_ISL_19051602, EPI_ISL_19051976, EPI_ISL_19052024, EPI_ISL_19052034, EPI_ISL_19052090, EPI_ISL_19052102, EPI_ISL_19052117, EPI_ISL_19053803, EPI_ISL_19053806, EPI_ISL_19053808, EPI_ISL_19053809, EPI_ISL_19054784, EPI_ISL_19055364, EPI_ISL_19055368, EPI_ISL_19055382, EPI_ISL_19055909, EPI_ISL_19058120, EPI_ISL_19060002, EPI_ISL_19060327, EPI_ISL_19060922, EPI_ISL_19061139, EPI_ISL_19061646, EPI_ISL_19062162, EPI_ISL_19062523, EPI_ISL_19062524, EPI_ISL_19064175, EPI_ISL_19065060, EPI_ISL_19065882, EPI_ISL_19066171, EPI_ISL_19066817, EPI_ISL_19067782, EPI_ISL_19067788, EPI_ISL_19070471, EPI_ISL_19071618, EPI_ISL_19073169, EPI_ISL_19073731, EPI_ISL_19073732, EPI_ISL_19073800, EPI_ISL_19073807, EPI_ISL_19074673, EPI_ISL_19075261, EPI_ISL_19075267, EPI_ISL_19075350, EPI_ISL_19076088, EPI_ISL_19081415, EPI_ISL_19081416, EPI_ISL_19081419, EPI_ISL_19081421, EPI_ISL_19081422, EPI_ISL_19081425, EPI_ISL_19082080, EPI_ISL_19082092, EPI_ISL_19082200, EPI_ISL_19082275, EPI_ISL_19082487, EPI_ISL_19085083, EPI_ISL_19085329, EPI_ISL_19085381, EPI_ISL_19085539, EPI_ISL_19085540, EPI_ISL_19085542, EPI_ISL_19086378, EPI_ISL_19086393, EPI_ISL_19086523, EPI_ISL_19091018, EPI_ISL_19091019, EPI_ISL_19091036, EPI_ISL_19091144, EPI_ISL_19094153, EPI_ISL_19094369, EPI_ISL_19095154, EPI_ISL_19095156, EPI_ISL_19095501, EPI_ISL_19095606, EPI_ISL_19095734, EPI_ISL_19095768, EPI_ISL_19100255, EPI_ISL_19100981, EPI_ISL_19105055, EPI_ISL_19106368, EPI_ISL_19106842, EPI_ISL_19106844, EPI_ISL_19106999, EPI_ISL_19108198, EPI_ISL_19108199, EPI_ISL_19108200, EPI_ISL_19108208, EPI_ISL_19108340, EPI_ISL_19108650, EPI_ISL_19108704, EPI_ISL_19108706, EPI_ISL_19108725, EPI_ISL_19109579, EPI_ISL_19116728, EPI_ISL_19117393, EPI_ISL_19131415, EPI_ISL_19131416, EPI_ISL_19131417, EPI_ISL_19131418, EPI_ISL_19131419, EPI_ISL_19132785, EPI_ISL_19132846, EPI_ISL_19133800, EPI_ISL_19135448, EPI_ISL_19135481, EPI_ISL_19135505, EPI_ISL_19135510, EPI_ISL_19135511, EPI_ISL_19137782, EPI_ISL_19137844, EPI_ISL_19140762, EPI_ISL_19141756, EPI_ISL_19141912, EPI_ISL_19142916, EPI_ISL_19143060, EPI_ISL_19143411, EPI_ISL_19143412, EPI_ISL_19143551, EPI_ISL_19143854, EPI_ISL_19143866, EPI_ISL_19143953, EPI_ISL_19143954, EPI_ISL_19144318, EPI_ISL_19145335, EPI_ISL_19146181, EPI_ISL_19146282, EPI_ISL_19146283, EPI_ISL_19146308, EPI_ISL_19146747, EPI_ISL_19147578, EPI_ISL_19148775, EPI_ISL_19151438, EPI_ISL_19153859, EPI_ISL_19158778, EPI_ISL_19158782, EPI_ISL_19159275, EPI_ISL_19159918, EPI_ISL_19159922, EPI_ISL_19159923, EPI_ISL_19161805, EPI_ISL_19164072, EPI_ISL_19164956, EPI_ISL_19165347, EPI_ISL_19165721, EPI_ISL_19167420, EPI_ISL_19167714, EPI_ISL_19169236, EPI_ISL_19169239, EPI_ISL_19169240, EPI_ISL_19169597, EPI_ISL_19169598, EPI_ISL_19173640, EPI_ISL_19173786, EPI_ISL_19175160, EPI_ISL_19175165, EPI_ISL_19175555, EPI_ISL_19175939, EPI_ISL_19176642, EPI_ISL_19176853, EPI_ISL_19176930, EPI_ISL_19177237, EPI_ISL_19177366, EPI_ISL_19177549, EPI_ISL_19177604, EPI_ISL_19177633, EPI_ISL_19178043, EPI_ISL_19178311, EPI_ISL_19179820, EPI_ISL_19182830, EPI_ISL_19182893, EPI_ISL_19183095, EPI_ISL_19183664, EPI_ISL_19183910, EPI_ISL_19183917, EPI_ISL_19183924, EPI_ISL_19184000, EPI_ISL_19184031, EPI_ISL_19184858, EPI_ISL_19185391, EPI_ISL_19186372, EPI_ISL_19187674, EPI_ISL_19187980, EPI_ISL_19188471, EPI_ISL_19189250, EPI_ISL_19190171, EPI_ISL_19190915, EPI_ISL_19191046, EPI_ISL_19192623, EPI_ISL_19192806, EPI_ISL_19192819, EPI_ISL_19193007, EPI_ISL_19193543, EPI_ISL_19193606, EPI_ISL_19193607, EPI_ISL_19193617, EPI_ISL_19195977, EPI_ISL_19195978, EPI_ISL_19195979, EPI_ISL_19195980, EPI_ISL_19195981, EPI_ISL_19195982, EPI_ISL_19196018, EPI_ISL_19196019, EPI_ISL_19196020, EPI_ISL_19196021, EPI_ISL_19197906, EPI_ISL_19198127, EPI_ISL_19198259, EPI_ISL_19199717, EPI_ISL_19199719, EPI_ISL_19200980, EPI_ISL_19202065, EPI_ISL_19202066, EPI_ISL_19203296, EPI_ISL_19205927, EPI_ISL_19209656, EPI_ISL_19210836, EPI_ISL_19211459, EPI_ISL_19213222, EPI_ISL_19213351, EPI_ISL_19214251, EPI_ISL_19214614, EPI_ISL_19216679, EPI_ISL_19216828, EPI_ISL_19217899, EPI_ISL_19219811, EPI_ISL_19221626, EPI_ISL_19223353, EPI_ISL_19225911, EPI_ISL_19226495, EPI_ISL_19227219, EPI_ISL_19228129, EPI_ISL_19229097, EPI_ISL_19229565, EPI_ISL_19230757, EPI_ISL_19230859, EPI_ISL_19230867, EPI_ISL_19230971, EPI_ISL_19232972, EPI_ISL_19234843, EPI_ISL_19237942, EPI_ISL_19239019, EPI_ISL_19239628, EPI_ISL_19239713, EPI_ISL_19243158, EPI_ISL_19243166, EPI_ISL_19243171, EPI_ISL_19243231, EPI_ISL_19243516, EPI_ISL_19243810, EPI_ISL_19244002, EPI_ISL_19245397, EPI_ISL_19245398, EPI_ISL_19245399, EPI_ISL_19251154, EPI_ISL_19254798, EPI_ISL_19256132, EPI_ISL_19256137, EPI_ISL_19256138, EPI_ISL_19256151, EPI_ISL_19256152, EPI_ISL_19257013, EPI_ISL_19257014, EPI_ISL_19257017, EPI_ISL_19257018, EPI_ISL_19257108, EPI_ISL_19257977, EPI_ISL_19259365, EPI_ISL_19259369, EPI_ISL_19259383, EPI_ISL_19260849, EPI_ISL_19260850, EPI_ISL_19261083, EPI_ISL_19261711, EPI_ISL_19264977, EPI_ISL_19265076, EPI_ISL_19267631, EPI_ISL_19268349, EPI_ISL_19268541, EPI_ISL_19269083, EPI_ISL_19269483, EPI_ISL_19271166, EPI_ISL_19271182, EPI_ISL_19271183, EPI_ISL_19271567, EPI_ISL_19271746, EPI_ISL_19273074, EPI_ISL_19277027, EPI_ISL_19277033, EPI_ISL_19280260, EPI_ISL_19281017, EPI_ISL_19282183, EPI_ISL_19282184, EPI_ISL_19283996, EPI_ISL_19286133, EPI_ISL_19286135, EPI_ISL_19286138, EPI_ISL_19286139, EPI_ISL_19286260, EPI_ISL_19287266, EPI_ISL_19287684, EPI_ISL_19288763, EPI_ISL_19290906, EPI_ISL_19292343, EPI_ISL_19292838, EPI_ISL_19292841, EPI_ISL_19297116, EPI_ISL_19297576, EPI_ISL_19298817, EPI_ISL_19299051, EPI_ISL_19300378, EPI_ISL_19300385, EPI_ISL_19300400, EPI_ISL_19301040, EPI_ISL_19302363, EPI_ISL_19302407, EPI_ISL_19304883, EPI_ISL_19308329, EPI_ISL_19308670, EPI_ISL_19308767, EPI_ISL_19308875, EPI_ISL_19308876, EPI_ISL_19308877, EPI_ISL_19308906, EPI_ISL_19309906, EPI_ISL_19310234, EPI_ISL_19311057, EPI_ISL_19311769, EPI_ISL_19312520, EPI_ISL_19318476, EPI_ISL_19319006, EPI_ISL_19320511, EPI_ISL_19322462, EPI_ISL_19324934, EPI_ISL_19326359, EPI_ISL_19331433, EPI_ISL_19332294, EPI_ISL_19333087, EPI_ISL_19333144, EPI_ISL_19335441, EPI_ISL_19340140, EPI_ISL_19341128, EPI_ISL_19341144, EPI_ISL_19344211, EPI_ISL_19344746, EPI_ISL_19345465, EPI_ISL_19345845, EPI_ISL_19346102, EPI_ISL_19346538, EPI_ISL_19348673, EPI_ISL_19351027, EPI_ISL_19351032, EPI_ISL_19351033, EPI_ISL_19351648, EPI_ISL_19351927, EPI_ISL_19359999, EPI_ISL_19360941, EPI_ISL_19362874, EPI_ISL_19362955, EPI_ISL_19363093, EPI_ISL_19363366, EPI_ISL_19364322, EPI_ISL_19364675, EPI_ISL_19365917, EPI_ISL_19369553, EPI_ISL_19369713, EPI_ISL_19374392, EPI_ISL_19374843, EPI_ISL_19374844, EPI_ISL_19374845, EPI_ISL_19380467, EPI_ISL_19381264, EPI_ISL_19381428, EPI_ISL_19381638, EPI_ISL_19381992, EPI_ISL_19381994, EPI_ISL_19382602, EPI_ISL_19383694, EPI_ISL_19384121, EPI_ISL_19385914, EPI_ISL_19385980, EPI_ISL_19387703, EPI_ISL_19388165, EPI_ISL_19388758, EPI_ISL_19391206, EPI_ISL_19391216, EPI_ISL_19393434, EPI_ISL_19393708, EPI_ISL_19398369, EPI_ISL_19405517, EPI_ISL_19405918, EPI_ISL_19408692, EPI_ISL_19408693, EPI_ISL_19408911, EPI_ISL_19409171, EPI_ISL_19410044, EPI_ISL_19410056, EPI_ISL_19410058, EPI_ISL_19411869, EPI_ISL_19412054, EPI_ISL_19412418, EPI_ISL_19414059, EPI_ISL_19414060, EPI_ISL_19414061, EPI_ISL_19414452, EPI_ISL_19414669, EPI_ISL_19414842, EPI_ISL_19415183, EPI_ISL_19415272, EPI_ISL_19415273, EPI_ISL_19415333, EPI_ISL_19417987, EPI_ISL_19418385, EPI_ISL_19418789, EPI_ISL_19419991, EPI_ISL_19425659, EPI_ISL_19427049, EPI_ISL_19427050, EPI_ISL_19427051, EPI_ISL_19428450, EPI_ISL_19428673, EPI_ISL_19431719, EPI_ISL_19433335, EPI_ISL_19434973, EPI_ISL_19438222, EPI_ISL_19438699, EPI_ISL_19441794, EPI_ISL_19446721, EPI_ISL_19446726, EPI_ISL_19447859, EPI_ISL_19450094, EPI_ISL_19451639, EPI_ISL_19452022, EPI_ISL_19454521, EPI_ISL_19456759, EPI_ISL_19457982, EPI_ISL_19458104, EPI_ISL_19458866, EPI_ISL_19459469, EPI_ISL_19463531, EPI_ISL_19463787, EPI_ISL_19463811, EPI_ISL_19464534, EPI_ISL_19465468, EPI_ISL_19467713, EPI_ISL_19467725, EPI_ISL_19468710, EPI_ISL_19473728, EPI_ISL_19474606, EPI_ISL_19474613, EPI_ISL_19477107, EPI_ISL_19478383, EPI_ISL_19478598, EPI_ISL_19479514, EPI_ISL_19480237, EPI_ISL_19482235, EPI_ISL_19483178, EPI_ISL_19483184, EPI_ISL_19483313, EPI_ISL_19486141, EPI_ISL_19488917, EPI_ISL_19491347, EPI_ISL_19495855, EPI_ISL_19498393, EPI_ISL_19499089, EPI_ISL_19499133, EPI_ISL_19499199, EPI_ISL_19499228, EPI_ISL_19499640, EPI_ISL_19499789, EPI_ISL_19500407, EPI_ISL_19502648, EPI_ISL_19503869, EPI_ISL_19506293, EPI_ISL_19506337, EPI_ISL_19506387, EPI_ISL_19506391, EPI_ISL_19506410, EPI_ISL_19506412, EPI_ISL_19506489, EPI_ISL_19506517, EPI_ISL_19506572, EPI_ISL_19506574, EPI_ISL_19508878, EPI_ISL_19510518, EPI_ISL_19511239, EPI_ISL_19511242, EPI_ISL_19511245, EPI_ISL_19511259, EPI_ISL_19511353, EPI_ISL_19512930, EPI_ISL_19513189, EPI_ISL_19513373, EPI_ISL_19513376, EPI_ISL_19517945, EPI_ISL_19519860, EPI_ISL_19521406, EPI_ISL_19522101, EPI_ISL_19526975, EPI_ISL_19527216, EPI_ISL_19527499, EPI_ISL_19529679, EPI_ISL_19529699, EPI_ISL_19530201, EPI_ISL_19530213, EPI_ISL_19530220, EPI_ISL_19532518, EPI_ISL_19535971, EPI_ISL_19536358, EPI_ISL_19537504, EPI_ISL_19537517, EPI_ISL_19537806, EPI_ISL_19539586, EPI_ISL_19539602, EPI_ISL_19541318, EPI_ISL_19543648, EPI_ISL_19543801, EPI_ISL_19546384, EPI_ISL_19550039, EPI_ISL_19550107, EPI_ISL_19551038, EPI_ISL_19551134, EPI_ISL_19555102, EPI_ISL_19555114, EPI_ISL_19555115, EPI_ISL_19557993, EPI_ISL_19558034, EPI_ISL_19560436, EPI_ISL_19560828, EPI_ISL_19560838, EPI_ISL_19561072, EPI_ISL_19561079, EPI_ISL_19562666, EPI_ISL_19575844, EPI_ISL_19575892, EPI_ISL_19577900, EPI_ISL_19578009, EPI_ISL_19578384, EPI_ISL_19585411, EPI_ISL_19585472, EPI_ISL_19588050, EPI_ISL_19588617, EPI_ISL_19588630, EPI_ISL_19589209, EPI_ISL_19589226, EPI_ISL_19589644, EPI_ISL_19591990, EPI_ISL_19600727, EPI_ISL_19601813, EPI_ISL_19603076, EPI_ISL_19603396, EPI_ISL_19603766, EPI_ISL_19606872, EPI_ISL_19606873, EPI_ISL_19606893, EPI_ISL_19613477, EPI_ISL_19613481, EPI_ISL_19613618, EPI_ISL_19615536, EPI_ISL_19615541, EPI_ISL_19615759, EPI_ISL_19616027, EPI_ISL_19616479, EPI_ISL_19618593, EPI_ISL_19619689, EPI_ISL_19619734, EPI_ISL_19619757, EPI_ISL_19620010, EPI_ISL_19621085, EPI_ISL_19621954, EPI_ISL_19622266, EPI_ISL_19623358, EPI_ISL_19623503, EPI_ISL_19627737, EPI_ISL_19627776, EPI_ISL_19627778, EPI_ISL_19630975, EPI_ISL_19631111, EPI_ISL_19632962, EPI_ISL_19634022, EPI_ISL_19636173, EPI_ISL_19638051, EPI_ISL_19639319, EPI_ISL_19640624, EPI_ISL_19641389, EPI_ISL_19641840, EPI_ISL_19642771, EPI_ISL_19643417, EPI_ISL_19643444, EPI_ISL_19643542, EPI_ISL_19643616, EPI_ISL_19643666, EPI_ISL_19645977, EPI_ISL_19646339, EPI_ISL_19648297, EPI_ISL_19655163, EPI_ISL_19656451, EPI_ISL_19657328, EPI_ISL_19657519, EPI_ISL_19657528, EPI_ISL_19660180, EPI_ISL_19661881, EPI_ISL_19661997, EPI_ISL_19665170, EPI_ISL_19666574, EPI_ISL_19671026, EPI_ISL_19671598, EPI_ISL_19671605, EPI_ISL_19671606, EPI_ISL_19671607, EPI_ISL_19671887, EPI_ISL_19673194, EPI_ISL_19673201, EPI_ISL_19676136, EPI_ISL_19680534, EPI_ISL_19683226, EPI_ISL_19683235, EPI_ISL_19683886, EPI_ISL_19685790, EPI_ISL_19685811, EPI_ISL_19685815, EPI_ISL_19685993, EPI_ISL_19688052, EPI_ISL_19689294, EPI_ISL_19690695, EPI_ISL_19693383, EPI_ISL_19693416, EPI_ISL_19693419, EPI_ISL_19694042, EPI_ISL_19696986, EPI_ISL_19703902, EPI_ISL_19704129, EPI_ISL_19704643, EPI_ISL_19704646, EPI_ISL_19706712, EPI_ISL_19706713, EPI_ISL_19710331, EPI_ISL_19710359, EPI_ISL_19711118, EPI_ISL_19711122, EPI_ISL_19713810, EPI_ISL_19713821, EPI_ISL_19713832, EPI_ISL_19714066, EPI_ISL_19715063, EPI_ISL_19715322, EPI_ISL_19717002, EPI_ISL_19719176, EPI_ISL_19719216, EPI_ISL_19719218, EPI_ISL_19720736, EPI_ISL_19720749, EPI_ISL_19720751, EPI_ISL_19720757, EPI_ISL_19720765, EPI_ISL_19720774, EPI_ISL_19720781, EPI_ISL_19721494, EPI_ISL_19721498, EPI_ISL_19729765, EPI_ISL_19729811, EPI_ISL_19729840, EPI_ISL_19730527, EPI_ISL_19735517, EPI_ISL_19736097, EPI_ISL_19736266, EPI_ISL_19736299, EPI_ISL_19736349, EPI_ISL_19736527, EPI_ISL_19739887, EPI_ISL_19742521, EPI_ISL_19743439, EPI_ISL_19743558, EPI_ISL_19748970, EPI_ISL_19749286, EPI_ISL_19749348, EPI_ISL_19750081, EPI_ISL_19750087, EPI_ISL_19750479, EPI_ISL_19751001, EPI_ISL_19752288, EPI_ISL_19755631, EPI_ISL_19760571, EPI_ISL_19761402, EPI_ISL_19761928, EPI_ISL_19765520, EPI_ISL_19765527, EPI_ISL_19766581, EPI_ISL_19766606, EPI_ISL_19770265, EPI_ISL_19770267, EPI_ISL_19771106, EPI_ISL_19773769, EPI_ISL_19773770, EPI_ISL_19775232, EPI_ISL_19775886, EPI_ISL_19776542, EPI_ISL_19782005, EPI_ISL_19783524, EPI_ISL_19786156, EPI_ISL_19788439, EPI_ISL_19791384, EPI_ISL_19794365, EPI_ISL_19795466, EPI_ISL_19797371, EPI_ISL_19798087, EPI_ISL_19799368, EPI_ISL_19799392, EPI_ISL_19799712, EPI_ISL_19799810, EPI_ISL_19800009, EPI_ISL_19800089, EPI_ISL_19800094, EPI_ISL_19800098, EPI_ISL_19800151, EPI_ISL_19801979, EPI_ISL_19803172, EPI_ISL_19803215, EPI_ISL_19803225, EPI_ISL_19803382, EPI_ISL_19804293, EPI_ISL_19804491, EPI_ISL_19805234, EPI_ISL_19806441, EPI_ISL_19806502, EPI_ISL_19806836, EPI_ISL_19806981, EPI_ISL_19807128, EPI_ISL_19807229, EPI_ISL_19808706, EPI_ISL_19810078, EPI_ISL_19810444, EPI_ISL_19810684, EPI_ISL_19812168, EPI_ISL_19814897, EPI_ISL_19815994, EPI_ISL_19816020, EPI_ISL_19816100, EPI_ISL_19816126, EPI_ISL_19816143, EPI_ISL_19816182, EPI_ISL_19816207, EPI_ISL_19816266, EPI_ISL_19816276, EPI_ISL_19816297, EPI_ISL_19816344, EPI_ISL_19816356, EPI_ISL_19816434, EPI_ISL_19819753, EPI_ISL_19819801, EPI_ISL_19820321, EPI_ISL_19822739, EPI_ISL_19824345, EPI_ISL_19824493, EPI_ISL_19826284, EPI_ISL_19826623, EPI_ISL_19828130, EPI_ISL_19828714, EPI_ISL_19831372, EPI_ISL_19831976, EPI_ISL_19836303, EPI_ISL_19838390, EPI_ISL_19841148, EPI_ISL_19845799, EPI_ISL_19845890, EPI_ISL_19846096, EPI_ISL_19846112, EPI_ISL_19846130, EPI_ISL_19846377, EPI_ISL_19846620, EPI_ISL_19846657, EPI_ISL_19846658, EPI_ISL_19848250, EPI_ISL_19848532, EPI_ISL_19848571, EPI_ISL_19850000, EPI_ISL_19851510, EPI_ISL_19852484, EPI_ISL_19853694, EPI_ISL_19853731, EPI_ISL_19854645, EPI_ISL_19856109, EPI_ISL_19856123, EPI_ISL_19858418, EPI_ISL_19859249, EPI_ISL_19859991, EPI_ISL_19860001, EPI_ISL_19860628, EPI_ISL_19860631, EPI_ISL_19860648, EPI_ISL_19860650, EPI_ISL_19860663, EPI_ISL_19862012, EPI_ISL_19867834, EPI_ISL_19867960, EPI_ISL_19869240, EPI_ISL_19869461, EPI_ISL_19869479, EPI_ISL_19870045, EPI_ISL_19872997, EPI_ISL_19873015, EPI_ISL_19874592, EPI_ISL_19875482, EPI_ISL_19875516, EPI_ISL_19875907, EPI_ISL_19875923, EPI_ISL_19875929, EPI_ISL_19876467, EPI_ISL_19876468, EPI_ISL_19876475, EPI_ISL_19876688, EPI_ISL_19878325, EPI_ISL_19878543, EPI_ISL_19878655, EPI_ISL_19881774, EPI_ISL_19881782, EPI_ISL_19885459, EPI_ISL_19888289, EPI_ISL_19890790, EPI_ISL_19892080, EPI_ISL_19893141, EPI_ISL_19895139, EPI_ISL_19896098, EPI_ISL_19900414, EPI_ISL_19904802, EPI_ISL_19905384, EPI_ISL_19906764, EPI_ISL_19907172, EPI_ISL_19907176, EPI_ISL_19908300, EPI_ISL_19909161, EPI_ISL_19909472, EPI_ISL_19912125, EPI_ISL_19913944, EPI_ISL_19913962, EPI_ISL_19914005, EPI_ISL_19914095, EPI_ISL_19914209, EPI_ISL_19914518, EPI_ISL_19915549, EPI_ISL_19916900, EPI_ISL_19921952, EPI_ISL_19922099, EPI_ISL_19922152, EPI_ISL_19922153, EPI_ISL_19923571, EPI_ISL_19924741, EPI_ISL_19925769, EPI_ISL_19927172, EPI_ISL_19927619, EPI_ISL_19930618, EPI_ISL_19931371, EPI_ISL_19931603, EPI_ISL_19931761, EPI_ISL_19932828, EPI_ISL_19932862, EPI_ISL_19933840, EPI_ISL_19937235, EPI_ISL_19939096, EPI_ISL_19939840, EPI_ISL_19940375, EPI_ISL_19941809, EPI_ISL_19942866, EPI_ISL_19944411, EPI_ISL_19945844, EPI_ISL_19946231, EPI_ISL_19946695, EPI_ISL_19948061, EPI_ISL_19964231, EPI_ISL_19971003, EPI_ISL_19972226, EPI_ISL_19977768, EPI_ISL_19978422, EPI_ISL_19981086, EPI_ISL_19981113, EPI_ISL_19981563, EPI_ISL_19987040, EPI_ISL_19988382, EPI_ISL_19988544, EPI_ISL_19988982, EPI_ISL_19988985, EPI_ISL_19989313, EPI_ISL_19989447, EPI_ISL_19989546, EPI_ISL_19989696, EPI_ISL_19989888, EPI_ISL_19991141, EPI_ISL_19993030, EPI_ISL_19996562, EPI_ISL_19998366, EPI_ISL_19998531, EPI_ISL_20000997, EPI_ISL_20001882, EPI_ISL_20003020, EPI_ISL_20003783, EPI_ISL_20004263, EPI_ISL_20004532, EPI_ISL_20005159, EPI_ISL_20005257, EPI_ISL_20005829, EPI_ISL_20006380, EPI_ISL_20007517, EPI_ISL_20007659, EPI_ISL_20010416, EPI_ISL_20011218, EPI_ISL_20012094, EPI_ISL_20012599, EPI_ISL_20013003, EPI_ISL_20013131, EPI_ISL_20013626, EPI_ISL_20013761, EPI_ISL_20013799, EPI_ISL_20013935, EPI_ISL_20014644, EPI_ISL_20014744, EPI_ISL_20014843, EPI_ISL_20015117, EPI_ISL_20015176, EPI_ISL_20015192, EPI_ISL_20015246, EPI_ISL_20015340, EPI_ISL_20015538, EPI_ISL_20016121, EPI_ISL_20016529, EPI_ISL_20016537, EPI_ISL_20016564, EPI_ISL_20016684, EPI_ISL_20016782, EPI_ISL_20016815, EPI_ISL_20016838, EPI_ISL_20017364, EPI_ISL_20017606, EPI_ISL_20017773, EPI_ISL_20018061, EPI_ISL_20019035, EPI_ISL_20019468, EPI_ISL_20024326, EPI_ISL_20025108, EPI_ISL_20025341, EPI_ISL_20025539, EPI_ISL_20026339, EPI_ISL_20027365, EPI_ISL_20027593, EPI_ISL_20027953, EPI_ISL_20030503, EPI_ISL_20040387, EPI_ISL_20040730, EPI_ISL_20043225, EPI_ISL_20043488, EPI_ISL_20045296, EPI_ISL_20046476, EPI_ISL_20047831, EPI_ISL_20047892, EPI_ISL_20048939, EPI_ISL_20048957, EPI_ISL_20053144, EPI_ISL_20053774, EPI_ISL_20054078, EPI_ISL_20056146, EPI_ISL_20062254, EPI_ISL_20063868, EPI_ISL_20066350, EPI_ISL_20066351, EPI_ISL_20066352, EPI_ISL_20066353, EPI_ISL_20066354, EPI_ISL_20066355, EPI_ISL_20066356, EPI_ISL_20066357, EPI_ISL_20066358, EPI_ISL_20066359, EPI_ISL_20066360, EPI_ISL_20066487, EPI_ISL_20070801, EPI_ISL_20070849, EPI_ISL_20074733, EPI_ISL_20076185, EPI_ISL_20080050, EPI_ISL_20081356, EPI_ISL_20081602, EPI_ISL_20081628, EPI_ISL_20082525, EPI_ISL_20088904, EPI_ISL_20096211, EPI_ISL_20096224, EPI_ISL_20097661, EPI_ISL_20097746, EPI_ISL_20097747, EPI_ISL_20098386, EPI_ISL_20099967, EPI_ISL_20113880, EPI_ISL_20114442, EPI_ISL_20114467, EPI_ISL_20114468, EPI_ISL_20114469, EPI_ISL_20115114, EPI_ISL_20127947, EPI_ISL_20131508, EPI_ISL_20132435, EPI_ISL_20136586, EPI_ISL_20137604, EPI_ISL_20138079, EPI_ISL_20138923, EPI_ISL_20139405, EPI_ISL_20140153, EPI_ISL_20140838, EPI_ISL_20140845, EPI_ISL_20141945, EPI_ISL_20141955, EPI_ISL_20141982, EPI_ISL_20143974, EPI_ISL_20146793, EPI_ISL_20146795, EPI_ISL_20146858, EPI_ISL_20148475, EPI_ISL_20152555, EPI_ISL_20155076, EPI_ISL_20155864, EPI_ISL_20157513, EPI_ISL_20166121, EPI_ISL_20167020, EPI_ISL_20175070, EPI_ISL_20176704, EPI_ISL_20176761, EPI_ISL_20178658, EPI_ISL_20179076, EPI_ISL_20179179, EPI_ISL_20181458, EPI_ISL_20181461, EPI_ISL_20182683, EPI_ISL_20182757, EPI_ISL_20184766, EPI_ISL_20185296, EPI_ISL_20185471, EPI_ISL_20185528, EPI_ISL_20193978, EPI_ISL_20193979, EPI_ISL_20193993, EPI_ISL_20193997, EPI_ISL_20194032, EPI_ISL_20196214, EPI_ISL_20196709, EPI_ISL_20196755, EPI_ISL_20196768, EPI_ISL_20196838, EPI_ISL_20198727, EPI_ISL_20212488, EPI_ISL_20223218, EPI_ISL_20223363, EPI_ISL_20223369, EPI_ISL_20225608, EPI_ISL_20227249, EPI_ISL_20231309, EPI_ISL_20231310, EPI_ISL_20231311, EPI_ISL_20231312, EPI_ISL_20231313, EPI_ISL_20231314, EPI_ISL_20231315, EPI_ISL_20231316, EPI_ISL_20233754, EPI_ISL_20237211, EPI_ISL_20238076, EPI_ISL_20238184, EPI_ISL_20239758")

mov_old_doc = stringlist_to_set("EPI_ISL_10512732, EPI_ISL_10513884, EPI_ISL_11022200, EPI_ISL_11174220, EPI_ISL_11224116, EPI_ISL_11353416, EPI_ISL_11664910, EPI_ISL_11782428, EPI_ISL_12014556, EPI_ISL_12559054, EPI_ISL_12830302, EPI_ISL_13276436, EPI_ISL_13294593, EPI_ISL_13358749, EPI_ISL_13358804, EPI_ISL_13723805, EPI_ISL_14138850, EPI_ISL_14346370, EPI_ISL_14429354, EPI_ISL_14466939, EPI_ISL_14467031, EPI_ISL_14526287, EPI_ISL_14573187, EPI_ISL_14580651, EPI_ISL_14601448, EPI_ISL_14616144, EPI_ISL_14667656, EPI_ISL_14788088, EPI_ISL_14809350, EPI_ISL_14818681, EPI_ISL_14830201, EPI_ISL_14916287, EPI_ISL_14916417, EPI_ISL_14916471, EPI_ISL_14929774, EPI_ISL_14945818, EPI_ISL_14946908, EPI_ISL_15081472, EPI_ISL_15106590, EPI_ISL_15111552, EPI_ISL_15113210, EPI_ISL_15118652, EPI_ISL_15119550, EPI_ISL_15119586, EPI_ISL_15119587, EPI_ISL_15119899, EPI_ISL_15125352, EPI_ISL_15157642, EPI_ISL_15160596, EPI_ISL_15212355, EPI_ISL_15212402, EPI_ISL_15258694, EPI_ISL_15258697, EPI_ISL_15258702, EPI_ISL_15258714, EPI_ISL_15258754, EPI_ISL_15260772, EPI_ISL_15276586, EPI_ISL_15341577, EPI_ISL_15343310, EPI_ISL_15352141, EPI_ISL_15354679, EPI_ISL_15356322, EPI_ISL_15357173, EPI_ISL_15357186, EPI_ISL_15357564, EPI_ISL_15385709, EPI_ISL_15385769, EPI_ISL_15395718, EPI_ISL_15410430, EPI_ISL_15471118, EPI_ISL_15471127, EPI_ISL_15476838, EPI_ISL_15476890, EPI_ISL_15477002, EPI_ISL_15477006, EPI_ISL_15477014, EPI_ISL_15477080, EPI_ISL_15477087, EPI_ISL_15477089, EPI_ISL_15477122, EPI_ISL_15477123, EPI_ISL_15477133, EPI_ISL_15477167, EPI_ISL_15523458, EPI_ISL_15532754, EPI_ISL_15532755, EPI_ISL_15532776, EPI_ISL_15532778, EPI_ISL_15533551, EPI_ISL_15546710, EPI_ISL_15578381, EPI_ISL_15629811, EPI_ISL_15641087, EPI_ISL_15652740, EPI_ISL_15666161, EPI_ISL_15670984, EPI_ISL_15673084, EPI_ISL_15673196, EPI_ISL_15673587, EPI_ISL_15693720, EPI_ISL_15697288, EPI_ISL_15729147, EPI_ISL_15731617, EPI_ISL_15731631, EPI_ISL_15732415, EPI_ISL_15741410, EPI_ISL_15755946, EPI_ISL_15755997, EPI_ISL_15756004, EPI_ISL_15756044, EPI_ISL_15765732, EPI_ISL_15797427, EPI_ISL_15797464, EPI_ISL_15797751, EPI_ISL_15797823, EPI_ISL_15797872, EPI_ISL_15805129, EPI_ISL_15808072, EPI_ISL_15820336, EPI_ISL_15820414, EPI_ISL_15825627, EPI_ISL_15829108, EPI_ISL_15830052, EPI_ISL_15838528, EPI_ISL_15857468, EPI_ISL_15870952, EPI_ISL_15871047, EPI_ISL_15894376, EPI_ISL_15895664, EPI_ISL_15895805, EPI_ISL_15906500, EPI_ISL_15907744, EPI_ISL_15910110, EPI_ISL_15910797, EPI_ISL_15926723, EPI_ISL_15938365, EPI_ISL_15938543, EPI_ISL_15941958, EPI_ISL_15955458, EPI_ISL_15955461, EPI_ISL_15956078, EPI_ISL_15957375, EPI_ISL_15959194, EPI_ISL_15979054, EPI_ISL_15979220, EPI_ISL_15980454, EPI_ISL_15980534, EPI_ISL_16004112, EPI_ISL_16004139, EPI_ISL_16004173, EPI_ISL_16018930, EPI_ISL_16036592, EPI_ISL_16046816, EPI_ISL_16048672, EPI_ISL_16050568, EPI_ISL_16058939, EPI_ISL_16074147, EPI_ISL_16077721, EPI_ISL_16077819, EPI_ISL_16077827, EPI_ISL_16078136, EPI_ISL_16081297, EPI_ISL_16093705, EPI_ISL_16101357, EPI_ISL_16119461, EPI_ISL_16119652, EPI_ISL_16119687, EPI_ISL_16119697, EPI_ISL_16131605, EPI_ISL_16133537, EPI_ISL_16133850, EPI_ISL_16134022, EPI_ISL_16137724, EPI_ISL_16172206, EPI_ISL_16172628, EPI_ISL_16173251, EPI_ISL_16175780, EPI_ISL_16191277, EPI_ISL_16193751, EPI_ISL_16196595, EPI_ISL_16205170, EPI_ISL_16210278, EPI_ISL_16210314, EPI_ISL_16210315, EPI_ISL_16210321, EPI_ISL_16210344, EPI_ISL_16210352, EPI_ISL_16210356, EPI_ISL_16257294, EPI_ISL_16272927, EPI_ISL_16273053, EPI_ISL_16273094, EPI_ISL_16273106, EPI_ISL_16273691, EPI_ISL_16273904, EPI_ISL_16279440, EPI_ISL_16279922, EPI_ISL_16298573, EPI_ISL_16298574, EPI_ISL_16315643, EPI_ISL_16315645, EPI_ISL_16315710, EPI_ISL_16342216, EPI_ISL_16359576, EPI_ISL_16359648, EPI_ISL_16359676, EPI_ISL_16359695, EPI_ISL_16359707, EPI_ISL_16359708, EPI_ISL_16359709, EPI_ISL_16359721, EPI_ISL_16359724, EPI_ISL_16359730, EPI_ISL_16359737, EPI_ISL_16359739, EPI_ISL_16360560, EPI_ISL_16360578, EPI_ISL_16360615, EPI_ISL_16360631, EPI_ISL_16364267, EPI_ISL_16366337, EPI_ISL_16374061, EPI_ISL_16374062, EPI_ISL_16374063, EPI_ISL_16374064, EPI_ISL_16374077, EPI_ISL_16374103, EPI_ISL_16378479, EPI_ISL_16386695, EPI_ISL_16391752, EPI_ISL_16394844, EPI_ISL_16394849, EPI_ISL_16433986, EPI_ISL_16434219, EPI_ISL_16434682, EPI_ISL_16435411, EPI_ISL_16435758, EPI_ISL_16435776, EPI_ISL_16435834, EPI_ISL_16435848, EPI_ISL_16436045, EPI_ISL_16455847, EPI_ISL_16466074, EPI_ISL_16474400, EPI_ISL_16480279, EPI_ISL_16482060, EPI_ISL_16491962, EPI_ISL_16493846, EPI_ISL_16493881, EPI_ISL_16494105, EPI_ISL_16495643, EPI_ISL_16507760, EPI_ISL_16521266, EPI_ISL_16521681, EPI_ISL_16522889, EPI_ISL_16523533, EPI_ISL_16523565, EPI_ISL_16523595, EPI_ISL_16527181, EPI_ISL_16527529, EPI_ISL_16546118, EPI_ISL_16548794, EPI_ISL_16554670, EPI_ISL_16559981, EPI_ISL_16568182, EPI_ISL_16568197, EPI_ISL_16570236, EPI_ISL_16570568, EPI_ISL_16570602, EPI_ISL_16581363, EPI_ISL_16581388, EPI_ISL_16581578, EPI_ISL_16585008, EPI_ISL_16585433, EPI_ISL_16611373, EPI_ISL_16611412, EPI_ISL_16611422, EPI_ISL_16611506, EPI_ISL_16611514, EPI_ISL_16611550, EPI_ISL_16611563, EPI_ISL_16611624, EPI_ISL_16611667, EPI_ISL_16611673, EPI_ISL_16611682, EPI_ISL_16611698, EPI_ISL_16611778, EPI_ISL_16611838, EPI_ISL_16611848, EPI_ISL_16611855, EPI_ISL_16612037, EPI_ISL_16616330, EPI_ISL_16617719, EPI_ISL_16617775, EPI_ISL_16633089, EPI_ISL_16633129, EPI_ISL_16639468, EPI_ISL_16639472, EPI_ISL_16641545, EPI_ISL_16645057, EPI_ISL_16645120, EPI_ISL_16664013, EPI_ISL_16675393, EPI_ISL_16678683, EPI_ISL_16691130, EPI_ISL_16691138, EPI_ISL_16691144, EPI_ISL_16691244, EPI_ISL_16702617, EPI_ISL_16702731, EPI_ISL_16711169, EPI_ISL_16716050, EPI_ISL_16716112, EPI_ISL_16716666, EPI_ISL_16716667, EPI_ISL_16717190, EPI_ISL_16723860, EPI_ISL_16729430, EPI_ISL_16742981, EPI_ISL_16746019, EPI_ISL_16783884, EPI_ISL_16806139, EPI_ISL_16824117, EPI_ISL_16825886, EPI_ISL_16825887, EPI_ISL_16825902, EPI_ISL_16825911, EPI_ISL_16825973, EPI_ISL_16825975, EPI_ISL_16826011, EPI_ISL_16826162, EPI_ISL_16827461, EPI_ISL_16829544, EPI_ISL_16830824, EPI_ISL_16831676, EPI_ISL_16832813, EPI_ISL_16836551, EPI_ISL_16840631, EPI_ISL_16846006, EPI_ISL_16850006, EPI_ISL_16850282, EPI_ISL_16856565, EPI_ISL_16886081, EPI_ISL_16887730, EPI_ISL_16888824, EPI_ISL_16889192, EPI_ISL_16900219, EPI_ISL_16901108, EPI_ISL_16902309, EPI_ISL_16902361, EPI_ISL_16902363, EPI_ISL_16902379, EPI_ISL_16904536, EPI_ISL_16909958, EPI_ISL_16910033, EPI_ISL_16910057, EPI_ISL_16910058, EPI_ISL_16910139, EPI_ISL_16910202, EPI_ISL_16910382, EPI_ISL_16910409, EPI_ISL_16912192, EPI_ISL_16912224, EPI_ISL_16912699, EPI_ISL_16925018, EPI_ISL_16931898, EPI_ISL_16939136, EPI_ISL_16939502, EPI_ISL_16939509, EPI_ISL_16939520, EPI_ISL_16947370, EPI_ISL_16949310, EPI_ISL_16949324, EPI_ISL_16949325, EPI_ISL_16949387, EPI_ISL_16949791, EPI_ISL_16950012, EPI_ISL_16950629, EPI_ISL_16953777, EPI_ISL_16953781, EPI_ISL_16953798, EPI_ISL_16953799, EPI_ISL_16953818, EPI_ISL_16953822, EPI_ISL_16953828, EPI_ISL_16953830, EPI_ISL_16953838, EPI_ISL_16953839, EPI_ISL_16953840, EPI_ISL_16953847, EPI_ISL_16953922, EPI_ISL_16954432, EPI_ISL_16957344, EPI_ISL_16957451, EPI_ISL_16964941, EPI_ISL_16969564, EPI_ISL_16970275, EPI_ISL_16971354, EPI_ISL_16973054, EPI_ISL_16973491, EPI_ISL_16973514, EPI_ISL_16974198, EPI_ISL_16974315, EPI_ISL_16974382, EPI_ISL_16974837, EPI_ISL_16976164, EPI_ISL_16977847, EPI_ISL_16983895, EPI_ISL_16993160, EPI_ISL_16994359, EPI_ISL_16994360, EPI_ISL_16994528, EPI_ISL_16994534, EPI_ISL_16994641, EPI_ISL_16994642, EPI_ISL_16994643, EPI_ISL_16994655, EPI_ISL_16997635, EPI_ISL_17000779, EPI_ISL_17001818, EPI_ISL_17013912, EPI_ISL_17013920, EPI_ISL_17014551, EPI_ISL_17014690, EPI_ISL_17015386, EPI_ISL_17015403, EPI_ISL_17015508, EPI_ISL_17024862, EPI_ISL_17027606, EPI_ISL_17027639, EPI_ISL_17027948, EPI_ISL_17027949, EPI_ISL_17028105, EPI_ISL_17028190, EPI_ISL_17029568, EPI_ISL_17029620, EPI_ISL_17040153, EPI_ISL_17040170, EPI_ISL_17040185, EPI_ISL_17042013, EPI_ISL_17046849, EPI_ISL_17047075, EPI_ISL_17047300, EPI_ISL_17053152, EPI_ISL_17053374, EPI_ISL_17058072, EPI_ISL_17058073, EPI_ISL_17062470, EPI_ISL_17062513, EPI_ISL_17068606, EPI_ISL_17076266, EPI_ISL_17079912, EPI_ISL_17089619, EPI_ISL_17090505, EPI_ISL_17090748, EPI_ISL_17091144, EPI_ISL_17092036, EPI_ISL_17092037, EPI_ISL_17092038, EPI_ISL_17120889, EPI_ISL_17124976, EPI_ISL_17131649, EPI_ISL_17144768, EPI_ISL_17146391, EPI_ISL_17150326, EPI_ISL_17150374, EPI_ISL_17150380, EPI_ISL_17154893, EPI_ISL_17156691, EPI_ISL_17162825, EPI_ISL_17162929, EPI_ISL_17163260, EPI_ISL_17164036, EPI_ISL_17164425, EPI_ISL_17168296, EPI_ISL_17171052, EPI_ISL_17171054, EPI_ISL_17171074, EPI_ISL_17173504, EPI_ISL_17180537, EPI_ISL_17185359, EPI_ISL_17185391, EPI_ISL_17185393, EPI_ISL_17185521, EPI_ISL_17187061, EPI_ISL_17187082, EPI_ISL_17187115, EPI_ISL_17187405, EPI_ISL_17190415, EPI_ISL_17193320, EPI_ISL_17193409, EPI_ISL_17197288, EPI_ISL_17198749, EPI_ISL_17198796, EPI_ISL_17198959, EPI_ISL_17198971, EPI_ISL_17205507, EPI_ISL_17212042, EPI_ISL_17212254, EPI_ISL_17212450, EPI_ISL_17213648, EPI_ISL_17213677, EPI_ISL_17233016, EPI_ISL_17237758, EPI_ISL_17239970, EPI_ISL_17241376, EPI_ISL_17243067, EPI_ISL_17244428, EPI_ISL_17244430, EPI_ISL_17245777, EPI_ISL_17249116, EPI_ISL_17259971, EPI_ISL_17273231, EPI_ISL_17273546, EPI_ISL_17279072, EPI_ISL_17283738, EPI_ISL_17288373, EPI_ISL_17292738, EPI_ISL_17292778, EPI_ISL_17292812, EPI_ISL_17295274, EPI_ISL_17299688, EPI_ISL_17305358, EPI_ISL_17322295, EPI_ISL_17326431, EPI_ISL_17329306, EPI_ISL_17342779, EPI_ISL_17350941, EPI_ISL_17372713, EPI_ISL_17373645, EPI_ISL_17373894, EPI_ISL_17374733, EPI_ISL_17374799, EPI_ISL_17374806, EPI_ISL_17374807, EPI_ISL_17375007, EPI_ISL_17382768, EPI_ISL_17387866, EPI_ISL_17388895, EPI_ISL_17388906, EPI_ISL_17390815, EPI_ISL_17393458, EPI_ISL_17404084, EPI_ISL_17409157, EPI_ISL_17413069, EPI_ISL_17413882, EPI_ISL_17419568, EPI_ISL_17422869, EPI_ISL_17430511, EPI_ISL_17433569, EPI_ISL_17463210, EPI_ISL_17464836, EPI_ISL_17465459, EPI_ISL_17468440, EPI_ISL_17477106, EPI_ISL_17485616, EPI_ISL_17485727, EPI_ISL_17485728, EPI_ISL_17493590, EPI_ISL_17501569, EPI_ISL_17503268, EPI_ISL_17504112, EPI_ISL_17511739, EPI_ISL_17516925, EPI_ISL_17517664, EPI_ISL_17521432, EPI_ISL_17527442, EPI_ISL_17527471, EPI_ISL_17527478, EPI_ISL_17527567, EPI_ISL_17539348, EPI_ISL_17544585, EPI_ISL_17544916, EPI_ISL_17547750, EPI_ISL_17551316, EPI_ISL_17556166, EPI_ISL_17558242, EPI_ISL_17558487, EPI_ISL_17558870, EPI_ISL_17559934, EPI_ISL_17592407, EPI_ISL_17604727, EPI_ISL_17604844, EPI_ISL_17605096, EPI_ISL_17621484, EPI_ISL_17622137, EPI_ISL_17628855, EPI_ISL_17642930, EPI_ISL_17647324, EPI_ISL_17649109, EPI_ISL_17656014, EPI_ISL_17656052, EPI_ISL_17657007, EPI_ISL_17657210, EPI_ISL_17660818, EPI_ISL_17661045, EPI_ISL_17661237, EPI_ISL_17661261, EPI_ISL_17661270, EPI_ISL_17662899, EPI_ISL_17666331, EPI_ISL_17682891, EPI_ISL_17682942, EPI_ISL_17683117, EPI_ISL_17683214, EPI_ISL_17683243, EPI_ISL_17685027, EPI_ISL_17692258, EPI_ISL_17695126, EPI_ISL_17695180, EPI_ISL_17696632, EPI_ISL_17696673, EPI_ISL_17696719, EPI_ISL_17698434, EPI_ISL_17703866, EPI_ISL_17714932, EPI_ISL_17716296, EPI_ISL_17718464, EPI_ISL_17719091, EPI_ISL_17720695, EPI_ISL_17722014, EPI_ISL_17724271, EPI_ISL_17724304, EPI_ISL_17737041, EPI_ISL_17740508, EPI_ISL_17742125, EPI_ISL_17762760, EPI_ISL_17762820, EPI_ISL_17764084, EPI_ISL_17765632, EPI_ISL_17765706, EPI_ISL_17765717, EPI_ISL_17775797, EPI_ISL_17775798, EPI_ISL_17776606, EPI_ISL_17777067, EPI_ISL_17784540, EPI_ISL_17784548, EPI_ISL_17784550, EPI_ISL_17784552, EPI_ISL_17784569, EPI_ISL_17784593, EPI_ISL_17790241, EPI_ISL_17791006, EPI_ISL_17793597, EPI_ISL_17793709, EPI_ISL_17796704, EPI_ISL_17802162, EPI_ISL_17802448, EPI_ISL_17802471, EPI_ISL_17802863, EPI_ISL_17803152, EPI_ISL_17806624, EPI_ISL_17806780, EPI_ISL_17806785, EPI_ISL_17807899, EPI_ISL_17821941, EPI_ISL_17822275, EPI_ISL_17822422, EPI_ISL_17822859, EPI_ISL_17822940, EPI_ISL_17823538, EPI_ISL_17824267, EPI_ISL_17824292, EPI_ISL_17824608, EPI_ISL_17824670, EPI_ISL_17826031, EPI_ISL_17826225, EPI_ISL_17826355, EPI_ISL_17826392, EPI_ISL_17827176, EPI_ISL_17827221, EPI_ISL_17827261, EPI_ISL_17829319, EPI_ISL_17829398, EPI_ISL_17832368, EPI_ISL_17849659, EPI_ISL_17851798, EPI_ISL_17857957, EPI_ISL_17859873, EPI_ISL_17859887, EPI_ISL_17879222, EPI_ISL_17882742, EPI_ISL_17960272, EPI_ISL_17975460, EPI_ISL_17977889, EPI_ISL_17977890, EPI_ISL_17977939, EPI_ISL_17980375, EPI_ISL_17980376, EPI_ISL_17980377, EPI_ISL_17985070, EPI_ISL_17987531, EPI_ISL_17994215, EPI_ISL_17997273, EPI_ISL_17999525, EPI_ISL_17999535, EPI_ISL_17999631, EPI_ISL_17999795, EPI_ISL_17999836, EPI_ISL_18000297, EPI_ISL_18001169, EPI_ISL_18010323, EPI_ISL_18014681, EPI_ISL_18031645, EPI_ISL_18032375, EPI_ISL_18039667, EPI_ISL_18040500, EPI_ISL_18041690, EPI_ISL_18042063, EPI_ISL_18043809, EPI_ISL_18051511, EPI_ISL_18052776, EPI_ISL_18057695, EPI_ISL_18060832, EPI_ISL_18063266, EPI_ISL_18063275, EPI_ISL_18063343, EPI_ISL_18063346, EPI_ISL_18063361, EPI_ISL_18063384, EPI_ISL_18063710, EPI_ISL_18063741, EPI_ISL_18063779, EPI_ISL_18063898, EPI_ISL_18064050, EPI_ISL_18064055, EPI_ISL_18064103, EPI_ISL_18064142, EPI_ISL_18064155, EPI_ISL_18064235, EPI_ISL_18064253, EPI_ISL_18064299, EPI_ISL_18064307, EPI_ISL_18064328, EPI_ISL_18064337, EPI_ISL_18064383, EPI_ISL_18065323, EPI_ISL_18066648, EPI_ISL_18066669, EPI_ISL_18066701, EPI_ISL_18076272, EPI_ISL_18077730, EPI_ISL_18082776, EPI_ISL_18093842, EPI_ISL_18100370, EPI_ISL_18100403, EPI_ISL_18100404, EPI_ISL_18100416, EPI_ISL_18100417, EPI_ISL_18100418, EPI_ISL_18108242, EPI_ISL_18112575, EPI_ISL_18112668, EPI_ISL_18115399, EPI_ISL_18117671, EPI_ISL_18127619, EPI_ISL_18127909, EPI_ISL_18127961, EPI_ISL_18134274, EPI_ISL_18134756, EPI_ISL_18138526, EPI_ISL_18139339, EPI_ISL_18139659, EPI_ISL_18166414, EPI_ISL_18208967, EPI_ISL_18213578, EPI_ISL_18215762, EPI_ISL_18215772, EPI_ISL_18216065, EPI_ISL_18218555, EPI_ISL_18218775, EPI_ISL_18218935, EPI_ISL_18218936, EPI_ISL_18225292, EPI_ISL_18226914, EPI_ISL_18237064, EPI_ISL_18247242, EPI_ISL_18265252, EPI_ISL_18272809, EPI_ISL_18280058, EPI_ISL_18285922, EPI_ISL_18285953, EPI_ISL_18300222, EPI_ISL_18300587, EPI_ISL_18306489, EPI_ISL_18316786, EPI_ISL_18320556, EPI_ISL_18343894, EPI_ISL_18347703, EPI_ISL_18350437, EPI_ISL_18352397, EPI_ISL_18354145, EPI_ISL_18355510, EPI_ISL_18365833, EPI_ISL_18367086, EPI_ISL_18375185, EPI_ISL_18375538, EPI_ISL_18376234, EPI_ISL_18382930, EPI_ISL_18382993, EPI_ISL_18383001, EPI_ISL_18390814, EPI_ISL_18423785, EPI_ISL_18432517, EPI_ISL_18432566, EPI_ISL_18434659, EPI_ISL_18435860, EPI_ISL_18440660, EPI_ISL_18449837, EPI_ISL_18461692, EPI_ISL_18466251, EPI_ISL_18471046, EPI_ISL_18475421, EPI_ISL_18475423, EPI_ISL_18475424, EPI_ISL_18475534, EPI_ISL_18498513, EPI_ISL_18502004, EPI_ISL_18505899, EPI_ISL_18509397, EPI_ISL_18509398, EPI_ISL_18509399, EPI_ISL_18512167, EPI_ISL_18512168, EPI_ISL_18512169, EPI_ISL_18520761, EPI_ISL_18520956, EPI_ISL_18551236, EPI_ISL_18551245, EPI_ISL_18566482, EPI_ISL_18566696, EPI_ISL_18575585, EPI_ISL_18623910, EPI_ISL_18624229, EPI_ISL_18624532, EPI_ISL_18633916, EPI_ISL_18639764, EPI_ISL_18650558, EPI_ISL_18650648, EPI_ISL_18662705, EPI_ISL_18687076, EPI_ISL_18687807, EPI_ISL_18687813, EPI_ISL_18687815, EPI_ISL_18697820, EPI_ISL_18711358, EPI_ISL_18712220, EPI_ISL_18715624, EPI_ISL_18715897, EPI_ISL_18720533, EPI_ISL_18720935, EPI_ISL_18730535, EPI_ISL_18742836, EPI_ISL_18743070, EPI_ISL_18744493, EPI_ISL_18747739, EPI_ISL_18749619, EPI_ISL_18751663, EPI_ISL_18751845, EPI_ISL_18751862, EPI_ISL_18758080, EPI_ISL_18760281, EPI_ISL_18760288, EPI_ISL_18760322, EPI_ISL_18792280, EPI_ISL_18792992, EPI_ISL_18793204, EPI_ISL_18798820, EPI_ISL_18799141, EPI_ISL_18843955, EPI_ISL_18857091, EPI_ISL_18877237, EPI_ISL_18878774, EPI_ISL_18886497, EPI_ISL_18895660, EPI_ISL_18904633, EPI_ISL_18904635, EPI_ISL_18922726, EPI_ISL_18924174, EPI_ISL_18931648, EPI_ISL_18931705, EPI_ISL_18931859, EPI_ISL_18932551, EPI_ISL_18960210, EPI_ISL_18967490, EPI_ISL_18992540, EPI_ISL_18994815, EPI_ISL_19005505, EPI_ISL_19009620, EPI_ISL_19009621, EPI_ISL_19009623, EPI_ISL_19009624, EPI_ISL_19015609, EPI_ISL_19021722, EPI_ISL_19024035, EPI_ISL_19024044, EPI_ISL_19024068, EPI_ISL_19024081, EPI_ISL_19037171, EPI_ISL_19050132, EPI_ISL_19054784, EPI_ISL_19057886, EPI_ISL_19059047, EPI_ISL_19071524, EPI_ISL_19080112, EPI_ISL_19082638, EPI_ISL_19084871, EPI_ISL_19084883, EPI_ISL_19085033, EPI_ISL_19085070, EPI_ISL_19085078, EPI_ISL_19085135, EPI_ISL_19096287, EPI_ISL_19108340, EPI_ISL_19122484, EPI_ISL_19135284, EPI_ISL_19140391, EPI_ISL_19150867, EPI_ISL_19150870, EPI_ISL_19168138, EPI_ISL_19169477, EPI_ISL_19169870, EPI_ISL_19169976, EPI_ISL_19175684, EPI_ISL_19180359, EPI_ISL_19180367, EPI_ISL_19197718, EPI_ISL_19226538, EPI_ISL_19239735, EPI_ISL_19269562, EPI_ISL_19269713, EPI_ISL_19281165, EPI_ISL_19289747, EPI_ISL_19292254, EPI_ISL_19298398, EPI_ISL_19300597, EPI_ISL_19305340, EPI_ISL_19308874, EPI_ISL_19308875, EPI_ISL_19308876, EPI_ISL_19308877, EPI_ISL_19323028, EPI_ISL_19328684, EPI_ISL_19333890, EPI_ISL_19337826, EPI_ISL_19340884, EPI_ISL_19363557")    
mov_new_doc = stringlist_to_set("EPI_ISL_1092344, EPI_ISL_2768315, EPI_ISL_2968466, EPI_ISL_3304784, EPI_ISL_3326806, EPI_ISL_3464544, EPI_ISL_3758018, EPI_ISL_4012065, EPI_ISL_4171202, EPI_ISL_4201810, EPI_ISL_4536418, EPI_ISL_5115827, EPI_ISL_5511463, EPI_ISL_5581788, EPI_ISL_6082708, EPI_ISL_6162906, EPI_ISL_6298830, EPI_ISL_6831874, EPI_ISL_7472964, EPI_ISL_7614031, EPI_ISL_8243566, EPI_ISL_8261058, EPI_ISL_8401858, EPI_ISL_9292221, EPI_ISL_9352281, EPI_ISL_9382677, EPI_ISL_9560035, EPI_ISL_9564988, EPI_ISL_9775279, EPI_ISL_9961511, EPI_ISL_10012883, EPI_ISL_10127751, EPI_ISL_10213024, EPI_ISL_10280217, EPI_ISL_10441882, EPI_ISL_10625202, EPI_ISL_10651538, EPI_ISL_10654545, EPI_ISL_10668128, EPI_ISL_10738796, EPI_ISL_10766361, EPI_ISL_10896078, EPI_ISL_10910076, EPI_ISL_11030076, EPI_ISL_11030366, EPI_ISL_11050901, EPI_ISL_11065112, EPI_ISL_11084474, EPI_ISL_11230018, EPI_ISL_11255409, EPI_ISL_11265321, EPI_ISL_11282729, EPI_ISL_11310855, EPI_ISL_11416915, EPI_ISL_11442788, EPI_ISL_11514566, EPI_ISL_11621053, EPI_ISL_11630027, EPI_ISL_11631155, EPI_ISL_11855687, EPI_ISL_11855713, EPI_ISL_11879008, EPI_ISL_11879025, EPI_ISL_11892804, EPI_ISL_11956947, EPI_ISL_12043591, EPI_ISL_12080356, EPI_ISL_12107836, EPI_ISL_12180755, EPI_ISL_12252373, EPI_ISL_12260199, EPI_ISL_12338023, EPI_ISL_12352282, EPI_ISL_12391660, EPI_ISL_12447732, EPI_ISL_12456735, EPI_ISL_12486036, EPI_ISL_12582161, EPI_ISL_12583464, EPI_ISL_12601110, EPI_ISL_12626284, EPI_ISL_12626387, EPI_ISL_12630743, EPI_ISL_12636130, EPI_ISL_12640988, EPI_ISL_12666444, EPI_ISL_12666642, EPI_ISL_12674754, EPI_ISL_12681076, EPI_ISL_12837375, EPI_ISL_12845916, EPI_ISL_12849707, EPI_ISL_12850240, EPI_ISL_12850262, EPI_ISL_12883279, EPI_ISL_12905490, EPI_ISL_12933035, EPI_ISL_12943505, EPI_ISL_12980006, EPI_ISL_13021351, EPI_ISL_13037364, EPI_ISL_13056045, EPI_ISL_13087460, EPI_ISL_13106533, EPI_ISL_13107201, EPI_ISL_13214811, EPI_ISL_13217407, EPI_ISL_13261618, EPI_ISL_13274125, EPI_ISL_13274357, EPI_ISL_13276187, EPI_ISL_13289724, EPI_ISL_13291196, EPI_ISL_13358801, EPI_ISL_13376289, EPI_ISL_13390266, EPI_ISL_13449336, EPI_ISL_13458696, EPI_ISL_13464576, EPI_ISL_13465017, EPI_ISL_13467791, EPI_ISL_13496740, EPI_ISL_13498905, EPI_ISL_13501768, EPI_ISL_13577993, EPI_ISL_13613050, EPI_ISL_13614366, EPI_ISL_13614367, EPI_ISL_13614401, EPI_ISL_13614610, EPI_ISL_13691164, EPI_ISL_13694672, EPI_ISL_13698383, EPI_ISL_13709091, EPI_ISL_13710220, EPI_ISL_13723805, EPI_ISL_13734677, EPI_ISL_13737005, EPI_ISL_13745743, EPI_ISL_13763841, EPI_ISL_13859056, EPI_ISL_13867663, EPI_ISL_13884501, EPI_ISL_13884648, EPI_ISL_13890658, EPI_ISL_13892609, EPI_ISL_13893030, EPI_ISL_13907925, EPI_ISL_13947241, EPI_ISL_13968198, EPI_ISL_13979237, EPI_ISL_13992801, EPI_ISL_14001467, EPI_ISL_14016773, EPI_ISL_14021631, EPI_ISL_14030106, EPI_ISL_14069091, EPI_ISL_14120400, EPI_ISL_14136918, EPI_ISL_14163050, EPI_ISL_14179475, EPI_ISL_14189255, EPI_ISL_14218792, EPI_ISL_14251033, EPI_ISL_14268791, EPI_ISL_14278233, EPI_ISL_14285562, EPI_ISL_14285646, EPI_ISL_14298694, EPI_ISL_14312503, EPI_ISL_14377405, EPI_ISL_14382457, EPI_ISL_14388832, EPI_ISL_14409577, EPI_ISL_14429373, EPI_ISL_14466720, EPI_ISL_14509715, EPI_ISL_14516309, EPI_ISL_14536391, EPI_ISL_14541651, EPI_ISL_14548126, EPI_ISL_14548186, EPI_ISL_14548996, EPI_ISL_14589022, EPI_ISL_14617298, EPI_ISL_14620632, EPI_ISL_14636356, EPI_ISL_14654283, EPI_ISL_14709471, EPI_ISL_14716868, EPI_ISL_14717521, EPI_ISL_14789011, EPI_ISL_14794264, EPI_ISL_14804388, EPI_ISL_14810025, EPI_ISL_14810089, EPI_ISL_14835350, EPI_ISL_14913266, EPI_ISL_14924994, EPI_ISL_14930743, EPI_ISL_14932144, EPI_ISL_14937861, EPI_ISL_14952616, EPI_ISL_14989193, EPI_ISL_15010302, EPI_ISL_15020116, EPI_ISL_15058728, EPI_ISL_15077793, EPI_ISL_15081501, EPI_ISL_15090054, EPI_ISL_15098796, EPI_ISL_15106739, EPI_ISL_15106758, EPI_ISL_15111369, EPI_ISL_15118530, EPI_ISL_15118652, EPI_ISL_15118683, EPI_ISL_15119798, EPI_ISL_15119826, EPI_ISL_15119830, EPI_ISL_15120114, EPI_ISL_15120173, EPI_ISL_15145266, EPI_ISL_15150344, EPI_ISL_15165861, EPI_ISL_15166985, EPI_ISL_15167502, EPI_ISL_15176036, EPI_ISL_15185400, EPI_ISL_15185557, EPI_ISL_15185627, EPI_ISL_15192815, EPI_ISL_15192964, EPI_ISL_15228995, EPI_ISL_15240404, EPI_ISL_15248977, EPI_ISL_15260783, EPI_ISL_15268354, EPI_ISL_15270718, EPI_ISL_15276610, EPI_ISL_15292147, EPI_ISL_15301009, EPI_ISL_15301025, EPI_ISL_15343208, EPI_ISL_15343310, EPI_ISL_15377262, EPI_ISL_15386088, EPI_ISL_15416260, EPI_ISL_15416332, EPI_ISL_15422263, EPI_ISL_15428954, EPI_ISL_15471292, EPI_ISL_15472544, EPI_ISL_15475787, EPI_ISL_15476096, EPI_ISL_15477124, EPI_ISL_15488495, EPI_ISL_15488731, EPI_ISL_15489332, EPI_ISL_15494887, EPI_ISL_15531670, EPI_ISL_15532764, EPI_ISL_15541106, EPI_ISL_15546790, EPI_ISL_15581363, EPI_ISL_15581659, EPI_ISL_15582663, EPI_ISL_15609973, EPI_ISL_15610206, EPI_ISL_15614395, EPI_ISL_15616465, EPI_ISL_15617522, EPI_ISL_15627130, EPI_ISL_15642469, EPI_ISL_15653115, EPI_ISL_15653305, EPI_ISL_15673829, EPI_ISL_15693069, EPI_ISL_15718497, EPI_ISL_15722243, EPI_ISL_15729046, EPI_ISL_15731918, EPI_ISL_15734089, EPI_ISL_15736020, EPI_ISL_15736490, EPI_ISL_15749655, EPI_ISL_15756058, EPI_ISL_15758276, EPI_ISL_15803022, EPI_ISL_15803045, EPI_ISL_15820336, EPI_ISL_15820574, EPI_ISL_15825048, EPI_ISL_15829598, EPI_ISL_15865651, EPI_ISL_15870196, EPI_ISL_15871064, EPI_ISL_15875255, EPI_ISL_15876857, EPI_ISL_15891417, EPI_ISL_15895830, EPI_ISL_15896932, EPI_ISL_15896947, EPI_ISL_15898266, EPI_ISL_15905654, EPI_ISL_15908757, EPI_ISL_15917272, EPI_ISL_15938365, EPI_ISL_15940018, EPI_ISL_15956334, EPI_ISL_15958000, EPI_ISL_15958880, EPI_ISL_15980444, EPI_ISL_15983623, EPI_ISL_15991137, EPI_ISL_16004201, EPI_ISL_16004221, EPI_ISL_16004267, EPI_ISL_16008150, EPI_ISL_16008489, EPI_ISL_16008877, EPI_ISL_16026658, EPI_ISL_16037058, EPI_ISL_16050127, EPI_ISL_16050317, EPI_ISL_16054786, EPI_ISL_16059102, EPI_ISL_16059617, EPI_ISL_16070082, EPI_ISL_16072907, EPI_ISL_16077673, EPI_ISL_16092096, EPI_ISL_16121748, EPI_ISL_16121818, EPI_ISL_16133519, EPI_ISL_16138501, EPI_ISL_16151550, EPI_ISL_16172642, EPI_ISL_16193751, EPI_ISL_16197780, EPI_ISL_16207275, EPI_ISL_16208574, EPI_ISL_16210148, EPI_ISL_16219790, EPI_ISL_16226994, EPI_ISL_16230801, EPI_ISL_16243417, EPI_ISL_16246433, EPI_ISL_16246605, EPI_ISL_16267849, EPI_ISL_16285032, EPI_ISL_16285249, EPI_ISL_16287235, EPI_ISL_16289354, EPI_ISL_16290866, EPI_ISL_16297102, EPI_ISL_16315710, EPI_ISL_16332693, EPI_ISL_16341231, EPI_ISL_16342178, EPI_ISL_16342762, EPI_ISL_16360495, EPI_ISL_16364779, EPI_ISL_16386434, EPI_ISL_16394918, EPI_ISL_16419696, EPI_ISL_16448121, EPI_ISL_16449474, EPI_ISL_16468471, EPI_ISL_16495627, EPI_ISL_16495643, EPI_ISL_16523552, EPI_ISL_16535383, EPI_ISL_16542512, EPI_ISL_16557447, EPI_ISL_16579916, EPI_ISL_16581425, EPI_ISL_16581431, EPI_ISL_16581432, EPI_ISL_16590525, EPI_ISL_16593038, EPI_ISL_16607627, EPI_ISL_16611238, EPI_ISL_16611841, EPI_ISL_16856565, EPI_ISL_16887870, EPI_ISL_17076266, EPI_ISL_17292778, EPI_ISL_17740508, EPI_ISL_17776606, EPI_ISL_17806780, EPI_ISL_17818059, EPI_ISL_17830192, EPI_ISL_18215772, EPI_ISL_18300587, EPI_ISL_18352397, EPI_ISL_18435050, EPI_ISL_18520761, EPI_ISL_18678771, EPI_ISL_18687078, EPI_ISL_18687079, EPI_ISL_18687224, EPI_ISL_18687225, EPI_ISL_18711978, EPI_ISL_18712035, EPI_ISL_18733406, EPI_ISL_18931705, EPI_ISL_18932507, EPI_ISL_19085355, EPI_ISL_19169477, EPI_ISL_19205639, EPI_ISL_19300597, EPI_ISL_19305340, EPI_ISL_19308874, EPI_ISL_19308875, EPI_ISL_19308876, EPI_ISL_19308877, EPI_ISL_19323028, EPI_ISL_19328684, EPI_ISL_19333890, EPI_ISL_19337826, EPI_ISL_19340412, EPI_ISL_19340513, EPI_ISL_19340884, EPI_ISL_19344734, EPI_ISL_19363557, EPI_ISL_19369244, EPI_ISL_19374121, EPI_ISL_19374151, EPI_ISL_19374152, EPI_ISL_19374153, EPI_ISL_19374154, EPI_ISL_19377637, EPI_ISL_19377761, EPI_ISL_19377778, EPI_ISL_19383580, EPI_ISL_19383680, EPI_ISL_19392805, EPI_ISL_19405488, EPI_ISL_19427278, EPI_ISL_19434372, EPI_ISL_19444130, EPI_ISL_19472214, EPI_ISL_19473889, EPI_ISL_19506224, EPI_ISL_19515856, EPI_ISL_19517266, EPI_ISL_19522346, EPI_ISL_19526058, EPI_ISL_19526073, EPI_ISL_19526074, EPI_ISL_19527099, EPI_ISL_19531769, EPI_ISL_19531772, EPI_ISL_19531776, EPI_ISL_19534803, EPI_ISL_19548086, EPI_ISL_19555103, EPI_ISL_19572558, EPI_ISL_19585472, EPI_ISL_19588464, EPI_ISL_19591990, EPI_ISL_19620039, EPI_ISL_19621085, EPI_ISL_19697717, EPI_ISL_19707682, EPI_ISL_19750637, EPI_ISL_19778907, EPI_ISL_19783495, EPI_ISL_19783524, EPI_ISL_19788441, EPI_ISL_19799392, EPI_ISL_19799712, EPI_ISL_19799810, EPI_ISL_19800089, EPI_ISL_19800094, EPI_ISL_19800098, EPI_ISL_19816731, EPI_ISL_19852829, EPI_ISL_19856862, EPI_ISL_19878325, EPI_ISL_19913944, EPI_ISL_19914209, EPI_ISL_20048939, EPI_ISL_20056146, EPI_ISL_20067911, EPI_ISL_20076112, EPI_ISL_20084491, EPI_ISL_20133315, EPI_ISL_20166121, EPI_ISL_20213240, EPI_ISL_20213410, EPI_ISL_20251366, EPI_ISL_20292415")
MOV = union(mov_old_doc, mov_new_doc)
Angola_A = stringlist_to_set("EPI_ISL_1347940, EPI_ISL_1347941, EPI_ISL_1347942, EPI_ISL_2419105, EPI_ISL_2746031")
AT1 = stringlist_to_set("EPI_ISL_2385089, EPI_ISL_18014433, EPI_ISL_18014355, EPI_ISL_3454837, EPI_ISL_2245684, EPI_ISL_2385137, EPI_ISL_18016392, EPI_ISL_18027763, EPI_ISL_1652622, EPI_ISL_18027770, EPI_ISL_2245728, EPI_ISL_2258756, EPI_ISL_2450459, EPI_ISL_2245675, EPI_ISL_2698607, EPI_ISL_3026520, EPI_ISL_2385127, EPI_ISL_11246080, EPI_ISL_1652580, EPI_ISL_2245775, EPI_ISL_2385430, EPI_ISL_2385240, EPI_ISL_18014354, EPI_ISL_2258757, EPI_ISL_2385264, EPI_ISL_2245737, EPI_ISL_2385132, EPI_ISL_2038887, EPI_ISL_2245644, EPI_ISL_11267839, EPI_ISL_1491610, EPI_ISL_1919575, EPI_ISL_1652607, EPI_ISL_3101247, EPI_ISL_2816166, EPI_ISL_3101347, EPI_ISL_3101296, EPI_ISL_2385268, EPI_ISL_1491611, EPI_ISL_2245680, EPI_ISL_1652608, EPI_ISL_18014306, EPI_ISL_3076755, EPI_ISL_2694579, EPI_ISL_3040116, EPI_ISL_2385306, EPI_ISL_2385222, EPI_ISL_3021542, EPI_ISL_1652639, EPI_ISL_2385317, EPI_ISL_2609255, EPI_ISL_2609191, EPI_ISL_2609371, EPI_ISL_18018216, EPI_ISL_2258962, EPI_ISL_2385285, EPI_ISL_18015138, EPI_ISL_2385294, EPI_ISL_11267844, EPI_ISL_2523392, EPI_ISL_2385305, EPI_ISL_12389148, EPI_ISL_11267840, EPI_ISL_2385426, EPI_ISL_2385141, EPI_ISL_1652545, EPI_ISL_3101302, EPI_ISL_2363834, EPI_ISL_2616208, EPI_ISL_2385265, EPI_ISL_2642249, EPI_ISL_18014342, EPI_ISL_2363825, EPI_ISL_2523425, EPI_ISL_18018316, EPI_ISL_3040109, EPI_ISL_2385427, EPI_ISL_2245760, EPI_ISL_1548041, EPI_ISL_4093675, EPI_ISL_2609203, EPI_ISL_2385115, EPI_ISL_2385223, EPI_ISL_2385086, EPI_ISL_2967111, EPI_ISL_18014405, EPI_ISL_2757968, EPI_ISL_18015053, EPI_ISL_11267843, EPI_ISL_18017208, EPI_ISL_11225838, EPI_ISL_2657493, EPI_ISL_2385327, EPI_ISL_1652585, EPI_ISL_2032642, EPI_ISL_2523721, EPI_ISL_2245670, EPI_ISL_18029419, EPI_ISL_1823179, EPI_ISL_7850437, EPI_ISL_2385286, EPI_ISL_3076760, EPI_ISL_2485133, EPI_ISL_18018233, EPI_ISL_18027824, EPI_ISL_2385376, EPI_ISL_2038912, EPI_ISL_2385269, EPI_ISL_3040113, EPI_ISL_2385325, EPI_ISL_2887541, EPI_ISL_3454841, EPI_ISL_2245716, EPI_ISL_2523667, EPI_ISL_2385234, EPI_ISL_2689418, EPI_ISL_2663108, EPI_ISL_2609134, EPI_ISL_2967104, EPI_ISL_5049878, EPI_ISL_2245717, EPI_ISL_2644858, EPI_ISL_2523696, EPI_ISL_2385238, EPI_ISL_18014312, EPI_ISL_2887421, EPI_ISL_2245679, EPI_ISL_2385171, EPI_ISL_13287205, EPI_ISL_3798354, EPI_ISL_3101286, EPI_ISL_2523759, EPI_ISL_3101315, EPI_ISL_1919459, EPI_ISL_2385201, EPI_ISL_8112527, EPI_ISL_1259282, EPI_ISL_2245761, EPI_ISL_2385198, EPI_ISL_2385307, EPI_ISL_1263541, EPI_ISL_3135785, EPI_ISL_5415169, EPI_ISL_18014259, EPI_ISL_2038886, EPI_ISL_2263172, EPI_ISL_2385216, EPI_ISL_2245665, EPI_ISL_2245649, EPI_ISL_3040110, EPI_ISL_2245663, EPI_ISL_7850438, EPI_ISL_11246081, EPI_ISL_1259283, EPI_ISL_11267841, EPI_ISL_2608656, EPI_ISL_2694559, EPI_ISL_2385421, EPI_ISL_18014137, EPI_ISL_2450562, EPI_ISL_2038871, EPI_ISL_2609088, EPI_ISL_2385082, EPI_ISL_11267842, EPI_ISL_3076749, EPI_ISL_6686962, EPI_ISL_2608657, EPI_ISL_1652635, EPI_ISL_12389150, EPI_ISL_2385109, EPI_ISL_3101381, EPI_ISL_2385229, EPI_ISL_1664699, EPI_ISL_2245763, EPI_ISL_2385110, EPI_ISL_11267838, EPI_ISL_18015144, EPI_ISL_2450534, EPI_ISL_1664729, EPI_ISL_2450556, EPI_ISL_2523448, EPI_ISL_2626347, EPI_ISL_2245677, EPI_ISL_1548040, EPI_ISL_2363838, EPI_ISL_2523400, EPI_ISL_2698617, EPI_ISL_3101291, EPI_ISL_2609233, EPI_ISL_1823178, EPI_ISL_2887542, EPI_ISL_18014235, EPI_ISL_18014407, EPI_ISL_2457502, EPI_ISL_2816398, EPI_ISL_2385263, EPI_ISL_2608651, EPI_ISL_2385431, EPI_ISL_2523611, EPI_ISL_2450517, EPI_ISL_1652557")
AV1 = stringlist_to_set("EPI_ISL_4566929, EPI_ISL_2125482, EPI_ISL_2239779, EPI_ISL_2004704, EPI_ISL_2240090, EPI_ISL_2240151, EPI_ISL_2122546, EPI_ISL_2198382, EPI_ISL_1987067, EPI_ISL_2021628, EPI_ISL_2624764, EPI_ISL_2434445, EPI_ISL_2127047, EPI_ISL_2236823, EPI_ISL_2520581, EPI_ISL_2236967, EPI_ISL_2705045, EPI_ISL_2120895, EPI_ISL_1857951, EPI_ISL_2237157, EPI_ISL_2239580, EPI_ISL_2171756, EPI_ISL_2434675, EPI_ISL_2237958, EPI_ISL_2239457, EPI_ISL_2518284, EPI_ISL_2240292, EPI_ISL_2570109, EPI_ISL_2457326, EPI_ISL_2706055, EPI_ISL_2625296, EPI_ISL_2518253, EPI_ISL_2518281, EPI_ISL_2517875, EPI_ISL_2021658, EPI_ISL_2625337, EPI_ISL_2625442, EPI_ISL_2318105, EPI_ISL_2518496, EPI_ISL_2721970, EPI_ISL_2236548, EPI_ISL_1857666, EPI_ISL_2253970, EPI_ISL_2092569, EPI_ISL_2437210, EPI_ISL_2514457, EPI_ISL_2517895, EPI_ISL_1857609, EPI_ISL_2434746, EPI_ISL_2355937, EPI_ISL_2127052, EPI_ISL_2240008, EPI_ISL_2434695, EPI_ISL_2240274, EPI_ISL_2240118, EPI_ISL_2518887, EPI_ISL_2004581, EPI_ISL_2239978, EPI_ISL_2518712, EPI_ISL_2717065, EPI_ISL_2138347, EPI_ISL_2126474, EPI_ISL_2517941, EPI_ISL_2347631, EPI_ISL_2356078, EPI_ISL_2004900, EPI_ISL_2356178, EPI_ISL_2171691, EPI_ISL_2239677, EPI_ISL_2707108, EPI_ISL_2356235, EPI_ISL_2240054, EPI_ISL_2240678, EPI_ISL_2517872, EPI_ISL_2518852, EPI_ISL_2625597, EPI_ISL_2005439, EPI_ISL_3291235, EPI_ISL_2434355, EPI_ISL_2238485, EPI_ISL_2239850, EPI_ISL_1912100, EPI_ISL_2517910, EPI_ISL_2355982, EPI_ISL_2092318, EPI_ISL_2664976, EPI_ISL_2518249, EPI_ISL_2122397, EPI_ISL_2394252, EPI_ISL_2434509, EPI_ISL_2742613, EPI_ISL_2240131, EPI_ISL_2355981, EPI_ISL_2434451, EPI_ISL_2625398, EPI_ISL_2394056, EPI_ISL_2355966, EPI_ISL_2120972, EPI_ISL_2353408, EPI_ISL_2199581, EPI_ISL_2240019, EPI_ISL_2743887, EPI_ISL_2623351, EPI_ISL_2487128, EPI_ISL_2120960, EPI_ISL_2138441, EPI_ISL_2240680, EPI_ISL_1980365, EPI_ISL_2128461, EPI_ISL_2240079, EPI_ISL_2119669, EPI_ISL_2518053, EPI_ISL_1698537, EPI_ISL_2434465, EPI_ISL_2236937, EPI_ISL_2239489, EPI_ISL_2348421, EPI_ISL_2518704, EPI_ISL_2355974, EPI_ISL_2236504, EPI_ISL_2356883, EPI_ISL_2434473, EPI_ISL_2518367, EPI_ISL_2004376, EPI_ISL_2518126, EPI_ISL_2518270, EPI_ISL_1857753, EPI_ISL_2199394, EPI_ISL_2239848, EPI_ISL_2240116, EPI_ISL_2366158, EPI_ISL_2022294, EPI_ISL_2120273, EPI_ISL_2723162, EPI_ISL_2138170, EPI_ISL_2356106, EPI_ISL_2239443, EPI_ISL_1980414, EPI_ISL_2240058, EPI_ISL_2240677, EPI_ISL_2121628, EPI_ISL_2356045, EPI_ISL_1980353, EPI_ISL_2392572, EPI_ISL_2706009, EPI_ISL_2437243, EPI_ISL_2022182, EPI_ISL_2138472, EPI_ISL_2199542, EPI_ISL_1921237, EPI_ISL_2239816")
B1_243_2 = stringlist_to_set("EPI_ISL_5059975, EPI_ISL_5778139, EPI_ISL_6253803, EPI_ISL_6437714, EPI_ISL_6437731, EPI_ISL_6810194, EPI_ISL_4370844, EPI_ISL_19101329, EPI_ISL_19101331, EPI_ISL_19101352, EPI_ISL_2341030, EPI_ISL_2341032, EPI_ISL_2341042, EPI_ISL_2598009, EPI_ISL_2779620, EPI_ISL_2790212, EPI_ISL_2861192, EPI_ISL_2966365, EPI_ISL_2966381, EPI_ISL_2966385, EPI_ISL_2966388, EPI_ISL_2966389, EPI_ISL_2966390, EPI_ISL_2966391, EPI_ISL_2966393, EPI_ISL_2966394, EPI_ISL_2966395, EPI_ISL_2966397, EPI_ISL_2966398, EPI_ISL_2970391, EPI_ISL_3010940, EPI_ISL_3010945, EPI_ISL_3010951, EPI_ISL_3062847, EPI_ISL_3062855, EPI_ISL_3062858, EPI_ISL_3133773, EPI_ISL_3133784, EPI_ISL_3162228, EPI_ISL_3162233, EPI_ISL_3162237, EPI_ISL_3162259, EPI_ISL_3268387, EPI_ISL_3268388, EPI_ISL_3371410, EPI_ISL_3371411, EPI_ISL_3371412, EPI_ISL_3371413, EPI_ISL_3509975, EPI_ISL_3557000, EPI_ISL_3557006, EPI_ISL_3568782, EPI_ISL_3568783, EPI_ISL_3743213, EPI_ISL_3743216, EPI_ISL_3753944, EPI_ISL_3769234, EPI_ISL_3827105, EPI_ISL_3848024, EPI_ISL_3857817, EPI_ISL_5655176, EPI_ISL_5217681, EPI_ISL_5925120, EPI_ISL_6327614, EPI_ISL_4299659, EPI_ISL_4370973, EPI_ISL_4959931, EPI_ISL_4706700, EPI_ISL_4706704, EPI_ISL_4571920, EPI_ISL_4571923, EPI_ISL_4508892, EPI_ISL_4508893, EPI_ISL_4508918, EPI_ISL_4359868, EPI_ISL_2245655, EPI_ISL_2385087, EPI_ISL_2385112, EPI_ISL_2385117, EPI_ISL_2385134, EPI_ISL_2450564, EPI_ISL_3015043, EPI_ISL_17105781, EPI_ISL_17105784, EPI_ISL_17105799, EPI_ISL_17105800, EPI_ISL_17105801")
B1_616 = stringlist_to_set("EPI_ISL_1259297, EPI_ISL_1262772, EPI_ISL_1292807, EPI_ISL_1292808, EPI_ISL_1443902, EPI_ISL_1526787, EPI_ISL_1526780, EPI_ISL_2178439, EPI_ISL_2178448, EPI_ISL_2178432, EPI_ISL_1655920, EPI_ISL_1739315, EPI_ISL_1655921, EPI_ISL_1118893, EPI_ISL_2178447, EPI_ISL_1443880, EPI_ISL_1526779, EPI_ISL_1111064, EPI_ISL_1409708, EPI_ISL_1118892, EPI_ISL_1381829, EPI_ISL_1259307, EPI_ISL_1489729, EPI_ISL_1526783, EPI_ISL_1739314, EPI_ISL_1739317, EPI_ISL_1526778, EPI_ISL_1318289, EPI_ISL_1526786, EPI_ISL_1655918, EPI_ISL_1540445, EPI_ISL_2281746, EPI_ISL_1526793, EPI_ISL_1526794, EPI_ISL_2178418, EPI_ISL_1381830, EPI_ISL_1739313, EPI_ISL_2178417, EPI_ISL_1110211, EPI_ISL_1655919, EPI_ISL_1239370, EPI_ISL_1525263")
B1_632 = stringlist_to_set("EPI_ISL_3326947, EPI_ISL_2942176, EPI_ISL_3610014, EPI_ISL_20118501, EPI_ISL_2343008, EPI_ISL_2402068, EPI_ISL_2479894, EPI_ISL_2479898, EPI_ISL_2479906, EPI_ISL_2490409, EPI_ISL_2490411, EPI_ISL_2616958, EPI_ISL_2658951, EPI_ISL_2658952, EPI_ISL_2658953, EPI_ISL_2671630, EPI_ISL_2681236, EPI_ISL_2681271, EPI_ISL_2681324, EPI_ISL_2778011, EPI_ISL_2801662, EPI_ISL_2801665, EPI_ISL_2801886, EPI_ISL_2835074, EPI_ISL_2835075, EPI_ISL_2835076, EPI_ISL_2835085, EPI_ISL_2835086, EPI_ISL_2835087, EPI_ISL_2835088, EPI_ISL_2920739, EPI_ISL_2926853, EPI_ISL_2928207, EPI_ISL_2942474, EPI_ISL_2942481, EPI_ISL_2942483, EPI_ISL_2942635, EPI_ISL_2942683, EPI_ISL_2942695, EPI_ISL_2970046, EPI_ISL_2978610, EPI_ISL_2987720, EPI_ISL_3048080, EPI_ISL_3067876, EPI_ISL_3067941, EPI_ISL_3084636, EPI_ISL_3084750, EPI_ISL_3085041, EPI_ISL_3085047, EPI_ISL_3110921, EPI_ISL_3155372, EPI_ISL_3155430, EPI_ISL_3155433, EPI_ISL_3160927, EPI_ISL_3219930, EPI_ISL_3239305, EPI_ISL_3239820, EPI_ISL_3265665, EPI_ISL_3265672, EPI_ISL_3265675, EPI_ISL_3277764, EPI_ISL_3277772, EPI_ISL_3327183, EPI_ISL_3328423, EPI_ISL_3330563, EPI_ISL_3330564, EPI_ISL_3347599, EPI_ISL_3347600, EPI_ISL_3347601, EPI_ISL_3347602, EPI_ISL_3347603, EPI_ISL_3347604, EPI_ISL_3347611, EPI_ISL_3385228, EPI_ISL_3388731, EPI_ISL_3393484, EPI_ISL_3408494, EPI_ISL_3408975, EPI_ISL_3428767, EPI_ISL_3431392, EPI_ISL_3491927, EPI_ISL_3492216, EPI_ISL_3494886, EPI_ISL_3495427, EPI_ISL_3496863, EPI_ISL_3501228, EPI_ISL_3503159, EPI_ISL_3503427, EPI_ISL_3503636, EPI_ISL_3520843, EPI_ISL_3556922, EPI_ISL_3556959, EPI_ISL_3556963, EPI_ISL_3610010, EPI_ISL_3610012, EPI_ISL_3713850, EPI_ISL_3713864, EPI_ISL_3715035, EPI_ISL_3721729, EPI_ISL_3721730, EPI_ISL_3721785, EPI_ISL_3721792, EPI_ISL_3721953, EPI_ISL_3757608, EPI_ISL_3769078, EPI_ISL_3805701, EPI_ISL_3805702, EPI_ISL_3814483, EPI_ISL_3817319, EPI_ISL_3817320, EPI_ISL_3848578, EPI_ISL_3848863, EPI_ISL_3851025, EPI_ISL_3855894, EPI_ISL_3905876, EPI_ISL_3912953, EPI_ISL_3913297, EPI_ISL_3945907, EPI_ISL_4006304, EPI_ISL_4006313, EPI_ISL_4035603, EPI_ISL_4071438, EPI_ISL_4158860, EPI_ISL_4158861, EPI_ISL_4232402, EPI_ISL_4232459, EPI_ISL_4276964, EPI_ISL_4298883, EPI_ISL_4298887, EPI_ISL_4412175, EPI_ISL_4455120, EPI_ISL_4466556, EPI_ISL_4507562, EPI_ISL_4550213, EPI_ISL_4743240, EPI_ISL_4743997, EPI_ISL_4917854, EPI_ISL_4917855, EPI_ISL_5014464, EPI_ISL_5193918, EPI_ISL_5337621, EPI_ISL_5338716, EPI_ISL_5340253, EPI_ISL_5340620, EPI_ISL_5342459, EPI_ISL_5344003, EPI_ISL_5344344, EPI_ISL_5344451, EPI_ISL_5517977, EPI_ISL_5781530, EPI_ISL_6475949, EPI_ISL_6483957, EPI_ISL_6486767, EPI_ISL_7812871, EPI_ISL_7813075, EPI_ISL_7813116, EPI_ISL_7813229, EPI_ISL_7813230, EPI_ISL_7813236, EPI_ISL_9540705, EPI_ISL_9545466, EPI_ISL_9546569, EPI_ISL_9548140, EPI_ISL_11090742, EPI_ISL_19570451, EPI_ISL_19587372")
B1_633 = stringlist_to_set("EPI_ISL_4132923, EPI_ISL_3006978, EPI_ISL_5978028, EPI_ISL_2556441, EPI_ISL_2253528, EPI_ISL_3006966, EPI_ISL_2880232, EPI_ISL_3277096, EPI_ISL_2461370, EPI_ISL_2556087, EPI_ISL_6024311, EPI_ISL_2881152, EPI_ISL_3267406, EPI_ISL_3267216, EPI_ISL_2461371, EPI_ISL_2880231, EPI_ISL_3277103")
B1_637_1 = stringlist_to_set("EPI_ISL_8062698, EPI_ISL_7769956, EPI_ISL_7025451, EPI_ISL_7350400, EPI_ISL_7580569, EPI_ISL_7718156, EPI_ISL_6210089, EPI_ISL_6603205, EPI_ISL_6754699, EPI_ISL_6792660, EPI_ISL_6813070, EPI_ISL_6826473, EPI_ISL_6888292, EPI_ISL_6892560, EPI_ISL_7025451, EPI_ISL_7126024, EPI_ISL_7126268, EPI_ISL_7126596, EPI_ISL_7128688, EPI_ISL_7145825, EPI_ISL_7158721, EPI_ISL_7251933, EPI_ISL_7350400, EPI_ISL_7447025, EPI_ISL_7447510, EPI_ISL_7447532, EPI_ISL_7498572, EPI_ISL_7567411, EPI_ISL_7580569, EPI_ISL_7602745, EPI_ISL_7602755, EPI_ISL_7649030, EPI_ISL_7698214, EPI_ISL_7698366, EPI_ISL_7698724, EPI_ISL_7698761, EPI_ISL_7699601, EPI_ISL_7699963, EPI_ISL_7718156, EPI_ISL_7742408, EPI_ISL_7751808, EPI_ISL_7751853, EPI_ISL_7851125, EPI_ISL_7851482, EPI_ISL_7851594, EPI_ISL_7851775, EPI_ISL_7851928, EPI_ISL_7852675, EPI_ISL_7907991-7907993, EPI_ISL_7908113, EPI_ISL_7929082, EPI_ISL_7946679, EPI_ISL_7948596, EPI_ISL_7962082, EPI_ISL_7981526, EPI_ISL_8011329, EPI_ISL_8017095, EPI_ISL_8017382, EPI_ISL_8017451, EPI_ISL_8026270, EPI_ISL_8060502, EPI_ISL_8062840, EPI_ISL_8093540, EPI_ISL_8093960, EPI_ISL_8146663, EPI_ISL_8146698-8146699, EPI_ISL_8186328, EPI_ISL_8207383, EPI_ISL_8211035, EPI_ISL_8304874, EPI_ISL_8318401, EPI_ISL_8374349, EPI_ISL_8749229, EPI_ISL_8937903, EPI_ISL_9394541, EPI_ISL_13787627, EPI_ISL_15328188, EPI_ISL_15328191, EPI_ISL_15328196")
B1_638 = stringlist_to_set("EPI_ISL_2693841, EPI_ISL_2693834, EPI_ISL_2955458, EPI_ISL_3717990, EPI_ISL_3451089, EPI_ISL_2693841, EPI_ISL_3718070, EPI_ISL_3451090, EPI_ISL_2955458, EPI_ISL_3717989, EPI_ISL_3717991, EPI_ISL_3451094, EPI_ISL_3451091, EPI_ISL_3451093, EPI_ISL_3451092")
C12 = stringlist_to_set("EPI_ISL_12934369, EPI_ISL_18141392, EPI_ISL_5510293, EPI_ISL_6966915, EPI_ISL_13830454, EPI_ISL_4877022, EPI_ISL_4572846, EPI_ISL_3451295, EPI_ISL_4505691, EPI_ISL_4029943, EPI_ISL_6201919, EPI_ISL_17118749, EPI_ISL_5416314, EPI_ISL_4301822, EPI_ISL_6422305, EPI_ISL_4029946, EPI_ISL_3411457, EPI_ISL_4649835, EPI_ISL_6966431, EPI_ISL_3237233, EPI_ISL_6010139, EPI_ISL_3451195, EPI_ISL_3132623, EPI_ISL_3746871, EPI_ISL_4301836, EPI_ISL_3149300, EPI_ISL_2770450, EPI_ISL_9439179, EPI_ISL_5250003, EPI_ISL_4030023, EPI_ISL_5918093, EPI_ISL_3451362, EPI_ISL_3482536, EPI_ISL_3730315, EPI_ISL_3717993, EPI_ISL_7310266, EPI_ISL_3261918, EPI_ISL_3236953, EPI_ISL_6327852, EPI_ISL_3643862, EPI_ISL_5196557, EPI_ISL_6202863, EPI_ISL_2988404, EPI_ISL_17118744, EPI_ISL_4498625, EPI_ISL_4575101, EPI_ISL_3643966, EPI_ISL_6422315, EPI_ISL_6966423, EPI_ISL_11435082, EPI_ISL_6704861, EPI_ISL_5416342, EPI_ISL_5196625, EPI_ISL_4301774, EPI_ISL_2988405, EPI_ISL_5510296, EPI_ISL_3451144, EPI_ISL_3799102, EPI_ISL_3729072, EPI_ISL_4651575, EPI_ISL_4121731, EPI_ISL_4816930, EPI_ISL_5501810, EPI_ISL_3838520, EPI_ISL_7015167, EPI_ISL_3838614, EPI_ISL_4030024, EPI_ISL_5196561, EPI_ISL_3411459, EPI_ISL_2803815, EPI_ISL_3451214, EPI_ISL_3746811, EPI_ISL_3342732, EPI_ISL_10716937, EPI_ISL_5249928, EPI_ISL_4029912, EPI_ISL_4301764, EPI_ISL_4651914, EPI_ISL_9201638, EPI_ISL_3281602, EPI_ISL_5416309, EPI_ISL_7971616, EPI_ISL_5918098, EPI_ISL_8801146, EPI_ISL_6327942, EPI_ISL_5510294, EPI_ISL_6308315, EPI_ISL_16078910, EPI_ISL_6900134, EPI_ISL_6585209, EPI_ISL_5603133, EPI_ISL_6966611, EPI_ISL_16073449, EPI_ISL_5510477, EPI_ISL_5416370, EPI_ISL_7971718, EPI_ISL_5918082, EPI_ISL_4572320, EPI_ISL_7751200, EPI_ISL_3859880, EPI_ISL_6693554, EPI_ISL_10878448, EPI_ISL_2989113, EPI_ISL_4029941, EPI_ISL_3746752, EPI_ISL_17118742, EPI_ISL_11376225, EPI_ISL_4029927, EPI_ISL_3236186, EPI_ISL_6203560, EPI_ISL_4572840, EPI_ISL_7456442, EPI_ISL_5917927, EPI_ISL_5918091, EPI_ISL_3859890, EPI_ISL_3838321, EPI_ISL_12954333, EPI_ISL_3827640, EPI_ISL_3643965, EPI_ISL_3451544, EPI_ISL_4891676, EPI_ISL_7337403, EPI_ISL_4891529, EPI_ISL_4253663, EPI_ISL_5250000, EPI_ISL_8182904, EPI_ISL_3722231, EPI_ISL_5416341, EPI_ISL_8801148, EPI_ISL_4473177, EPI_ISL_7605669, EPI_ISL_3447713, EPI_ISL_3411467, EPI_ISL_5429037, EPI_ISL_12954319, EPI_ISL_4572340, EPI_ISL_8182902, EPI_ISL_6202177, EPI_ISL_8801145, EPI_ISL_7852873, EPI_ISL_3411463, EPI_ISL_16073445, EPI_ISL_3342730, EPI_ISL_3236951, EPI_ISL_6585226, EPI_ISL_6202861, EPI_ISL_4301828, EPI_ISL_3451391, EPI_ISL_6327902, EPI_ISL_11673422, EPI_ISL_6966874, EPI_ISL_4030022, EPI_ISL_4473159, EPI_ISL_6327957, EPI_ISL_4572321, EPI_ISL_4572845, EPI_ISL_5099204, EPI_ISL_11978044, EPI_ISL_7456446, EPI_ISL_3237063, EPI_ISL_3447717, EPI_ISL_5099171, EPI_ISL_4884442, EPI_ISL_3411681, EPI_ISL_3149306, EPI_ISL_9439165, EPI_ISL_7337381, EPI_ISL_4572347, EPI_ISL_3237084, EPI_ISL_4121681, EPI_ISL_3799162, EPI_ISL_5510297, EPI_ISL_5250001, EPI_ISL_7740878, EPI_ISL_7337385, EPI_ISL_7605582, EPI_ISL_4891546, EPI_ISL_4650213, EPI_ISL_2942287, EPI_ISL_3838556, EPI_ISL_4968419, EPI_ISL_5416369, EPI_ISL_5098944, EPI_ISL_3033753, EPI_ISL_4891545, EPI_ISL_4253610, EPI_ISL_3261970, EPI_ISL_6010138, EPI_ISL_5416297, EPI_ISL_3447773, EPI_ISL_5098812, EPI_ISL_3447714, EPI_ISL_3451378, EPI_ISL_17758039, EPI_ISL_3149313, EPI_ISL_5098807, EPI_ISL_17758080, EPI_ISL_6203099, EPI_ISL_4968413, EPI_ISL_6308256, EPI_ISL_2841668, EPI_ISL_4968415, EPI_ISL_8801151, EPI_ISL_3447758, EPI_ISL_4193905, EPI_ISL_3722270, EPI_ISL_5249995, EPI_ISL_3722264, EPI_ISL_5510295, EPI_ISL_5416523, EPI_ISL_2931281, EPI_ISL_7456406, EPI_ISL_7337388, EPI_ISL_7337401, EPI_ISL_6327890, EPI_ISL_4575118, EPI_ISL_6327851, EPI_ISL_13037920, EPI_ISL_3838375, EPI_ISL_5196617, EPI_ISL_5540191, EPI_ISL_3281600, EPI_ISL_3149299, EPI_ISL_5501902, EPI_ISL_14594777, EPI_ISL_3149307, EPI_ISL_3838569, EPI_ISL_5196616, EPI_ISL_5249952, EPI_ISL_12954336, EPI_ISL_8957611, EPI_ISL_3451358, EPI_ISL_7337390, EPI_ISL_5018646, EPI_ISL_3287712, EPI_ISL_4253656, EPI_ISL_4029935, EPI_ISL_3746874, EPI_ISL_3643842, EPI_ISL_6327948, EPI_ISL_6585204, EPI_ISL_12954342, EPI_ISL_4968411, EPI_ISL_6202316, EPI_ISL_4336900, EPI_ISL_2984801, EPI_ISL_9439008, EPI_ISL_5603137, EPI_ISL_4253637, EPI_ISL_5196531, EPI_ISL_3717995, EPI_ISL_4030021, EPI_ISL_3860015, EPI_ISL_6436394, EPI_ISL_5429026, EPI_ISL_3237098, EPI_ISL_3697115, EPI_ISL_6308257, EPI_ISL_3132566, EPI_ISL_3799095, EPI_ISL_3342734, EPI_ISL_9230060, EPI_ISL_3342733, EPI_ISL_3451569, EPI_ISL_3237092, EPI_ISL_2718062, EPI_ISL_6327850, EPI_ISL_4876926, EPI_ISL_3717994, EPI_ISL_4029948, EPI_ISL_17758081, EPI_ISL_5918092, EPI_ISL_3149312, EPI_ISL_5917886, EPI_ISL_11435083, EPI_ISL_12954141, EPI_ISL_6201944, EPI_ISL_3746788, EPI_ISL_7881950, EPI_ISL_8801147, EPI_ISL_3219805, EPI_ISL_4650090, EPI_ISL_4029923, EPI_ISL_7310320, EPI_ISL_3663539, EPI_ISL_4891705, EPI_ISL_3074033, EPI_ISL_3799035, EPI_ISL_3838541, EPI_ISL_17758085, EPI_ISL_6585202, EPI_ISL_6782048, EPI_ISL_18215881, EPI_ISL_3482519, EPI_ISL_3451222, EPI_ISL_13963277, EPI_ISL_8616711, EPI_ISL_4891659, EPI_ISL_5249934, EPI_ISL_4572381, EPI_ISL_3411458, EPI_ISL_4301791, EPI_ISL_3451369, EPI_ISL_3453877, EPI_ISL_9439176, EPI_ISL_6436401, EPI_ISL_3799163, EPI_ISL_3643860, EPI_ISL_3506424, EPI_ISL_3128775, EPI_ISL_5416371, EPI_ISL_3746772, EPI_ISL_2828749, EPI_ISL_6203297, EPI_ISL_5477673, EPI_ISL_4003563, EPI_ISL_2827937, EPI_ISL_3237237, EPI_ISL_3746842, EPI_ISL_3838512, EPI_ISL_3132608, EPI_ISL_3838515, EPI_ISL_6203350, EPI_ISL_2726854, EPI_ISL_6327853, EPI_ISL_4474416, EPI_ISL_3860060, EPI_ISL_3859884, EPI_ISL_6966660, EPI_ISL_4003556, EPI_ISL_6693553, EPI_ISL_5098934, EPI_ISL_4651145, EPI_ISL_7971590, EPI_ISL_4108306, EPI_ISL_3643903, EPI_ISL_4968412, EPI_ISL_4651357, EPI_ISL_3451301, EPI_ISL_6966959, EPI_ISL_4301837, EPI_ISL_11516060, EPI_ISL_3219868, EPI_ISL_12954357, EPI_ISL_4968418, EPI_ISL_3149301, EPI_ISL_4029964, EPI_ISL_3411589, EPI_ISL_11435084, EPI_ISL_5540181, EPI_ISL_5917846, EPI_ISL_7456429, EPI_ISL_5250002, EPI_ISL_2988409, EPI_ISL_4650166, EPI_ISL_3838489, EPI_ISL_3722284, EPI_ISL_4473174, EPI_ISL_6900144, EPI_ISL_3838621, EPI_ISL_5099194, EPI_ISL_17118750, EPI_ISL_12954142, EPI_ISL_2695610, EPI_ISL_9439051, EPI_ISL_6906407, EPI_ISL_6201931, EPI_ISL_11435085, EPI_ISL_3717932, EPI_ISL_6203343, EPI_ISL_3101505, EPI_ISL_6201939, EPI_ISL_6327854, EPI_ISL_6913983, EPI_ISL_3281601, EPI_ISL_6966691, EPI_ISL_4575181, EPI_ISL_4030025, EPI_ISL_4474432, EPI_ISL_4336918, EPI_ISL_2726855, EPI_ISL_3799137")







    
manual = stringlist_to_set("EPI_ISL_5049921, EPI_ISL_15316450, EPI_ISL_5049921, EPI_ISL_16407945, EPI_ISL_2304079, EPI_ISL_20090417, EPI_ISL_19189685, EPI_ISL_19192500, EPI_ISL_17057500, EPI_ISL_8788657, EPI_ISL_804043, EPI_ISL_18141461, EPI_ISL_11565253, EPI_ISL_11070505, EPI_ISL_8035694, EPI_ISL_12171528, EPI_ISL_9261356, EPI_ISL_14239213, EPI_ISL_4945775, EPI_ISL_3464481, EPI_ISL_14689915, EPI_ISL_20125502, EPI_ISL_10306135, EPI_ISL_2222502, EPI_ISL_4186372, EPI_ISL_2221132, EPI_ISL_2190385, EPI_ISL_20125501, EPI_ISL_17047780, EPI_ISL_11355408, EPI_ISL_18936212, EPI_ISL_12914782, EPI_ISL_12911275, EPI_ISL_12914780, EPI_ISL_8812967, EPI_ISL_12934364, EPI_ISL_8806479, EPI_ISL_8812973, EPI_ISL_7551897, EPI_ISL_12934332, EPI_ISL_17213928, EPI_ISL_11355275, EPI_ISL_8804024, EPI_ISL_6262430, EPI_ISL_6938247, EPI_ISL_10306135, EPI_ISL_14680808, EPI_ISL_19243172, EPI_ISL_8812983, EPI_ISL_8813006, EPI_ISL_8813057, EPI_ISL_479747, EPI_ISL_8813018, EPI_ISL_8813069, EPI_ISL_15292382, EPI_ISL_13493579, EPI_ISL_15157267, EPI_ISL_12582783, EPI_ISL_15292690, EPI_ISL_9205859, EPI_ISL_14595398, EPI_ISL_14677326, EPI_ISL_2296053, EPI_ISL_2650472, EPI_ISL_7892890, EPI_ISL_16814977, EPI_ISL_1312352, EPI_ISL_4186372, EPI_ISL_2221132, EPI_ISL_13114890, EPI_ISL_2193347, EPI_ISL_13992956, EPI_ISL_10831828, EPI_ISL_10823290, EPI_ISL_11236803, EPI_ISL_10993946, EPI_ISL_9205057, EPI_ISL_9405597, EPI_ISL_10707897, EPI_ISL_10246702, EPI_ISL_5400333, EPI_ISL_5664187, EPI_ISL_7757661, EPI_ISL_7757684, EPI_ISL_8222762, EPI_ISL_8222967, EPI_ISL_8670567, EPI_ISL_9213667, EPI_ISL_9213692, EPI_ISL_9229258, EPI_ISL_9443826, EPI_ISL_9444069, EPI_ISL_9732707, EPI_ISL_10030009, EPI_ISL_10030176, EPI_ISL_10310444, EPI_ISL_10765993, EPI_ISL_10905742, EPI_ISL_11460188, EPI_ISL_12014754, EPI_ISL_12015959, EPI_ISL_12363744, EPI_ISL_12364499, EPI_ISL_12524891, EPI_ISL_12735789, EPI_ISL_12818860, EPI_ISL_12823932, EPI_ISL_13236709, EPI_ISL_13237591, EPI_ISL_13384126, EPI_ISL_13384129, EPI_ISL_13384137, EPI_ISL_13384148-13384149, EPI_ISL_13384153, EPI_ISL_13384156, EPI_ISL_13384160, EPI_ISL_13384167, EPI_ISL_13384175, EPI_ISL_13384180, EPI_ISL_13384183, EPI_ISL_13384192, EPI_ISL_13384211-13384212, EPI_ISL_13384216, EPI_ISL_13384220, EPI_ISL_13384230, EPI_ISL_13384250, EPI_ISL_13384252, EPI_ISL_13384272, EPI_ISL_13384285, EPI_ISL_13384292-13384293, EPI_ISL_13384305, EPI_ISL_13384320, EPI_ISL_13384323, EPI_ISL_13384325, EPI_ISL_13384328, EPI_ISL_13384336, EPI_ISL_13384342, EPI_ISL_13384349, EPI_ISL_13384366, EPI_ISL_13384371, EPI_ISL_13384374, EPI_ISL_13384395-13384396, EPI_ISL_13384401-13384402, EPI_ISL_13384407, EPI_ISL_13384413, EPI_ISL_13384416, EPI_ISL_13384420, EPI_ISL_13384425, EPI_ISL_10069072, EPI_ISL_4470082, EPI_ISL_15457520, EPI_ISL_17712917, EPI_ISL_12658797, EPI_ISL_17712420, EPI_ISL_17057556, EPI_ISL_13714340, EPI_ISL_13633130, EPI_ISL_12829598, EPI_ISL_17642296, EPI_ISL_17708677, EPI_ISL_7733214, EPI_ISL_8146261, EPI_ISL_11449568, EPI_ISL_15457520, EPI_ISL_18572269, EPI_ISL_5049871, EPI_ISL_19013131, EPI_ISL_17372563, EPI_ISL_8164976, EPI_ISL_17372562, EPI_ISL_17698885, EPI_ISL_14936053, EPI_ISL_11584217, EPI_ISL_3613544, EPI_ISL_18979607, EPI_ISL_8422512, EPI_ISL_15588441, EPI_ISL_16613984, EPI_ISL_15588441, EPI_ISL_12065504, EPI_ISL_12065507, EPI_ISL_12065513, EPI_ISL_12071130, EPI_ISL_5522696, EPI_ISL_19465491, EPI_ISL_5031397, EPI_ISL_16993458, EPI_ISL_20244086, EPI_ISL_1755027, EPI_ISL_15588324, EPI_ISL_8688192, EPI_ISL_10551553, EPI_ISL_20248007, EPI_ISL_2966185, EPI_ISL_10834859, EPI_ISL_17583367, EPI_ISL_18365480, EPI_ISL_18365364, EPI_ISL_16946489, EPI_ISL_2221881, EPI_ISL_3299465, EPI_ISL_3299471, EPI_ISL_5309835, EPI_ISL_3299192, EPI_ISL_18341474, EPI_ISL_18328446, EPI_ISL_951315, EPI_ISL_2654003, EPI_ISL_4112436, EPI_ISL_1960932, EPI_ISL_11345967, EPI_ISL_1401804, EPI_ISL_1415145, EPI_ISL_8812981, EPI_ISL_10943972, EPI_ISL_10944001, EPI_ISL_10950962, EPI_ISL_10951229, EPI_ISL_12914785, EPI_ISL_12914805, EPI_ISL_18418390, EPI_ISL_17110455, EPI_ISL_16355474, EPI_ISL_3033651, EPI_ISL_1905078, EPI_ISL_7733492, EPI_ISL_8422475, EPI_ISL_8422486, EPI_ISL_19837250, EPI_ISL_19837306, EPI_ISL_17323946, EPI_ISL_12792509, EPI_ISL_12792527, EPI_ISL_12792542, EPI_ISL_12792549, EPI_ISL_12792598, EPI_ISL_12792604, EPI_ISL_16699250, EPI_ISL_14192854, EPI_ISL_16314191, EPI_ISL_17812445, EPI_ISL_16546030, EPI_ISL_17989364, EPI_ISL_19704256, EPI_ISL_2382994, EPI_ISL_17505863, EPI_ISL_17989356, EPI_ISL_17505855, EPI_ISL_17989361, EPI_ISL_10013207, EPI_ISL_10013313, EPI_ISL_17697547, EPI_ISL_5297454, EPI_ISL_2332780, EPI_ISL_10925740, EPI_ISL_17664185, EPI_ISL_3242601, EPI_ISL_9288672, EPI_ISL_10925732, EPI_ISL_10758363, EPI_ISL_9835876, EPI_ISL_2382940, EPI_ISL_16680126, EPI_ISL_11469663, EPI_ISL_10882762, EPI_ISL_2304098, EPI_ISL_1936614, EPI_ISL_9835815, EPI_ISL_9835863, EPI_ISL_9223704, EPI_ISL_3236785, EPI_ISL_3775918, EPI_ISL_17047962, EPI_ISL_6684991, EPI_ISL_12792520, EPI_ISL_10013207, EPI_ISL_7547082, EPI_ISL_1475614, EPI_ISL_11355259, EPI_ISL_12174847, EPI_ISL_10560968, EPI_ISL_14570652, EPI_ISL_10830779, EPI_ISL_11933005, EPI_ISL_15588423, EPI_ISL_2230703, EPI_ISL_11254727, EPI_ISL_18098347, EPI_ISL_16258489, EPI_ISL_17530377, EPI_ISL_16394748, EPI_ISL_18936196, EPI_ISL_18936257, EPI_ISL_11887814, EPI_ISL_18936201, EPI_ISL_18255176, EPI_ISL_18831502, EPI_ISL_11214367, EPI_ISL_6631807, EPI_ISL_1253646, EPI_ISL_8766773, EPI_ISL_3020336, EPI_ISL_470837, EPI_ISL_12005443, EPI_ISL_9286565, EPI_ISL_11254727, EPI_ISL_7232349, EPI_ISL_2392166, EPI_ISL_11824693, EPI_ISL_3150227, EPI_ISL_3150233, EPI_ISL_3150234, EPI_ISL_3246603, EPI_ISL_3254309, EPI_ISL_14474356, EPI_ISL_754195, EPI_ISL_20125507, EPI_ISL_20125617, EPI_ISL_15091235, EPI_ISL_1133158, EPI_ISL_5384514, EPI_ISL_10729421, EPI_ISL_10835322, EPI_ISL_5881213, EPI_ISL_11514235, EPI_ISL_463893, EPI_ISL_1406836, EPI_ISL_6938332, EPI_ISL_7551888, EPI_ISL_11355432, EPI_ISL_12911269, EPI_ISL_1036813, EPI_ISL_1036822-1036823, EPI_ISL_1036933, EPI_ISL_1711176, EPI_ISL_1711203, EPI_ISL_2715298, EPI_ISL_2715306, EPI_ISL_2715310, EPI_ISL_2715355, EPI_ISL_2966759, EPI_ISL_3940164, EPI_ISL_11294820, EPI_ISL_8579199, EPI_ISL_19348679, EPI_ISL_17275324, EPI_ISL_1475561, EPI_ISL_1249228, EPI_ISL_8724990, EPI_ISL_1051710, EPI_ISL_1051768, EPI_ISL_20082490, EPI_ISL_20082488, EPI_ISL_15637858, EPI_ISL_5484593, EPI_ISL_17759731, EPI_ISL_6810514, EPI_ISL_10671881, EPI_ISL_12955779, EPI_ISL_18736013, EPI_ISL_17565160, EPI_ISL_18936264, EPI_ISL_18255100, EPI_ISL_16258490, EPI_ISL_18098341, EPI_ISL_12914784, EPI_ISL_11776067, EPI_ISL_4170073, EPI_ISL_12582706, EPI_ISL_12914795, EPI_ISL_17057936, EPI_ISL_8813083, EPI_ISL_8813033, EPI_ISL_11746640, EPI_ISL_16424022, EPI_ISL_15846162, EPI_ISL_18936264, EPI_ISL_18255100, EPI_ISL_18115504, EPI_ISL_16407946, EPI_ISL_12146072, EPI_ISL_14298842, EPI_ISL_10149346, EPI_ISL_10149333, EPI_ISL_17601585, EPI_ISL_961353, EPI_ISL_19804505, EPI_ISL_13895599, EPI_ISL_17236435, EPI_ISL_18056797, EPI_ISL_18071577, EPI_ISL_10889661, EPI_ISL_15756983, EPI_ISL_12319878, EPI_ISL_13988923, EPI_ISL_11023738, EPI_ISL_4056154, EPI_ISL_17236482, EPI_ISL_11022632, EPI_ISL_11881600, EPI_ISL_5050881, EPI_ISL_7509807, EPI_ISL_1079237, EPI_ISL_2445044, EPI_ISL_16761948, EPI_ISL_11776056, EPI_ISL_12836675, EPI_ISL_2628241, EPI_ISL_18840307, EPI_ISL_1696737, EPI_ISL_14753968, EPI_ISL_1051606, EPI_ISL_10443097, EPI_ISL_2690331, EPI_ISL_4104706, EPI_ISL_5802416, EPI_ISL_10830359, EPI_ISL_2628258, EPI_ISL_5433662, EPI_ISL_1840693, EPI_ISL_16687715, EPI_ISL_14699173, EPI_ISL_10826836, EPI_ISL_2695990, EPI_ISL_1077253, EPI_ISL_10826719, EPI_ISL_14954510, EPI_ISL_14871997, EPI_ISL_2227569, EPI_ISL_561368, EPI_ISL_13787687, EPI_ISL_18347556, EPI_ISL_9526798, EPI_ISL_1904297, EPI_ISL_2080874, EPI_ISL_2174728, EPI_ISL_2468944, EPI_ISL_2650481, EPI_ISL_2652051, EPI_ISL_3299465, EPI_ISL_3299521, EPI_ISL_16560204, EPI_ISL_17028423, EPI_ISL_3299528, EPI_ISL_3792382, EPI_ISL_18829937, EPI_ISL_12308695, EPI_ISL_11736389, EPI_ISL_13544269, EPI_ISL_19212892, EPI_ISL_14749461, EPI_ISL_18936191, EPI_ISL_12934315, EPI_ISL_14211821, EPI_ISL_14211671, EPI_ISL_17275299, EPI_ISL_7547071, EPI_ISL_11291877, EPI_ISL_1979030, EPI_ISL_8812993, EPI_ISL_3790381, EPI_ISL_14211783, EPI_ISL_6262420, EPI_ISL_999172, EPI_ISL_10951217, EPI_ISL_1097025, EPI_ISL_3794870, EPI_ISL_14211793, EPI_ISL_8193918, EPI_ISL_7547083, EPI_ISL_14749035, EPI_ISL_4198509, EPI_ISL_12658798, EPI_ISL_6262415, EPI_ISL_951358, EPI_ISL_12658867, EPI_ISL_3793003, EPI_ISL_7507374, EPI_ISL_18226327, EPI_ISL_3393273, EPI_ISL_7551912, EPI_ISL_10729410, EPI_ISL_7551929, EPI_ISL_17565330, EPI_ISL_5049870, EPI_ISL_3421721, EPI_ISL_6938327, EPI_ISL_12658806, EPI_ISL_14211649, EPI_ISL_3794866, EPI_ISL_14749439, EPI_ISL_12934326, EPI_ISL_14749463, EPI_ISL_3393126, EPI_ISL_19212649, EPI_ISL_413695, EPI_ISL_14211663, EPI_ISL_17427519, EPI_ISL_3421717, EPI_ISL_8813039, EPI_ISL_12934445, EPI_ISL_17372611, EPI_ISL_1167194, EPI_ISL_19212796, EPI_ISL_14749448, EPI_ISL_17275312, EPI_ISL_1167188, EPI_ISL_17236581, EPI_ISL_999097, EPI_ISL_8813042, EPI_ISL_9114647, EPI_ISL_4530134, EPI_ISL_14749465, EPI_ISL_14749025, EPI_ISL_5049922, EPI_ISL_3164296, EPI_ISL_17372618, EPI_ISL_8812990, EPI_ISL_11355423, EPI_ISL_3078866, EPI_ISL_17592069, EPI_ISL_14211728, EPI_ISL_4254661, EPI_ISL_14211661, EPI_ISL_5934822, EPI_ISL_17020828, EPI_ISL_19013431, EPI_ISL_14770231, EPI_ISL_12934291, EPI_ISL_8812965, EPI_ISL_17275321, EPI_ISL_4198510, EPI_ISL_19212916, EPI_ISL_953938, EPI_ISL_3793007, EPI_ISL_2903010, EPI_ISL_6262428, EPI_ISL_1109908, EPI_ISL_6357208, EPI_ISL_7509709, EPI_ISL_10944012, EPI_ISL_17425904, EPI_ISL_8803764, EPI_ISL_8812978, EPI_ISL_17565368, EPI_ISL_3421852, EPI_ISL_14211838, EPI_ISL_17020822, EPI_ISL_3792996, EPI_ISL_3793009, EPI_ISL_950844, EPI_ISL_14749011, EPI_ISL_1910755, EPI_ISL_15157245, EPI_ISL_13736428, EPI_ISL_12934366, EPI_ISL_12934246, EPI_ISL_14211711, EPI_ISL_6262412, EPI_ISL_14749469, EPI_ISL_951124, EPI_ISL_3386462, EPI_ISL_16258470, EPI_ISL_867117, EPI_ISL_17049000, EPI_ISL_950782, EPI_ISL_17275314, EPI_ISL_1165084, EPI_ISL_14211773, EPI_ISL_8803733, EPI_ISL_6938386, EPI_ISL_4198507, EPI_ISL_17372609, EPI_ISL_3794580, EPI_ISL_17592058, EPI_ISL_19192594, EPI_ISL_2399424, EPI_ISL_12694374, EPI_ISL_17592097, EPI_ISL_14211787, EPI_ISL_12934328, EPI_ISL_2707330, EPI_ISL_10943998, EPI_ISL_7493985, EPI_ISL_12658800, EPI_ISL_951455, EPI_ISL_15292716, EPI_ISL_1979036, EPI_ISL_6938399, EPI_ISL_10080731, EPI_ISL_6262427, EPI_ISL_3421731, EPI_ISL_19212725, EPI_ISL_3078917, EPI_ISL_3052905, EPI_ISL_12658803, EPI_ISL_14211788, EPI_ISL_12934421, EPI_ISL_1097027, EPI_ISL_17275330, EPI_ISL_19192556, EPI_ISL_17252820, EPI_ISL_14211785, EPI_ISL_7547072, EPI_ISL_20125615, EPI_ISL_3793004, EPI_ISL_3793312, EPI_ISL_1387214, EPI_ISL_17275327, EPI_ISL_3078951, EPI_ISL_3079049, EPI_ISL_14749430, EPI_ISL_3393200, EPI_ISL_14211665, EPI_ISL_4198508, EPI_ISL_16258498, EPI_ISL_17020817, EPI_ISL_840352, EPI_ISL_2404761, EPI_ISL_12934358, EPI_ISL_3792995, EPI_ISL_7547069, EPI_ISL_12658810, EPI_ISL_3568588, EPI_ISL_1663023, EPI_ISL_10944002, EPI_ISL_3393163, EPI_ISL_1178701, EPI_ISL_12934314, EPI_ISL_17744298, EPI_ISL_12954355, EPI_ISL_950902, EPI_ISL_6262431, EPI_ISL_14211790, EPI_ISL_3052909, EPI_ISL_1914289, EPI_ISL_12934360, EPI_ISL_906121, EPI_ISL_8812963, EPI_ISL_12934313, EPI_ISL_14770243, EPI_ISL_15334776, EPI_ISL_1475544, EPI_ISL_7508846, EPI_ISL_1165085, EPI_ISL_18831499, EPI_ISL_1178703, EPI_ISL_16258468, EPI_ISL_6262421, EPI_ISL_14211856, EPI_ISL_1415157, EPI_ISL_8803750, EPI_ISL_17502968, EPI_ISL_999421, EPI_ISL_12658812, EPI_ISL_3393236, EPI_ISL_15157236, EPI_ISL_20125505, EPI_ISL_3079030, EPI_ISL_12934162, EPI_ISL_954750, EPI_ISL_13857951, EPI_ISL_6125398, EPI_ISL_7509667, EPI_ISL_1167190, EPI_ISL_12658794, EPI_ISL_8193604, EPI_ISL_7792652, EPI_ISL_13374359, EPI_ISL_12248761, EPI_ISL_9090788, EPI_ISL_8085714, EPI_ISL_9487744, EPI_ISL_10746637, EPI_ISL_10996872, EPI_ISL_12363357, EPI_ISL_14018741, EPI_ISL_14260636, EPI_ISL_14793479, EPI_ISL_14936557, EPI_ISL_13469794, EPI_ISL_417446, EPI_ISL_406592, EPI_ISL_2775392, EPI_ISL_2775401, EPI_ISL_12754223, EPI_ISL_12754227, EPI_ISL_12754233, EPI_ISL_18052875, EPI_ISL_18936206, EPI_ISL_18936263, EPI_ISL_18936280, EPI_ISL_2312645, EPI_ISL_2349009, EPI_ISL_7637886, EPI_ISL_7644315, EPI_ISL_7644402, EPI_ISL_9130454-9130455, EPI_ISL_9189916-9189917, EPI_ISL_10079271, EPI_ISL_10079312, EPI_ISL_10786190, EPI_ISL_10808713, EPI_ISL_10811722, EPI_ISL_11383862, EPI_ISL_16258481, EPI_ISL_16370707, EPI_ISL_16741333, EPI_ISL_17275329, EPI_ISL_17275340, EPI_ISL_17275345, EPI_ISL_17275364, EPI_ISL_5049884, EPI_ISL_1167187, EPI_ISL_1909249, EPI_ISL_4106141, EPI_ISL_1908152, EPI_ISL_2965871, EPI_ISL_2775396, EPI_ISL_4646106, EPI_ISL_1039244, EPI_ISL_2775409, EPI_ISL_4085143, EPI_ISL_2775408, EPI_ISL_3717099, EPI_ISL_3806776, EPI_ISL_13351575, EPI_ISL_18743060, EPI_ISL_10858520, EPI_ISL_12145040, EPI_ISL_18436051, EPI_ISL_18228604, EPI_ISL_12584044, EPI_ISL_4050580, EPI_ISL_12740221, EPI_ISL_2885435, EPI_ISL_10858451, EPI_ISL_1585044, EPI_ISL_19507701, EPI_ISL_11541387, EPI_ISL_14718231, EPI_ISL_19507698, EPI_ISL_12999280, EPI_ISL_3664332, EPI_ISL_12955767, EPI_ISL_12955757, EPI_ISL_12999198, EPI_ISL_12955759, EPI_ISL_12320470, EPI_ISL_17057905, EPI_ISL_18743097, EPI_ISL_7591211, EPI_ISL_12146282, EPI_ISL_5510405, EPI_ISL_18228600, EPI_ISL_19130834, EPI_ISL_18228592, EPI_ISL_1257834, EPI_ISL_12152776, EPI_ISL_12955309, EPI_ISL_12955323, EPI_ISL_12955358, EPI_ISL_12955755, EPI_ISL_2185097, EPI_ISL_18228572, EPI_ISL_12955362, EPI_ISL_3047545, EPI_ISL_2861894, EPI_ISL_2375796, EPI_ISL_17675888, EPI_ISL_4367333, EPI_ISL_2762420, EPI_ISL_3547039, EPI_ISL_15348882, EPI_ISL_2462557, EPI_ISL_12999295, EPI_ISL_2185139, EPI_ISL_19721400, EPI_ISL_13131330, EPI_ISL_4635662, EPI_ISL_12955798, EPI_ISL_3305534, EPI_ISL_12320477, EPI_ISL_3032638, EPI_ISL_10523958, EPI_ISL_3410096, EPI_ISL_1684491, EPI_ISL_12999303, EPI_ISL_14495416, EPI_ISL_18912398, EPI_ISL_12223844, EPI_ISL_7711714, EPI_ISL_2987814, EPI_ISL_3315446, EPI_ISL_3402433, EPI_ISL_3402507, EPI_ISL_18821489, EPI_ISL_18821483, EPI_ISL_18821485, EPI_ISL_18821484, EPI_ISL_18909127, EPI_ISL_18677245, EPI_ISL_18743285, EPI_ISL_1703642, EPI_ISL_7337934, EPI_ISL_14148380, EPI_ISL_14148392, EPI_ISL_14148370, EPI_ISL_19213104, EPI_ISL_1585048, EPI_ISL_5049881, EPI_ISL_19014027, EPI_ISL_1096997, EPI_ISL_1109909, EPI_ISL_1305139, EPI_ISL_11415298, EPI_ISL_19022021, EPI_ISL_11359469, EPI_ISL_5237178, EPI_ISL_1811222, EPI_ISL_10657798, EPI_ISL_14934379, EPI_ISL_3164401, EPI_ISL_5069757, EPI_ISL_5056578, EPI_ISL_2634081, EPI_ISL_2761448, EPI_ISL_18268996, EPI_ISL_11776065, EPI_ISL_9305848, EPI_ISL_19906489, EPI_ISL_7810567, EPI_ISL_17048351, EPI_ISL_17048373, EPI_ISL_7459802, EPI_ISL_7459808, EPI_ISL_7459826, EPI_ISL_7459829, EPI_ISL_5324097, EPI_ISL_8165195, EPI_ISL_19143502, EPI_ISL_17017049, EPI_ISL_17017054, EPI_ISL_17017060, EPI_ISL_17612350, EPI_ISL_17821657, EPI_ISL_1127139, EPI_ISL_10905976, EPI_ISL_8151965, EPI_ISL_15213516, EPI_ISL_13905991, EPI_ISL_2989612, EPI_ISL_2990625, EPI_ISL_2990638, EPI_ISL_2990640, EPI_ISL_2990657, EPI_ISL_2990682, EPI_ISL_2990723, EPI_ISL_2991008, EPI_ISL_3655594, EPI_ISL_3656320, EPI_ISL_3657281, EPI_ISL_4366250, EPI_ISL_4562321, EPI_ISL_4562335, EPI_ISL_4562352, EPI_ISL_4562623, EPI_ISL_4562651, EPI_ISL_4921913, EPI_ISL_13092657, EPI_ISL_6313521, EPI_ISL_3003316, EPI_ISL_3178943, EPI_ISL_18394018, EPI_ISL_2804368, EPI_ISL_14217903, EPI_ISL_14218187, EPI_ISL_19614951, EPI_ISL_17522539, EPI_ISL_15316164, EPI_ISL_17245100, EPI_ISL_5339021, EPI_ISL_19536023, EPI_ISL_15015723, EPI_ISL_16397957, EPI_ISL_17549547, EPI_ISL_13462014, EPI_ISL_4473854, EPI_ISL_14578586, EPI_ISL_16397957, EPI_ISL_18138937, EPI_ISL_13912700, EPI_ISL_14361317, EPI_ISL_14432028, EPI_ISL_10380147, EPI_ISL_15154103, EPI_ISL_9215228, EPI_ISL_19573850, EPI_ISL_11760260, EPI_ISL_11017250, EPI_ISL_14551457, EPI_ISL_9940216, EPI_ISL_15038834, EPI_ISL_16125619, EPI_ISL_16528621, EPI_ISL_16528622, EPI_ISL_13368486, EPI_ISL_19838229, EPI_ISL_18274129, EPI_ISL_9261274, EPI_ISL_9261275, EPI_ISL_9261315, EPI_ISL_9261333, EPI_ISL_9261343, EPI_ISL_9261351, EPI_ISL_9261365, EPI_ISL_19032958, EPI_ISL_5388899, EPI_ISL_2557866, EPI_ISL_6262413, EPI_ISL_6262414, EPI_ISL_6262417, EPI_ISL_6262422, EPI_ISL_6262422, EPI_ISL_8812980, EPI_ISL_8812987, EPI_ISL_8812988, EPI_ISL_8812994, EPI_ISL_8812998, EPI_ISL_8813000, EPI_ISL_8813010, EPI_ISL_8813011, EPI_ISL_8813020, EPI_ISL_8813036, EPI_ISL_8813037, EPI_ISL_8813043, EPI_ISL_8813047, EPI_ISL_8813049, EPI_ISL_8813061, EPI_ISL_8813062, EPI_ISL_8813063, EPI_ISL_8813071, EPI_ISL_6938252, EPI_ISL_6938255, EPI_ISL_6938305, EPI_ISL_6938365, EPI_ISL_6938417, EPI_ISL_7493988, EPI_ISL_7547073, EPI_ISL_7547080, EPI_ISL_7547084, EPI_ISL_7551871, EPI_ISL_12658842, EPI_ISL_12658843, EPI_ISL_12658853, EPI_ISL_12934134, EPI_ISL_12934143, EPI_ISL_12934185, EPI_ISL_12934261, EPI_ISL_12934297, EPI_ISL_12934298, EPI_ISL_12934299, EPI_ISL_12934321, EPI_ISL_12934333, EPI_ISL_12934336, EPI_ISL_12934369, EPI_ISL_12934450, EPI_ISL_12934455, EPI_ISL_12934456, EPI_ISL_17562669, EPI_ISL_3588762, EPI_ISL_3743152, EPI_ISL_3588812, EPI_ISL_3756845, EPI_ISL_3743157, EPI_ISL_3588790, EPI_ISL_12583842, EPI_ISL_3543373, EPI_ISL_3588749, EPI_ISL_3588802, EPI_ISL_5511689, EPI_ISL_14701411, EPI_ISL_16120995, EPI_ISL_16553517, EPI_ISL_16814940, EPI_ISL_16814949, EPI_ISL_16815034, EPI_ISL_16599636, EPI_ISL_16271048, EPI_ISL_17275297, EPI_ISL_16699252, EPI_ISL_15473936, EPI_ISL_9629866, EPI_ISL_1109976, EPI_ISL_482775, EPI_ISL_12511485, EPI_ISL_7945646, EPI_ISL_7944977, EPI_ISL_7792628, EPI_ISL_5470943, EPI_ISL_467299, EPI_ISL_1517192, EPI_ISL_13384362, EPI_ISL_1034151, EPI_ISL_7947025, EPI_ISL_2754678, EPI_ISL_1625031, EPI_ISL_9629922, EPI_ISL_9077212, EPI_ISL_17597388, EPI_ISL_13544323, EPI_ISL_955140, EPI_ISL_12736844, EPI_ISL_2887404, EPI_ISL_5934793, EPI_ISL_17034838, EPI_ISL_12955760, EPI_ISL_7591215, EPI_ISL_7591217, EPI_ISL_7591219, EPI_ISL_7591222, EPI_ISL_1696687, EPI_ISL_1696692, EPI_ISL_1696714, EPI_ISL_1696715, EPI_ISL_1133157, EPI_ISL_1704072, EPI_ISL_17238817, EPI_ISL_3299585, EPI_ISL_2095083, EPI_ISL_2095085, EPI_ISL_2095099, EPI_ISL_2095111, EPI_ISL_2095114, EPI_ISL_2095125, EPI_ISL_19013684, EPI_ISL_18110550, EPI_ISL_17785884, EPI_ISL_17821633, EPI_ISL_17821640, EPI_ISL_17821665, EPI_ISL_1960965, EPI_ISL_1960981, EPI_ISL_1960987, EPI_ISL_1961000, EPI_ISL_8422490, EPI_ISL_12582703, EPI_ISL_18572143, EPI_ISL_2931767, EPI_ISL_12146277, EPI_ISL_12320529, EPI_ISL_12320662, EPI_ISL_10697706, EPI_ISL_10697708, EPI_ISL_10697709, EPI_ISL_10697711, EPI_ISL_10697714, EPI_ISL_10697716, EPI_ISL_10697718, EPI_ISL_10697720, EPI_ISL_10697721, EPI_ISL_10697722, EPI_ISL_10697723, EPI_ISL_10697724, EPI_ISL_18352009, EPI_ISL_18139468, EPI_ISL_15316453, EPI_ISL_10432651, EPI_ISL_3941723, EPI_ISL_1133149, EPI_ISL_12152767, EPI_ISL_12321743, EPI_ISL_12906443, EPI_ISL_17372560, EPI_ISL_19014042, EPI_ISL_17097728, EPI_ISL_14552727, EPI_ISL_14584747, EPI_ISL_14584752, EPI_ISL_14584768, EPI_ISL_14584783, EPI_ISL_961464, EPI_ISL_1704083, EPI_ISL_2106988, EPI_ISL_2547087, EPI_ISL_3631655, EPI_ISL_7873234, EPI_ISL_8445235, EPI_ISL_8445256, EPI_ISL_8975271, EPI_ISL_9093175, EPI_ISL_9112420, EPI_ISL_9305884, EPI_ISL_9348945, EPI_ISL_10121550, EPI_ISL_10716792, EPI_ISL_10716808, EPI_ISL_10716815, EPI_ISL_11336335, EPI_ISL_11799946, EPI_ISL_12906191, EPI_ISL_12906192, EPI_ISL_12906193, EPI_ISL_12906194, EPI_ISL_12906195, EPI_ISL_12906196, EPI_ISL_12906197, EPI_ISL_12906198, EPI_ISL_12906199, EPI_ISL_12906200, EPI_ISL_12906201, EPI_ISL_12906202, EPI_ISL_12906203, EPI_ISL_12906204, EPI_ISL_12906205, EPI_ISL_12906206, EPI_ISL_12906207, EPI_ISL_12906208, EPI_ISL_12906209, EPI_ISL_12906213, EPI_ISL_12906215, EPI_ISL_12906216, EPI_ISL_12906217, EPI_ISL_12906221, EPI_ISL_12906223, EPI_ISL_12906224, EPI_ISL_12906225, EPI_ISL_12906226, EPI_ISL_12906227, EPI_ISL_12906228, EPI_ISL_12906229, EPI_ISL_12906230, EPI_ISL_12906231, EPI_ISL_12906232, EPI_ISL_12906233, EPI_ISL_12906234, EPI_ISL_12906236, EPI_ISL_12906239, EPI_ISL_12906241, EPI_ISL_12906245, EPI_ISL_12906246, EPI_ISL_12906252, EPI_ISL_12906256, EPI_ISL_12906263, EPI_ISL_12906264, EPI_ISL_12906268, EPI_ISL_12906270, EPI_ISL_12906271, EPI_ISL_12906272, EPI_ISL_12906273, EPI_ISL_12906276, EPI_ISL_12906281, EPI_ISL_12906282, EPI_ISL_12906283, EPI_ISL_12906284, EPI_ISL_12906285, EPI_ISL_12906287, EPI_ISL_12906288, EPI_ISL_12906289, EPI_ISL_12906292, EPI_ISL_12906294, EPI_ISL_12906300, EPI_ISL_12906301, EPI_ISL_12906302, EPI_ISL_12906305, EPI_ISL_12906306, EPI_ISL_12906307, EPI_ISL_12906310, EPI_ISL_12906311, EPI_ISL_12906312, EPI_ISL_12906315, EPI_ISL_12906316, EPI_ISL_12906317, EPI_ISL_12906318, EPI_ISL_12906319, EPI_ISL_12906320, EPI_ISL_12906321, EPI_ISL_12906322, EPI_ISL_12906323, EPI_ISL_12906325, EPI_ISL_12906339, EPI_ISL_12906442, EPI_ISL_12906445, EPI_ISL_12906446, EPI_ISL_12906447, EPI_ISL_12906448, EPI_ISL_12906449, EPI_ISL_12906450, EPI_ISL_12906589, EPI_ISL_12906591, EPI_ISL_12906593, EPI_ISL_12906596, EPI_ISL_12906598, EPI_ISL_12906610, EPI_ISL_12906621, EPI_ISL_12906626, EPI_ISL_12906635, EPI_ISL_12906639, EPI_ISL_12934416, EPI_ISL_12955769, EPI_ISL_12999272, EPI_ISL_13304547, EPI_ISL_15038277, EPI_ISL_15251283, EPI_ISL_15348547, EPI_ISL_16407943, EPI_ISL_17275302, EPI_ISL_17301938, EPI_ISL_17301940, EPI_ISL_17301944, EPI_ISL_17302062, EPI_ISL_17302064, EPI_ISL_17302068, EPI_ISL_17302070, EPI_ISL_17302071, EPI_ISL_17466156, EPI_ISL_17466157, EPI_ISL_17466158, EPI_ISL_17466159, EPI_ISL_17466160, EPI_ISL_17466161, EPI_ISL_17466162, EPI_ISL_17466163, EPI_ISL_17466164, EPI_ISL_17466165, EPI_ISL_17466166, EPI_ISL_17466167, EPI_ISL_17466168, EPI_ISL_17466169, EPI_ISL_17466170, EPI_ISL_17466171, EPI_ISL_17466172, EPI_ISL_17466173, EPI_ISL_17466174, EPI_ISL_17466175, EPI_ISL_17466176, EPI_ISL_17466177, EPI_ISL_17466178, EPI_ISL_17466179, EPI_ISL_12583843, EPI_ISL_12146044, EPI_ISL_12146227, EPI_ISL_1957278, EPI_ISL_16061073, EPI_ISL_1173555, EPI_ISL_17682075, EPI_ISL_17989401, EPI_ISL_14561987, EPI_ISL_12194616, EPI_ISL_449800, EPI_ISL_8688196, EPI_ISL_2389452, EPI_ISL_2375846, EPI_ISL_1825065, EPI_ISL_2375511, EPI_ISL_11468163, EPI_ISL_10551554, EPI_ISL_10551562, EPI_ISL_13478523, EPI_ISL_10314946, EPI_ISL_5484223, EPI_ISL_17445536, EPI_ISL_18385487, EPI_ISL_1703777, EPI_ISL_1703826, EPI_ISL_16354122, EPI_ISL_15348741, EPI_ISL_15415391, EPI_ISL_9861349, EPI_ISL_1133148, EPI_ISL_1166096, EPI_ISL_1166102, EPI_ISL_1166104, EPI_ISL_1469239, EPI_ISL_10523160, EPI_ISL_10851659, EPI_ISL_10851661, EPI_ISL_10858436, EPI_ISL_10858527, EPI_ISL_10858541, EPI_ISL_11333353, EPI_ISL_11507483, EPI_ISL_11507489, EPI_ISL_11507524, EPI_ISL_11793451, EPI_ISL_12146193, EPI_ISL_12291347, EPI_ISL_12308587, EPI_ISL_12321607, EPI_ISL_12583926, EPI_ISL_12694427, EPI_ISL_12955756, EPI_ISL_12955775, EPI_ISL_12955828, EPI_ISL_13176027, EPI_ISL_13307898, EPI_ISL_13847367, EPI_ISL_14999066, EPI_ISL_15038548, EPI_ISL_1185776, EPI_ISL_4551799, EPI_ISL_1184558, EPI_ISL_1185776, EPI_ISL_18098351, EPI_ISL_8827492, EPI_ISL_9196492, EPI_ISL_8193909, EPI_ISL_8251487, EPI_ISL_5686636, EPI_ISL_3743165, EPI_ISL_1915533, EPI_ISL_11887232, EPI_ISL_11336333, EPI_ISL_12753427, EPI_ISL_12753104, EPI_ISL_3775926, EPI_ISL_6599771, EPI_ISL_12955519, EPI_ISL_14847621, EPI_ISL_9658674, EPI_ISL_9202505, EPI_ISL_9202539, EPI_ISL_9030329, EPI_ISL_9746968, EPI_ISL_9518504, EPI_ISL_9133317, EPI_ISL_9300549, EPI_ISL_9437783, EPI_ISL_9669302, EPI_ISL_9669737, EPI_ISL_9519903, EPI_ISL_9437783, EPI_ISL_9133317, EPI_ISL_9518504, EPI_ISL_9746968, EPI_ISL_9030329, EPI_ISL_9202505, EPI_ISL_9202539, EPI_ISL_9437783, EPI_ISL_9093609, EPI_ISL_9093690, EPI_ISL_7566355, EPI_ISL_6949207, EPI_ISL_6117425, EPI_ISL_6898986, EPI_ISL_6208342, EPI_ISL_6208343, EPI_ISL_6208344, EPI_ISL_6208345, EPI_ISL_6208346, EPI_ISL_6368268, EPI_ISL_6582943, EPI_ISL_5736276, EPI_ISL_5736289, EPI_ISL_5736297, EPI_ISL_5736302, EPI_ISL_5736307, EPI_ISL_5736314, EPI_ISL_5736811, EPI_ISL_5738176, EPI_ISL_5367954, EPI_ISL_5123044, EPI_ISL_5384223, EPI_ISL_5344440, EPI_ISL_5854760, EPI_ISL_5536462, EPI_ISL_5536463, EPI_ISL_5599480, EPI_ISL_5599487, EPI_ISL_5599496, EPI_ISL_5599500, EPI_ISL_5867029, EPI_ISL_5324096, EPI_ISL_5070742, EPI_ISL_5073213, EPI_ISL_5390697, EPI_ISL_2428882, EPI_ISL_2428956, EPI_ISL_7370200, EPI_ISL_7380510, EPI_ISL_7456430, EPI_ISL_7456435, EPI_ISL_7456462, EPI_ISL_7547036, EPI_ISL_7566326, EPI_ISL_7566334, EPI_ISL_7566340, EPI_ISL_7566342, EPI_ISL_7566350, EPI_ISL_7566352, EPI_ISL_7566354, EPI_ISL_7566356, EPI_ISL_7566358, EPI_ISL_7566359, EPI_ISL_7566361, EPI_ISL_7566362, EPI_ISL_7566367, EPI_ISL_7566369, EPI_ISL_7566372, EPI_ISL_7566374, EPI_ISL_7566375, EPI_ISL_7566381, EPI_ISL_7566383, EPI_ISL_7566386, EPI_ISL_7568581, EPI_ISL_7568583, EPI_ISL_7568585, EPI_ISL_7568586, EPI_ISL_7568588, EPI_ISL_7568590, EPI_ISL_7568595, EPI_ISL_7568596, EPI_ISL_7568598, EPI_ISL_7575090, EPI_ISL_7594185, EPI_ISL_7605583, EPI_ISL_7605585, EPI_ISL_7615994, EPI_ISL_7634556, EPI_ISL_7660560, EPI_ISL_7698118, EPI_ISL_7716422, EPI_ISL_7716481, EPI_ISL_7719334, EPI_ISL_7729239, EPI_ISL_7729406, EPI_ISL_7734039, EPI_ISL_7734070, EPI_ISL_7734081, EPI_ISL_7738202, EPI_ISL_7743919, EPI_ISL_7747278, EPI_ISL_7762218, EPI_ISL_7763437, EPI_ISL_7795567, EPI_ISL_7807933, EPI_ISL_7808511, EPI_ISL_7818414, EPI_ISL_7842429, EPI_ISL_7848702, EPI_ISL_7850777, EPI_ISL_7850785, EPI_ISL_7860076, EPI_ISL_7862534, EPI_ISL_7862924, EPI_ISL_7863030, EPI_ISL_7863256, EPI_ISL_7868872, EPI_ISL_7869228, EPI_ISL_7882980, EPI_ISL_7895735, EPI_ISL_7948596, EPI_ISL_7952860, EPI_ISL_7957890, EPI_ISL_7961502, EPI_ISL_7961514, EPI_ISL_7961912, EPI_ISL_7963486, EPI_ISL_7969832, EPI_ISL_7971033, EPI_ISL_7976101, EPI_ISL_7977800, EPI_ISL_7977825, EPI_ISL_7977909, EPI_ISL_7978204, EPI_ISL_7978212, EPI_ISL_7978362, EPI_ISL_7978364, EPI_ISL_7978604, EPI_ISL_7979198, EPI_ISL_7979224, EPI_ISL_7980072, EPI_ISL_7992055, EPI_ISL_7994652, EPI_ISL_7994984, EPI_ISL_7605585, EPI_ISL_7337601, EPI_ISL_7850777, EPI_ISL_7850785, EPI_ISL_7698118, EPI_ISL_7734039, EPI_ISL_7734070, EPI_ISL_7734081, EPI_ISL_7977800, EPI_ISL_7977825, EPI_ISL_7977909, EPI_ISL_7978204, EPI_ISL_7978212, EPI_ISL_7978362, EPI_ISL_7978364, EPI_ISL_7978604, EPI_ISL_7979198, EPI_ISL_7979224, EPI_ISL_7980072, EPI_ISL_7738202, EPI_ISL_7848702, EPI_ISL_7842429, EPI_ISL_7969832, EPI_ISL_7868872, EPI_ISL_7807933, EPI_ISL_7808511, EPI_ISL_7615994, EPI_ISL_7594185, EPI_ISL_7265940, EPI_ISL_7952860, EPI_ISL_7869228, EPI_ISL_7994984, EPI_ISL_7994652, EPI_ISL_7992055, EPI_ISL_7795567, EPI_ISL_7895735, EPI_ISL_7862534, EPI_ISL_7862924, EPI_ISL_7863030, EPI_ISL_7863256, EPI_ISL_7575090, EPI_ISL_7963486, EPI_ISL_7762218, EPI_ISL_7763437, EPI_ISL_7818414, EPI_ISL_7634556, EPI_ISL_7719334, EPI_ISL_7860076, EPI_ISL_7605583, EPI_ISL_7337522, EPI_ISL_7337591, EPI_ISL_7167723, EPI_ISL_7747278, EPI_ISL_7716422, EPI_ISL_7716481, EPI_ISL_7976101, EPI_ISL_7566355, EPI_ISL_14210694, EPI_ISL_15846254, EPI_ISL_15945485, EPI_ISL_9456848, EPI_ISL_9456983, EPI_ISL_9156804, EPI_ISL_15709506, EPI_ISL_15025376, EPI_ISL_15940837, EPI_ISL_15975710, EPI_ISL_15025373, EPI_ISL_15270610, EPI_ISL_15670087, EPI_ISL_15250585, EPI_ISL_15709506, EPI_ISL_15880471, EPI_ISL_15879843, EPI_ISL_15897084, EPI_ISL_15897053, EPI_ISL_15886020, EPI_ISL_15866417, EPI_ISL_15874062, EPI_ISL_15179344, EPI_ISL_15237955, EPI_ISL_15399812, EPI_ISL_15494652, EPI_ISL_15482029, EPI_ISL_15947183, EPI_ISL_15982525, EPI_ISL_15846260, EPI_ISL_15831957")




                



        
Russian_Kaluga_duplicate = Set(["EPI_ISL_18647516"])
interesting_wastewater = stringlist_to_set("EPI_ISL_18111774, EPI_ISL_12836539")

India = stringlist_to_set("EPI_ISL_18936212, EPI_ISL_17047867, EPI_ISL_9091138, EPI_ISL_15157217, EPI_ISL_14441047, EPI_ISL_14227227, EPI_ISL_10943846, EPI_ISL_14227415, EPI_ISL_14227383, EPI_ISL_14227399, EPI_ISL_14227382, EPI_ISL_14227381, EPI_ISL_13391738, EPI_ISL_14441094, EPI_ISL_10924576, EPI_ISL_11333412, EPI_ISL_17372596, EPI_ISL_17565160, EPI_ISL_10205296, EPI_ISL_18094389, EPI_ISL_14999073, EPI_ISL_17372568, EPI_ISL_9419009, EPI_ISL_10729443, EPI_ISL_10729438, EPI_ISL_10729442, EPI_ISL_10729421, EPI_ISL_10066198, EPI_ISL_14679032, EPI_ISL_12906191-12906640, EPI_ISL_17466156-17466179, EPI_ISL_10716749-10716816, EPI_ISL_10772350, EPI_ISL_13718890, EPI_ISL_13847287, EPI_ISL_8527556, EPI_ISL_9782216, EPI_ISL_10080769, EPI_ISL_14211650-14211651, EPI_ISL_14211653, EPI_ISL_14211655, EPI_ISL_14211668, EPI_ISL_14211686, EPI_ISL_14211693, EPI_ISL_14211701, EPI_ISL_14211740, EPI_ISL_14211775, EPI_ISL_14211779, EPI_ISL_14211813, EPI_ISL_14211816, EPI_ISL_14211820, EPI_ISL_14211828, EPI_ISL_8806428, EPI_ISL_15293844, EPI_ISL_14677091, EPI_ISL_14681074, EPI_ISL_14595732, EPI_ISL_10306128, EPI_ISL_14443132, EPI_ISL_8882813, EPI_ISL_11468761, EPI_ISL_11468762, EPI_ISL_479776, EPI_ISL_2547229, EPI_ISL_1703710, EPI_ISL_541731, EPI_ISL_577654, EPI_ISL_2966131, EPI_ISL_11106543, EPI_ISL_11887241, EPI_ISL_11887313, EPI_ISL_11887728, EPI_ISL_11887754, EPI_ISL_11887820, EPI_ISL_12291249, EPI_ISL_12291339, EPI_ISL_9782271, EPI_ISL_8049924, EPI_ISL_8049928, EPI_ISL_17372400, EPI_ISL_17372447, EPI_ISL_17372505, EPI_ISL_17372556, EPI_ISL_17372557, EPI_ISL_17372567, EPI_ISL_15294031, EPI_ISL_15363265, EPI_ISL_14910936, EPI_ISL_14495310, EPI_ISL_14495319, EPI_ISL_14495329, EPI_ISL_14495334, EPI_ISL_14148984, EPI_ISL_13158620, EPI_ISL_13158678, EPI_ISL_13158687, EPI_ISL_13158700, EPI_ISL_12494547, EPI_ISL_12494558, EPI_ISL_12494582, EPI_ISL_12494585, EPI_ISL_12494587, EPI_ISL_12494596, EPI_ISL_12494638, EPI_ISL_12494874, EPI_ISL_10149349, EPI_ISL_14975997, EPI_ISL_14975999, EPI_ISL_14976001, EPI_ISL_14976002, EPI_ISL_14976004, EPI_ISL_14976005, EPI_ISL_14976006, EPI_ISL_14976008, EPI_ISL_14976014, EPI_ISL_14976017, EPI_ISL_14976018, EPI_ISL_14976094, EPI_ISL_14976096, EPI_ISL_14976099, EPI_ISL_14976106, EPI_ISL_14976107, EPI_ISL_14976109, EPI_ISL_14976111, EPI_ISL_14976112, EPI_ISL_14976113, EPI_ISL_14976123, EPI_ISL_14976129, EPI_ISL_14976139, EPI_ISL_14976142, EPI_ISL_14976146, EPI_ISL_14976147, EPI_ISL_14976148, EPI_ISL_14976150, EPI_ISL_14976153, EPI_ISL_14976154, EPI_ISL_14986408, EPI_ISL_14986472, EPI_ISL_14986473, EPI_ISL_14986476, EPI_ISL_14986478, EPI_ISL_14986480, EPI_ISL_14986481, EPI_ISL_14986482, EPI_ISL_14986484, EPI_ISL_14986486, EPI_ISL_14986487, EPI_ISL_14986488, EPI_ISL_14986489, EPI_ISL_14986490, EPI_ISL_14986493, EPI_ISL_14986505, EPI_ISL_14986525, EPI_ISL_14986529, EPI_ISL_14986530, EPI_ISL_14986532, EPI_ISL_14986533, EPI_ISL_14986539, EPI_ISL_14986542, EPI_ISL_14986543, EPI_ISL_14986544, EPI_ISL_14986545, EPI_ISL_14986547, EPI_ISL_14986550, EPI_ISL_14986551, EPI_ISL_14986554, EPI_ISL_14986555, EPI_ISL_14986556, EPI_ISL_14986560, EPI_ISL_14986563, EPI_ISL_14986567, EPI_ISL_14986568, EPI_ISL_14986570, EPI_ISL_14986574, EPI_ISL_14986580, EPI_ISL_14986583, EPI_ISL_14986591, EPI_ISL_14986594, EPI_ISL_14986600, EPI_ISL_14986601, EPI_ISL_14986606, EPI_ISL_14986608, EPI_ISL_14986612, EPI_ISL_14986613, EPI_ISL_14986619, EPI_ISL_14986623, EPI_ISL_14986624, EPI_ISL_14986625, EPI_ISL_14986630, EPI_ISL_14986631, EPI_ISL_14986633, EPI_ISL_14992926, EPI_ISL_14992971, EPI_ISL_14993025, EPI_ISL_14993053, EPI_ISL_14993093, EPI_ISL_14993097, EPI_ISL_14993098, EPI_ISL_14993100, EPI_ISL_14993111, EPI_ISL_14993113, EPI_ISL_14993114, EPI_ISL_14993139, EPI_ISL_14993140, EPI_ISL_14993141, EPI_ISL_14993142, EPI_ISL_14993146, EPI_ISL_14993173, EPI_ISL_14993174, EPI_ISL_15172564, EPI_ISL_15172596, EPI_ISL_15172598, EPI_ISL_15172634, EPI_ISL_15172657, EPI_ISL_15172658, EPI_ISL_15172662, EPI_ISL_15172663, EPI_ISL_15172678, EPI_ISL_15172680, EPI_ISL_15172702, EPI_ISL_15172709, EPI_ISL_15172719, EPI_ISL_15172720, EPI_ISL_15172726, EPI_ISL_15172727, EPI_ISL_15172729, EPI_ISL_15172730, EPI_ISL_15172733, EPI_ISL_15172741, EPI_ISL_15172743, EPI_ISL_15172745, EPI_ISL_15190291, EPI_ISL_15262286, EPI_ISL_15293825, EPI_ISL_15293919, EPI_ISL_15293920, EPI_ISL_15293921, EPI_ISL_15293922, EPI_ISL_15293923, EPI_ISL_15293924, EPI_ISL_15293951, EPI_ISL_15293996, EPI_ISL_15294000, EPI_ISL_15294013, EPI_ISL_15294014, EPI_ISL_15294017, EPI_ISL_15294021, EPI_ISL_15294023, EPI_ISL_15294024, EPI_ISL_15294025, EPI_ISL_15294027, EPI_ISL_15294028, EPI_ISL_11336342, EPI_ISL_11336416, EPI_ISL_11359809, EPI_ISL_11359818, EPI_ISL_11373282, EPI_ISL_11373303, EPI_ISL_11373308, EPI_ISL_11373310, EPI_ISL_11373330, EPI_ISL_11373343, EPI_ISL_11373365, EPI_ISL_11373384, EPI_ISL_11373430, EPI_ISL_11507507, EPI_ISL_11736106, EPI_ISL_11736364, EPI_ISL_11736368, EPI_ISL_11736370, EPI_ISL_11736383, EPI_ISL_11736392, EPI_ISL_11736399, EPI_ISL_11736401, EPI_ISL_11737568, EPI_ISL_11737648, EPI_ISL_11737649, EPI_ISL_11737670, EPI_ISL_11737703, EPI_ISL_11737704, EPI_ISL_11737705, EPI_ISL_11737757, EPI_ISL_11738154, EPI_ISL_11743906, EPI_ISL_11743950, EPI_ISL_11743951, EPI_ISL_11743952, EPI_ISL_11743956, EPI_ISL_11743957, EPI_ISL_11743964, EPI_ISL_11743967, EPI_ISL_11743971, EPI_ISL_11743972, EPI_ISL_11743980, EPI_ISL_11743981, EPI_ISL_11743982, EPI_ISL_11743984, EPI_ISL_11743985, EPI_ISL_11743987, EPI_ISL_11743992, EPI_ISL_11744005, EPI_ISL_11744006, EPI_ISL_11744012, EPI_ISL_11744047, EPI_ISL_11744059, EPI_ISL_11746406, EPI_ISL_11746517, EPI_ISL_11746523, EPI_ISL_11746556, EPI_ISL_11746561, EPI_ISL_11746579, EPI_ISL_11746642, EPI_ISL_11746682, EPI_ISL_11746691, EPI_ISL_11767469, EPI_ISL_11767883, EPI_ISL_11777028, EPI_ISL_11777627, EPI_ISL_11777640, EPI_ISL_11777683, EPI_ISL_11777799, EPI_ISL_11777817, EPI_ISL_11789967, EPI_ISL_11790070, EPI_ISL_11790073, EPI_ISL_11790078, EPI_ISL_11790082, EPI_ISL_11790083, EPI_ISL_11790088, EPI_ISL_11790094, EPI_ISL_11790096, EPI_ISL_11790105, EPI_ISL_11790135, EPI_ISL_11790203, EPI_ISL_11790222, EPI_ISL_11790247, EPI_ISL_11790250, EPI_ISL_11790253, EPI_ISL_11790259, EPI_ISL_11790274, EPI_ISL_11790278, EPI_ISL_11790290, EPI_ISL_11790294, EPI_ISL_11790305, EPI_ISL_11790310, EPI_ISL_11802438, EPI_ISL_11802439, EPI_ISL_11802546, EPI_ISL_11802599, EPI_ISL_11802600, EPI_ISL_11802602, EPI_ISL_11802603, EPI_ISL_11802608, EPI_ISL_11802616, EPI_ISL_11802639, EPI_ISL_11803132, EPI_ISL_11803143, EPI_ISL_11803157, EPI_ISL_11803274, EPI_ISL_11803277, EPI_ISL_11803351, EPI_ISL_11803374, EPI_ISL_11803386, EPI_ISL_12906222, EPI_ISL_12906238, EPI_ISL_12906247, EPI_ISL_12906251, EPI_ISL_12906255, EPI_ISL_12906258, EPI_ISL_12906259, EPI_ISL_12906261, EPI_ISL_12906265, EPI_ISL_12906267, EPI_ISL_12906274, EPI_ISL_12906280, EPI_ISL_12906286, EPI_ISL_12906290, EPI_ISL_12906293, EPI_ISL_12906296, EPI_ISL_12906299, EPI_ISL_12906303, EPI_ISL_12906304, EPI_ISL_12906309, EPI_ISL_12906313, EPI_ISL_12906314, EPI_ISL_12906369, EPI_ISL_12906374, EPI_ISL_12906383, EPI_ISL_12906455, EPI_ISL_12906464, EPI_ISL_12906470, EPI_ISL_12906479, EPI_ISL_12906498, EPI_ISL_12906503, EPI_ISL_12906506, EPI_ISL_12906509, EPI_ISL_12906511, EPI_ISL_12906518, EPI_ISL_12906542, EPI_ISL_12906559, EPI_ISL_12906562, EPI_ISL_12906590, EPI_ISL_12906592, EPI_ISL_12906597, EPI_ISL_12906599, EPI_ISL_12906600, EPI_ISL_12906601, EPI_ISL_12906602, EPI_ISL_12906604, EPI_ISL_12906609, EPI_ISL_12906618, EPI_ISL_12906619, EPI_ISL_12906622, EPI_ISL_12906627, EPI_ISL_12906628, EPI_ISL_12906632, EPI_ISL_12906633, EPI_ISL_12906636, EPI_ISL_12906637, EPI_ISL_12906638, EPI_ISL_12906640, EPI_ISL_12906210, EPI_ISL_12906211, EPI_ISL_12906212, EPI_ISL_12906214, EPI_ISL_12906218, EPI_ISL_12906219, EPI_ISL_12906220, EPI_ISL_12906235, EPI_ISL_12906237, EPI_ISL_12906240, EPI_ISL_12906242, EPI_ISL_12906243, EPI_ISL_12906244, EPI_ISL_12906248, EPI_ISL_12906249, EPI_ISL_12906250, EPI_ISL_12906253, EPI_ISL_12906254, EPI_ISL_12906257, EPI_ISL_12906260, EPI_ISL_12906262, EPI_ISL_12906266, EPI_ISL_12906269, EPI_ISL_12906275, EPI_ISL_12906277, EPI_ISL_12906278, EPI_ISL_12906279, EPI_ISL_12906291, EPI_ISL_12906295, EPI_ISL_12906297, EPI_ISL_12906298, EPI_ISL_12906308, EPI_ISL_12906324, EPI_ISL_12906326, EPI_ISL_12906327, EPI_ISL_12906328, EPI_ISL_12906329, EPI_ISL_12906330, EPI_ISL_12906331, EPI_ISL_12906332, EPI_ISL_12906333, EPI_ISL_12906334, EPI_ISL_12906335, EPI_ISL_12906336, EPI_ISL_12906337, EPI_ISL_12906338, EPI_ISL_12906340, EPI_ISL_12906341, EPI_ISL_12906342, EPI_ISL_12906343, EPI_ISL_12906344, EPI_ISL_12906345, EPI_ISL_12906346, EPI_ISL_12906347, EPI_ISL_12906348, EPI_ISL_12906349, EPI_ISL_12906350, EPI_ISL_12906351, EPI_ISL_12906352, EPI_ISL_12906353, EPI_ISL_12906354, EPI_ISL_12906355, EPI_ISL_12906356, EPI_ISL_12906357, EPI_ISL_12906358, EPI_ISL_12906359, EPI_ISL_12906360, EPI_ISL_12906361, EPI_ISL_12906362, EPI_ISL_12906363, EPI_ISL_12906364, EPI_ISL_12906365, EPI_ISL_12906366, EPI_ISL_12906367, EPI_ISL_12906368, EPI_ISL_12906370, EPI_ISL_12906371, EPI_ISL_12906372, EPI_ISL_12906373, EPI_ISL_12906375, EPI_ISL_12906376, EPI_ISL_12906377, EPI_ISL_12906378, EPI_ISL_12906379, EPI_ISL_12906380, EPI_ISL_12906381, EPI_ISL_12906382, EPI_ISL_12906384, EPI_ISL_12906385, EPI_ISL_12906386, EPI_ISL_12906387, EPI_ISL_12906388, EPI_ISL_12906389, EPI_ISL_12906390, EPI_ISL_12906391, EPI_ISL_12906392, EPI_ISL_12906393, EPI_ISL_12906394, EPI_ISL_12906395, EPI_ISL_12906396, EPI_ISL_12906397, EPI_ISL_12906398, EPI_ISL_12906399, EPI_ISL_12906400, EPI_ISL_12906401, EPI_ISL_12906402, EPI_ISL_12906403, EPI_ISL_12906404, EPI_ISL_12906405, EPI_ISL_12906406, EPI_ISL_12906407, EPI_ISL_12906408, EPI_ISL_12906409, EPI_ISL_12906410, EPI_ISL_12906411, EPI_ISL_12906412, EPI_ISL_12906413, EPI_ISL_12906414, EPI_ISL_12906415, EPI_ISL_12906416, EPI_ISL_12906417, EPI_ISL_12906418, EPI_ISL_12906419, EPI_ISL_12906420, EPI_ISL_12906421, EPI_ISL_12906422, EPI_ISL_12906423, EPI_ISL_12906424, EPI_ISL_12906425, EPI_ISL_12906426, EPI_ISL_12906427, EPI_ISL_12906428, EPI_ISL_12906429, EPI_ISL_12906430, EPI_ISL_12906431, EPI_ISL_12906432, EPI_ISL_12906433, EPI_ISL_12906434, EPI_ISL_12906435, EPI_ISL_12906436, EPI_ISL_12906437, EPI_ISL_12906438, EPI_ISL_12906439, EPI_ISL_12906440, EPI_ISL_12906441, EPI_ISL_12906444, EPI_ISL_12906451, EPI_ISL_12906452, EPI_ISL_12906453, EPI_ISL_12906454, EPI_ISL_12906456, EPI_ISL_12906457, EPI_ISL_12906458, EPI_ISL_12906459, EPI_ISL_12906460, EPI_ISL_12906461, EPI_ISL_12906462, EPI_ISL_12906463, EPI_ISL_12906465, EPI_ISL_12906466, EPI_ISL_12906467, EPI_ISL_12906468, EPI_ISL_12906469, EPI_ISL_12906471, EPI_ISL_12906472, EPI_ISL_12906473, EPI_ISL_12906474, EPI_ISL_12906475, EPI_ISL_12906476, EPI_ISL_12906477, EPI_ISL_12906478, EPI_ISL_12906480, EPI_ISL_12906481, EPI_ISL_12906482, EPI_ISL_12906483, EPI_ISL_12906484, EPI_ISL_12906485, EPI_ISL_12906486, EPI_ISL_12906487, EPI_ISL_12906488, EPI_ISL_12906489, EPI_ISL_12906490, EPI_ISL_12906491, EPI_ISL_12906492, EPI_ISL_12906493, EPI_ISL_12906494, EPI_ISL_12906495, EPI_ISL_12906496, EPI_ISL_12906497, EPI_ISL_12906499, EPI_ISL_12906500, EPI_ISL_12906501, EPI_ISL_12906502, EPI_ISL_12906504, EPI_ISL_12906505, EPI_ISL_12906507, EPI_ISL_12906508, EPI_ISL_12906510, EPI_ISL_12906512, EPI_ISL_12906513, EPI_ISL_12906514, EPI_ISL_12906515, EPI_ISL_12906516, EPI_ISL_12906517, EPI_ISL_12906519, EPI_ISL_12906520, EPI_ISL_12906521, EPI_ISL_12906522, EPI_ISL_12906523, EPI_ISL_12906524, EPI_ISL_12906525, EPI_ISL_12906526, EPI_ISL_12906527, EPI_ISL_12906528, EPI_ISL_12906529, EPI_ISL_12906530, EPI_ISL_12906531, EPI_ISL_12906532, EPI_ISL_12906533, EPI_ISL_12906534, EPI_ISL_12906535, EPI_ISL_12906536, EPI_ISL_12906537, EPI_ISL_12906538, EPI_ISL_12906539, EPI_ISL_12906540, EPI_ISL_12906541, EPI_ISL_12906543, EPI_ISL_12906544, EPI_ISL_12906545, EPI_ISL_12906546, EPI_ISL_12906547, EPI_ISL_12906548, EPI_ISL_12906549, EPI_ISL_12906550, EPI_ISL_12906551, EPI_ISL_12906552, EPI_ISL_12906553, EPI_ISL_12906554, EPI_ISL_12906555, EPI_ISL_12906556, EPI_ISL_12906557, EPI_ISL_12906558, EPI_ISL_12906560, EPI_ISL_12906561, EPI_ISL_12906563, EPI_ISL_12906564, EPI_ISL_12906565, EPI_ISL_12906566, EPI_ISL_12906567, EPI_ISL_12906568, EPI_ISL_12906569, EPI_ISL_12906570, EPI_ISL_12906571, EPI_ISL_12906572, EPI_ISL_12906573, EPI_ISL_12906574, EPI_ISL_12906575, EPI_ISL_12906576, EPI_ISL_12906577, EPI_ISL_12906578, EPI_ISL_12906579, EPI_ISL_12906580, EPI_ISL_12906581, EPI_ISL_12906582, EPI_ISL_12906583, EPI_ISL_12906584, EPI_ISL_12906585, EPI_ISL_12906586, EPI_ISL_12906587, EPI_ISL_12906588, EPI_ISL_12906594, EPI_ISL_12906595, EPI_ISL_12906603, EPI_ISL_12906605, EPI_ISL_12906606, EPI_ISL_12906607, EPI_ISL_12906608, EPI_ISL_12906611, EPI_ISL_12906612, EPI_ISL_12906613, EPI_ISL_12906614, EPI_ISL_12906615, EPI_ISL_12906616, EPI_ISL_12906617, EPI_ISL_12906620, EPI_ISL_12906623, EPI_ISL_12906624, EPI_ISL_12906625, EPI_ISL_12906629, EPI_ISL_12906630, EPI_ISL_12906631, EPI_ISL_12906634")
Idaho = stringlist_to_set("EPI_ISL_3588790, EPI_ISL_3923181, EPI_ISL_6117526, EPI_ISL_9352281, EPI_ISL_632847-632889, EPI_ISL_1164782-1164793, EPI_ISL_1164795-1164815, EPI_ISL_1190729-1190737, EPI_ISL_1219911-1219923, EPI_ISL_1363168-1363171, EPI_ISL_1363175-1363194, EPI_ISL_1366701-1366713, EPI_ISL_1405711-1405728, EPI_ISL_1492620-1492636, EPI_ISL_1510260-1510277, EPI_ISL_1532066-1532088, EPI_ISL_1627285-1627319, EPI_ISL_1696491-1696496, EPI_ISL_1806183, EPI_ISL_1806186, EPI_ISL_1806188, EPI_ISL_1806191, EPI_ISL_1806194, EPI_ISL_1806197, EPI_ISL_1806200, EPI_ISL_1806203, EPI_ISL_1806206, EPI_ISL_1806209, EPI_ISL_1806212, EPI_ISL_1806215, EPI_ISL_1806218, EPI_ISL_1806220, EPI_ISL_1806223, EPI_ISL_1806226, EPI_ISL_1806229, EPI_ISL_1806232, EPI_ISL_1806235, EPI_ISL_1806238, EPI_ISL_1806241, EPI_ISL_1806244, EPI_ISL_1806247, EPI_ISL_1806250, EPI_ISL_1806253, EPI_ISL_1806256, EPI_ISL_1806259, EPI_ISL_1806262, EPI_ISL_1806265, EPI_ISL_1806268, EPI_ISL_1806270, EPI_ISL_1806274, EPI_ISL_1806277, EPI_ISL_1806280, EPI_ISL_1806283, EPI_ISL_1806286, EPI_ISL_1806289, EPI_ISL_1806292, EPI_ISL_1806296, EPI_ISL_1806298, EPI_ISL_1806301, EPI_ISL_1806305, EPI_ISL_1806307, EPI_ISL_1806310, EPI_ISL_1806313, EPI_ISL_1806317, EPI_ISL_1806320, EPI_ISL_1806323, EPI_ISL_1806326, EPI_ISL_1806329, EPI_ISL_1806332, EPI_ISL_1806335, EPI_ISL_1806338, EPI_ISL_1806341, EPI_ISL_1806345, EPI_ISL_1806347, EPI_ISL_1825119-1825140, EPI_ISL_1969000, EPI_ISL_2031593-2031635, EPI_ISL_2090629-2090647, EPI_ISL_2095693-2095700, EPI_ISL_2140147-2140169, EPI_ISL_2229207, EPI_ISL_2229209-2229210, EPI_ISL_2229212-2229215, EPI_ISL_2229217-2229218, EPI_ISL_2229220, EPI_ISL_2229222, EPI_ISL_2229224-2229225, EPI_ISL_2229227-2229228, EPI_ISL_2229230-2229231, EPI_ISL_2229233-2229234, EPI_ISL_2598120-2598134, EPI_ISL_2689737-2689754, EPI_ISL_2834621-2834678, EPI_ISL_2934924-2934987, EPI_ISL_2976189-2976299, EPI_ISL_3010500-3010583, EPI_ISL_3052920, EPI_ISL_3064153-3064241, EPI_ISL_3066910-3066987, EPI_ISL_3086849-3086850, EPI_ISL_3090099, EPI_ISL_3090757-3090848, EPI_ISL_3098710, EPI_ISL_3098719, EPI_ISL_3098762, EPI_ISL_3100632-3100677, EPI_ISL_3156454-3156546, EPI_ISL_3164497, EPI_ISL_3190942-3191016, EPI_ISL_3191021, EPI_ISL_3191024, EPI_ISL_3191027, EPI_ISL_3191029-3191030, EPI_ISL_3191035-3191044, EPI_ISL_3217177-3217269, EPI_ISL_3242917-3242999, EPI_ISL_3272351-3272449, EPI_ISL_3272452, EPI_ISL_3272454, EPI_ISL_3272457, EPI_ISL_3272460, EPI_ISL_3272463, EPI_ISL_3272466, EPI_ISL_3272469, EPI_ISL_3272471, EPI_ISL_3272474, EPI_ISL_3272477, EPI_ISL_3272480, EPI_ISL_3272483, EPI_ISL_3272485, EPI_ISL_3272489, EPI_ISL_3272491, EPI_ISL_3272494, EPI_ISL_3272497, EPI_ISL_3272500, EPI_ISL_3272503, EPI_ISL_3272506, EPI_ISL_3272509, EPI_ISL_3272512, EPI_ISL_3272515, EPI_ISL_3272518, EPI_ISL_3272521, EPI_ISL_3272523, EPI_ISL_3272527, EPI_ISL_3272530, EPI_ISL_3272972, EPI_ISL_3273142, EPI_ISL_3273147, EPI_ISL_3273175-3273176, EPI_ISL_3273180, EPI_ISL_3273187, EPI_ISL_3273193, EPI_ISL_3273206, EPI_ISL_3273222, EPI_ISL_3275054-3275138, EPI_ISL_3385856-3385941, EPI_ISL_3404610-3404730, EPI_ISL_3464593-3464683, EPI_ISL_3502770-3502844, EPI_ISL_3502887, EPI_ISL_3543363-3543450, EPI_ISL_3556822-3556911, EPI_ISL_3588721-3588813, EPI_ISL_3640689-3640701, EPI_ISL_3640705, EPI_ISL_3640709, EPI_ISL_3640713, EPI_ISL_3640717, EPI_ISL_3640720, EPI_ISL_3640722, EPI_ISL_3640726, EPI_ISL_3640730, EPI_ISL_3640734, EPI_ISL_3640738, EPI_ISL_3640742, EPI_ISL_3640747, EPI_ISL_3640752, EPI_ISL_3640755, EPI_ISL_3640760, EPI_ISL_3640765, EPI_ISL_3640769, EPI_ISL_3640773, EPI_ISL_3640777, EPI_ISL_3640781, EPI_ISL_3640784, EPI_ISL_3640788, EPI_ISL_3640792, EPI_ISL_3640796, EPI_ISL_3640800, EPI_ISL_3640804, EPI_ISL_3640809, EPI_ISL_3640814, EPI_ISL_3640818, EPI_ISL_3640823, EPI_ISL_3640827, EPI_ISL_3640831, EPI_ISL_3640835, EPI_ISL_3640839, EPI_ISL_3640844, EPI_ISL_3640849, EPI_ISL_3640853, EPI_ISL_3640857, EPI_ISL_3640862, EPI_ISL_3640866, EPI_ISL_3640870, EPI_ISL_3640874, EPI_ISL_3640878, EPI_ISL_3640883, EPI_ISL_3640887, EPI_ISL_3640892, EPI_ISL_3640896, EPI_ISL_3640900, EPI_ISL_3640904, EPI_ISL_3640909, EPI_ISL_3640911, EPI_ISL_3640915, EPI_ISL_3640919, EPI_ISL_3640923, EPI_ISL_3640927, EPI_ISL_3640932, EPI_ISL_3640937, EPI_ISL_3640942, EPI_ISL_3640947, EPI_ISL_3640953, EPI_ISL_3640958, EPI_ISL_3640965, EPI_ISL_3640971, EPI_ISL_3640976, EPI_ISL_3640981, EPI_ISL_3640986, EPI_ISL_3640991, EPI_ISL_3640996, EPI_ISL_3641001, EPI_ISL_3641007, EPI_ISL_3641012, EPI_ISL_3641017, EPI_ISL_3641022, EPI_ISL_3641027, EPI_ISL_3641032, EPI_ISL_3641037, EPI_ISL_3641042, EPI_ISL_3641046, EPI_ISL_3641051, EPI_ISL_3641056, EPI_ISL_3641062, EPI_ISL_3641066, EPI_ISL_3641071, EPI_ISL_3641076, EPI_ISL_3641080, EPI_ISL_3641084, EPI_ISL_3641088, EPI_ISL_3641092, EPI_ISL_3641097, EPI_ISL_3641103, EPI_ISL_3641108, EPI_ISL_3641113, EPI_ISL_3641118, EPI_ISL_3641123, EPI_ISL_3641128, EPI_ISL_3641133, EPI_ISL_3641138, EPI_ISL_3641144, EPI_ISL_3641148, EPI_ISL_3641153, EPI_ISL_3641157, EPI_ISL_3641162, EPI_ISL_3641167, EPI_ISL_3641171, EPI_ISL_3641176, EPI_ISL_3641181, EPI_ISL_3641185, EPI_ISL_3641191, EPI_ISL_3641195, EPI_ISL_3641200, EPI_ISL_3641204, EPI_ISL_3641209, EPI_ISL_3641212, EPI_ISL_3641217, EPI_ISL_3642831-3642867, EPI_ISL_3642872-3642921, EPI_ISL_3664290-3664293, EPI_ISL_3664321-3664328, EPI_ISL_3722342-3722427, EPI_ISL_3743004, EPI_ISL_3743006, EPI_ISL_3743008-3743009, EPI_ISL_3743011, EPI_ISL_3743013-3743014, EPI_ISL_3743016, EPI_ISL_3743018-3743019, EPI_ISL_3743021, EPI_ISL_3743023-3743024, EPI_ISL_3743026, EPI_ISL_3743028-3743029, EPI_ISL_3743031, EPI_ISL_3743033-3743034, EPI_ISL_3743037, EPI_ISL_3743039-3743040, EPI_ISL_3743042-3743044, EPI_ISL_3743046, EPI_ISL_3743048, EPI_ISL_3743050, EPI_ISL_3743053, EPI_ISL_3743056-3743057, EPI_ISL_3743059, EPI_ISL_3743061, EPI_ISL_3743063-3743064, EPI_ISL_3743066, EPI_ISL_3743068, EPI_ISL_3743070-3743071, EPI_ISL_3743074-3743075, EPI_ISL_3743077, EPI_ISL_3743080, EPI_ISL_3743083, EPI_ISL_3743085, EPI_ISL_3743088, EPI_ISL_3743090, EPI_ISL_3743092, EPI_ISL_3743095, EPI_ISL_3743098, EPI_ISL_3743101-3743102, EPI_ISL_3743104, EPI_ISL_3743106, EPI_ISL_3743108-3743109, EPI_ISL_3743111, EPI_ISL_3743113-3743114, EPI_ISL_3743116, EPI_ISL_3743118, EPI_ISL_3743120, EPI_ISL_3743122, EPI_ISL_3743125, EPI_ISL_3743128, EPI_ISL_3743130-3743131, EPI_ISL_3743133, EPI_ISL_3743135, EPI_ISL_3743137-3743138, EPI_ISL_3743140, EPI_ISL_3743142-3743143, EPI_ISL_3743145, EPI_ISL_3743147-3743148, EPI_ISL_3743150, EPI_ISL_3743152, EPI_ISL_3743154-3743155, EPI_ISL_3743157, EPI_ISL_3743159-3743160, EPI_ISL_3743162-3743163, EPI_ISL_3743165, EPI_ISL_3743167, EPI_ISL_3743169-3743170, EPI_ISL_3743172, EPI_ISL_3743174, EPI_ISL_3756615-3756699, EPI_ISL_3756838-3756923, EPI_ISL_3759954-3760044, EPI_ISL_3905577-3905658, EPI_ISL_3905666-3905748, EPI_ISL_3922775-3922943, EPI_ISL_3923011-3923095, EPI_ISL_3923164-3923189, EPI_ISL_3923232-3923337, EPI_ISL_4084621-4084700, EPI_ISL_4091126-4091212, EPI_ISL_4175470, EPI_ISL_4175472-4175473, EPI_ISL_4175475, EPI_ISL_4175477, EPI_ISL_4175479, EPI_ISL_4175481, EPI_ISL_4175483, EPI_ISL_4175485-4175486, EPI_ISL_4175488, EPI_ISL_4175490, EPI_ISL_4175492, EPI_ISL_4175494, EPI_ISL_4175496, EPI_ISL_4175498, EPI_ISL_4175500-4175501, EPI_ISL_4175503, EPI_ISL_4175505, EPI_ISL_4175507, EPI_ISL_4175509, EPI_ISL_4175511, EPI_ISL_4175513-4175514, EPI_ISL_4175516, EPI_ISL_4175518, EPI_ISL_4175520-4175521, EPI_ISL_4175524-4175525, EPI_ISL_4175527, EPI_ISL_4175529, EPI_ISL_4175531, EPI_ISL_4175533, EPI_ISL_4175535-4175536, EPI_ISL_4175538, EPI_ISL_4175540, EPI_ISL_4175542, EPI_ISL_4175544-4175545, EPI_ISL_4175547, EPI_ISL_4175549, EPI_ISL_4175551, EPI_ISL_4175553, EPI_ISL_4175555, EPI_ISL_4175557, EPI_ISL_4175559, EPI_ISL_4175561, EPI_ISL_4175563, EPI_ISL_4175565, EPI_ISL_4175567, EPI_ISL_4175569-4175570, EPI_ISL_4175572, EPI_ISL_4175574, EPI_ISL_4175576, EPI_ISL_4175578, EPI_ISL_4175580, EPI_ISL_4175582, EPI_ISL_4175584, EPI_ISL_4175586, EPI_ISL_4175588, EPI_ISL_4175590, EPI_ISL_4175592, EPI_ISL_4175594-4175595, EPI_ISL_4175597, EPI_ISL_4175599, EPI_ISL_4175601, EPI_ISL_4175603-4175604, EPI_ISL_4175606, EPI_ISL_4175608, EPI_ISL_4175610, EPI_ISL_4175612, EPI_ISL_4175614, EPI_ISL_4175616, EPI_ISL_4175618-4175619, EPI_ISL_4175621, EPI_ISL_4175623, EPI_ISL_4175625, EPI_ISL_4175627, EPI_ISL_4211457-4211476, EPI_ISL_4219243-4219259, EPI_ISL_4256841-4256887, EPI_ISL_4435380-4435467, EPI_ISL_4469547-4469582, EPI_ISL_4474929-4475022, EPI_ISL_4494135-4494208, EPI_ISL_4496487-4496542, EPI_ISL_4730623-4730627, EPI_ISL_4730633, EPI_ISL_4730636-4730641, EPI_ISL_4730645-4730648, EPI_ISL_4878312, EPI_ISL_4889660-4889729, EPI_ISL_4945672-4945674, EPI_ISL_4945679-4945735, EPI_ISL_4945737, EPI_ISL_4955745-4955746, EPI_ISL_4978374-4978449, EPI_ISL_5057881-5057962, EPI_ISL_5071787-5071831, EPI_ISL_5071834-5071881, EPI_ISL_5072680-5072681, EPI_ISL_5072683-5072684, EPI_ISL_5072687, EPI_ISL_5072689, EPI_ISL_5072691-5072692, EPI_ISL_5072694, EPI_ISL_5072696, EPI_ISL_5072698-5072699, EPI_ISL_5072701, EPI_ISL_5072703-5072704, EPI_ISL_5072706, EPI_ISL_5072708, EPI_ISL_5072710-5072711, EPI_ISL_5072713, EPI_ISL_5072715-5072716, EPI_ISL_5072718-5072719, EPI_ISL_5072721-5072722, EPI_ISL_5072724, EPI_ISL_5072726, EPI_ISL_5072728, EPI_ISL_5072730-5072731, EPI_ISL_5072733-5072734, EPI_ISL_5072736-5072738, EPI_ISL_5072740-5072741, EPI_ISL_5072743-5072744, EPI_ISL_5072746, EPI_ISL_5072748-5072749, EPI_ISL_5072751-5072752, EPI_ISL_5072754, EPI_ISL_5072756, EPI_ISL_5072759, EPI_ISL_5072762, EPI_ISL_5072767, EPI_ISL_5072769, EPI_ISL_5072771-5072772, EPI_ISL_5072774-5072775, EPI_ISL_5072777-5072780, EPI_ISL_5072782-5072783, EPI_ISL_5072785, EPI_ISL_5072787-5072788, EPI_ISL_5072790-5072791, EPI_ISL_5072793, EPI_ISL_5072795-5072796, EPI_ISL_5072798, EPI_ISL_5072800-5072801, EPI_ISL_5072803-5072804, EPI_ISL_5072806, EPI_ISL_5072808, EPI_ISL_5072810-5072811, EPI_ISL_5072813, EPI_ISL_5072815, EPI_ISL_5072817-5072818, EPI_ISL_5072820, EPI_ISL_5072822-5072823, EPI_ISL_5072825, EPI_ISL_5072827, EPI_ISL_5072829, EPI_ISL_5072831, EPI_ISL_5072833-5072834, EPI_ISL_5072836, EPI_ISL_5095655-5095734, EPI_ISL_5114936-5115013, EPI_ISL_5115092-5115134, EPI_ISL_5115136-5115172, EPI_ISL_5334248, EPI_ISL_5334587, EPI_ISL_5348547, EPI_ISL_5348554-5348555, EPI_ISL_5348560-5348561, EPI_ISL_5348566-5348567, EPI_ISL_5348572-5348573, EPI_ISL_5348575, EPI_ISL_5348582-5348583, EPI_ISL_5348591-5348592, EPI_ISL_5348595-5348596, EPI_ISL_5348599-5348600, EPI_ISL_5348605-5348606, EPI_ISL_5348612-5348613, EPI_ISL_5348619-5348620, EPI_ISL_5348628-5348629, EPI_ISL_5348636-5348637, EPI_ISL_5348643-5348644, EPI_ISL_5348649-5348650, EPI_ISL_5348657, EPI_ISL_5348662, EPI_ISL_5348666, EPI_ISL_5348673-5348674, EPI_ISL_5348679-5348680, EPI_ISL_5348687-5348688, EPI_ISL_5348692-5348693, EPI_ISL_5348702-5348704, EPI_ISL_5348708-5348709, EPI_ISL_5348713, EPI_ISL_5348720-5348721, EPI_ISL_5348728, EPI_ISL_5348732-5348733, EPI_ISL_5348739-5348740, EPI_ISL_5348746-5348748, EPI_ISL_5348757-5348758, EPI_ISL_5348763, EPI_ISL_5348769-5348770, EPI_ISL_5348774-5348775, EPI_ISL_5348782-5348783, EPI_ISL_5348787-5348788, EPI_ISL_5441241-5441315, EPI_ISL_5511689-5511695, EPI_ISL_5696764-5696843, EPI_ISL_5845388, EPI_ISL_5850374-5850416, EPI_ISL_5850421-5850446, EPI_ISL_5851328-5851417, EPI_ISL_5851420-5851498, EPI_ISL_5851502-5851603, EPI_ISL_5851605-5851655, EPI_ISL_5851723-5851815, EPI_ISL_5851820-5851910, EPI_ISL_5851915-5851938, EPI_ISL_5868823-5868907, EPI_ISL_5868910-5869000, EPI_ISL_5869003-5869082, EPI_ISL_5869084-5869122, EPI_ISL_5924689-5924693, EPI_ISL_5931619-5931621, EPI_ISL_5934840-5934848, EPI_ISL_5990147-5990204, EPI_ISL_6102162, EPI_ISL_6340373-6340424, EPI_ISL_6340462-6340523, EPI_ISL_6436860-6436862, EPI_ISL_6477199, EPI_ISL_6477201, EPI_ISL_6477203, EPI_ISL_6477205-6477206, EPI_ISL_6477208-6477209, EPI_ISL_6477211, EPI_ISL_6477213, EPI_ISL_6477215-6477216, EPI_ISL_6477218, EPI_ISL_6477220, EPI_ISL_6477222-6477223, EPI_ISL_6477225, EPI_ISL_6477227, EPI_ISL_6477229-6477230, EPI_ISL_6477232, EPI_ISL_6477234, EPI_ISL_6477236-6477237, EPI_ISL_6585694, EPI_ISL_6606438, EPI_ISL_6606446, EPI_ISL_6606455, EPI_ISL_6606466, EPI_ISL_6606470, EPI_ISL_6606476, EPI_ISL_6606485, EPI_ISL_6606492, EPI_ISL_6606495, EPI_ISL_6606500, EPI_ISL_6606510, EPI_ISL_6606516, EPI_ISL_6606523, EPI_ISL_6606529, EPI_ISL_6606533, EPI_ISL_6606539, EPI_ISL_6606549, EPI_ISL_6606559, EPI_ISL_6606568, EPI_ISL_6606575, EPI_ISL_6606580, EPI_ISL_6606586, EPI_ISL_6606594, EPI_ISL_6606601, EPI_ISL_6606607, EPI_ISL_6606614, EPI_ISL_6606618, EPI_ISL_6606621, EPI_ISL_6606628, EPI_ISL_6606632, EPI_ISL_6606636, EPI_ISL_6606644, EPI_ISL_6606648, EPI_ISL_6606656, EPI_ISL_6606660, EPI_ISL_6606664, EPI_ISL_6606675, EPI_ISL_6606678, EPI_ISL_6606688, EPI_ISL_6606696, EPI_ISL_6606704, EPI_ISL_6606710, EPI_ISL_6606717, EPI_ISL_6606724, EPI_ISL_6606733, EPI_ISL_6606737, EPI_ISL_6606743, EPI_ISL_6606747, EPI_ISL_6606757, EPI_ISL_6606763, EPI_ISL_6606769, EPI_ISL_6606777, EPI_ISL_6606786, EPI_ISL_6606793, EPI_ISL_6606800, EPI_ISL_6606810, EPI_ISL_6606815, EPI_ISL_6606822, EPI_ISL_6606831, EPI_ISL_6606840, EPI_ISL_6606848, EPI_ISL_6606856, EPI_ISL_6647949-6647951, EPI_ISL_6794949-6794979, EPI_ISL_6795087-6795148, EPI_ISL_6913893-6913894, EPI_ISL_7017893-7017966, EPI_ISL_7017968, EPI_ISL_7163309-7163367, EPI_ISL_7163370-7163380, EPI_ISL_7229474, EPI_ISL_7229486, EPI_ISL_7313992-7314004, EPI_ISL_7314006-7314067, EPI_ISL_7314153-7314192, EPI_ISL_7314194-7314205, EPI_ISL_7314208-7314228, EPI_ISL_7314233-7314252, EPI_ISL_7314257-7314268, EPI_ISL_7314273-7314278, EPI_ISL_7314281-7314297, EPI_ISL_7314303-7314304, EPI_ISL_7338767-7338768, EPI_ISL_7355476, EPI_ISL_7355482, EPI_ISL_7355490, EPI_ISL_7355497, EPI_ISL_7355526, EPI_ISL_7355554, EPI_ISL_7355581, EPI_ISL_7355606, EPI_ISL_7355634, EPI_ISL_7355659, EPI_ISL_7355690, EPI_ISL_7355719, EPI_ISL_7355739, EPI_ISL_7355768, EPI_ISL_7355785, EPI_ISL_7355799, EPI_ISL_7355825, EPI_ISL_7355848, EPI_ISL_7355879, EPI_ISL_7355899, EPI_ISL_7355922, EPI_ISL_7355946, EPI_ISL_7355968, EPI_ISL_7355986, EPI_ISL_7356008, EPI_ISL_7356032, EPI_ISL_7356055, EPI_ISL_7356080, EPI_ISL_7356110, EPI_ISL_7356132, EPI_ISL_7356167, EPI_ISL_7356194, EPI_ISL_7356220, EPI_ISL_7356245, EPI_ISL_7356269, EPI_ISL_7356291, EPI_ISL_7356318, EPI_ISL_7356335, EPI_ISL_7356360, EPI_ISL_7356382, EPI_ISL_7356405, EPI_ISL_7356433, EPI_ISL_7356458, EPI_ISL_7356478, EPI_ISL_7356495, EPI_ISL_7356520, EPI_ISL_7356535, EPI_ISL_7356564, EPI_ISL_7356594, EPI_ISL_7356614, EPI_ISL_7356636, EPI_ISL_7356664, EPI_ISL_7356688, EPI_ISL_7356706, EPI_ISL_7356724, EPI_ISL_7356747, EPI_ISL_7356770, EPI_ISL_7356808, EPI_ISL_7356834, EPI_ISL_7356838, EPI_ISL_7356846, EPI_ISL_7356851, EPI_ISL_7356859, EPI_ISL_7356862, EPI_ISL_7356871, EPI_ISL_7356877, EPI_ISL_7356887, EPI_ISL_7356894, EPI_ISL_7356903, EPI_ISL_7356912, EPI_ISL_7356921, EPI_ISL_7356930, EPI_ISL_7356938, EPI_ISL_7356947, EPI_ISL_7356954, EPI_ISL_7356964, EPI_ISL_7356975, EPI_ISL_7356984, EPI_ISL_7356992, EPI_ISL_7456985-7457002, EPI_ISL_7457006-7457060, EPI_ISL_7492647-7492671, EPI_ISL_7492675-7492698, EPI_ISL_7492700-7492724, EPI_ISL_7492728-7492735, EPI_ISL_7496679-7496724, EPI_ISL_7496729-7496734, EPI_ISL_7496737-7496739, EPI_ISL_7622544-7622596, EPI_ISL_7622598-7622618, EPI_ISL_7622620-7622671, EPI_ISL_7622676-7622734, EPI_ISL_7622737-7622755, EPI_ISL_7622760-7622860, EPI_ISL_7648887, EPI_ISL_7648903, EPI_ISL_7648905, EPI_ISL_7648908, EPI_ISL_7648910, EPI_ISL_7648912, EPI_ISL_7648914, EPI_ISL_7648916, EPI_ISL_7648919, EPI_ISL_7648921, EPI_ISL_7648924, EPI_ISL_7648926, EPI_ISL_7648929, EPI_ISL_7648931, EPI_ISL_7648934, EPI_ISL_7648936, EPI_ISL_7648938, EPI_ISL_7648941, EPI_ISL_7648943, EPI_ISL_7648946, EPI_ISL_7648948, EPI_ISL_7648951, EPI_ISL_7648953, EPI_ISL_7648956, EPI_ISL_7648959, EPI_ISL_7648962, EPI_ISL_7648965, EPI_ISL_7648968, EPI_ISL_7648971, EPI_ISL_7648974, EPI_ISL_7648977, EPI_ISL_7648986, EPI_ISL_7648994, EPI_ISL_7648996, EPI_ISL_7648999, EPI_ISL_7671313, EPI_ISL_7674334-7674342, EPI_ISL_7674344-7674538, EPI_ISL_7674540-7674627, EPI_ISL_7778211-7778286, EPI_ISL_7792103, EPI_ISL_7832645-7832693, EPI_ISL_7950352-7950400, EPI_ISL_7983814, EPI_ISL_8130198-8130248, EPI_ISL_8131550-8131628, EPI_ISL_8166892-8166971, EPI_ISL_8184775-8184777, EPI_ISL_8186683-8186688, EPI_ISL_8186703-8186707, EPI_ISL_8318248-8318314, EPI_ISL_8318980-8318983, EPI_ISL_8324898-8324970, EPI_ISL_8329270-8329290, EPI_ISL_8329292-8329357, EPI_ISL_8351203-8351216, EPI_ISL_8629099-8629173, EPI_ISL_8629214-8629274, EPI_ISL_8639330-8639333, EPI_ISL_8639360-8639364, EPI_ISL_8681649, EPI_ISL_8681651-8681686, EPI_ISL_8681688-8681711, EPI_ISL_8717997-8718070, EPI_ISL_8750817-8750866, EPI_ISL_8883001-8883179, EPI_ISL_8900516-8900852, EPI_ISL_8901883-8901884, EPI_ISL_8975267-8975320, EPI_ISL_9030095-9030105, EPI_ISL_9047087-9047116, EPI_ISL_9093135-9093196, EPI_ISL_9095780-9095858, EPI_ISL_9100789-9100798, EPI_ISL_9112401-9112441, EPI_ISL_9112443-9112488, EPI_ISL_9143403-9143405, EPI_ISL_9152928-9153007, EPI_ISL_9203155-9203227, EPI_ISL_9271651-9271736, EPI_ISL_9348922-9348994, EPI_ISL_9352263-9352342, EPI_ISL_9356683-9356764, EPI_ISL_9410495-9410498, EPI_ISL_9410500-9410575, EPI_ISL_9437768-9437843, EPI_ISL_9471578-9471588, EPI_ISL_9471590-9471633, EPI_ISL_9487023-9487112, EPI_ISL_9520523-9520538, EPI_ISL_9520540-9520604, EPI_ISL_9551856-9551940, EPI_ISL_9636984-9637044, EPI_ISL_9805649, EPI_ISL_9873088-9873103, EPI_ISL_9873105-9873126, EPI_ISL_9873128-9873162, EPI_ISL_9873507-9873560, EPI_ISL_9873562-9873578, EPI_ISL_9928485-9928549, EPI_ISL_9928551-9928553, EPI_ISL_10057438-10057491, EPI_ISL_10057493-10057510, EPI_ISL_10082603-10082686, EPI_ISL_10239441-10239503, EPI_ISL_10239657-10239720, EPI_ISL_10239779-10239859, EPI_ISL_10239861-10239944, EPI_ISL_10283576, EPI_ISL_10284172, EPI_ISL_10284716-10284718, EPI_ISL_10402739-10402769, EPI_ISL_10575103-10575384, EPI_ISL_10706201-10706250, EPI_ISL_10792673-10792679, EPI_ISL_11096036-11096037, EPI_ISL_11096039-11096077, EPI_ISL_11096079-11096102, EPI_ISL_11567772-11567824, EPI_ISL_11799908-11799969, EPI_ISL_12001704-12001732, EPI_ISL_12036973-12038956, EPI_ISL_12166970-12167007, EPI_ISL_12167009-12167021, EPI_ISL_12471173-12471217, EPI_ISL_12573620-12573681, EPI_ISL_12599428-12599506, EPI_ISL_12617586-12617667, EPI_ISL_12869399-12869426, EPI_ISL_13044520-13044590, EPI_ISL_13125307-13125374, EPI_ISL_13191623-13191689, EPI_ISL_13282059-13282111, EPI_ISL_13346203-13346242, EPI_ISL_13413796-13413798, EPI_ISL_13436070-13436089, EPI_ISL_13453572-13453655, EPI_ISL_13563917-13563933, EPI_ISL_13647726-13647802, EPI_ISL_13811828-13811835, EPI_ISL_13812248-13812307, EPI_ISL_13829359-13829445, EPI_ISL_13871229-13871292, EPI_ISL_14064584, EPI_ISL_14069795-14069801, EPI_ISL_14069805-14069806, EPI_ISL_14224066-14224143, EPI_ISL_14253402-14253470, EPI_ISL_14287960-14288034, EPI_ISL_14333784-14333843, EPI_ISL_14488398-14488461, EPI_ISL_14588225-14588292, EPI_ISL_14881049-14881080, EPI_ISL_14907032-14907098, EPI_ISL_15068083-15068135, EPI_ISL_15279926-15279960, EPI_ISL_15377008-15377077, EPI_ISL_15415338-15415391, EPI_ISL_15416614-15416655, EPI_ISL_15659017-15659058, EPI_ISL_15692996-15693038, EPI_ISL_15734312-15734353, EPI_ISL_15783595-15783654, EPI_ISL_15998820-15998859, EPI_ISL_16049757-16049846, EPI_ISL_16137517-16137564, EPI_ISL_16225759-16225826, EPI_ISL_16314961-16315008, EPI_ISL_16372293-16372321, EPI_ISL_16553477-16553528, EPI_ISL_16852846-16852883, EPI_ISL_16893894-16893896, EPI_ISL_16946504-16946522, EPI_ISL_16947535-16947556, EPI_ISL_16987766-16987798, EPI_ISL_17089345-17089388, EPI_ISL_17162889-17162920, EPI_ISL_17260602-17260660, EPI_ISL_17273394-17273433, EPI_ISL_17463038-17463055, EPI_ISL_17546498-17546515, EPI_ISL_18231364-18231376, EPI_ISL_18550551-18550565, EPI_ISL_18551259-18551267, EPI_ISL_18744051-18744057, EPI_ISL_18840357-18840365, EPI_ISL_19315398-19315403, EPI_ISL_19364518-19364520")
Indonesia = stringlist_to_set("EPI_ISL_3019945, EPI_ISL_2854687, EPI_ISL_2854775, EPI_ISL_2868873, EPI_ISL_2868876, EPI_ISL_1824617, EPI_ISL_8889354, EPI_ISL_14583732, EPI_ISL_14584687, EPI_ISL_14584697")
Mexico = stringlist_to_set("EPI_ISL_13398383, EPI_ISL_7716412, EPI_ISL_7716415, EPI_ISL_7716416, EPI_ISL_7716417, EPI_ISL_7716418, EPI_ISL_7716419, EPI_ISL_7716421, EPI_ISL_7716425, EPI_ISL_7716427, EPI_ISL_7716429, EPI_ISL_7716431, EPI_ISL_7716432, EPI_ISL_7716433, EPI_ISL_7716435, EPI_ISL_7716439, EPI_ISL_7716441, EPI_ISL_7716442, EPI_ISL_7716450, EPI_ISL_7716454, EPI_ISL_7716455, EPI_ISL_7716456, EPI_ISL_7716459, EPI_ISL_7716460, EPI_ISL_7716482, EPI_ISL_7716483, EPI_ISL_7716485, EPI_ISL_7716486, EPI_ISL_7716487, EPI_ISL_7716488, EPI_ISL_7716489, EPI_ISL_7730969")
Texas = stringlist_to_set("EPI_ISL_1236653, EPI_ISL_1078985, EPI_ISL_1079742, EPI_ISL_17061113, EPI_ISL_1079482, EPI_ISL_18817042, EPI_ISL_1079887, EPI_ISL_16762249, EPI_ISL_16762269, EPI_ISL_16542938, EPI_ISL_16398020, EPI_ISL_16012710, EPI_ISL_16397783, EPI_ISL_16762458, EPI_ISL_16762297, EPI_ISL_16599844, EPI_ISL_16599878, EPI_ISL_16599980, EPI_ISL_16599842, EPI_ISL_16599826, EPI_ISL_16599959, EPI_ISL_16599794, EPI_ISL_16599636, EPI_ISL_16599538, EPI_ISL_16599540, EPI_ISL_16599492, EPI_ISL_16599498, EPI_ISL_16599495, EPI_ISL_16599649, EPI_ISL_16599554, EPI_ISL_16599460, EPI_ISL_16599380, EPI_ISL_16599389, EPI_ISL_16599299, EPI_ISL_16599467, EPI_ISL_16542453, EPI_ISL_16454140, EPI_ISL_16454168, EPI_ISL_16398063, EPI_ISL_16398072, EPI_ISL_16398022, EPI_ISL_16397883, EPI_ISL_16398016, EPI_ISL_16397988, EPI_ISL_16397860, EPI_ISL_16397721, EPI_ISL_16397554, EPI_ISL_16397720, EPI_ISL_16397699, EPI_ISL_16271386, EPI_ISL_16271102, EPI_ISL_16271048, EPI_ISL_16271255, EPI_ISL_16113938, EPI_ISL_16114486, EPI_ISL_16113336, EPI_ISL_16113255, EPI_ISL_16113399, EPI_ISL_16013449, EPI_ISL_16013309, EPI_ISL_16013227, EPI_ISL_16012987, EPI_ISL_16012954, EPI_ISL_16012943, EPI_ISL_16013209, EPI_ISL_16013267, EPI_ISL_16599525, EPI_ISL_16762084, EPI_ISL_16599288, EPI_ISL_16599747, EPI_ISL_16270893, EPI_ISL_16762700, EPI_ISL_16762753, EPI_ISL_16762699, EPI_ISL_16762631, EPI_ISL_16762654, EPI_ISL_16762714, EPI_ISL_16762735, EPI_ISL_16762653, EPI_ISL_16762559, EPI_ISL_16762726, EPI_ISL_16762652, EPI_ISL_16762600, EPI_ISL_16762614, EPI_ISL_16762407, EPI_ISL_16762427, EPI_ISL_16762327, EPI_ISL_16762455, EPI_ISL_16762310, EPI_ISL_16762611, EPI_ISL_16762475, EPI_ISL_16762245, EPI_ISL_16762132, EPI_ISL_16762035, EPI_ISL_16762052, EPI_ISL_16762273, EPI_ISL_16762070, EPI_ISL_16762022, EPI_ISL_16762148, EPI_ISL_16761962, EPI_ISL_16762255, EPI_ISL_16762066, EPI_ISL_16761978, EPI_ISL_16761930, EPI_ISL_16761912, EPI_ISL_16542189, EPI_ISL_16542369, EPI_ISL_16542267, EPI_ISL_16762176, EPI_ISL_16762028, EPI_ISL_16762462, EPI_ISL_16599325, EPI_ISL_16542552, EPI_ISL_16271333, EPI_ISL_16398035, EPI_ISL_16398060, EPI_ISL_16919306, EPI_ISL_16919337, EPI_ISL_16919315, EPI_ISL_16919296, EPI_ISL_16919348, EPI_ISL_16919365, EPI_ISL_16919303, EPI_ISL_16919368, EPI_ISL_16919380, EPI_ISL_16919362, EPI_ISL_16919367, EPI_ISL_16919388, EPI_ISL_16919412, EPI_ISL_16919396, EPI_ISL_16919435, EPI_ISL_16919455, EPI_ISL_16919430, EPI_ISL_16919399, EPI_ISL_16919496, EPI_ISL_16919468, EPI_ISL_16919526, EPI_ISL_16919523, EPI_ISL_16919467, EPI_ISL_16919558, EPI_ISL_16919504, EPI_ISL_16919456, EPI_ISL_16919592, EPI_ISL_16919598, EPI_ISL_16919609, EPI_ISL_16919586, EPI_ISL_16919660, EPI_ISL_16919546, EPI_ISL_16919662, EPI_ISL_16919695, EPI_ISL_16919680, EPI_ISL_16919670, EPI_ISL_16919640, EPI_ISL_16919718, EPI_ISL_16919762, EPI_ISL_16919794, EPI_ISL_16919804, EPI_ISL_16919796, EPI_ISL_16919806, EPI_ISL_16919772, EPI_ISL_16919841, EPI_ISL_16919706, EPI_ISL_16919843, EPI_ISL_16919845, EPI_ISL_16919810, EPI_ISL_16919894, EPI_ISL_16919852, EPI_ISL_16919902, EPI_ISL_16919932, EPI_ISL_16919919, EPI_ISL_16919944, EPI_ISL_16919906, EPI_ISL_16919897, EPI_ISL_16919870, EPI_ISL_16919856, EPI_ISL_16919962")
Thailand = stringlist_to_set("EPI_ISL_11720069, EPI_ISL_17785884, EPI_ISL_6136058, EPI_ISL_6599859, EPI_ISL_5916960, EPI_ISL_5916998, EPI_ISL_3636272, EPI_ISL_4348475")
Turkey = stringlist_to_set("EPI_ISL_12146153, EPI_ISL_12321719, EPI_ISL_14298818, EPI_ISL_14456697, EPI_ISL_12610886, EPI_ISL_10858468, EPI_ISL_11507543, EPI_ISL_7874635, EPI_ISL_10858493, EPI_ISL_12146142, EPI_ISL_12320637, EPI_ISL_13112964, EPI_ISL_13622800, EPI_ISL_10397169, EPI_ISL_10397203, EPI_ISL_10678671, EPI_ISL_10678679, EPI_ISL_12319870, EPI_ISL_12584102, EPI_ISL_13074466, EPI_ISL_13112909, EPI_ISL_13176001, EPI_ISL_13621057, EPI_ISL_15927979, EPI_ISL_2102044, EPI_ISL_15681715, EPI_ISL_15295325")
Good_But_Not_Chronic = stringlist_to_set("EPI_ISL_5336026, EPI_ISL_1419150, EPI_ISL_894228, EPI_ISL_894229, EPI_ISL_1165744, EPI_ISL_1232443, EPI_ISL_1370821")
EPI_8mil_9mil = stringlist_to_set("EPI_ISL_8001937, EPI_ISL_8014799, EPI_ISL_8017918, EPI_ISL_8029713, EPI_ISL_8030450, EPI_ISL_8030625, EPI_ISL_8038409, EPI_ISL_8044393, EPI_ISL_8046586, EPI_ISL_8049906, EPI_ISL_8049915, EPI_ISL_8049917, EPI_ISL_8049918, EPI_ISL_8049919, EPI_ISL_8060345, EPI_ISL_8060406, EPI_ISL_8060833, EPI_ISL_8060894, EPI_ISL_8060974, EPI_ISL_8062145, EPI_ISL_8063083, EPI_ISL_8068107, EPI_ISL_8078285, EPI_ISL_8079424, EPI_ISL_8079488, EPI_ISL_8079555, EPI_ISL_8080096, EPI_ISL_8080419, EPI_ISL_8082607, EPI_ISL_8082623, EPI_ISL_8082825, EPI_ISL_8087834, EPI_ISL_8091177, EPI_ISL_8092036, EPI_ISL_8097113, EPI_ISL_8097119, EPI_ISL_8106398, EPI_ISL_8109581, EPI_ISL_8111073, EPI_ISL_8112683, EPI_ISL_8115748, EPI_ISL_8116089, EPI_ISL_8116090, EPI_ISL_8122388, EPI_ISL_8126914, EPI_ISL_8132420, EPI_ISL_8133923, EPI_ISL_8149958, EPI_ISL_8157487, EPI_ISL_8159346, EPI_ISL_8163293, EPI_ISL_8163307, EPI_ISL_8163414, EPI_ISL_8166517, EPI_ISL_8170854, EPI_ISL_8180529, EPI_ISL_8182884, EPI_ISL_8183017, EPI_ISL_8183131, EPI_ISL_8183263, EPI_ISL_8185416, EPI_ISL_8186670, EPI_ISL_8188582, EPI_ISL_8188583, EPI_ISL_8188584, EPI_ISL_8188585, EPI_ISL_8188586, EPI_ISL_8188587, EPI_ISL_8188588, EPI_ISL_8188589, EPI_ISL_8189141, EPI_ISL_8193597, EPI_ISL_8200642, EPI_ISL_8206829, EPI_ISL_8206840, EPI_ISL_8206864, EPI_ISL_8207740, EPI_ISL_8229935, EPI_ISL_8229990, EPI_ISL_8233602, EPI_ISL_8236893, EPI_ISL_8238189, EPI_ISL_8238279, EPI_ISL_8241972, EPI_ISL_8242673, EPI_ISL_8243729, EPI_ISL_8248465, EPI_ISL_8248962, EPI_ISL_8255974, EPI_ISL_8257133, EPI_ISL_8257994, EPI_ISL_8258023, EPI_ISL_8259443, EPI_ISL_8260129, EPI_ISL_8260467, EPI_ISL_8260727, EPI_ISL_8260969, EPI_ISL_8261006, EPI_ISL_8261145, EPI_ISL_8261170, EPI_ISL_8261175, EPI_ISL_8261182, EPI_ISL_8261191, EPI_ISL_8261196, EPI_ISL_8261207, EPI_ISL_8261636, EPI_ISL_8262817, EPI_ISL_8263131, EPI_ISL_8265180, EPI_ISL_8267137, EPI_ISL_8276676, EPI_ISL_8280181, EPI_ISL_8281741, EPI_ISL_8282119, EPI_ISL_8282662, EPI_ISL_8287151, EPI_ISL_8287534, EPI_ISL_8289103, EPI_ISL_8289305, EPI_ISL_8289538, EPI_ISL_8289922, EPI_ISL_8290718, EPI_ISL_8290776, EPI_ISL_8293444, EPI_ISL_8294105, EPI_ISL_8295523, EPI_ISL_8303487, EPI_ISL_8304935, EPI_ISL_8305718, EPI_ISL_8306738, EPI_ISL_8307368, EPI_ISL_8307404, EPI_ISL_8307812, EPI_ISL_8308045, EPI_ISL_8308655, EPI_ISL_8308964, EPI_ISL_8309887, EPI_ISL_8309894, EPI_ISL_8310980, EPI_ISL_8311137, EPI_ISL_8311149, EPI_ISL_8311897, EPI_ISL_8314392, EPI_ISL_8315214, EPI_ISL_8317836, EPI_ISL_8321859, EPI_ISL_8328897, EPI_ISL_8328910, EPI_ISL_8329948, EPI_ISL_8336265, EPI_ISL_8336433, EPI_ISL_8336449, EPI_ISL_8336450, EPI_ISL_8337681, EPI_ISL_8337736, EPI_ISL_8338155, EPI_ISL_8351717, EPI_ISL_8352253, EPI_ISL_8353002, EPI_ISL_8353108, EPI_ISL_8354726, EPI_ISL_8355349, EPI_ISL_8355392, EPI_ISL_8356546, EPI_ISL_8356698, EPI_ISL_8357683, EPI_ISL_8358788, EPI_ISL_8364636, EPI_ISL_8365056, EPI_ISL_8365900, EPI_ISL_8373815, EPI_ISL_8373830, EPI_ISL_8373837, EPI_ISL_8374787, EPI_ISL_8374830, EPI_ISL_8374840, EPI_ISL_8374865, EPI_ISL_8374872, EPI_ISL_8374901, EPI_ISL_8377503, EPI_ISL_8381143, EPI_ISL_8381594, EPI_ISL_8381916, EPI_ISL_8381956, EPI_ISL_8384493, EPI_ISL_8392598, EPI_ISL_8393372, EPI_ISL_8393961, EPI_ISL_8394061, EPI_ISL_8396883, EPI_ISL_8402134, EPI_ISL_8402159, EPI_ISL_8404859, EPI_ISL_8404938, EPI_ISL_8404940, EPI_ISL_8404941, EPI_ISL_8405256, EPI_ISL_8405273, EPI_ISL_8405275, EPI_ISL_8405278, EPI_ISL_8405315, EPI_ISL_8405318, EPI_ISL_8405319, EPI_ISL_8405320, EPI_ISL_8405322, EPI_ISL_8405323, EPI_ISL_8405380, EPI_ISL_8405381, EPI_ISL_8405386, EPI_ISL_8410467, EPI_ISL_8410970, EPI_ISL_8411771, EPI_ISL_8416220, EPI_ISL_8416492, EPI_ISL_8418256, EPI_ISL_8420864, EPI_ISL_8424966, EPI_ISL_8425444, EPI_ISL_8425523, EPI_ISL_8425592, EPI_ISL_8425672, EPI_ISL_8425704, EPI_ISL_8425738, EPI_ISL_8426120, EPI_ISL_8426360, EPI_ISL_8427960, EPI_ISL_8428965, EPI_ISL_8433278, EPI_ISL_8433519, EPI_ISL_8434288, EPI_ISL_8434905, EPI_ISL_8436270, EPI_ISL_8449160, EPI_ISL_8456090, EPI_ISL_8457545, EPI_ISL_8458445, EPI_ISL_8462049, EPI_ISL_8465951, EPI_ISL_8467603, EPI_ISL_8468086, EPI_ISL_8468812, EPI_ISL_8470809, EPI_ISL_8471061, EPI_ISL_8471158, EPI_ISL_8471718, EPI_ISL_8471777, EPI_ISL_8472812, EPI_ISL_8472952, EPI_ISL_8475117, EPI_ISL_8480319, EPI_ISL_8482448, EPI_ISL_8484894, EPI_ISL_8494317, EPI_ISL_8494323, EPI_ISL_8494345, EPI_ISL_8495309, EPI_ISL_8508586, EPI_ISL_8512920, EPI_ISL_8513858, EPI_ISL_8513971, EPI_ISL_8514031, EPI_ISL_8517222, EPI_ISL_8517765, EPI_ISL_8517847, EPI_ISL_8521955, EPI_ISL_8524441, EPI_ISL_8524457, EPI_ISL_8525646, EPI_ISL_8525691, EPI_ISL_8527625, EPI_ISL_8527626, EPI_ISL_8529690, EPI_ISL_8530292, EPI_ISL_8530350, EPI_ISL_8530555, EPI_ISL_8530614, EPI_ISL_8538969, EPI_ISL_8541464, EPI_ISL_8541468, EPI_ISL_8541479, EPI_ISL_8546707, EPI_ISL_8546869, EPI_ISL_8546880, EPI_ISL_8546881, EPI_ISL_8546882, EPI_ISL_8546887, EPI_ISL_8546891, EPI_ISL_8546904, EPI_ISL_8546914, EPI_ISL_8546921, EPI_ISL_8546924, EPI_ISL_8546931, EPI_ISL_8546941, EPI_ISL_8546942, EPI_ISL_8546943, EPI_ISL_8546950, EPI_ISL_8546961, EPI_ISL_8546971, EPI_ISL_8546974, EPI_ISL_8546975, EPI_ISL_8546976, EPI_ISL_8546980, EPI_ISL_8546991, EPI_ISL_8546994, EPI_ISL_8547002, EPI_ISL_8547004, EPI_ISL_8551701, EPI_ISL_8551761, EPI_ISL_8552453, EPI_ISL_8552532, EPI_ISL_8552581, EPI_ISL_8552772, EPI_ISL_8554484, EPI_ISL_8554672, EPI_ISL_8554757, EPI_ISL_8555445, EPI_ISL_8555469, EPI_ISL_8556379, EPI_ISL_8557231, EPI_ISL_8573331, EPI_ISL_8578788, EPI_ISL_8581082, EPI_ISL_8581168, EPI_ISL_8581612, EPI_ISL_8581634, EPI_ISL_8584284, EPI_ISL_8584288, EPI_ISL_8584342, EPI_ISL_8584781, EPI_ISL_8584953, EPI_ISL_8585021, EPI_ISL_8588323, EPI_ISL_8588361, EPI_ISL_8588364, EPI_ISL_8588380, EPI_ISL_8588388, EPI_ISL_8588391, EPI_ISL_8588427, EPI_ISL_8588430, EPI_ISL_8588432, EPI_ISL_8588435, EPI_ISL_8591364, EPI_ISL_8592531, EPI_ISL_8593220, EPI_ISL_8603155, EPI_ISL_8605978, EPI_ISL_8606026, EPI_ISL_8611160, EPI_ISL_8623916, EPI_ISL_8624527, EPI_ISL_8625737, EPI_ISL_8632712, EPI_ISL_8637660, EPI_ISL_8637762, EPI_ISL_8647028, EPI_ISL_8647980, EPI_ISL_8648923, EPI_ISL_8650390, EPI_ISL_8652362, EPI_ISL_8652364, EPI_ISL_8652383, EPI_ISL_8652384, EPI_ISL_8654313, EPI_ISL_8658036, EPI_ISL_8662478, EPI_ISL_8665279, EPI_ISL_8665287, EPI_ISL_8668093, EPI_ISL_8674922, EPI_ISL_8674923, EPI_ISL_8674925, EPI_ISL_8674927, EPI_ISL_8674928, EPI_ISL_8674929, EPI_ISL_8674930, EPI_ISL_8674933, EPI_ISL_8674934, EPI_ISL_8674936, EPI_ISL_8674938, EPI_ISL_8676067, EPI_ISL_8676068, EPI_ISL_8676069, EPI_ISL_8676070, EPI_ISL_8676071, EPI_ISL_8676072, EPI_ISL_8676073, EPI_ISL_8676074, EPI_ISL_8676075, EPI_ISL_8676076, EPI_ISL_8676078, EPI_ISL_8676079, EPI_ISL_8676080, EPI_ISL_8676081, EPI_ISL_8676082, EPI_ISL_8676083, EPI_ISL_8676084, EPI_ISL_8676085, EPI_ISL_8676086, EPI_ISL_8676087, EPI_ISL_8676088, EPI_ISL_8676089, EPI_ISL_8676091, EPI_ISL_8676092, EPI_ISL_8676093, EPI_ISL_8676094, EPI_ISL_8676095, EPI_ISL_8676096, EPI_ISL_8678185, EPI_ISL_8678971, EPI_ISL_8679259, EPI_ISL_8679268, EPI_ISL_8679715, EPI_ISL_8679849, EPI_ISL_8681601, EPI_ISL_8683706, EPI_ISL_8683806, EPI_ISL_8684846, EPI_ISL_8685784, EPI_ISL_8686045, EPI_ISL_8686128, EPI_ISL_8686207, EPI_ISL_8686269, EPI_ISL_8686300, EPI_ISL_8686328, EPI_ISL_8686493, EPI_ISL_8686597, EPI_ISL_8686682, EPI_ISL_8686690, EPI_ISL_8686754, EPI_ISL_8686761, EPI_ISL_8686789, EPI_ISL_8686819, EPI_ISL_8686839, EPI_ISL_8686847, EPI_ISL_8687000, EPI_ISL_8687449, EPI_ISL_8687931, EPI_ISL_8687964, EPI_ISL_8688006, EPI_ISL_8688043, EPI_ISL_8688224, EPI_ISL_8688225, EPI_ISL_8688245, EPI_ISL_8700224, EPI_ISL_8703745, EPI_ISL_8703751, EPI_ISL_8703820, EPI_ISL_8704528, EPI_ISL_8705856, EPI_ISL_8707751, EPI_ISL_8707755, EPI_ISL_8707757, EPI_ISL_8707765, EPI_ISL_8713502, EPI_ISL_8717411, EPI_ISL_8717997, EPI_ISL_8718031, EPI_ISL_8718035, EPI_ISL_8718060, EPI_ISL_8718446, EPI_ISL_8719196, EPI_ISL_8719218, EPI_ISL_8719280, EPI_ISL_8719337, EPI_ISL_8719483, EPI_ISL_8719489, EPI_ISL_8719543, EPI_ISL_8719614, EPI_ISL_8720079, EPI_ISL_8720127, EPI_ISL_8733411, EPI_ISL_8733650, EPI_ISL_8739680, EPI_ISL_8739918, EPI_ISL_8742288, EPI_ISL_8742542, EPI_ISL_8744107, EPI_ISL_8746666, EPI_ISL_8746676, EPI_ISL_8746677, EPI_ISL_8746824, EPI_ISL_8748244, EPI_ISL_8754017, EPI_ISL_8766584, EPI_ISL_8766632, EPI_ISL_8766766, EPI_ISL_8767028, EPI_ISL_8767346, EPI_ISL_8768280, EPI_ISL_8769246, EPI_ISL_8769320, EPI_ISL_8769322, EPI_ISL_8769379, EPI_ISL_8769462, EPI_ISL_8769569, EPI_ISL_8769627, EPI_ISL_8769628, EPI_ISL_8769648, EPI_ISL_8769680, EPI_ISL_8769788, EPI_ISL_8769790, EPI_ISL_8769824, EPI_ISL_8769847, EPI_ISL_8769857, EPI_ISL_8769918, EPI_ISL_8770192, EPI_ISL_8781930, EPI_ISL_8781931, EPI_ISL_8784470, EPI_ISL_8784572, EPI_ISL_8790835, EPI_ISL_8790931, EPI_ISL_8792919, EPI_ISL_8799640, EPI_ISL_8800910, EPI_ISL_8800997, EPI_ISL_8801344, EPI_ISL_8801345, EPI_ISL_8802474, EPI_ISL_8802508, EPI_ISL_8804682, EPI_ISL_8809124, EPI_ISL_8809149, EPI_ISL_8809538, EPI_ISL_8809542, EPI_ISL_8809554, EPI_ISL_8809566, EPI_ISL_8809593, EPI_ISL_8809600, EPI_ISL_8809623, EPI_ISL_8809629, EPI_ISL_8809822, EPI_ISL_8809828, EPI_ISL_8809841, EPI_ISL_8809843, EPI_ISL_8809845, EPI_ISL_8809848, EPI_ISL_8809852, EPI_ISL_8809853, EPI_ISL_8809855, EPI_ISL_8809857, EPI_ISL_8809858, EPI_ISL_8809859, EPI_ISL_8809861, EPI_ISL_8809862, EPI_ISL_8809864, EPI_ISL_8809897, EPI_ISL_8809906, EPI_ISL_8809914, EPI_ISL_8809918, EPI_ISL_8809920, EPI_ISL_8809921, EPI_ISL_8809922, EPI_ISL_8809925, EPI_ISL_8809929, EPI_ISL_8809933, EPI_ISL_8809939, EPI_ISL_8809941, EPI_ISL_8809973, EPI_ISL_8809974, EPI_ISL_8809978, EPI_ISL_8809989, EPI_ISL_8809995, EPI_ISL_8814315, EPI_ISL_8814384, EPI_ISL_8814429, EPI_ISL_8814445, EPI_ISL_8814732, EPI_ISL_8815416, EPI_ISL_8815595, EPI_ISL_8815638, EPI_ISL_8815659, EPI_ISL_8818869, EPI_ISL_8819629, EPI_ISL_8825863, EPI_ISL_8826031, EPI_ISL_8826840, EPI_ISL_8827045, EPI_ISL_8827050, EPI_ISL_8827103, EPI_ISL_8827128, EPI_ISL_8827454, EPI_ISL_8827488, EPI_ISL_8827489, EPI_ISL_8827490, EPI_ISL_8827491, EPI_ISL_8827728, EPI_ISL_8827959, EPI_ISL_8827960, EPI_ISL_8828192, EPI_ISL_8828260, EPI_ISL_8829541, EPI_ISL_8829676, EPI_ISL_8829684, EPI_ISL_8830413, EPI_ISL_8830421, EPI_ISL_8830422, EPI_ISL_8830425, EPI_ISL_8830429, EPI_ISL_8830757, EPI_ISL_8830989, EPI_ISL_8831792, EPI_ISL_8831848, EPI_ISL_8831882, EPI_ISL_8834694, EPI_ISL_8835906, EPI_ISL_8835936, EPI_ISL_8837359, EPI_ISL_8837952, EPI_ISL_8841708, EPI_ISL_8843285, EPI_ISL_8851373, EPI_ISL_8867091, EPI_ISL_8867332, EPI_ISL_8869019, EPI_ISL_8869364, EPI_ISL_8870629, EPI_ISL_8870728, EPI_ISL_8871465, EPI_ISL_8871471, EPI_ISL_8872057, EPI_ISL_8872528, EPI_ISL_8876645, EPI_ISL_8876658, EPI_ISL_8877091, EPI_ISL_8878410, EPI_ISL_8881555, EPI_ISL_8881557, EPI_ISL_8881656, EPI_ISL_8881657, EPI_ISL_8881665, EPI_ISL_8881732, EPI_ISL_8881761, EPI_ISL_8881764, EPI_ISL_8881774, EPI_ISL_8881783, EPI_ISL_8881789, EPI_ISL_8881790, EPI_ISL_8881792, EPI_ISL_8881795, EPI_ISL_8881796, EPI_ISL_8881802, EPI_ISL_8881803, EPI_ISL_8881825, EPI_ISL_8881826, EPI_ISL_8881828, EPI_ISL_8881832, EPI_ISL_8881834, EPI_ISL_8881836, EPI_ISL_8881845, EPI_ISL_8881846, EPI_ISL_8881850, EPI_ISL_8881852, EPI_ISL_8881854, EPI_ISL_8885399, EPI_ISL_8887525, EPI_ISL_8887824, EPI_ISL_8892298, EPI_ISL_8897108, EPI_ISL_8898058, EPI_ISL_8901948, EPI_ISL_8901957, EPI_ISL_8911076, EPI_ISL_8917553, EPI_ISL_8917567, EPI_ISL_8917574, EPI_ISL_8917631, EPI_ISL_8917683, EPI_ISL_8917706, EPI_ISL_8917711, EPI_ISL_8917729, EPI_ISL_8917754, EPI_ISL_8917822, EPI_ISL_8917831, EPI_ISL_8921555, EPI_ISL_8921859, EPI_ISL_8921872, EPI_ISL_8921892, EPI_ISL_8921935, EPI_ISL_8922141, EPI_ISL_8934076, EPI_ISL_8940932, EPI_ISL_8941919, EPI_ISL_8942426, EPI_ISL_8942952, EPI_ISL_8943696, EPI_ISL_8944656, EPI_ISL_8945151, EPI_ISL_8945557, EPI_ISL_8946014, EPI_ISL_8947563, EPI_ISL_8949486, EPI_ISL_8950554, EPI_ISL_8950655, EPI_ISL_8950704, EPI_ISL_8950735, EPI_ISL_8951104, EPI_ISL_8951359, EPI_ISL_8957606, EPI_ISL_8960442, EPI_ISL_8962449, EPI_ISL_8974204, EPI_ISL_8975553, EPI_ISL_8976190, EPI_ISL_8978128, EPI_ISL_8978129, EPI_ISL_8984121, EPI_ISL_8990258, EPI_ISL_8991619")
EPI_9mil_10mil = stringlist_to_set("EPI_ISL_9005352, EPI_ISL_9005356, EPI_ISL_9009178, EPI_ISL_9010456, EPI_ISL_9011268, EPI_ISL_9011275, EPI_ISL_9011283, EPI_ISL_9011289, EPI_ISL_9011290, EPI_ISL_9011298, EPI_ISL_9011310, EPI_ISL_9012199, EPI_ISL_9012200, EPI_ISL_9012288, EPI_ISL_9012305, EPI_ISL_9013737, EPI_ISL_9025020, EPI_ISL_9028109, EPI_ISL_9029649, EPI_ISL_9029749, EPI_ISL_9030071, EPI_ISL_9031908, EPI_ISL_9034666, EPI_ISL_9034671, EPI_ISL_9034712, EPI_ISL_9034715, EPI_ISL_9035097, EPI_ISL_9036190, EPI_ISL_9036685, EPI_ISL_9036935, EPI_ISL_9037616, EPI_ISL_9037649, EPI_ISL_9038072, EPI_ISL_9040293, EPI_ISL_9041163, EPI_ISL_9044369, EPI_ISL_9044458, EPI_ISL_9044535, EPI_ISL_9044608, EPI_ISL_9045074, EPI_ISL_9045075, EPI_ISL_9045087, EPI_ISL_9045225, EPI_ISL_9046021, EPI_ISL_9046155, EPI_ISL_9046511, EPI_ISL_9046768, EPI_ISL_9047380, EPI_ISL_9047385, EPI_ISL_9047396, EPI_ISL_9047413, EPI_ISL_9047424, EPI_ISL_9047438, EPI_ISL_9047479, EPI_ISL_9047560, EPI_ISL_9047929, EPI_ISL_9048577, EPI_ISL_9048645, EPI_ISL_9048813, EPI_ISL_9048832, EPI_ISL_9049061, EPI_ISL_9049124, EPI_ISL_9051207, EPI_ISL_9053050, EPI_ISL_9053093, EPI_ISL_9058521, EPI_ISL_9061430, EPI_ISL_9063947, EPI_ISL_9064017, EPI_ISL_9064261, EPI_ISL_9065596, EPI_ISL_9066315, EPI_ISL_9068610, EPI_ISL_9074323, EPI_ISL_9074392, EPI_ISL_9074669, EPI_ISL_9075663, EPI_ISL_9076234, EPI_ISL_9078947, EPI_ISL_9080382, EPI_ISL_9086997, EPI_ISL_9087004, EPI_ISL_9087015, EPI_ISL_9087019, EPI_ISL_9090751, EPI_ISL_9090837, EPI_ISL_9090929, EPI_ISL_9090961, EPI_ISL_9090962, EPI_ISL_9090967, EPI_ISL_9090980, EPI_ISL_9090984, EPI_ISL_9090991, EPI_ISL_9090992, EPI_ISL_9090993, EPI_ISL_9090995, EPI_ISL_9091000, EPI_ISL_9091036, EPI_ISL_9091066, EPI_ISL_9091067, EPI_ISL_9091077, EPI_ISL_9091088, EPI_ISL_9091091, EPI_ISL_9091097, EPI_ISL_9091117, EPI_ISL_9091195, EPI_ISL_9091242, EPI_ISL_9092486, EPI_ISL_9092543, EPI_ISL_9093024, EPI_ISL_9093324, EPI_ISL_9100643, EPI_ISL_9100660, EPI_ISL_9100671, EPI_ISL_9100690, EPI_ISL_9103139, EPI_ISL_9104320, EPI_ISL_9108186, EPI_ISL_9109762, EPI_ISL_9109898, EPI_ISL_9110303, EPI_ISL_9120288, EPI_ISL_9121666, EPI_ISL_9121730, EPI_ISL_9126398, EPI_ISL_9129541, EPI_ISL_9131755, EPI_ISL_9133265, EPI_ISL_9133362, EPI_ISL_9133533, EPI_ISL_9133534, EPI_ISL_9133560, EPI_ISL_9133651, EPI_ISL_9133676, EPI_ISL_9133770, EPI_ISL_9133811, EPI_ISL_9133912, EPI_ISL_9134047, EPI_ISL_9134101, EPI_ISL_9134165, EPI_ISL_9134203, EPI_ISL_9134204, EPI_ISL_9134714, EPI_ISL_9135550, EPI_ISL_9135558, EPI_ISL_9135571, EPI_ISL_9139305, EPI_ISL_9139589, EPI_ISL_9139682, EPI_ISL_9139789, EPI_ISL_9139793, EPI_ISL_9139806, EPI_ISL_9139841, EPI_ISL_9139906, EPI_ISL_9139953, EPI_ISL_9146287, EPI_ISL_9147675, EPI_ISL_9151196, EPI_ISL_9151405, EPI_ISL_9151417, EPI_ISL_9151754, EPI_ISL_9152643, EPI_ISL_9156804, EPI_ISL_9157476, EPI_ISL_9163218, EPI_ISL_9166254, EPI_ISL_9168068, EPI_ISL_9169546, EPI_ISL_9169574, EPI_ISL_9169859, EPI_ISL_9170032, EPI_ISL_9172208, EPI_ISL_9176038, EPI_ISL_9180127, EPI_ISL_9182632, EPI_ISL_9182645, EPI_ISL_9182662, EPI_ISL_9187834, EPI_ISL_9187843, EPI_ISL_9187872, EPI_ISL_9187891, EPI_ISL_9191078, EPI_ISL_9191147, EPI_ISL_9191309, EPI_ISL_9196222, EPI_ISL_9196330, EPI_ISL_9196339, EPI_ISL_9196351, EPI_ISL_9196358, EPI_ISL_9196364, EPI_ISL_9196485, EPI_ISL_9196486, EPI_ISL_9196487, EPI_ISL_9196489, EPI_ISL_9196490, EPI_ISL_9196491, EPI_ISL_9196493, EPI_ISL_9196494, EPI_ISL_9196495, EPI_ISL_9196496, EPI_ISL_9196497, EPI_ISL_9196498, EPI_ISL_9196499, EPI_ISL_9196500, EPI_ISL_9196502, EPI_ISL_9196508, EPI_ISL_9197135, EPI_ISL_9197952, EPI_ISL_9198020, EPI_ISL_9198230, EPI_ISL_9198767, EPI_ISL_9199183, EPI_ISL_9200048, EPI_ISL_9200866, EPI_ISL_9202367, EPI_ISL_9202404, EPI_ISL_9202411, EPI_ISL_9202498, EPI_ISL_9202934, EPI_ISL_9202999, EPI_ISL_9204848, EPI_ISL_9205605, EPI_ISL_9206643, EPI_ISL_9210610, EPI_ISL_9210834, EPI_ISL_9210948, EPI_ISL_9211469, EPI_ISL_9214019, EPI_ISL_9214153, EPI_ISL_9217742, EPI_ISL_9218276, EPI_ISL_9227643, EPI_ISL_9227757, EPI_ISL_9229700, EPI_ISL_9229876, EPI_ISL_9230962, EPI_ISL_9232065, EPI_ISL_9232175, EPI_ISL_9232315, EPI_ISL_9232396, EPI_ISL_9232404, EPI_ISL_9232441, EPI_ISL_9232442, EPI_ISL_9233990, EPI_ISL_9237445, EPI_ISL_9244079, EPI_ISL_9244229, EPI_ISL_9244241, EPI_ISL_9244346, EPI_ISL_9244347, EPI_ISL_9244349, EPI_ISL_9244352, EPI_ISL_9244354, EPI_ISL_9244356, EPI_ISL_9244361, EPI_ISL_9244362, EPI_ISL_9244364, EPI_ISL_9244365, EPI_ISL_9244366, EPI_ISL_9244367, EPI_ISL_9244368, EPI_ISL_9244371, EPI_ISL_9244393, EPI_ISL_9244407, EPI_ISL_9244414, EPI_ISL_9244430, EPI_ISL_9244436, EPI_ISL_9244438, EPI_ISL_9244443, EPI_ISL_9257372, EPI_ISL_9263448, EPI_ISL_9268007, EPI_ISL_9272940, EPI_ISL_9273139, EPI_ISL_9275801, EPI_ISL_9287159, EPI_ISL_9301892, EPI_ISL_9303894, EPI_ISL_9320000, EPI_ISL_9320048, EPI_ISL_9320701, EPI_ISL_9322799, EPI_ISL_9323194, EPI_ISL_9324184, EPI_ISL_9324261, EPI_ISL_9324445, EPI_ISL_9325956, EPI_ISL_9329574, EPI_ISL_9329597, EPI_ISL_9329613, EPI_ISL_9329709, EPI_ISL_9329949, EPI_ISL_9332338, EPI_ISL_9340194, EPI_ISL_9346264, EPI_ISL_9357293, EPI_ISL_9361245, EPI_ISL_9366656, EPI_ISL_9366674, EPI_ISL_9366695, EPI_ISL_9366701, EPI_ISL_9366748, EPI_ISL_9369408, EPI_ISL_9369483, EPI_ISL_9369654, EPI_ISL_9369696, EPI_ISL_9369748, EPI_ISL_9369840, EPI_ISL_9374568, EPI_ISL_9375639, EPI_ISL_9375644, EPI_ISL_9393238, EPI_ISL_9393371, EPI_ISL_9393552, EPI_ISL_9393561, EPI_ISL_9393590, EPI_ISL_9393601, EPI_ISL_9393617, EPI_ISL_9393658, EPI_ISL_9397486, EPI_ISL_9397545, EPI_ISL_9397580, EPI_ISL_9397591, EPI_ISL_9402130, EPI_ISL_9402568, EPI_ISL_9402729, EPI_ISL_9402868, EPI_ISL_9405606, EPI_ISL_9406530, EPI_ISL_9406661, EPI_ISL_9406745, EPI_ISL_9407327, EPI_ISL_9408663, EPI_ISL_9408966, EPI_ISL_9408994, EPI_ISL_9409022, EPI_ISL_9409061, EPI_ISL_9409074, EPI_ISL_9409098, EPI_ISL_9409875, EPI_ISL_9409936, EPI_ISL_9410965, EPI_ISL_9412364, EPI_ISL_9416349, EPI_ISL_9416383, EPI_ISL_9416496, EPI_ISL_9416811, EPI_ISL_9417829, EPI_ISL_9417856, EPI_ISL_9418351, EPI_ISL_9429907, EPI_ISL_9429910, EPI_ISL_9429914, EPI_ISL_9429916, EPI_ISL_9429917, EPI_ISL_9429928, EPI_ISL_9429953, EPI_ISL_9430010, EPI_ISL_9430074, EPI_ISL_9430075, EPI_ISL_9430083, EPI_ISL_9430088, EPI_ISL_9430093, EPI_ISL_9430095, EPI_ISL_9430097, EPI_ISL_9430100, EPI_ISL_9430104, EPI_ISL_9430105, EPI_ISL_9430142, EPI_ISL_9430149, EPI_ISL_9430156, EPI_ISL_9430807, EPI_ISL_9433224, EPI_ISL_9436219, EPI_ISL_9436655, EPI_ISL_9436667, EPI_ISL_9436800, EPI_ISL_9436819, EPI_ISL_9439014, EPI_ISL_9443344, EPI_ISL_9445585, EPI_ISL_9445587, EPI_ISL_9448294, EPI_ISL_9448303, EPI_ISL_9450747, EPI_ISL_9451725, EPI_ISL_9454967, EPI_ISL_9455817, EPI_ISL_9455959, EPI_ISL_9456039, EPI_ISL_9456611, EPI_ISL_9457664, EPI_ISL_9457860, EPI_ISL_9458247, EPI_ISL_9459219, EPI_ISL_9460511, EPI_ISL_9465685, EPI_ISL_9466125, EPI_ISL_9466330, EPI_ISL_9466376, EPI_ISL_9466488, EPI_ISL_9466563, EPI_ISL_9466712, EPI_ISL_9466730, EPI_ISL_9466742, EPI_ISL_9466762, EPI_ISL_9466822, EPI_ISL_9466824, EPI_ISL_9466827, EPI_ISL_9467584, EPI_ISL_9468165, EPI_ISL_9471815, EPI_ISL_9471816, EPI_ISL_9471828, EPI_ISL_9471831, EPI_ISL_9471852, EPI_ISL_9471897, EPI_ISL_9471996, EPI_ISL_9471999, EPI_ISL_9472306, EPI_ISL_9472308, EPI_ISL_9472322, EPI_ISL_9472338, EPI_ISL_9472385, EPI_ISL_9472416, EPI_ISL_9472426, EPI_ISL_9472506, EPI_ISL_9472507, EPI_ISL_9472531, EPI_ISL_9472535, EPI_ISL_9475129, EPI_ISL_9487490, EPI_ISL_9487619, EPI_ISL_9487621, EPI_ISL_9487630, EPI_ISL_9487635, EPI_ISL_9487640, EPI_ISL_9487643, EPI_ISL_9487651, EPI_ISL_9487652, EPI_ISL_9487655, EPI_ISL_9487657, EPI_ISL_9487660, EPI_ISL_9487662, EPI_ISL_9487676, EPI_ISL_9487692, EPI_ISL_9487693, EPI_ISL_9487694, EPI_ISL_9487703, EPI_ISL_9487706, EPI_ISL_9487716, EPI_ISL_9487722, EPI_ISL_9487739, EPI_ISL_9487741, EPI_ISL_9487742, EPI_ISL_9487743, EPI_ISL_9487750, EPI_ISL_9487753, EPI_ISL_9487768, EPI_ISL_9487787, EPI_ISL_9487793, EPI_ISL_9487803, EPI_ISL_9487831, EPI_ISL_9487857, EPI_ISL_9487898, EPI_ISL_9487928, EPI_ISL_9487970, EPI_ISL_9487982, EPI_ISL_9487983, EPI_ISL_9488006, EPI_ISL_9488029, EPI_ISL_9488474, EPI_ISL_9488477, EPI_ISL_9488501, EPI_ISL_9488515, EPI_ISL_9488519, EPI_ISL_9488524, EPI_ISL_9488545, EPI_ISL_9488570, EPI_ISL_9488575, EPI_ISL_9488581, EPI_ISL_9488619, EPI_ISL_9488636, EPI_ISL_9488661, EPI_ISL_9488677, EPI_ISL_9488757, EPI_ISL_9488782, EPI_ISL_9488784, EPI_ISL_9488805, EPI_ISL_9488811, EPI_ISL_9488829, EPI_ISL_9488834, EPI_ISL_9489247, EPI_ISL_9489252, EPI_ISL_9497113, EPI_ISL_9504005, EPI_ISL_9504259, EPI_ISL_9510032, EPI_ISL_9510046, EPI_ISL_9511693, EPI_ISL_9511841, EPI_ISL_9511897, EPI_ISL_9511947, EPI_ISL_9511949, EPI_ISL_9511951, EPI_ISL_9512035, EPI_ISL_9512183, EPI_ISL_9512188, EPI_ISL_9512193, EPI_ISL_9512195, EPI_ISL_9512200, EPI_ISL_9512213, EPI_ISL_9512376, EPI_ISL_9519799, EPI_ISL_9523248, EPI_ISL_9532527, EPI_ISL_9533075, EPI_ISL_9533563, EPI_ISL_9533609, EPI_ISL_9549192, EPI_ISL_9549456, EPI_ISL_9561760, EPI_ISL_9569515, EPI_ISL_9569535, EPI_ISL_9572644, EPI_ISL_9573034, EPI_ISL_9573172, EPI_ISL_9587585, EPI_ISL_9587875, EPI_ISL_9589057, EPI_ISL_9594728, EPI_ISL_9594734, EPI_ISL_9594736, EPI_ISL_9594746, EPI_ISL_9594748, EPI_ISL_9594790, EPI_ISL_9594793, EPI_ISL_9594813, EPI_ISL_9594816, EPI_ISL_9594822, EPI_ISL_9594823, EPI_ISL_9594825, EPI_ISL_9596778, EPI_ISL_9596984, EPI_ISL_9601628, EPI_ISL_9601700, EPI_ISL_9601744, EPI_ISL_9604897, EPI_ISL_9609009, EPI_ISL_9609069, EPI_ISL_9610381, EPI_ISL_9610390, EPI_ISL_9610393, EPI_ISL_9610402, EPI_ISL_9613674, EPI_ISL_9614658, EPI_ISL_9617758, EPI_ISL_9620216, EPI_ISL_9623090, EPI_ISL_9625693, EPI_ISL_9625727, EPI_ISL_9625751, EPI_ISL_9625759, EPI_ISL_9625770, EPI_ISL_9625891, EPI_ISL_9636300, EPI_ISL_9637481, EPI_ISL_9640020, EPI_ISL_9640740, EPI_ISL_9642609, EPI_ISL_9647293, EPI_ISL_9650799, EPI_ISL_9651096, EPI_ISL_9651353, EPI_ISL_9653335, EPI_ISL_9656370, EPI_ISL_9666874, EPI_ISL_9668614, EPI_ISL_9673566, EPI_ISL_9674935, EPI_ISL_9678134, EPI_ISL_9678135, EPI_ISL_9679129, EPI_ISL_9679144, EPI_ISL_9679158, EPI_ISL_9679269, EPI_ISL_9679403, EPI_ISL_9683434, EPI_ISL_9683447, EPI_ISL_9683466, EPI_ISL_9683467, EPI_ISL_9683478, EPI_ISL_9683483, EPI_ISL_9683486, EPI_ISL_9683487, EPI_ISL_9689797, EPI_ISL_9691942, EPI_ISL_9695242, EPI_ISL_9695386, EPI_ISL_9696667, EPI_ISL_9696682, EPI_ISL_9696732, EPI_ISL_9696744, EPI_ISL_9696756, EPI_ISL_9696794, EPI_ISL_9696809, EPI_ISL_9697283, EPI_ISL_9697342, EPI_ISL_9697401, EPI_ISL_9697409, EPI_ISL_9697412, EPI_ISL_9697419, EPI_ISL_9697420, EPI_ISL_9697622, EPI_ISL_9697669, EPI_ISL_9697674, EPI_ISL_9697678, EPI_ISL_9697695, EPI_ISL_9697740, EPI_ISL_9697742, EPI_ISL_9697750, EPI_ISL_9697760, EPI_ISL_9697779, EPI_ISL_9697780, EPI_ISL_9697786, EPI_ISL_9697797, EPI_ISL_9697801, EPI_ISL_9697802, EPI_ISL_9697803, EPI_ISL_9697810, EPI_ISL_9697813, EPI_ISL_9697815, EPI_ISL_9697816, EPI_ISL_9702292, EPI_ISL_9705038, EPI_ISL_9705152, EPI_ISL_9707770, EPI_ISL_9709081, EPI_ISL_9709115, EPI_ISL_9709126, EPI_ISL_9709128, EPI_ISL_9709139, EPI_ISL_9713712, EPI_ISL_9714143, EPI_ISL_9714174, EPI_ISL_9715503, EPI_ISL_9715636, EPI_ISL_9715645, EPI_ISL_9721311, EPI_ISL_9721351, EPI_ISL_9721359, EPI_ISL_9721365, EPI_ISL_9721370, EPI_ISL_9721377, EPI_ISL_9721381, EPI_ISL_9721390, EPI_ISL_9727117, EPI_ISL_9727142, EPI_ISL_9727150, EPI_ISL_9727169, EPI_ISL_9727186, EPI_ISL_9727194, EPI_ISL_9727223, EPI_ISL_9727268, EPI_ISL_9728278, EPI_ISL_9729033, EPI_ISL_9729063, EPI_ISL_9729127, EPI_ISL_9729128, EPI_ISL_9736232, EPI_ISL_9743775, EPI_ISL_9746931, EPI_ISL_9754259, EPI_ISL_9754610, EPI_ISL_9770596, EPI_ISL_9770856, EPI_ISL_9770870, EPI_ISL_9771473, EPI_ISL_9773064, EPI_ISL_9775562, EPI_ISL_9781644, EPI_ISL_9781649, EPI_ISL_9781675, EPI_ISL_9781690, EPI_ISL_9783712, EPI_ISL_9788022, EPI_ISL_9791141, EPI_ISL_9792573, EPI_ISL_9806761, EPI_ISL_9806777, EPI_ISL_9806780, EPI_ISL_9806795, EPI_ISL_9806825, EPI_ISL_9806831, EPI_ISL_9806849, EPI_ISL_9806890, EPI_ISL_9806897, EPI_ISL_9806909, EPI_ISL_9806941, EPI_ISL_9806943, EPI_ISL_9806949, EPI_ISL_9806960, EPI_ISL_9806977, EPI_ISL_9806985, EPI_ISL_9806992, EPI_ISL_9806995, EPI_ISL_9808141, EPI_ISL_9808146, EPI_ISL_9808159, EPI_ISL_9808311, EPI_ISL_9808318, EPI_ISL_9813896, EPI_ISL_9814061, EPI_ISL_9817504, EPI_ISL_9818106, EPI_ISL_9818145, EPI_ISL_9831433, EPI_ISL_9831863, EPI_ISL_9832011, EPI_ISL_9832049, EPI_ISL_9832223, EPI_ISL_9832265, EPI_ISL_9832266, EPI_ISL_9833929, EPI_ISL_9835767, EPI_ISL_9835963, EPI_ISL_9837460, EPI_ISL_9837718, EPI_ISL_9837719, EPI_ISL_9843631, EPI_ISL_9844747, EPI_ISL_9844816, EPI_ISL_9844820, EPI_ISL_9849476, EPI_ISL_9849487, EPI_ISL_9849501, EPI_ISL_9854654, EPI_ISL_9854923, EPI_ISL_9863925, EPI_ISL_9873050, EPI_ISL_9873051, EPI_ISL_9873829, EPI_ISL_9873872, EPI_ISL_9873933, EPI_ISL_9873964, EPI_ISL_9877792, EPI_ISL_9880735, EPI_ISL_9882111, EPI_ISL_9884555, EPI_ISL_9886029, EPI_ISL_9886100, EPI_ISL_9911424, EPI_ISL_9911578, EPI_ISL_9919496, EPI_ISL_9925948, EPI_ISL_9925959, EPI_ISL_9925965, EPI_ISL_9925989, EPI_ISL_9925993, EPI_ISL_9925995, EPI_ISL_9925997, EPI_ISL_9925999, EPI_ISL_9926001, EPI_ISL_9926003, EPI_ISL_9926004, EPI_ISL_9926007, EPI_ISL_9926010, EPI_ISL_9926017, EPI_ISL_9926019, EPI_ISL_9926022, EPI_ISL_9926024, EPI_ISL_9926030, EPI_ISL_9926058, EPI_ISL_9926078, EPI_ISL_9926099, EPI_ISL_9926112, EPI_ISL_9926118, EPI_ISL_9926127, EPI_ISL_9926154, EPI_ISL_9926162, EPI_ISL_9926327, EPI_ISL_9926334, EPI_ISL_9940202, EPI_ISL_9961706, EPI_ISL_9961927, EPI_ISL_9963072, EPI_ISL_9966882, EPI_ISL_9970992, EPI_ISL_9979109, EPI_ISL_9983756, EPI_ISL_9983883, EPI_ISL_9987524, EPI_ISL_9994180")
EPI_10mil_11mil = stringlist_to_set("EPI_ISL_10005959, EPI_ISL_10006826, EPI_ISL_10008204, EPI_ISL_10008459, EPI_ISL_10008460, EPI_ISL_10008465, EPI_ISL_10009329, EPI_ISL_10011197, EPI_ISL_10013260, EPI_ISL_10015927, EPI_ISL_10021110, EPI_ISL_10022743, EPI_ISL_10023162, EPI_ISL_10023287, EPI_ISL_10023339, EPI_ISL_10025493, EPI_ISL_10026752, EPI_ISL_10026977, EPI_ISL_10027022, EPI_ISL_10027035, EPI_ISL_10027100, EPI_ISL_10027276, EPI_ISL_10028336, EPI_ISL_10029029, EPI_ISL_10034133, EPI_ISL_10034481, EPI_ISL_10034593, EPI_ISL_10034637, EPI_ISL_10045301, EPI_ISL_10058780, EPI_ISL_10058783, EPI_ISL_10063882, EPI_ISL_10065608, EPI_ISL_10065768, EPI_ISL_10065974, EPI_ISL_10065986, EPI_ISL_10065987, EPI_ISL_10065994, EPI_ISL_10066000, EPI_ISL_10066001, EPI_ISL_10066002, EPI_ISL_10066010, EPI_ISL_10066015, EPI_ISL_10066022, EPI_ISL_10066023, EPI_ISL_10066029, EPI_ISL_10066034, EPI_ISL_10066036, EPI_ISL_10066048, EPI_ISL_10066050, EPI_ISL_10066056, EPI_ISL_10066057, EPI_ISL_10066058, EPI_ISL_10066059, EPI_ISL_10066061, EPI_ISL_10066062, EPI_ISL_10066068, EPI_ISL_10066070, EPI_ISL_10066073, EPI_ISL_10066084, EPI_ISL_10066090, EPI_ISL_10066091, EPI_ISL_10066093, EPI_ISL_10066102, EPI_ISL_10066130, EPI_ISL_10066134, EPI_ISL_10066137, EPI_ISL_10066150, EPI_ISL_10066153, EPI_ISL_10066164, EPI_ISL_10066168, EPI_ISL_10066186, EPI_ISL_10066213, EPI_ISL_10066243, EPI_ISL_10066246, EPI_ISL_10066254, EPI_ISL_10066267, EPI_ISL_10072285, EPI_ISL_10072674, EPI_ISL_10072698, EPI_ISL_10073148, EPI_ISL_10074087, EPI_ISL_10074702, EPI_ISL_10074707, EPI_ISL_10077456, EPI_ISL_10077457, EPI_ISL_10077740, EPI_ISL_10077789, EPI_ISL_10077805, EPI_ISL_10077939, EPI_ISL_10077958, EPI_ISL_10077960, EPI_ISL_10077963, EPI_ISL_10077965, EPI_ISL_10077966, EPI_ISL_10077980, EPI_ISL_10077987, EPI_ISL_10078149, EPI_ISL_10078155, EPI_ISL_10078205, EPI_ISL_10078216, EPI_ISL_10078218, EPI_ISL_10078365, EPI_ISL_10078399, EPI_ISL_10078400, EPI_ISL_10078405, EPI_ISL_10078406, EPI_ISL_10078749, EPI_ISL_10078881, EPI_ISL_10078882, EPI_ISL_10078977, EPI_ISL_10079008, EPI_ISL_10079124, EPI_ISL_10079482, EPI_ISL_10079485, EPI_ISL_10079730, EPI_ISL_10080088, EPI_ISL_10080659, EPI_ISL_10081351, EPI_ISL_10081394, EPI_ISL_10081402, EPI_ISL_10081662, EPI_ISL_10081729, EPI_ISL_10081881, EPI_ISL_10081886, EPI_ISL_10082052, EPI_ISL_10082630, EPI_ISL_10082989, EPI_ISL_10083756, EPI_ISL_10083777, EPI_ISL_10083787, EPI_ISL_10083812, EPI_ISL_10083866, EPI_ISL_10083944, EPI_ISL_10083960, EPI_ISL_10083972, EPI_ISL_10083979, EPI_ISL_10083981, EPI_ISL_10083985, EPI_ISL_10083988, EPI_ISL_10084004, EPI_ISL_10084007, EPI_ISL_10084008, EPI_ISL_10084093, EPI_ISL_10085377, EPI_ISL_10087734, EPI_ISL_10087752, EPI_ISL_10092537, EPI_ISL_10093299, EPI_ISL_10093694, EPI_ISL_10101118, EPI_ISL_10101120, EPI_ISL_10101124, EPI_ISL_10121115, EPI_ISL_10121312, EPI_ISL_10121522, EPI_ISL_10123102, EPI_ISL_10123125, EPI_ISL_10123126, EPI_ISL_10123127, EPI_ISL_10123136, EPI_ISL_10123138, EPI_ISL_10123144, EPI_ISL_10123149, EPI_ISL_10123151, EPI_ISL_10123184, EPI_ISL_10123237, EPI_ISL_10123242, EPI_ISL_10123246, EPI_ISL_10123256, EPI_ISL_10123278, EPI_ISL_10129960, EPI_ISL_10138346, EPI_ISL_10138347, EPI_ISL_10138351, EPI_ISL_10138509, EPI_ISL_10149308, EPI_ISL_10149344, EPI_ISL_10149570, EPI_ISL_10173852, EPI_ISL_10174207, EPI_ISL_10174960, EPI_ISL_10175390, EPI_ISL_10182087, EPI_ISL_10182644, EPI_ISL_10182902, EPI_ISL_10184975, EPI_ISL_10185121, EPI_ISL_10185137, EPI_ISL_10185347, EPI_ISL_10188771, EPI_ISL_10188772, EPI_ISL_10188801, EPI_ISL_10188808, EPI_ISL_10188819, EPI_ISL_10188829, EPI_ISL_10192722, EPI_ISL_10192735, EPI_ISL_10198777, EPI_ISL_10204977, EPI_ISL_10204983, EPI_ISL_10204986, EPI_ISL_10204992, EPI_ISL_10205004, EPI_ISL_10205039, EPI_ISL_10205073, EPI_ISL_10205094, EPI_ISL_10205098, EPI_ISL_10205106, EPI_ISL_10205142, EPI_ISL_10205178, EPI_ISL_10205180, EPI_ISL_10205182, EPI_ISL_10205253, EPI_ISL_10205289, EPI_ISL_10205338, EPI_ISL_10205375, EPI_ISL_10207392, EPI_ISL_10207536, EPI_ISL_10207539, EPI_ISL_10207777, EPI_ISL_10210308, EPI_ISL_10210437, EPI_ISL_10210486, EPI_ISL_10212487, EPI_ISL_10212488, EPI_ISL_10213774, EPI_ISL_10213775, EPI_ISL_10214094, EPI_ISL_10214108, EPI_ISL_10214427, EPI_ISL_10214452, EPI_ISL_10214463, EPI_ISL_10220260, EPI_ISL_10224838, EPI_ISL_10246896, EPI_ISL_10246913, EPI_ISL_10246929, EPI_ISL_10247077, EPI_ISL_10247741, EPI_ISL_10248161, EPI_ISL_10251088, EPI_ISL_10251304, EPI_ISL_10255340, EPI_ISL_10255376, EPI_ISL_10283352, EPI_ISL_10283711, EPI_ISL_10283724, EPI_ISL_10290459, EPI_ISL_10293291, EPI_ISL_10293513, EPI_ISL_10295769, EPI_ISL_10296275, EPI_ISL_10298437, EPI_ISL_10298438, EPI_ISL_10298464, EPI_ISL_10298491, EPI_ISL_10298494, EPI_ISL_10300795, EPI_ISL_10306024, EPI_ISL_10306120, EPI_ISL_10306134, EPI_ISL_10306140, EPI_ISL_10306148, EPI_ISL_10306161, EPI_ISL_10306164, EPI_ISL_10306559, EPI_ISL_10306869, EPI_ISL_10309965, EPI_ISL_10311702, EPI_ISL_10311711, EPI_ISL_10311753, EPI_ISL_10312030, EPI_ISL_10312031, EPI_ISL_10312032, EPI_ISL_10312033, EPI_ISL_10312042, EPI_ISL_10312044, EPI_ISL_10312052, EPI_ISL_10312056, EPI_ISL_10312057, EPI_ISL_10312059, EPI_ISL_10312060, EPI_ISL_10312064, EPI_ISL_10312065, EPI_ISL_10312066, EPI_ISL_10312081, EPI_ISL_10312085, EPI_ISL_10312086, EPI_ISL_10312087, EPI_ISL_10312094, EPI_ISL_10312096, EPI_ISL_10312370, EPI_ISL_10312525, EPI_ISL_10312548, EPI_ISL_10312553, EPI_ISL_10312557, EPI_ISL_10312559, EPI_ISL_10312565, EPI_ISL_10312568, EPI_ISL_10312573, EPI_ISL_10312582, EPI_ISL_10312584, EPI_ISL_10312585, EPI_ISL_10312592, EPI_ISL_10312596, EPI_ISL_10312597, EPI_ISL_10312598, EPI_ISL_10312615, EPI_ISL_10312617, EPI_ISL_10312618, EPI_ISL_10312623, EPI_ISL_10312624, EPI_ISL_10312626, EPI_ISL_10312627, EPI_ISL_10312635, EPI_ISL_10312638, EPI_ISL_10312754, EPI_ISL_10313149, EPI_ISL_10313150, EPI_ISL_10313183, EPI_ISL_10313845, EPI_ISL_10314883, EPI_ISL_10314886, EPI_ISL_10314887, EPI_ISL_10314899, EPI_ISL_10314904, EPI_ISL_10314914, EPI_ISL_10314915, EPI_ISL_10314916, EPI_ISL_10314924, EPI_ISL_10314930, EPI_ISL_10314931, EPI_ISL_10314940, EPI_ISL_10314941, EPI_ISL_10314942, EPI_ISL_10314943, EPI_ISL_10314944, EPI_ISL_10316983, EPI_ISL_10317379, EPI_ISL_10327362, EPI_ISL_10327716, EPI_ISL_10328262, EPI_ISL_10328772, EPI_ISL_10369251, EPI_ISL_10379783, EPI_ISL_10382204, EPI_ISL_10386226, EPI_ISL_10391525, EPI_ISL_10392765, EPI_ISL_10398645, EPI_ISL_10401810, EPI_ISL_10401837, EPI_ISL_10401856, EPI_ISL_10401865, EPI_ISL_10401868, EPI_ISL_10401984, EPI_ISL_10404067, EPI_ISL_10404103, EPI_ISL_10404131, EPI_ISL_10405495, EPI_ISL_10431884, EPI_ISL_10431934, EPI_ISL_10431989, EPI_ISL_10433657, EPI_ISL_10435634, EPI_ISL_10437641, EPI_ISL_10441356, EPI_ISL_10443386, EPI_ISL_10443404, EPI_ISL_10443409, EPI_ISL_10443411, EPI_ISL_10443425, EPI_ISL_10443430, EPI_ISL_10443433, EPI_ISL_10443434, EPI_ISL_10443436, EPI_ISL_10443439, EPI_ISL_10443440, EPI_ISL_10443441, EPI_ISL_10443444, EPI_ISL_10443446, EPI_ISL_10443447, EPI_ISL_10443448, EPI_ISL_10443450, EPI_ISL_10443452, EPI_ISL_10443454, EPI_ISL_10443456, EPI_ISL_10443458, EPI_ISL_10443460, EPI_ISL_10443461, EPI_ISL_10443466, EPI_ISL_10443469, EPI_ISL_10443471, EPI_ISL_10443490, EPI_ISL_10443494, EPI_ISL_10443495, EPI_ISL_10443503, EPI_ISL_10443505, EPI_ISL_10443506, EPI_ISL_10443507, EPI_ISL_10443510, EPI_ISL_10443511, EPI_ISL_10443512, EPI_ISL_10443516, EPI_ISL_10443517, EPI_ISL_10443518, EPI_ISL_10443519, EPI_ISL_10443520, EPI_ISL_10443523, EPI_ISL_10443525, EPI_ISL_10443526, EPI_ISL_10443533, EPI_ISL_10443535, EPI_ISL_10443563, EPI_ISL_10443572, EPI_ISL_10443573, EPI_ISL_10443575, EPI_ISL_10443577, EPI_ISL_10443584, EPI_ISL_10443592, EPI_ISL_10443595, EPI_ISL_10443599, EPI_ISL_10443601, EPI_ISL_10443603, EPI_ISL_10443605, EPI_ISL_10443606, EPI_ISL_10443608, EPI_ISL_10443611, EPI_ISL_10443613, EPI_ISL_10443615, EPI_ISL_10443617, EPI_ISL_10443623, EPI_ISL_10443627, EPI_ISL_10443632, EPI_ISL_10443636, EPI_ISL_10443647, EPI_ISL_10443648, EPI_ISL_10443649, EPI_ISL_10443656, EPI_ISL_10443658, EPI_ISL_10443661, EPI_ISL_10443671, EPI_ISL_10443674, EPI_ISL_10443675, EPI_ISL_10443678, EPI_ISL_10450198, EPI_ISL_10451247, EPI_ISL_10455028, EPI_ISL_10457041, EPI_ISL_10481797, EPI_ISL_10482336, EPI_ISL_10482850, EPI_ISL_10497475, EPI_ISL_10502293, EPI_ISL_10504027, EPI_ISL_10504064, EPI_ISL_10504107, EPI_ISL_10504108, EPI_ISL_10504167, EPI_ISL_10504187, EPI_ISL_10504203, EPI_ISL_10504205, EPI_ISL_10504239, EPI_ISL_10504258, EPI_ISL_10504269, EPI_ISL_10504319, EPI_ISL_10504339, EPI_ISL_10504345, EPI_ISL_10504347, EPI_ISL_10504381, EPI_ISL_10504387, EPI_ISL_10504420, EPI_ISL_10504441, EPI_ISL_10504471, EPI_ISL_10504485, EPI_ISL_10504647, EPI_ISL_10514956, EPI_ISL_10519096, EPI_ISL_10523937, EPI_ISL_10524685, EPI_ISL_10524790, EPI_ISL_10524808, EPI_ISL_10543999, EPI_ISL_10544042, EPI_ISL_10544102, EPI_ISL_10544144, EPI_ISL_10544474, EPI_ISL_10544507, EPI_ISL_10544984, EPI_ISL_10544986, EPI_ISL_10545423, EPI_ISL_10545452, EPI_ISL_10546178, EPI_ISL_10546476, EPI_ISL_10547155, EPI_ISL_10547219, EPI_ISL_10547244, EPI_ISL_10547258, EPI_ISL_10547266, EPI_ISL_10547268, EPI_ISL_10547296, EPI_ISL_10547299, EPI_ISL_10547314, EPI_ISL_10547344, EPI_ISL_10547683, EPI_ISL_10548055, EPI_ISL_10548071, EPI_ISL_10548077, EPI_ISL_10548443, EPI_ISL_10548461, EPI_ISL_10549286, EPI_ISL_10555394, EPI_ISL_10555425, EPI_ISL_10555459, EPI_ISL_10555540, EPI_ISL_10555586, EPI_ISL_10555615, EPI_ISL_10555636, EPI_ISL_10555668, EPI_ISL_10559721, EPI_ISL_10560171, EPI_ISL_10565822, EPI_ISL_10567103, EPI_ISL_10568889, EPI_ISL_10571268, EPI_ISL_10579550, EPI_ISL_10579638, EPI_ISL_10579782, EPI_ISL_10580591, EPI_ISL_10580610, EPI_ISL_10580791, EPI_ISL_10580798, EPI_ISL_10590704, EPI_ISL_10594394, EPI_ISL_10596188, EPI_ISL_10596244, EPI_ISL_10623440, EPI_ISL_10630173, EPI_ISL_10630208, EPI_ISL_10630286, EPI_ISL_10630294, EPI_ISL_10630387, EPI_ISL_10630426, EPI_ISL_10630509, EPI_ISL_10631057, EPI_ISL_10631087, EPI_ISL_10631441, EPI_ISL_10632161, EPI_ISL_10633951, EPI_ISL_10634024, EPI_ISL_10638400, EPI_ISL_10638438, EPI_ISL_10638461, EPI_ISL_10638464, EPI_ISL_10638466, EPI_ISL_10638483, EPI_ISL_10638490, EPI_ISL_10638493, EPI_ISL_10638507, EPI_ISL_10638513, EPI_ISL_10638522, EPI_ISL_10638528, EPI_ISL_10638529, EPI_ISL_10638534, EPI_ISL_10638544, EPI_ISL_10638548, EPI_ISL_10638569, EPI_ISL_10638681, EPI_ISL_10638682, EPI_ISL_10638684, EPI_ISL_10638687, EPI_ISL_10639921, EPI_ISL_10640459, EPI_ISL_10645365, EPI_ISL_10646183, EPI_ISL_10646223, EPI_ISL_10646224, EPI_ISL_10646625, EPI_ISL_10646659, EPI_ISL_10646660, EPI_ISL_10646664, EPI_ISL_10646678, EPI_ISL_10652475, EPI_ISL_10661114, EPI_ISL_10661119, EPI_ISL_10661136, EPI_ISL_10667120, EPI_ISL_10673551, EPI_ISL_10676563, EPI_ISL_10678674, EPI_ISL_10682237, EPI_ISL_10682241, EPI_ISL_10682298, EPI_ISL_10682367, EPI_ISL_10682374, EPI_ISL_10699409, EPI_ISL_10703587, EPI_ISL_10703923, EPI_ISL_10707346, EPI_ISL_10714913, EPI_ISL_10714914, EPI_ISL_10715650, EPI_ISL_10716749, EPI_ISL_10716750, EPI_ISL_10716751, EPI_ISL_10716752, EPI_ISL_10716753, EPI_ISL_10716754, EPI_ISL_10716755, EPI_ISL_10716756, EPI_ISL_10716757, EPI_ISL_10716758, EPI_ISL_10716759, EPI_ISL_10716760, EPI_ISL_10716761, EPI_ISL_10716762, EPI_ISL_10716763, EPI_ISL_10716764, EPI_ISL_10716765, EPI_ISL_10716766, EPI_ISL_10716767, EPI_ISL_10716768, EPI_ISL_10716769, EPI_ISL_10716770, EPI_ISL_10716771, EPI_ISL_10716772, EPI_ISL_10716773, EPI_ISL_10716774, EPI_ISL_10716775, EPI_ISL_10716776, EPI_ISL_10716777, EPI_ISL_10716778, EPI_ISL_10716779, EPI_ISL_10716780, EPI_ISL_10716781, EPI_ISL_10716782, EPI_ISL_10716783, EPI_ISL_10716784, EPI_ISL_10716785, EPI_ISL_10716786, EPI_ISL_10716787, EPI_ISL_10716788, EPI_ISL_10716789, EPI_ISL_10716790, EPI_ISL_10716791, EPI_ISL_10716793, EPI_ISL_10716794, EPI_ISL_10716795, EPI_ISL_10716796, EPI_ISL_10716797, EPI_ISL_10716798, EPI_ISL_10716799, EPI_ISL_10716800, EPI_ISL_10716801, EPI_ISL_10716802, EPI_ISL_10716803, EPI_ISL_10716804, EPI_ISL_10716805, EPI_ISL_10716806, EPI_ISL_10716807, EPI_ISL_10716809, EPI_ISL_10716810, EPI_ISL_10716811, EPI_ISL_10716812, EPI_ISL_10716813, EPI_ISL_10716814, EPI_ISL_10716816, EPI_ISL_10720898, EPI_ISL_10720899, EPI_ISL_10720917, EPI_ISL_10720921, EPI_ISL_10720922, EPI_ISL_10720936, EPI_ISL_10720939, EPI_ISL_10720942, EPI_ISL_10720945, EPI_ISL_10720959, EPI_ISL_10720964, EPI_ISL_10720967, EPI_ISL_10720972, EPI_ISL_10720980, EPI_ISL_10720994, EPI_ISL_10721000, EPI_ISL_10721008, EPI_ISL_10721019, EPI_ISL_10721027, EPI_ISL_10721028, EPI_ISL_10721030, EPI_ISL_10721034, EPI_ISL_10721045, EPI_ISL_10721046, EPI_ISL_10721063, EPI_ISL_10721065, EPI_ISL_10721071, EPI_ISL_10721081, EPI_ISL_10721083, EPI_ISL_10721087, EPI_ISL_10721089, EPI_ISL_10721092, EPI_ISL_10721094, EPI_ISL_10721096, EPI_ISL_10721097, EPI_ISL_10721098, EPI_ISL_10721101, EPI_ISL_10721107, EPI_ISL_10721109, EPI_ISL_10721112, EPI_ISL_10721115, EPI_ISL_10721130, EPI_ISL_10721131, EPI_ISL_10721136, EPI_ISL_10721137, EPI_ISL_10721139, EPI_ISL_10721144, EPI_ISL_10721149, EPI_ISL_10721152, EPI_ISL_10721155, EPI_ISL_10721157, EPI_ISL_10721163, EPI_ISL_10721178, EPI_ISL_10721181, EPI_ISL_10721192, EPI_ISL_10721199, EPI_ISL_10721211, EPI_ISL_10721221, EPI_ISL_10721236, EPI_ISL_10721237, EPI_ISL_10721238, EPI_ISL_10724195, EPI_ISL_10729907, EPI_ISL_10729913, EPI_ISL_10729914, EPI_ISL_10729923, EPI_ISL_10729942, EPI_ISL_10729960, EPI_ISL_10729962, EPI_ISL_10729987, EPI_ISL_10730596, EPI_ISL_10737360, EPI_ISL_10737642, EPI_ISL_10737702, EPI_ISL_10737703, EPI_ISL_10737959, EPI_ISL_10738130, EPI_ISL_10738152, EPI_ISL_10739357, EPI_ISL_10743603, EPI_ISL_10744853, EPI_ISL_10745444, EPI_ISL_10745973, EPI_ISL_10746047, EPI_ISL_10746077, EPI_ISL_10746272, EPI_ISL_10746292, EPI_ISL_10746326, EPI_ISL_10746390, EPI_ISL_10746411, EPI_ISL_10746428, EPI_ISL_10746508, EPI_ISL_10746552, EPI_ISL_10746590, EPI_ISL_10746695, EPI_ISL_10746698, EPI_ISL_10747159, EPI_ISL_10747334, EPI_ISL_10748103, EPI_ISL_10748266, EPI_ISL_10748279, EPI_ISL_10748300, EPI_ISL_10748460, EPI_ISL_10748490, EPI_ISL_10748497, EPI_ISL_10748499, EPI_ISL_10748731, EPI_ISL_10748742, EPI_ISL_10748765, EPI_ISL_10748789, EPI_ISL_10748796, EPI_ISL_10749120, EPI_ISL_10749363, EPI_ISL_10751290, EPI_ISL_10752622, EPI_ISL_10752779, EPI_ISL_10753373, EPI_ISL_10761887, EPI_ISL_10765065, EPI_ISL_10766003, EPI_ISL_10767929, EPI_ISL_10774009, EPI_ISL_10778001, EPI_ISL_10786839, EPI_ISL_10787136, EPI_ISL_10792746, EPI_ISL_10794919, EPI_ISL_10794943, EPI_ISL_10795137, EPI_ISL_10804609, EPI_ISL_10808697, EPI_ISL_10816282, EPI_ISL_10818640, EPI_ISL_10824995, EPI_ISL_10825050, EPI_ISL_10825072, EPI_ISL_10825080, EPI_ISL_10827078, EPI_ISL_10827098, EPI_ISL_10827110, EPI_ISL_10827811, EPI_ISL_10828375, EPI_ISL_10830097, EPI_ISL_10832248, EPI_ISL_10832490, EPI_ISL_10832873, EPI_ISL_10834277, EPI_ISL_10834286, EPI_ISL_10834301, EPI_ISL_10834303, EPI_ISL_10834339, EPI_ISL_10835529, EPI_ISL_10835646, EPI_ISL_10836497, EPI_ISL_10839829, EPI_ISL_10846979, EPI_ISL_10846993, EPI_ISL_10847003, EPI_ISL_10847047, EPI_ISL_10856645, EPI_ISL_10858431, EPI_ISL_10858434, EPI_ISL_10858504, EPI_ISL_10858534, EPI_ISL_10858576, EPI_ISL_10859516, EPI_ISL_10863825, EPI_ISL_10863916, EPI_ISL_10864352, EPI_ISL_10864402, EPI_ISL_10864691, EPI_ISL_10869496, EPI_ISL_10875769, EPI_ISL_10876749, EPI_ISL_10885204, EPI_ISL_10889622, EPI_ISL_10889699, EPI_ISL_10894052, EPI_ISL_10899907, EPI_ISL_10901492, EPI_ISL_10901542, EPI_ISL_10901650, EPI_ISL_10902039, EPI_ISL_10902076, EPI_ISL_10903411, EPI_ISL_10905381, EPI_ISL_10905390, EPI_ISL_10906700, EPI_ISL_10907939, EPI_ISL_10908621, EPI_ISL_10909171, EPI_ISL_10910544, EPI_ISL_10910885, EPI_ISL_10911406, EPI_ISL_10911449, EPI_ISL_10916131, EPI_ISL_10916325, EPI_ISL_10916342, EPI_ISL_10916454, EPI_ISL_10916811, EPI_ISL_10916964, EPI_ISL_10917724, EPI_ISL_10918548, EPI_ISL_10918663, EPI_ISL_10918757, EPI_ISL_10918877, EPI_ISL_10918897, EPI_ISL_10919073, EPI_ISL_10919348, EPI_ISL_10919390, EPI_ISL_10919417, EPI_ISL_10920188, EPI_ISL_10920207, EPI_ISL_10920295, EPI_ISL_10920318, EPI_ISL_10920323, EPI_ISL_10920387, EPI_ISL_10920420, EPI_ISL_10920485, EPI_ISL_10929576, EPI_ISL_10929737, EPI_ISL_10931583, EPI_ISL_10940696, EPI_ISL_10944316, EPI_ISL_10944320, EPI_ISL_10944473, EPI_ISL_10944567, EPI_ISL_10944816, EPI_ISL_10945552, EPI_ISL_10952144, EPI_ISL_10952168, EPI_ISL_10952171, EPI_ISL_10953120, EPI_ISL_10958600, EPI_ISL_10958671, EPI_ISL_10958693, EPI_ISL_10958704, EPI_ISL_10958730, EPI_ISL_10969481, EPI_ISL_10981425, EPI_ISL_10984859, EPI_ISL_10985594, EPI_ISL_10985626, EPI_ISL_10989445, EPI_ISL_10989720, EPI_ISL_10989838, EPI_ISL_10989839, EPI_ISL_10993276, EPI_ISL_10993277, EPI_ISL_10993342, EPI_ISL_10993606, EPI_ISL_10993614, EPI_ISL_10993840, EPI_ISL_10996854, EPI_ISL_10996875, EPI_ISL_10996887, EPI_ISL_10996895, EPI_ISL_10997448, EPI_ISL_10999171")
EPI_11mil_12mil = stringlist_to_set("EPI_ISL_11522614, EPI_ISL_11001378, EPI_ISL_11001397, EPI_ISL_11001406, EPI_ISL_11001410, EPI_ISL_11001422, EPI_ISL_11001431, EPI_ISL_11001508, EPI_ISL_11001616, EPI_ISL_11001834, EPI_ISL_11001864, EPI_ISL_11001898, EPI_ISL_11001909, EPI_ISL_11001910, EPI_ISL_11001940, EPI_ISL_11002001, EPI_ISL_11002017, EPI_ISL_11002051, EPI_ISL_11002068, EPI_ISL_11007865, EPI_ISL_11011841, EPI_ISL_11011842, EPI_ISL_11011899, EPI_ISL_11011902, EPI_ISL_11011912, EPI_ISL_11011916, EPI_ISL_11011921, EPI_ISL_11011925, EPI_ISL_11011943, EPI_ISL_11011947, EPI_ISL_11011948, EPI_ISL_11011954, EPI_ISL_11011956, EPI_ISL_11011959, EPI_ISL_11011977, EPI_ISL_11011979, EPI_ISL_11011981, EPI_ISL_11011985, EPI_ISL_11012065, EPI_ISL_11012067, EPI_ISL_11012071, EPI_ISL_11012083, EPI_ISL_11012095, EPI_ISL_11012096, EPI_ISL_11012105, EPI_ISL_11012146, EPI_ISL_11012162, EPI_ISL_11012215, EPI_ISL_11012226, EPI_ISL_11012236, EPI_ISL_11012240, EPI_ISL_11012245, EPI_ISL_11012261, EPI_ISL_11012315, EPI_ISL_11012413, EPI_ISL_11012456, EPI_ISL_11012472, EPI_ISL_11012512, EPI_ISL_11012531, EPI_ISL_11012543, EPI_ISL_11012621, EPI_ISL_11012652, EPI_ISL_11012696, EPI_ISL_11012965, EPI_ISL_11015460, EPI_ISL_11015499, EPI_ISL_11016046, EPI_ISL_11022678, EPI_ISL_11023750, EPI_ISL_11023776, EPI_ISL_11023778, EPI_ISL_11027868, EPI_ISL_11035832, EPI_ISL_11036960, EPI_ISL_11036979, EPI_ISL_11037012, EPI_ISL_11037077, EPI_ISL_11057571, EPI_ISL_11057573, EPI_ISL_11057577, EPI_ISL_11057578, EPI_ISL_11057581, EPI_ISL_11057586, EPI_ISL_11057587, EPI_ISL_11057588, EPI_ISL_11057589, EPI_ISL_11057593, EPI_ISL_11057599, EPI_ISL_11057601, EPI_ISL_11057616, EPI_ISL_11057681, EPI_ISL_11057697, EPI_ISL_11057712, EPI_ISL_11057720, EPI_ISL_11057741, EPI_ISL_11057775, EPI_ISL_11057776, EPI_ISL_11057791, EPI_ISL_11057819, EPI_ISL_11057998, EPI_ISL_11062632, EPI_ISL_11063127, EPI_ISL_11063642, EPI_ISL_11063656, EPI_ISL_11064020, EPI_ISL_11068208, EPI_ISL_11069952, EPI_ISL_11070263, EPI_ISL_11072733, EPI_ISL_11072760, EPI_ISL_11072840, EPI_ISL_11073708, EPI_ISL_11074533, EPI_ISL_11074547, EPI_ISL_11074949, EPI_ISL_11075203, EPI_ISL_11075204, EPI_ISL_11075208, EPI_ISL_11075232, EPI_ISL_11095421, EPI_ISL_11101743, EPI_ISL_11102862, EPI_ISL_11102868, EPI_ISL_11102875, EPI_ISL_11102889, EPI_ISL_11102890, EPI_ISL_11102891, EPI_ISL_11102900, EPI_ISL_11102924, EPI_ISL_11102931, EPI_ISL_11102937, EPI_ISL_11102940, EPI_ISL_11102941, EPI_ISL_11102942, EPI_ISL_11102950, EPI_ISL_11102956, EPI_ISL_11102957, EPI_ISL_11103042, EPI_ISL_11103086, EPI_ISL_11104664, EPI_ISL_11104679, EPI_ISL_11104750, EPI_ISL_11104787, EPI_ISL_11104817, EPI_ISL_11104830, EPI_ISL_11104878, EPI_ISL_11104879, EPI_ISL_11104936, EPI_ISL_11104946, EPI_ISL_11104949, EPI_ISL_11104969, EPI_ISL_11105281, EPI_ISL_11105304, EPI_ISL_11105376, EPI_ISL_11105377, EPI_ISL_11105589, EPI_ISL_11105593, EPI_ISL_11105661, EPI_ISL_11105665, EPI_ISL_11106012, EPI_ISL_11106554, EPI_ISL_11106847, EPI_ISL_11108553, EPI_ISL_11108569, EPI_ISL_11108594, EPI_ISL_11108623, EPI_ISL_11108654, EPI_ISL_11108683, EPI_ISL_11108722, EPI_ISL_11108748, EPI_ISL_11108767, EPI_ISL_11108772, EPI_ISL_11108774, EPI_ISL_11108803, EPI_ISL_11108815, EPI_ISL_11108817, EPI_ISL_11133720, EPI_ISL_11133721, EPI_ISL_11133722, EPI_ISL_11134084, EPI_ISL_11134695, EPI_ISL_11141677, EPI_ISL_11141682, EPI_ISL_11144326, EPI_ISL_11145334, EPI_ISL_11148047, EPI_ISL_11168875, EPI_ISL_11168895, EPI_ISL_11168902, EPI_ISL_11168912, EPI_ISL_11168920, EPI_ISL_11168923, EPI_ISL_11168926, EPI_ISL_11168929, EPI_ISL_11168930, EPI_ISL_11168932, EPI_ISL_11168988, EPI_ISL_11168992, EPI_ISL_11168995, EPI_ISL_11169013, EPI_ISL_11169014, EPI_ISL_11169015, EPI_ISL_11169017, EPI_ISL_11169022, EPI_ISL_11169028, EPI_ISL_11173013, EPI_ISL_11173017, EPI_ISL_11173018, EPI_ISL_11173041, EPI_ISL_11173049, EPI_ISL_11173060, EPI_ISL_11173065, EPI_ISL_11173069, EPI_ISL_11173070, EPI_ISL_11173076, EPI_ISL_11173079, EPI_ISL_11173084, EPI_ISL_11173088, EPI_ISL_11173353, EPI_ISL_11173361, EPI_ISL_11173417, EPI_ISL_11179918, EPI_ISL_11179924, EPI_ISL_11209814, EPI_ISL_11220075, EPI_ISL_11220597, EPI_ISL_11222931, EPI_ISL_11225546, EPI_ISL_11225824, EPI_ISL_11228601, EPI_ISL_11228602, EPI_ISL_11228630, EPI_ISL_11228665, EPI_ISL_11228678, EPI_ISL_11228700, EPI_ISL_11235456, EPI_ISL_11235471, EPI_ISL_11235483, EPI_ISL_11235496, EPI_ISL_11235499, EPI_ISL_11235563, EPI_ISL_11235578, EPI_ISL_11235580, EPI_ISL_11235581, EPI_ISL_11235583, EPI_ISL_11235627, EPI_ISL_11235843, EPI_ISL_11235856, EPI_ISL_11235861, EPI_ISL_11235862, EPI_ISL_11235863, EPI_ISL_11235865, EPI_ISL_11235879, EPI_ISL_11235890, EPI_ISL_11235919, EPI_ISL_11235925, EPI_ISL_11235929, EPI_ISL_11235937, EPI_ISL_11236108, EPI_ISL_11236172, EPI_ISL_11236449, EPI_ISL_11237359, EPI_ISL_11246565, EPI_ISL_11260036, EPI_ISL_11260040, EPI_ISL_11260049, EPI_ISL_11260136, EPI_ISL_11260148, EPI_ISL_11260154, EPI_ISL_11262602, EPI_ISL_11268266, EPI_ISL_11268277, EPI_ISL_11268631, EPI_ISL_11270606, EPI_ISL_11271349, EPI_ISL_11278381, EPI_ISL_11278382, EPI_ISL_11284205, EPI_ISL_11291795, EPI_ISL_11299954, EPI_ISL_11299975, EPI_ISL_11311005, EPI_ISL_11323423, EPI_ISL_11323501, EPI_ISL_11323538, EPI_ISL_11323548, EPI_ISL_11323552, EPI_ISL_11323584, EPI_ISL_11323592, EPI_ISL_11323651, EPI_ISL_11323672, EPI_ISL_11323696, EPI_ISL_11323720, EPI_ISL_11325780, EPI_ISL_11330485, EPI_ISL_11333623, EPI_ISL_11333761, EPI_ISL_11344101, EPI_ISL_11349130, EPI_ISL_11351998, EPI_ISL_11369287, EPI_ISL_11381121, EPI_ISL_11381144, EPI_ISL_11397865, EPI_ISL_11404058, EPI_ISL_11404085, EPI_ISL_11404186, EPI_ISL_11404235, EPI_ISL_11404242, EPI_ISL_11404262, EPI_ISL_11404274, EPI_ISL_11404733, EPI_ISL_11410657, EPI_ISL_11414660, EPI_ISL_11414941, EPI_ISL_11415188, EPI_ISL_11416082, EPI_ISL_11436707, EPI_ISL_11441807, EPI_ISL_11442637, EPI_ISL_11450574, EPI_ISL_11451045, EPI_ISL_11451135, EPI_ISL_11451317, EPI_ISL_11451340, EPI_ISL_11451721, EPI_ISL_11451747, EPI_ISL_11452037, EPI_ISL_11452179, EPI_ISL_11452204, EPI_ISL_11454385, EPI_ISL_11454553, EPI_ISL_11454667, EPI_ISL_11454932, EPI_ISL_11455246, EPI_ISL_11466767, EPI_ISL_11468033, EPI_ISL_11468040, EPI_ISL_11468560, EPI_ISL_11468703, EPI_ISL_11468704, EPI_ISL_11468712, EPI_ISL_11468998, EPI_ISL_11469089, EPI_ISL_11471268, EPI_ISL_11471391, EPI_ISL_11471530, EPI_ISL_11471595, EPI_ISL_11472799, EPI_ISL_11473010, EPI_ISL_11473294, EPI_ISL_11473344, EPI_ISL_11474040, EPI_ISL_11474069, EPI_ISL_11474248, EPI_ISL_11474345, EPI_ISL_11474631, EPI_ISL_11474689, EPI_ISL_11477582, EPI_ISL_11477660, EPI_ISL_11478575, EPI_ISL_11480313, EPI_ISL_11483474, EPI_ISL_11483944, EPI_ISL_11483969, EPI_ISL_11486118, EPI_ISL_11486134, EPI_ISL_11486916, EPI_ISL_11487008, EPI_ISL_11490008, EPI_ISL_11494181, EPI_ISL_11495508, EPI_ISL_11503117, EPI_ISL_11505283, EPI_ISL_11505390, EPI_ISL_11507500, EPI_ISL_11507601, EPI_ISL_11519651, EPI_ISL_11520066, EPI_ISL_11521151, EPI_ISL_11521416, EPI_ISL_11521422, EPI_ISL_11521427, EPI_ISL_11521431, EPI_ISL_11521432, EPI_ISL_11521468, EPI_ISL_11521537, EPI_ISL_11521679, EPI_ISL_11522492, EPI_ISL_11522623, EPI_ISL_11522626, EPI_ISL_11525794, EPI_ISL_11525892, EPI_ISL_11525987, EPI_ISL_11526041, EPI_ISL_11532512, EPI_ISL_11533032, EPI_ISL_11533037, EPI_ISL_11533038, EPI_ISL_11533039, EPI_ISL_11533042, EPI_ISL_11533048, EPI_ISL_11533055, EPI_ISL_11533059, EPI_ISL_11533060, EPI_ISL_11533069, EPI_ISL_11533082, EPI_ISL_11533083, EPI_ISL_11533084, EPI_ISL_11533107, EPI_ISL_11533112, EPI_ISL_11533950, EPI_ISL_11534031, EPI_ISL_11534153, EPI_ISL_11534154, EPI_ISL_11538263, EPI_ISL_11543286, EPI_ISL_11543382, EPI_ISL_11543420, EPI_ISL_11543431, EPI_ISL_11555534, EPI_ISL_11555548, EPI_ISL_11555594, EPI_ISL_11559326, EPI_ISL_11559410, EPI_ISL_11559413, EPI_ISL_11559417, EPI_ISL_11559448, EPI_ISL_11563413, EPI_ISL_11563417, EPI_ISL_11563418, EPI_ISL_11565882, EPI_ISL_11565885, EPI_ISL_11566152, EPI_ISL_11566753, EPI_ISL_11575298, EPI_ISL_11575324, EPI_ISL_11575447, EPI_ISL_11575462, EPI_ISL_11575466, EPI_ISL_11575478, EPI_ISL_11575481, EPI_ISL_11578018, EPI_ISL_11579649, EPI_ISL_11579784, EPI_ISL_11579804, EPI_ISL_11579812, EPI_ISL_11579831, EPI_ISL_11580138, EPI_ISL_11580145, EPI_ISL_11582258, EPI_ISL_11582259, EPI_ISL_11582428, EPI_ISL_11582444, EPI_ISL_11582544, EPI_ISL_11582650, EPI_ISL_11582673, EPI_ISL_11583316, EPI_ISL_11583338, EPI_ISL_11583341, EPI_ISL_11584302, EPI_ISL_11584306, EPI_ISL_11584307, EPI_ISL_11584311, EPI_ISL_11584334, EPI_ISL_11584335, EPI_ISL_11586351, EPI_ISL_11586743, EPI_ISL_11588831, EPI_ISL_11588876, EPI_ISL_11591856, EPI_ISL_11593249, EPI_ISL_11594770, EPI_ISL_11595467, EPI_ISL_11598753, EPI_ISL_11610925, EPI_ISL_11612817, EPI_ISL_11612972, EPI_ISL_11612981, EPI_ISL_11613645, EPI_ISL_11614059, EPI_ISL_11615640, EPI_ISL_11616224, EPI_ISL_11618940, EPI_ISL_11619152, EPI_ISL_11619182, EPI_ISL_11619311, EPI_ISL_11619523, EPI_ISL_11621075, EPI_ISL_11636756, EPI_ISL_11651621, EPI_ISL_11651726, EPI_ISL_11651735, EPI_ISL_11654722, EPI_ISL_11656053, EPI_ISL_11656172, EPI_ISL_11656610, EPI_ISL_11656659, EPI_ISL_11657596, EPI_ISL_11657892, EPI_ISL_11657965, EPI_ISL_11662146, EPI_ISL_11663835, EPI_ISL_11666737, EPI_ISL_11666773, EPI_ISL_11667158, EPI_ISL_11667216, EPI_ISL_11672579, EPI_ISL_11672755, EPI_ISL_11672970, EPI_ISL_11677016, EPI_ISL_11681633, EPI_ISL_11682511, EPI_ISL_11687875, EPI_ISL_11687889, EPI_ISL_11687901, EPI_ISL_11735262, EPI_ISL_11738966, EPI_ISL_11738979, EPI_ISL_11738981, EPI_ISL_11739588, EPI_ISL_11744191, EPI_ISL_11744197, EPI_ISL_11744203, EPI_ISL_11744225, EPI_ISL_11744241, EPI_ISL_11745458, EPI_ISL_11750312, EPI_ISL_11754783, EPI_ISL_11755178, EPI_ISL_11764066, EPI_ISL_11764145, EPI_ISL_11764370, EPI_ISL_11765223, EPI_ISL_11765233, EPI_ISL_11765239, EPI_ISL_11765314, EPI_ISL_11776032, EPI_ISL_11777698, EPI_ISL_11778037, EPI_ISL_11780300, EPI_ISL_11780326, EPI_ISL_11780428, EPI_ISL_11792579, EPI_ISL_11792580, EPI_ISL_11799231, EPI_ISL_11799253, EPI_ISL_11799254, EPI_ISL_11799412, EPI_ISL_11799422, EPI_ISL_11799425, EPI_ISL_11799426, EPI_ISL_11799427, EPI_ISL_11799428, EPI_ISL_11799429, EPI_ISL_11799448, EPI_ISL_11799451, EPI_ISL_11799459, EPI_ISL_11799460, EPI_ISL_11799463, EPI_ISL_11799472, EPI_ISL_11799473, EPI_ISL_11799482, EPI_ISL_11799483, EPI_ISL_11800533, EPI_ISL_11805895, EPI_ISL_11816334, EPI_ISL_11817226, EPI_ISL_11817393, EPI_ISL_11818911, EPI_ISL_11822283, EPI_ISL_11822392, EPI_ISL_11822405, EPI_ISL_11822410, EPI_ISL_11822441, EPI_ISL_11822445, EPI_ISL_11822452, EPI_ISL_11833006, EPI_ISL_11833041, EPI_ISL_11833043, EPI_ISL_11833045, EPI_ISL_11834694, EPI_ISL_11834787, EPI_ISL_11835704, EPI_ISL_11838824, EPI_ISL_11838984, EPI_ISL_11839213, EPI_ISL_11839231, EPI_ISL_11839266, EPI_ISL_11840926, EPI_ISL_11841862, EPI_ISL_11851894, EPI_ISL_11851906, EPI_ISL_11852879, EPI_ISL_11854563, EPI_ISL_11857754, EPI_ISL_11857995, EPI_ISL_11858417, EPI_ISL_11858533, EPI_ISL_11858709, EPI_ISL_11858787, EPI_ISL_11858901, EPI_ISL_11860869, EPI_ISL_11861621, EPI_ISL_11873762, EPI_ISL_11887370, EPI_ISL_11887719, EPI_ISL_11887726, EPI_ISL_11890159, EPI_ISL_11890160, EPI_ISL_11890193, EPI_ISL_11897161, EPI_ISL_11897201, EPI_ISL_11897281, EPI_ISL_11897326, EPI_ISL_11897379, EPI_ISL_11898663, EPI_ISL_11898850, EPI_ISL_11899077, EPI_ISL_11901662, EPI_ISL_11901676, EPI_ISL_11901688, EPI_ISL_11901702, EPI_ISL_11901703, EPI_ISL_11901787, EPI_ISL_11905620, EPI_ISL_11908129, EPI_ISL_11924873, EPI_ISL_11927468, EPI_ISL_11927549, EPI_ISL_11927559, EPI_ISL_11927927, EPI_ISL_11929258, EPI_ISL_11929260, EPI_ISL_11929325, EPI_ISL_11929348, EPI_ISL_11929386, EPI_ISL_11930116, EPI_ISL_11936029, EPI_ISL_11937846, EPI_ISL_11946506, EPI_ISL_11946951, EPI_ISL_11947876, EPI_ISL_11947967, EPI_ISL_11948994, EPI_ISL_11950245, EPI_ISL_11952162, EPI_ISL_11953205, EPI_ISL_11953779, EPI_ISL_11954832, EPI_ISL_11969387, EPI_ISL_11969394, EPI_ISL_11971083, EPI_ISL_11976102, EPI_ISL_11988824, EPI_ISL_11999927")
EPI_12mil_13mil = stringlist_to_strings("")
EPI_13mil_14mil = stringlist_to_strings("")
EPI_16mil_17mil = stringlist_to_set("EPI_ISL_16815237, EPI_ISL_16897661, EPI_ISL_16371526, EPI_ISL_16654387, EPI_ISL_16654388, EPI_ISL_16670567, EPI_ISL_16742612, EPI_ISL_16991534, EPI_ISL_16876388, EPI_ISL_16206521, EPI_ISL_16097562, EPI_ISL_16201324, EPI_ISL_16472666, EPI_ISL_16897651, EPI_ISL_16012710, EPI_ISL_16012943, EPI_ISL_16012954, EPI_ISL_16012987, EPI_ISL_16013209, EPI_ISL_16013227, EPI_ISL_16013267, EPI_ISL_16013309, EPI_ISL_16013449, EPI_ISL_16113255, EPI_ISL_16113336, EPI_ISL_16113399, EPI_ISL_16113938, EPI_ISL_16114486, EPI_ISL_16270893, EPI_ISL_16271048, EPI_ISL_16271102, EPI_ISL_16271255, EPI_ISL_16271333, EPI_ISL_16271386, EPI_ISL_16397554, EPI_ISL_16397699, EPI_ISL_16397720, EPI_ISL_16397721, EPI_ISL_16397783, EPI_ISL_16397860, EPI_ISL_16397883, EPI_ISL_16397988, EPI_ISL_16398016, EPI_ISL_16398020, EPI_ISL_16398022, EPI_ISL_16398035, EPI_ISL_16398060, EPI_ISL_16398063, EPI_ISL_16398072, EPI_ISL_16454140, EPI_ISL_16454168, EPI_ISL_16542189, EPI_ISL_16542267, EPI_ISL_16542369, EPI_ISL_16542453, EPI_ISL_16542552, EPI_ISL_16542938, EPI_ISL_16599288, EPI_ISL_16599299, EPI_ISL_16599325, EPI_ISL_16599380, EPI_ISL_16599389, EPI_ISL_16599460, EPI_ISL_16599467, EPI_ISL_16599492, EPI_ISL_16599495, EPI_ISL_16599498, EPI_ISL_16599525, EPI_ISL_16599538, EPI_ISL_16599540, EPI_ISL_16599554, EPI_ISL_16599636, EPI_ISL_16599649, EPI_ISL_16599747, EPI_ISL_16599794, EPI_ISL_16599826, EPI_ISL_16599842, EPI_ISL_16599844, EPI_ISL_16599878, EPI_ISL_16599959, EPI_ISL_16599980, EPI_ISL_16761912, EPI_ISL_16761930, EPI_ISL_16761962, EPI_ISL_16761978, EPI_ISL_16762022, EPI_ISL_16762028, EPI_ISL_16762035, EPI_ISL_16762052, EPI_ISL_16762066, EPI_ISL_16762070, EPI_ISL_16762084, EPI_ISL_16762132, EPI_ISL_16762148, EPI_ISL_16762176, EPI_ISL_16762245, EPI_ISL_16762249, EPI_ISL_16762255, EPI_ISL_16762269, EPI_ISL_16762273, EPI_ISL_16762297, EPI_ISL_16762310, EPI_ISL_16762327, EPI_ISL_16762407, EPI_ISL_16762427, EPI_ISL_16762455, EPI_ISL_16762458, EPI_ISL_16762462, EPI_ISL_16762475, EPI_ISL_16762559, EPI_ISL_16762600, EPI_ISL_16762611, EPI_ISL_16762614, EPI_ISL_16762631, EPI_ISL_16762652, EPI_ISL_16762653, EPI_ISL_16762654, EPI_ISL_16762699, EPI_ISL_16762700, EPI_ISL_16762714, EPI_ISL_16762726, EPI_ISL_16762735, EPI_ISL_16762753, EPI_ISL_16919296, EPI_ISL_16919303, EPI_ISL_16919306, EPI_ISL_16919315, EPI_ISL_16919337, EPI_ISL_16919348, EPI_ISL_16919362, EPI_ISL_16919365, EPI_ISL_16919367, EPI_ISL_16919368, EPI_ISL_16919380, EPI_ISL_16919388, EPI_ISL_16919396, EPI_ISL_16919399, EPI_ISL_16919412, EPI_ISL_16919430, EPI_ISL_16919435, EPI_ISL_16919455, EPI_ISL_16919456, EPI_ISL_16919467, EPI_ISL_16919468, EPI_ISL_16919496, EPI_ISL_16919504, EPI_ISL_16919523, EPI_ISL_16919526, EPI_ISL_16919546, EPI_ISL_16919558, EPI_ISL_16919586, EPI_ISL_16919592, EPI_ISL_16919598, EPI_ISL_16919609, EPI_ISL_16919640, EPI_ISL_16919660, EPI_ISL_16919662, EPI_ISL_16919670, EPI_ISL_16919680, EPI_ISL_16919695, EPI_ISL_16919706, EPI_ISL_16919718, EPI_ISL_16919762, EPI_ISL_16919772, EPI_ISL_16919794, EPI_ISL_16919796, EPI_ISL_16919804, EPI_ISL_16919806, EPI_ISL_16919810, EPI_ISL_16919841, EPI_ISL_16919843, EPI_ISL_16919845, EPI_ISL_16919852, EPI_ISL_16919856, EPI_ISL_16919870, EPI_ISL_16919894, EPI_ISL_16919897, EPI_ISL_16919902, EPI_ISL_16919906, EPI_ISL_16919919, EPI_ISL_16919932, EPI_ISL_16919944, EPI_ISL_16919962, EPI_ISL_16001302, EPI_ISL_16020838, EPI_ISL_16024683, EPI_ISL_16047006, EPI_ISL_16049706, EPI_ISL_16088909, EPI_ISL_16089016, EPI_ISL_16092776, EPI_ISL_16097810, EPI_ISL_16119629, EPI_ISL_16148354, EPI_ISL_16169012, EPI_ISL_16169559, EPI_ISL_16171387, EPI_ISL_16173291, EPI_ISL_16173302, EPI_ISL_16189719, EPI_ISL_16204016, EPI_ISL_16205980, EPI_ISL_16206468, EPI_ISL_16248743, EPI_ISL_16260886, EPI_ISL_16268007, EPI_ISL_16268596, EPI_ISL_16276826, EPI_ISL_16281961, EPI_ISL_16281966, EPI_ISL_16293327, EPI_ISL_16302637, EPI_ISL_16327676, EPI_ISL_16364514, EPI_ISL_16368859, EPI_ISL_16368863, EPI_ISL_16368870, EPI_ISL_16377998, EPI_ISL_16378441, EPI_ISL_16381367, EPI_ISL_16381431, EPI_ISL_16382369, EPI_ISL_16382423, EPI_ISL_16382491, EPI_ISL_16390727, EPI_ISL_16402352, EPI_ISL_16431012, EPI_ISL_16440289, EPI_ISL_16449531, EPI_ISL_16449569, EPI_ISL_16465346, EPI_ISL_16465349, EPI_ISL_16475710, EPI_ISL_16485170, EPI_ISL_16496661, EPI_ISL_16499231, EPI_ISL_16507911, EPI_ISL_16510336, EPI_ISL_16513678, EPI_ISL_16524791, EPI_ISL_16527367, EPI_ISL_16528798, EPI_ISL_16534033, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570256, EPI_ISL_16570393, EPI_ISL_16574023, EPI_ISL_16574144, EPI_ISL_16574237, EPI_ISL_16598797, EPI_ISL_16608560, EPI_ISL_16612080, EPI_ISL_16613416, EPI_ISL_16649909, EPI_ISL_16649933, EPI_ISL_16655461, EPI_ISL_16655462, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16660760, EPI_ISL_16660761, EPI_ISL_16665774, EPI_ISL_16665862, EPI_ISL_16668033, EPI_ISL_16668112, EPI_ISL_16668536, EPI_ISL_16668786, EPI_ISL_16702345, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702391, EPI_ISL_16729840, EPI_ISL_16734294, EPI_ISL_16742171, EPI_ISL_16742359, EPI_ISL_16742486, EPI_ISL_16742561, EPI_ISL_16760522, EPI_ISL_16760525, EPI_ISL_16760534, EPI_ISL_16760542, EPI_ISL_16760544, EPI_ISL_16760553, EPI_ISL_16760564, EPI_ISL_16760565, EPI_ISL_16763262, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16825422, EPI_ISL_16835014, EPI_ISL_16845486, EPI_ISL_16847576, EPI_ISL_16860429, EPI_ISL_16860526, EPI_ISL_16861003, EPI_ISL_16870757, EPI_ISL_16879371, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16887308, EPI_ISL_16893748, EPI_ISL_16895710, EPI_ISL_16895713, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907578, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907588, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907598, EPI_ISL_16907601, EPI_ISL_16907602, EPI_ISL_16921497, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943841, EPI_ISL_16943847, EPI_ISL_16943850, EPI_ISL_16943855, EPI_ISL_16943857, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16943896, EPI_ISL_16947614, EPI_ISL_16966278, EPI_ISL_16968369, EPI_ISL_16976030, EPI_ISL_16986705, EPI_ISL_16997282, EPI_ISL_16020838, EPI_ISL_16041723, EPI_ISL_16047006, EPI_ISL_16049706, EPI_ISL_16089016, EPI_ISL_16097810, EPI_ISL_16119629, EPI_ISL_16169012, EPI_ISL_16169559, EPI_ISL_16171387, EPI_ISL_16189719, EPI_ISL_16205980, EPI_ISL_16206468, EPI_ISL_16212482, EPI_ISL_16226002, EPI_ISL_16248743, EPI_ISL_16268007, EPI_ISL_16268596, EPI_ISL_16292066, EPI_ISL_16292121, EPI_ISL_16293327, EPI_ISL_16328978, EPI_ISL_16368859, EPI_ISL_16368863, EPI_ISL_16368870, EPI_ISL_16377998, EPI_ISL_16381431, EPI_ISL_16382369, EPI_ISL_16382423, EPI_ISL_16390727, EPI_ISL_16431012, EPI_ISL_16440289, EPI_ISL_16449525, EPI_ISL_16449531, EPI_ISL_16449569, EPI_ISL_16459438, EPI_ISL_16465329, EPI_ISL_16465341, EPI_ISL_16465346, EPI_ISL_16465349, EPI_ISL_16475710, EPI_ISL_16485170, EPI_ISL_16510336, EPI_ISL_16513678, EPI_ISL_16513739, EPI_ISL_16527367, EPI_ISL_16534033, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570393, EPI_ISL_16574144, EPI_ISL_16574237, EPI_ISL_16598797, EPI_ISL_16607325, EPI_ISL_16612080, EPI_ISL_16612088, EPI_ISL_16613416, EPI_ISL_16638962, EPI_ISL_16649909, EPI_ISL_16649933, EPI_ISL_16655461, EPI_ISL_16655462, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16665774, EPI_ISL_16665862, EPI_ISL_16668112, EPI_ISL_16668536, EPI_ISL_16668786, EPI_ISL_16702345, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702383, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702391, EPI_ISL_16724327, EPI_ISL_16729840, EPI_ISL_16734284, EPI_ISL_16742359, EPI_ISL_16742486, EPI_ISL_16742561, EPI_ISL_16760522, EPI_ISL_16760527, EPI_ISL_16760534, EPI_ISL_16760542, EPI_ISL_16760544, EPI_ISL_16760553, EPI_ISL_16760557, EPI_ISL_16760565, EPI_ISL_16760567, EPI_ISL_16765385, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16835014, EPI_ISL_16837809, EPI_ISL_16847576, EPI_ISL_16860429, EPI_ISL_16861003, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16887308, EPI_ISL_16895710, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16900417, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907578, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907598, EPI_ISL_16907601, EPI_ISL_16907602, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943847, EPI_ISL_16943855, EPI_ISL_16943857, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16943896, EPI_ISL_16947614, EPI_ISL_16976030, EPI_ISL_16986705, EPI_ISL_16997282, EPI_ISL_16097754, EPI_ISL_16119602, EPI_ISL_16119610, EPI_ISL_16119629, EPI_ISL_16171387, EPI_ISL_16204016, EPI_ISL_16212482, EPI_ISL_16307563, EPI_ISL_16357154, EPI_ISL_16368857, EPI_ISL_16369646, EPI_ISL_16377998, EPI_ISL_16382423, EPI_ISL_16390790, EPI_ISL_16402352, EPI_ISL_16440038, EPI_ISL_16441289, EPI_ISL_16449517, EPI_ISL_16457740, EPI_ISL_16465341, EPI_ISL_16475710, EPI_ISL_16483723, EPI_ISL_16496661, EPI_ISL_16510324, EPI_ISL_16524791, EPI_ISL_16527367, EPI_ISL_16529241, EPI_ISL_16536287, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570256, EPI_ISL_16574237, EPI_ISL_16606996, EPI_ISL_16607319, EPI_ISL_16607322, EPI_ISL_16607325, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16660760, EPI_ISL_16660761, EPI_ISL_16670793, EPI_ISL_16702339, EPI_ISL_16702350, EPI_ISL_16702360, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702383, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702393, EPI_ISL_16702396, EPI_ISL_16733307, EPI_ISL_16734294, EPI_ISL_16760522, EPI_ISL_16760527, EPI_ISL_16760534, EPI_ISL_16760541, EPI_ISL_16760553, EPI_ISL_16760557, EPI_ISL_16760562, EPI_ISL_16760565, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16823432, EPI_ISL_16845486, EPI_ISL_16870757, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16907559, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907602, EPI_ISL_16915893, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943841, EPI_ISL_16943847, EPI_ISL_16943857, EPI_ISL_16943858, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16993581, EPI_ISL_16073446, EPI_ISL_16861820, EPI_ISL_16510293, EPI_ISL_16815208, EPI_ISL_16457333, EPI_ISL_16937010, EPI_ISL_16536298, EPI_ISL_16536270, EPI_ISL_16536294, EPI_ISL_16536310, EPI_ISL_16536267, EPI_ISL_16913912, EPI_ISL_16170920, EPI_ISL_16201285, EPI_ISL_16201296, EPI_ISL_16201312, EPI_ISL_16201314, EPI_ISL_16201319, EPI_ISL_16201326, EPI_ISL_16201331, EPI_ISL_16201278, EPI_ISL_16201334, EPI_ISL_16940694, EPI_ISL_16861820, EPI_ISL_16012710, EPI_ISL_16012943, EPI_ISL_16012954, EPI_ISL_16012987, EPI_ISL_16013209, EPI_ISL_16013227, EPI_ISL_16013267, EPI_ISL_16013309, EPI_ISL_16013449, EPI_ISL_16113255, EPI_ISL_16113336, EPI_ISL_16113399, EPI_ISL_16113938, EPI_ISL_16114486, EPI_ISL_16270893, EPI_ISL_16271048, EPI_ISL_16271102, EPI_ISL_16271255, EPI_ISL_16271333, EPI_ISL_16271386, EPI_ISL_16397554, EPI_ISL_16397699, EPI_ISL_16397720, EPI_ISL_16397721, EPI_ISL_16397783, EPI_ISL_16397860, EPI_ISL_16397883, EPI_ISL_16397988, EPI_ISL_16398016, EPI_ISL_16398020, EPI_ISL_16398022, EPI_ISL_16398035, EPI_ISL_16398060, EPI_ISL_16398063, EPI_ISL_16398072, EPI_ISL_16454140, EPI_ISL_16454168, EPI_ISL_16542189, EPI_ISL_16542267, EPI_ISL_16542369, EPI_ISL_16542453, EPI_ISL_16542552, EPI_ISL_16542938, EPI_ISL_16599288, EPI_ISL_16599299, EPI_ISL_16599325, EPI_ISL_16599380, EPI_ISL_16599389, EPI_ISL_16599460, EPI_ISL_16599467, EPI_ISL_16599492, EPI_ISL_16599495, EPI_ISL_16599498, EPI_ISL_16599525, EPI_ISL_16599538, EPI_ISL_16599540, EPI_ISL_16599554, EPI_ISL_16599636, EPI_ISL_16599649, EPI_ISL_16599747, EPI_ISL_16599794, EPI_ISL_16599826, EPI_ISL_16599842, EPI_ISL_16599844, EPI_ISL_16599878, EPI_ISL_16599959, EPI_ISL_16599980, EPI_ISL_16761912, EPI_ISL_16761930, EPI_ISL_16761962, EPI_ISL_16761978, EPI_ISL_16762022, EPI_ISL_16762028, EPI_ISL_16762035, EPI_ISL_16762052, EPI_ISL_16762066, EPI_ISL_16762070, EPI_ISL_16762084, EPI_ISL_16762132, EPI_ISL_16762148, EPI_ISL_16762176, EPI_ISL_16762245, EPI_ISL_16762249, EPI_ISL_16762255, EPI_ISL_16762269, EPI_ISL_16762273, EPI_ISL_16762297, EPI_ISL_16762310, EPI_ISL_16762327, EPI_ISL_16762407, EPI_ISL_16762427, EPI_ISL_16762455, EPI_ISL_16762458, EPI_ISL_16762462, EPI_ISL_16762475, EPI_ISL_16762559, EPI_ISL_16762600, EPI_ISL_16762611, EPI_ISL_16762614, EPI_ISL_16762631, EPI_ISL_16762652, EPI_ISL_16762653, EPI_ISL_16762654, EPI_ISL_16762699, EPI_ISL_16762700, EPI_ISL_16762714, EPI_ISL_16762726, EPI_ISL_16762735, EPI_ISL_16762753, EPI_ISL_16919296, EPI_ISL_16919303, EPI_ISL_16919306, EPI_ISL_16919315, EPI_ISL_16919337, EPI_ISL_16919348, EPI_ISL_16919362, EPI_ISL_16919365, EPI_ISL_16919367, EPI_ISL_16919368, EPI_ISL_16919380, EPI_ISL_16919388, EPI_ISL_16919396, EPI_ISL_16919399, EPI_ISL_16919412, EPI_ISL_16919430, EPI_ISL_16919435, EPI_ISL_16919455, EPI_ISL_16919456, EPI_ISL_16919467, EPI_ISL_16919468, EPI_ISL_16919496, EPI_ISL_16919504, EPI_ISL_16919523, EPI_ISL_16919526, EPI_ISL_16919546, EPI_ISL_16919558, EPI_ISL_16919586, EPI_ISL_16919592, EPI_ISL_16919598, EPI_ISL_16919609, EPI_ISL_16919640, EPI_ISL_16919660, EPI_ISL_16919662, EPI_ISL_16919670, EPI_ISL_16919680, EPI_ISL_16919695, EPI_ISL_16919706, EPI_ISL_16919718, EPI_ISL_16919762, EPI_ISL_16919772, EPI_ISL_16919794, EPI_ISL_16919796, EPI_ISL_16919804, EPI_ISL_16919806, EPI_ISL_16919810, EPI_ISL_16919841, EPI_ISL_16919843, EPI_ISL_16919845, EPI_ISL_16919852, EPI_ISL_16919856, EPI_ISL_16919870, EPI_ISL_16919894, EPI_ISL_16919897, EPI_ISL_16919902, EPI_ISL_16919906, EPI_ISL_16919919, EPI_ISL_16919932, EPI_ISL_16919944, EPI_ISL_16919962, EPI_ISL_16001302, EPI_ISL_16020838, EPI_ISL_16024683, EPI_ISL_16047006, EPI_ISL_16049706, EPI_ISL_16088909, EPI_ISL_16089016, EPI_ISL_16092776, EPI_ISL_16097810, EPI_ISL_16119629, EPI_ISL_16148354, EPI_ISL_16169012, EPI_ISL_16169559, EPI_ISL_16171387, EPI_ISL_16173291, EPI_ISL_16173302, EPI_ISL_16189719, EPI_ISL_16204016, EPI_ISL_16205980, EPI_ISL_16206468, EPI_ISL_16248743, EPI_ISL_16260886, EPI_ISL_16268007, EPI_ISL_16268596, EPI_ISL_16276826, EPI_ISL_16281961, EPI_ISL_16281966, EPI_ISL_16293327, EPI_ISL_16302637, EPI_ISL_16327676, EPI_ISL_16364514, EPI_ISL_16368859, EPI_ISL_16368863, EPI_ISL_16368870, EPI_ISL_16377998, EPI_ISL_16378441, EPI_ISL_16381367, EPI_ISL_16381431, EPI_ISL_16382369, EPI_ISL_16382423, EPI_ISL_16382491, EPI_ISL_16390727, EPI_ISL_16402352, EPI_ISL_16431012, EPI_ISL_16440289, EPI_ISL_16449531, EPI_ISL_16449569, EPI_ISL_16465346, EPI_ISL_16465349, EPI_ISL_16475710, EPI_ISL_16485170, EPI_ISL_16496661, EPI_ISL_16499231, EPI_ISL_16507911, EPI_ISL_16510336, EPI_ISL_16513678, EPI_ISL_16524791, EPI_ISL_16527367, EPI_ISL_16528798, EPI_ISL_16534033, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570256, EPI_ISL_16570393, EPI_ISL_16574023, EPI_ISL_16574144, EPI_ISL_16574237, EPI_ISL_16598797, EPI_ISL_16608560, EPI_ISL_16612080, EPI_ISL_16613416, EPI_ISL_16649909, EPI_ISL_16649933, EPI_ISL_16655461, EPI_ISL_16655462, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16660760, EPI_ISL_16660761, EPI_ISL_16665774, EPI_ISL_16665862, EPI_ISL_16668033, EPI_ISL_16668112, EPI_ISL_16668536, EPI_ISL_16668786, EPI_ISL_16702345, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702391, EPI_ISL_16729840, EPI_ISL_16734294, EPI_ISL_16742171, EPI_ISL_16742359, EPI_ISL_16742486, EPI_ISL_16742561, EPI_ISL_16760522, EPI_ISL_16760525, EPI_ISL_16760534, EPI_ISL_16760542, EPI_ISL_16760544, EPI_ISL_16760553, EPI_ISL_16760564, EPI_ISL_16760565, EPI_ISL_16763262, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16825422, EPI_ISL_16835014, EPI_ISL_16845486, EPI_ISL_16847576, EPI_ISL_16860429, EPI_ISL_16860526, EPI_ISL_16861003, EPI_ISL_16870757, EPI_ISL_16879371, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16887308, EPI_ISL_16893748, EPI_ISL_16895710, EPI_ISL_16895713, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907578, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907588, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907598, EPI_ISL_16907601, EPI_ISL_16907602, EPI_ISL_16921497, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943841, EPI_ISL_16943847, EPI_ISL_16943850, EPI_ISL_16943855, EPI_ISL_16943857, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16943896, EPI_ISL_16947614, EPI_ISL_16966278, EPI_ISL_16968369, EPI_ISL_16976030, EPI_ISL_16986705, EPI_ISL_16997282, EPI_ISL_16020838, EPI_ISL_16041723, EPI_ISL_16047006, EPI_ISL_16049706, EPI_ISL_16089016, EPI_ISL_16097810, EPI_ISL_16119629, EPI_ISL_16169012, EPI_ISL_16169559, EPI_ISL_16171387, EPI_ISL_16189719, EPI_ISL_16205980, EPI_ISL_16206468, EPI_ISL_16212482, EPI_ISL_16226002, EPI_ISL_16248743, EPI_ISL_16268007, EPI_ISL_16268596, EPI_ISL_16292066, EPI_ISL_16292121, EPI_ISL_16293327, EPI_ISL_16328978, EPI_ISL_16368859, EPI_ISL_16368863, EPI_ISL_16368870, EPI_ISL_16377998, EPI_ISL_16381431, EPI_ISL_16382369, EPI_ISL_16382423, EPI_ISL_16390727, EPI_ISL_16431012, EPI_ISL_16440289, EPI_ISL_16449525, EPI_ISL_16449531, EPI_ISL_16449569, EPI_ISL_16459438, EPI_ISL_16465329, EPI_ISL_16465341, EPI_ISL_16465346, EPI_ISL_16465349, EPI_ISL_16475710, EPI_ISL_16485170, EPI_ISL_16510336, EPI_ISL_16513678, EPI_ISL_16513739, EPI_ISL_16527367, EPI_ISL_16534033, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570393, EPI_ISL_16574144, EPI_ISL_16574237, EPI_ISL_16598797, EPI_ISL_16607325, EPI_ISL_16612080, EPI_ISL_16612088, EPI_ISL_16613416, EPI_ISL_16638962, EPI_ISL_16649909, EPI_ISL_16649933, EPI_ISL_16655461, EPI_ISL_16655462, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16665774, EPI_ISL_16665862, EPI_ISL_16668112, EPI_ISL_16668536, EPI_ISL_16668786, EPI_ISL_16702345, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702383, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702391, EPI_ISL_16724327, EPI_ISL_16729840, EPI_ISL_16734284, EPI_ISL_16742359, EPI_ISL_16742486, EPI_ISL_16742561, EPI_ISL_16760522, EPI_ISL_16760527, EPI_ISL_16760534, EPI_ISL_16760542, EPI_ISL_16760544, EPI_ISL_16760553, EPI_ISL_16760557, EPI_ISL_16760565, EPI_ISL_16760567, EPI_ISL_16765385, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16835014, EPI_ISL_16837809, EPI_ISL_16847576, EPI_ISL_16860429, EPI_ISL_16861003, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16887308, EPI_ISL_16895710, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16900417, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907578, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907598, EPI_ISL_16907601, EPI_ISL_16907602, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943847, EPI_ISL_16943855, EPI_ISL_16943857, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16943896, EPI_ISL_16947614, EPI_ISL_16976030, EPI_ISL_16986705, EPI_ISL_16997282, EPI_ISL_16097754, EPI_ISL_16119602, EPI_ISL_16119610, EPI_ISL_16119629, EPI_ISL_16171387, EPI_ISL_16204016, EPI_ISL_16212482, EPI_ISL_16307563, EPI_ISL_16357154, EPI_ISL_16368857, EPI_ISL_16369646, EPI_ISL_16377998, EPI_ISL_16382423, EPI_ISL_16390790, EPI_ISL_16402352, EPI_ISL_16440038, EPI_ISL_16441289, EPI_ISL_16449517, EPI_ISL_16457740, EPI_ISL_16465341, EPI_ISL_16475710, EPI_ISL_16483723, EPI_ISL_16496661, EPI_ISL_16510324, EPI_ISL_16524791, EPI_ISL_16527367, EPI_ISL_16529241, EPI_ISL_16536287, EPI_ISL_16547367, EPI_ISL_16549005, EPI_ISL_16570256, EPI_ISL_16574237, EPI_ISL_16606996, EPI_ISL_16607319, EPI_ISL_16607322, EPI_ISL_16607325, EPI_ISL_16656602, EPI_ISL_16656603, EPI_ISL_16660760, EPI_ISL_16660761, EPI_ISL_16670793, EPI_ISL_16702339, EPI_ISL_16702350, EPI_ISL_16702360, EPI_ISL_16702362, EPI_ISL_16702365, EPI_ISL_16702381, EPI_ISL_16702383, EPI_ISL_16702388, EPI_ISL_16702390, EPI_ISL_16702393, EPI_ISL_16702396, EPI_ISL_16733307, EPI_ISL_16734294, EPI_ISL_16760522, EPI_ISL_16760527, EPI_ISL_16760534, EPI_ISL_16760541, EPI_ISL_16760553, EPI_ISL_16760557, EPI_ISL_16760562, EPI_ISL_16760565, EPI_ISL_16823403, EPI_ISL_16823418, EPI_ISL_16823432, EPI_ISL_16845486, EPI_ISL_16870757, EPI_ISL_16882917, EPI_ISL_16882928, EPI_ISL_16883034, EPI_ISL_16883038, EPI_ISL_16883059, EPI_ISL_16883062, EPI_ISL_16883095, EPI_ISL_16883117, EPI_ISL_16883170, EPI_ISL_16895724, EPI_ISL_16895777, EPI_ISL_16907559, EPI_ISL_16907561, EPI_ISL_16907564, EPI_ISL_16907572, EPI_ISL_16907575, EPI_ISL_16907579, EPI_ISL_16907581, EPI_ISL_16907587, EPI_ISL_16907591, EPI_ISL_16907597, EPI_ISL_16907602, EPI_ISL_16915893, EPI_ISL_16940242, EPI_ISL_16940243, EPI_ISL_16943841, EPI_ISL_16943847, EPI_ISL_16943857, EPI_ISL_16943858, EPI_ISL_16943859, EPI_ISL_16943861, EPI_ISL_16943862, EPI_ISL_16943865, EPI_ISL_16943874, EPI_ISL_16943888, EPI_ISL_16943889, EPI_ISL_16993581")
Spike_I794N = stringlist_to_set("EPI_ISL_1407032, EPI_ISL_1646281, EPI_ISL_2221068, EPI_ISL_2661550-2661609, EPI_ISL_2681488-2681550, EPI_ISL_2970972, EPI_ISL_2970981, EPI_ISL_2970989, EPI_ISL_2970996, EPI_ISL_2970999, EPI_ISL_2971051, EPI_ISL_2971057, EPI_ISL_2971077, EPI_ISL_2971085-2971086, EPI_ISL_2971089, EPI_ISL_2971099, EPI_ISL_2971123, EPI_ISL_2971141, EPI_ISL_2971157, EPI_ISL_2971165, EPI_ISL_2996654, EPI_ISL_2996657, EPI_ISL_2996702, EPI_ISL_2996712, EPI_ISL_2996720, EPI_ISL_3373334, EPI_ISL_3613280, EPI_ISL_3613461, EPI_ISL_3613545, EPI_ISL_3613599, EPI_ISL_3613648, EPI_ISL_3613711, EPI_ISL_3613807, EPI_ISL_3614113, EPI_ISL_3759977, EPI_ISL_3801323, EPI_ISL_3923183, EPI_ISL_3998693-3998695, EPI_ISL_3998698-3998700, EPI_ISL_3998702, EPI_ISL_3998704-3998705, EPI_ISL_3998708, EPI_ISL_3998710-3998711, EPI_ISL_3998713-3998714, EPI_ISL_3998716, EPI_ISL_3998718-3998720, EPI_ISL_3998722-3998724, EPI_ISL_3998726, EPI_ISL_3998728, EPI_ISL_3998730-3998735, EPI_ISL_3998737-3998741, EPI_ISL_3998743-3998745, EPI_ISL_3998747-3998748, EPI_ISL_3998751-3998752, EPI_ISL_3998754-3998757, EPI_ISL_3998760-3998768, EPI_ISL_3998771-3998772, EPI_ISL_3998775-3998777, EPI_ISL_3998781, EPI_ISL_3998784, EPI_ISL_4470628-4470629, EPI_ISL_4470633, EPI_ISL_4470646-4470647, EPI_ISL_5350573, EPI_ISL_6566104, EPI_ISL_6596059, EPI_ISL_6596063, EPI_ISL_6596067-6596072, EPI_ISL_6596074-6596076, EPI_ISL_6596079-6596080, EPI_ISL_6596082-6596088, EPI_ISL_6596090-6596091, EPI_ISL_6596094, EPI_ISL_6596096, EPI_ISL_6596098, EPI_ISL_6596100-6596101, EPI_ISL_6596103, EPI_ISL_6596106-6596107, EPI_ISL_6596109, EPI_ISL_6596111-6596117, EPI_ISL_6596119, EPI_ISL_6596122-6596124, EPI_ISL_6596127, EPI_ISL_6596129-6596131, EPI_ISL_6596133, EPI_ISL_6596135, EPI_ISL_6596137, EPI_ISL_6601688, EPI_ISL_6601692-6601693, EPI_ISL_6601695-6601698, EPI_ISL_6601701-6601708, EPI_ISL_6601710, EPI_ISL_6601712-6601713, EPI_ISL_6601715-6601716, EPI_ISL_6601718, EPI_ISL_6601720-6601722, EPI_ISL_6601724, EPI_ISL_6601727-6601731, EPI_ISL_6601733, EPI_ISL_6601735-6601740, EPI_ISL_6601742-6601744, EPI_ISL_6601747-6601748, EPI_ISL_6601750-6601759, EPI_ISL_6601762-6601766, EPI_ISL_6601770, EPI_ISL_6601774, EPI_ISL_6601776, EPI_ISL_6601778-6601780, EPI_ISL_6601782-6601783, EPI_ISL_6601785-6601786, EPI_ISL_6601788, EPI_ISL_6601790-6601793, EPI_ISL_6601795-6601797, EPI_ISL_6601799-6601802, EPI_ISL_6601805, EPI_ISL_6601807-6601808, EPI_ISL_6601810-6601822, EPI_ISL_6601824-6601833, EPI_ISL_6601835, EPI_ISL_6601837-6601839, EPI_ISL_6601841-6601842, EPI_ISL_6601844-6601846, EPI_ISL_6601849, EPI_ISL_6601851-6601858, EPI_ISL_6601860, EPI_ISL_6601862-6601866, EPI_ISL_6601872-6601880, EPI_ISL_6601884-6601886, EPI_ISL_6601891, EPI_ISL_6601894, EPI_ISL_6601899-6601906, EPI_ISL_6601908-6601909, EPI_ISL_6601911-6601912, EPI_ISL_6601914-6601922, EPI_ISL_6601924, EPI_ISL_6601926-6601929, EPI_ISL_6601931-6601932, EPI_ISL_6601935-6601936, EPI_ISL_6601938-6601939, EPI_ISL_6601941-6601944, EPI_ISL_6601948-6601949, EPI_ISL_6601951-6601957, EPI_ISL_6601961-6601965, EPI_ISL_6601969-6601979, EPI_ISL_6601981-6601982, EPI_ISL_6601985-6601991, EPI_ISL_6601993-6602003, EPI_ISL_6602005-6602006, EPI_ISL_6602008, EPI_ISL_6602010, EPI_ISL_6602012-6602021, EPI_ISL_6602023-6602028, EPI_ISL_6602032-6602033, EPI_ISL_6602035-6602036, EPI_ISL_6602038-6602039, EPI_ISL_6602041, EPI_ISL_6603775-6603778, EPI_ISL_6603780-6603782, EPI_ISL_6603784-6603786, EPI_ISL_6603790-6603793, EPI_ISL_6603795-6603800, EPI_ISL_6603802-6603809, EPI_ISL_6603813-6603823, EPI_ISL_6603825-6603829, EPI_ISL_6603831-6603833, EPI_ISL_6603835, EPI_ISL_6603837-6603842, EPI_ISL_6603847, EPI_ISL_6603850-6603857, EPI_ISL_6603860-6603862, EPI_ISL_6603864, EPI_ISL_6603866-6603874, EPI_ISL_6603876-6603880, EPI_ISL_6603882-6603884, EPI_ISL_6603886-6603888, EPI_ISL_6603891-6603894, EPI_ISL_6603896-6603898, EPI_ISL_6603901-6603903, EPI_ISL_6603905-6603906, EPI_ISL_6603911-6603925, EPI_ISL_6603927-6603940, EPI_ISL_6603943, EPI_ISL_6603945-6603949, EPI_ISL_6645158, EPI_ISL_6645161, EPI_ISL_6645169, EPI_ISL_6645175, EPI_ISL_6645183, EPI_ISL_6645192, EPI_ISL_6645196, EPI_ISL_6645211, EPI_ISL_6645216, EPI_ISL_6645223, EPI_ISL_6645228, EPI_ISL_6645246-6645247, EPI_ISL_6645252, EPI_ISL_6645260, EPI_ISL_6645266, EPI_ISL_6645279, EPI_ISL_6645291, EPI_ISL_6645307, EPI_ISL_6645326, EPI_ISL_6645338, EPI_ISL_6645344, EPI_ISL_6645360, EPI_ISL_6645371, EPI_ISL_6645381, EPI_ISL_6645396, EPI_ISL_6645403, EPI_ISL_6645409, EPI_ISL_6645419, EPI_ISL_6645434, EPI_ISL_6645441, EPI_ISL_6645462, EPI_ISL_6645471, EPI_ISL_6645487, EPI_ISL_6645499, EPI_ISL_6645504, EPI_ISL_6645513, EPI_ISL_6645547, EPI_ISL_6645554, EPI_ISL_6645560, EPI_ISL_6645570, EPI_ISL_6645574, EPI_ISL_6645590, EPI_ISL_6645600, EPI_ISL_6645603, EPI_ISL_6645613, EPI_ISL_6645625, EPI_ISL_6645635, EPI_ISL_6645645, EPI_ISL_6645653, EPI_ISL_6645664, EPI_ISL_6645673, EPI_ISL_6645678, EPI_ISL_6645685, EPI_ISL_6645695, EPI_ISL_6645700, EPI_ISL_6645705, EPI_ISL_6645714, EPI_ISL_6645742, EPI_ISL_6645749, EPI_ISL_6645769, EPI_ISL_6645778, EPI_ISL_6645785, EPI_ISL_6645801, EPI_ISL_6645814, EPI_ISL_6645821, EPI_ISL_6645827, EPI_ISL_6645836, EPI_ISL_6645845, EPI_ISL_6645852, EPI_ISL_6645861, EPI_ISL_6645868, EPI_ISL_6645875, EPI_ISL_6645881, EPI_ISL_6645885, EPI_ISL_6645891, EPI_ISL_6645896, EPI_ISL_6645909, EPI_ISL_6645916, EPI_ISL_6645924, EPI_ISL_6645932, EPI_ISL_6645937, EPI_ISL_6645943, EPI_ISL_6645950, EPI_ISL_6645971, EPI_ISL_6645984-6645985, EPI_ISL_6645997, EPI_ISL_6646022, EPI_ISL_6646030, EPI_ISL_6646034, EPI_ISL_6646044, EPI_ISL_6646052, EPI_ISL_6646057, EPI_ISL_6646065, EPI_ISL_6646084, EPI_ISL_6646092, EPI_ISL_6646101, EPI_ISL_6646107, EPI_ISL_6646110, EPI_ISL_6646123, EPI_ISL_6646127, EPI_ISL_6646135, EPI_ISL_6646145, EPI_ISL_6646152, EPI_ISL_6646160, EPI_ISL_6646168, EPI_ISL_6646180, EPI_ISL_6646188-6646189, EPI_ISL_6646199, EPI_ISL_6646211, EPI_ISL_6646224-6646225, EPI_ISL_6711328, EPI_ISL_6711340, EPI_ISL_6711344, EPI_ISL_6711349, EPI_ISL_6711357, EPI_ISL_6711362, EPI_ISL_6711380, EPI_ISL_6711386, EPI_ISL_6711392, EPI_ISL_6711400, EPI_ISL_6711422, EPI_ISL_6711433, EPI_ISL_6711448, EPI_ISL_6711452, EPI_ISL_6711458, EPI_ISL_6711460-6711463, EPI_ISL_6711465-6711467, EPI_ISL_6711469-6711471, EPI_ISL_6711473, EPI_ISL_6711476, EPI_ISL_6711478, EPI_ISL_6711482, EPI_ISL_6711484, EPI_ISL_6711486-6711489, EPI_ISL_6711492-6711494, EPI_ISL_6711497-6711498, EPI_ISL_6711500-6711501, EPI_ISL_6711515-6711516, EPI_ISL_6711527, EPI_ISL_6711533, EPI_ISL_6711537, EPI_ISL_6711547, EPI_ISL_6711562, EPI_ISL_6711569, EPI_ISL_6711571, EPI_ISL_6930232-6930233, EPI_ISL_6954331-6954332, EPI_ISL_6954334-6954337, EPI_ISL_6954339-6954341, EPI_ISL_6954344-6954345, EPI_ISL_6954347, EPI_ISL_6954349, EPI_ISL_6954351-6954354, EPI_ISL_6954356-6954362, EPI_ISL_6954367, EPI_ISL_6954370-6954372, EPI_ISL_6954374-6954377, EPI_ISL_6954379-6954381, EPI_ISL_6954383-6954397, EPI_ISL_6954399, EPI_ISL_6954401-6954409, EPI_ISL_6954411-6954414, EPI_ISL_6954416-6954422, EPI_ISL_6954424-6954425, EPI_ISL_6954427-6954429, EPI_ISL_6954431-6954443, EPI_ISL_6954445-6954449, EPI_ISL_6954451-6954455, EPI_ISL_6954458-6954459, EPI_ISL_6954461-6954470, EPI_ISL_6954472-6954492, EPI_ISL_7239290, EPI_ISL_7239302-7239303, EPI_ISL_7239307, EPI_ISL_7239312, EPI_ISL_7239332, EPI_ISL_7239344, EPI_ISL_7239351, EPI_ISL_7239381-7239382, EPI_ISL_7239408, EPI_ISL_7239420-7239421, EPI_ISL_7239427, EPI_ISL_7239446, EPI_ISL_7239455, EPI_ISL_7239463, EPI_ISL_7239472, EPI_ISL_7239489, EPI_ISL_7239498, EPI_ISL_7239506, EPI_ISL_7239513, EPI_ISL_7239522, EPI_ISL_7239529, EPI_ISL_7239572, EPI_ISL_7239578, EPI_ISL_7239596, EPI_ISL_7239605, EPI_ISL_7239618, EPI_ISL_7239628-7239629, EPI_ISL_7239636, EPI_ISL_7239646, EPI_ISL_7239657, EPI_ISL_7239662, EPI_ISL_7239673, EPI_ISL_7239684-7239685, EPI_ISL_7239694, EPI_ISL_7239701, EPI_ISL_7239708, EPI_ISL_7239721, EPI_ISL_7239738, EPI_ISL_7239756, EPI_ISL_7239761-7239762, EPI_ISL_7338768, EPI_ISL_8476117-8476121, EPI_ISL_8476124-8476125, EPI_ISL_8674922-8674940, EPI_ISL_8674971, EPI_ISL_8676018, EPI_ISL_8676067-8676096, EPI_ISL_10678677, EPI_ISL_10834506, EPI_ISL_11096096, EPI_ISL_11173595, EPI_ISL_11323484, EPI_ISL_12293481-12293484, EPI_ISL_12293486-12293532, EPI_ISL_12293534-12293546, EPI_ISL_12293548, EPI_ISL_12293550-12293572, EPI_ISL_12293574-12293575, EPI_ISL_12293577-12293584, EPI_ISL_12293586-12293607, EPI_ISL_12293609-12293622, EPI_ISL_12293624-12293633, EPI_ISL_12367081, EPI_ISL_12623196-12623201, EPI_ISL_12623203-12623204, EPI_ISL_12623206-12623212, EPI_ISL_12623214-12623218, EPI_ISL_12623220-12623227, EPI_ISL_12623229-12623249, EPI_ISL_12623251-12623252, EPI_ISL_12623254-12623255, EPI_ISL_12623257-12623258, EPI_ISL_12623260-12623263, EPI_ISL_12623266-12623268, EPI_ISL_12623270, EPI_ISL_12771908, EPI_ISL_12784697-12784707, EPI_ISL_12784709-12784711, EPI_ISL_12784713-12784742, EPI_ISL_12784744-12784754, EPI_ISL_12784756-12784766, EPI_ISL_12784768-12784839, EPI_ISL_12784841-12784849, EPI_ISL_12784851-12784863, EPI_ISL_12911909-12911910, EPI_ISL_13773508-13773509, EPI_ISL_13773511-13773512, EPI_ISL_13773515, EPI_ISL_13773518, EPI_ISL_13773520-13773521, EPI_ISL_13774111, EPI_ISL_13774114-13774115, EPI_ISL_13774118-13774119, EPI_ISL_13774121, EPI_ISL_13774123, EPI_ISL_13774127-13774132, EPI_ISL_13774135-13774136, EPI_ISL_13774138-13774141, EPI_ISL_13778407, EPI_ISL_13778414, EPI_ISL_13812267, EPI_ISL_14253453, EPI_ISL_14643049, EPI_ISL_14643088-14643093, EPI_ISL_16680115, EPI_ISL_16762665, EPI_ISL_17546505, EPI_ISL_18032876, EPI_ISL_18936208, EPI_ISL_18936222")
K113N_Q115H = stringlist_to_set("EPI_ISL_16932114, EPI_ISL_16869750, EPI_ISL_16931402, EPI_ISL_16201312, EPI_ISL_16536270, EPI_ISL_16957026, EPI_ISL_16675021, EPI_ISL_16675029, EPI_ISL_16675040, EPI_ISL_16675042, EPI_ISL_16675044, EPI_ISL_16675045, EPI_ISL_16675046, EPI_ISL_16675048, EPI_ISL_16675051, EPI_ISL_16675052, EPI_ISL_16675054, EPI_ISL_16675056, EPI_ISL_16675058, EPI_ISL_16675061, EPI_ISL_16675066, EPI_ISL_16675070, EPI_ISL_16201278, EPI_ISL_16201285, EPI_ISL_16201296, EPI_ISL_16201314, EPI_ISL_16201319, EPI_ISL_16201326, EPI_ISL_16201331, EPI_ISL_16201334, EPI_ISL_16913912, EPI_ISL_16536267, EPI_ISL_16536288, EPI_ISL_16536294, EPI_ISL_16536298, EPI_ISL_16536307, EPI_ISL_16536310, EPI_ISL_16177586, EPI_ISL_16176426, EPI_ISL_16177390, EPI_ISL_16177585, EPI_ISL_16177885, EPI_ISL_16182290, EPI_ISL_16181062, EPI_ISL_16181097, EPI_ISL_16182291, EPI_ISL_16182292, EPI_ISL_16194844, EPI_ISL_16927816, EPI_ISL_16927912, EPI_ISL_16928225, EPI_ISL_16928351, EPI_ISL_16929433, EPI_ISL_16929568, EPI_ISL_16933210, EPI_ISL_16933982, EPI_ISL_16934074, EPI_ISL_16934187, EPI_ISL_16934367, EPI_ISL_16930495, EPI_ISL_16396196, EPI_ISL_16041963, EPI_ISL_16193316, EPI_ISL_16510293, EPI_ISL_16117273, EPI_ISL_16117275, EPI_ISL_16504003, EPI_ISL_16019601, EPI_ISL_16766839, EPI_ISL_16564702, EPI_ISL_16638878, EPI_ISL_16564608, EPI_ISL_16548907, EPI_ISL_16760558, EPI_ISL_16038755, EPI_ISL_16139659, EPI_ISL_16371647, EPI_ISL_16370921, EPI_ISL_16396871, EPI_ISL_16896486, EPI_ISL_16896496, EPI_ISL_16170925, EPI_ISL_16170920, EPI_ISL_16386231, EPI_ISL_16118427, EPI_ISL_16535377, EPI_ISL_16260047, EPI_ISL_16401734, EPI_ISL_16815607, EPI_ISL_16269297, EPI_ISL_16815605, EPI_ISL_16815315, EPI_ISL_16977445, EPI_ISL_672597, EPI_ISL_12293485, EPI_ISL_12293533, EPI_ISL_12293547, EPI_ISL_12293573, EPI_ISL_12293576, EPI_ISL_12293585, EPI_ISL_12293623, EPI_ISL_12623202, EPI_ISL_12623213, EPI_ISL_12623228, EPI_ISL_12623250, EPI_ISL_12623253, EPI_ISL_12623256, EPI_ISL_12623264, EPI_ISL_12623269, EPI_ISL_12784708, EPI_ISL_12784712, EPI_ISL_12784743, EPI_ISL_12784755, EPI_ISL_12784767, EPI_ISL_12955745, EPI_ISL_12955833, EPI_ISL_13135506, EPI_ISL_13612385, EPI_ISL_13774112-13774113, EPI_ISL_13774116-13774117, EPI_ISL_13774120, EPI_ISL_13774122, EPI_ISL_13774124-13774126, EPI_ISL_13774133-13774134, EPI_ISL_13774137, EPI_ISL_13774142-13774143, EPI_ISL_14807247, EPI_ISL_17372592")
mink = stringlist_to_set("EPI_ISL_7082794, EPI_ISL_3494590, EPI_ISL_3494726, EPI_ISL_3496945, EPI_ISL_3496948, EPI_ISL_7082794, EPI_ISL_7083492, EPI_ISL_8619831, EPI_ISL_13107810, EPI_ISL_8174956, EPI_ISL_8189787-8189788, EPI_ISL_13107809, EPI_ISL_13107811, EPI_ISL_13110241, EPI_ISL_8174953-8174955, EPI_ISL_8175130, EPI_ISL_8175132-8175133, EPI_ISL_8514993-8514997, EPI_ISL_8527347-8527353, EPI_ISL_8564587, EPI_ISL_8564590, EPI_ISL_8619819-8619830, EPI_ISL_8786093-8786094, EPI_ISL_10217391-10217398, EPI_ISL_10217400-10217424, EPI_ISL_10437323-10437361, EPI_ISL_10509691, EPI_ISL_10571267, EPI_ISL_10576571, EPI_ISL_10580472, EPI_ISL_11826874-11826875, EPI_ISL_11826908, EPI_ISL_11827547-11827549, EPI_ISL_11827576, EPI_ISL_11827583, EPI_ISL_11827615, EPI_ISL_11827892, EPI_ISL_11827942-11827943, EPI_ISL_11828060, EPI_ISL_11828282, EPI_ISL_11828393, EPI_ISL_13107812-13107816, EPI_ISL_13107818-13107822, EPI_ISL_13110242-13110249, EPI_ISL_13466466-13466470, EPI_ISL_13467028-13467031, EPI_ISL_19096357, EPI_ISL_16811138, EPI_ISL_16811146, EPI_ISL_16811151, EPI_ISL_16811154, EPI_ISL_16994016, EPI_ISL_16994017, EPI_ISL_16994018, EPI_ISL_16994087")
combined_lists = union(MOV, chronics_2025_11_09_update, all_seqs_set, Russian_Kaluga_duplicate, interesting_wastewater, Angola_A, AT1, AV1, B1_243_2, B1_616, B1_632, B1_633, B1_637_1, B1_638, C12, manual, EPI_8mil_9mil, EPI_9mil_10mil, EPI_10mil_11mil, EPI_11mil_12mil, EPI_16mil_17mil, Spike_I794N, K113N_Q115H, Idaho, India, Indonesia, Mexico, Texas, Thailand, Turkey, mink)
baddies = union(Angola_A, AT1, AV1, B1_243_2, B1_616, B1_632, B1_633, B1_637_1, B1_638, C12, manual, EPI_8mil_9mil, EPI_9mil_10mil, EPI_10mil_11mil, EPI_11mil_12mil, EPI_16mil_17mil, Spike_I794N, K113N_Q115H, Idaho, India, Indonesia, Mexico, Texas, Thailand, Turkey, mink)
####################################################################################################################################
####################################################################################################################################
print("\n"^1)
major_VOC_muts = Set(["S:L18F", "S:L176F", "S:Q146K", "S:H445R", "S:F59S", "S:T22N", "S:Q493E", "E:T9I", "M:D3G", "M:Q19E", "M:A63T", "N:P13L", "N:R203K", "N:G204R", "ORF1a:K856R", "ORF1a:L2084I", "ORF1a:T3255I", "ORF1a:P3395H", "ORF1a:I3758V", "ORF1b:P314L", "ORF1b:I1566V", "ORF9b:P10S", "S:A67V", "S:T95I", "S:Y145D", "S:L212I", "S:G339D", "S:S371L", "S:S373P", "S:S375F", "S:K417N", "S:N440K", "S:G446S", "S:S477N", "S:T478K", "S:E484A", "S:Q493R", "S:G496S", "S:Q498R", "S:N501Y", "S:Y505H", "S:T547K", "S:D614G", "S:H655Y", "S:N679K", "S:P681H", "S:N764K", "S:D796Y", "S:N856K", "S:Q954H", "S:N969K", "S:L981F", "E:T9I", "M:Q19E", "M:A63T", "N:P13L", "N:R203K", "N:G204R", "N:S413R", "ORF1a:S135R", "ORF1a:T842I", "ORF1a:G1307S", "ORF1a:L3027F", "ORF1a:T3090I", "ORF1a:T3255I", "ORF1a:P3395H", "ORF1b:P314L", "ORF1b:R1315C", "ORF1b:I1566V", "ORF1b:T2163I", "ORF3a:T223I", "ORF6:D61L", "ORF9b:P10S", "S:T19I", "S:A27S", "S:G142D", "S:V213G", "S:G339D", "S:S371F", "S:S373P", "S:S375F", "S:T376A", "S:D405N", "S:R408S", "S:K417N", "S:N440K", "S:S477N", "S:T478K", "S:E484A", "S:Q493R", "S:Q498R", "S:N501Y", "S:Y505H", "S:D614G", "S:H655Y", "S:N679K", "S:P681H", "S:N764K", "S:D796Y", "S:Q954H", "S:N969K", "E:T9I", "E:T11A", "M:Q19E", "M:A63T", "N:P13L", "N:R203K", "N:G204R", "N:S413R", "ORF1a:K47R", "ORF1a:S135R", "ORF1a:T842I", "ORF1a:G1307S", "ORF1a:L3027F", "ORF1a:T3090I", "ORF1a:L3201F", "ORF1a:T3255I", "ORF1a:P3395H", "ORF1b:P314L", "ORF1b:G662S", "ORF1b:S959P", "ORF1b:R1315C", "ORF1b:I1566V", "ORF1b:T2163I", "ORF3a:T223I", "ORF6:D61L", "ORF9b:P10S", "S:T19I", "S:A27S", "S:V83A", "S:G142D", "S:H146Q", "S:Q183E", "S:V213E", "S:G339H", "S:R346T", "S:L368I", "S:S371F", "S:S373P", "S:S375F", "S:T376A", "S:D405N", "S:R408S", "S:K417N", "S:N440K", "S:V445P", "S:G446S", "S:N460K", "S:S477N", "S:T478K", "S:E484A", "S:F486S", "S:F490S", "S:Q498R", "S:N501Y", "S:Y505H", "S:D614G", "S:H655Y", "S:N679K", "S:P681H", "S:N764K", "S:D796Y", "S:Q954H", "S:N969K", "E:T9I", "M:D3H", "M:Q19E", "M:T30A", "M:A63T", "M:A104V", "N:P13L", "N:R203K", "N:G204R", "N:Q229K", "N:S413R", "ORF1a:S135R", "ORF1a:A211D", "ORF1a:T842I", "ORF1a:V1056L", "ORF1a:G1307S", "ORF1a:K1973R", "ORF1a:N2526S", "ORF1a:L3027F", "ORF1a:T3090I", "ORF1a:T3255I", "ORF1a:P3395H", "ORF1a:V3593F", "ORF1a:T4175I", "ORF1b:P314L", "ORF1b:R1315C", "ORF1b:I1566V", "ORF1b:T2163I", "ORF3a:T223I", "ORF6:D61L", "ORF9b:P10S", "S:T19I", "S:R21T", "S:A27S", "S:S50L", "S:V127F", "S:G142D", "S:F157S", "S:R158G", "S:L212I", "S:V213G", "S:L216F", "S:H245N", "S:A264D", "S:I332V", "S:G339H", "S:K356T", "S:S371F", "S:S373P", "S:S375F", "S:T376A", "S:R403K", "S:D405N", "S:R408S", "S:K417N", "S:N440K", "S:V445H", "S:G446S", "S:N450D", "S:L452W", "S:N460K", "S:S477N", "S:T478K", "S:N481K", "S:E484K", "S:F486P", "S:Q498R", "S:N501Y", "S:Y505H", "S:E554K", "S:A570V", "S:D614G", "S:P621S", "S:H655Y", "S:N679K", "S:P681R", "S:N764K", "S:D796Y", "S:S939F", "S:Q954H", "S:N969K", "S:P1143L", "N:D3L", "N:R203K", "N:G204R", "N:S235F", "ORF1a:T1001I", "ORF1a:A1708D", "ORF1a:I2230T", "ORF1b:P314L", "ORF8:Q27*", "ORF8:R52I", "ORF8:Y73C", "S:N501Y", "S:A570D", "S:D614G", "S:P681H", "S:T716I", "S:S982A", "S:D1118H", "M:I82T", "N:D63G", "N:R203M", "N:D377Y", "ORF1b:P314L", "ORF1b:G662S", "ORF1b:P1000L", "ORF3a:S26L", "ORF7a:V82A", "ORF7a:T120I", "ORF9b:T60A", "S:T19R", "S:G142D", "S:R158G", "S:L452R", "S:T478K", "S:D614G", "S:P681R", "S:D950N"])
other_excluded_muts = Set(["E:S50I"])
S2_muts_set = Set{String}()
ratio_min = 10
#ratio_min2 = 
high_ratio_muts_10_plus = Set{String}()
#high_ratio_muts_15_plus = Set{String}()
for (mut, ratio) in AA_muts_ct_no_dels_no_revs_chr_all_ratio
    mut_tot_ct = AA_muts_ct[mut]
    gene = aa_gene_comprehensive_dict[mut]
    pos = aa_pos_comprehensive_dict[mut]
    if !(mut in major_VOC_muts)
        if ratio ≥ ratio_min && mut_tot_ct ≥ 2 || ratio*mut_tot_ct ≥ 70
            push!(high_ratio_muts_10_plus, mut)
            print("$(mut), ")
        end
    end
end
manual_adds = list_to_set("S:T859I, ORF1a:P1220S, ORF1b:S2339F, ORF1a:P3952S, ORF1b:L820F, ORF1a:S3195G, ORF1a:S2972F, ORF1a:S3158G, ORF1b:F2314L, ORF1a:D2136N, ORF1b:T76I, M:T7I, ORF9b:S6N, ORF1b:D870G, ORF1b:K2340N, S:Q836R, ORF1a:P959L, ORF1b:L1220F, ORF1b:A1219T, S:A688D, ORF1b:S2339P, ORF1a:A239V, ORF1a:C3059S, S:E554D, ORF1a:A3049V, S:D936N, ORF1b:I1181T, S:I1179L, S:D1153V, ORF1a:T1572I, S:I1169F, ORF1a:F3085C, ORF1b:T984N, S:S691P, ORF1b:T2537R, ORF3a:G196E, ORF1a:I1326T, ORF1a:R1566M, E:V5I, N:S105N, ORF1a:T4090N, M:E12D, ORF1a:Q4110R, S:V615I, ORF9b:M1I, M:D215N, ORF1a:I1274T, E:F26S, ORF1a:S1539N, S:I693F, S:T415N, ORF1b:N1540S, S:Y796H, ORF9b:L71S, S:F338L")
for mut in manual_adds 
    push!(high_ratio_muts_10_plus, mut)
end
print("\n"^2)
#println("Length of high_ratio_muts_$(ratio_min)_plus Muts = $(length(high_ratio_muts_10_plus))")
#println("Length of Chronic-Specific S2 Muts = $(length(S2_muts_set))")
print("\n"^2)
#S2_muts_set_sort = sort(collect(S2_muts_set), by = x -> aa_pos_comprehensive_dict[x])
high_ratio_muts_10_plus_sort = sort(collect(high_ratio_muts_10_plus), by = x -> aa_pos_comprehensive_dict[x])
#for mut in S2_muts_set_sort
#    print(mut, ", ")
#end
print("\n"^3)
min_mut_ct = 5
JN1_chk1 = ["S:A435S", "S:R190S", "S:T572I"]
JN1_chk2 = ["S:L441R", "S:K444R", "S:Q493E"]
AAmuts_10_ratio_dict = Dict{String, Int}()
allseq_mut_count = Dict{String, Int}()
for mut in high_ratio_muts_10_plus
    if haskey(AA_muts_seq_all_8000qc_filter, mut)
        for seq in AA_muts_seq_all_8000qc_filter[mut]
            if !(seq in combined_lists)
                if !(seq in AA_muts_seq_all_8000qc_filter["S:R190S"] && seq in AA_muts_seq_all_8000qc_filter["S:K182R"])
                    if !(seq in AA_muts_seq_all_8000qc_filter["S:R190S"] && seq in AA_muts_seq_all_8000qc_filter["S:T572I"])
                        if !(seq in AA_muts_seq_all_8000qc_filter["S:A435S"] && seq in AA_muts_seq_all_8000qc_filter["S:K444R"] && seq in AA_muts_seq_all_8000qc_filter["S:T572I"])
                            if !(seq in AA_muts_seq_all_8000qc_filter["S:R190S"] && seq in AA_muts_seq_all_8000qc_filter["S:K444R"] && seq in AA_muts_seq_all_8000qc_filter["S:A435S"])
                                AAmuts_10_ratio_dict[mut] = get(AAmuts_10_ratio_dict, mut, 0) + 1
                                allseq_mut_count[seq] = get(allseq_mut_count, seq, 0) + 1
                            end
                        end
                    end
                end
            end
        end
    end
end
#if haskey(AA_muts_seq_all_8000qc_filter, mut)
#########################################################################################
total_chr_candidates = length(high_ratio_muts_10_plus)                                 
allseq_mut_count_sort = sort(collect(allseq_mut_count), by = x -> (length(x[1]), x[1]))

new_chr_candidates = 0
for seq____ct in allseq_mut_count_sort
    seq = seq____ct[1]
    ct = seq____ct[2]
    if ct ≥ min_mut_ct
        new_chr_candidates += 1
    end
end
#########################################################################################
date = Dates.format(today(), "yyyy_mm_dd")
open("seqs_with_$(min_mut_ct)muts_chr2circ_ratio_$(ratio_min)_or_above_8000QC_$(new_chr_candidates)seq__$(date_now).txt", "w") do g
    seq10_ct = 0
    for seq____ct in allseq_mut_count_sort
        seq = seq____ct[1]
        ct = seq____ct[2]
        if ct ≥ min_mut_ct
            print(g, "$(seq), ")
            seq10_ct += 1
        end
    end
    print("\n"^2)
    println("Total AA muts with Chr-to-Circ Ratio ≥ $(ratio_min) = $(length(high_ratio_muts_10_plus))")
    println("Total Sequences with ≥$(min_mut_ct) $(ratio_min)+ Ratio Muts = $(seq10_ct)")
    print("\n"^2)
    print(g, "\n"^2)
    println(g, "Total AA muts with Chr-to-Circ Ratio ≥ $(ratio_min) = $(length(high_ratio_muts_10_plus))")
    println(g, "Total Sequences with ≥$(min_mut_ct) $(ratio_min)+ Ratio Muts = $(seq10_ct)")
    print(g, "\n"^2)
    AAmuts_10_ratio_dict_sort = sort(collect(AAmuts_10_ratio_dict), by = x -> x[2], rev=true)
    for mut____ct in AAmuts_10_ratio_dict_sort
        mut = mut____ct[1]
        ct = mut____ct[2]
        if ct > 1
            mut_pad = rpad(mut, 12)
            ctpad = lpad(ct, 4)
            println("$(mut_pad) = $(ctpad)")
            println(g, "$(mut_pad) = $(ctpad)")
        end
    end
    println("\n"^4)
    println(g, "\n"^9)
end
#################################################################

2026_01_03__535PM

S:R214S, ORF7a:N43T, S:N405G, S:N440D, S:K187E, ORF1a:T3058I, S:D339A, S:G838A, S:V952F, S:W152S, S:Y508F, ORF1a:L1368I, ORF1a:A2325V, S:Y28L, ORF1a:G2073K, ORF1b:P2487S, S:D737E, E:N15K, M:N3L, N:R413L, S:S446R, N:G321N, S:P230S, S:N960D, ORF1a:S3732Y, S:L48S, ORF1a:D2650T, ORF1a:T1395I, ORF7a:N43S, E:L27S, ORF1a:N1709R, S:H66Q, S:K854R, ORF1a:A1420G, S:V551I, ORF1a:N1709H, S:K986T, E:S50C, S:S982L, ORF1b:D824S, ORF6:Q56*, S:T299I, ORF3a:I118A, S:V327A, ORF1b:N682K, ORF1b:L986V, S:E988K, ORF9b:V94L, S:V382A, S:A570G, S:E619V, ORF1a:D448A, S:C136R, ORF1a:S2972P, ORF1b:S826A, S:N388S, ORF1a:T1638V, ORF1a:V2943L, ORF1a:F143M, ORF1a:Q4099H, ORF1a:Q1365L, ORF8:V117L, ORF1a:E172R, M:H148R, S:L981V, S:D737A, ORF1b:L2019I, S:L977F, ORF1b:D824E, M:F28C, S:I68Y, ORF1a:L3641I, ORF1b:I1074T, S:N657T, ORF1a:E93K, ORF1a:P2899Q, S:D294N, S:A376V, S:M740I, S:N417H, ORF1b:K2340E, ORF1a:E2468D, ORF9b:D89G, ORF1b:R1084G, S:K378R, S:S640P, M:A38P, M:F28V, ORF1a:E3441N, ORF1b:I1074M, OR

In [123]:
### ORF9b stop codon counts, all HQCS sequences ############
for (mut, ct) in AA_muts_ct_all
    if mut_gene(mut) == "ORF9b" && mut[end] == '*'
        println("$(mut) = $(ct)")
    end
end

ORF9b:E86* = 273
ORF9b:E65* = 327
ORF9b:E7* = 114
ORF9b:Q20* = 408
ORF9b:Q77* = 541
ORF9b:Q34* = 322
ORF9b:E77* = 3


In [19]:
### Finds all EPCI sequences with a given mutations and counts up all the private mutations in those sequences; sorts & prints results
#### Fx: AA_mutation_sequences_and_mutations(mutation::String, min_ct::Int) ################################
# 20I, 21A, 21B, 21C, 21D, 21E, 21F, 21G, 21H, 21I, 21J, 21K, 21L, 21M
AA_gene_sortKey(n) = (mut_gene_Dict[aa_gene_comprehensive_dict[n[1]]], aa_pos_comprehensive_dict[n[1]])
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function AA_mutation_sequences_and_mutations(mutation::String, min_ct::Int)
    mutation_pos_only = aa_gene_and_pos_comprehensive_dict[mutation]
    date = Dates.format(today(), "yyyy_mm_dd")
    AA_cts = Dict{String, Int}()
    AA_cts_no_dels = Dict{String, Int}()
    AA_cts_pos_only = Dict{String, Int}()
    AA_cts_pos_only_no_dels = Dict{String, Int}()
#############################################################################################################
    allMutSeqs = Vector{String}()
    allMutSeqs_pos_only = Vector{String}()
    repGrpsSeqs = Vector{Vector{String}}()
    repGrpsSeqs_pos_only = Vector{Vector{String}}()
    for (grpNum, mutSet) in rep_seq_grps_maxmut_AA
        if mutation in mutSet
            for AAmut in mutSet
                if haskey(AA_cts, AAmut)
                    AA_cts[AAmut] += 1
                else
                    AA_cts[AAmut] = 1
                end
                if AAmut[end] ≠ '-'
                    if haskey(AA_cts_no_dels, AAmut)
                        AA_cts_no_dels[AAmut] += 1
                    else
                        AA_cts_no_dels[AAmut] = 1
                    end
                end
            end
        end
        seq_arr = Vector{String}()
        for seq in rep_seq_grps_seqs[grpNum]
            if mutation in seq_AA_muts[seq]
                push!(seq_arr, seq)
                push!(allMutSeqs, seq)
            end
        end
        if !(isempty(seq_arr))
            seq_arr_sort = sort(seq_arr, by = x -> (length(x), x))
            push!(repGrpsSeqs, seq_arr_sort)
        end
        mutSet_pos_only = Set{String}()
############## mutpos_mut_dict records the full mutations (e.g., "S:P330S") that correspond to each pos_only mut (e.g., S:330) in each sequence
        mutpos_mut_dict = Dict{String, String}(m=>"" for m in mutSet)
        for mut in mutSet
            mutpos = aa_gene_and_pos_comprehensive_dict[mut]
            push!(mutSet_pos_only, mutpos)
            mutpos_mut_dict[mutpos] = mut
        end
        seq_arr_pos_only = Vector{String}()
        if mutation_pos_only in mutSet_pos_only
            for AAmutpos in mutSet_pos_only
                if haskey(AA_cts_pos_only, AAmutpos)
                    AA_cts_pos_only[AAmutpos] += 1
                else
                    AA_cts_pos_only[AAmutpos] = 1
                end
                OG_mut = mutpos_mut_dict[AAmutpos]
                if OG_mut[end] ≠ '-'
                    if haskey(AA_cts_pos_only_no_dels, AAmutpos)
                        AA_cts_pos_only_no_dels[AAmutpos] += 1
                    else
                        AA_cts_pos_only_no_dels[AAmutpos] = 1
                    end
                end
            end
        end
        for seq in rep_seq_grps_seqs[grpNum]
            if mutation_pos_only in seq_AA_muts_pos_only_no_dels[seq]
                push!(seq_arr_pos_only, seq)
                push!(allMutSeqs_pos_only, seq)
            end
            if mutation in seq_AA_muts_pos_only_no_dels[seq]
                push!(seq_arr_pos_only_no_dels, seq)
                push!(allMutSeqs_pos_only_no_dels, seq)
            end
        end
        if !(isempty(seq_arr_pos_only))
            seq_arr_pos_only_sort = sort(seq_arr_pos_only, by = x -> (length(x), x) )
            push!(repGrpsSeqs_pos_only, seq_arr_pos_only_sort)
        end
    end
    repGrpsSeqs_sort = sort(repGrpsSeqs, by = x -> length(x) )
    repGrpsSeqs_pos_only_sort = sort(repGrpsSeqs_pos_only, by = x -> length(x) )
#############################################################################################################
    nonRepSeq_arr = Vector{String}()
    nonRepSeq_arr_pos_only = Vector{String}()
    for seq in non_rep_seqs
        mutSet = seq_AA_muts[seq]
        mutSet_pos_only = seq_AA_muts_pos_only[seq]
        if mutation in mutSet
            push!(nonRepSeq_arr, seq)
            push!(allMutSeqs, seq)
            for AAmut in mutSet
                if haskey(AA_cts, AAmut)
                    AA_cts[AAmut] += 1
                else
                    AA_cts[AAmut] = 1
                end
                if AAmut[end] ≠ '-'
                    if haskey(AA_cts_no_dels, AAmut)
                        AA_cts_no_dels[AAmut] += 1
                    else
                        AA_cts_no_dels[AAmut] = 1
                    end
                end
            end
        end
        if mutation_pos_only in mutSet_pos_only
            push!(nonRepSeq_arr_pos_only, seq)
            push!(allMutSeqs_pos_only, seq)
            for AAmutpos in mutSet_pos_only
                if haskey(AA_cts_pos_only, AAmutpos)
                    AA_cts_pos_only[AAmutpos] += 1
                else
                    AA_cts_pos_only[AAmutpos] = 1
                end
                if AAmutpos[end] ≠ '-'
                    if haskey(AA_cts_pos_only_no_dels, AAmutpos)
                        AA_cts_pos_only_no_dels[AAmutpos] += 1
                    else
                        AA_cts_pos_only_no_dels[AAmutpos] = 1
                    end
                end
            end
        end
    end
    nonRepSeq_arr_sort = sort(nonRepSeq_arr, by = x -> (length(x), x) )
    allMutSeqs_sort = sort(allMutSeqs, by = x -> (length(x), x) )
    nonRepSeq_arr_pos_only_sort = sort(nonRepSeq_arr_pos_only, by = x -> (length(x), x) )
    allMutSeqs_pos_only_sort = sort(allMutSeqs_pos_only, by = x -> (length(x), x) )
#############################################################################################################
    prop_dict = Dict{String, Tuple{Float64, Float64, Float64, Int}}()
    for (AAmut, ct) in AA_cts
        qry_prop = round(digits=1, 100*ct/AA_muts_ct[AAmut])
        ref_prop = round(digits=1, 100*ct/AA_muts_ct[mutation])
        total_prop = round(digits=1, ref_prop + qry_prop)
        prop_dict[AAmut] = (qry_prop, ref_prop, total_prop, ct)
    end
############################################################################################################# 
    prop_dict_pos_only = Dict{String, Tuple{Float64, Float64, Float64, Int}}()
    mutation_pos_only_total = AA_muts_ct_pos_only[mutation_pos_only]
    for (AAmutPos, ct) in AA_cts_pos_only
        qry_prop = round(digits=1, 100*ct/AA_muts_ct_pos_only[AAmutPos])
        ref_prop = round(digits=1, 100*ct/AA_muts_ct_pos_only[mutation_pos_only])
        total_prop = round(digits=1, ref_prop + qry_prop)
        props = (qry_prop, ref_prop, total_prop, ct)
        prop_dict_pos_only[AAmutPos] = props
    end
############################################################################################################# 
    mut_gene(n) = split(n, ":")[1]
    mut1(n) = mut_gene(n[1])
    mut_only(n) = split(n, ":")[2]
    mut_numPre(n) = mut_only(n)[2:end-1]
    gene___AAnum_vec(n) = [mut_gene(n), aa_pos_comprehensive_dict[n]]
    mut_num_pos_only(n) = aa_pos_comprehensive_dict[n]
    gene_num___AApos_only_vec(n) = [mut_gene(n), aa_pos_comprehensive_dict[n]]
    mut_gene_Dict = Dict{String, Int}("ORF1a"=>1, "ORF1b"=>2, "S"=>3, "E"=>4, "M"=>5, "N"=>6, "ORF3a"=>7, "ORF6"=>8, "ORF7a"=>9, "ORF7b"=>10, "ORF8"=>11, "ORF9b"=>12)
    AA_gene_sortKey(n) = (mut_gene_Dict[mut_gene(n[1])], aa_pos_comprehensive_dict[n[1]])
    AA_ct_sortKey1(n) = (1000÷mut_gene_Dict[mut_gene(n[1])], aa_pos_comprehensive_dict[n[1]])
    AA_ct_sortKey2(n) = (n[2], AA_ct_sortKey1(n))
    AA_gene_pos_sortKey(n) = (mut_gene_Dict[mut_gene(n[1])], aa_pos_comprehensive_dict[n[1]])
    AA_ct_pos_sortKey1(n) = (1000÷mut_gene_Dict[mut_gene(n[1])], aa_pos_comprehensive_dict[n[1]])
    AA_ct_pos_sortKey2(n) = (n[2], AA_ct_pos_sortKey1(n))
    AA_cts_gene_sort = sort(collect(AA_cts), by = x -> AA_gene_sortKey(x))
    AA_cts_count_sort = sort(collect(AA_cts), by = x -> AA_ct_sortKey2(x), rev=true)
    AA_cts_pos_only_gene_sort = sort(collect(AA_cts_pos_only), by = x -> AA_gene_pos_sortKey(x))
    AA_cts_pos_only_count_sort = sort(collect(AA_cts_pos_only), by = x -> AA_ct_pos_sortKey2(x), rev=true)
    # prop_dict = Dict{String, Tuple{Float64, Float64, Float64, Int}}()
    # prop_dict form = prop_dict[AAmutPos] = [refProp, qryProp, totProp, ct]
    prop_dict_sort_Count = sort(collect(prop_dict), by = x -> (AA_cts[x[1]], x[2][1]), rev=true)
#    prop_dict_sort_TotPerc = sort(collect(prop_dict), by = x -> (x[2][3], x[2][4]), rev=true)
    prop_dict_sort_QryPerc = sort(collect(prop_dict), by = x -> (x[2][1], x[2][4]), rev=true)
    prop_dict_sort_gene = sort(collect(prop_dict), by = x -> AA_gene_sortKey(x))
    
    prop_dict_pos_only_sort_Count = sort(collect(prop_dict_pos_only), by = x -> AA_cts_pos_only[x[1]], rev=true)
#    prop_dict_pos_only_sort_TotPerc = sort(collect(prop_dict_pos_only), by = x -> (x[2][3], x[2][4]), rev=true)
    prop_dict_pos_only_sort_QryPerc = sort(collect(prop_dict_pos_only), by = x -> (x[2][1], x[2][4]), rev=true)
    prop_dict_pos_only_sort_gene = sort(collect(prop_dict_pos_only), by = x -> AA_gene_pos_sortKey(x))
    
    rep_seq_grps_tot = length(keys(rep_seq_grps_seqs))
    singlet_tot = length(non_rep_seqs)
    singlet_grps_tot = singlet_tot + rep_seq_grps_tot
    singlets = 0
    for seq in non_rep_seqs
        if mutation in seq_AA_muts[seq]
            singlets += 1
        end
    end
    maxmuts = 0
    for (i, mutSet) in rep_seq_grps_maxmut_AA
        if mutation in mutSet
            maxmuts += 1
        end
    end
    groups = 0
    for (i, mutSet) in rep_seq_grps_AA
        if mutation in mutSet
            for seq in rep_seq_grps_seqs[i]
                groups += 1
            end
        end
    end
    singlets_plus_grps = singlets + groups
    singlets_plus_maxmuts = singlets + maxmuts

    mutPerc = 0.0
    mutPerc_pos_only = 0.0
    if haskey(AA_muts_ct, mutation) 
        mutPerc = round(digits=1, 100*AA_muts_ct[mutation]/singlet_grps_tot)
    end
    if haskey(AA_muts_ct_pos_only, mutation_pos_only) 
        mutPerc_pos_only = round(digits=1, 100*AA_muts_ct_pos_only[mutation_pos_only]/singlet_grps_tot)
    end

    padd = 15
    seq_title_pad = 24
    clade_title_pad = 5
##############################################################################################################
##############################################################################################################
##############################################################################################################
##############################################################################################################
##############################################################################################################
    open("mutation_analysis_output/$(mutation)_seqs_with_priv_muts_PERCENTS_$(date)_minCt$(min_ct).txt", "w") do g
        if haskey(AA_muts_ct, mutation)
            println("Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
            println(g, "Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
        end
        if haskey(AA_muts_ct_pos_only, mutation_pos_only)
            println("Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
            println(g, "Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
        end
        println()
        println(g)
        println(g, "Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println("Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println("Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
        println(g, "Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
        println()
        println(g)
        println("##################################**** Repeat (Related) Sequence Groups ##################################****")
        println(g, "##################################**** Repeat (Related) Sequence Groups ##################################****")
        for grp in repGrpsSeqs_sort
            for EPI in grp
                print("$(EPI), ")
                print(g, "$(EPI), ")
            end
            println()
            println(g)
        end
        print("\n"^2)
        print(g, "\n"^2)
        println("#############################################* Sequences with $(mutation) UNIQUE ONLY #############################################*")
        println(g, "#############################################* Sequences with $(mutation) UNIQUE ONLY #############################################*")
        for seq in allMutSeqs_sort
            if seq in all_unique_chr_seqs
                print(seq, ", ")
                print(g, seq, ", ")
            end
        end
        print("\n"^2)
        print(g, "\n"^2)
        println("###################################################* Sequences with $(mutation) v2 ###################################################*")
        println(g, "###################################################* Sequences with $(mutation) v2 ###################################################*")
        if haskey(AA_muts_seq, mutation)
            for seq in AA_muts_seq[mutation]
                print(seq, ", ")
                print(g, seq, ", ")
                if !(seq in allMutSeqs)
                    println("Sequence in AA_muts_seq[mutation] but not in allMutSeqs = $seq")
                end
            end
        end
        print("\n"^2)
        print(g, "\n"^2)
        pango_dict_for_mut = Dict{String, Int}()
        clade_dict_for_mut = Dict{String, Int}()
        clade_dict_for_mut_ADJ = Dict{String, Float64}()
        clade_ADJ_factor_dict = Dict{String, Float64}()
        for clade in clade_set_complete
            if !haskey(seq_clade_ct, clade)
                seq_clade_ct[clade] = 0
            end
        end
        for clade in clade_set_complete
            if seq_clade_ct[clade] == 0
                clade_ADJ_factor_dict[clade] = 0
            end
            if seq_clade_ct[clade] ≠ 0
                clade_ADJ_factor_dict[clade] = 1000/seq_clade_ct[clade]
            end
        end
        for clade in clade_set_complete
            clade_dict_for_mut[clade] = 0
        end
        if haskey(AA_muts_seq, mutation)
            for seq in AA_muts_seq[mutation]
                if seq in non_rep_seqs
                    if haskey(pango_dict_for_mut, seq_pango[seq])
                        pango_dict_for_mut[seq_pango[seq]] += 1
                    else
                        pango_dict_for_mut[seq_pango[seq]] = 1
                    end
                end
            end
            for seq in AA_muts_seq[mutation]
                if seq in non_rep_seqs
                    if haskey(clade_dict_for_mut, seq_clade[seq])
                        clade_dict_for_mut[seq_clade[seq]] += 1
                    else
                        clade_dict_for_mut[seq_clade[seq]] = 1
                    end
                end
            end
        end
#############################################################################################################
############################################################################################################# 
        for (grp_num, clade) in rep_seq_grps_clade
            if mutation in rep_seq_grps_maxmut_AA[grp_num]
                if haskey(clade_dict_for_mut, clade)
                    clade_dict_for_mut[clade] += 1
                else
                    clade_dict_for_mut[clade] = 1
                end
            end
        end
        for (grp_num, pango) in rep_seq_grps_pango
            if mutation in rep_seq_grps_maxmut_AA[grp_num]
                if haskey(pango_dict_for_mut, pango)
                    pango_dict_for_mut[pango] += 1
                else
                    pango_dict_for_mut[pango] = 1
                end
            end
        end

        for clade in clade_set_complete
            clade_dict_for_mut_ADJ[clade] = 0
        end
        for clade in clade_set_complete
            if haskey(clade_dict_for_mut_ADJ, clade)
                clade_dict_for_mut_ADJ[clade] = round(digits=1, clade_dict_for_mut[clade]*clade_ADJ_factor_dict[clade])
            else
                clade_dict_for_mut_ADJ[clade] = 0
            end
        end
        pango_dict_for_mut_sort_by_ct = sort(collect(pango_dict_for_mut), by = x -> x[2], rev=true)
        pango_dict_for_mut_sort_by_pango = sort(collect(pango_dict_for_mut), by = x -> x[1])
        clade_dict_for_mut_sort_by_ct = sort(collect(clade_dict_for_mut), by = x -> x[2], rev=true)
        clade_dict_for_mut_sort_by_clade = sort(collect(clade_dict_for_mut), by = x -> x[1])
        clade_dict_ADJ_for_mut_sort_by_ct = sort(collect(clade_dict_for_mut_ADJ), by = x -> x[2], rev=true)
        clade_dict_ADJ_for_mut_sort_by_clade = sort(collect(clade_dict_for_mut_ADJ), by = x -> x[1])
        ct_str = ""
        if haskey(AA_muts_ct, mutation)
            ct_str = string(AA_muts_ct[mutation])
        end
        len = length(ct_str)
#############################################################################################################
#############################################################################################################
        pango_dict_for_mut_pos_only = Dict{String, Int}()
        clade_dict_for_mut_pos_only = Dict{String, Int}()
        clade_dict_for_mut_pos_only_ADJ = Dict{String, Float64}()
        for clade in clade_set_complete
            clade_dict_for_mut_pos_only[clade] = 0
        end
        for seq in AA_muts_seq_pos_only[mutation_pos_only]
            if seq in non_rep_seqs
                if haskey(pango_dict_for_mut_pos_only, seq_pango[seq])
                    pango_dict_for_mut_pos_only[seq_pango[seq]] += 1
                else
                    pango_dict_for_mut_pos_only[seq_pango[seq]] = 1
                end
            end
        end
        for seq in AA_muts_seq_pos_only[mutation_pos_only]
            if seq in non_rep_seqs
                if haskey(clade_dict_for_mut_pos_only, seq_clade[seq])
                    clade_dict_for_mut_pos_only[seq_clade[seq]] += 1
                else
                    clade_dict_for_mut_pos_only[seq_clade[seq]] = 1
                end
            end
        end
#################################################### 
        for (grp_num, clade) in rep_seq_grps_clade
            if mutation_pos_only in rep_seq_grps_maxmut_AA_pos_only[grp_num]
                if haskey(clade_dict_for_mut_pos_only, clade)
                    clade_dict_for_mut_pos_only[clade] += 1
                else
                    clade_dict_for_mut_pos_only[clade] = 1
                end
            end
        end
        for (grp_num, pango) in rep_seq_grps_pango
            if mutation_pos_only in rep_seq_grps_maxmut_AA_pos_only[grp_num]
                if haskey(pango_dict_for_mut_pos_only, pango)
                    pango_dict_for_mut_pos_only[pango] += 1
                else
                    pango_dict_for_mut_pos_only[pango] = 1
                end
            end
        end
        for clade in clade_set_complete
            if haskey(clade_dict_for_mut_pos_only_ADJ, clade)
                clade_dict_for_mut_pos_only_ADJ[clade] = round(digits=1, clade_dict_for_mut_pos_only[clade]*clade_ADJ_factor_dict[clade])
            else
                clade_dict_for_mut_pos_only_ADJ[clade] = 0
            end
        end
############################################################################################################# 
        pango_dict_for_mut_pos_only_sort_by_ct = sort(collect(pango_dict_for_mut_pos_only), by = x -> x[2], rev=true)
        pango_dict_for_mut_pos_only_sort_by_pango = sort(collect(pango_dict_for_mut_pos_only), by = x -> x[1])
        clade_dict_for_mut_pos_only_sort_by_ct = sort(collect(clade_dict_for_mut_pos_only), by = x -> x[2], rev=true)
        clade_dict_for_mut_pos_only_sort_by_clade = sort(collect(clade_dict_for_mut_pos_only), by = x -> x[1])
        clade_dict_ADJ_for_mut_pos_only_sort_by_ct = sort(collect(clade_dict_for_mut_pos_only_ADJ), by = x -> x[2], rev=true)
        clade_dict_ADJ_for_mut_pos_only_sort_by_clade = sort(collect(clade_dict_for_mut_pos_only_ADJ), by = x -> x[1])
############################################################################################################# 
#############################################################################################################
        println("###################################################* Clade/Pango Count for $(mutation), sorted by Clade ###################################################*")
        println(g, "###################################################* Clade/Pango Count for $(mutation), sorted by Clade ###################################################*")
            pct_seq_title = lpad("| Seqs % |", seq_title_pad)
            pct_clade_title = lpad(" Clade % |", clade_title_pad)
            println("Clade/Pango", pct_seq_title, pct_clade_title)    
        for w in clade_dict_for_mut_sort_by_clade
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = 0.0
            if haskey(AA_muts_ct, mutation)
                clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/AA_muts_ct[mutation])
            end
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango) ", padd)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad("$(clade_ct)", 2)
                fracBot = rpad("$(AA_muts_ct[mutation])", 3)
                fract = "$(fracTop)/$(fracBot)"
                println("$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)  |")
                println(g, "$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)  |")
            end
        end
        println()
        println(g)
        println("###################################################* Clade/Pango Count for $(mutation), sorted by Count ###################################################*")
        println(g, "###################################################* Clade/Pango Count for $(mutation), sorted by Count ###################################################*")
            pct_seq_title = lpad("| Seqs % |", seq_title_pad)
            pct_clade_title = lpad(" Clade % |", clade_title_pad)
            println("Clade/Pango", pct_seq_title, pct_clade_title)    
        for w in clade_dict_for_mut_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = 0.0
            if haskey(AA_muts_ct, mutation)
                clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/AA_muts_ct[mutation])
            end
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango) ", padd)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad("$(clade_ct)", 2)
                fracBot = lpad("$(AA_muts_ct[mutation])", 1)
                fract = "$(fracTop)/$(fracBot)"
                println("$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
                println(g, "$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
            end
        end
        println()
        println(g)
        println("###################################################* Normalized Clade/Pango Count for $(mutation), sorted by Count ###################################################*")
        println(g, "###################################################* Normalized Clade/Pango Count for $(mutation), sorted by Count ###################################################*")
        for w in clade_dict_ADJ_for_mut_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            if clade_ct > 0
                println(rpad("$(clade)|$(pango)", padd) * " = $(clade_ct) [$(clade_dict_for_mut[clade])]")
                println(g, rpad("$(clade)|$(pango)", padd) * " = $(clade_ct) [$(clade_dict_for_mut[clade])]")
            end
        end
        println()
        println(g)
        println("########################################################################################################################################################################################**")
        println("#################***** Mutation cts for seq with $(mutation) Sorted by Count #################*****")
        println("########################################################################################################################################################################################**")
        println()
        println(rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(rpad("Total number of Independent Chronic Singlets/Groups = ", 55) * "$(singlet_grps_tot)")
        println()
        if haskey(AA_muts_ct, mutation)
            println("Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
        else
            println("Total Number of $(mutation) seqs = 0")
        end
        println("Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println()
        println(g, "########################################################################################################################################################################################**")
        println(g, "#################***** Mutation cts for seq with $(mutation) Sorted by Count #################*****")
        println(g, "########################################################################################################################################################################################**")
        println(g)
        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
        println(g)
        if haskey(AA_muts_ct, mutation)
            println(g, "Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
        else
            println(g, "Total Number of $(mutation) seqs = 0")
        end
        println(g, "Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println(g)
        for m in prop_dict_sort_Count
            if AA_cts[m[1]] ≥ min_ct
                m_perc_of_all = round(digits=1, 100*AA_muts_ct[m[1]]/singlet_grps_tot)
                aamut = rpad("$(m[1])", padd)
#                totPerc = lpad("$(m[2][3])%", 6)
                fracTop = lpad("$(AA_cts[m[1]])", 2)
                fracBot = lpad("$(AA_muts_ct[m[1]])", 3)
                fracBot2 = lpad("$(AA_muts_ct[mutation])", 3)
                qryPerc = lpad("$(m[2][1])%", 6)
                refPerc = lpad("$(m[2][2])%", 6)
                mPercAll = lpad("$(m_perc_of_all)%", 5)
                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
                println(g, "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
            end
        end 
        print("\n"^2)
        print(g, "\n"^2)
#############################################################################################################
#        println("########################################################################################################################################################################################**")
#        println("#################***** Mutation cts for seq with $(mutation) Sorted by QRY PERCENTS #################*****")
#        println("########################################################################################################################################################################################**")
#        println()
#        println(rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
#        println(rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
#        println(rpad("Total number of Independent Chronic Singlets/Groups = ", 55) * "$(singlet_grps_tot)")
#        println()
#        if haskey(AA_muts_ct, mutation)
#            println("Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
#        else
#            println("Total Number of $(mutation) seqs = 0")
#        end
#        println("Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
#        println()
#        println(g, "########################################################################################################################################################################################**")
#        println(g, "#################***** Mutation cts for seq with $(mutation) Sorted by QRY PERCENTS #################*****")
#        println(g, "########################################################################################################################################################################################**")
#        println(g)
#        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
#        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
#        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
#        println(g)
#        if haskey(AA_muts_ct, mutation)
#            println(g, "Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
#        else
#            println(g, "Total Number of $(mutation) seqs = 0")
#        end
#        println(g, "Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
#        println(g)
#        for m in prop_dict_sort_QryPerc
#            if AA_cts[m[1]] ≥ min_ct
#                m_perc_of_all = round(digits=1, 100*AA_muts_ct[m[1]]/singlet_grps_tot)
#                aamut = rpad("$(m[1])", 13)
#                qryPerc = lpad("$(m[2][1])%", 6)
#                fracTop = lpad("$(AA_cts[m[1]])", 2)
#                fracBot = lpad("$(AA_muts_ct[m[1]])", 3)
#                fracBot2 = lpad("$(AA_muts_ct[mutation])", 3)
#                #fraction = lpad("$(AA_cts[m[1]])/$(AA_muts_ct[m[1]])", 6)
#                refPerc = lpad("$(m[2][2])%", 6)
#                mPercAll = lpad("$(m_perc_of_all)%", 5)
#                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
#                println(g, "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
#            end 
#        end
#        print("\n"^2)
#        print(g, "\n"^2)
#############################################################################################################
#############################################################################################################
        println("########################################################################################################################################################################################**")
        println("#################**** Mutation cts for seq with $(mutation) Sorted by Gene & Position #################*****")
        println("########################################################################################################################################################################################**")
        println()
        println(rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(rpad("Total number of Independent Chronic Singlets/Groups = ", 55) * "$(singlet_grps_tot)")
        println()
        if haskey(AA_muts_ct, mutation)
            println("Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
        else
            println("Total Number of $(mutation) seqs = 0")
        end
        println("Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println()
        println(g, "########################################################################################################################################################################################**")
        println(g, "#################**** Mutation cts for seq with $(mutation) Sorted by Gene & Position #################*****")
        println(g, "########################################################################################################################################################################################**")
        println(g)
        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
        println(g)
        if haskey(AA_muts_ct, mutation)
            println(g, "Total Number of $(mutation) seqs = $(AA_muts_ct[mutation])")
        else
            println(g, "Total Number of $(mutation) seqs = 0")
        end
        println(g, "Total Percentage of all Chronics with $(mutation) = $(mutPerc)%")
        println(g)
        for m in prop_dict_sort_gene
            if AA_cts[m[1]] ≥ min_ct
                m_perc_of_all = round(digits=1, 100*AA_muts_ct[m[1]]/singlet_grps_tot)
                aamut = rpad("$(m[1])", padd)
#                totPerc = lpad("$(m[2][3])%", 6)
                fracTop = lpad("$(AA_cts[m[1]])", 2)
                fracBot = lpad("$(AA_muts_ct[m[1]])", 3)
                fracBot2 = lpad("$(AA_muts_ct[mutation])", 3)
                qryPerc = lpad("$(m[2][1])%", 6)
                refPerc = lpad("$(m[2][2])%", 6)
                mPercAll = lpad("$(m_perc_of_all)%", 5)
                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
                println(g, "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
           end
        end 
        print("\n"^2)
        print(g, "\n"^2)
#############################################################################################################
############################################################################################################# 
        println("##################################**** Repeat (Related) Sequence Groups ##################################****")
        println(g, "##################################**** Repeat (Related) Sequence Groups ##################################****")
        for grp in repGrpsSeqs_pos_only
            for EPI in grp
                print("$(EPI), ")
                print(g, "$(EPI), ")
            end
            println()
            println(g)
        end
        print("\n"^2)
        print(g, "\n"^2)
        println("#############################################* Sequences with $(mutation_pos_only) UNIQUE ONLY #############################################*")
        println(g, "#############################################* Sequences with $(mutation_pos_only) UNIQUE ONLY #############################################*")
        for seq in allMutSeqs_pos_only_sort
            if seq in all_unique_chr_seqs
                print(seq, ", ")
                print(g, seq, ", ")
            end
        end
        print("\n"^2)
        print(g, "\n"^2)
        println("###################################################* Sequences with $(mutation_pos_only) ###################################################*")
        println(g, "###################################################* Sequences with $(mutation_pos_only) ###################################################*")
        for seq in AA_muts_seq_pos_only[mutation_pos_only]
            print(seq, ", ")
            print(g, seq, ", ")
            if AA_muts_seq_pos_only[mutation_pos_only] == ""
                println("Empty Key in AA_muts_seq_pos_only[mutation_pos_only]")
            end
            if !(seq in allMutSeqs_pos_only)
                println("Sequence in AA_muts_seq_pos_only[mutation] but not in allMutSeqs_pos_only = $seq")
            end
        end
        print("\n"^2)
        print(g, "\n"^2)
#############################################################################################################
        ct_str = ""
        if haskey(AA_muts_ct, mutation)
            ct_str = string(AA_muts_ct[mutation])
        end
        len = length(ct_str)
        println("###################################################* Clade/Pango Count for $(mutation_pos_only), sorted by Clade ###################################################*")
        println(g, "###################################################* Clade/Pango Count for $(mutation_pos_only), sorted by Clade ###################################################*")
        for w in clade_dict_for_mut_pos_only_sort_by_clade
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/AA_muts_ct_pos_only[mutation_pos_only])
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango) ", padd)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad("$(clade_ct)", 2)
                fracBot = lpad("$(AA_muts_ct_pos_only[mutation_pos_only])", len)
                fract = "$(fracTop)/$(fracBot)"
                println("$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
                println(g, "$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
            end
        end
        println()
        println(g)
        println("############################################* Clade/Pango Count for $(mutation_pos_only), sorted by Count ############################################*")
        println(g, "############################################* Clade/Pango Count for $(mutation_pos_only), sorted by Count ############################################*")
        for w in clade_dict_for_mut_pos_only_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            clade_pct_of_mutSeqs = round(digits=1, 100*clade_ct/AA_muts_ct_pos_only[mutation_pos_only])
            mutSeqs_pct_of_clade = round(digits=1, 100*clade_ct/seq_clade_ct[clade])
            if clade_ct > 0
                cladePango = rpad("$(clade)|$(pango)", padd)
                pctOfMutseqs = lpad("$(clade_pct_of_mutSeqs)%", 6)
                pctOfClade = lpad("$(mutSeqs_pct_of_clade)%", 6)
                fracTop = lpad("$(clade_ct)", 2)
                fracBot = lpad("$(AA_muts_ct_pos_only[mutation_pos_only])", len)
                fract = "$(fracTop)/$(fracBot)"
                println("$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
                println(g, "$(cladePango) = $(fract) | $(pctOfMutseqs) | $(pctOfClade)")
            end
        end
#        println()
        println(g)
#        println("###################################################* Normalized Clade/Pango Count for $(mutation_pos_only), sorted by Count ###################################################*")
        println(g, "###################################################* Normalized Clade/Pango Count for $(mutation_pos_only), sorted by Count ###################################################*")
        for w in clade_dict_ADJ_for_mut_pos_only_sort_by_ct
            clade = w[1]
            pango = clade_to_pango[w[1]]
            clade_ct = w[2]
            if clade_ct > 0
#                println(rpad("$(clade)|$(pango)", padd) * " = $(clade_ct) [$(clade_dict_for_mut_pos_only[clade])]")
                println(g, rpad("$(clade)|$(pango)", padd) * " = $(clade_ct) [$(clade_dict_for_mut_pos_only[clade])]")
            end
        end
        println()
        println(g)
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println(g, "#############################################################################################################")
        println("Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
        println("Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
        println()
        println(g, "########################################################################################################################################################################################**")
        println(g, "#################***** Mutation cts for seq with $(mutation_pos_only) Sorted by Count #################*****")
        println(g, "########################################################################################################################################################################################**")
        println(g)
        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
        println(g)
        println(g, "Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
        println(g, "Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
        println(g)
        for m in prop_dict_pos_only_sort_Count
            if AA_cts_pos_only[m[1]] ≥ min_ct
                m_perc_of_all = round(digits=1, 100*AA_muts_ct_pos_only[m[1]]/singlet_grps_tot)
                aamut = rpad("$(m[1])", padd)
                totPerc = lpad("$(m[2][3])%", 6)
                fracTop = lpad("$(AA_cts_pos_only[m[1]])", 2)
                fracBot = lpad("$(AA_muts_ct_pos_only[m[1]])", 3)
                fracBot2 = lpad("$(AA_muts_ct_pos_only[mutation_pos_only])", 3)
                qryPerc = lpad("$(m[2][1])%", 6)
                refPerc = lpad("$(m[2][2])%", 6)
                mPercAll = lpad("$(m_perc_of_all)%", 5)
#                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
                println(g, "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
            end
        end 
#        print("\n"^2)
#        print(g, "\n"^2)
#        println("########################################################################################################################################################################################**")
#        println("#################***** Mutation cts for seq with $(mutation_pos_only) Sorted by QRY PERCENTS #################*****")
#        println("########################################################################################################################################################################################**")
#        println()
#        println(rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
#        println(rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
#        println(rpad("Total number of Independent Chronic Singlets/Groups = ", 55) * "$(singlet_grps_tot)")
#        println()
#        if haskey(AA_muts_ct_pos_only, mutation_pos_only)
#            println("Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
#        else
#            println("Total Number of $(mutation_pos_only) seqs = 0")
#        end
#        println("Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
#        println()
#        println(g, "########################################################################################################################################################################################**")
#        println(g, "#################***** Mutation cts for seq with $(mutation_pos_only) Sorted by QRY PERCENTS #################*****")
#        println(g, "########################################################################################################################################################################################**")
#        println(g)
#        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
#        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
#        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
#        println(g)
#        if haskey(AA_muts_ct_pos_only, mutation_pos_only)
#            println(g, "Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
#        else
#            println(g, "Total Number of $(mutation_pos_only) seqs = 0")
#        end
#        println(g, "Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
#        println(g)
#        for m in prop_dict_pos_only_sort_QryPerc
#            if AA_cts_pos_only[m[1]] ≥ min_ct
#                m_perc_of_all = round(digits=1, 100*AA_muts_ct_pos_only[m[1]]/singlet_grps_tot)
#                aamut = rpad("$(m[1])", 13)
#                qryPerc = lpad("$(m[2][1])%", 6)
#                fracTop = lpad("$(AA_cts_pos_only[m[1]])", 2)
#                fracBot = lpad("$(AA_muts_ct_pos_only[m[1]])", 3)
#                fracBot2 = lpad("$(AA_muts_ct_pos_only[mutation_pos_only])", 3)
#                refPerc = lpad("$(m[2][2])%", 6)
#                mPercAll = lpad("$(m_perc_of_all)%", 5)
#                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
#                println(g, "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
#            end 
#        end
#        print("\n"^2)
#        print(g, "\n"^2)
#############################################################################################################
#############################################################################################################
#        println("########################################################################################################################################################################################**")
#        println("#################**** Mutation cts for seq with $(mutation_pos_only) Sorted by Gene & Position #################*****")
#        println("########################################################################################################################################################################################**")
#        println()
#        println(rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
#        println(rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
#        println(rpad("Total number of Independent Chronic Singlets/Groups = ", 55) * "$(singlet_grps_tot)")
#        println()
#        if haskey(AA_muts_ct_pos_only, mutation_pos_only)
#            println("Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
#        else
#            println("Total Number of $(mutation_pos_only) seqs = 0")
#        end
#        println("Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
#        println()
        println(g, "########################################################################################################################################################################################**")
        println(g, "#################**** Mutation cts for seq with $(mutation_pos_only) Sorted by Gene & Position #################*****")
        println(g, "########################################################################################################################################################################################**")
        println(g)
        println(g, rpad("Total number of Chronic Singlets = ", 55) * "$(singlet_tot)")
        println(g, rpad("Total number of Unrelated Chronic Groups = ", 55) * "$(rep_seq_grps_tot)")
        println(g, rpad("Total number of Independent Chronic Singlets/Groups = ", 53) * "$(singlet_grps_tot)")
        println(g)
        if haskey(AA_muts_ct_pos_only, mutation_pos_only)
            println(g, "Total Number of $(mutation_pos_only) seqs = $(AA_muts_ct_pos_only[mutation_pos_only])")
        else
            println(g, "Total Number of $(mutation_pos_only) seqs = 0")
        end
        println(g, "Total Percentage of all Chronics with $(mutation_pos_only) = $(mutPerc_pos_only)%")
        println(g)
        for m in prop_dict_pos_only_sort_gene
            if AA_cts_pos_only[m[1]] ≥ min_ct
                m_perc_of_all = round(digits=1, 100*AA_muts_ct_pos_only[m[1]]/singlet_grps_tot)
                aamut = rpad("$(m[1])", padd)
                totPerc = lpad("$(m[2][3])%", 6)
                fracTop = lpad("$(AA_cts_pos_only[m[1]])", 2)
                fracBot = lpad("$(AA_muts_ct_pos_only[m[1]])", 3)
                fracBot2 = lpad("$(AA_muts_ct_pos_only[mutation_pos_only])", 3)
                qryPerc = lpad("$(m[2][1])%", 6)
                refPerc = lpad("$(m[2][2])%", 6)
                mPercAll = lpad("$(m_perc_of_all)%", 5)
#                println("$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
                println(g,  "$(aamut) = $(fracTop)/$(fracBot) =$(qryPerc) $(fracTop)/$(fracBot2) =$(refPerc)  PctAll = $(mPercAll)")
            end
        end 
        print("\n"^2)
        print(g, "\n"^2) 
    end
##############################################################################################################
    date = Dates.format(today(), "yyyy_mm_dd")
    geneK(n) = split(n, ":")[1]
    mutK(n) = split(n, ":")[2][2:end-1]
    gene_number(n) = mut_gene_Dict[geneK(n)]
    mutnum_sortKey(n) = (length(mutK(n)), mutK(n))
    finalKey(n) = (gene_number(n), mutnum_sortKey(n))
    seq_muts_sortKey(n) = ()
#    mutK_pos(n) = split(n, ":")[2]
#    mutnum_sortKey_pos_only(n) = (length(mutK_pos(n)), mutK_pos(n))
#    finalKey_pos_only(n) = (gene_number(n), mutnum_sortKey_pos_only(n))
    open("mutation_analysis_output/$(mutation)_seqs_individually_priv_muts_$(date)_minCt$(min_ct).txt", "w") do g
        println("########################################################################################################################################################################"^3)
        println("###################################################****** $(mutation)_seqs-individual_private_muts ####################################################################**")
        println("########################################################################################################################################################################"^3)
        println(g, "################################################################################################################################################################################"^12)
        println(g, "###################################################****** $(mutation)_seqs-individual_private_muts ####################################################################**")
        println(g, "################################################################################################################################################################################"^12)
        
        vec_vec_header_mutsStr = Vector{Vector{String}}()
        for seq_EPI in allMutSeqs_sort
            if seq_EPI in all_unique_chr_seqs_set
                print_all_seq_info(seq_EPI, "all_seq_info/all_seq_info_$(seq_EPI)_$(date_now).txt")
                vec_header_mutsStr = Vector{String}()
                seq_muts_sort = sort(collect(seq_AA_muts[seq_EPI]), by = x -> finalKey(x))
                pango = seq_pango[seq_EPI]
                cntry = seq_country[seq_EPI]
                coll_date = seq_collection_date[seq_EPI]
                header = "$(pango), $(cntry), $(coll_date), $(seq_EPI)"
                seq_muts_sort_arr = Vector{String}()
                for seq_mut in seq_muts_sort
                    push!(seq_muts_sort_arr, seq_mut)
                end
                seq_muts_sort_str = join(seq_muts_sort_arr, ", ")
                push!(vec_header_mutsStr, header)
                push!(vec_header_mutsStr, seq_muts_sort_str)
                push!(vec_vec_header_mutsStr, vec_header_mutsStr)
            end
        end
        vec_vec_header_mutsStr_sort = sort(vec_vec_header_mutsStr, by = x -> x[1])
        for vec_header in vec_vec_header_mutsStr_sort
            println(vec_header[1])
            println(vec_header[2])
            println()
            println(g, vec_header[1])
            println(g, vec_header[2])
            println(g)
        end
    end
##############################################################################################################
    open("mutation_analysis_output/$(mutation)_seqs_individually_priv_muts_POS_ONLY_$(date)_minCt$(min_ct).txt", "w") do g
#        println("##########################################################################################################################################################################################################################"^12)
#        println("##################################*** $(mutation_pos_only)_seqs_individually_priv_muts_POS_ONLY ###################################################**")
#        println("##########################################################################################################################################################################################################################"^12)
        println(g, "##########################################################################################################################################################################################################################"^12)
        println(g, "##################################*** $(mutation_pos_only)_seqs_individually_priv_muts_POS_ONLY ###################################################**")
        println(g, "##########################################################################################################################################################################################################################"^12)
        vec_vec_header_mutsStr_pos_only = Vector{Vector{String}}()
        for seq_EPI in allMutSeqs_pos_only_sort
            if seq_EPI in all_unique_chr_seqs_set
#                print_all_seq_info(seq_EPI, "all_seq_info/all_seq_info_$(seq_EPI)_$(date_now).txt")
                vec_header_mutsStr_pos_only = Vector{String}()
    #            seq_muts_pos_only_sort = sort(collect(seq_AA_muts_pos_only[seq_EPI]), by = x -> finalKey_pos_only(x))
                seq_muts_pos_only_sort = sort(collect(seq_AA_muts[seq_EPI]), by = x -> finalKey(x))
                pango = seq_pango[seq_EPI]
                cntry = seq_country[seq_EPI]
                coll_date = seq_collection_date[seq_EPI]
                header = "$(pango), $(cntry), $(coll_date), $(seq_EPI)"
                seq_muts_pos_only_sort_arr = Vector{String}()
                for seq_mut_pos in seq_muts_pos_only_sort
                    push!(seq_muts_pos_only_sort_arr, seq_mut_pos)
                end
                seq_mut_pos_sort_str = join(seq_muts_pos_only_sort_arr, ", ")
                push!(vec_header_mutsStr_pos_only, header)
                push!(vec_header_mutsStr_pos_only, seq_mut_pos_sort_str)
                push!(vec_vec_header_mutsStr_pos_only, vec_header_mutsStr_pos_only)
            end
            vec_vec_header_mutsStr_pos_only_sort = sort(vec_vec_header_mutsStr_pos_only, by = x -> x[1])
#            for vec_header in vec_vec_header_mutsStr_pos_only_sort
#                println(vec_header[1])
#                println(vec_header[2])
#                println()
#                println(g, vec_header[1])
#                println(g, vec_header[2])
#                println(g)
#            end
        end
    end
##############################################################################################################
end

# CovSpectrum search: 
# Remdesivir-resistance muts: Q435K/H, C455F/Y, V783I, M785I/V, E787K, C790F/Y, V811G
# candidates: I97V, E127A, V157L, V157A, D161Y, I162M, I162T, T635A, I748V, F773L, L796I,  
# Very likely based on chronic muts = Y610F, L777I, L777V, T792I 

# ["ORF1a:3284", "ORF1a:3284", "ORF1a:3429", "ORF1a:3429", "ORF1a:3432", "ORF1a:Y3502"]
# ORF1a:T3284I, ORF1a:C3423F, ORF1a:E3429A, ORF1a:E3429V  ORF1a:T3432,  ORF1a:Y3502
# Paxlovid-resistance muts: T21I, C160F, E166A/V, T169, Y239C (all in BA.1.1 Japanese chronic)
# Possibles:  R4K, Y239C 

function nuc_mutation(mut)
    AA_sort_key_pos_only(m) = (aa_gene_comprehensive_dict[m], aa_pos_comprehensive_dict[m])
    jene(n) = aa_gene_comprehensive_dict[n]
    date = Dates.format(today(), "yyyy_mm_dd")
    open("mutation_analysis_output/$(mut)_seqs_with_private_mutations_$(date).txt", "w") do g
        seqs_indy = Vector{String}()
        for (grp_num, nuc_set) in rep_seq_grps_maxmut_nuc
            if mut in nuc_set
                push!(seqs_indy, rep_seq_grps_maxmut_seqs[grp_num])
            end
        end
        for seq in non_rep_seqs
            if mut in seq_nuc_muts[seq]
                push!(seqs_indy, seq)
            end
        end
        indy_seq_total = length(seqs_indy)
        seqs_indy_sort = sort(collect(seqs_indy), by = x -> (length(x), x))              
##############################
        println("Total Number of Independent Sequences with ", mut, " = ", indy_seq_total)
        println(); println()
        println(g, "Total Number of Independent Sequences with ", mut, " = ", indy_seq_total)
        println(g); println(g)
        for seq in seqs_indy_sort
#            println("$(seq_pango[seq]), $seq, $(seq_country[seq]), $(seq_collection_date[seq])")
            println(g, "$(seq_pango[seq]), $seq, $(seq_country[seq]), $(seq_collection_date[seq])")
            seq_AA_muts_sort = sort(collect(seq_AA_muts[seq]), by = x -> AA_sort_key_pos_only(x))        
            seq_nuc_muts_sort = sort(collect(seq_nuc_muts[seq]), by = x -> nuc_mut_int_comprehensive_dict[x])
#           print(seq_AA_muts_sort[1], ", ")
            print(g, seq_AA_muts_sort[1], ", ")
            for i in 2:length(seq_AA_muts_sort)
                premute = seq_AA_muts_sort[i-1]
                mute = seq_AA_muts_sort[i]
                if jene(mute) == jene(premute)
#                   print("$(mute), ")
                    print(g, "$(mute), ")
                else
#                    println()
                    println(g)
#                    print("$(mute), ")
                    print(g, "$(mute), ")
                end
            end
#            for mute in seq_AA_muts_sort
#                print("$(mute), ")
#                print(g, "$(mute), ")
#            end
#            print("\n"^2)
            print(g, "\n"^2)
            for nucmut in seq_nuc_muts_sort
#                print("$(nucmut), ")
                print(g, "$(nucmut), ")
            end
#            print("\n"^2)
            print(g, "\n"^2)
        end
#####################################################################################################################################
        println("****** ", mut, " sequences ******") 
        println(g, "****** ", mut, " sequences ******") 
        b = join(collect(nuc_muts_seq[mut]), ", ")
        println(b, ", ")
        println(g, b, ", ")
        c = length(nuc_muts_seq[mut])
        println()
        println(g)
        println("Total Number of Sequences with ", mut, " = ", c)
        println()
        println(g, "Total Number of Sequences with ", mut, " = ", c)
        println(g)
######################################
        seqList_all = nuc_muts_seq[mut]
        seqList_all_sort = sort(collect(seqList_all), by = x -> (length(x), x))
        for seq in seqList_all_sort
            println("$(seq_pango[seq]), $seq, $(seq_country[seq]), $(seq_collection_date[seq])")
            println(g, "$(seq_pango[seq]), $seq, $(seq_country[seq]), $(seq_collection_date[seq])")
            print_all_seq_info(seq, "all_seq_info/all_seq_info_$(seq)_$(date_now).txt")
#            print_all_seq_info(g, seq)
#            seq_AA_muts_sort = sort(collect(seq_AA_muts[seq]), by = x -> AA_sort_key_pos_only(x))        
#            seq_nuc_muts_sort = sort(collect(seq_nuc_muts[seq]), by = x -> nuc_mut_int_comprehensive_dict[x])
#            print(seq_AA_muts_sort[1], ", ")
#            print(g, seq_AA_muts_sort[1], ", ")
#            for i in 2:length(seq_AA_muts_sort)
#                premute = seq_AA_muts_sort[i-1]
#                mute = seq_AA_muts_sort[i]
#                if jene(mute) == jene(premute)
#                    print("$(mute), ")
#                    print(g, "$(mute), ")
#                else
#                    println()
#                    println(g)
#                    print("$(mute), ")
#                    print(g, "$(mute), ")
#                end
#            end
#            for mute in seq_AA_muts_sort
#                print("$(mute), ")
#                print(g, "$(mute), ")
#            end
            print("\n"^2)
            print(g, "\n"^2)
#            for nucmut in seq_nuc_muts_sort
#                print("$(nucmut), ")
#                print(g, "$(nucmut), ")
#            end
#            print("\n"^2)
#            print(g, "\n"^2)
        end
    end
end
function AA_mutation_sequences(mut)
    x = collect(AA_muts_seq[mut])
    for s in x
        print(s, ", ")
    end
end
# select :1648-1657       # color sel red

2026_03_06__832AM


AA_mutation_sequences (generic function with 1 method)

In [252]:
### Fx: avg_bal_muts_per_seq_v2 + avg_bal_muts_per_seq_no_print_v2
### This function takes a set of mutations and collects all sequences that have ≥1 of those mutations. It then calculates the number
### avg # of BAL-MP mutations (as determined by the main BAL-MP function) per sequence in the list (excluding all mutations used to select the sequences,
### i.e., if E:T30I is in the selected set of mutations used, it will not count as a BAL-MP mut even though it is on the BAL-MP mutation list.)
### This number is then compared to the avg # of BAL-MP mutations for all EPCI sequences that **DO NOT** possess one of the mutations in the set, and a ratio is calculated.
#################### Below: Alternative calculation that compares all EPCI seqs WITHOUT ≥1 mut in AAmutset ########################################
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["ORF7b:I2-", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
NSP15_K2340 = Set(["ORF1b:K2340N", "ORF1b:K2340E", "ORF1b:K2340T"])
NSP15_endonuclease_killers = Set(["ORF1b:S2339A", "ORF1b:S2339F", "ORF1b:S2339Y", "ORF1a:S2339P", "ORF1b:K2340N", "ORF1b:K2340E", "ORF1b:K2340T"])
ORF9b_Killers = list_to_set("ORF9b:M1K, ORF9b:M1T, ORF9b:M1I, ORF9b:M1L, ORF9b:M1V, ORF9b:E7*, ORF9b:Q18*, ORF9b:Q20*, ORF9b:Q34*, ORF9b:E65*, ORF9b:Q77*")
ORF6_killers = list_to_set("ORF6:M1T, ORF6:M1I, ORF6:M1V, ORF6:Q8*, ORF6:W27*, ORF6:Y49*, ORF6:Q56*, ORF6:M58K, ORF6:E59*")
ORF7a_extensions = list_to_set("ORF7a:*122W, ORF7a:*122R, ORF7a:*122S, ORF7a:*122-")
### The sequence below has ORF7a:*122- but as part of an extensive deletion that inludes ORF7a:96-122 and all of ORF7b and ORF8.
### It is therefore excluded from the ORF7a_extension sequences as it does not actually have the extended C-terminal tail (as all other ORF7a:*122- seqs do).
ORF7a_extension_excluded_seqs = list_to_set("EPI_ISL_17298321")
empty_string_set = Set{String}()
BAL_mut_list_2025_12_08 = list_to_set("ORF1a:T2183I, ORF1a:T2967I, ORF1a:S2972P, ORF1a:S2972F, ORF1a:A3023V, ORF1a:T3032I, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:V3077A, ORF1a:F3089L, ORF1a:L3123F, ORF1a:S3195G, ORF1a:I3255V, ORF1a:I3255T, ORF1a:P3359L, ORF1a:K3365R, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:K4176R, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:K1957R, ORF1b:I2147T, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:Y369H, S:A372-, S:A372T, S:F374L, S:F375-, S:A376V, S:N405S, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A647T, S:A653V, S:N657D, S:N657K, S:S659P, S:S659A, S:A668V, S:S691P, S:S691F, S:I693V, S:T859I, S:A944T, S:A944V, S:A958D, S:N978D, S:I1169T, S:I1179T, S:I1183T, S:L1186F, E:V5A, E:V5F, E:S6L, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:L19I, E:L27S, E:T30I, E:A32V, E:A36V, E:Y42C, E:I46V, E:S68F, E:P71L, M:A2V, M:N3I, M:N3K, M:S4P, M:F28S, M:T77N, M:G78S, M:S94N, M:S94R, M:H125Y, M:H148R, M:S173P, M:A188T, M:G189S, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*")
total_unique_chr_seqs = length(all_unique_chr_seqs_set)
function avg_bal_muts_per_seq_v2(AAmutset::Set{String}, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String})
#    mutset_BALmuts_overlap = intersect(AAmutset, EPCI_BALmut_list)
    EPCI_BALmut_list_adjusted = setdiff(EPCI_BALmut_list, AAmutset)
    mut_set_tot_BAL_muts = 0
    mut_set_tot_AA_subs = 0
    mut_set_tot_seqs = 0
    mut_set_seqs = Set{String}()
    for seq in all_unique_chr_seqs_set
        if !(seq in excluded_seqs)
            if !isempty(intersect(seq_AA_muts[seq], AAmutset))
                mut_set_tot_seqs += 1
                total_seq_AA_subs = length(seq_AA_muts_no_dels[seq])
                mut_set_tot_AA_subs += total_seq_AA_subs
                BAL_muts = intersect(seq_AA_muts[seq], EPCI_BALmut_list_adjusted)
                BAL_mut_tot = length(BAL_muts)
                mut_set_tot_BAL_muts += BAL_mut_tot
                push!(mut_set_seqs, seq)
            end
        end
    end
    avg_BALmuts_per_seq = round(digits=2, mut_set_tot_BAL_muts/mut_set_tot_seqs)
    mut_set_avg_AA_subs_per_seq = round(digits=2, mut_set_tot_AA_subs/mut_set_tot_seqs)
    EPCI_total_AA_subs = sum(length.(seq_AA_muts_no_dels[seq] for seq in all_unique_chr_seqs_set))
    EPCI_avg_AA_subs_per_seq = round(digits=2, EPCI_total_AA_subs/total_unique_chr_seqs)
    totAAsubs_ratio = round(digits=2, mut_set_avg_AA_subs_per_seq/EPCI_avg_AA_subs_per_seq)
    println("Avg BAL Muts per mut_set seq = $(avg_BALmuts_per_seq)")
    println("Avg Total AA Subs/mut_set seq = $(mut_set_avg_AA_subs_per_seq)")
    println("Avg Total AA Subs/EPCI seq = $(EPCI_avg_AA_subs_per_seq)")
    println("Ratio of mutset/EPCI AA Subs/seq = $(totAAsubs_ratio)")
    print("\n"^1)
##############################################################################
    all_unique_chr_seqs_set_minus_seqs_on_list = setdiff(all_unique_chr_seqs_set, mut_set_seqs)
    total_unique_chr_seqs_minus_seqs_on_list = length(all_unique_chr_seqs_set_minus_seqs_on_list)
    total_unique_chr_seqs_minus_seqs_on_list_v2 = total_unique_chr_seqs - mut_set_tot_seqs
    println("Total EPCI seqs = $(total_unique_chr_seqs)")
    println("Total mut_set seqs = $(mut_set_tot_seqs)")
    println("Total EPCI seqs - mut_set_seqs    = $(total_unique_chr_seqs_minus_seqs_on_list)")
    println("Total EPCI seqs - mut_set_seqs v2 = $(total_unique_chr_seqs_minus_seqs_on_list_v2)")
    EPCI_BALmut_list_adjusted_total = 0
    for seq in all_unique_chr_seqs_set_minus_seqs_on_list
        for mut in seq_AA_muts[seq]
            if mut in EPCI_BALmut_list_adjusted
                EPCI_BALmut_list_adjusted_total += 1
            end
        end
    end
    avg_BAL_muts_per_unique_chr_seq = round(digits=2, EPCI_BALmut_list_adjusted_total/total_unique_chr_seqs_minus_seqs_on_list)
#############
    BALmut_ratio = round(digits=2, avg_BALmuts_per_seq/avg_BAL_muts_per_unique_chr_seq)
    println("Avg BAL muts per EPCI seq not on List = $(avg_BAL_muts_per_unique_chr_seq)")
    print("\n"^1)
    println("BALmuts/mutsetSeq to BALmuts/EPCIseq Ratio = $(BALmut_ratio)")
    print("\n"^1)
    return avg_BALmuts_per_seq, BALmut_ratio, totAAsubs_ratio, mut_set_tot_seqs
end
############################################################################################################
############################################################################################################
### Fx: avg_bal_muts_per_seq_no_print_v2 (for seqs with ≥ 1 of a list of muts)
function avg_bal_muts_per_seq_no_print_v2(AAmutset::Set{String}, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String})
#    mutset_BALmuts_overlap = intersect(AAmutset, EPCI_BALmut_list)
    EPCI_BALmut_list_adjusted = setdiff(EPCI_BALmut_list, AAmutset)
    mut_set_tot_BAL_muts = 0
    mut_set_tot_AA_subs = 0
    mut_set_tot_seqs = 0
    mut_set_seqs = Set{String}()
    for seq in all_unique_chr_seqs_set
        if !(seq in excluded_seqs)
            if !isempty(intersect(seq_AA_muts[seq], AAmutset))
                mut_set_tot_seqs += 1
                total_seq_AA_subs = length(seq_AA_muts_no_dels[seq])
                mut_set_tot_AA_subs += total_seq_AA_subs
                BAL_muts = intersect(seq_AA_muts[seq], EPCI_BALmut_list_adjusted)
                BAL_mut_tot = length(BAL_muts)
                mut_set_tot_BAL_muts += BAL_mut_tot
                push!(mut_set_seqs, seq)
            end
        end
    end
    avg_BALmuts_per_seq = round(digits=2, mut_set_tot_BAL_muts/mut_set_tot_seqs)
    mut_set_avg_AA_subs_per_seq = round(digits=2, mut_set_tot_AA_subs/mut_set_tot_seqs)
    EPCI_total_AA_subs = sum(length.(seq_AA_muts_no_dels[seq] for seq in all_unique_chr_seqs_set))
    EPCI_avg_AA_subs_per_seq = round(digits=2, EPCI_total_AA_subs/total_unique_chr_seqs)
    totAAsubs_ratio = round(digits=2, mut_set_avg_AA_subs_per_seq/EPCI_avg_AA_subs_per_seq)
##############################################################################
    all_unique_chr_seqs_set_minus_seqs_on_list = setdiff(all_unique_chr_seqs_set, mut_set_seqs)
    total_unique_chr_seqs_minus_seqs_on_list = length(all_unique_chr_seqs_set_minus_seqs_on_list)
    total_unique_chr_seqs_minus_seqs_on_list_v2 = total_unique_chr_seqs - mut_set_tot_seqs
    EPCI_BALmut_list_adjusted_total = 0
    for seq in all_unique_chr_seqs_set_minus_seqs_on_list
        for mut in seq_AA_muts[seq]
            if mut in EPCI_BALmut_list_adjusted
                EPCI_BALmut_list_adjusted_total += 1
            end
        end
    end
    avg_BAL_muts_per_unique_chr_seq = round(digits=2, EPCI_BALmut_list_adjusted_total/total_unique_chr_seqs_minus_seqs_on_list)
#############
    BALmut_ratio = round(digits=2, avg_BALmuts_per_seq/avg_BAL_muts_per_unique_chr_seq)
    return avg_BALmuts_per_seq, BALmut_ratio, totAAsubs_ratio, mut_set_tot_seqs
end
############################################################################################################
############################################################################################################  

2026_01_30__523PM


avg_bal_muts_per_seq_no_print_v2 (generic function with 1 method)

In [253]:
### Fx: avg_bal_muts_per_seq_v1 + avg_bal_muts_per_seq_no_print_v1 (used to find avg BAL muts for seqs with ≥1 muts in a set of mutations)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
NSP15_K2340 = Set(["ORF1b:K2340N", "ORF1b:K2340E", "ORF1b:K2340T"])
NSP15_endonuclease_killers = Set(["ORF1b:S2339A", "ORF1b:S2339F", "ORF1b:S2339Y", "ORF1a:S2339P", "ORF1b:K2340N", "ORF1b:K2340E", "ORF1b:K2340T"])
ORF9b_Killers = list_to_set("ORF9b:M1K, ORF9b:M1T, ORF9b:M1I, ORF9b:M1L, ORF9b:M1V, ORF9b:E7*, ORF9b:Q18*, ORF9b:Q20*, ORF9b:Q34*, ORF9b:E65*, ORF9b:Q77*")
ORF6_killers = list_to_set("ORF6:M1T, ORF6:M1I, ORF6:M1V, ORF6:Q8*, ORF6:W27*, ORF6:Y49*, ORF6:Q56*, ORF6:M58K, ORF6:E59*")
ORF7a_extensions = list_to_set("ORF7a:*122W, ORF7a:*122R, ORF7a:*122S, ORF7a:*122-")
### The sequence below has ORF7a:*122- but as part of an extensive deletion that inludes ORF7a:96-122 and all of ORF7b and ORF8.
### It is therefore excluded from the ORF7a_extension sequences as it does not actually have the extended C-terminal tail (as all other ORF7a:*122- seqs do).
ORF7a_extension_excluded_seqs = list_to_set("EPI_ISL_17298321")
empty_string_set = Set{String}()
BAL_mut_list_2025_12_08 = list_to_set("ORF1a:T2183I, ORF1a:T2967I, ORF1a:S2972P, ORF1a:S2972F, ORF1a:A3023V, ORF1a:T3032I, ORF1a:A3049V, ORF1a:T3058I, ORF1a:A3070V, ORF1a:G3072C, ORF1a:H3076Y, ORF1a:V3077A, ORF1a:F3089L, ORF1a:L3123F, ORF1a:S3195G, ORF1a:I3255V, ORF1a:I3255T, ORF1a:P3359L, ORF1a:K3365R, ORF1a:A3454V, ORF1a:A3456V, ORF1a:Q4110R, ORF1a:T4175I, ORF1a:K4176R, ORF1a:P4197S, ORF1a:I4205V, ORF1a:T4207A, ORF1a:R4387S, ORF1b:L314P, ORF1b:I1181T, ORF1b:Y1247C, ORF1b:T1424I, ORF1b:K1957R, ORF1b:I2147T, ORF1b:S2339F, ORF1b:K2340N, ORF1b:T2537I, S:S50L, S:P330S, S:N354D, S:V367F, S:Y369H, S:A372-, S:A372T, S:F374L, S:F375-, S:A376V, S:N405S, S:N405D, S:Y508H, S:V551I, S:T573I, S:A647S, S:A647T, S:A653V, S:N657D, S:N657K, S:S659P, S:S659A, S:A668V, S:S691P, S:S691F, S:I693V, S:T859I, S:A944T, S:A944V, S:A958D, S:N978D, S:I1169T, S:I1179T, S:I1183T, S:L1186F, E:V5A, E:V5F, E:S6L, E:T9I, E:G10S, E:G10C, E:I13-, E:V14-, E:S16N, E:L19I, E:L27S, E:T30I, E:A32V, E:A36V, E:Y42C, E:I46V, E:S68F, E:P71L, M:A2V, M:N3I, M:N3K, M:S4P, M:F28S, M:T77N, M:G78S, M:S94N, M:S94R, M:H125Y, M:H148R, M:S173P, M:A188T, M:G189S, N:S416L, N:T417I, ORF7a:T115I, ORF9b:M1T, ORF9b:Q77*")
tot_seqs_uniq_chr = length(all_unique_chr_seqs_set)
function avg_bal_muts_per_seq_v1(AAmutset::Set{String}, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String})
#    mutset_BALmuts_overlap = intersect(AAmutset, EPCI_BALmut_list)
    EPCI_BALmut_list_adjusted = setdiff(EPCI_BALmut_list, AAmutset)
    mut_set_tot_BAL_muts = 0
    mut_set_tot_AA_subs = 0
    mut_set_tot_seqs = 0
    for seq in all_unique_chr_seqs_set
        if !(seq in excluded_seqs)
            if !isempty(intersect(seq_AA_muts[seq], AAmutset))
                mut_set_tot_seqs += 1
                total_seq_AA_subs = length(seq_AA_muts_no_dels[seq])
                mut_set_tot_AA_subs += total_seq_AA_subs
                BAL_muts = intersect(seq_AA_muts[seq], EPCI_BALmut_list_adjusted)
                BAL_mut_tot = length(BAL_muts)
                mut_set_tot_BAL_muts += BAL_mut_tot
            end
        end
    end
    avg_BALmuts_per_seq = round(digits=2, mut_set_tot_BAL_muts/mut_set_tot_seqs)
    mut_set_avg_AA_subs_per_seq = round(digits=2, mut_set_tot_AA_subs/mut_set_tot_seqs)
    EPCI_total_AA_subs = sum(length.(seq_AA_muts_no_dels[seq] for seq in all_unique_chr_seqs_set))
    EPCI_avg_AA_subs_per_seq = round(digits=2, EPCI_total_AA_subs/tot_seqs_uniq_chr)
    totAAsubs_ratio = round(digits=2, mut_set_avg_AA_subs_per_seq/EPCI_avg_AA_subs_per_seq)
    println("Avg BAL Muts per mut_set seq = $(avg_BALmuts_per_seq)")
    println("Avg Total AA Subs/mut_set seq = $(mut_set_avg_AA_subs_per_seq)")
    println("Avg Total AA Subs/EPCI seq = $(EPCI_avg_AA_subs_per_seq)")
    println("Ratio of mutset/EPCI AA Subs/seq = $(totAAsubs_ratio)")
    print("\n"^1)
#############
    EPCI_BALmut_list_adjusted_total = 0
    for mut in EPCI_BALmut_list_adjusted
        BALmut_tot = get!(AA_muts_ct, mut, 0)
        EPCI_BALmut_list_adjusted_total += BALmut_tot
    end
    avg_BAL_muts_per_unique_chr_seq = round(digits=2, EPCI_BALmut_list_adjusted_total/tot_seqs_uniq_chr)
#############
    BALmut_ratio = round(digits=2, avg_BALmuts_per_seq/avg_BAL_muts_per_unique_chr_seq)
    println("Avg BAL muts per EPCI seq = $(avg_BAL_muts_per_unique_chr_seq)")
    print("\n"^1)
    println("BALmuts/mutsetSeq to BALmuts/EPCIseq Ratio = $(BALmut_ratio)")
    print("\n"^1)
    return avg_BALmuts_per_seq, BALmut_ratio, totAAsubs_ratio, mut_set_tot_seqs
end
############################################################################################################
############################################################################################################
### Fx: avg_bal_muts_per_seq_no_print_v1 (for seqs with ≥ 1 of a list of muts)
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
function avg_bal_muts_per_seq_no_print_v1(AAmutset::Set{String}, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String})
#    mutset_BALmuts_overlap = intersect(AAmutset, EPCI_BALmut_list)
    EPCI_BALmut_list_adjusted = setdiff(EPCI_BALmut_list, AAmutset)
    mut_set_tot_BAL_muts = 0
    mut_set_tot_AA_subs = 0
    mut_set_tot_seqs = 0
    for seq in all_unique_chr_seqs_set
        if !(seq in excluded_seqs)
            if !isempty(intersect(seq_AA_muts[seq], AAmutset))
                mut_set_tot_seqs += 1
                total_seq_AA_subs = length(seq_AA_muts_no_dels[seq])
                mut_set_tot_AA_subs += total_seq_AA_subs
                BAL_muts = intersect(seq_AA_muts[seq], EPCI_BALmut_list_adjusted)
                BAL_mut_tot = length(BAL_muts)
                mut_set_tot_BAL_muts += BAL_mut_tot
            end
        end
    end
    avg_BALmuts_per_seq = round(digits=2, mut_set_tot_BAL_muts/mut_set_tot_seqs)
    mut_set_avg_AA_subs_per_seq = round(digits=2, mut_set_tot_AA_subs/mut_set_tot_seqs)
    EPCI_total_AA_subs = sum(length.(seq_AA_muts_no_dels[seq] for seq in all_unique_chr_seqs_set))
    EPCI_avg_AA_subs_per_seq = round(digits=2, EPCI_total_AA_subs/tot_seqs_uniq_chr)
    totAAsubs_ratio = round(digits=2, mut_set_avg_AA_subs_per_seq/EPCI_avg_AA_subs_per_seq)
#############
    EPCI_BALmut_list_adjusted_total = 0
    for mut in EPCI_BALmut_list_adjusted
        BALmut_tot = get!(AA_muts_ct, mut, 0)
        EPCI_BALmut_list_adjusted_total += BALmut_tot
    end
    avg_BAL_muts_per_unique_chr_seq = round(digits=2, EPCI_BALmut_list_adjusted_total/tot_seqs_uniq_chr)
#############
    BALmut_ratio = round(digits=2, avg_BALmuts_per_seq/avg_BAL_muts_per_unique_chr_seq)
    return avg_BALmuts_per_seq, BALmut_ratio, totAAsubs_ratio, mut_set_tot_seqs
end
############################################################################################################
############################################################################################################  

2026_01_30__523PM
2026_01_30__523PM


avg_bal_muts_per_seq_no_print_v1 (generic function with 1 method)

In [214]:
### meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2 + DataFrame, CSV creation
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
start = time()
meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2 = Dict{Int, Dict{String, Tuple{Float64, Float64, Float64, Int64}}}()
meta_BAL_ratio_rank_dict_v2 = Dict{Int, Dict{String, Int}}()
meta_BAL_ratio_percentile_dict_v2 = Dict{Int, Dict{String, Float64}}()
for i in 3:20
    mut_ct = 0
    meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2[i] = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ i && !(mut in Double_N_ORF9b_muts)
            mut_ct += 1
            if mut_ct%400 == 0
                println("meta_dict $(i), $(mut_ct) muts finished.")
            end
            mutset = Set([mut])
            mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(mutset, BAL_mut_list_2025_12_08, ORF7a_extension_excluded_seqs)
            meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2[i][mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
        end
    end; dict_runtime = round(digits=1, time() - start)
    println("meta_dict #$(i) complete: $(dict_runtime) seconds")
end
df_dict = Dict{Int, DataFrame}()
for i in 3:20
    meta_BAL_ratio_rank_dict_v2[i] = Dict{String, Int}()
    meta_BAL_ratio_percentile_dict_v2[i] = Dict{String, Int}()
    df_dict[i] = DataFrame(
        Mutation = String[],
        BAL_avg = Float64[],
        BAL_ratio = Float64[],
        Ratio_Rank = Int64[],
        Percentile = Float64[],
        AAsub_Ratio = Float64[],
        TotalSeqs = Int64[])
    total_iPlus_muts = length(meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2[i])
    meta_dict_i = meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v2[i]
    meta_dict_i_sort = sort(collect(meta_dict_i), by = x -> x[2][2], rev=true)
    for j in 1:length(meta_dict_i_sort)
        mut = meta_dict_i_sort[j][1]
        AvgBALct = meta_dict_i_sort[j][2][1]
        BALratio = meta_dict_i_sort[j][2][2]
        AAsubRatio = meta_dict_i_sort[j][2][3]
        seqtot = meta_dict_i_sort[j][2][4]
        meta_BAL_ratio_rank_dict_v2[i][mut] = j
        percentile = round(digits=3, 100 - 100*(j-1)/total_iPlus_muts)
        meta_BAL_ratio_percentile_dict_v2[i][mut] = percentile
        push!(df_dict[i], (mut, AvgBALct, BALratio, j, percentile, AAsubRatio, seqtot))
    end
    runtime = time() - start
    runtime_rd = round(digits=2, runtime)
    runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
    println("$(i)+ Mut Dict Finished Time = $(runtime2)")
##########
    CSV.write("df_BALct_BALratio_totAAratio_seqTot__min$(i)__$(date_hour)_v2.csv", df_dict[i])
end
################################################################################################################################################################
################################################################################################################################################################
################################################################################################################################################################

meta_dict 3, 400 muts finished.
meta_dict 3, 800 muts finished.
meta_dict 3, 1200 muts finished.
meta_dict 3, 1600 muts finished.
meta_dict 3, 2000 muts finished.
meta_dict 3, 2400 muts finished.
meta_dict 3, 2800 muts finished.
meta_dict 3, 3200 muts finished.
meta_dict 3, 3600 muts finished.
meta_dict 3, 4000 muts finished.
meta_dict #3 complete: 57.7 seconds
meta_dict 4, 400 muts finished.
meta_dict 4, 800 muts finished.
meta_dict 4, 1200 muts finished.
meta_dict 4, 1600 muts finished.
meta_dict 4, 2000 muts finished.
meta_dict 4, 2400 muts finished.
meta_dict 4, 2800 muts finished.
meta_dict #4 complete: 100.0 seconds
meta_dict 5, 400 muts finished.
meta_dict 5, 800 muts finished.
meta_dict 5, 1200 muts finished.
meta_dict 5, 1600 muts finished.
meta_dict 5, 2000 muts finished.
meta_dict #5 complete: 133.1 seconds
meta_dict 6, 400 muts finished.
meta_dict 6, 800 muts finished.
meta_dict 6, 1200 muts finished.
meta_dict 6, 1600 muts finished.
meta_dict #6 complete: 160.6 seconds
met

In [215]:
### meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1 + DataFrame, CSV creation
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
start = time()
meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1 = Dict{Int, Dict{String, Tuple{Float64, Float64, Float64, Int64}}}()
meta_BAL_ratio_rank_dict_v1 = Dict{Int, Dict{String, Int}}()
meta_BAL_ratio_percentile_dict_v1 = Dict{Int, Dict{String, Float64}}()
for i in 3:20
    mut_ct = 0
    meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1[i] = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ i && !(mut in Double_N_ORF9b_muts)
            mut_ct += 1
            if mut_ct%500 == 0
                println("meta_dict $(i), $(mut_ct) muts finished.")
            end
            mutset = Set([mut])
            mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(mutset, BAL_mut_list_2025_12_08, ORF7a_extension_excluded_seqs)
            meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1[i][mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
        end
    end; dict_runtime = round(digits=1, time() - start)
    println("meta_dict #$(i) complete: $(dict_runtime) seconds")
end
df_dict = Dict{Int, DataFrame}()
for i in 3:20
    meta_BAL_ratio_rank_dict_v1[i] = Dict{String, Int}()
    meta_BAL_ratio_percentile_dict_v1[i] = Dict{String, Int}()
    df_dict[i] = DataFrame(
        Mutation = String[],
        BAL_avg = Float64[],
        BAL_ratio = Float64[],
        Ratio_Rank = Int64[],
        Percentile = Float64[],
        AAsub_Ratio = Float64[],
        TotalSeqs = Int64[])
    total_iPlus_muts = length(meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1[i])
    meta_dict_i = meta__mut__BALctAvg_BALratio_totAAsubRatio_totSeqs_v1[i]
    meta_dict_i_sort = sort(collect(meta_dict_i), by = x -> x[2][2], rev=true)
    for j in 1:length(meta_dict_i_sort)
        mut = meta_dict_i_sort[j][1]
        AvgBALct = meta_dict_i_sort[j][2][1]
        BALratio = meta_dict_i_sort[j][2][2]
        AAsubRatio = meta_dict_i_sort[j][2][3]
        seqtot = meta_dict_i_sort[j][2][4]
        meta_BAL_ratio_rank_dict_v1[i][mut] = j
        percentile = round(digits=3, 100 - 100*(j-1)/total_iPlus_muts)
        meta_BAL_ratio_percentile_dict_v1[i][mut] = percentile
        push!(df_dict[i], (mut, AvgBALct, BALratio, j, percentile, AAsubRatio, seqtot))
    end
    runtime = time() - start
    runtime_rd = round(digits=2, runtime)
    runtime1, runtime2 = seconds_to_hrs_min_sec(runtime)
    println("$(i)+ Mut Dict Finished Time = $(runtime2)")
##########
    CSV.write("df_BALct_BALratio_totAAratio_seqTot__min$(i)__$(date_hour)_v1.csv", df_dict[i])
end
################################################################################################################################################################
################################################################################################################################################################
################################################################################################################################################################

meta_dict 3, 500 muts finished.
meta_dict 3, 1000 muts finished.
meta_dict 3, 1500 muts finished.
meta_dict 3, 2000 muts finished.
meta_dict 3, 2500 muts finished.
meta_dict 3, 3000 muts finished.
meta_dict 3, 3500 muts finished.
meta_dict 3, 4000 muts finished.
meta_dict #3 complete: 7.9 seconds
meta_dict 4, 500 muts finished.
meta_dict 4, 1000 muts finished.
meta_dict 4, 1500 muts finished.
meta_dict 4, 2000 muts finished.
meta_dict 4, 2500 muts finished.
meta_dict 4, 3000 muts finished.
meta_dict #4 complete: 13.2 seconds
meta_dict 5, 500 muts finished.
meta_dict 5, 1000 muts finished.
meta_dict 5, 1500 muts finished.
meta_dict 5, 2000 muts finished.
meta_dict #5 complete: 17.2 seconds
meta_dict 6, 500 muts finished.
meta_dict 6, 1000 muts finished.
meta_dict 6, 1500 muts finished.
meta_dict #6 complete: 20.5 seconds
meta_dict 7, 500 muts finished.
meta_dict 7, 1000 muts finished.
meta_dict 7, 1500 muts finished.
meta_dict #7 complete: 23.3 seconds
meta_dict 8, 500 muts finished.
me

In [184]:
println("########## Muts in at least X EPCI seqs ##########")
number_plus_ct_dict = Dict{Int, Int}(i=>0 for i in 1:15)
for (mut, ct) in AA_muts_ct_no_dels
    for i in 1:15
        if ct ≥ i
            number_plus_ct_dict[i] += 1
        end
    end
end
for i in 15:-1:1
    i_pad = lpad(i, 2)
    println("$(i_pad)+ muts = $(number_plus_ct_dict[i])")
end

########## Muts in at least X EPCI seqs ##########
15+ muts = 735
14+ muts = 787
13+ muts = 851
12+ muts = 929
11+ muts = 1038
10+ muts = 1151
 9+ muts = 1281
 8+ muts = 1470
 7+ muts = 1725
 6+ muts = 2005
 5+ muts = 2427
 4+ muts = 3081
 3+ muts = 4134
 2+ muts = 6385
 1+ muts = 12340


In [185]:
println("########## Pos_Only Muts in at least X EPCI seqs ##########")
number_plus_ct_pos_only_dict = Dict{Int, Int}(i=>0 for i in 1:15)
for (mut, ct) in AA_muts_ct_pos_only_no_dels
    for i in 1:15
        if ct ≥ i
            number_plus_ct_pos_only_dict[i] += 1
        end
    end
end
for i in 15:-1:1
    i_pad = lpad(i, 2)
    println("$(i_pad)+ muts = $(number_plus_ct_pos_only_dict[i])")
end

########## Pos_Only Muts in at least X EPCI seqs ##########
15+ muts = 822
14+ muts = 880
13+ muts = 955
12+ muts = 1036
11+ muts = 1141
10+ muts = 1234
 9+ muts = 1367
 8+ muts = 1540
 7+ muts = 1767
 6+ muts = 2021
 5+ muts = 2387
 4+ muts = 2857
 3+ muts = 3479
 2+ muts = 4474
 1+ muts = 5997


In [174]:
### Making: mut_BALctAvg_BALratio_totalSeqs_totAAsubRatio__min5_dict_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2 = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 5
        mutset = Set([mut])
        mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(mutset, BAL_mut_list_2025_12_08, empty_string_set)
        mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2[mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
    end
end
########################################################################################################
########################################################################################################

2026_01_29__1028PM


In [220]:
### Fx: finding_BAL_mutset_ratio_percentile_v2 + finding_BAL_mutset_ratio_percentile_no_print_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["ORF7b:I2-", "N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_v2(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio, ref_total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_Xplus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ min_mut_ct && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_Xplus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_Xplus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_Xplus_muts))
    println("$(AAmutset_name) [$(ref_BAL_mut_seq_avg)] BALmuts/seq percentile = $(percentile_mut_ref)%")
    print("\n"^1)
    println("$(AAmutset_name) [$(ref_BAL_mutseq_to_EPCI_ratio)] BALmutsRatio percentile = $(percentile_ratio_ref)%")
    print("\n"^1)
    return percentile_ratio_ref, ref_BAL_mutseq_to_EPCI_ratio, ref_BAL_mut_seq_avg, ref_total_mut_seqs, ref_tot_AAsub_ratio
end
############################################################################################################
############################################################################################################
### Fx: finding_BAL_mutset_ratio_percentile_no_print_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_no_print_v2(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio, ref_total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_Xplus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ min_mut_ct && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_Xplus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_Xplus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_Xplus_muts))
    return percentile_ratio_ref, ref_BAL_mutseq_to_EPCI_ratio, ref_BAL_mut_seq_avg, ref_total_mut_seqs, ref_tot_AAsub_ratio
end
############################################################################################################
############################################################################################################  

2026_01_30__418PM
2026_01_30__418PM


finding_BAL_mutset_ratio_percentile_no_print_v2 (generic function with 2 methods)

In [231]:
### Fx: finding_BAL_mutset_ratio_percentile_v1 + finding_BAL_mutset_ratio_percentile_no_print_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_v1(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio, ref_total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_Xplus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ min_mut_ct && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_Xplus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_Xplus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_Xplus_muts))
    println("$(AAmutset_name) [$(ref_BAL_mut_seq_avg)] BALmuts/seq percentile = $(percentile_mut_ref)%")
    print("\n"^1)
    println("$(AAmutset_name) [$(ref_BAL_mutseq_to_EPCI_ratio)] BALmutsRatio percentile = $(percentile_ratio_ref)%")
    print("\n"^1)
    return percentile_ratio_ref, ref_BAL_mutseq_to_EPCI_ratio, ref_BAL_mut_seq_avg, ref_total_mut_seqs, ref_tot_AAsub_ratio
end
############################################################################################################
############################################################################################################
### Fx: finding_BAL_mutset_ratio_percentile_no_print_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_no_print_v1(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio, ref_total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_Xplus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ min_mut_ct && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_Xplus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_Xplus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_Xplus_muts))
    return percentile_ratio_ref, ref_BAL_mutseq_to_EPCI_ratio, ref_BAL_mut_seq_avg, ref_total_mut_seqs, ref_tot_AAsub_ratio
end
############################################################################################################
############################################################################################################  

2026_01_30__431PM
2026_01_30__431PM


finding_BAL_mutset_ratio_percentile_no_print_v1 (generic function with 2 methods)

In [217]:
### OLD   finding_BAL_mutset_ratio_percentile + finding_BAL_mutset_ratio_percentile_no_print_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_v1(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio = avg_bal_muts_per_seq_no_print_v1(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_5plus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ 5 && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_5plus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio = avg_bal_muts_per_seq_no_print_v1(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_5plus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_5plus_muts))
    println("$(AAmutset_name) [$(ref_BAL_mut_seq_avg)] BALmuts/seq percentile = $(percentile_mut_ref)%")
    print("\n"^1)
    println("$(AAmutset_name) [$(ref_BAL_mutseq_to_EPCI_ratio)] BALmutsRatio percentile = $(percentile_ratio_ref)%")
    print("\n"^1)
end
############################################################################################################
############################################################################################################  
### Finding BAL-mut total percentile
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
function finding_BAL_mutset_ratio_percentile_no_print_v1(AAmutset::Set{String}, AAmutset_name::String, EPCI_BALmut_list::Set{String}, excluded_seqs::Set{String}, min_mut_ct::Int)
    ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio, ref_tot_AAsub_ratio = avg_bal_muts_per_seq_no_print_v1(AAmutset, EPCI_BALmut_list, excluded_seqs)
    total_5plus_muts = 0
    atLeast_mutSet_BALmut_ct = 0
    atLeast_mutSet_BAL_ratio = 0
    total_unique_chr_seqs = length(all_unique_chr_seqs_set) - 1
    for (mut, ct) in AA_muts_ct_no_dels
        if ct ≥ 5 && !(mut in Double_N_ORF9b_muts) && !(mut in AAmutset)
            total_5plus_muts += 1
            mutset = Set([mut])
            BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, tot_AAsub_ratio = avg_bal_muts_per_seq_no_print_v1(mutset, EPCI_BALmut_list, excluded_seqs)
            if BAL_mutseq_to_EPCI_ratio > ref_BAL_mutseq_to_EPCI_ratio
                atLeast_mutSet_BAL_ratio += 1
            end
            if BAL_mut_seq_avg > ref_BAL_mut_seq_avg
                atLeast_mutSet_BALmut_ct += 1
            end 
    #############
        end
    end
    percentile_mut_ref = round(digits=2, 100*(1 - atLeast_mutSet_BALmut_ct/total_5plus_muts))
    percentile_ratio_ref = round(digits=2, 100*(1 - atLeast_mutSet_BAL_ratio/total_5plus_muts))
    return percentile_ratio_ref, ref_BAL_mut_seq_avg, ref_BAL_mutseq_to_EPCI_ratio
end
############################################################################################################
############################################################################################################  

2026_01_30__409PM
2026_01_30__409PM


finding_BAL_mutset_ratio_percentile_no_print_v1 (generic function with 1 method)

In [ ]:
### BAL_mut_EPCI_ratio_dict_min5_v2 for all muts with ct ≥5 in EPCI
ct_finished = 0
BAL_mut_EPCI_ratio_dict_min5_v2 = Dict{String, Tuple{Float64, Float64, Float64, Int64, Float64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 5 && !(mut in Double_N_ORF9b_muts)
        mutset = Set([mut])
        mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs, totAAsub_to_EPCI_ratio = finding_BAL_mutset_ratio_percentile_no_print_v2(mutset, "$(mut)", BAL_mut_list_2025_12_08, empty_string_set)
        BAL_mut_EPCI_ratio_dict_v2[mut] = (mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs)
        ct_finished += 1
        if ct_finished%50 = 0
            println("$(ct_finished) seqs finished")
        end
        if mut_ratio_percentile ≥ 99
            mutpad = rpad(mut, 12)
            mut_ratio_percentile_pad = lpad(mut_ratio_percentile, 6)
            println("$(mutpad) Ratio Percentile = $(mut_ratio_percentile_pad)% | BAL Ratio = $(mut_BAL_ratio) | Avg BAL Muts = $(mut_avg_BAL_ct) | Mut Seq Ct = $() | TotAASub_EPCI_Ratio = $(totAAsub_to_EPCI_ratio)")
        end
    end
end
##############################################################################################################
##############################################################################################################

In [ ]:
### BAL_mut_EPCI_ratio_dict_min10_v2 for all muts with ct ≥10 in EPCI
ct_finished = 0
BAL_mut_EPCI_ratio_dict_min10_v2 = Dict{String, Tuple{Float64, Float64, Float64, Int64, Float64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 10 && !(mut in Double_N_ORF9b_muts)
        mutset = Set([mut])
        mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs, totAAsub_to_EPCI_ratio = finding_BAL_mutset_ratio_percentile_no_print_v2(mutset, "$(mut)", BAL_mut_list_2025_12_08, empty_string_set)
        BAL_mut_EPCI_ratio_dict_v2[mut] = (mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs, totAAsub_to_EPCI_ratio)
        ct_finished += 1
        if ct_finished%50 = 0
            println("$(ct_finished) seqs finished")
        end
        if mut_ratio_percentile ≥ 99
            mutpad = rpad(mut, 12)
            mut_ratio_percentile_pad = lpad(mut_ratio_percentile, 6)
            println("$(mutpad) Ratio Percentile = $(mut_ratio_percentile_pad)% | BAL Ratio = $(mut_BAL_ratio) | Avg BAL Muts = $(mut_avg_BAL_ct) | Mut Seq Ct = $() | TotAASub_EPCI_Ratio = $(totAAsub_to_EPCI_ratio)")
        end
    end
end
##############################################################################################################
##############################################################################################################

In [157]:
### BAL_mut_EPCI_ratio_dict_min5_v1  (for all muts with ct ≥5 in EPCI)
ct_finished = 0
BAL_mut_EPCI_ratio_dict_min5_v1 = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 5 && !(mut in Double_N_ORF9b_muts)
        mutset = Set([mut])
        mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs = finding_BAL_mutset_ratio_percentile_no_print_v1(mutset, "$(mut)", BAL_mut_list_2025_12_08, empty_string_set)
        BAL_mut_EPCI_ratio_dict_v1[mut] = (mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs)
        ct_finished += 1
        if ct_finished%50 = 0
            println("$(ct_finished) seqs finished")
        end
        if mut_ratio_percentile ≥ 99
            mutpad = rpad(mut, 12)
            mut_ratio_percentile_pad = lpad(mut_ratio_percentile, 6)
            println("$(mutpad) Ratio Percentile = $(mut_ratio_percentile_pad)% | BAL Ratio = $(mut_BAL_ratio) | Avg BAL Muts = $(mut_avg_BAL_ct) | Mut Seq Ct = $() | ")
        end
    end
end
##############################################################################################################
##############################################################################################################

M:C64S       Ratio Percentile =  100.0%
M:A188T      Ratio Percentile =  99.83%
ORF1b:I1181T Ratio Percentile =  99.58%
M:F28L       Ratio Percentile =  99.54%
ORF1a:A3023V Ratio Percentile =  99.71%
M:N3S        Ratio Percentile =  99.87%
S:I693V      Ratio Percentile =  99.71%
ORF1a:S3950F Ratio Percentile =  99.16%
M:S173P      Ratio Percentile =  99.16%
ORF9b:Q77*   Ratio Percentile =  99.75%
M:T77N       Ratio Percentile =  99.29%
M:N3K        Ratio Percentile =  99.37%
S:A944T      Ratio Percentile =  99.92%
ORF1b:A1302V Ratio Percentile =  99.46%
N:S416L      Ratio Percentile =  99.33%
S:L1186F     Ratio Percentile =   99.2%
M:V10I       Ratio Percentile =  99.25%
ORF1a:F1248S Ratio Percentile =  99.16%
ORF1a:L3123F Ratio Percentile =  99.16%
ORF1a:A3049V Ratio Percentile =  99.79%
ORF9b:M1T    Ratio Percentile =  99.41%
N:T417I      Ratio Percentile =  99.54%
S:A647T      Ratio Percentile =  99.62%
N:P80L       Ratio Percentile =  99.92%
M:S4P        Ratio Percentile =  99.96%


In [ ]:
### BAL_mut_EPCI_ratio_dict_min5_v1 for all muts with ct ≥5 in EPCI
ct_finished = 0
BAL_mut_EPCI_ratio_dict_min5_v1 = Dict{String, Tuple{Float64, Float64, Float64, Int64, Float64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 5 && !(mut in Double_N_ORF9b_muts)
        mutset = Set([mut])
        mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs, totAAsub_to_EPCI_ratio = finding_BAL_mutset_ratio_percentile_no_print_v1(mutset, "$(mut)", BAL_mut_list_2025_12_08, empty_string_set)
        BAL_mut_EPCI_ratio_dict_v1[mut] = (mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs)
        ct_finished += 1
        if ct_finished%50 = 0
            println("$(ct_finished) seqs finished")
        end
        if mut_ratio_percentile ≥ 99
            mutpad = rpad(mut, 12)
            mut_ratio_percentile_pad = lpad(mut_ratio_percentile, 6)
            println("$(mutpad) Ratio Percentile = $(mut_ratio_percentile_pad)% | BAL Ratio = $(mut_BAL_ratio) | Avg BAL Muts = $(mut_avg_BAL_ct) | Mut Seq Ct = $() | TotAASub_EPCI_Ratio = $(totAAsub_to_EPCI_ratio)")
        end
    end
end
##############################################################################################################
##############################################################################################################

In [ ]:
### BAL_mut_EPCI_ratio_dict_min10_v1 for all muts with ct ≥10 in EPCI
ct_finished = 0
BAL_mut_EPCI_ratio_dict_min10_v1 = Dict{String, Tuple{Float64, Float64, Float64, Int64, Float64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 10 && !(mut in Double_N_ORF9b_muts)
        mutset = Set([mut])
        mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs, totAAsub_to_EPCI_ratio = finding_BAL_mutset_ratio_percentile_no_print_v1(mutset, "$(mut)", BAL_mut_list_2025_12_08, empty_string_set)
        BAL_mut_EPCI_ratio_dict_v1[mut] = (mut_ratio_percentile, mut_BAL_ratio, mut_avg_BAL_ct, mut_total_seqs)
        ct_finished += 1
        if ct_finished%50 = 0
            println("$(ct_finished) seqs finished")
        end
        if mut_ratio_percentile ≥ 99
            mutpad = rpad(mut, 12)
            mut_ratio_percentile_pad = lpad(mut_ratio_percentile, 6)
            println("$(mutpad) Ratio Percentile = $(mut_ratio_percentile_pad)% | BAL Ratio = $(mut_BAL_ratio) | Avg BAL Muts = $(mut_avg_BAL_ct) | Mut Seq Ct = $() | TotAASub_EPCI_Ratio = $(totAAsub_to_EPCI_ratio)")
        end
    end
end
##############################################################################################################
##############################################################################################################

In [113]:
### Finding BAL-mut total percentile
atLeast_240_avg = 0
atLeast_261_avg = 0
atLeast_298_avg = 0
atLeast_340_avg = 0
atLeast_231_ratio = 0
atLeast_253_ratio = 0
atLeast_289_ratio = 0
atLeast_327_ratio = 0
total_5plus_muts = 0
tot_seqs_uniq_chr = length(all_unique_chr_seqs_set)
Double_N_ORF9b_muts = Set(["N:N4I", "N:N4T", "N:N4S", "N:N4K", "N:N4K", "N:G5*", "N:G5R", "N:G5R", "N:G5V", "N:G5A", "N:G5E", "N:P6T", "N:P6A", "N:P6L", "N:P6H", "N:P6R", "N:Q7L", "N:Q7P", "N:Q7R", "N:Q7H", "N:Q7H", "N:N8Y", "N:N8H", "N:N8I", "N:N8T", "N:N8S", "N:N8K", "N:N8K", "N:Q9E", "N:Q9L", "N:Q9P", "N:Q9R", "N:Q9H", "N:Q9H", "N:R10G", "N:R10L", "N:R10P", "N:R10Q", "N:N11Y", "N:N11H", "N:N11I", "N:N11T", "N:N11S", "N:N11K", "N:N11K", "N:A12S", "N:A12P", "N:A12T", "N:A12V", "N:A12E", "N:A12G", "N:P13T", "N:P13A", "N:P13L", "N:L13P", "N:L13I", "N:L13F", "N:L13S", "N:P13H", "N:P13R", "N:R14L", "N:R14P", "N:R14H", "N:I15N", "N:I15S", "N:I15M", "N:T16S", "N:T16P", "N:T16M", "N:T16K", "N:T16R", "N:F17Y", "N:F17C", "N:F17L", "N:F17L", "N:G18C", "N:G18R", "N:G18V", "N:G18A", "N:G18D", "N:G19V", "N:G19A", "N:G19E", "N:P20T", "N:P20A", "N:P20L", "N:P20H", "N:P20R", "N:S21L", "N:S21*", "N:S21*", "N:D22Y", "N:D22H", "N:D22V", "N:D22A", "N:D22G", "N:D22E", "N:D22E", "N:S23A", "N:S23L", "N:S23*", "N:S23*", "N:T24S", "N:T24P", "N:T24N", "N:T24S", "N:G25V", "N:G25A", "N:G25D", "N:S26I", "N:S26T", "N:S26N", "N:S26R", "N:S26R", "N:N27I", "N:N27T", "N:N27S", "N:N27K", "N:N27K", "N:Q28L", "N:Q28R", "N:Q28H", "N:Q28H", "N:N29Y", "N:N29H", "N:N29I", "N:N29T", "N:N29S", "N:N29K", "N:N29K", "N:G30*", "N:G30R", "N:G30R", "N:G30V", "N:G30A", "N:G30E", "N:E31*", "N:E31Q", "N:E31V", "N:E31A", "N:E31G", "N:E31D", "N:E31D", "N:R32S", "N:R32G", "N:R32L", "N:R32P", "N:R32H", "N:S33I", "N:S33T", "N:S33N", "N:S33R", "N:S33R", "N:G34V", "N:G34A", "N:G34E", "N:A35V", "N:A35E", "N:A35G", "N:R36L", "N:R36P", "N:R36Q", "N:S37T", "N:S37A", "N:S37L", "N:S37*", "N:S37*", "N:K38*", "N:K38Q", "N:K38I", "N:K38T", "N:K38R", "N:K38N", "N:K38N", "N:Q39K", "N:Q39E", "N:Q39L", "N:Q39P", "N:Q39R", "N:Q39H", "N:Q39H", "N:R40S", "N:R40G", "N:R40L", "N:R40P", "N:R40H", "N:R41L", "N:R41P", "N:R41Q", "N:P42L", "N:P42H", "N:P42R", "N:Q43L", "N:Q43P", "N:Q43R", "N:Q43H", "N:Q43H", "N:G44C", "N:G44R", "N:G44V", "N:G44A", "N:G44D", "N:L45S", "N:L45*", "N:L45*", "N:L45F", "N:L45F", "N:P46T", "N:P46A", "N:P46L", "N:P46H", "N:P46R", "N:N47I", "N:N47T", "N:N47S", "N:N47K", "N:N47K", "N:N48D", "N:N48I", "N:N48T", "N:N48S", "N:N48K", "N:N48K", "N:T49A", "N:T49N", "N:T49S", "N:A50V", "N:A50E", "N:A50G", "N:S51F", "N:S51Y", "N:S51C", "N:W52L", "N:W52S", "N:W52*", "N:W52C", "N:W52C", "N:W52*", "N:F53S", "N:F53Y", "N:F53C", "N:F53L", "N:F53L", "N:T54I", "N:T54N", "N:T54S", "N:A55V", "N:A55D", "N:A55G", "N:L56P", "N:L56H", "N:L56R", "N:T57I", "N:T57N", "N:T57S", "N:Q58L", "N:Q58P", "N:Q58R", "N:Q58H", "N:Q58H", "N:H59N", "N:H59D", "N:H59L", "N:H59P", "N:H59R", "N:H59Q", "N:H59Q", "N:G60C", "N:G60R", "N:G60S", "N:G60V", "N:G60A", "N:G60D", "N:K61M", "N:K61R", "N:K61N", "N:K61N", "N:E62*", "N:E62Q", "N:E62V", "N:E62A", "N:E62G", "N:E62D", "N:E62D", "N:D63Y", "N:D63H", "N:D63V", "N:D63A", "N:D63G", "N:D63E", "N:D63E", "N:L64H", "N:L64R", "N:K65*", "N:K65Q", "N:K65I", "N:K65T", "N:K65R", "N:K65N", "N:K65N", "N:F66I", "N:F66V", "N:F66S", "N:F66Y", "N:F66C", "N:F66L", "N:F66L", "N:P67L", "N:P67H", "N:P67R", "N:R68L", "N:R68P", "N:R68Q", "N:G69*", "N:G69R", "N:G69V", "N:G69A", "N:G69E", "N:Q70K", "N:Q70E", "N:Q70L", "N:Q70P", "N:Q70R", "N:Q70H", "N:Q70H", "N:G71C", "N:G71R", "N:G71V", "N:G71A", "N:G71D", "N:V72A", "N:V72D", "N:V72G", "N:P73T", "N:P73A", "N:P73L", "N:P73Q", "N:P73R", "N:I74F", "N:I74L", "N:I74N", "N:I74S", "N:I74M", "N:N75Y", "N:N75H", "N:N75I", "N:N75T", "N:N75S", "N:N75K", "N:N75K", "N:T76I", "N:T76N", "N:T76S", "N:N77I", "N:N77T", "N:N77S", "N:N77K", "N:N77K", "N:S78G", "N:S78I", "N:S78T", "N:S78N", "N:S78R", "N:S78R", "N:S79I", "N:S79T", "N:S79N", "N:S79R", "N:S79R", "N:P80L", "N:P80Q", "N:P80R", "N:D81Y", "N:D81H", "N:D81V", "N:D81A", "N:D81G", "N:D81E", "N:D81E", "N:D82Y", "N:D82H", "N:D82N", "N:D82V", "N:D82A", "N:D82G", "N:D82E", "N:D82E", "N:Q83L", "N:Q83P", "N:Q83R", "N:Q83H", "N:Q83H", "N:I84F", "N:I84L", "N:I84N", "N:I84S", "N:I84M", "N:G85C", "N:G85R", "N:G85V", "N:G85A", "N:G85D", "N:Y86F", "N:Y86S", "N:Y86C", "N:Y86*", "N:Y86*", "N:Y87F", "N:Y87S", "N:Y87C", "N:Y87*", "N:Y87*", "N:R88L", "N:R88P", "N:R88Q", "N:R89*", "N:R89I", "N:R89T", "N:R89K", "N:R89S", "N:R89S", "N:A90S", "N:A90P", "N:A90D", "N:A90G", "N:T91I", "N:T91N", "N:T91S", "N:R92I", "N:R92T", "N:R92K", "N:R92S", "N:R92S", "N:R93G", "N:R93L", "N:R93P", "N:R93Q", "N:I94F", "N:I94L", "N:I94T", "N:I94N", "N:I94S", "N:I94M", "N:R95S", "N:R95G", "N:R95L", "N:R95P", "N:R95H", "N:G96V", "N:G96A", "N:G96D", "N:G97V", "N:G97A", "N:G97D", "N:D98V", "N:D98A", "N:D98G", "N:D98E", "N:D98E", "N:G99V", "N:G99A", "N:G99D", "N:K100I", "N:K100T", "N:K100R", "N:K100N", "N:K100N", "N:M101L", "N:M101L", "N:M101T", "N:M101K", "N:M101R", "N:M101I", "N:M101I", "N:K102*", "N:K102Q", "N:K102E"])    
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 8 && !(mut in Double_N_ORF9b_muts)
        total_5plus_muts += 1
        mutset = Set([mut])
        BAL_mut_seq_avg, BAL_mutseq_to_EPCI_ratio, AAsub_ratio = avg_bal_muts_per_seq_no_print_v1(mutset, BAL_mut_list_2025_12_08, ORF7a_extension_excluded_seqs)
        if BAL_mut_seq_avg ≥ 2.40
            atLeast_240_avg += 1
        end
        if BAL_mut_seq_avg ≥ 2.61
            atLeast_261_avg += 1
        end
        if BAL_mut_seq_avg ≥ 2.98
            atLeast_298_avg += 1
        end
        if BAL_mut_seq_avg ≥ 3.40
            atLeast_340_avg += 1
        end
#############
        if BAL_mutseq_to_EPCI_ratio ≥ 2.31
            atLeast_231_ratio += 1
        end
        if BAL_mutseq_to_EPCI_ratio ≥ 2.53
            atLeast_253_ratio += 1
        end
        if BAL_mutseq_to_EPCI_ratio ≥ 2.89
            atLeast_289_ratio += 1
        end
        if BAL_mutseq_to_EPCI_ratio ≥ 3.27
            atLeast_327_ratio += 1
        end
        if BAL_mutseq_to_EPCI_ratio ≥ 4
            println("$(mut), $(BAL_mutseq_to_EPCI_ratio)")
        end
    end
end
percentile_240 = round(digits=2, 100*(1 - atLeast_240_avg/tot_seqs_uniq_chr))
percentile_261 = round(digits=2, 100*(1 - atLeast_261_avg/tot_seqs_uniq_chr))
percentile_298 = round(digits=2, 100*(1 - atLeast_298_avg/tot_seqs_uniq_chr))
percentile_340 = round(digits=2, 100*(1 - atLeast_340_avg/tot_seqs_uniq_chr))
percentile_231_ratio = round(digits=2, 100*(1 - atLeast_231_ratio/tot_seqs_uniq_chr))
percentile_253_ratio = round(digits=2, 100*(1 - atLeast_253_ratio/tot_seqs_uniq_chr))
percentile_289_ratio = round(digits=2, 100*(1 - atLeast_289_ratio/tot_seqs_uniq_chr))
percentile_327_ratio = round(digits=2, 100*(1 - atLeast_327_ratio/tot_seqs_uniq_chr))
println("2.40 BALmuts/seq percentile = $(percentile_240)%")
println("2.61 BALmuts/seq percentile = $(percentile_261)%")
println("2.98 BALmuts/seq percentile = $(percentile_298)%")
println("3.40 BALmuts/seq percentile = $(percentile_340)%")
print("\n"^1)
println("2.31 BALmutsRatio percentile = $(percentile_231_ratio)%")
println("2.53 BALmutsRatio percentile = $(percentile_253_ratio)%")
println("2.89 BALmutsRatio percentile = $(percentile_289_ratio)%")
println("3.27 BALmutsRatio percentile = $(percentile_327_ratio)%")
print("\n"^1)
############################################################################################################
############################################################################################################  

LoadError: BoundsError: attempt to access Tuple{Float64, Float64} at index [3]

In [167]:
### Making: mut_BALctAvg_BALratio_totalSeqs_totAAsubRatio__min10_dict_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2 = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 10
        mutset = Set([mut])
        mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v2(mutset, BAL_mut_list_2025_12_08, empty_string_set)
        mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2[mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
    end
end
########################################################################################################
########################################################################################################

In [197]:
### CSV file for mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort = sort(collect(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2), by = x -> x[2][2], rev=true)
println(typeof(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort))
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][1])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][2])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][2][1])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][2][2])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][2][3])
println(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[1][2][4])
BAL_ratio_rank_dict_v2 = Dict{String, Int}()
BAL_ratio_percentile_dict_v2 = Dict{String, Float64}()
total_5plus_muts_v2 = length(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort)
println("Total 5+ muts = $(total_5plus_muts)")
##############################################################################
df_BALct_BALratio_totAAratio_seqTot_v2 = DataFrame(
    Mutation = String[],
    BAL_avg = Float64[],
    BAL_ratio = Float64[],
    Ratio_Rank = Int64[],
    Percentile = Float64[],
    AAsub_Ratio = Float64[],
    TotalSeqs = Int64[])
##############################################################################
for i in 1:total_5plus_muts_v2
    mut = mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[i][1]
    AvgBALct = mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[i][2][1]
    BALratio = mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[i][2][2]
    AAsubRatio = mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[i][2][3]
    seqtot = mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort[i][2][4]
    BAL_ratio_rank_dict_v2[mut] = i
    percentile = round(digits=3, 100 - 100*(i-1)/total_5plus_muts)
    BAL_ratio_percentile_dict_v2[mut] = percentile
    push!(df_BALct_BALratio_totAAratio_seqTot_v2, (mut, AvgBALct, BALratio, i, percentile, AAsubRatio, seqtot))
end
### Below: Old version (keeping until sure new version works properly) 
#for (mut, ct__ratio__totAAratio__totseqs) in mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v2_RatioSort
#    AvgBALct = ct__ratio__totAAratio__totseqs[1]
#    BALratio = ct__ratio__totAAratio__totseqs[2]
#    AAsubRatio = ct__ratio__totAAratio__totseqs[3]
#    seqtot = ct__ratio__totAAratio__totseqs[4]
#    push!(df_BALct_BALratio_totAAratio_seqTot, (mut, AvgBALct, BALratio, AAsubRatio, seqtot))
#end
################################################################################################################################################################
CSV.write("df_BALct_BALratio_totAAratio_seqTot__min5__$(date_hour)_v2.csv", df_BALct_BALratio_totAAratio_seqTot_v2)
################################################################################################################################################################
################################################################################################################################################################

2026_01_30__1215PM
Vector{Pair{String, Tuple{Float64, Float64, Float64, Int64}}}
"M:C64S" => (6.2, 6.26, 1.7, 5)
M:C64S
(6.2, 6.26, 1.7, 5)
6.2
6.26
1.7
5
Total 5+ muts = 2427


"df_BALct_BALratio_totAAratio_seqTot__min5__2026_01_30_12PM_v2.csv"

In [176]:
### CSV file for mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2_RatioSort = sort(collect(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2), by = x -> x[2][2], rev=true)
df_BALct_BALratio_totAAratio_seqTot = DataFrame(
    Mut = String[],
    BAL_avg = Float64[],
    BAL_ratio = Float64[],
    AAsubRatio = Float64[],
    TotalSeqs = Int64[])
for (mut, ct__ratio__totAAratio__totseqs) in mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v2_RatioSort
    AvgBALct = ct__ratio__totAAratio__totseqs[1]
    BALratio = ct__ratio__totAAratio__totseqs[2]
    AAsubRatio = ct__ratio__totAAratio__totseqs[3]
    seqtot = ct__ratio__totAAratio__totseqs[4]
    push!(df_BALct_BALratio_totAAratio_seqTot, (mut, AvgBALct, BALratio, AAsubRatio, seqtot))
end
CSV.write("df_BALct_BALratio_totAAratio_seqTot__min10__$(date_hour)_v2.csv", df_BALct_BALratio_totAAratio_seqTot)
########################################################################################################
########################################################################################################

2026_01_29__1029PM


"df_BALct_BALratio_totAAratio_seqTot__min10__2026_01_29_10PM_v2.csv"

In [190]:
### Making: mut_BALctAvg_BALratio_totalSeqs_totAAsubRatio__min5_dict_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1 = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 5
        mutset = Set([mut])
        mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(mutset, BAL_mut_list_2025_12_08, empty_string_set)
        mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1[mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
    end
end
########################################################################################################
########################################################################################################

2026_01_29__1042PM


In [191]:
### Making: mut_BALctAvg_BALratio_totalSeqs_totAAsubRatio__min10_dict_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1 = Dict{String, Tuple{Float64, Float64, Float64, Int64}}()
for (mut, ct) in AA_muts_ct_no_dels
    if ct ≥ 10
        mutset = Set([mut])
        mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs = avg_bal_muts_per_seq_no_print_v1(mutset, BAL_mut_list_2025_12_08, empty_string_set)
        mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1[mut] = (mut_avgBALmuts, mutBALratio, totAAsubRatio, total_mut_seqs)
    end
end
########################################################################################################
########################################################################################################

2026_01_29__1042PM


In [192]:
### CSV file for mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1_RatioSort = sort(collect(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1), by = x -> x[2][2], rev=true)
df_BALct_BALratio_totAAratio_seqTot = DataFrame(
    Mut = String[],
    BAL_avg = Float64[],
    BAL_ratio = Float64[],
    AAsubRatio = Float64[],
    TotalSeqs = Int64[])
for (mut, ct__ratio__totAAratio__totseqs) in mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min5_dict_v1_RatioSort
    AvgBALct = ct__ratio__totAAratio__totseqs[1]
    BALratio = ct__ratio__totAAratio__totseqs[2]
    AAsubRatio = ct__ratio__totAAratio__totseqs[3]
    seqtot = ct__ratio__totAAratio__totseqs[4]
    push!(df_BALct_BALratio_totAAratio_seqTot, (mut, AvgBALct, BALratio, AAsubRatio, seqtot))
end
CSV.write("df_BALct_BALratio_totAAratio_seqTot__min5__$(date_hour)_v1.csv", df_BALct_BALratio_totAAratio_seqTot)
########################################################################################################
########################################################################################################

2026_01_29__1042PM


"df_BALct_BALratio_totAAratio_seqTot__min5__2026_01_29_10PM_v1.csv"

In [194]:
### CSV file for mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1
date_now = Dates.format(now(), "yyyy_mm_dd__IMMp"); println(date_now)
date_hour = Dates.format(now(), "yyyy_mm_dd_Ip")
mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1_RatioSort = sort(collect(mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1), by = x -> x[2][2], rev=true)
df_BALct_BALratio_totAAratio_seqTot = DataFrame(
    Mut = String[],
    BAL_avg = Float64[],
    BAL_ratio = Float64[],
    AAsubRatio = Float64[],
    TotalSeqs = Int64[])
for (mut, ct__ratio__totAAratio__totseqs) in mut__BALctAvg_BALratio_totAAsubRatio_totalSeqs__min10_dict_v1_RatioSort
    AvgBALct = ct__ratio__totAAratio__totseqs[1]
    BALratio = ct__ratio__totAAratio__totseqs[2]
    AAsubRatio = ct__ratio__totAAratio__totseqs[3]
    seqtot = ct__ratio__totAAratio__totseqs[4]
    push!(df_BALct_BALratio_totAAratio_seqTot, (mut, AvgBALct, BALratio, AAsubRatio, seqtot))
end
CSV.write("df_BALct_BALratio_totAAratio_seqTot__min10__$(date_hour)_v1.csv", df_BALct_BALratio_totAAratio_seqTot)
########################################################################################################
########################################################################################################

2026_01_29__1043PM


"df_BALct_BALratio_totAAratio_seqTot__min10__2026_01_29_10PM_v1.csv"